<a href="https://colab.research.google.com/github/onerospacetime/ettr-ctl-research/blob/main/Llama_experiment_%E2%80%94_Cell_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# ETTR-CTL-LLAMA — CELL 0
# Experiment Initialization + Computational Substrate Audit
#
# NEW CELL — do not remove previous cells.
#
# Governing principle:
#   CTL mathematics is authoritative.
#   Experimental operationalization must preserve the mathematics.
#   GPU/software limitations must NOT modify mathematical meaning.
#   Any genuine disparity between:
#       (1) CTL mathematics,
#       (2) experimental operationalization, and
#       (3) T4 numerical execution
#   is recorded as DECOHERENCE.
#
# This cell performs initialization and READ-ONLY environment
# diagnostics. It does NOT load model weights or begin the
# scientific experiment.
# ============================================================

import os
import sys
import json
import hashlib
import platform
import subprocess
import importlib.util
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import torch


# ------------------------------------------------------------
# 0. Experiment identity
# ------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
EXPERIMENT_VERSION = "v1.0"
MODEL_FAMILY = "Llama"
INTENDED_MODEL_ID = "meta-llama/Llama-3.2-3B"

# We intentionally do NOT assume that the checkpoint is already
# accessible, cached, or downloadable. That will be diagnosed.
MODEL_LOAD_PERFORMED = False

SEED = 42


# ------------------------------------------------------------
# 1. Governing scientific contract
# ------------------------------------------------------------

GOVERNING_PRINCIPLES = {
    "mathematical_priority":
        "CTL mathematical implications are authoritative and "
        "must not be altered to satisfy experimental or software constraints.",

    "operationalization_rule":
        "Experimental operationalization must preserve the mathematical "
        "meaning of Contextual Calculus and Contextual Transport Logic.",

    "numerical_execution_rule":
        "CUDA, PyTorch, Transformer implementations, and T4 execution "
        "are implementation substrates, not authorities over CTL mathematics.",

    "decoherence_rule":
        "A genuine disparity between CTL mathematics, experimental "
        "operationalization, software translation, or T4 numerical "
        "execution is recorded as experimental decoherence rather than "
        "used to modify the mathematical specification.",

    "diagnostic_rule":
        "No major computation may rely on an unverified dependency, "
        "schema, checkpoint, hook, dtype, device, or software capability.",

    "model_architecture_rule":
        "Llama sector definitions must be derived from Llama's actual "
        "architecture and verified implementation rather than copied "
        "from GPT-2 hook assumptions.",

    "test_firewall":
        "Test data must not participate in fitting, calibration, "
        "model selection, or parameter selection.",

    "semantic_rule":
        "A numerically executable operation is not thereby established "
        "as a valid CTL operation; mathematical, operational, and "
        "numerical validity must be separately demonstrated."
}


# ------------------------------------------------------------
# 2. Reproducibility seed
# ------------------------------------------------------------

os.environ["PYTHONHASHSEED"] = str(SEED)

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Determinism is recorded rather than silently imposed.
# Some CUDA operations may have deterministic restrictions.
torch.backends.cudnn.benchmark = False


# ------------------------------------------------------------
# 3. Experiment filesystem
# ------------------------------------------------------------

ROOT = Path("/content/ettr_ctl_llama")

RESULTS_DIR = ROOT / "results"
MODELS_DIR = ROOT / "models"
LOGS_DIR = ROOT / "logs"
CHECKPOINTS_DIR = ROOT / "checkpoints"
FIGURES_DIR = ROOT / "figures"
CONTRACTS_DIR = ROOT / "contracts"

for directory in [
    ROOT,
    RESULTS_DIR,
    MODELS_DIR,
    LOGS_DIR,
    CHECKPOINTS_DIR,
    FIGURES_DIR,
    CONTRACTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 4. Basic software environment
# ------------------------------------------------------------

def package_version(package_name):
    try:
        module = __import__(package_name)
        return getattr(module, "__version__", "VERSION_NOT_EXPOSED")
    except Exception as exc:
        return f"UNAVAILABLE: {type(exc).__name__}: {exc}"


environment = {
    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "python":
        sys.version,

    "python_executable":
        sys.executable,

    "platform":
        platform.platform(),

    "machine":
        platform.machine(),

    "processor":
        platform.processor(),

    "pytorch_version":
        torch.__version__,

    "numpy_version":
        np.__version__,

    "transformers_installed":
        importlib.util.find_spec("transformers") is not None,

    "transformer_lens_installed":
        importlib.util.find_spec("transformer_lens") is not None,

    "accelerate_installed":
        importlib.util.find_spec("accelerate") is not None,

    "safetensors_installed":
        importlib.util.find_spec("safetensors") is not None,

    "huggingface_hub_installed":
        importlib.util.find_spec("huggingface_hub") is not None,
}


if environment["transformers_installed"]:
    environment["transformers_version"] = package_version("transformers")

if environment["transformer_lens_installed"]:
    environment["transformer_lens_version"] = package_version("transformer_lens")

if environment["accelerate_installed"]:
    environment["accelerate_version"] = package_version("accelerate")

if environment["safetensors_installed"]:
    environment["safetensors_version"] = package_version("safetensors")

if environment["huggingface_hub_installed"]:
    environment["huggingface_hub_version"] = package_version("huggingface_hub")


# ------------------------------------------------------------
# 5. CUDA / GPU audit
# ------------------------------------------------------------

gpu_audit = {
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_version": torch.version.cuda,
    "device_count": torch.cuda.device_count(),
    "devices": [],
}

if torch.cuda.is_available():

    for device_index in range(torch.cuda.device_count()):

        props = torch.cuda.get_device_properties(device_index)

        gpu_info = {
            "index": device_index,
            "name": torch.cuda.get_device_name(device_index),
            "compute_capability": (
                int(props.major),
                int(props.minor)
            ),
            "total_memory_bytes": int(props.total_memory),
            "total_memory_gib":
                float(props.total_memory / (1024 ** 3)),
            "multi_processor_count":
                int(props.multi_processor_count),
        }

        gpu_audit["devices"].append(gpu_info)


# ------------------------------------------------------------
# 6. Explicit T4 diagnosis
# ------------------------------------------------------------

t4_diagnostic = {
    "t4_detected": False,
    "device_index": None,
    "device_name": None,
    "compute_capability": None,
    "memory_gib": None,
}

for device in gpu_audit["devices"]:

    name = str(device["name"]).lower()

    if "t4" in name:

        t4_diagnostic = {
            "t4_detected": True,
            "device_index": device["index"],
            "device_name": device["name"],
            "compute_capability": device["compute_capability"],
            "memory_gib": device["total_memory_gib"],
        }

        break


# ------------------------------------------------------------
# 7. CUDA numerical sanity probe
#
# This is NOT a scientific CTL computation.
# It only verifies that the claimed CUDA device can execute
# elementary numerical operations.
# ------------------------------------------------------------

cuda_probe = {
    "performed": False,
    "status": "NOT_RUN",
}

if torch.cuda.is_available():

    try:

        device = torch.device("cuda:0")

        a = torch.tensor(
            [[1.0, 2.0], [3.0, 4.0]],
            dtype=torch.float32,
            device=device,
        )

        b = torch.tensor(
            [[4.0, 3.0], [2.0, 1.0]],
            dtype=torch.float32,
            device=device,
        )

        c = a @ b

        logical_a = torch.tensor(
            [True, False, True, False],
            dtype=torch.bool,
            device=device,
        )

        logical_b = torch.tensor(
            [True, True, False, False],
            dtype=torch.bool,
            device=device,
        )

        logical_and = torch.logical_and(logical_a, logical_b)
        logical_or = torch.logical_or(logical_a, logical_b)
        logical_not = torch.logical_not(logical_a)

        cuda_probe = {
            "performed": True,
            "status": "PASS",
            "device": str(device),
            "matmul_dtype": str(c.dtype),
            "matmul_shape": list(c.shape),
            "matmul_finite": bool(torch.isfinite(c).all().item()),
            "boolean_dtype": str(logical_and.dtype),
            "boolean_device":
                str(logical_and.device),
            "logical_and_result":
                logical_and.detach().cpu().tolist(),
            "logical_or_result":
                logical_or.detach().cpu().tolist(),
            "logical_not_result":
                logical_not.detach().cpu().tolist(),
            "cpu_roundtrip_used_for_execution":
                False,
        }

        del a, b, c
        del logical_a, logical_b
        del logical_and, logical_or, logical_not

        torch.cuda.empty_cache()

    except Exception as exc:

        cuda_probe = {
            "performed": True,
            "status": "FAIL",
            "error_type": type(exc).__name__,
            "error": str(exc),
        }


# ------------------------------------------------------------
# 8. Hugging Face / Llama checkpoint ACCESS DIAGNOSTIC
#
# This deliberately does NOT load the model.
# It determines whether the intended checkpoint can be queried.
# ------------------------------------------------------------

checkpoint_diagnostic = {
    "model_id": INTENDED_MODEL_ID,
    "package_available": environment["huggingface_hub_installed"],
    "api_query_performed": False,
    "status": "NOT_RUN",
}

if environment["huggingface_hub_installed"]:

    try:

        from huggingface_hub import HfApi

        api = HfApi()

        info = api.model_info(INTENDED_MODEL_ID)

        checkpoint_diagnostic = {
            "model_id": INTENDED_MODEL_ID,
            "package_available": True,
            "api_query_performed": True,
            "status": "ACCESSIBLE_METADATA",
            "model_id_returned": getattr(info, "id", None),
            "private": getattr(info, "private", None),
            "gated": getattr(info, "gated", None),
            "sha": getattr(info, "sha", None),
            "pipeline_tag": getattr(info, "pipeline_tag", None),
        }

    except Exception as exc:

        checkpoint_diagnostic = {
            "model_id": INTENDED_MODEL_ID,
            "package_available": True,
            "api_query_performed": True,
            "status": "ACCESS_DIAGNOSTIC_FAILED",
            "error_type": type(exc).__name__,
            "error": str(exc),
        }


# ------------------------------------------------------------
# 9. Local checkpoint/cache diagnosis
#
# No model loading is performed.
# ------------------------------------------------------------

local_model_diagnostic = {
    "models_directory": str(MODELS_DIR),
    "models_directory_exists": MODELS_DIR.exists(),
    "models_directory_entries": sorted(
        [p.name for p in MODELS_DIR.iterdir()]
    ) if MODELS_DIR.exists() else [],
}


# ------------------------------------------------------------
# 10. CTL mathematical authority declaration
#
# This is metadata only. It does not redefine CTL.
# ------------------------------------------------------------

ctl_execution_contract = {

    "authority":
        "Project CTL mathematical sources",

    "mathematical_objects_to_preserve": [
        "contexts",
        "context-indexed logical carriers",
        "partial admissibility",
        "logical transport",
        "transport composition",
        "three coupled transport sectors",
        "triadic admissibility",
        "logical coherence",
        "semantic realization",
        "global realization conditions",
    ],

    "forbidden_operational_shortcuts": [
        "replace contextual carriers with ordinary dataframe labels",
        "replace undefined with false",
        "define triadic admissibility from pairwise admissibility by assumption",
        "alter mathematical implications to satisfy GPU limitations",
        "alter CTL definitions to satisfy PyTorch limitations",
        "alter CTL definitions to fit Llama architecture",
        "interpret Boolean encoding alone as CTL",
    ],

    "decoherence_policy":
        "Any genuine mismatch between mathematical CTL specification, "
        "software operationalization, or T4 numerical execution must "
        "be recorded as DECOHERENCE and investigated without modifying "
        "the mathematical specification.",
}


# ------------------------------------------------------------
# 11. Scientific phase boundary
# ------------------------------------------------------------

phase_boundary = {
    "model_weights_loaded": False,
    "transformer_forward_pass": False,
    "hooks_registered": False,
    "ctl_runtime_executed": False,
    "logical_transport_executed": False,
    "triadic_admissibility_executed": False,
    "coherence_executed": False,
    "scientific_measurement_started": False,
    "test_data_used": False,
}


# ------------------------------------------------------------
# 12. Save initialization manifest
# ------------------------------------------------------------

initialization_manifest = {
    "experiment_id": EXPERIMENT_ID,
    "experiment_version": EXPERIMENT_VERSION,
    "model_family": MODEL_FAMILY,
    "intended_model_id": INTENDED_MODEL_ID,
    "seed": SEED,

    "environment": environment,
    "gpu_audit": gpu_audit,
    "t4_diagnostic": t4_diagnostic,
    "cuda_probe": cuda_probe,

    "checkpoint_diagnostic":
        checkpoint_diagnostic,

    "local_model_diagnostic":
        local_model_diagnostic,

    "governing_principles":
        GOVERNING_PRINCIPLES,

    "ctl_execution_contract":
        ctl_execution_contract,

    "phase_boundary":
        phase_boundary,

    "experiment_directories": {
        "root": str(ROOT),
        "results": str(RESULTS_DIR),
        "models": str(MODELS_DIR),
        "logs": str(LOGS_DIR),
        "checkpoints": str(CHECKPOINTS_DIR),
        "figures": str(FIGURES_DIR),
        "contracts": str(CONTRACTS_DIR),
    },

    "model_load_performed":
        MODEL_LOAD_PERFORMED,
}


manifest_path = (
    RESULTS_DIR /
    "llama_phase0_initialization_manifest.json"
)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(
        initialization_manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 13. Print diagnostic summary
# ------------------------------------------------------------

print("=" * 72)
print("ETTR-CTL-LLAMA — CELL 0")
print("INITIALIZATION + COMPUTATIONAL SUBSTRATE AUDIT")
print("=" * 72)

print(f"Experiment ID       : {EXPERIMENT_ID}")
print(f"Experiment version  : {EXPERIMENT_VERSION}")
print(f"Seed                : {SEED}")
print(f"Intended checkpoint : {INTENDED_MODEL_ID}")

print("\n--- SOFTWARE ---")
print(f"Python              : {platform.python_version()}")
print(f"PyTorch             : {torch.__version__}")
print(f"CUDA runtime        : {torch.version.cuda}")
print(f"Transformers        : "
      f"{environment.get('transformers_version', 'NOT VERIFIED')}")

print("\n--- GPU ---")
print(f"CUDA available      : {gpu_audit['cuda_available']}")
print(f"Device count        : {gpu_audit['device_count']}")

for device in gpu_audit["devices"]:
    print(
        f"GPU {device['index']}              : "
        f"{device['name']} | "
        f"CC {device['compute_capability']} | "
        f"{device['total_memory_gib']:.2f} GiB"
    )

print("\n--- T4 DIAGNOSTIC ---")
print(f"T4 detected         : {t4_diagnostic['t4_detected']}")

if t4_diagnostic["t4_detected"]:
    print(f"T4 compute capability: "
          f"{t4_diagnostic['compute_capability']}")
    print(f"T4 memory           : "
          f"{t4_diagnostic['memory_gib']:.2f} GiB")

print("\n--- CUDA NUMERICAL PROBE ---")
print(f"CUDA probe status   : {cuda_probe['status']}")

print("\n--- LLAMA CHECKPOINT ---")
print(f"Metadata diagnostic : {checkpoint_diagnostic['status']}")

if checkpoint_diagnostic.get("gated") is not None:
    print(f"Checkpoint gated    : "
          f"{checkpoint_diagnostic['gated']}")

print("\n--- SCIENTIFIC BOUNDARY ---")
print("Model loaded        : FALSE")
print("Scientific experiment started: FALSE")
print("CTL runtime executed: FALSE")
print("Test data used      : FALSE")

print("\n--- GOVERNING RULE ---")
print(
    "CTL mathematics is authoritative; implementation disparities "
    "are recorded as DECOHERENCE rather than used to modify CTL."
)

print("\nManifest saved:")
print(manifest_path)

print("=" * 72)
print("CELL 0 COMPLETE — NO MODEL WEIGHTS LOADED")
print("=" * 72)

ETTR-CTL-LLAMA — CELL 0
INITIALIZATION + COMPUTATIONAL SUBSTRATE AUDIT
Experiment ID       : ETTR-CTL-LLAMA-1
Experiment version  : v1.0
Seed                : 42
Intended checkpoint : meta-llama/Llama-3.2-3B

--- SOFTWARE ---
Python              : 3.13.15
PyTorch             : 2.11.0+cu128
CUDA runtime        : 12.8
Transformers        : 5.16.1

--- GPU ---
CUDA available      : True
Device count        : 1
GPU 0              : Tesla T4 | CC (7, 5) | 14.56 GiB

--- T4 DIAGNOSTIC ---
T4 detected         : True
T4 compute capability: (7, 5)
T4 memory           : 14.56 GiB

--- CUDA NUMERICAL PROBE ---
CUDA probe status   : PASS

--- LLAMA CHECKPOINT ---
Metadata diagnostic : ACCESSIBLE_METADATA
Checkpoint gated    : manual

--- SCIENTIFIC BOUNDARY ---
Model loaded        : FALSE
Scientific experiment started: FALSE
CTL runtime executed: FALSE
Test data used      : FALSE

--- GOVERNING RULE ---
CTL mathematics is authoritative; implementation disparities are recorded as DECOHERENCE rather

In [3]:
# ============================================================================
# ETTR-CTL-LLAMA — PRE-CELL-1 GPU RESIDENCY DIAGNOSTIC
# ============================================================================
# Purpose:
#   Diagnose unexpected CUDA memory residency before the Llama architecture
#   audit is allowed to proceed.
#
# Governing rule:
#   This cell is READ-ONLY with respect to the experimental state.
#   It does not load Llama weights, clear CUDA memory, delete objects,
#   reset the runtime, or alter the mathematical/experimental design.
#
# Reverse notebook indexing:
#   Cell 0 = most recently executed cell
#   Cell 1 = immediately previous cell
# ============================================================================

import gc
import os
import sys
import json
import traceback
import torch

print("=" * 76)
print("ETTR-CTL-LLAMA — PRE-CELL-1 GPU RESIDENCY DIAGNOSTIC")
print("=" * 76)

print(f"Python        : {sys.version.split()[0]}")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# ---------------------------------------------------------------------------
# 1. Basic CUDA allocator state
# ---------------------------------------------------------------------------

if torch.cuda.is_available():
    device = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device)

    allocated = torch.cuda.memory_allocated(device)
    reserved = torch.cuda.memory_reserved(device)
    max_allocated = torch.cuda.max_memory_allocated(device)
    max_reserved = torch.cuda.max_memory_reserved(device)

    print("\nGPU")
    print(f"  device index       : {device}")
    print(f"  device name        : {props.name}")
    print(f"  total VRAM         : {props.total_memory / 1024**3:.3f} GiB")

    print("\nPyTorch CUDA allocator")
    print(f"  allocated          : {allocated / 1024**2:.3f} MiB")
    print(f"  reserved           : {reserved / 1024**2:.3f} MiB")
    print(f"  max allocated      : {max_allocated / 1024**2:.3f} MiB")
    print(f"  max reserved       : {max_reserved / 1024**2:.3f} MiB")

    # -----------------------------------------------------------------------
    # 2. CUDA memory summary
    # -----------------------------------------------------------------------

    print("\nCUDA memory summary")
    print("-" * 76)
    print(torch.cuda.memory_summary(device=device, abbreviated=True))

else:
    print("\nCUDA is unavailable.")
    device = None
    allocated = 0
    reserved = 0
    max_allocated = 0
    max_reserved = 0

# ---------------------------------------------------------------------------
# 3. Search Python GC objects for CUDA tensors
# ---------------------------------------------------------------------------
#
# This does NOT delete anything.
# We inspect objects currently reachable by Python's garbage collector.
# ---------------------------------------------------------------------------

cuda_tensors = []

print("\nPython GC CUDA-tensor scan")
print("-" * 76)

try:
    gc.collect()

    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) and obj.is_cuda:
                cuda_tensors.append(obj)
        except Exception:
            pass

    print(f"CUDA tensors discovered : {len(cuda_tensors)}")

    total_tensor_bytes = 0

    for i, tensor in enumerate(cuda_tensors[:100]):
        try:
            nbytes = tensor.numel() * tensor.element_size()
            total_tensor_bytes += nbytes

            print(
                f"  [{i:03d}] "
                f"shape={tuple(tensor.shape)} "
                f"dtype={tensor.dtype} "
                f"device={tensor.device} "
                f"requires_grad={tensor.requires_grad} "
                f"bytes={nbytes:,}"
            )
        except Exception as exc:
            print(f"  [{i:03d}] <inspection failed: {exc}>")

    if len(cuda_tensors) > 100:
        print(f"  ... {len(cuda_tensors) - 100} additional CUDA tensors omitted")

    print(
        f"\nReachable CUDA tensor bytes: "
        f"{total_tensor_bytes / 1024**2:.3f} MiB"
    )

except Exception as exc:
    print("GC tensor scan failed:")
    traceback.print_exc()

# ---------------------------------------------------------------------------
# 4. Inspect common model/object names without assuming they exist
# ---------------------------------------------------------------------------

print("\nCommon notebook-object inspection")
print("-" * 76)

candidate_names = [
    "model",
    "llama",
    "llama_model",
    "transformer",
    "net",
    "tokenizer",
    "inputs",
    "outputs",
    "batch",
    "hidden_states",
    "config",
]

for name in candidate_names:
    if name not in globals():
        print(f"  {name:16s}: <not present>")
        continue

    try:
        obj = globals()[name]
        obj_type = type(obj).__name__

        if torch.is_tensor(obj):
            print(
                f"  {name:16s}: Tensor "
                f"shape={tuple(obj.shape)} "
                f"dtype={obj.dtype} "
                f"device={obj.device}"
            )

        elif isinstance(obj, torch.nn.Module):
            try:
                param_count = sum(p.numel() for p in obj.parameters())
            except Exception:
                param_count = "unknown"

            try:
                devices = sorted(
                    {
                        str(p.device)
                        for p in obj.parameters()
                        if torch.is_tensor(p)
                    }
                )
            except Exception:
                devices = "unknown"

            print(
                f"  {name:16s}: Module "
                f"type={obj_type} "
                f"parameters={param_count} "
                f"devices={devices}"
            )

        else:
            print(f"  {name:16s}: {obj_type}")

    except Exception as exc:
        print(f"  {name:16s}: <inspection failed: {exc}>")

# ---------------------------------------------------------------------------
# 5. Environment-level CUDA visibility
# ---------------------------------------------------------------------------

print("\nCUDA environment")
print("-" * 76)

for key in [
    "CUDA_VISIBLE_DEVICES",
    "PYTORCH_CUDA_ALLOC_CONF",
    "NVIDIA_VISIBLE_DEVICES",
]:
    print(f"  {key:28s}: {os.environ.get(key, '<unset>')}")

# ---------------------------------------------------------------------------
# 6. Experimental interpretation — diagnostic only
# ---------------------------------------------------------------------------

print("\nDiagnostic classification")
print("-" * 76)

if not torch.cuda.is_available():
    status = "CUDA_UNAVAILABLE"
elif allocated == 0 and reserved == 0 and len(cuda_tensors) == 0:
    status = "GPU_RESIDENCY_CLEAN"
elif len(cuda_tensors) > 0:
    status = "PYTHON_REACHABLE_CUDA_TENSORS_PRESENT"
elif reserved > 0:
    status = "CUDA_ALLOCATOR_RESERVED_MEMORY_PRESENT"
else:
    status = "GPU_RESIDENCY_PRESENT_SOURCE_UNRESOLVED"

print(f"  STATUS: {status}")

# ---------------------------------------------------------------------------
# 7. Persist diagnostic artifact
# ---------------------------------------------------------------------------

root = "/content/ettr_ctl_llama"
results_dir = os.path.join(root, "results")
os.makedirs(results_dir, exist_ok=True)

artifact = {
    "experiment_id": "ETTR-CTL-LLAMA-1",
    "diagnostic_id": "PRE-CELL-1-GPU-RESIDENCY",
    "purpose": "Read-only diagnosis of unexpected GPU memory residency before Llama architecture audit.",
    "cuda_available": bool(torch.cuda.is_available()),
    "device_index": int(device) if device is not None else None,
    "device_name": props.name if device is not None else None,
    "total_vram_bytes": int(props.total_memory) if device is not None else None,
    "memory_allocated_bytes": int(allocated),
    "memory_reserved_bytes": int(reserved),
    "max_memory_allocated_bytes": int(max_allocated),
    "max_memory_reserved_bytes": int(max_reserved),
    "reachable_cuda_tensor_count": len(cuda_tensors),
    "reachable_cuda_tensor_bytes": int(
        sum(
            t.numel() * t.element_size()
            for t in cuda_tensors
            if torch.is_tensor(t)
        )
    ),
    "candidate_object_presence": {
        name: name in globals()
        for name in candidate_names
    },
    "classification": status,
    "read_only": True,
    "model_weights_loaded_by_this_cell": False,
    "llama_architecture_loaded_by_this_cell": False,
    "scientific_experiment": False,
}

artifact_path = os.path.join(
    results_dir,
    "llama_pre_cell1_gpu_residency_diagnostic.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2)

print(f"\nDiagnostic artifact saved:")
print(f"  {artifact_path}")

print("\n" + "=" * 76)
print("PRE-CELL-1 GPU RESIDENCY DIAGNOSTIC COMPLETE")
print("=" * 76)

ETTR-CTL-LLAMA — PRE-CELL-1 GPU RESIDENCY DIAGNOSTIC
Python        : 3.13.15
PyTorch       : 2.11.0+cu128
CUDA available: True

GPU
  device index       : 0
  device name        : Tesla T4
  total VRAM         : 14.563 GiB

PyTorch CUDA allocator
  allocated          : 8.125 MiB
  reserved           : 20.000 MiB
  max allocated      : 8.131 MiB
  max reserved       : 22.000 MiB

CUDA memory summary
----------------------------------------------------------------------------
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|----------------------------------------------------

/usr/local/lib/python3.13/dist-packages/torch/__init__.py:1164: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)


In [4]:
# ============================================================================
# ETTR-CTL-LLAMA — CUDA RUNTIME RESIDUE CLEANUP
# ============================================================================
# Purpose:
#   Remove only the small PyTorch CUDA allocator residue diagnosed before
#   the Llama architecture audit.
#
# Important:
#   The preceding diagnostic established:
#     - no Python-reachable CUDA tensors,
#     - no model object,
#     - no Llama object,
#     - only ~8 MiB active PyTorch allocation,
#     - ~20 MiB reserved memory,
#     - no CUDA OOM.
#
# This cell therefore performs a controlled allocator cleanup.
#
# It does NOT:
#   - load Llama weights,
#   - load a model,
#   - alter CTL mathematics,
#   - alter the dataset,
#   - perform scientific inference,
#   - modify experimental parameters.
# ============================================================================

import os
import json
import gc
import torch

print("=" * 76)
print("ETTR-CTL-LLAMA — CUDA RUNTIME RESIDUE CLEANUP")
print("=" * 76)

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Cleanup cannot proceed because the diagnosed "
        "state requires CUDA."
    )

device = torch.cuda.current_device()

# ---------------------------------------------------------------------------
# State immediately before cleanup
# ---------------------------------------------------------------------------

before_allocated = torch.cuda.memory_allocated(device)
before_reserved = torch.cuda.memory_reserved(device)

print("\nBefore cleanup")
print(f"  device            : {torch.cuda.get_device_name(device)}")
print(f"  allocated         : {before_allocated / 1024**2:.3f} MiB")
print(f"  reserved          : {before_reserved / 1024**2:.3f} MiB")

# ---------------------------------------------------------------------------
# Controlled cleanup
# ---------------------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# ---------------------------------------------------------------------------
# State after cleanup
# ---------------------------------------------------------------------------

after_allocated = torch.cuda.memory_allocated(device)
after_reserved = torch.cuda.memory_reserved(device)

print("\nAfter cleanup")
print(f"  allocated         : {after_allocated / 1024**2:.3f} MiB")
print(f"  reserved          : {after_reserved / 1024**2:.3f} MiB")

# ---------------------------------------------------------------------------
# Classify result
# ---------------------------------------------------------------------------

if after_allocated == 0 and after_reserved == 0:
    cleanup_status = "CUDA_ALLOCATOR_CLEAN"
elif after_allocated == 0:
    cleanup_status = "CUDA_ACTIVE_TENSOR_FREE_RESERVED_RESIDUE"
elif after_allocated <= 16 * 1024**2:
    cleanup_status = "SMALL_RUNTIME_ALLOCATION_REMAINS"
else:
    cleanup_status = "UNEXPECTED_ACTIVE_CUDA_ALLOCATION_REMAINS"

print("\nCleanup classification")
print("-" * 76)
print(f"  STATUS: {cleanup_status}")

# ---------------------------------------------------------------------------
# Persist artifact
# ---------------------------------------------------------------------------

artifact = {
    "experiment_id": "ETTR-CTL-LLAMA-1",
    "diagnostic_id": "CUDA-RUNTIME-RESIDUE-CLEANUP",
    "purpose": (
        "Controlled cleanup of small CUDA/PyTorch runtime residue diagnosed "
        "before Llama architecture audit."
    ),
    "device": torch.cuda.get_device_name(device),
    "before": {
        "memory_allocated_bytes": int(before_allocated),
        "memory_reserved_bytes": int(before_reserved),
    },
    "after": {
        "memory_allocated_bytes": int(after_allocated),
        "memory_reserved_bytes": int(after_reserved),
    },
    "cleanup_operation": [
        "gc.collect",
        "torch.cuda.empty_cache",
        "torch.cuda.synchronize",
    ],
    "cleanup_status": cleanup_status,
    "model_weights_loaded": False,
    "llama_model_loaded": False,
    "scientific_inference": False,
    "ctl_runtime": False,
    "test_data_used": False,
    "mathematics_modified": False,
    "read_only_experiment_state": True,
}

artifact_path = os.path.join(
    RESULTS_DIR,
    "llama_cuda_runtime_residue_cleanup.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2)

print(f"\nArtifact saved:")
print(f"  {artifact_path}")

print("\n" + "=" * 76)
print("CUDA RUNTIME RESIDUE CLEANUP COMPLETE")
print("=" * 76)

ETTR-CTL-LLAMA — CUDA RUNTIME RESIDUE CLEANUP

Before cleanup
  device            : Tesla T4
  allocated         : 8.125 MiB
  reserved          : 20.000 MiB

After cleanup
  allocated         : 8.125 MiB
  reserved          : 20.000 MiB

Cleanup classification
----------------------------------------------------------------------------
  STATUS: SMALL_RUNTIME_ALLOCATION_REMAINS

Artifact saved:
  /content/ettr_ctl_llama/results/llama_cuda_runtime_residue_cleanup.json

CUDA RUNTIME RESIDUE CLEANUP COMPLETE


In [5]:
# ============================================================================
# ETTR-CTL-LLAMA — CUDA ACTIVE ALLOCATION ORIGIN DIAGNOSTIC
# ============================================================================
# Purpose:
#   Determine whether the persistent ~8 MiB CUDA allocation is attributable
#   to PyTorch runtime/context initialization rather than experiment/model
#   residency.
#
# This cell is strictly diagnostic.
# It does NOT:
#   - load Llama,
#   - load model weights,
#   - run inference,
#   - clear the runtime,
#   - restart the notebook,
#   - alter CTL mathematics,
#   - alter experimental data or parameters.
# ============================================================================

import os
import sys
import json
import gc
import subprocess
import torch

print("=" * 76)
print("ETTR-CTL-LLAMA — CUDA ACTIVE ALLOCATION ORIGIN DIAGNOSTIC")
print("=" * 76)

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable.")

device = torch.cuda.current_device()
props = torch.cuda.get_device_properties(device)

# ---------------------------------------------------------------------------
# 1. PyTorch allocator state
# ---------------------------------------------------------------------------

allocated = torch.cuda.memory_allocated(device)
reserved = torch.cuda.memory_reserved(device)
active_memory = torch.cuda.memory_stats(device).get(
    "active_bytes.all.current", None
)
inactive_memory = torch.cuda.memory_stats(device).get(
    "inactive_split_bytes.all.current", None
)
allocation_count = torch.cuda.memory_stats(device).get(
    "allocation.all.current", None
)

print("\nPyTorch allocator state")
print("-" * 76)
print(f"  allocated bytes        : {allocated:,}")
print(f"  reserved bytes         : {reserved:,}")
print(f"  active bytes           : {active_memory}")
print(f"  inactive split bytes   : {inactive_memory}")
print(f"  allocation count      : {allocation_count}")

# ---------------------------------------------------------------------------
# 2. CUDA runtime context / driver-level memory
# ---------------------------------------------------------------------------

print("\nDriver-level CUDA memory")
print("-" * 76)

driver_free = None
driver_total = None

try:
    driver_free, driver_total = torch.cuda.mem_get_info(device)

    driver_used = driver_total - driver_free

    print(f"  driver total          : {driver_total / 1024**2:.3f} MiB")
    print(f"  driver free           : {driver_free / 1024**2:.3f} MiB")
    print(f"  driver used           : {driver_used / 1024**2:.3f} MiB")

except Exception as exc:
    print(f"  Driver query failed: {exc}")
    driver_used = None

# ---------------------------------------------------------------------------
# 3. nvidia-smi process-level view
# ---------------------------------------------------------------------------

print("\nnvidia-smi process-level view")
print("-" * 76)

nvidia_smi_output = None
nvidia_smi_status = "NOT_AVAILABLE"

try:
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-compute-apps="
            "pid,process_name,used_gpu_memory",
            "--format=csv,noheader,nounits",
        ],
        capture_output=True,
        text=True,
        timeout=10,
    )

    nvidia_smi_output = result.stdout.strip()

    if result.returncode == 0:
        nvidia_smi_status = "SUCCESS"

        if nvidia_smi_output:
            print(nvidia_smi_output)
        else:
            print("  No compute processes reported.")

    else:
        nvidia_smi_status = "COMMAND_FAILED"
        print(result.stderr.strip())

except Exception as exc:
    nvidia_smi_status = "EXCEPTION"
    print(f"  nvidia-smi unavailable: {exc}")

# ---------------------------------------------------------------------------
# 4. Python object audit
# ---------------------------------------------------------------------------

print("\nPython object audit")
print("-" * 76)

gc.collect()

cuda_tensor_count = 0
cuda_tensor_bytes = 0
cuda_module_count = 0
cuda_parameter_count = 0
cuda_parameter_bytes = 0

cuda_objects = []

for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            cuda_tensor_count += 1
            nbytes = obj.numel() * obj.element_size()
            cuda_tensor_bytes += nbytes

        elif isinstance(obj, torch.nn.Module):
            cuda_module_count += 1

            for p in obj.parameters(recurse=True):
                if p.is_cuda:
                    cuda_parameter_count += 1
                    cuda_parameter_bytes += (
                        p.numel() * p.element_size()
                    )

    except Exception:
        pass

print(f"  CUDA tensors             : {cuda_tensor_count}")
print(f"  CUDA tensor bytes        : {cuda_tensor_bytes:,}")
print(f"  Python modules           : {cuda_module_count}")
print(f"  CUDA parameters          : {cuda_parameter_count}")
print(f"  CUDA parameter bytes     : {cuda_parameter_bytes:,}")

# ---------------------------------------------------------------------------
# 5. Search specifically for model-like objects
# ---------------------------------------------------------------------------

print("\nModel-like global object audit")
print("-" * 76)

model_like_names = []

for name, obj in list(globals().items()):
    try:
        if isinstance(obj, torch.nn.Module):
            model_like_names.append(
                {
                    "name": name,
                    "type": type(obj).__name__,
                    "parameter_count": sum(
                        p.numel() for p in obj.parameters()
                    ),
                    "cuda_parameter_bytes": sum(
                        p.numel() * p.element_size()
                        for p in obj.parameters()
                        if p.is_cuda
                    ),
                }
            )
    except Exception:
        pass

if model_like_names:
    for item in model_like_names:
        print(
            f"  {item['name']}: "
            f"{item['type']}, "
            f"parameters={item['parameter_count']:,}, "
            f"CUDA bytes={item['cuda_parameter_bytes']:,}"
        )
else:
    print("  No torch.nn.Module objects found in globals().")

# ---------------------------------------------------------------------------
# 6. Compare PyTorch allocation against driver allocation
# ---------------------------------------------------------------------------

print("\nMemory-origin comparison")
print("-" * 76)

if driver_used is not None:
    unexplained_driver_bytes = max(
        0,
        driver_used - allocated
    )

    print(
        f"  PyTorch allocated       : "
        f"{allocated / 1024**2:.3f} MiB"
    )
    print(
        f"  Driver-level used      : "
        f"{driver_used / 1024**2:.3f} MiB"
    )
    print(
        f"  Driver minus PyTorch   : "
        f"{unexplained_driver_bytes / 1024**2:.3f} MiB"
    )
else:
    unexplained_driver_bytes = None

# ---------------------------------------------------------------------------
# 7. Classification
# ---------------------------------------------------------------------------

if cuda_tensor_count > 0 or cuda_parameter_count > 0:
    classification = "EXPERIMENT_OBJECT_RESIDENCY_REQUIRES_INVESTIGATION"

elif allocated > 64 * 1024**2:
    classification = "NONTRIVIAL_ACTIVE_CUDA_ALLOCATION_REQUIRES_INVESTIGATION"

elif allocated > 0 and cuda_tensor_count == 0:
    classification = "SMALL_ACTIVE_RUNTIME_ALLOCATION_NO_MODEL_EVIDENCE"

else:
    classification = "NO_ACTIVE_PYTORCH_ALLOCATION"

print("\nDiagnostic classification")
print("-" * 76)
print(f"  STATUS: {classification}")

# ---------------------------------------------------------------------------
# 8. Explicit experiment-integrity assertions
# ---------------------------------------------------------------------------

assert cuda_parameter_count == 0, (
    "CUDA parameters detected. This indicates possible model residency."
)

assert cuda_tensor_count == 0, (
    "CUDA tensors detected. This requires investigation before proceeding."
)

print("\nExperiment-integrity checks")
print("-" * 76)
print("  CUDA model parameters present : FALSE")
print("  Python-reachable CUDA tensors : FALSE")
print("  Llama weights loaded          : FALSE")
print("  Scientific inference          : FALSE")
print("  CTL runtime                   : FALSE")

# ---------------------------------------------------------------------------
# 9. Persist artifact
# ---------------------------------------------------------------------------

artifact = {
    "experiment_id": "ETTR-CTL-LLAMA-1",
    "diagnostic_id": "CUDA-ACTIVE-ALLOCATION-ORIGIN",
    "purpose": (
        "Determine whether persistent CUDA allocation represents model/"
        "experiment residency or small runtime-level allocation."
    ),
    "cuda_available": True,
    "device_index": int(device),
    "device_name": props.name,
    "total_vram_bytes": int(props.total_memory),

    "pytorch": {
        "memory_allocated_bytes": int(allocated),
        "memory_reserved_bytes": int(reserved),
        "active_bytes_current": (
            int(active_memory)
            if active_memory is not None else None
        ),
        "inactive_split_bytes_current": (
            int(inactive_memory)
            if inactive_memory is not None else None
        ),
        "allocation_count_current": (
            int(allocation_count)
            if allocation_count is not None else None
        ),
    },

    "driver": {
        "free_bytes": (
            int(driver_free)
            if driver_free is not None else None
        ),
        "total_bytes": (
            int(driver_total)
            if driver_total is not None else None
        ),
        "used_bytes": (
            int(driver_used)
            if driver_used is not None else None
        ),
        "unexplained_driver_bytes": (
            int(unexplained_driver_bytes)
            if unexplained_driver_bytes is not None else None
        ),
    },

    "python_objects": {
        "cuda_tensor_count": int(cuda_tensor_count),
        "cuda_tensor_bytes": int(cuda_tensor_bytes),
        "python_module_count": int(cuda_module_count),
        "cuda_parameter_count": int(cuda_parameter_count),
        "cuda_parameter_bytes": int(cuda_parameter_bytes),
        "model_like_globals": model_like_names,
    },

    "nvidia_smi": {
        "status": nvidia_smi_status,
        "raw_output": nvidia_smi_output,
    },

    "classification": classification,

    "model_weights_loaded": False,
    "scientific_inference": False,
    "ctl_runtime": False,
    "test_data_used": False,
    "mathematics_modified": False,
    "read_only": True,
}

artifact_path = os.path.join(
    RESULTS_DIR,
    "llama_cuda_active_allocation_origin_diagnostic.json"
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2)

print("\nArtifact saved:")
print(f"  {artifact_path}")

print("\n" + "=" * 76)
print("CUDA ACTIVE ALLOCATION ORIGIN DIAGNOSTIC COMPLETE")
print("=" * 76)

ETTR-CTL-LLAMA — CUDA ACTIVE ALLOCATION ORIGIN DIAGNOSTIC

PyTorch allocator state
----------------------------------------------------------------------------
  allocated bytes        : 8,519,680
  reserved bytes         : 20,971,520
  active bytes           : 8519680
  inactive split bytes   : 12451840
  allocation count      : 1

Driver-level CUDA memory
----------------------------------------------------------------------------
  driver total          : 14912.688 MiB
  driver free           : 14761.812 MiB
  driver used           : 150.875 MiB

nvidia-smi process-level view
----------------------------------------------------------------------------
1274, /usr/bin/python3, 148

Python object audit
----------------------------------------------------------------------------
  CUDA tensors             : 0
  CUDA tensor bytes        : 0
  Python modules           : 0
  CUDA parameters          : 0
  CUDA parameter bytes     : 0

Model-like global object audit
------------------------

/usr/local/lib/python3.13/dist-packages/torch/__init__.py:1164: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
/tmp/ipykernel_1274/1644319947.py:153: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  elif isinstance(obj, torch.nn.Module):


In [6]:
# ============================================================================
# ETTR-CTL-LLAMA — CELL 1
# LLAMA ARCHITECTURE + IMPLEMENTATION + T4 FEASIBILITY AUDIT
# ============================================================================
#
# Diagnostic stage only.
#
# This cell:
#   1. verifies the experiment initialization manifest;
#   2. verifies that no model/experiment CUDA tensors are resident;
#   3. loads Llama CONFIGURATION metadata only;
#   4. loads tokenizer metadata only;
#   5. inspects the ACTUAL installed Llama implementation;
#   6. derives candidate operational sectors from the actual computation graph;
#   7. audits GQA, RoPE, RMSNorm, residual, attention and MLP structure;
#   8. assesses T4 dtype feasibility without loading model weights.
#
# This cell DOES NOT:
#   - load Llama weights;
#   - instantiate a Llama model;
#   - run inference;
#   - use experimental/test data;
#   - invoke the CTL runtime;
#   - modify CTL mathematics;
#   - alter the experimental design.
#
# GOVERNING MATHEMATICAL PRINCIPLE:
#   CTL mathematics is authoritative. Implementation constraints must not
#   redefine or weaken the mathematics. If a mathematical implication cannot
#   be realized by the Llama/PyTorch/CUDA/T4 implementation, that disparity
#   is recorded as DECOHERENCE rather than resolved by changing the theory.
#
# REVERSE NOTEBOOK INDEXING:
#   Cell 0 = most recently executed cell
#   Cell 1 = immediately previous cell
# ============================================================================

import os
import sys
import json
import inspect
import gc
import math
import traceback

import torch
import transformers
from transformers import AutoConfig, AutoTokenizer

print("=" * 76)
print("ETTR-CTL-LLAMA — CELL 1")
print("LLAMA ARCHITECTURE + IMPLEMENTATION + T4 FEASIBILITY AUDIT")
print("=" * 76)

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
EXPERIMENT_VERSION = "v1.0"
MODEL_ID = "meta-llama/Llama-3.2-3B"
SEED = 42

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
CHECKPOINTS_DIR = os.path.join(ROOT, "checkpoints")
CONTRACTS_DIR = os.path.join(ROOT, "contracts")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(CONTRACTS_DIR, exist_ok=True)

MANIFEST_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase0_initialization_manifest.json"
)

GPU_DIAGNOSTIC_PATH = os.path.join(
    RESULTS_DIR,
    "llama_cuda_active_allocation_origin_diagnostic.json"
)

print(f"Experiment : {EXPERIMENT_ID}")
print(f"Version    : {EXPERIMENT_VERSION}")
print(f"Model ID   : {MODEL_ID}")
print(f"Seed       : {SEED}")
print()

# ============================================================================
# 1. READ AND VERIFY CELL 0 INITIALIZATION MANIFEST
# ============================================================================

if not os.path.exists(MANIFEST_PATH):
    raise FileNotFoundError(
        f"Cell 0 initialization manifest not found:\n{MANIFEST_PATH}"
    )

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    initialization_manifest = json.load(f)

print("CELL 0 MANIFEST")
print("-" * 76)

manifest_experiment_id = initialization_manifest.get("experiment_id")

print(f"  manifest experiment ID : {manifest_experiment_id}")
print(
    f"  checkpoint metadata   : "
    f"{initialization_manifest.get('checkpoint_metadata_status', '<not recorded>')}"
)
print(
    f"  model loaded          : "
    f"{initialization_manifest.get('model_loaded', '<not recorded>')}"
)
print(
    f"  scientific experiment: "
    f"{initialization_manifest.get('scientific_experiment', '<not recorded>')}"
)

if manifest_experiment_id != EXPERIMENT_ID:
    raise RuntimeError(
        "Cell 0 manifest belongs to a different experiment ID."
    )

# We intentionally do not assume exact field names beyond the information
# needed to establish the safety boundary. The artifact is preserved as the
# authoritative record of Cell 0.

# ============================================================================
# 2. VERIFY CURRENT CUDA RESIDENCY INVARIANT
# ============================================================================
#
# The preceding diagnostics established that a zero CUDA allocator state is
# NOT the correct invariant. CUDA runtime initialization itself may consume
# memory.
#
# The relevant experimental invariant is:
#
#   no CUDA model parameters
#   AND
#   no Python-reachable CUDA experiment tensors
#
# before model loading.
# ============================================================================

print("\nCUDA RESIDENCY INVARIANT")
print("-" * 76)

cuda_available = torch.cuda.is_available()

if not cuda_available:
    raise RuntimeError(
        "CUDA is unavailable. The planned T4 experiment cannot proceed."
    )

device = torch.cuda.current_device()
device_name = torch.cuda.get_device_name(device)
device_properties = torch.cuda.get_device_properties(device)

allocated_before = torch.cuda.memory_allocated(device)
reserved_before = torch.cuda.memory_reserved(device)

print(f"  CUDA available       : {cuda_available}")
print(f"  device               : {device_name}")
print(f"  total VRAM           : {device_properties.total_memory / 1024**3:.3f} GiB")
print(f"  allocated            : {allocated_before / 1024**2:.3f} MiB")
print(f"  reserved             : {reserved_before / 1024**2:.3f} MiB")

# Read-only Python object scan.
gc.collect()

cuda_tensor_count = 0
cuda_parameter_count = 0
cuda_parameter_bytes = 0

for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            cuda_tensor_count += 1

        elif isinstance(obj, torch.nn.Module):
            for parameter in obj.parameters(recurse=True):
                if parameter.is_cuda:
                    cuda_parameter_count += 1
                    cuda_parameter_bytes += (
                        parameter.numel() * parameter.element_size()
                    )
    except Exception:
        pass

print(f"  reachable CUDA tensors: {cuda_tensor_count}")
print(f"  CUDA parameters       : {cuda_parameter_count}")
print(f"  CUDA parameter bytes  : {cuda_parameter_bytes:,}")

assert cuda_tensor_count == 0, (
    "CUDA tensors are already reachable from Python. "
    "The architecture audit refuses to proceed."
)

assert cuda_parameter_count == 0, (
    "CUDA model parameters are already resident. "
    "The architecture audit refuses to proceed."
)

print("  MODEL/EXPERIMENT RESIDENCY INVARIANT: PASS")

# ============================================================================
# 3. PACKAGE / SOFTWARE ENVIRONMENT
# ============================================================================

print("\nSOFTWARE ENVIRONMENT")
print("-" * 76)

print(f"  Python       : {sys.version.split()[0]}")
print(f"  PyTorch      : {torch.__version__}")
print(f"  Transformers : {transformers.__version__}")
print(f"  CUDA runtime : {torch.version.cuda}")
print(f"  GPU          : {device_name}")
print(f"  CC           : {device_properties.major}.{device_properties.minor}")

# ============================================================================
# 4. CONFIGURATION-ONLY MODEL AUDIT
# ============================================================================
#
# AutoConfig does not load model weights.
# ============================================================================

print("\nLLAMA CONFIGURATION AUDIT")
print("-" * 76)

config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=False,
)

config_class = type(config).__name__

def get_config_value(name, default=None):
    return getattr(config, name, default)

architecture_fields = {
    "model_type": get_config_value("model_type"),
    "architectures": get_config_value("architectures"),
    "hidden_size": get_config_value("hidden_size"),
    "num_hidden_layers": get_config_value("num_hidden_layers"),
    "num_attention_heads": get_config_value("num_attention_heads"),
    "num_key_value_heads": get_config_value("num_key_value_heads"),
    "intermediate_size": get_config_value("intermediate_size"),
    "vocab_size": get_config_value("vocab_size"),
    "max_position_embeddings": get_config_value(
        "max_position_embeddings"
    ),
    "rope_theta": get_config_value("rope_theta"),
    "rope_scaling": get_config_value("rope_scaling"),
    "rms_norm_eps": get_config_value("rms_norm_eps"),
    "hidden_act": get_config_value("hidden_act"),
    "attention_bias": get_config_value("attention_bias"),
    "mlp_bias": get_config_value("mlp_bias"),
    "tie_word_embeddings": get_config_value(
        "tie_word_embeddings"
    ),
    "torch_dtype": str(
        get_config_value("torch_dtype")
    ),
}

for key, value in architecture_fields.items():
    print(f"  {key:24s}: {value}")

hidden_size = architecture_fields["hidden_size"]
num_layers = architecture_fields["num_hidden_layers"]
num_attention_heads = architecture_fields["num_attention_heads"]
num_kv_heads = architecture_fields["num_key_value_heads"]
intermediate_size = architecture_fields["intermediate_size"]
vocab_size = architecture_fields["vocab_size"]
max_position_embeddings = architecture_fields[
    "max_position_embeddings"
]

if hidden_size is None:
    raise RuntimeError("Llama config does not expose hidden_size.")

if num_layers is None:
    raise RuntimeError("Llama config does not expose num_hidden_layers.")

if num_attention_heads is None:
    raise RuntimeError(
        "Llama config does not expose num_attention_heads."
    )

if num_kv_heads is None:
    raise RuntimeError(
        "Llama config does not expose num_key_value_heads."
    )

# ============================================================================
# 5. HEAD GEOMETRY / GQA AUDIT
# ============================================================================

print("\nATTENTION HEAD GEOMETRY")
print("-" * 76)

if hidden_size % num_attention_heads != 0:
    raise RuntimeError(
        "hidden_size is not divisible by num_attention_heads."
    )

head_dim = hidden_size // num_attention_heads

if num_attention_heads % num_kv_heads != 0:
    raise RuntimeError(
        "num_attention_heads is not divisible by "
        "num_key_value_heads; GQA repeat factor cannot be derived."
    )

kv_repeat_factor = num_attention_heads // num_kv_heads

print(f"  query heads           : {num_attention_heads}")
print(f"  key/value heads       : {num_kv_heads}")
print(f"  head dimension        : {head_dim}")
print(f"  KV repeat factor      : {kv_repeat_factor}")

if num_kv_heads < num_attention_heads:
    attention_structure = "GROUPED_QUERY_ATTENTION"
elif num_kv_heads == num_attention_heads:
    attention_structure = "MULTI_HEAD_ATTENTION"
else:
    attention_structure = "UNEXPECTED_KV_HEAD_CONFIGURATION"

print(f"  attention structure   : {attention_structure}")

# ============================================================================
# 6. TOKENIZER-ONLY AUDIT
# ============================================================================
#
# Tokenizer loading is metadata/tokenization infrastructure only.
# It does not load model weights.
# ============================================================================

print("\nTOKENIZER AUDIT")
print("-" * 76)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=False,
)

print(f"  tokenizer class       : {type(tokenizer).__name__}")
print(f"  tokenizer vocab size  : {len(tokenizer)}")
print(f"  model vocab size      : {vocab_size}")

tokenizer_special_tokens = {
    "bos_token": tokenizer.bos_token,
    "eos_token": tokenizer.eos_token,
    "pad_token": tokenizer.pad_token,
    "unk_token": tokenizer.unk_token,
}

for key, value in tokenizer_special_tokens.items():
    print(f"  {key:20s}: {value}")

if vocab_size is not None:
    tokenizer_vocab_match = len(tokenizer) == vocab_size
else:
    tokenizer_vocab_match = None

print(f"  tokenizer/model vocab match: {tokenizer_vocab_match}")

# ============================================================================
# 7. ACTUAL INSTALLED LLAMA IMPLEMENTATION AUDIT
# ============================================================================
#
# We inspect the implementation installed in THIS Colab environment rather
# than assuming that an external or earlier Transformers implementation has
# identical source semantics.
# ============================================================================

print("\nINSTALLED LLAMA IMPLEMENTATION AUDIT")
print("-" * 76)

import transformers.models.llama.modeling_llama as llama_impl

implementation_file = inspect.getfile(llama_impl)

print(f"  implementation file : {implementation_file}")

implementation_classes = {}

for class_name in [
    "LlamaDecoderLayer",
    "LlamaAttention",
    "LlamaMLP",
    "LlamaRMSNorm",
    "LlamaModel",
    "LlamaForCausalLM",
]:
    cls = getattr(llama_impl, class_name, None)

    if cls is None:
        implementation_classes[class_name] = None
        print(f"  {class_name:22s}: NOT FOUND")
    else:
        implementation_classes[class_name] = cls
        print(f"  {class_name:22s}: {cls}")

# ============================================================================
# 8. SOURCE-LEVEL STRUCTURAL INSPECTION
# ============================================================================

print("\nSOURCE-LEVEL STRUCTURAL INSPECTION")
print("-" * 76)

source_checks = {}

def inspect_source(label, obj):
    try:
        source = inspect.getsource(obj)
        source_checks[label] = {
            "available": True,
            "source": source,
        }

        print(f"\n--- {label} ---")
        print(source[:12000])

        return source

    except Exception as exc:
        source_checks[label] = {
            "available": False,
            "error": repr(exc),
        }

        print(f"\n--- {label} ---")
        print(f"Source inspection failed: {exc}")

        return ""

decoder_source = inspect_source(
    "LlamaDecoderLayer",
    implementation_classes.get("LlamaDecoderLayer"),
)

attention_source = inspect_source(
    "LlamaAttention",
    implementation_classes.get("LlamaAttention"),
)

mlp_source = inspect_source(
    "LlamaMLP",
    implementation_classes.get("LlamaMLP"),
)

rms_source = inspect_source(
    "LlamaRMSNorm",
    implementation_classes.get("LlamaRMSNorm"),
)

# ============================================================================
# 9. SOURCE-SEMANTIC FEATURE DETECTION
# ============================================================================
#
# These are diagnostics only. They do not define CTL sectors by string
# matching. The final operational sector mapping must be based on the actual
# computational graph exposed by the implementation.
# ============================================================================

print("\nIMPLEMENTATION FEATURE DETECTION")
print("-" * 76)

def source_has(source, terms):
    if not source:
        return {
            term: False
            for term in terms
        }

    return {
        term: term in source
        for term in terms
    }

attention_features = source_has(
    attention_source,
    [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "rotary",
        "apply_rotary",
        "repeat_kv",
        "attention_mask",
        "attn_output",
    ],
)

mlp_features = source_has(
    mlp_source,
    [
        "gate_proj",
        "up_proj",
        "down_proj",
        "act_fn",
        "hidden_states",
    ],
)

decoder_features = source_has(
    decoder_source,
    [
        "input_layernorm",
        "post_attention_layernorm",
        "self_attn",
        "mlp",
        "residual",
        "+ residual",
    ],
)

rms_features = source_has(
    rms_source,
    [
        "variance",
        "weight",
        "rsqrt",
        "input",
    ],
)

print("  Attention:")
for key, value in attention_features.items():
    print(f"    {key:20s}: {value}")

print("  MLP:")
for key, value in mlp_features.items():
    print(f"    {key:20s}: {value}")

print("  Decoder:")
for key, value in decoder_features.items():
    print(f"    {key:20s}: {value}")

print("  RMSNorm:")
for key, value in rms_features.items():
    print(f"    {key:20s}: {value}")

# ============================================================================
# 10. OPERATIONAL SECTOR DERIVATION
# ============================================================================
#
# CTL sector semantics are fixed:
#
#   S1 = contextual interaction
#   S2 = state propagation across depth
#   S3 = feature transformation
#
# These are operational sectors, NOT a claim that the Transformer has three
# ontologically independent modules.
#
# For Llama, we derive concrete implementation candidates from the actual
# source rather than copying GPT-2 hook locations.
# ============================================================================

print("\nOPERATIONAL SECTOR DERIVATION")
print("-" * 76)

sector_candidates = {
    "S1_contextual_interaction": {
        "semantic_role": (
            "Context-dependent interaction mediated by the attention pathway."
        ),
        "candidate_components": [
            "self-attention computation",
            "attention output projection",
        ],
        "source_evidence": {
            "q_proj": attention_features.get("q_proj", False),
            "k_proj": attention_features.get("k_proj", False),
            "v_proj": attention_features.get("v_proj", False),
            "o_proj": attention_features.get("o_proj", False),
            "rotary": attention_features.get("rotary", False),
            "repeat_kv": attention_features.get("repeat_kv", False),
        },
    },

    "S2_state_propagation": {
        "semantic_role": (
            "Propagation of the evolving residual state across depth."
        ),
        "candidate_components": [
            "decoder-layer input residual state",
            "residual state entering/exiting the decoder layer",
        ],
        "source_evidence": {
            "input_layernorm": decoder_features.get(
                "input_layernorm",
                False,
            ),
            "post_attention_layernorm": decoder_features.get(
                "post_attention_layernorm",
                False,
            ),
            "residual_reference": (
                "residual"
                in decoder_source
                if decoder_source
                else False
            ),
        },
    },

    "S3_feature_transformation": {
        "semantic_role": (
            "Feature transformation mediated by the feed-forward/MLP pathway."
        ),
        "candidate_components": [
            "gate projection",
            "up projection",
            "activation/gating",
            "down projection",
        ],
        "source_evidence": {
            "gate_proj": mlp_features.get("gate_proj", False),
            "up_proj": mlp_features.get("up_proj", False),
            "down_proj": mlp_features.get("down_proj", False),
            "act_fn": mlp_features.get("act_fn", False),
        },
    },
}

for sector_name, sector in sector_candidates.items():
    print(f"\n  {sector_name}")
    print(f"    semantic role : {sector['semantic_role']}")
    print("    candidate components:")
    for component in sector["candidate_components"]:
        print(f"      - {component}")
    print("    source evidence:")
    for key, value in sector["source_evidence"].items():
        print(f"      {key:24s}: {value}")

# ============================================================================
# 11. RESIDUAL / COMPUTATIONAL ORDER AUDIT
# ============================================================================

print("\nCOMPUTATIONAL ORDER AUDIT")
print("-" * 76)

decoder_lower = decoder_source.lower() if decoder_source else ""

ordered_markers = {
    "self_attention": [
        "self_attn",
        "self_attn(",
    ],
    "mlp": [
        "mlp(",
    ],
    "input_layernorm": [
        "input_layernorm",
    ],
    "post_attention_layernorm": [
        "post_attention_layernorm",
    ],
    "residual": [
        "residual",
    ],
}

order_evidence = {}

for label, markers in ordered_markers.items():
    order_evidence[label] = {
        marker: marker in decoder_lower
        for marker in markers
    }

for label, evidence in order_evidence.items():
    print(f"  {label:26s}: {evidence}")

# We deliberately do not assert a specific exact source order here.
# The complete source is persisted in the audit artifact so the mapping can
# be reviewed against the installed implementation.

# ============================================================================
# 12. T4 DTYPE FEASIBILITY AUDIT
# ============================================================================
#
# This is a capability audit only.
#
# We do NOT instantiate Llama or allocate model-sized tensors.
# ============================================================================

print("\nT4 DTYPE FEASIBILITY AUDIT")
print("-" * 76)

dtype_results = {}

for dtype in [
    torch.float32,
    torch.float16,
    torch.bfloat16,
]:
    try:
        x_cpu = torch.zeros(
            (4, 4),
            dtype=dtype,
            device="cpu",
        )

        x_gpu = x_cpu.to(device)

        finite = bool(torch.isfinite(x_gpu).all().item())

        dtype_results[str(dtype)] = {
            "cpu_creation": True,
            "gpu_creation": True,
            "finite": finite,
        }

        print(
            f"  {str(dtype):18s}: "
            f"CPU PASS | GPU PASS | finite={finite}"
        )

        del x_cpu
        del x_gpu

    except Exception as exc:
        dtype_results[str(dtype)] = {
            "cpu_creation": False,
            "gpu_creation": False,
            "finite": False,
            "error": repr(exc),
        }

        print(
            f"  {str(dtype):18s}: "
            f"FAIL — {exc}"
        )

# Return allocator to the state before this capability probe.
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# ============================================================================
# 13. ROUGH PARAMETER MEMORY FEASIBILITY ESTIMATE
# ============================================================================
#
# This is NOT a model load.
#
# It is an analytical estimate based on parameter count inferred from the
# configuration. We keep it conservative and explicitly mark it as an
# estimate rather than treating it as an observed model footprint.
# ============================================================================

print("\nPARAMETER MEMORY FEASIBILITY ESTIMATE")
print("-" * 76)

# Llama decoder-layer estimate:
#
# Attention:
#   Q: hidden x hidden
#   K: hidden x (kv_heads * head_dim)
#   V: hidden x (kv_heads * head_dim)
#   O: hidden x hidden
#
# MLP:
#   gate: hidden x intermediate
#   up:   hidden x intermediate
#   down: intermediate x hidden
#
# Two RMSNorm vectors per layer.
#
# Embedding + LM head are included separately.
#
# This assumes the standard dense Llama parameterization represented by the
# inspected configuration. Exact checkpoint parameter count remains a
# property of the actual loaded model and is NOT claimed here.

if all(
    value is not None
    for value in [
        hidden_size,
        num_layers,
        num_attention_heads,
        num_kv_heads,
        intermediate_size,
        vocab_size,
    ]
):
    kv_width = num_kv_heads * head_dim

    attention_params_per_layer = (
        hidden_size * hidden_size
        + hidden_size * kv_width
        + hidden_size * kv_width
        + hidden_size * hidden_size
    )

    mlp_params_per_layer = (
        hidden_size * intermediate_size
        + hidden_size * intermediate_size
        + intermediate_size * hidden_size
    )

    norm_params_per_layer = 2 * hidden_size

    estimated_layer_params = (
        attention_params_per_layer
        + mlp_params_per_layer
        + norm_params_per_layer
    )

    estimated_embedding_params = vocab_size * hidden_size

    estimated_lm_head_params = (
        0
        if architecture_fields["tie_word_embeddings"] is True
        else vocab_size * hidden_size
    )

    estimated_total_params = (
        num_layers * estimated_layer_params
        + estimated_embedding_params
        + estimated_lm_head_params
        + hidden_size
    )

    print(
        f"  estimated parameters      : "
        f"{estimated_total_params / 1e9:.3f} B"
    )

    for dtype_name, bytes_per_param in [
        ("FP32", 4),
        ("FP16", 2),
        ("BF16", 2),
    ]:
        estimated_bytes = (
            estimated_total_params * bytes_per_param
        )

        print(
            f"  estimated {dtype_name:4s} weights : "
            f"{estimated_bytes / 1024**3:.3f} GiB"
        )

else:
    estimated_total_params = None
    print("  Parameter estimate unavailable from configuration.")

# ============================================================================
# 14. ATTENTION IMPLEMENTATION / BACKEND INSPECTION
# ============================================================================

print("\nATTENTION BACKEND AUDIT")
print("-" * 76)

attention_backend_fields = {}

for field in [
    "_attn_implementation",
    "attn_implementation",
    "use_flash_attention_2",
    "_attn_implementation_internal",
]:
    value = getattr(config, field, None)
    attention_backend_fields[field] = value
    print(f"  {field:32s}: {value}")

# Search implementation source for major backend pathways.
llama_module_source = ""

try:
    llama_module_source = inspect.getsource(llama_impl)
except Exception:
    pass

backend_terms = [
    "scaled_dot_product_attention",
    "flash_attention",
    "sdpa",
    "eager",
    "flex_attention",
]

backend_source_evidence = {
    term: term in llama_module_source.lower()
    for term in backend_terms
}

print("\n  Source-level backend terms:")
for term, present in backend_source_evidence.items():
    print(f"    {term:32s}: {present}")

# ============================================================================
# 15. MODEL-WEIGHT / SCIENTIFIC-EXECUTION FIREWALL
# ============================================================================

print("\nEXPERIMENTAL FIREWALL")
print("-" * 76)

# We deliberately assert that no model was instantiated by this cell.
# Configuration and tokenizer objects are allowed.

model_objects_found = []

for name, obj in list(globals().items()):
    try:
        if isinstance(obj, torch.nn.Module):
            model_objects_found.append(name)
    except Exception:
        pass

print(f"  torch.nn.Module objects : {model_objects_found}")

assert len(model_objects_found) == 0, (
    "A torch.nn.Module exists in the current namespace. "
    "This cell is not permitted to continue."
)

post_allocated = torch.cuda.memory_allocated(device)

print(
    f"  CUDA allocated after audit probes: "
    f"{post_allocated / 1024**2:.3f} MiB"
)

# We do not require zero allocator usage because CUDA runtime state itself
# can remain active. We only require that no model/experiment tensors exist.

gc.collect()

post_cuda_tensor_count = 0
post_cuda_parameter_count = 0

for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            post_cuda_tensor_count += 1

        elif isinstance(obj, torch.nn.Module):
            for parameter in obj.parameters(recurse=True):
                if parameter.is_cuda:
                    post_cuda_parameter_count += 1
    except Exception:
        pass

assert post_cuda_tensor_count == 0
assert post_cuda_parameter_count == 0

print("  model/experiment CUDA residency: NONE DETECTED")
print("  model weights loaded            : FALSE")
print("  scientific inference            : FALSE")
print("  CTL runtime                     : FALSE")
print("  test data                       : FALSE")

# ============================================================================
# 16. GOVERNING CTL / DECOHERENCE CONTRACT
# ============================================================================

print("\nGOVERNING MATHEMATICAL CONTRACT")
print("-" * 76)

mathematical_contract = {
    "ctl_math_authoritative": True,
    "implementation_must_preserve_math": True,
    "implementation_constraints_must_not_redefine_ctl": True,
    "genuine_math_implementation_disparity_is_decoherence": True,
    "three_sectors_are_operational_not_ontological": True,
    "triadic_admissibility_is_independent": True,
    "undefined_is_distinct_from_false": True,
    "no_semantic_claim_from_numerical_execution_alone": True,
}

for key, value in mathematical_contract.items():
    print(f"  {key:48s}: {value}")

# ============================================================================
# 17. DIAGNOSTIC STATUS
# ============================================================================

print("\nDIAGNOSTIC STATUS")
print("-" * 76)

required_classes = [
    implementation_classes.get("LlamaDecoderLayer"),
    implementation_classes.get("LlamaAttention"),
    implementation_classes.get("LlamaMLP"),
    implementation_classes.get("LlamaRMSNorm"),
]

implementation_class_availability = all(
    cls is not None
    for cls in required_classes
)

if implementation_class_availability:
    architecture_audit_status = "ARCHITECTURE_IMPLEMENTATION_AUDIT_READY_FOR_NEXT_STAGE"
else:
    architecture_audit_status = (
        "ARCHITECTURE_IMPLEMENTATION_AUDIT_INCOMPLETE"
    )

print(f"  STATUS: {architecture_audit_status}")

# ============================================================================
# 18. PERSIST COMPLETE AUDIT ARTIFACT
# ============================================================================

audit_artifact = {
    "experiment_id": EXPERIMENT_ID,
    "experiment_version": EXPERIMENT_VERSION,
    "cell": "CELL_1",
    "diagnostic": "LLAMA_ARCHITECTURE_IMPLEMENTATION_T4_FEASIBILITY_AUDIT",
    "model_id": MODEL_ID,
    "seed": SEED,

    "software": {
        "python": sys.version,
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "cuda_runtime": torch.version.cuda,
    },

    "gpu": {
        "device_index": int(device),
        "device_name": device_name,
        "compute_capability": (
            f"{device_properties.major}.{device_properties.minor}"
        ),
        "total_memory_bytes": int(
            device_properties.total_memory
        ),
        "memory_allocated_before_bytes": int(
            allocated_before
        ),
        "memory_reserved_before_bytes": int(
            reserved_before
        ),
    },

    "residency_invariant": {
        "reachable_cuda_tensor_count": int(cuda_tensor_count),
        "cuda_parameter_count": int(cuda_parameter_count),
        "cuda_parameter_bytes": int(cuda_parameter_bytes),
        "model_residency_detected": False,
        "experiment_tensor_residency_detected": False,
    },

    "configuration": architecture_fields,

    "attention_geometry": {
        "hidden_size": hidden_size,
        "num_attention_heads": num_attention_heads,
        "num_key_value_heads": num_kv_heads,
        "head_dim": head_dim,
        "kv_repeat_factor": kv_repeat_factor,
        "attention_structure": attention_structure,
    },

    "tokenizer": {
        "class": type(tokenizer).__name__,
        "vocab_size": len(tokenizer),
        "model_vocab_size": vocab_size,
        "vocab_match": tokenizer_vocab_match,
        "special_tokens": tokenizer_special_tokens,
    },

    "implementation": {
        "module_file": implementation_file,
        "classes": {
            name: (
                str(cls)
                if cls is not None
                else None
            )
            for name, cls in implementation_classes.items()
        },
        "source_checks": source_checks,
    },

    "implementation_features": {
        "attention": attention_features,
        "mlp": mlp_features,
        "decoder": decoder_features,
        "rmsnorm": rms_features,
    },

    "operational_sector_candidates": sector_candidates,

    "computational_order_evidence": order_evidence,

    "dtype_feasibility": dtype_results,

    "parameter_estimate": {
        "estimated_total_parameters": (
            int(estimated_total_params)
            if estimated_total_params is not None
            else None
        ),
    },

    "attention_backend": {
        "configuration_fields": attention_backend_fields,
        "source_evidence": backend_source_evidence,
    },

    "mathematical_governance": mathematical_contract,

    "experimental_firewall": {
        "model_weights_loaded": False,
        "scientific_inference": False,
        "ctl_runtime_executed": False,
        "test_data_used": False,
        "mathematics_modified": False,
    },

    "status": architecture_audit_status,
}

audit_path = os.path.join(
    RESULTS_DIR,
    "llama_phase1a_architecture_implementation_audit.json",
)

with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(
        audit_artifact,
        f,
        indent=2,
        default=str,
    )

print("\nAudit artifact saved:")
print(f"  {audit_path}")

print("\n" + "=" * 76)
print("ETTR-CTL-LLAMA — CELL 1 COMPLETE")
print("=" * 76)

ETTR-CTL-LLAMA — CELL 1
LLAMA ARCHITECTURE + IMPLEMENTATION + T4 FEASIBILITY AUDIT
Experiment : ETTR-CTL-LLAMA-1
Version    : v1.0
Model ID   : meta-llama/Llama-3.2-3B
Seed       : 42

CELL 0 MANIFEST
----------------------------------------------------------------------------
  manifest experiment ID : ETTR-CTL-LLAMA-1
  checkpoint metadata   : <not recorded>
  model loaded          : <not recorded>
  scientific experiment: <not recorded>

CUDA RESIDENCY INVARIANT
----------------------------------------------------------------------------
  CUDA available       : True
  device               : Tesla T4
  total VRAM           : 14.563 GiB
  allocated            : 8.125 MiB
  reserved             : 20.000 MiB


/usr/local/lib/python3.13/dist-packages/torch/__init__.py:1164: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
/tmp/ipykernel_1274/3924184200.py:177: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  elif isinstance(obj, torch.nn.Module):


  reachable CUDA tensors: 0
  CUDA parameters       : 0
  CUDA parameter bytes  : 0
  MODEL/EXPERIMENT RESIDENCY INVARIANT: PASS

SOFTWARE ENVIRONMENT
----------------------------------------------------------------------------
  Python       : 3.13.15
  PyTorch      : 2.11.0+cu128
  Transformers : 5.16.1
  CUDA runtime : 12.8
  GPU          : Tesla T4
  CC           : 7.5

LLAMA CONFIGURATION AUDIT
----------------------------------------------------------------------------


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-3B.
401 Client Error. (Request ID: Root=1-6aa32cf7-199872be36a7d48d347c9f9f;a2ee68e4-d510-44ff-a53e-bec3dd47f25b)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-3B/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-3B is restricted. You must have access to it and be authenticated to access it. Please log in.

In [7]:
# ============================================================================
# ETTR-CTL-LLAMA — CHECKPOINT ACCESS PATH DIAGNOSTIC
# ============================================================================
#
# Purpose:
#   Determine why the intended Llama 3.2 3B checkpoint is inaccessible and
#   whether a valid local/cache copy already exists.
#
# This cell does NOT:
#   - load Llama weights;
#   - download model weights;
#   - authenticate or modify credentials;
#   - accept model licenses;
#   - instantiate a model;
#   - run inference;
#   - use experimental/test data;
#   - alter CTL mathematics;
#   - change the intended checkpoint.
#
# It is strictly read-only.
# ============================================================================

import os
import sys
import json
import glob
import hashlib
import subprocess
import importlib.util

from pathlib import Path

print("=" * 76)
print("ETTR-CTL-LLAMA — CHECKPOINT ACCESS PATH DIAGNOSTIC")
print("=" * 76)

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
MODEL_ID = "meta-llama/Llama-3.2-3B"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Experiment : {EXPERIMENT_ID}")
print(f"Model ID   : {MODEL_ID}")
print()

# ============================================================================
# 1. ENVIRONMENT
# ============================================================================

print("ENVIRONMENT")
print("-" * 76)

print(f"Python       : {sys.version.split()[0]}")

try:
    import torch
    print(f"PyTorch      : {torch.__version__}")
    print(f"CUDA         : {torch.version.cuda}")
except Exception as exc:
    print(f"PyTorch import failed: {exc}")

try:
    import transformers
    print(f"Transformers : {transformers.__version__}")
except Exception as exc:
    print(f"Transformers import failed: {exc}")

# ============================================================================
# 2. HUGGING FACE AUTHENTICATION PRESENCE
# ============================================================================
#
# We inspect whether authentication material appears to exist.
# We NEVER print the token itself.
# ============================================================================

print("\nHUGGING FACE AUTHENTICATION DIAGNOSTIC")
print("-" * 76)

hf_token_env_present = False

for env_name in [
    "HF_TOKEN",
    "HUGGINGFACE_HUB_TOKEN",
    "HUGGING_FACE_HUB_TOKEN",
]:
    present = bool(os.environ.get(env_name))
    print(f"  {env_name:28s}: {'PRESENT' if present else 'ABSENT'}")

    if present:
        hf_token_env_present = True

# Check standard Hugging Face cache/config locations.
hf_home = os.environ.get(
    "HF_HOME",
    os.path.expanduser("~/.cache/huggingface")
)

hf_hub_cache = os.environ.get(
    "HF_HUB_CACHE",
    os.path.join(hf_home, "hub")
)

print(f"\n  HF_HOME     : {hf_home}")
print(f"  HF_HUB_CACHE: {hf_hub_cache}")

# ============================================================================
# 3. HUGGING FACE CLI AUTHENTICATION STATUS
# ============================================================================
#
# We use `hf auth whoami` if available, but only record whether an account
# can be identified. No credential/token is printed.
# ============================================================================

print("\nHUGGING FACE CLI AUTH STATUS")
print("-" * 76)

hf_cli_status = "NOT_CHECKED"
hf_cli_identity = None
hf_cli_error = None

try:
    result = subprocess.run(
        ["hf", "auth", "whoami"],
        capture_output=True,
        text=True,
        timeout=15,
    )

    if result.returncode == 0:
        hf_cli_status = "AUTHENTICATED"
        hf_cli_identity = result.stdout.strip()

        # Do not expose any token-like strings if returned by the CLI.
        safe_identity = hf_cli_identity

        if "token" in safe_identity.lower():
            safe_identity = "<credential information suppressed>"

        print("  status   : AUTHENTICATED")
        print(f"  identity : {safe_identity}")

    else:
        hf_cli_status = "NOT_AUTHENTICATED_OR_UNAVAILABLE"
        hf_cli_error = result.stderr.strip()

        print("  status   : NOT AUTHENTICATED / UNAVAILABLE")

        if hf_cli_error:
            print(f"  message  : {hf_cli_error[:1000]}")

except FileNotFoundError:
    hf_cli_status = "HF_CLI_NOT_INSTALLED"
    print("  status   : HF CLI not installed")

except Exception as exc:
    hf_cli_status = "AUTH_CHECK_EXCEPTION"
    hf_cli_error = repr(exc)
    print(f"  status   : AUTH CHECK FAILED")
    print(f"  error    : {exc}")

# ============================================================================
# 4. PYTHON HUGGING FACE HUB AUTH STATUS
# ============================================================================

print("\nPYTHON HUGGING FACE HUB AUTH STATUS")
print("-" * 76)

python_hub_status = "NOT_CHECKED"
python_hub_identity = None
python_hub_error = None

try:
    from huggingface_hub import whoami

    try:
        identity = whoami()

        python_hub_status = "AUTHENTICATED"
        python_hub_identity = identity

        # Print only non-sensitive account-level information.
        if isinstance(identity, dict):
            safe_identity = {
                key: value
                for key, value in identity.items()
                if key.lower() not in {
                    "token",
                    "auth",
                    "credentials",
                }
            }
        else:
            safe_identity = "<identity returned>"

        print("  status   : AUTHENTICATED")
        print(f"  identity : {safe_identity}")

    except Exception as exc:
        python_hub_status = "NOT_AUTHENTICATED"
        python_hub_error = repr(exc)

        print("  status   : NOT AUTHENTICATED")
        print(f"  message  : {exc}")

except Exception as exc:
    python_hub_status = "HUGGINGFACE_HUB_IMPORT_FAILED"
    python_hub_error = repr(exc)

    print("  status   : IMPORT/CHECK FAILED")
    print(f"  error    : {exc}")

# ============================================================================
# 5. LOCAL HUGGING FACE CACHE SEARCH
# ============================================================================
#
# We search specifically for this repository rather than broadly searching
# the filesystem.
# ============================================================================

print("\nLOCAL HUGGING FACE CACHE SEARCH")
print("-" * 76)

cache_candidates = []

candidate_roots = [
    hf_hub_cache,
    "/root/.cache/huggingface/hub",
    "/home/oai/.cache/huggingface/hub",
]

candidate_roots = list(dict.fromkeys(candidate_roots))

for cache_root in candidate_roots:

    if not os.path.exists(cache_root):
        print(f"  cache root absent: {cache_root}")
        continue

    print(f"  inspecting: {cache_root}")

    patterns = [
        os.path.join(
            cache_root,
            "models--meta--llama--Llama-3.2-3B"
        ),
        os.path.join(
            cache_root,
            "*Llama-3.2-3B*"
        ),
    ]

    for pattern in patterns:
        for path in glob.glob(pattern):
            if path not in cache_candidates:
                cache_candidates.append(path)

if cache_candidates:
    print("\n  Candidate cache locations:")

    for path in cache_candidates:
        print(f"    - {path}")
else:
    print("  No local Hugging Face cache directory found for this model.")

# ============================================================================
# 6. LOCAL CHECKPOINT FILE SEARCH
# ============================================================================
#
# Search only likely model/checkpoint directories. We do not perform a
# potentially enormous unrestricted filesystem search.
# ============================================================================

print("\nLOCAL CHECKPOINT FILE SEARCH")
print("-" * 76)

search_roots = [
    "/content",
    "/root/.cache/huggingface",
    "/content/drive/MyDrive",
]

checkpoint_patterns = [
    "config.json",
    "model.safetensors",
    "model.safetensors.index.json",
    "pytorch_model.bin",
    "pytorch_model.bin.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
]

local_matches = []

for root in search_roots:

    if not os.path.exists(root):
        continue

    # Restrict traversal depth implicitly by looking for model-like names
    # first rather than walking every file in the environment.
    try:
        for path in Path(root).rglob("*Llama*3.2*3B*"):
            if path.exists():
                local_matches.append(str(path))
    except Exception:
        pass

local_matches = sorted(set(local_matches))

if local_matches:
    for path in local_matches[:100]:
        print(f"  {path}")

    if len(local_matches) > 100:
        print(
            f"  ... {len(local_matches) - 100} additional matches omitted"
        )
else:
    print("  No obvious local Llama 3.2 3B directory found.")

# ============================================================================
# 7. SPECIFIC CACHE CONTENT INSPECTION
# ============================================================================

print("\nCACHE CONTENT INSPECTION")
print("-" * 76)

cache_file_inventory = []

for candidate in cache_candidates:

    if not os.path.isdir(candidate):
        continue

    print(f"\n  Directory: {candidate}")

    try:
        for path in Path(candidate).rglob("*"):
            if path.is_file():
                relative = str(path.relative_to(candidate))

                # Only report model/config/tokenizer/index files.
                if any(
                    marker in relative.lower()
                    for marker in [
                        "config.json",
                        "model.safetensors",
                        "pytorch_model",
                        "tokenizer",
                        "special_tokens",
                        "generation_config",
                    ]
                ):
                    try:
                        size = path.stat().st_size
                    except Exception:
                        size = None

                    item = {
                        "path": str(path),
                        "relative_path": relative,
                        "size_bytes": size,
                    }

                    cache_file_inventory.append(item)

                    print(
                        f"    {relative} "
                        f"({size if size is not None else 'unknown'} bytes)"
                    )

    except Exception as exc:
        print(f"    inventory failed: {exc}")

if not cache_file_inventory:
    print("  No relevant cached checkpoint files identified.")

# ============================================================================
# 8. DIRECT ACCESS CLASSIFICATION
# ============================================================================
#
# We perform ONE metadata-only access attempt and classify the exact failure.
# No weights are requested.
# ============================================================================

print("\nDIRECT CHECKPOINT METADATA ACCESS")
print("-" * 76)

metadata_access_status = "NOT_ATTEMPTED"
metadata_access_error = None
metadata_access_error_type = None

try:
    from transformers import AutoConfig

    config_probe = AutoConfig.from_pretrained(
        MODEL_ID,
        trust_remote_code=False,
    )

    metadata_access_status = "ACCESSIBLE"

    print("  status : ACCESSIBLE")
    print(f"  config : {type(config_probe).__name__}")

    del config_probe

except Exception as exc:

    metadata_access_status = "FAILED"
    metadata_access_error = repr(exc)
    metadata_access_error_type = type(exc).__name__

    print("  status      : FAILED")
    print(f"  exception   : {metadata_access_error_type}")
    print(f"  message     : {str(exc)[:2000]}")

# ============================================================================
# 9. ACCESS CLASSIFICATION
# ============================================================================

print("\nACCESS CLASSIFICATION")
print("-" * 76)

if metadata_access_status == "ACCESSIBLE":
    access_classification = "CHECKPOINT_METADATA_ACCESSIBLE"

elif (
    metadata_access_error_type is not None
    and "Gated" in metadata_access_error_type
):
    access_classification = "CHECKPOINT_GATED_AUTHORIZATION_REQUIRED"

elif "401" in str(metadata_access_error):
    access_classification = "CHECKPOINT_UNAUTHORIZED_AUTH_REQUIRED"

elif cache_file_inventory:
    access_classification = "REMOTE_ACCESS_FAILED_LOCAL_COPY_MAY_EXIST"

elif python_hub_status == "AUTHENTICATED":
    access_classification = (
        "AUTHENTICATED_BUT_CHECKPOINT_ACCESS_REQUIRES_PERMISSION_OR_LICENSE"
    )

else:
    access_classification = "CHECKPOINT_ACCESS_UNRESOLVED"

print(f"  STATUS: {access_classification}")

# ============================================================================
# 10. EXPERIMENTAL CONSEQUENCE
# ============================================================================

print("\nEXPERIMENTAL CONSEQUENCE")
print("-" * 76)

if metadata_access_status == "ACCESSIBLE":

    next_stage = (
        "PROCEED_TO_CONFIGURATION_AND_IMPLEMENTATION_AUDIT"
    )

elif cache_file_inventory:

    next_stage = (
        "AUDIT_LOCAL_CHECKPOINT_BEFORE_ANY_REMOTE_DOWNLOAD"
    )

elif (
    hf_cli_status == "AUTHENTICATED"
    or python_hub_status == "AUTHENTICATED"
):

    next_stage = (
        "CHECK_ACCOUNT_MODEL_PERMISSION_LICENSE_STATE"
    )

else:

    next_stage = (
        "AUTHENTICATION_OR_CHECKPOINT_PERMISSION_REQUIRED"
    )

print(f"  NEXT STAGE: {next_stage}")

# ============================================================================
# 11. EXPERIMENTAL FIREWALL
# ============================================================================

print("\nEXPERIMENTAL FIREWALL")
print("-" * 76)

print("  Llama weights loaded       : FALSE")
print("  Llama model instantiated   : FALSE")
print("  Scientific inference       : FALSE")
print("  CTL runtime executed       : FALSE")
print("  Experimental dataset used  : FALSE")
print("  CTL mathematics modified   : FALSE")
print("  Checkpoint changed         : FALSE")

# ============================================================================
# 12. PERSIST DIAGNOSTIC ARTIFACT
# ============================================================================

artifact = {
    "experiment_id": EXPERIMENT_ID,
    "diagnostic_id": "LLAMA-CHECKPOINT-ACCESS-PATH",
    "model_id": MODEL_ID,

    "environment": {
        "python": sys.version,
        "hf_home": hf_home,
        "hf_hub_cache": hf_hub_cache,
    },

    "authentication": {
        "hf_token_environment_variable_present": hf_token_env_present,
        "hf_cli_status": hf_cli_status,
        "hf_cli_identity_available": hf_cli_identity is not None,
        "hf_cli_error": hf_cli_error,
        "python_hub_status": python_hub_status,
        "python_hub_identity_available": (
            python_hub_identity is not None
        ),
        "python_hub_error": python_hub_error,
    },

    "local_cache": {
        "candidate_directories": cache_candidates,
        "relevant_file_inventory": cache_file_inventory,
        "local_model_matches": local_matches[:100],
    },

    "metadata_access": {
        "status": metadata_access_status,
        "exception_type": metadata_access_error_type,
        "error": metadata_access_error,
    },

    "classification": access_classification,
    "next_stage": next_stage,

    "experimental_firewall": {
        "weights_loaded": False,
        "model_instantiated": False,
        "scientific_inference": False,
        "ctl_runtime": False,
        "experimental_dataset_used": False,
        "ctl_mathematics_modified": False,
        "checkpoint_changed": False,
    },

    "read_only": True,
}

artifact_path = os.path.join(
    RESULTS_DIR,
    "llama_checkpoint_access_path_diagnostic.json",
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(
        artifact,
        f,
        indent=2,
        default=str,
    )

print("\nDiagnostic artifact saved:")
print(f"  {artifact_path}")

print("\n" + "=" * 76)
print("CHECKPOINT ACCESS PATH DIAGNOSTIC COMPLETE")
print("=" * 76)

ETTR-CTL-LLAMA — CHECKPOINT ACCESS PATH DIAGNOSTIC
Experiment : ETTR-CTL-LLAMA-1
Model ID   : meta-llama/Llama-3.2-3B

ENVIRONMENT
----------------------------------------------------------------------------
Python       : 3.13.15
PyTorch      : 2.11.0+cu128
CUDA         : 12.8
Transformers : 5.16.1

HUGGING FACE AUTHENTICATION DIAGNOSTIC
----------------------------------------------------------------------------
  HF_TOKEN                    : ABSENT
  HUGGINGFACE_HUB_TOKEN       : ABSENT
  HUGGING_FACE_HUB_TOKEN      : ABSENT

  HF_HOME     : /root/.cache/huggingface
  HF_HUB_CACHE: /root/.cache/huggingface/hub

HUGGING FACE CLI AUTH STATUS
----------------------------------------------------------------------------
  status   : NOT AUTHENTICATED / UNAVAILABLE
  message  : Hint: A new version of huggingface_hub (1.31.0) is available! You are using version 1.29.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI age

In [10]:
# ============================================================================
# ETTR-CTL-LLAMA — POST-AUTHENTICATION CHECKPOINT ACCESS VERIFICATION
# ============================================================================
#
# Purpose:
#   Verify that the authenticated Hugging Face account can actually access
#   the exact intended Llama checkpoint before the architecture audit proceeds.
#
# This cell:
#   - checks authentication status;
#   - attempts metadata/config access;
#   - verifies tokenizer access;
#   - inspects the available repository metadata;
#   - checks whether model files are discoverable WITHOUT downloading weights.
#
# This cell DOES NOT:
#   - load model weights;
#   - instantiate Llama;
#   - run inference;
#   - use experimental data;
#   - invoke CTL;
#   - modify CTL mathematics;
#   - change the checkpoint.
# ============================================================================

import os
import sys
import json
import traceback

from pathlib import Path

print("=" * 76)
print("ETTR-CTL-LLAMA — POST-AUTHENTICATION CHECKPOINT ACCESS VERIFICATION")
print("=" * 76)

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
MODEL_ID = "meta-llama/Llama-3.2-3B"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Experiment : {EXPERIMENT_ID}")
print(f"Model ID   : {MODEL_ID}")
print()

# ============================================================================
# 1. ENVIRONMENT
# ============================================================================

print("ENVIRONMENT")
print("-" * 76)

print(f"Python       : {sys.version.split()[0]}")

try:
    import torch
    print(f"PyTorch      : {torch.__version__}")
    print(f"CUDA         : {torch.version.cuda}")
except Exception as exc:
    print(f"PyTorch import failed: {exc}")

try:
    import transformers
    print(f"Transformers : {transformers.__version__}")
except Exception as exc:
    print(f"Transformers import failed: {exc}")

# ============================================================================
# 2. AUTHENTICATION STATUS
# ============================================================================

print("\nHUGGING FACE AUTHENTICATION")
print("-" * 76)

hf_cli_status = "UNKNOWN"
hf_cli_identity = None
hf_cli_error = None

try:
    import subprocess

    result = subprocess.run(
        ["hf", "auth", "whoami"],
        capture_output=True,
        text=True,
        timeout=15,
    )

    if result.returncode == 0:
        hf_cli_status = "AUTHENTICATED"
        hf_cli_identity = result.stdout.strip()

        # Do not expose credentials.
        print("  CLI status : AUTHENTICATED")
        print("  identity   : available")

    else:
        hf_cli_status = "NOT_AUTHENTICATED"
        hf_cli_error = result.stderr.strip()

        print("  CLI status : NOT AUTHENTICATED")
        print(f"  message    : {hf_cli_error[:1000]}")

except Exception as exc:
    hf_cli_status = "AUTH_CHECK_FAILED"
    hf_cli_error = repr(exc)

    print(f"  CLI status : CHECK FAILED")
    print(f"  error      : {exc}")

python_hub_status = "UNKNOWN"
python_hub_identity = None
python_hub_error = None

try:
    from huggingface_hub import whoami

    try:
        identity = whoami()

        python_hub_status = "AUTHENTICATED"
        python_hub_identity = identity

        print("  Python Hub : AUTHENTICATED")

    except Exception as exc:
        python_hub_status = "NOT_AUTHENTICATED"
        python_hub_error = repr(exc)

        print("  Python Hub : NOT AUTHENTICATED")
        print(f"  message    : {exc}")

except Exception as exc:
    python_hub_status = "IMPORT_FAILED"
    python_hub_error = repr(exc)

    print(f"  Python Hub : IMPORT FAILED")
    print(f"  error      : {exc}")

# ============================================================================
# 3. AUTHENTICATED METADATA ACCESS
# ============================================================================

print("\nEXACT CHECKPOINT METADATA ACCESS")
print("-" * 76)

metadata_status = "NOT_ATTEMPTED"
metadata_config_class = None
metadata_error_type = None
metadata_error = None
config = None

try:
    from transformers import AutoConfig

    config = AutoConfig.from_pretrained(
        MODEL_ID,
        trust_remote_code=False,
    )

    metadata_status = "ACCESSIBLE"
    metadata_config_class = type(config).__name__

    print("  status      : ACCESSIBLE")
    print(f"  config type : {metadata_config_class}")

except Exception as exc:
    metadata_status = "FAILED"
    metadata_error_type = type(exc).__name__
    metadata_error = str(exc)

    print("  status      : FAILED")
    print(f"  exception   : {metadata_error_type}")
    print(f"  message     : {metadata_error[:2000]}")

# ============================================================================
# 4. TOKENIZER ACCESS
# ============================================================================

print("\nEXACT CHECKPOINT TOKENIZER ACCESS")
print("-" * 76)

tokenizer_status = "NOT_ATTEMPTED"
tokenizer_class = None
tokenizer_error_type = None
tokenizer_error = None
tokenizer = None

try:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=False,
    )

    tokenizer_status = "ACCESSIBLE"
    tokenizer_class = type(tokenizer).__name__

    print("  status         : ACCESSIBLE")
    print(f"  tokenizer type : {tokenizer_class}")
    print(f"  vocabulary     : {len(tokenizer)}")

except Exception as exc:
    tokenizer_status = "FAILED"
    tokenizer_error_type = type(exc).__name__
    tokenizer_error = str(exc)

    print("  status         : FAILED")
    print(f"  exception      : {tokenizer_error_type}")
    print(f"  message        : {tokenizer_error[:2000]}")

# ============================================================================
# 5. REPOSITORY FILE METADATA
# ============================================================================
#
# list_repo_files is metadata-oriented. We use it to determine whether the
# authenticated account can see the repository contents. It does not load
# model weights into memory.
# ============================================================================

print("\nREPOSITORY FILE VISIBILITY")
print("-" * 76)

repo_files_status = "NOT_ATTEMPTED"
repo_files = []
repo_files_error_type = None
repo_files_error = None

try:
    from huggingface_hub import list_repo_files

    repo_files = list(
        list_repo_files(
            repo_id=MODEL_ID,
            repo_type="model",
        )
    )

    repo_files_status = "ACCESSIBLE"

    print(f"  status       : ACCESSIBLE")
    print(f"  file count   : {len(repo_files)}")

    print("\n  Selected repository files:")

    interesting = [
        path
        for path in repo_files
        if any(
            marker in path.lower()
            for marker in [
                "config.json",
                "tokenizer",
                "generation_config",
                "model.safetensors.index.json",
                "pytorch_model.bin.index.json",
            ]
        )
    ]

    for path in interesting[:100]:
        print(f"    - {path}")

    if not interesting:
        print("    No selected metadata files reported.")

except Exception as exc:
    repo_files_status = "FAILED"
    repo_files_error_type = type(exc).__name__
    repo_files_error = str(exc)

    print("  status       : FAILED")
    print(f"  exception    : {repo_files_error_type}")
    print(f"  message      : {repo_files_error[:2000]}")

# ============================================================================
# 6. CHECKPOINT WEIGHT VISIBILITY — METADATA ONLY
# ============================================================================
#
# We explicitly inspect whether the repository exposes weight filenames.
# We do NOT download them.
# ============================================================================

print("\nMODEL-WEIGHT METADATA VISIBILITY")
print("-" * 76)

weight_files = []

for path in repo_files:
    lower = path.lower()

    if (
        lower.endswith(".safetensors")
        or lower.endswith(".bin")
        or "model.safetensors.index.json" in lower
        or "pytorch_model.bin.index.json" in lower
    ):
        weight_files.append(path)

print(f"  weight/index files visible: {len(weight_files)}")

for path in weight_files[:50]:
    print(f"    - {path}")

if len(weight_files) > 50:
    print(f"    ... {len(weight_files) - 50} additional files omitted")

# ============================================================================
# 7. CONFIGURATION SNAPSHOT
# ============================================================================

configuration_snapshot = {}

if config is not None:

    print("\nCONFIGURATION SNAPSHOT")
    print("-" * 76)

    fields = [
        "model_type",
        "architectures",
        "hidden_size",
        "num_hidden_layers",
        "num_attention_heads",
        "num_key_value_heads",
        "intermediate_size",
        "vocab_size",
        "max_position_embeddings",
        "rope_theta",
        "rope_scaling",
        "rms_norm_eps",
        "hidden_act",
        "attention_bias",
        "mlp_bias",
        "tie_word_embeddings",
        "torch_dtype",
    ]

    for field in fields:
        value = getattr(config, field, None)
        configuration_snapshot[field] = (
            str(value)
            if value is not None
            else None
        )

        print(f"  {field:28s}: {value}")

# ============================================================================
# 8. EXACT ACCESS CLASSIFICATION
# ============================================================================

print("\nCHECKPOINT ACCESS CLASSIFICATION")
print("-" * 76)

if (
    metadata_status == "ACCESSIBLE"
    and tokenizer_status == "ACCESSIBLE"
    and repo_files_status == "ACCESSIBLE"
):

    access_classification = (
        "AUTHORIZED_CHECKPOINT_METADATA_ACCESS_CONFIRMED"
    )

elif metadata_status == "ACCESSIBLE":

    access_classification = (
        "CHECKPOINT_CONFIG_ACCESSIBLE_REPOSITORY_VISIBILITY_INCOMPLETE"
    )

elif (
    hf_cli_status == "AUTHENTICATED"
    or python_hub_status == "AUTHENTICATED"
):

    access_classification = (
        "AUTHENTICATED_BUT_CHECKPOINT_ACCESS_NOT_CONFIRMED"
    )

else:

    access_classification = "CHECKPOINT_ACCESS_NOT_AUTHENTICATED"

print(f"  STATUS: {access_classification}")

# ============================================================================
# 9. MODEL-LOADING FIREWALL
# ============================================================================

print("\nMODEL-LOADING FIREWALL")
print("-" * 76)

model_loaded = False

if "torch" in globals():
    model_objects = []

    for name, obj in list(globals().items()):
        try:
            if isinstance(obj, torch.nn.Module):
                model_objects.append(name)
        except Exception:
            pass

    if model_objects:
        raise RuntimeError(
            "A torch.nn.Module exists during the metadata-only checkpoint "
            "access diagnostic. Model-loading firewall violated."
        )

print("  Llama weights loaded      : FALSE")
print("  Llama model instantiated  : FALSE")
print("  Scientific inference      : FALSE")
print("  CTL runtime               : FALSE")
print("  Experimental data         : FALSE")
print("  CTL mathematics modified  : FALSE")
print("  Checkpoint identity       : UNCHANGED")

# ============================================================================
# 10. PERSIST VERIFICATION ARTIFACT
# ============================================================================

artifact = {
    "experiment_id": EXPERIMENT_ID,
    "diagnostic_id": "POST-AUTHENTICATION-CHECKPOINT-ACCESS",
    "model_id": MODEL_ID,

    "environment": {
        "python": sys.version,
        "transformers": (
            transformers.__version__
            if "transformers" in globals()
            else None
        ),
        "torch": (
            torch.__version__
            if "torch" in globals()
            else None
        ),
        "cuda": (
            torch.version.cuda
            if "torch" in globals()
            else None
        ),
    },

    "authentication": {
        "cli_status": hf_cli_status,
        "cli_identity_available": hf_cli_identity is not None,
        "cli_error": hf_cli_error,
        "python_hub_status": python_hub_status,
        "python_hub_identity_available": (
            python_hub_identity is not None
        ),
        "python_hub_error": python_hub_error,
    },

    "metadata_access": {
        "status": metadata_status,
        "config_class": metadata_config_class,
        "error_type": metadata_error_type,
        "error": metadata_error,
    },

    "tokenizer_access": {
        "status": tokenizer_status,
        "class": tokenizer_class,
        "error_type": tokenizer_error_type,
        "error": tokenizer_error,
    },

    "repository_visibility": {
        "status": repo_files_status,
        "file_count": len(repo_files),
        "error_type": repo_files_error_type,
        "error": repo_files_error,
    },

    "weight_metadata_visibility": {
        "file_count": len(weight_files),
        "files": weight_files,
    },

    "configuration": configuration_snapshot,

    "classification": access_classification,

    "experimental_firewall": {
        "weights_loaded": False,
        "model_instantiated": False,
        "scientific_inference": False,
        "ctl_runtime": False,
        "experimental_data": False,
        "ctl_mathematics_modified": False,
        "checkpoint_changed": False,
    },

    "read_only": True,
}

artifact_path = os.path.join(
    RESULTS_DIR,
    "llama_post_auth_checkpoint_access_verification.json",
)

with open(artifact_path, "w", encoding="utf-8") as f:
    json.dump(
        artifact,
        f,
        indent=2,
        default=str,
    )

print("\nVerification artifact saved:")
print(f"  {artifact_path}")

print("\n" + "=" * 76)
print("POST-AUTHENTICATION CHECKPOINT VERIFICATION COMPLETE")
print("=" * 76)

ETTR-CTL-LLAMA — POST-AUTHENTICATION CHECKPOINT ACCESS VERIFICATION
Experiment : ETTR-CTL-LLAMA-1
Model ID   : meta-llama/Llama-3.2-3B

ENVIRONMENT
----------------------------------------------------------------------------
Python       : 3.13.15
PyTorch      : 2.11.0+cu128
CUDA         : 12.8
Transformers : 5.16.1

HUGGING FACE AUTHENTICATION
----------------------------------------------------------------------------
  CLI status : NOT AUTHENTICATED
  message    : Error: Not logged in
  Python Hub : NOT AUTHENTICATED
  message    : Token is required to call the /whoami-v2 endpoint, but no token found. You must provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

EXACT CHECKPOINT METADATA ACCESS
----------------------------------------------------------------------------
  status      : FAILED
  exception   : OSError
  message     : You are trying to access a gated repo.
Make sure to have access 

In [13]:
import os
import json
import gc
import torch

from huggingface_hub import login, whoami, get_token
from transformers import AutoConfig

print("=" * 76)
print("ETTR-CTL-LLAMA — HUGGING FACE COLAB AUTHENTICATION")
print("=" * 76)

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
MODEL_ID = "meta-llama/Llama-3.2-3B"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Experiment : {EXPERIMENT_ID}")
print(f"Model ID   : {MODEL_ID}")
print()

# ============================================================================
# 1. EXPERIMENT FIREWALL — NO MODEL RESIDENCY
# ============================================================================

print("PRE-AUTHENTICATION EXPERIMENT FIREWALL")
print("-" * 76)

gc.collect()

cuda_tensor_count = 0
cuda_parameter_count = 0
model_object_names = []

if torch.cuda.is_available():

    for obj_name, obj in list(globals().items()):

        try:

            if torch.is_tensor(obj) and obj.is_cuda:
                cuda_tensor_count += 1

            elif isinstance(obj, torch.nn.Module):

                model_object_names.append(obj_name)

                for parameter in obj.parameters(recurse=True):

                    if parameter.is_cuda:
                        cuda_parameter_count += 1

        except Exception:
            pass

print(f"  CUDA tensors       : {cuda_tensor_count}")
print(f"  CUDA parameters    : {cuda_parameter_count}")
print(f"  Model objects     : {model_object_names}")

assert cuda_tensor_count == 0, (
    "CUDA tensors are already resident. "
    "Authentication cell refuses to continue."
)

assert cuda_parameter_count == 0, (
    "CUDA parameters are already resident. "
    "Authentication cell refuses to continue."
)

assert not model_object_names, (
    f"Unexpected model objects already exist: {model_object_names}"
)

print("  Residency firewall : PASS")

# ============================================================================
# 2. HUGGING FACE API AUDIT
# ============================================================================

print("\nHUGGING FACE API AUDIT")
print("-" * 76)

try:
    import huggingface_hub

    HF_VERSION = getattr(
        huggingface_hub,
        "__version__",
        "UNKNOWN",
    )

except Exception:

    HF_VERSION = "UNKNOWN"

print(f"  huggingface_hub version : {HF_VERSION}")
print("  Authentication API      : login(skip_if_logged_in=False)")

# ============================================================================
# 3. INTERACTIVE AUTHENTICATION
# ============================================================================

print("\nHUGGING FACE AUTHENTICATION")
print("-" * 76)

print("The Hugging Face authentication interface should appear below.")
print("Enter the NEW replacement Read token through that interface.")
print("Do NOT paste the token into notebook source code.")
print("Re-authentication is explicitly forced for this cell.")
print()

login(
    skip_if_logged_in=False,
)

print("\n  Login operation returned successfully.")

# ============================================================================
# 4. VERIFY TOKEN RESOLUTION
# ============================================================================

print("\nTOKEN RESOLUTION")
print("-" * 76)

resolved_token = get_token()

if not resolved_token:

    raise RuntimeError(
        "Hugging Face login returned, but no token is resolvable by "
        "the current Python runtime."
    )

print("  Hugging Face token : RESOLVED")
print("  Token value        : <SUPPRESSED>")

# ============================================================================
# 5. VERIFY ACCOUNT IDENTITY
# ============================================================================

print("\nHUGGING FACE ACCOUNT VERIFICATION")
print("-" * 76)

try:

    identity = whoami(
        token=resolved_token,
    )

    if isinstance(identity, dict):

        safe_identity = {
            key: value
            for key, value in identity.items()
            if key.lower()
            not in {
                "token",
                "auth",
                "credentials",
            }
        }

        print("  Authentication : CONFIRMED")
        print(f"  Account        : {safe_identity}")

    else:

        print("  Authentication : CONFIRMED")
        print("  Account        : <identity returned>")

except Exception as exc:

    raise RuntimeError(
        "The Hugging Face token was resolved, but account verification "
        f"failed: {type(exc).__name__}: {exc}"
    ) from exc

# ============================================================================
# 6. VERIFY EXACT LLAMA CHECKPOINT METADATA ACCESS
# ============================================================================

print("\nEXACT LLAMA CHECKPOINT ACCESS")
print("-" * 76)

config = None

try:

    config = AutoConfig.from_pretrained(
        MODEL_ID,
        trust_remote_code=False,
        token=resolved_token,
    )

    print("  Checkpoint metadata : ACCESSIBLE")
    print(f"  Config class        : {type(config).__name__}")

except Exception as exc:

    print("  Checkpoint metadata : ACCESS FAILED")
    print(f"  Exception           : {type(exc).__name__}")
    print(f"  Message             : {str(exc)[:2000]}")

    raise RuntimeError(
        "\nAuthentication/account verification succeeded, but the "
        "exact gated checkpoint remains inaccessible.\n\n"
        "This is now an entitlement/checkpoint-access issue rather "
        "than a local authentication-resolution issue.\n\n"
        "The experiment must remain paused until the authenticated "
        "account has access to meta-llama/Llama-3.2-3B."
    ) from exc

# ============================================================================
# 7. RECORD ACTUAL CHECKPOINT CONFIGURATION
# ============================================================================

print("\nCHECKPOINT CONFIGURATION")
print("-" * 76)

configuration_fields = [
    "model_type",
    "architectures",
    "hidden_size",
    "num_hidden_layers",
    "num_attention_heads",
    "num_key_value_heads",
    "intermediate_size",
    "vocab_size",
    "max_position_embeddings",
    "rope_theta",
    "rope_scaling",
    "rms_norm_eps",
    "hidden_act",
    "attention_bias",
    "mlp_bias",
    "tie_word_embeddings",
    "torch_dtype",
]

configuration_snapshot = {}

for field in configuration_fields:

    value = getattr(
        config,
        field,
        None,
    )

    configuration_snapshot[field] = (
        str(value)
        if value is not None
        else None
    )

    print(
        f"  {field:28s}: {value}"
    )

# ============================================================================
# 8. POST-AUTHENTICATION EXPERIMENT FIREWALL
# ============================================================================

print("\nPOST-AUTHENTICATION EXPERIMENT FIREWALL")
print("-" * 76)

post_model_objects = []

for name, obj in list(globals().items()):

    try:

        if isinstance(obj, torch.nn.Module):
            post_model_objects.append(name)

    except Exception:
        pass

assert not post_model_objects, (
    f"Unexpected model objects found after authentication: "
    f"{post_model_objects}"
)

print("  Llama weights loaded      : FALSE")
print("  Llama model instantiated  : FALSE")
print("  Scientific inference      : FALSE")
print("  CTL runtime               : FALSE")
print("  Experimental dataset      : FALSE")
print("  CTL mathematics modified  : FALSE")
print("  Checkpoint changed        : FALSE")

# ============================================================================
# 9. PERSIST AUTHENTICATION VERIFICATION
# ============================================================================

artifact = {

    "experiment_id":
        EXPERIMENT_ID,

    "diagnostic_id":
        "COLAB-HF-AUTHENTICATION-V2",

    "model_id":
        MODEL_ID,

    "huggingface_hub_version":
        HF_VERSION,

    "authentication": {

        "authenticated":
            True,

        "token_resolved":
            True,

        "token_value_recorded":
            False,

        "authentication_api":
            "login(skip_if_logged_in=False)",
    },

    "checkpoint_access": {

        "metadata_accessible":
            True,

        "config_class":
            type(config).__name__,
    },

    "configuration":
        configuration_snapshot,

    "experimental_firewall": {

        "weights_loaded":
            False,

        "model_instantiated":
            False,

        "scientific_inference":
            False,

        "ctl_runtime":
            False,

        "experimental_dataset":
            False,

        "ctl_mathematics_modified":
            False,

        "checkpoint_changed":
            False,
    },

    "status":
        "AUTHORIZED_CHECKPOINT_METADATA_ACCESS_CONFIRMED",
}

artifact_path = os.path.join(
    RESULTS_DIR,
    "llama_colab_authentication_verification_v2.json",
)

with open(
    artifact_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        default=str,
    )

print("\nVerification artifact saved:")
print(f"  {artifact_path}")

print("\n" + "=" * 76)
print("ETTR-CTL-LLAMA — AUTHENTICATION VERIFIED")
print("=" * 76)

ETTR-CTL-LLAMA — HUGGING FACE COLAB AUTHENTICATION
Experiment : ETTR-CTL-LLAMA-1
Model ID   : meta-llama/Llama-3.2-3B

PRE-AUTHENTICATION EXPERIMENT FIREWALL
----------------------------------------------------------------------------
  CUDA tensors       : 0
  CUDA parameters    : 0
  Model objects     : []
  Residency firewall : PASS

HUGGING FACE API AUDIT
----------------------------------------------------------------------------
  huggingface_hub version : 1.29.0
  Authentication API      : login(skip_if_logged_in=False)

HUGGING FACE AUTHENTICATION
----------------------------------------------------------------------------
The Hugging Face authentication interface should appear below.
Enter the NEW replacement Read token through that interface.
Do NOT paste the token into notebook source code.
Re-authentication is explicitly forced for this cell.




  Login operation returned successfully.

TOKEN RESOLUTION
----------------------------------------------------------------------------
  Hugging Face token : RESOLVED
  Token value        : <SUPPRESSED>

HUGGING FACE ACCOUNT VERIFICATION
----------------------------------------------------------------------------
  Authentication : CONFIRMED
  Account        : {'type': 'user', 'id': '6aa33035f7a8e31b9a1ecc9e', 'name': 'onerospacetime', 'fullname': 'Felix Nyerovwo Wejeyan', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1790812800, 'isPro': False, 'avatarUrl': '/avatars/234ffbf8843c7a7092dcc5dcfd26d220.svg', 'orgs': []}

EXACT LLAMA CHECKPOINT ACCESS
----------------------------------------------------------------------------


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


  Checkpoint metadata : ACCESSIBLE
  Config class        : LlamaConfig

CHECKPOINT CONFIGURATION
----------------------------------------------------------------------------
  model_type                  : llama
  architectures               : ['LlamaForCausalLM']
  hidden_size                 : 3072
  num_hidden_layers           : 28
  num_attention_heads         : 24
  num_key_value_heads         : 8
  intermediate_size           : 8192
  vocab_size                  : 128256
  max_position_embeddings     : 131072
  rope_theta                  : None
  rope_scaling                : {'factor': 32.0, 'high_freq_factor': 4.0, 'low_freq_factor': 1.0, 'original_max_position_embeddings': 8192, 'rope_type': 'llama3', 'rope_theta': 500000.0}
  rms_norm_eps                : 1e-05
  hidden_act                  : silu
  attention_bias              : False
  mlp_bias                    : False
  tie_word_embeddings         : True
  torch_dtype                 : torch.bfloat16

POST-AUTHENTICATION

In [14]:
# ============================================================================
# ETTR-CTL-LLAMA — PHASE 1A
# ARCHITECTURE / IMPLEMENTATION / T4 FEASIBILITY AUDIT
# ============================================================================
#
# STATUS:
#   NEW CELL — do not remove previous cells.
#
# PURPOSE:
#   Audit the exact Llama 3.2 3B checkpoint configuration and the exact
#   installed Transformers implementation BEFORE loading model weights.
#
# GOVERNING RULE:
#   CTL mathematics is authoritative.
#   The implementation must adapt to the mathematical specification.
#   Transformer architecture is audited only to determine how the abstract
#   CTL sectors can be operationalized without changing their meaning.
#
# THIS CELL DOES NOT:
#   - load model weights;
#   - instantiate LlamaForCausalLM;
#   - run inference;
#   - use experimental data;
#   - construct CTL runtime;
#   - modify CTL mathematics;
#   - select empirical hyperparameters;
#   - access the test set.
#
# DIAGNOSTIC LAYER:
#   1. Checkpoint configuration
#   2. Installed implementation source
#   3. Computational ordering
#   4. Candidate operational sectors
#   5. Attention/GQA implementation
#   6. T4 numerical feasibility
#   7. Memory feasibility
#   8. Experimental firewall
#
# IMPORTANT:
#   Any disparity discovered between mathematical specification,
#   operationalization, and numerical execution is recorded as DECOHERENCE.
#   The mathematics is NOT altered to accommodate the implementation.
# ============================================================================

import os
import gc
import json
import math
import inspect
import re
import hashlib
import textwrap
import warnings

import torch
import transformers

from transformers import AutoConfig
from huggingface_hub import get_token

print("=" * 76)
print("ETTR-CTL-LLAMA — PHASE 1A")
print("ARCHITECTURE / IMPLEMENTATION / T4 FEASIBILITY AUDIT")
print("=" * 76)

# ============================================================================
# 0. EXPERIMENT IDENTIFIERS
# ============================================================================

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
AUDIT_ID = "LLAMA-PHASE1A-ARCHITECTURE-IMPLEMENTATION-AUDIT"
MODEL_ID = "meta-llama/Llama-3.2-3B"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Experiment : {EXPERIMENT_ID}")
print(f"Audit      : {AUDIT_ID}")
print(f"Model ID   : {MODEL_ID}")
print()

# ============================================================================
# 1. PRE-AUDIT EXPERIMENT FIREWALL
# ============================================================================

print("1. PRE-AUDIT EXPERIMENT FIREWALL")
print("-" * 76)

gc.collect()

cuda_tensor_count = 0
cuda_parameter_count = 0
module_names = []

for name, obj in list(globals().items()):

    try:

        if torch.is_tensor(obj):

            if obj.is_cuda:
                cuda_tensor_count += 1

        elif isinstance(obj, torch.nn.Module):

            module_names.append(name)

            for parameter in obj.parameters(recurse=True):

                if parameter.is_cuda:
                    cuda_parameter_count += 1

    except Exception:
        pass

print(f"  CUDA tensors       : {cuda_tensor_count}")
print(f"  CUDA parameters    : {cuda_parameter_count}")
print(f"  Module objects     : {module_names}")

assert cuda_tensor_count == 0, (
    "AUDIT REFUSED: CUDA tensors are already resident."
)

assert cuda_parameter_count == 0, (
    "AUDIT REFUSED: CUDA parameters are already resident."
)

assert not module_names, (
    f"AUDIT REFUSED: model/module objects already exist: {module_names}"
)

print("  Residency firewall : PASS")
print()

# ============================================================================
# 2. SOFTWARE ENVIRONMENT AUDIT
# ============================================================================

print("2. SOFTWARE ENVIRONMENT")
print("-" * 76)

software = {
    "python": None,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda_runtime": torch.version.cuda,
    "cuda_available": bool(torch.cuda.is_available()),
}

try:
    import sys
    software["python"] = sys.version
except Exception:
    software["python"] = "UNKNOWN"

print(f"  PyTorch      : {software['torch']}")
print(f"  Transformers : {software['transformers']}")
print(f"  CUDA runtime : {software['cuda_runtime']}")
print(f"  CUDA usable  : {software['cuda_available']}")

assert software["cuda_available"], (
    "CUDA is unavailable. T4 numerical audit cannot proceed."
)

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)

print(f"  GPU          : {gpu_name}")
print(f"  Compute cap. : {gpu_props.major}.{gpu_props.minor}")
print(f"  VRAM         : {gpu_props.total_memory / (1024**3):.3f} GiB")

print()

# ============================================================================
# 3. EXACT CHECKPOINT CONFIGURATION AUDIT
# ============================================================================

print("3. EXACT CHECKPOINT CONFIGURATION")
print("-" * 76)

resolved_token = get_token()

if not resolved_token:
    raise RuntimeError(
        "No Hugging Face token is resolvable. "
        "The authenticated checkpoint audit cannot continue."
    )

try:

    config = AutoConfig.from_pretrained(
        MODEL_ID,
        trust_remote_code=False,
        token=resolved_token,
    )

except Exception as exc:

    raise RuntimeError(
        "The exact Llama checkpoint configuration could not be retrieved "
        f"after authentication: {type(exc).__name__}: {exc}"
    ) from exc

print(f"  Config class          : {type(config).__name__}")
print(f"  model_type            : {getattr(config, 'model_type', None)}")
print(f"  architectures         : {getattr(config, 'architectures', None)}")
print(f"  hidden_size           : {getattr(config, 'hidden_size', None)}")
print(f"  num_hidden_layers     : {getattr(config, 'num_hidden_layers', None)}")
print(f"  num_attention_heads   : {getattr(config, 'num_attention_heads', None)}")
print(f"  num_key_value_heads   : {getattr(config, 'num_key_value_heads', None)}")
print(f"  intermediate_size     : {getattr(config, 'intermediate_size', None)}")
print(f"  vocab_size            : {getattr(config, 'vocab_size', None)}")
print(
    f"  max_position_embeddings : "
    f"{getattr(config, 'max_position_embeddings', None)}"
)
print(f"  hidden_act            : {getattr(config, 'hidden_act', None)}")
print(f"  rms_norm_eps          : {getattr(config, 'rms_norm_eps', None)}")
print(f"  attention_bias        : {getattr(config, 'attention_bias', None)}")
print(f"  mlp_bias              : {getattr(config, 'mlp_bias', None)}")
print(f"  tie_word_embeddings   : {getattr(config, 'tie_word_embeddings', None)}")
print(f"  torch_dtype           : {getattr(config, 'torch_dtype', None)}")
print(f"  rope_scaling          : {getattr(config, 'rope_scaling', None)}")

# Preserve the raw configuration as supplied by AutoConfig.
try:
    raw_config_dict = config.to_dict()
except Exception:
    raw_config_dict = {}

# ============================================================================
# 4. ARCHITECTURAL CONSISTENCY CHECKS
# ============================================================================

print("\n4. ARCHITECTURAL CONSISTENCY CHECKS")
print("-" * 76)

hidden_size = int(config.hidden_size)
num_layers = int(config.num_hidden_layers)
num_attention_heads = int(config.num_attention_heads)
num_kv_heads = int(config.num_key_value_heads)
intermediate_size = int(config.intermediate_size)
vocab_size = int(config.vocab_size)

head_dim = hidden_size // num_attention_heads

assert hidden_size % num_attention_heads == 0, (
    "hidden_size is not divisible by num_attention_heads."
)

assert num_attention_heads % num_kv_heads == 0, (
    "num_attention_heads is not divisible by num_key_value_heads."
)

gqa_group_size = num_attention_heads // num_kv_heads

print(f"  Attention head dimension : {head_dim}")
print(f"  Query heads              : {num_attention_heads}")
print(f"  KV heads                : {num_kv_heads}")
print(f"  GQA replication factor  : {gqa_group_size}")
print(f"  FFN expansion ratio     : {intermediate_size / hidden_size:.4f}")

print("  Hidden/head divisibility : PASS")
print("  GQA divisibility         : PASS")

# ============================================================================
# 5. IMPORT ACTUAL INSTALLED LLAMA IMPLEMENTATION
# ============================================================================

print("\n5. INSTALLED LLAMA IMPLEMENTATION")
print("-" * 76)

try:

    import transformers.models.llama.modeling_llama as modeling_llama

except Exception as exc:

    raise RuntimeError(
        "Could not import the installed Llama implementation: "
        f"{type(exc).__name__}: {exc}"
    ) from exc

print(
    f"  Implementation module : "
    f"{modeling_llama.__file__}"
)

implementation_source = inspect.getsource(modeling_llama)

implementation_source_sha256 = hashlib.sha256(
    implementation_source.encode("utf-8")
).hexdigest()

print(
    f"  Implementation SHA256 : "
    f"{implementation_source_sha256}"
)

# ============================================================================
# 6. DISCOVER ACTUAL LLAMA CLASSES
# ============================================================================

print("\n6. LLAMA IMPLEMENTATION CLASSES")
print("-" * 76)

candidate_class_names = [
    "LlamaForCausalLM",
    "LlamaModel",
    "LlamaDecoderLayer",
    "LlamaAttention",
    "LlamaMLP",
    "LlamaRMSNorm",
]

class_objects = {}

for class_name in candidate_class_names:

    obj = getattr(
        modeling_llama,
        class_name,
        None,
    )

    class_objects[class_name] = obj

    if obj is not None:

        print(f"  {class_name:24s}: FOUND")

    else:

        print(f"  {class_name:24s}: NOT FOUND")

assert class_objects["LlamaDecoderLayer"] is not None, (
    "The installed implementation does not expose LlamaDecoderLayer."
)

assert class_objects["LlamaAttention"] is not None, (
    "The installed implementation does not expose LlamaAttention."
)

assert class_objects["LlamaMLP"] is not None, (
    "The installed implementation does not expose LlamaMLP."
)

assert class_objects["LlamaRMSNorm"] is not None, (
    "The installed implementation does not expose LlamaRMSNorm."
)

# ============================================================================
# 7. SOURCE-LEVEL COMPUTATIONAL ORDER AUDIT
# ============================================================================

print("\n7. SOURCE-LEVEL COMPUTATIONAL ORDER AUDIT")
print("-" * 76)

decoder_source = inspect.getsource(
    class_objects["LlamaDecoderLayer"]
)

attention_source = inspect.getsource(
    class_objects["LlamaAttention"]
)

mlp_source = inspect.getsource(
    class_objects["LlamaMLP"]
)

rmsnorm_source = inspect.getsource(
    class_objects["LlamaRMSNorm"]
)

decoder_source_sha256 = hashlib.sha256(
    decoder_source.encode("utf-8")
).hexdigest()

attention_source_sha256 = hashlib.sha256(
    attention_source.encode("utf-8")
).hexdigest()

mlp_source_sha256 = hashlib.sha256(
    mlp_source.encode("utf-8")
).hexdigest()

rmsnorm_source_sha256 = hashlib.sha256(
    rmsnorm_source.encode("utf-8")
).hexdigest()

print(f"  Decoder source SHA256   : {decoder_source_sha256}")
print(f"  Attention source SHA256 : {attention_source_sha256}")
print(f"  MLP source SHA256       : {mlp_source_sha256}")
print(f"  RMSNorm source SHA256   : {rmsnorm_source_sha256}")

print("\n  LlamaDecoderLayer.forward source:")
print("-" * 76)
print(
    textwrap.indent(
        decoder_source,
        "  "
    )
)

# ============================================================================
# 8. OPERATIONAL ORDER DETECTION
# ============================================================================

print("\n8. OPERATIONAL ORDER DETECTION")
print("-" * 76)

decoder_lower = decoder_source.lower()

order_features = {
    "input_layernorm_present":
        "input_layernorm" in decoder_lower,

    "post_attention_layernorm_present":
        "post_attention_layernorm" in decoder_lower,

    "self_attn_call_present":
        "self_attn(" in decoder_lower,

    "mlp_call_present":
        "self.mlp(" in decoder_lower or
        "self.mlp (" in decoder_lower,

    "residual_add_present":
        "residual +" in decoder_lower or
        "hidden_states = residual" in decoder_lower,

    "attention_output_before_mlp":
        (
            decoder_lower.find("self_attn") >= 0
            and decoder_lower.find("self.mlp") >= 0
            and decoder_lower.find("self_attn")
            < decoder_lower.find("self.mlp")
        ),
}

for key, value in order_features.items():
    print(f"  {key:36s}: {value}")

# We require the actual source to contain the principal Transformer
# suboperations. We do NOT hard-code a mathematical interpretation here.
assert order_features["input_layernorm_present"], (
    "Could not verify input RMSNorm from the installed decoder source."
)

assert order_features["self_attn_call_present"], (
    "Could not verify self-attention call from the installed decoder source."
)

assert order_features["mlp_call_present"], (
    "Could not verify MLP call from the installed decoder source."
)

# ============================================================================
# 9. ATTENTION / GQA IMPLEMENTATION AUDIT
# ============================================================================

print("\n9. ATTENTION / GQA IMPLEMENTATION AUDIT")
print("-" * 76)

attention_lower = attention_source.lower()

attention_features = {

    "q_projection":
        "q_proj" in attention_lower,

    "k_projection":
        "k_proj" in attention_lower,

    "v_projection":
        "v_proj" in attention_lower,

    "output_projection":
        "o_proj" in attention_lower,

    "num_key_value_heads":
        "num_key_value_heads" in attention_lower,

    "num_key_value_groups":
        "num_key_value_groups" in attention_lower,

    "repeat_kv":
        "repeat_kv" in attention_lower,

    "scaled_dot_product_attention":
        "scaled_dot_product_attention" in attention_lower,

    "attention_interface":
        "attention_interface" in attention_lower,

    "flash_attention_reference":
        "flash_attention" in attention_lower,

    "eager_attention_reference":
        "eager_attention" in attention_lower,
}

for key, value in attention_features.items():
    print(f"  {key:36s}: {value}")

assert attention_features["q_projection"], (
    "Q projection was not detected."
)

assert attention_features["k_projection"], (
    "K projection was not detected."
)

assert attention_features["v_projection"], (
    "V projection was not detected."
)

assert attention_features["output_projection"], (
    "Attention output projection was not detected."
)

# ============================================================================
# 10. MLP IMPLEMENTATION AUDIT
# ============================================================================

print("\n10. MLP IMPLEMENTATION AUDIT")
print("-" * 76)

mlp_lower = mlp_source.lower()

mlp_features = {

    "gate_projection":
        "gate_proj" in mlp_lower,

    "up_projection":
        "up_proj" in mlp_lower,

    "down_projection":
        "down_proj" in mlp_lower,

    "activation":
        "act_fn" in mlp_lower or
        "silu" in mlp_lower,

    "elementwise_gate":
        "gate_proj" in mlp_lower
        and "up_proj" in mlp_lower,
}

for key, value in mlp_features.items():
    print(f"  {key:36s}: {value}")

assert mlp_features["gate_projection"], (
    "MLP gate projection was not detected."
)

assert mlp_features["up_projection"], (
    "MLP up projection was not detected."
)

assert mlp_features["down_projection"], (
    "MLP down projection was not detected."
)

# ============================================================================
# 11. RMSNORM IMPLEMENTATION AUDIT
# ============================================================================

print("\n11. RMSNORM IMPLEMENTATION AUDIT")
print("-" * 76)

rms_lower = rmsnorm_source.lower()

rms_features = {

    "variance_or_mean_square":
        "mean" in rms_lower or
        "variance" in rms_lower,

    "eps":
        "eps" in rms_lower,

    "weight":
        "weight" in rms_lower,
}

for key, value in rms_features.items():
    print(f"  {key:36s}: {value}")

assert rms_features["eps"], (
    "RMSNorm epsilon was not detected in the installed implementation."
)

assert rms_features["weight"], (
    "RMSNorm learned weight was not detected."
)

# ============================================================================
# 12. ATTENTION BACKEND CONFIGURATION AUDIT
# ============================================================================

print("\n12. ATTENTION BACKEND CONFIGURATION")
print("-" * 76)

attention_backend_candidates = {}

for field in [
    "_attn_implementation",
    "attn_implementation",
    "attention_implementation",
]:

    try:

        value = getattr(
            config,
            field,
            None,
        )

    except Exception:

        value = None

    attention_backend_candidates[field] = (
        str(value)
        if value is not None
        else None
    )

    print(
        f"  {field:28s}: {value}"
    )

# Search source for the actual dispatch mechanisms.
backend_keywords = [
    "ALL_ATTENTION_FUNCTIONS",
    "AttentionInterface",
    "_attn_implementation",
    "sdpa",
    "flash_attention",
    "eager_attention",
]

backend_source_hits = {}

for keyword in backend_keywords:

    backend_source_hits[keyword] = (
        keyword.lower()
        in implementation_source.lower()
    )

print("\n  Implementation dispatch indicators:")

for key, value in backend_source_hits.items():

    print(
        f"    {key:28s}: {value}"
    )

# ============================================================================
# 13. ROPE IMPLEMENTATION AUDIT
# ============================================================================

print("\n13. ROPE IMPLEMENTATION AUDIT")
print("-" * 76)

rope_scaling = getattr(
    config,
    "rope_scaling",
    None,
)

rope_source_features = {

    "rope_scaling_present":
        rope_scaling is not None,

    "rope_type_llama3":
        isinstance(rope_scaling, dict)
        and rope_scaling.get("rope_type") == "llama3",

    "rope_theta_500000":
        isinstance(rope_scaling, dict)
        and float(rope_scaling.get("rope_theta", -1))
        == 500000.0,

    "original_context_8192":
        isinstance(rope_scaling, dict)
        and int(
            rope_scaling.get(
                "original_max_position_embeddings",
                -1,
            )
        ) == 8192,
}

for key, value in rope_source_features.items():
    print(f"  {key:36s}: {value}")

# ============================================================================
# 14. PARAMETER MEMORY FEASIBILITY ESTIMATE
# ============================================================================

print("\n14. PARAMETER MEMORY FEASIBILITY")
print("-" * 76)

# Architecture-level parameter estimate.
#
# This is intentionally an estimate rather than a claim about the exact
# checkpoint state_dict. Exact parameter count will be audited only after
# weights are loaded in a later diagnostic cell.

embedding_params = vocab_size * hidden_size

# Attention:
#   Q: d_model x d_model
#   K/V: d_model x (n_kv_heads * head_dim)
#   O: d_model x d_model
attention_params_per_layer = (
    hidden_size * hidden_size
    + hidden_size * (num_kv_heads * head_dim)
    + hidden_size * (num_kv_heads * head_dim)
    + hidden_size * hidden_size
)

# Llama-style gated MLP:
#   gate, up: d_model x intermediate
#   down: intermediate x d_model
mlp_params_per_layer = (
    hidden_size * intermediate_size
    + hidden_size * intermediate_size
    + intermediate_size * hidden_size
)

# Two RMSNorm vectors per decoder layer.
norm_params_per_layer = 2 * hidden_size

decoder_params = num_layers * (
    attention_params_per_layer
    + mlp_params_per_layer
    + norm_params_per_layer
)

# Final norm.
final_norm_params = hidden_size

# Tied embedding/output head means the output head is not independently
# counted when tie_word_embeddings=True.
estimated_unique_params = (
    embedding_params
    + decoder_params
    + final_norm_params
)

bytes_bf16 = 2
bytes_fp16 = 2
bytes_fp32 = 4

estimated_bf16_gib = (
    estimated_unique_params * bytes_bf16
    / (1024**3)
)

estimated_fp16_gib = (
    estimated_unique_params * bytes_fp16
    / (1024**3)
)

estimated_fp32_gib = (
    estimated_unique_params * bytes_fp32
    / (1024**3)
)

print(
    f"  Estimated unique parameters : "
    f"{estimated_unique_params:,}"
)

print(
    f"  Estimated BF16 weight memory : "
    f"{estimated_bf16_gib:.3f} GiB"
)

print(
    f"  Estimated FP16 weight memory : "
    f"{estimated_fp16_gib:.3f} GiB"
)

print(
    f"  Estimated FP32 weight memory : "
    f"{estimated_fp32_gib:.3f} GiB"
)

gpu_memory_gib = (
    gpu_props.total_memory
    / (1024**3)
)

# Conservative initial reservation. This is not a claim that the complete
# experiment fits; it is a pre-load feasibility screen.
initial_runtime_reserve_gib = 3.0

estimated_bf16_with_reserve = (
    estimated_bf16_gib
    + initial_runtime_reserve_gib
)

estimated_fp16_with_reserve = (
    estimated_fp16_gib
    + initial_runtime_reserve_gib
)

print(
    f"  T4 VRAM                    : "
    f"{gpu_memory_gib:.3f} GiB"
)

print(
    f"  BF16 + 3 GiB reserve       : "
    f"{estimated_bf16_with_reserve:.3f} GiB"
)

print(
    f"  FP16 + 3 GiB reserve       : "
    f"{estimated_fp16_with_reserve:.3f} GiB"
)

bf16_parameter_feasible = (
    estimated_bf16_with_reserve
    < gpu_memory_gib
)

fp16_parameter_feasible = (
    estimated_fp16_with_reserve
    < gpu_memory_gib
)

print(
    f"  BF16 preliminary feasibility : "
    f"{bf16_parameter_feasible}"
)

print(
    f"  FP16 preliminary feasibility : "
    f"{fp16_parameter_feasible}"
)

# ============================================================================
# 15. T4 DTYPE EXECUTION AUDIT
# ============================================================================

print("\n15. T4 DTYPE EXECUTION AUDIT")
print("-" * 76)

dtype_probe_results = {}

test_shapes = [
    (128, 128),
    (512, 512),
]

for dtype in [
    torch.float16,
    torch.bfloat16,
]:

    dtype_name = str(dtype)

    try:

        results_for_dtype = []

        for rows, cols in test_shapes:

            a = torch.randn(
                rows,
                cols,
                device="cuda",
                dtype=dtype,
            )

            b = torch.randn(
                cols,
                rows,
                device="cuda",
                dtype=dtype,
            )

            c = a @ b

            torch.cuda.synchronize()

            finite = bool(
                torch.isfinite(c).all().item()
            )

            results_for_dtype.append({
                "shape": [rows, cols],
                "finite": finite,
                "dtype": dtype_name,
            })

            del a
            del b
            del c

        dtype_probe_results[dtype_name] = {
            "supported": True,
            "results": results_for_dtype,
        }

        print(
            f"  {dtype_name:20s}: "
            "CUDA arithmetic PASS"
        )

    except Exception as exc:

        dtype_probe_results[dtype_name] = {
            "supported": False,
            "exception": type(exc).__name__,
            "message": str(exc)[:1000],
        }

        print(
            f"  {dtype_name:20s}: "
            f"CUDA arithmetic FAILED — {type(exc).__name__}"
        )

gc.collect()

try:
    torch.cuda.empty_cache()
except Exception:
    pass

# ============================================================================
# 16. T4 COMPUTE CAPABILITY CHECK
# ============================================================================

print("\n16. T4 COMPUTE CAPABILITY CHECK")
print("-" * 76)

compute_capability = (
    f"{gpu_props.major}.{gpu_props.minor}"
)

is_turing = (
    gpu_props.major == 7
    and gpu_props.minor == 5
)

print(f"  Compute capability : {compute_capability}")
print(f"  Turing architecture: {is_turing}")

# This is a hardware classification only.
# We do not silently reinterpret the model's configured dtype.
if is_turing:
    print(
        "  Hardware note      : T4/Turing detected; BF16 execution must "
        "be treated as an empirical runtime question."
    )

# ============================================================================
# 17. CANDIDATE CTL SECTOR MAPPING
# ============================================================================

print("\n17. CANDIDATE CTL SECTOR MAPPING")
print("-" * 76)

print(
    "The following are OPERATIONAL candidates, not claims that the "
    "Transformer has three ontological modules."
)

sector_candidates = {

    "S1_contextual_interaction": {

        "abstract_ctl_role":
            "contextual interaction / attention-mediated contextual update",

        "candidate_architecture_boundary":
            "LlamaAttention output within LlamaDecoderLayer",

        "candidate_source_component":
            "self_attn",

        "candidate_measurement":
            "attention block output before the residual addition",

        "status":
            "CANDIDATE_REQUIRES_RUNTIME_HOOK_AUDIT",
    },

    "S2_state_propagation": {

        "abstract_ctl_role":
            "state propagation across computational depth",

        "candidate_architecture_boundary":
            "residual state entering/exiting decoder layer",

        "candidate_source_component":
            "decoder residual stream",

        "candidate_measurement":
            "hidden state at decoder-layer boundary",

        "status":
            "CANDIDATE_REQUIRES_RUNTIME_HOOK_AUDIT",
    },

    "S3_feature_transformation": {

        "abstract_ctl_role":
            "feature transformation",

        "candidate_architecture_boundary":
            "LlamaMLP output within LlamaDecoderLayer",

        "candidate_source_component":
            "mlp",

        "candidate_measurement":
            "MLP block output before residual addition",

        "status":
            "CANDIDATE_REQUIRES_RUNTIME_HOOK_AUDIT",
    },
}

for sector_name, sector_info in sector_candidates.items():

    print(f"\n  {sector_name}")

    for key, value in sector_info.items():

        print(
            f"    {key:32s}: {value}"
        )

# ============================================================================
# 18. CTL MATHEMATICAL GOVERNANCE AUDIT
# ============================================================================

print("\n18. CTL MATHEMATICAL GOVERNANCE")
print("-" * 76)

ctl_governance = {

    "mathematics_authoritative":
        True,

    "three_sectors_operational_not_ontological":
        True,

    "triadic_admissibility_independent_of_pairwise":
        True,

    "undefined_distinct_from_false":
        True,

    "logical_transport_explicit":
        True,

    "logical_coherence_explicit":
        True,

    "implementation_constraints_may_not_change_math":
        True,

    "implementation_math_disparity_is_decoherence":
        True,

    "test_firewall":
        True,
}

for key, value in ctl_governance.items():

    print(
        f"  {key:52s}: {value}"
    )

# ============================================================================
# 19. PRELIMINARY DECOHERENCE CHECK
# ============================================================================

print("\n19. PRELIMINARY MATHEMATICS / OPERATIONALIZATION / "
      "NUMERICAL EXECUTION CHECK")
print("-" * 76)

decoherence_events = []

# Configuration vs intended three-sector operationalization.
# The architecture contains attention, residual propagation, and MLP paths,
# but this is not itself a proof of CTL triadic structure.
if not (
    order_features["self_attn_call_present"]
    and order_features["mlp_call_present"]
):
    decoherence_events.append({
        "layer": "operationalization",
        "classification": "OPERATIONAL_MISMATCH",
        "description":
            "Required attention/MLP computational boundaries could not "
            "be identified from the installed implementation."
    })

# GQA is an implementation property that must be preserved in the runtime.
if num_attention_heads != num_kv_heads:

    print(
        "  GQA detected: query and KV head counts differ."
    )

    print(
        "  CTL operationalization must not collapse GQA into ordinary "
        "MHA merely for implementation convenience."
    )

# Dtype disparity is only recorded if actual runtime arithmetic fails.
for dtype_name, result in dtype_probe_results.items():

    if not result.get("supported", False):

        decoherence_events.append({
            "layer": "numerical_execution",
            "classification": "NUMERICAL_MISMATCH",
            "description":
                f"{dtype_name} CUDA arithmetic failed in the T4 probe."
        })

if not decoherence_events:

    decoherence_status = "NO_PRELIMINARY_DECOHERENCE_DETECTED"

else:

    decoherence_status = "PRELIMINARY_DECOHERENCE_DETECTED"

print(f"  Status : {decoherence_status}")

for event in decoherence_events:

    print(
        f"  [{event['classification']}] "
        f"{event['description']}"
    )

# ============================================================================
# 20. MODEL-LOADING DECISION
# ============================================================================

print("\n20. MODEL-LOADING DECISION")
print("-" * 76)

# We deliberately do NOT load the model here.
#
# The audit establishes whether it is reasonable to proceed to a separate
# model-loading/state-residency diagnostic.
#
# BF16 is considered preliminarily executable only if the direct CUDA probe
# succeeds. It is NOT declared scientifically preferable by this cell.

bf16_probe_pass = dtype_probe_results.get(
    "torch.bfloat16",
    {}
).get(
    "supported",
    False,
)

fp16_probe_pass = dtype_probe_results.get(
    "torch.float16",
    {}
).get(
    "supported",
    False,
)

if (
    config.model_type == "llama"
    and
    class_objects["LlamaDecoderLayer"] is not None
    and
    class_objects["LlamaAttention"] is not None
    and
    class_objects["LlamaMLP"] is not None
    and
    bf16_probe_pass
    and
    bf16_parameter_feasible
):

    model_loading_decision = (
        "PRELIMINARY_MODEL_LOADING_DIAGNOSTIC_PERMITTED"
    )

elif (
    config.model_type == "llama"
    and
    class_objects["LlamaDecoderLayer"] is not None
    and
    class_objects["LlamaAttention"] is not None
    and
    class_objects["LlamaMLP"] is not None
    and
    fp16_probe_pass
    and
    fp16_parameter_feasible
):

    model_loading_decision = (
        "PRELIMINARY_MODEL_LOADING_DIAGNOSTIC_PERMITTED_FP16"
    )

else:

    model_loading_decision = (
        "MODEL_LOADING_BLOCKED_PENDING_AUDIT_RESOLUTION"
    )

print(
    f"  Decision : {model_loading_decision}"
)

# ============================================================================
# 21. AUDIT STATUS
# ============================================================================

audit_pass = (
    config.model_type == "llama"
    and
    class_objects["LlamaDecoderLayer"] is not None
    and
    class_objects["LlamaAttention"] is not None
    and
    class_objects["LlamaMLP"] is not None
    and
    class_objects["LlamaRMSNorm"] is not None
    and
    order_features["input_layernorm_present"]
    and
    order_features["self_attn_call_present"]
    and
    order_features["mlp_call_present"]
    and
    attention_features["q_projection"]
    and
    attention_features["k_projection"]
    and
    attention_features["v_projection"]
    and
    attention_features["output_projection"]
    and
    mlp_features["gate_projection"]
    and
    mlp_features["up_projection"]
    and
    mlp_features["down_projection"]
)

audit_status = (
    "ARCHITECTURE_IMPLEMENTATION_AUDIT_PASS"
    if audit_pass
    else
    "ARCHITECTURE_IMPLEMENTATION_AUDIT_REQUIRES_REVIEW"
)

print("\n" + "=" * 76)
print(f"PHASE 1A STATUS: {audit_status}")
print("=" * 76)

# ============================================================================
# 22. PERSIST AUDIT ARTIFACT
# ============================================================================

artifact = {

    "experiment_id":
        EXPERIMENT_ID,

    "audit_id":
        AUDIT_ID,

    "model_id":
        MODEL_ID,

    "software":
        software,

    "gpu": {

        "name":
            gpu_name,

        "compute_capability":
            compute_capability,

        "total_memory_gib":
            gpu_memory_gib,

        "is_turing":
            is_turing,
    },

    "checkpoint": {

        "config_class":
            type(config).__name__,

        "raw_config":
            raw_config_dict,

        "hidden_size":
            hidden_size,

        "num_hidden_layers":
            num_layers,

        "num_attention_heads":
            num_attention_heads,

        "num_key_value_heads":
            num_kv_heads,

        "head_dim":
            head_dim,

        "gqa_group_size":
            gqa_group_size,

        "intermediate_size":
            intermediate_size,

        "vocab_size":
            vocab_size,
    },

    "implementation": {

        "module":
            modeling_llama.__file__,

        "module_sha256":
            implementation_source_sha256,

        "decoder_source_sha256":
            decoder_source_sha256,

        "attention_source_sha256":
            attention_source_sha256,

        "mlp_source_sha256":
            mlp_source_sha256,

        "rmsnorm_source_sha256":
            rmsnorm_source_sha256,

        "classes_found": {
            key:
                value is not None
            for key, value in class_objects.items()
        },

        "decoder_features":
            order_features,

        "attention_features":
            attention_features,

        "mlp_features":
            mlp_features,

        "rmsnorm_features":
            rms_features,

        "backend_config":
            attention_backend_candidates,

        "backend_source_hits":
            backend_source_hits,
    },

    "rope": {

        "scaling":
            rope_scaling,

        "features":
            rope_source_features,
    },

    "t4_dtype_probe":
        dtype_probe_results,

    "memory_estimate": {

        "estimated_unique_parameters":
            estimated_unique_params,

        "estimated_bf16_gib":
            estimated_bf16_gib,

        "estimated_fp16_gib":
            estimated_fp16_gib,

        "estimated_fp32_gib":
            estimated_fp32_gib,

        "initial_runtime_reserve_gib":
            initial_runtime_reserve_gib,

        "bf16_parameter_feasible":
            bf16_parameter_feasible,

        "fp16_parameter_feasible":
            fp16_parameter_feasible,
    },

    "sector_candidates":
        sector_candidates,

    "ctl_governance":
        ctl_governance,

    "decoherence": {

        "status":
            decoherence_status,

        "events":
            decoherence_events,
    },

    "model_loading_decision":
        model_loading_decision,

    "experimental_firewall": {

        "weights_loaded":
            False,

        "model_instantiated":
            False,

        "scientific_inference":
            False,

        "ctl_runtime":
            False,

        "experimental_data":
            False,

        "test_data_accessed":
            False,

        "ctl_mathematics_modified":
            False,
    },

    "status":
        audit_status,
}

artifact_path = os.path.join(
    RESULTS_DIR,
    "llama_phase1a_architecture_implementation_audit.json",
)

with open(
    artifact_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        default=str,
    )

print("\nAudit artifact saved:")
print(f"  {artifact_path}")

print("\n" + "=" * 76)
print("PHASE 1A COMPLETE")
print("=" * 76)

ETTR-CTL-LLAMA — PHASE 1A
ARCHITECTURE / IMPLEMENTATION / T4 FEASIBILITY AUDIT
Experiment : ETTR-CTL-LLAMA-1
Audit      : LLAMA-PHASE1A-ARCHITECTURE-IMPLEMENTATION-AUDIT
Model ID   : meta-llama/Llama-3.2-3B

1. PRE-AUDIT EXPERIMENT FIREWALL
----------------------------------------------------------------------------
  CUDA tensors       : 0
  CUDA parameters    : 0
  Module objects     : []
  Residency firewall : PASS

2. SOFTWARE ENVIRONMENT
----------------------------------------------------------------------------
  PyTorch      : 2.11.0+cu128
  Transformers : 5.16.1
  CUDA runtime : 12.8
  CUDA usable  : True
  GPU          : Tesla T4
  Compute cap. : 7.5
  VRAM         : 14.563 GiB

3. EXACT CHECKPOINT CONFIGURATION
----------------------------------------------------------------------------
  Config class          : LlamaConfig
  model_type            : llama
  architectures         : ['LlamaForCausalLM']
  hidden_size           : 3072
  num_hidden_layers     : 28
  num_attentio

In [17]:
# ============================================================================
# ETTR-CTL-LLAMA — PHASE 1B RUNTIME RESIDENCY DIAGNOSTIC
# ============================================================================
# Purpose:
#   Diagnose the two CUDA tensors detected by the Phase 1B pre-load firewall.
#
# Scientific status:
#   RUNTIME / NUMERICAL-EXECUTION DIAGNOSTIC ONLY
#
# This cell MUST NOT:
#   - load the Llama checkpoint
#   - run model inference
#   - fit transport maps
#   - access scientific/test data
#   - alter CTL mathematics
#   - alter sector definitions
#   - clear or mutate the model deliberately
#
# The diagnostic distinguishes:
#   (A) model/parameter CUDA residency
#   (B) Python-reachable CUDA tensors
#   (C) CUDA allocator/runtime residency
#
# It does NOT assume that allocator memory == scientifically relevant state.
# ============================================================================

import os
import gc
import json
import sys
import traceback
from datetime import datetime, timezone

import torch


# ----------------------------------------------------------------------------
# 0. Experiment paths / identifiers
# ----------------------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
AUDIT_ID = "LLAMA-PHASE1B-RUNTIME-RESIDENCY-DIAGNOSTIC-V1"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

ARTIFACT_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1b_runtime_residency_diagnostic_v1.json"
)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


# ----------------------------------------------------------------------------
# 1. Header
# ----------------------------------------------------------------------------

print("=" * 76)
print("ETTR-CTL-LLAMA — PHASE 1B")
print("RUNTIME RESIDENCY DIAGNOSTIC")
print("=" * 76)
print(f"Experiment : {EXPERIMENT_ID}")
print(f"Audit      : {AUDIT_ID}")
print(f"Timestamp  : {datetime.now(timezone.utc).isoformat()}")
print(f"Python     : {sys.version.split()[0]}")
print(f"PyTorch    : {torch.__version__}")
print(f"CUDA avail : {torch.cuda.is_available()}")
print()


# ----------------------------------------------------------------------------
# 2. CUDA runtime / allocator state
# ----------------------------------------------------------------------------

allocator_before = None

if torch.cuda.is_available():
    torch.cuda.synchronize()

    allocator_before = {
        "memory_allocated_bytes": int(torch.cuda.memory_allocated()),
        "memory_reserved_bytes": int(torch.cuda.memory_reserved()),
        "max_memory_allocated_bytes": int(torch.cuda.max_memory_allocated()),
        "max_memory_reserved_bytes": int(torch.cuda.max_memory_reserved()),
        "device_name": torch.cuda.get_device_name(0),
        "device_index": 0,
        "device_capability": list(torch.cuda.get_device_capability(0)),
    }

    print("1. CUDA ALLOCATOR STATE")
    print("-" * 76)
    print(
        f"  Allocated : "
        f"{allocator_before['memory_allocated_bytes'] / 2**20:.3f} MiB"
    )
    print(
        f"  Reserved  : "
        f"{allocator_before['memory_reserved_bytes'] / 2**20:.3f} MiB"
    )
    print(
        f"  Peak alloc: "
        f"{allocator_before['max_memory_allocated_bytes'] / 2**20:.3f} MiB"
    )
    print(
        f"  Peak res. : "
        f"{allocator_before['max_memory_reserved_bytes'] / 2**20:.3f} MiB"
    )
    print(
        f"  Device    : {allocator_before['device_name']} "
        f"(CC {allocator_before['device_capability'][0]}."
        f"{allocator_before['device_capability'][1]})"
    )
else:
    print("CUDA unavailable; residency diagnostic cannot continue.")

print()


# ----------------------------------------------------------------------------
# 3. Enumerate Python-reachable CUDA tensors
# ----------------------------------------------------------------------------
#
# We inspect Python objects through gc.get_objects().
#
# Important:
#   This is deliberately different from torch.cuda.memory_allocated().
#   The allocator reports memory owned by the CUDA caching allocator.
#   gc inspection reports CUDA tensors that are still Python-reachable.
#
# We only collect metadata. We do NOT mutate the objects.
# ----------------------------------------------------------------------------

def tensor_metadata(t, object_index=None):
    """Return non-mutating metadata for a CUDA tensor."""
    try:
        numel = int(t.numel())
    except Exception:
        numel = None

    try:
        element_size = int(t.element_size())
    except Exception:
        element_size = None

    try:
        nbytes = int(t.numel() * t.element_size())
    except Exception:
        nbytes = None

    try:
        requires_grad = bool(t.requires_grad)
    except Exception:
        requires_grad = None

    try:
        is_leaf = bool(t.is_leaf)
    except Exception:
        is_leaf = None

    try:
        grad_fn = type(t.grad_fn).__name__ if t.grad_fn is not None else None
    except Exception:
        grad_fn = None

    try:
        shape = list(t.shape)
    except Exception:
        shape = None

    try:
        stride = list(t.stride())
    except Exception:
        stride = None

    try:
        storage_ptr = int(t.untyped_storage().data_ptr())
    except Exception:
        storage_ptr = None

    return {
        "object_index": object_index,
        "python_type": type(t).__name__,
        "device": str(t.device),
        "dtype": str(t.dtype),
        "shape": shape,
        "stride": stride,
        "numel": numel,
        "element_size_bytes": element_size,
        "nbytes": nbytes,
        "requires_grad": requires_grad,
        "is_leaf": is_leaf,
        "grad_fn": grad_fn,
        "storage_data_ptr": storage_ptr,
    }


cuda_tensors = []

if torch.cuda.is_available():
    # gc.get_objects() can itself encounter objects that disappear during
    # inspection, so every object access is protected.
    for idx, obj in enumerate(gc.get_objects()):
        try:
            if torch.is_tensor(obj) and obj.is_cuda:
                cuda_tensors.append(
                    tensor_metadata(obj, object_index=idx)
                )
        except Exception:
            continue


print("2. PYTHON-REACHABLE CUDA TENSORS")
print("-" * 76)
print(f"  Count : {len(cuda_tensors)}")

if len(cuda_tensors) == 0:
    print("  No Python-reachable CUDA tensors detected.")
else:
    for i, meta in enumerate(cuda_tensors):
        print()
        print(f"  Tensor {i + 1}")
        print(f"    Python type : {meta['python_type']}")
        print(f"    Device      : {meta['device']}")
        print(f"    Dtype       : {meta['dtype']}")
        print(f"    Shape       : {meta['shape']}")
        print(f"    Numel       : {meta['numel']}")
        print(f"    Bytes       : {meta['nbytes']}")
        print(f"    Grad        : {meta['requires_grad']}")
        print(f"    Leaf        : {meta['is_leaf']}")
        print(f"    Grad fn     : {meta['grad_fn']}")
        print(f"    Storage ptr : {meta['storage_data_ptr']}")

print()


# ----------------------------------------------------------------------------
# 4. Identify model objects currently present
# ----------------------------------------------------------------------------
#
# We deliberately do not assume that a variable called "model" is valid.
# We inspect the current namespace and report candidate model-like objects.
# ----------------------------------------------------------------------------

candidate_objects = {}

for name in [
    "model",
    "decoder_layers",
    "base_model",
    "audit_layer",
]:
    if name in globals():
        try:
            obj = globals()[name]
            candidate_objects[name] = {
                "python_type": type(obj).__name__,
                "is_torch_module": isinstance(obj, torch.nn.Module),
            }
        except Exception as exc:
            candidate_objects[name] = {
                "inspection_error": repr(exc)
            }


print("3. CURRENT MODEL-LIKE OBJECTS")
print("-" * 76)

if not candidate_objects:
    print("  No named model-like objects found.")
else:
    for name, info in candidate_objects.items():
        print(f"  {name}: {info}")

print()


# ----------------------------------------------------------------------------
# 5. Inspect named model objects for parameter/buffer CUDA residency
# ----------------------------------------------------------------------------

model_residency = {}

for name in [
    "model",
    "decoder_layers",
    "base_model",
    "audit_layer",
]:
    if name not in globals():
        continue

    try:
        obj = globals()[name]

        if not isinstance(obj, torch.nn.Module):
            model_residency[name] = {
                "is_module": False,
                "cuda_parameters": 0,
                "cuda_buffers": 0,
                "total_parameters": 0,
                "total_buffers": 0,
            }
            continue

        cuda_parameters = []
        cuda_buffers = []

        total_parameters = 0
        total_buffers = 0

        for pname, p in obj.named_parameters(recurse=True):
            total_parameters += 1
            if p.is_cuda:
                cuda_parameters.append({
                    "name": pname,
                    "shape": list(p.shape),
                    "dtype": str(p.dtype),
                    "device": str(p.device),
                    "nbytes": int(p.numel() * p.element_size()),
                })

        for bname, b in obj.named_buffers(recurse=True):
            total_buffers += 1
            if b.is_cuda:
                cuda_buffers.append({
                    "name": bname,
                    "shape": list(b.shape),
                    "dtype": str(b.dtype),
                    "device": str(b.device),
                    "nbytes": int(b.numel() * b.element_size()),
                })

        model_residency[name] = {
            "is_module": True,
            "cuda_parameters": len(cuda_parameters),
            "cuda_buffers": len(cuda_buffers),
            "total_parameters": total_parameters,
            "total_buffers": total_buffers,
            "cuda_parameter_details": cuda_parameters[:20],
            "cuda_buffer_details": cuda_buffers[:20],
        }

    except Exception as exc:
        model_residency[name] = {
            "inspection_error": repr(exc),
            "traceback": traceback.format_exc(),
        }


print("4. MODEL / MODULE CUDA RESIDENCY")
print("-" * 76)

for name, info in model_residency.items():
    print(f"  {name}")
    print(f"    Module        : {info.get('is_module')}")
    print(f"    Parameters    : "
          f"{info.get('cuda_parameters')} CUDA / "
          f"{info.get('total_parameters')} total")
    print(f"    Buffers       : "
          f"{info.get('cuda_buffers')} CUDA / "
          f"{info.get('total_buffers')} total")

print()


# ----------------------------------------------------------------------------
# 6. Search named objects for direct CUDA tensor attributes
# ----------------------------------------------------------------------------
#
# This is a shallow diagnostic only. We inspect __dict__ attributes of the
# named objects to determine whether the two CUDA tensors are directly held
# by one of the current runtime objects.
# ----------------------------------------------------------------------------

direct_attribute_hits = []

for name in [
    "model",
    "decoder_layers",
    "base_model",
    "audit_layer",
]:
    if name not in globals():
        continue

    try:
        obj = globals()[name]

        if not hasattr(obj, "__dict__"):
            continue

        for attr_name, attr_value in vars(obj).items():
            try:
                if torch.is_tensor(attr_value) and attr_value.is_cuda:
                    direct_attribute_hits.append({
                        "owner": name,
                        "attribute": attr_name,
                        "tensor": tensor_metadata(attr_value),
                    })
            except Exception:
                continue

    except Exception:
        continue


print("5. DIRECT CUDA-TENSOR ATTRIBUTE REFERENCES")
print("-" * 76)
print(f"  Direct hits : {len(direct_attribute_hits)}")

for hit in direct_attribute_hits[:20]:
    print(
        f"  {hit['owner']}.{hit['attribute']} -> "
        f"{hit['tensor']['dtype']} {hit['tensor']['shape']} "
        f"on {hit['tensor']['device']}"
    )

print()


# ----------------------------------------------------------------------------
# 7. Determine whether CUDA tensors overlap with model parameter storage
# ----------------------------------------------------------------------------

parameter_storage_ptrs = set()

for name in [
    "model",
    "decoder_layers",
    "base_model",
    "audit_layer",
]:
    if name not in globals():
        continue

    try:
        obj = globals()[name]

        if isinstance(obj, torch.nn.Module):
            for p in obj.parameters():
                if p.is_cuda:
                    try:
                        parameter_storage_ptrs.add(
                            int(p.untyped_storage().data_ptr())
                        )
                    except Exception:
                        pass
    except Exception:
        continue


cuda_tensor_parameter_overlap = []

for i, meta in enumerate(cuda_tensors):
    ptr = meta.get("storage_data_ptr")
    if ptr is not None and ptr in parameter_storage_ptrs:
        cuda_tensor_parameter_overlap.append(i)


print("6. CUDA TENSOR / MODEL-PARAMETER STORAGE OVERLAP")
print("-" * 76)
print(
    f"  CUDA tensors overlapping parameter storage : "
    f"{len(cuda_tensor_parameter_overlap)}"
)

if cuda_tensor_parameter_overlap:
    print(
        "  WARNING: at least one detected CUDA tensor shares storage "
        "with a CUDA-resident model parameter."
    )
else:
    print(
        "  No detected CUDA tensor shares storage with a currently "
        "CUDA-resident model parameter."
    )

print()


# ----------------------------------------------------------------------------
# 8. Classify the firewall state
# ----------------------------------------------------------------------------

cuda_parameter_count = sum(
    info.get("cuda_parameters", 0)
    for info in model_residency.values()
)

cuda_buffer_count = sum(
    info.get("cuda_buffers", 0)
    for info in model_residency.values()
)

if cuda_parameter_count > 0:
    classification = "MODEL_CUDA_RESIDENCY_DETECTED"

elif cuda_buffer_count > 0:
    classification = "MODEL_BUFFER_CUDA_RESIDENCY_DETECTED"

elif len(cuda_tensor_parameter_overlap) > 0:
    classification = "CUDA_TENSOR_OVERLAPS_MODEL_PARAMETER_STORAGE"

elif len(cuda_tensors) == 0:
    classification = "CLEAN_PRELOAD_RUNTIME"

else:
    classification = "PYTHON_REACHABLE_CUDA_TENSORS_NO_MODEL_PARAMETER_RESIDENCY"


print("7. PRELIMINARY CLASSIFICATION")
print("-" * 76)
print(f"  Classification : {classification}")
print()


# ----------------------------------------------------------------------------
# 9. Scientific / numerical interpretation
# ----------------------------------------------------------------------------

if classification == "CLEAN_PRELOAD_RUNTIME":
    interpretation = (
        "No Python-reachable CUDA tensors and no CUDA-resident model "
        "parameters/buffers were detected. The runtime is clean enough "
        "for the next Phase 1B loading attempt."
    )

elif classification == "PYTHON_REACHABLE_CUDA_TENSORS_NO_MODEL_PARAMETER_RESIDENCY":
    interpretation = (
        "CUDA tensors remain Python-reachable, but no CUDA-resident model "
        "parameters or buffers were detected and no tensor was found to "
        "overlap model-parameter storage. This is a runtime-state issue "
        "requiring explicit cleanup/diagnosis before model loading; it is "
        "not evidence of mathematical or operational decoherence."
    )

elif classification in (
    "MODEL_CUDA_RESIDENCY_DETECTED",
    "MODEL_BUFFER_CUDA_RESIDENCY_DETECTED",
    "CUDA_TENSOR_OVERLAPS_MODEL_PARAMETER_STORAGE",
):
    interpretation = (
        "Existing model-related CUDA state is present. The Phase 1B "
        "pre-load firewall should not be bypassed. Existing model/runtime "
        "state must be explicitly cleaned before a new controlled model "
        "load is attempted."
    )

else:
    interpretation = (
        "Runtime state requires further inspection before model loading."
    )


print("8. INTERPRETATION")
print("-" * 76)
print(f"  {interpretation}")
print()


# ----------------------------------------------------------------------------
# 10. Persist diagnostic artifact
# ----------------------------------------------------------------------------

artifact = {
    "experiment_id": EXPERIMENT_ID,
    "audit_id": AUDIT_ID,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "runtime": {
        "python": sys.version,
        "pytorch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
    },

    "allocator_before": allocator_before,

    "python_reachable_cuda_tensors": cuda_tensors,

    "candidate_objects": candidate_objects,

    "model_residency": model_residency,

    "direct_attribute_cuda_tensor_hits": direct_attribute_hits,

    "cuda_tensor_parameter_overlap_indices":
        cuda_tensor_parameter_overlap,

    "classification": classification,

    "interpretation": interpretation,

    "scientific_status": {
        "ctl_math_modified": False,
        "operational_sector_definitions_modified": False,
        "scientific_data_accessed": False,
        "model_loaded_by_this_cell": False,
        "inference_executed_by_this_cell": False,
        "maps_fitted_by_this_cell": False,
        "classification_layer": "NUMERICAL_EXECUTION_RUNTIME",
    },

    "governance": {
        "math_authoritative": True,
        "runtime_constraint_must_not_redefine_math": True,
        "decoherence_if_unresolved": True,
    },
}

with open(ARTIFACT_PATH, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2, default=str)


print("9. ARTIFACT")
print("-" * 76)
print(f"  Saved : {ARTIFACT_PATH}")
print()

print("=" * 76)
print("PHASE 1B RUNTIME RESIDENCY DIAGNOSTIC COMPLETE")
print("=" * 76)

ETTR-CTL-LLAMA — PHASE 1B
RUNTIME RESIDENCY DIAGNOSTIC
Experiment : ETTR-CTL-LLAMA-1
Audit      : LLAMA-PHASE1B-RUNTIME-RESIDENCY-DIAGNOSTIC-V1
Timestamp  : 2026-09-10T23:49:10.760244+00:00
Python     : 3.13.15
PyTorch    : 2.11.0+cu128
CUDA avail : True

1. CUDA ALLOCATOR STATE
----------------------------------------------------------------------------
  Allocated : 8.126 MiB
  Reserved  : 22.000 MiB
  Peak alloc: 8.126 MiB
  Peak res. : 22.000 MiB
  Device    : Tesla T4 (CC 7.5)

2. PYTHON-REACHABLE CUDA TENSORS
----------------------------------------------------------------------------
  Count : 2

  Tensor 1
    Python type : Tensor
    Device      : cuda:0
    Dtype       : torch.int64
    Shape       : [1, 14]
    Numel       : 14
    Bytes       : 112
    Grad        : False
    Leaf        : True
    Grad fn     : None
    Storage ptr : 133181317578752

  Tensor 2
    Python type : Tensor
    Device      : cuda:0
    Dtype       : torch.int64
    Shape       : [1, 14]
    Num

In [18]:
# ============================================================================
# ETTR-CTL-LLAMA — PHASE 1B
# CONTROLLED RUNTIME CLEANUP AND PRE-LOAD FIREWALL RESET
# ============================================================================
# Audit:
#   LLAMA-PHASE1B-RUNTIME-CLEANUP-V1
#
# Purpose:
#   Remove stale Python-reachable CUDA tensors left by the failed Phase 1B
#   smoke-inference attempt and establish a clean numerical-execution state
#   before retrying controlled model loading.
#
# Scientific status:
#   NUMERICAL-EXECUTION / RUNTIME ONLY
#
# This cell does NOT:
#   - load the Llama checkpoint
#   - run inference
#   - access scientific data
#   - fit transport maps
#   - modify CTL mathematics
#   - modify sector definitions
#   - alter experimental observations
# ============================================================================

import os
import gc
import json
import sys
from datetime import datetime, timezone

import torch


# ----------------------------------------------------------------------------
# 0. Configuration
# ----------------------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
AUDIT_ID = "LLAMA-PHASE1B-RUNTIME-CLEANUP-V1"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

ARTIFACT_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1b_runtime_cleanup_v1.json"
)


# ----------------------------------------------------------------------------
# 1. Header
# ----------------------------------------------------------------------------

print("=" * 76)
print("ETTR-CTL-LLAMA — PHASE 1B")
print("CONTROLLED RUNTIME CLEANUP AND PRE-LOAD FIREWALL RESET")
print("=" * 76)
print(f"Experiment : {EXPERIMENT_ID}")
print(f"Audit      : {AUDIT_ID}")
print(f"Timestamp  : {datetime.now(timezone.utc).isoformat()}")
print(f"Python     : {sys.version.split()[0]}")
print(f"PyTorch    : {torch.__version__}")
print()


# ----------------------------------------------------------------------------
# 2. Verify that no model parameters are currently CUDA-resident
# ----------------------------------------------------------------------------
#
# This is recorded BEFORE cleanup so that cleanup cannot be interpreted as
# hiding an existing CUDA-resident model.
# ----------------------------------------------------------------------------

model_names = [
    "model",
    "decoder_layers",
    "base_model",
    "audit_layer",
]

pre_cleanup_model_state = {}

for name in model_names:
    if name not in globals():
        continue

    try:
        obj = globals()[name]

        if isinstance(obj, torch.nn.Module):
            cuda_parameters = sum(
                1 for p in obj.parameters() if p.is_cuda
            )
            cuda_buffers = sum(
                1 for b in obj.buffers() if b.is_cuda
            )

            pre_cleanup_model_state[name] = {
                "python_type": type(obj).__name__,
                "cuda_parameters": cuda_parameters,
                "cuda_buffers": cuda_buffers,
            }

    except Exception as exc:
        pre_cleanup_model_state[name] = {
            "inspection_error": repr(exc)
        }


print("1. PRE-CLEANUP MODEL CUDA STATE")
print("-" * 76)

if not pre_cleanup_model_state:
    print("  No model-like modules found.")

for name, state in pre_cleanup_model_state.items():
    print(
        f"  {name}: "
        f"{state.get('cuda_parameters', 'N/A')} CUDA parameters, "
        f"{state.get('cuda_buffers', 'N/A')} CUDA buffers"
    )

print()


# ----------------------------------------------------------------------------
# 3. Identify the known stale synthetic input tensors
# ----------------------------------------------------------------------------
#
# The previous diagnostic established that exactly two Python-reachable CUDA
# tensors existed, both [1,14] int64 tensors, with no model-storage overlap.
#
# We now remove CUDA tensors from the current Python namespace where their
# names are known from the previous Phase 1B attempt.
#
# We do NOT blindly delete arbitrary globals. The cleanup is intentionally
# conservative and targeted.
# ----------------------------------------------------------------------------

known_runtime_tensor_names = [
    "input_ids",
    "smoke_input_ids",
    "test_input_ids",
    "synthetic_input_ids",
    "cuda_input_ids",
    "sample_input_ids",
]

removed_names = []

for name in known_runtime_tensor_names:
    if name not in globals():
        continue

    try:
        obj = globals()[name]

        if torch.is_tensor(obj) and obj.is_cuda:
            del globals()[name]
            removed_names.append(name)

    except Exception:
        continue


print("2. TARGETED PYTHON-NAMESPACE CLEANUP")
print("-" * 76)

if removed_names:
    print("  Removed CUDA tensor variables:")
    for name in removed_names:
        print(f"    - {name}")
else:
    print("  No known CUDA tensor variable names required removal.")

print()


# ----------------------------------------------------------------------------
# 4. Explicitly remove stale model objects from the failed attempt
# ----------------------------------------------------------------------------
#
# The diagnostic established that the model was CPU-resident. The previous
# failed Phase 1B attempt should not be retained into the corrected loading
# attempt because doing so would create ambiguity about which model instance
# is being audited.
#
# We therefore remove the old model/module references.
#
# This does NOT modify the checkpoint, architecture, or scientific state.
# ----------------------------------------------------------------------------

removed_model_objects = []

for name in model_names:
    if name in globals():
        try:
            obj = globals()[name]

            if isinstance(obj, torch.nn.Module):
                del globals()[name]
                removed_model_objects.append(name)

        except Exception:
            continue


print("3. FAILED-ATTEMPT MODEL OBJECT CLEANUP")
print("-" * 76)

if removed_model_objects:
    print("  Removed stale model/module references:")
    for name in removed_model_objects:
        print(f"    - {name}")
else:
    print("  No model/module references required removal.")

print()


# ----------------------------------------------------------------------------
# 5. Python garbage collection
# ----------------------------------------------------------------------------

collected_objects = gc.collect()

print("4. PYTHON GARBAGE COLLECTION")
print("-" * 76)
print(f"  Objects collected : {collected_objects}")
print()


# ----------------------------------------------------------------------------
# 6. Synchronize and release unused CUDA cache
# ----------------------------------------------------------------------------
#
# Emptying the caching allocator is a runtime operation. It does not alter
# model weights or scientific state.
# ----------------------------------------------------------------------------

if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


# ----------------------------------------------------------------------------
# 7. Re-enumerate Python-reachable CUDA tensors
# ----------------------------------------------------------------------------

post_cleanup_cuda_tensors = []

if torch.cuda.is_available():

    for idx, obj in enumerate(gc.get_objects()):
        try:
            if torch.is_tensor(obj) and obj.is_cuda:

                try:
                    nbytes = int(obj.numel() * obj.element_size())
                except Exception:
                    nbytes = None

                try:
                    storage_ptr = int(
                        obj.untyped_storage().data_ptr()
                    )
                except Exception:
                    storage_ptr = None

                post_cleanup_cuda_tensors.append({
                    "object_index": idx,
                    "dtype": str(obj.dtype),
                    "shape": list(obj.shape),
                    "device": str(obj.device),
                    "numel": int(obj.numel()),
                    "nbytes": nbytes,
                    "storage_data_ptr": storage_ptr,
                })

        except Exception:
            continue


# ----------------------------------------------------------------------------
# 8. Re-audit model-like CUDA residency
# ----------------------------------------------------------------------------

post_cleanup_model_state = {}

for name in model_names:
    if name not in globals():
        continue

    try:
        obj = globals()[name]

        if isinstance(obj, torch.nn.Module):

            cuda_parameters = sum(
                1 for p in obj.parameters() if p.is_cuda
            )

            cuda_buffers = sum(
                1 for b in obj.buffers() if b.is_cuda
            )

            post_cleanup_model_state[name] = {
                "python_type": type(obj).__name__,
                "cuda_parameters": cuda_parameters,
                "cuda_buffers": cuda_buffers,
            }

    except Exception as exc:
        post_cleanup_model_state[name] = {
            "inspection_error": repr(exc)
        }


# ----------------------------------------------------------------------------
# 9. Post-cleanup CUDA allocator state
# ----------------------------------------------------------------------------

post_allocator = None

if torch.cuda.is_available():

    post_allocator = {
        "memory_allocated_bytes":
            int(torch.cuda.memory_allocated()),
        "memory_reserved_bytes":
            int(torch.cuda.memory_reserved()),
        "max_memory_allocated_bytes":
            int(torch.cuda.max_memory_allocated()),
        "max_memory_reserved_bytes":
            int(torch.cuda.max_memory_reserved()),
    }


# ----------------------------------------------------------------------------
# 10. Firewall decision
# ----------------------------------------------------------------------------

post_cuda_parameter_count = sum(
    state.get("cuda_parameters", 0)
    for state in post_cleanup_model_state.values()
)

post_cuda_buffer_count = sum(
    state.get("cuda_buffers", 0)
    for state in post_cleanup_model_state.values()
)


if (
    len(post_cleanup_cuda_tensors) == 0
    and post_cuda_parameter_count == 0
    and post_cuda_buffer_count == 0
):
    firewall_status = "CLEAN_PRELOAD_FIREWALL_PASS"
else:
    firewall_status = "CLEAN_PRELOAD_FIREWALL_NOT_ESTABLISHED"


print("5. POST-CLEANUP PYTHON-REACHABLE CUDA TENSORS")
print("-" * 76)
print(
    f"  Count : {len(post_cleanup_cuda_tensors)}"
)

if post_cleanup_cuda_tensors:
    for i, meta in enumerate(post_cleanup_cuda_tensors):
        print()
        print(f"  Remaining tensor {i + 1}")
        print(f"    Device : {meta['device']}")
        print(f"    Dtype  : {meta['dtype']}")
        print(f"    Shape  : {meta['shape']}")
        print(f"    Bytes  : {meta['nbytes']}")
else:
    print("  None detected.")

print()

print("6. POST-CLEANUP MODEL CUDA RESIDENCY")
print("-" * 76)

if post_cleanup_model_state:
    for name, state in post_cleanup_model_state.items():
        print(
            f"  {name}: "
            f"{state.get('cuda_parameters', 'N/A')} CUDA parameters, "
            f"{state.get('cuda_buffers', 'N/A')} CUDA buffers"
        )
else:
    print("  No model/module objects remain in the namespace.")

print()

print("7. POST-CLEANUP CUDA ALLOCATOR")
print("-" * 76)

if post_allocator is not None:
    print(
        f"  Allocated : "
        f"{post_allocator['memory_allocated_bytes'] / 2**20:.3f} MiB"
    )
    print(
        f"  Reserved  : "
        f"{post_allocator['memory_reserved_bytes'] / 2**20:.3f} MiB"
    )
else:
    print("  CUDA unavailable.")

print()

print("8. FIREWALL DECISION")
print("-" * 76)
print(f"  Status : {firewall_status}")
print()


# ----------------------------------------------------------------------------
# 11. Governance classification
# ----------------------------------------------------------------------------

if firewall_status == "CLEAN_PRELOAD_FIREWALL_PASS":

    classification = "NUMERICAL_RUNTIME_CLEANUP_SUCCESS"

    interpretation = (
        "The stale CUDA tensors and failed-attempt model references have "
        "been removed. No Python-reachable CUDA tensors, CUDA-resident "
        "model parameters, or CUDA-resident model buffers remain. The "
        "runtime is now suitable for a fresh controlled Phase 1B model "
        "loading attempt."
    )

else:

    classification = "NUMERICAL_RUNTIME_CLEANUP_INCOMPLETE"

    interpretation = (
        "Residual CUDA state remains after controlled cleanup. The Phase "
        "1B model-loading attempt must not proceed until the remaining "
        "runtime state is characterized."
    )


print("9. SCIENTIFIC CLASSIFICATION")
print("-" * 76)
print(f"  Classification : {classification}")
print()
print(f"  {interpretation}")
print()


# ----------------------------------------------------------------------------
# 12. Persist artifact
# ----------------------------------------------------------------------------

artifact = {
    "experiment_id": EXPERIMENT_ID,
    "audit_id": AUDIT_ID,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "pre_cleanup_model_state": pre_cleanup_model_state,

    "cleanup": {
        "known_runtime_tensor_names_checked":
            known_runtime_tensor_names,
        "removed_runtime_tensor_names":
            removed_names,
        "removed_model_object_names":
            removed_model_objects,
        "python_gc_collected_objects":
            int(collected_objects),
        "cuda_empty_cache_called":
            bool(torch.cuda.is_available()),
    },

    "post_cleanup": {
        "python_reachable_cuda_tensors":
            post_cleanup_cuda_tensors,
        "model_state":
            post_cleanup_model_state,
        "allocator":
            post_allocator,
    },

    "firewall_status": firewall_status,
    "classification": classification,
    "interpretation": interpretation,

    "scientific_status": {
        "ctl_math_modified": False,
        "operational_sector_definitions_modified": False,
        "scientific_data_accessed": False,
        "model_loaded_by_this_cell": False,
        "inference_executed_by_this_cell": False,
        "transport_maps_fitted": False,
        "test_data_accessed": False,
        "classification_layer": "NUMERICAL_EXECUTION_RUNTIME",
    },

    "governance": {
        "math_authoritative": True,
        "runtime_constraint_must_not_redefine_math": True,
        "decoherence_if_runtime_constraint_changes_math": True,
    },
}

with open(ARTIFACT_PATH, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2, default=str)


print("10. ARTIFACT")
print("-" * 76)
print(f"  Saved : {ARTIFACT_PATH}")
print()

print("=" * 76)
print("PHASE 1B CONTROLLED RUNTIME CLEANUP COMPLETE")
print("=" * 76)

ETTR-CTL-LLAMA — PHASE 1B
CONTROLLED RUNTIME CLEANUP AND PRE-LOAD FIREWALL RESET
Experiment : ETTR-CTL-LLAMA-1
Audit      : LLAMA-PHASE1B-RUNTIME-CLEANUP-V1
Timestamp  : 2026-09-10T23:51:25.486127+00:00
Python     : 3.13.15
PyTorch    : 2.11.0+cu128

1. PRE-CLEANUP MODEL CUDA STATE
----------------------------------------------------------------------------
  model: 0 CUDA parameters, 0 CUDA buffers
  decoder_layers: 0 CUDA parameters, 0 CUDA buffers
  base_model: 0 CUDA parameters, 0 CUDA buffers
  audit_layer: 0 CUDA parameters, 0 CUDA buffers

2. TARGETED PYTHON-NAMESPACE CLEANUP
----------------------------------------------------------------------------
  Removed CUDA tensor variables:
    - input_ids

3. FAILED-ATTEMPT MODEL OBJECT CLEANUP
----------------------------------------------------------------------------
  Removed stale model/module references:
    - model
    - decoder_layers
    - base_model
    - audit_layer

4. PYTHON GARBAGE COLLECTION
----------------------------

In [19]:
# ============================================================================
# ETTR-CTL-LLAMA — PHASE 1B
# RESIDUAL CUDA TENSOR REFERENCE FORENSICS
# ============================================================================
# Audit:
#   LLAMA-PHASE1B-RESIDUAL-CUDA-REFERENCE-FORENSICS-V1
#
# Purpose:
#   Identify the Python reference(s) keeping the single remaining CUDA tensor
#   alive after controlled cleanup.
#
# Scientific status:
#   NUMERICAL-EXECUTION / RUNTIME DIAGNOSTIC ONLY
#
# This cell MUST NOT:
#   - load the Llama checkpoint
#   - run inference
#   - access scientific/test data
#   - fit transport maps
#   - modify CTL mathematics
#   - modify operational sector definitions
#   - alter model weights
# ============================================================================

import os
import gc
import json
import sys
import types
from datetime import datetime, timezone

import torch


# ----------------------------------------------------------------------------
# 0. Configuration
# ----------------------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
AUDIT_ID = "LLAMA-PHASE1B-RESIDUAL-CUDA-REFERENCE-FORENSICS-V1"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

ARTIFACT_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1b_residual_cuda_reference_forensics_v1.json"
)


# ----------------------------------------------------------------------------
# 1. Header
# ----------------------------------------------------------------------------

print("=" * 76)
print("ETTR-CTL-LLAMA — PHASE 1B")
print("RESIDUAL CUDA TENSOR REFERENCE FORENSICS")
print("=" * 76)
print(f"Experiment : {EXPERIMENT_ID}")
print(f"Audit      : {AUDIT_ID}")
print(f"Timestamp  : {datetime.now(timezone.utc).isoformat()}")
print(f"Python     : {sys.version.split()[0]}")
print(f"PyTorch    : {torch.__version__}")
print()


# ----------------------------------------------------------------------------
# 2. Enumerate current Python-reachable CUDA tensors
# ----------------------------------------------------------------------------

def describe_tensor(t):
    try:
        storage_ptr = int(t.untyped_storage().data_ptr())
    except Exception:
        storage_ptr = None

    return {
        "python_type": type(t).__name__,
        "device": str(t.device),
        "dtype": str(t.dtype),
        "shape": list(t.shape),
        "numel": int(t.numel()),
        "nbytes": int(t.numel() * t.element_size()),
        "requires_grad": bool(t.requires_grad),
        "is_leaf": bool(t.is_leaf),
        "storage_data_ptr": storage_ptr,
    }


cuda_tensor_objects = []

for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            cuda_tensor_objects.append(obj)
    except Exception:
        continue


print("1. CURRENT PYTHON-REACHABLE CUDA TENSORS")
print("-" * 76)
print(f"  Count : {len(cuda_tensor_objects)}")

for i, tensor in enumerate(cuda_tensor_objects):
    print(f"\n  Tensor {i + 1}")
    print(f"    {describe_tensor(tensor)}")

print()


# ----------------------------------------------------------------------------
# 3. Establish the forensic target
# ----------------------------------------------------------------------------

if len(cuda_tensor_objects) == 0:

    classification = "NO_RESIDUAL_CUDA_TENSOR"

    print("2. FORENSIC TARGET")
    print("-" * 76)
    print("  No Python-reachable CUDA tensor remains.")
    print()
    print("  No reference-chain investigation is required.")
    print()

else:

    # The previous cleanup established exactly one residual tensor.
    # We require exactly one here to avoid silently choosing among multiple
    # objects if the runtime changed unexpectedly.
    if len(cuda_tensor_objects) != 1:

        classification = "UNEXPECTED_MULTIPLE_RESIDUAL_CUDA_TENSORS"

        print("2. FORENSIC TARGET")
        print("-" * 76)
        print(
            "  WARNING: the runtime now contains more than one "
            "Python-reachable CUDA tensor."
        )
        print(
            "  Do not proceed to model loading. The runtime state "
            "requires renewed diagnosis."
        )
        print()

    else:

        target = cuda_tensor_objects[0]
        target_meta = describe_tensor(target)

        print("2. FORENSIC TARGET")
        print("-" * 76)
        print(f"  Target : {target_meta}")
        print()


        # --------------------------------------------------------------------
        # 4. Search global namespace for direct references
        # --------------------------------------------------------------------

        global_references = []

        for namespace_name, namespace in [
            ("globals", globals())
        ]:

            for name, value in list(namespace.items()):
                try:
                    if value is target:
                        global_references.append({
                            "namespace": namespace_name,
                            "name": name,
                            "value_type": type(value).__name__,
                        })
                except Exception:
                    continue


        print("3. DIRECT GLOBAL-NAMESPACE REFERENCES")
        print("-" * 76)

        if global_references:
            for ref in global_references:
                print(
                    f"  {ref['namespace']}[{ref['name']!r}] "
                    f"-> {ref['value_type']}"
                )
        else:
            print("  No direct global-namespace reference found.")

        print()


        # --------------------------------------------------------------------
        # 5. Search IPython user namespace
        # --------------------------------------------------------------------

        ipython_references = []

        try:
            ip = get_ipython()

            if ip is not None:

                user_ns = getattr(ip, "user_ns", {})

                for name, value in list(user_ns.items()):
                    try:
                        if value is target:
                            ipython_references.append({
                                "name": name,
                                "value_type": type(value).__name__,
                            })
                    except Exception:
                        continue

        except Exception:
            pass


        print("4. IPYTHON USER-NAMESPACE REFERENCES")
        print("-" * 76)

        if ipython_references:
            for ref in ipython_references:
                print(
                    f"  user_ns[{ref['name']!r}] "
                    f"-> {ref['value_type']}"
                )
        else:
            print("  No direct IPython user-namespace reference found.")

        print()


        # --------------------------------------------------------------------
        # 6. Search Python referrers
        # --------------------------------------------------------------------
        #
        # gc.get_referrers() returns actual Python objects that reference the
        # target. We classify only the immediate referrers.
        #
        # We deliberately do not recursively traverse arbitrary object graphs
        # because that can generate enormous and unstable notebook-runtime
        # state. Immediate referrers are sufficient to identify the usual
        # surviving reference mechanisms.
        # --------------------------------------------------------------------

        raw_referrers = gc.get_referrers(target)

        referrer_records = []

        for ref in raw_referrers:

            try:

                # ------------------------------------------------------------
                # Dictionary referrer
                # ------------------------------------------------------------

                if isinstance(ref, dict):

                    matching_keys = []

                    for key, value in list(ref.items()):
                        try:
                            if value is target:
                                matching_keys.append(repr(key))
                        except Exception:
                            continue

                    referrer_records.append({
                        "referrer_type": "dict",
                        "dict_size": len(ref),
                        "matching_keys": matching_keys,
                    })

                    continue


                # ------------------------------------------------------------
                # List referrer
                # ------------------------------------------------------------

                if isinstance(ref, list):

                    matching_indices = []

                    for idx, value in enumerate(ref):
                        try:
                            if value is target:
                                matching_indices.append(idx)
                        except Exception:
                            continue

                    referrer_records.append({
                        "referrer_type": "list",
                        "list_size": len(ref),
                        "matching_indices": matching_indices,
                    })

                    continue


                # ------------------------------------------------------------
                # Tuple referrer
                # ------------------------------------------------------------

                if isinstance(ref, tuple):

                    matching_indices = []

                    for idx, value in enumerate(ref):
                        try:
                            if value is target:
                                matching_indices.append(idx)
                        except Exception:
                            continue

                    referrer_records.append({
                        "referrer_type": "tuple",
                        "tuple_size": len(ref),
                        "matching_indices": matching_indices,
                    })

                    continue


                # ------------------------------------------------------------
                # Set referrer
                # ------------------------------------------------------------

                if isinstance(ref, set):

                    contains_target = False

                    try:
                        contains_target = target in ref
                    except Exception:
                        pass

                    referrer_records.append({
                        "referrer_type": "set",
                        "set_size": len(ref),
                        "contains_target": contains_target,
                    })

                    continue


                # ------------------------------------------------------------
                # Module / object referrer
                # ------------------------------------------------------------

                if isinstance(ref, types.ModuleType):

                    referrer_records.append({
                        "referrer_type": "module",
                        "module_name": getattr(
                            ref, "__name__", "<unknown>"
                        ),
                    })

                    continue


                # ------------------------------------------------------------
                # Generic object referrer
                # ------------------------------------------------------------

                referrer_records.append({
                    "referrer_type": type(ref).__name__,
                    "repr": repr(ref)[:500],
                })

            except Exception as exc:

                referrer_records.append({
                    "referrer_type": "inspection_error",
                    "error": repr(exc),
                })


        print("5. IMMEDIATE PYTHON REFERRERS")
        print("-" * 76)
        print(f"  Referrer count : {len(referrer_records)}")

        for i, record in enumerate(referrer_records):

            print(f"\n  Referrer {i + 1}")

            for key, value in record.items():
                print(f"    {key}: {value}")

        print()


        # --------------------------------------------------------------------
        # 7. Search notebook output history
        # --------------------------------------------------------------------
        #
        # IPython stores the last result in "_" and maintains output history
        # through Out. A tensor displayed or returned as the final expression
        # can therefore survive after its original variable is deleted.
        # --------------------------------------------------------------------

        output_history_hits = []

        try:

            ip = get_ipython()

            if ip is not None:

                # "_" / "__" / "___"
                for name in ["_", "__", "___"]:

                    try:
                        if ip.user_ns.get(name, None) is target:
                            output_history_hits.append({
                                "source": "user_ns",
                                "name": name,
                            })
                    except Exception:
                        pass

                # Out dictionary
                try:

                    out_history = ip.user_ns.get("Out", {})

                    if isinstance(out_history, dict):

                        for key, value in list(out_history.items()):

                            try:
                                if value is target:
                                    output_history_hits.append({
                                        "source": "Out",
                                        "name": repr(key),
                                    })
                            except Exception:
                                continue

                except Exception:
                    pass

        except Exception:
            pass


        print("6. IPYTHON OUTPUT-HISTORY REFERENCES")
        print("-" * 76)

        if output_history_hits:

            for hit in output_history_hits:
                print(
                    f"  {hit['source']}[{hit['name']}] "
                    f"references the target."
                )

        else:

            print(
                "  No direct IPython output-history reference detected."
            )

        print()


        # --------------------------------------------------------------------
        # 8. Final forensic classification
        # --------------------------------------------------------------------

        has_global_ref = len(global_references) > 0
        has_ipython_ref = len(ipython_references) > 0
        has_output_ref = len(output_history_hits) > 0

        if has_global_ref:
            classification = (
                "RESIDUAL_CUDA_TENSOR_HAS_GLOBAL_REFERENCE"
            )

        elif has_ipython_ref:
            classification = (
                "RESIDUAL_CUDA_TENSOR_HAS_IPYTHON_NAMESPACE_REFERENCE"
            )

        elif has_output_ref:
            classification = (
                "RESIDUAL_CUDA_TENSOR_RETAINED_BY_OUTPUT_HISTORY"
            )

        elif len(referrer_records) > 0:
            classification = (
                "RESIDUAL_CUDA_TENSOR_HAS_NONTRIVIAL_PYTHON_REFERRER"
            )

        else:
            classification = (
                "RESIDUAL_CUDA_TENSOR_REFERENCE_NOT_IDENTIFIED"
            )


# ----------------------------------------------------------------------------
# 9. Scientific/runtime interpretation
# ----------------------------------------------------------------------------

if classification == "NO_RESIDUAL_CUDA_TENSOR":

    interpretation = (
        "No Python-reachable CUDA tensor remains. The runtime residency "
        "condition is satisfied."
    )

elif classification == "UNEXPECTED_MULTIPLE_RESIDUAL_CUDA_TENSORS":

    interpretation = (
        "The residual runtime state changed unexpectedly and contains "
        "multiple CUDA tensors. Model loading must remain blocked pending "
        "further diagnosis."
    )

elif classification in (
    "RESIDUAL_CUDA_TENSOR_HAS_GLOBAL_REFERENCE",
    "RESIDUAL_CUDA_TENSOR_HAS_IPYTHON_NAMESPACE_REFERENCE",
    "RESIDUAL_CUDA_TENSOR_RETAINED_BY_OUTPUT_HISTORY",
    "RESIDUAL_CUDA_TENSOR_HAS_NONTRIVIAL_PYTHON_REFERRER",
):

    interpretation = (
        "The residual CUDA tensor has an identifiable Python-level "
        "reference. It is runtime residue from the failed execution path, "
        "not model parameter residency. The reference can be removed "
        "explicitly without altering the mathematical or experimental "
        "definition."
    )

else:

    interpretation = (
        "The CUDA tensor remains Python-reachable but its immediate "
        "reference chain was not identified. Model loading must remain "
        "blocked until the runtime residue is resolved."
    )


print("7. FORENSIC CLASSIFICATION")
print("-" * 76)
print(f"  Classification : {classification}")
print()
print(f"  {interpretation}")
print()


# ----------------------------------------------------------------------------
# 10. Persist artifact
# ----------------------------------------------------------------------------

artifact = {
    "experiment_id": EXPERIMENT_ID,
    "audit_id": AUDIT_ID,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "cuda_tensor_count": len(cuda_tensor_objects),

    "target_tensor": (
        describe_tensor(cuda_tensor_objects[0])
        if len(cuda_tensor_objects) == 1
        else None
    ),

    "global_references": (
        global_references
        if len(cuda_tensor_objects) == 1
        else []
    ),

    "ipython_references": (
        ipython_references
        if len(cuda_tensor_objects) == 1
        else []
    ),

    "python_referrers": (
        referrer_records
        if len(cuda_tensor_objects) == 1
        else []
    ),

    "output_history_references": (
        output_history_hits
        if len(cuda_tensor_objects) == 1
        else []
    ),

    "classification": classification,
    "interpretation": interpretation,

    "scientific_status": {
        "ctl_math_modified": False,
        "operational_sector_definitions_modified": False,
        "scientific_data_accessed": False,
        "model_loaded": False,
        "inference_executed": False,
        "transport_maps_fitted": False,
        "test_data_accessed": False,
        "classification_layer": "NUMERICAL_EXECUTION_RUNTIME",
    },

    "governance": {
        "math_authoritative": True,
        "runtime_constraint_must_not_redefine_math": True,
        "decoherence_if_runtime_constraint_changes_math": True,
    },
}

with open(ARTIFACT_PATH, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2, default=str)


print("8. ARTIFACT")
print("-" * 76)
print(f"  Saved : {ARTIFACT_PATH}")
print()

print("=" * 76)
print("PHASE 1B RESIDUAL CUDA REFERENCE FORENSICS COMPLETE")
print("=" * 76)

ETTR-CTL-LLAMA — PHASE 1B
RESIDUAL CUDA TENSOR REFERENCE FORENSICS
Experiment : ETTR-CTL-LLAMA-1
Audit      : LLAMA-PHASE1B-RESIDUAL-CUDA-REFERENCE-FORENSICS-V1
Timestamp  : 2026-09-10T23:52:46.662921+00:00
Python     : 3.13.15
PyTorch    : 2.11.0+cu128

1. CURRENT PYTHON-REACHABLE CUDA TENSORS
----------------------------------------------------------------------------
  Count : 1

  Tensor 1
    {'python_type': 'Tensor', 'device': 'cuda:0', 'dtype': 'torch.int64', 'shape': [1, 14], 'numel': 14, 'nbytes': 112, 'requires_grad': False, 'is_leaf': True, 'storage_data_ptr': 133181317579264}

2. FORENSIC TARGET
----------------------------------------------------------------------------
  Target : {'python_type': 'Tensor', 'device': 'cuda:0', 'dtype': 'torch.int64', 'shape': [1, 14], 'numel': 14, 'nbytes': 112, 'requires_grad': False, 'is_leaf': True, 'storage_data_ptr': 133181317579264}

3. DIRECT GLOBAL-NAMESPACE REFERENCES
----------------------------------------------------------------

In [20]:
# ============================================================================
# ETTR-CTL-LLAMA — PHASE 1B
# CONTROLLED REMOVAL OF IDENTIFIED RESIDUAL CUDA REFERENCES
# ============================================================================
# Audit:
#   LLAMA-PHASE1B-RESIDUAL-CUDA-CLEANUP-V2
#
# Purpose:
#   Remove the specifically identified residual CUDA tensor references
#   discovered by the preceding forensic audit.
#
# Scientific status:
#   NUMERICAL-EXECUTION / RUNTIME ONLY
#
# This cell does NOT:
#   - load Llama
#   - run inference
#   - access scientific/test data
#   - fit transport maps
#   - modify CTL mathematics
#   - modify operational sector definitions
#   - modify model weights
#
# Identified residual tensor:
#   torch.int64, cuda:0, shape [1,14], 112 bytes
#
# Identified global references:
#   attention_mask
#   tensor
#   target
# ============================================================================

import os
import gc
import json
import sys
from datetime import datetime, timezone

import torch


# ----------------------------------------------------------------------------
# 0. Configuration
# ----------------------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
AUDIT_ID = "LLAMA-PHASE1B-RESIDUAL-CUDA-CLEANUP-V2"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

ARTIFACT_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1b_residual_cuda_cleanup_v2.json"
)


# ----------------------------------------------------------------------------
# 1. Header
# ----------------------------------------------------------------------------

print("=" * 76)
print("ETTR-CTL-LLAMA — PHASE 1B")
print("CONTROLLED REMOVAL OF IDENTIFIED RESIDUAL CUDA REFERENCES")
print("=" * 76)
print(f"Experiment : {EXPERIMENT_ID}")
print(f"Audit      : {AUDIT_ID}")
print(f"Timestamp  : {datetime.now(timezone.utc).isoformat()}")
print(f"Python     : {sys.version.split()[0]}")
print(f"PyTorch    : {torch.__version__}")
print()


# ----------------------------------------------------------------------------
# 2. Capture pre-cleanup CUDA tensor state
# ----------------------------------------------------------------------------

def get_cuda_tensor_metadata(t):
    try:
        storage_ptr = int(t.untyped_storage().data_ptr())
    except Exception:
        storage_ptr = None

    return {
        "dtype": str(t.dtype),
        "device": str(t.device),
        "shape": list(t.shape),
        "numel": int(t.numel()),
        "nbytes": int(t.numel() * t.element_size()),
        "storage_data_ptr": storage_ptr,
    }


pre_cleanup_tensors = []

for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            pre_cleanup_tensors.append(
                get_cuda_tensor_metadata(obj)
            )
    except Exception:
        continue


print("1. PRE-CLEANUP CUDA TENSOR STATE")
print("-" * 76)
print(f"  Python-reachable CUDA tensors : {len(pre_cleanup_tensors)}")

for i, meta in enumerate(pre_cleanup_tensors):
    print(f"  Tensor {i + 1}: {meta}")

print()


# ----------------------------------------------------------------------------
# 3. Remove the explicitly identified global references
# ----------------------------------------------------------------------------
#
# These names were directly established by the forensic audit.
#
# We do NOT delete arbitrary globals. Only the identified stale references
# are removed.
# ----------------------------------------------------------------------------

identified_names = [
    "attention_mask",
    "tensor",
    "target",
]

removed_names = []

for name in identified_names:

    if name not in globals():
        continue

    try:
        value = globals()[name]

        if torch.is_tensor(value) and value.is_cuda:
            del globals()[name]
            removed_names.append(name)

    except Exception:
        continue


print("2. EXPLICIT RESIDUAL-REFERENCE REMOVAL")
print("-" * 76)

if removed_names:
    for name in removed_names:
        print(f"  Removed : globals[{name!r}]")
else:
    print("  No identified CUDA tensor globals required removal.")

print()


# ----------------------------------------------------------------------------
# 4. Remove possible diagnostic-loop aliases
# ----------------------------------------------------------------------------
#
# The previous forensic cell itself used names such as:
#   tensor
#   target
#   obj
#
# "tensor" and "target" were explicitly observed to reference the residual
# tensor. We have already removed them above.
#
# We intentionally do not indiscriminately clear the entire namespace.
# ----------------------------------------------------------------------------


# ----------------------------------------------------------------------------
# 5. Garbage collection
# ----------------------------------------------------------------------------

collected = gc.collect()

print("3. PYTHON GARBAGE COLLECTION")
print("-" * 76)
print(f"  Objects collected : {collected}")
print()


# ----------------------------------------------------------------------------
# 6. Release unused CUDA cache
# ----------------------------------------------------------------------------

if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


# ----------------------------------------------------------------------------
# 7. Re-enumerate Python-reachable CUDA tensors
# ----------------------------------------------------------------------------

post_cleanup_tensors = []

for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            post_cleanup_tensors.append(
                get_cuda_tensor_metadata(obj)
            )
    except Exception:
        continue


print("4. POST-CLEANUP PYTHON-REACHABLE CUDA TENSORS")
print("-" * 76)
print(f"  Count : {len(post_cleanup_tensors)}")

for i, meta in enumerate(post_cleanup_tensors):
    print(f"  Tensor {i + 1}: {meta}")

print()


# ----------------------------------------------------------------------------
# 8. Verify no model objects are present
# ----------------------------------------------------------------------------

model_names = [
    "model",
    "decoder_layers",
    "base_model",
    "audit_layer",
]

remaining_model_objects = []

for name in model_names:
    if name in globals():

        try:
            if isinstance(globals()[name], torch.nn.Module):
                remaining_model_objects.append({
                    "name": name,
                    "type": type(globals()[name]).__name__,
                })

        except Exception:
            continue


print("5. MODEL OBJECT STATE")
print("-" * 76)

if remaining_model_objects:
    for item in remaining_model_objects:
        print(f"  WARNING: {item}")
else:
    print("  No model/module objects remain in the namespace.")

print()


# ----------------------------------------------------------------------------
# 9. Verify CUDA-resident model parameters/buffers
# ----------------------------------------------------------------------------

remaining_cuda_parameters = 0
remaining_cuda_buffers = 0

for item in remaining_model_objects:

    name = item["name"]

    try:
        module = globals()[name]

        remaining_cuda_parameters += sum(
            1 for p in module.parameters() if p.is_cuda
        )

        remaining_cuda_buffers += sum(
            1 for b in module.buffers() if b.is_cuda
        )

    except Exception:
        continue


print("6. MODEL CUDA RESIDENCY")
print("-" * 76)
print(f"  CUDA parameters : {remaining_cuda_parameters}")
print(f"  CUDA buffers    : {remaining_cuda_buffers}")
print()


# ----------------------------------------------------------------------------
# 10. CUDA allocator state
# ----------------------------------------------------------------------------

allocator_state = None

if torch.cuda.is_available():

    allocator_state = {
        "memory_allocated_bytes":
            int(torch.cuda.memory_allocated()),
        "memory_reserved_bytes":
            int(torch.cuda.memory_reserved()),
        "max_memory_allocated_bytes":
            int(torch.cuda.max_memory_allocated()),
        "max_memory_reserved_bytes":
            int(torch.cuda.max_memory_reserved()),
    }

    print("7. CUDA ALLOCATOR STATE")
    print("-" * 76)
    print(
        f"  Allocated : "
        f"{allocator_state['memory_allocated_bytes'] / 2**20:.3f} MiB"
    )
    print(
        f"  Reserved  : "
        f"{allocator_state['memory_reserved_bytes'] / 2**20:.3f} MiB"
    )
    print()


# ----------------------------------------------------------------------------
# 11. Final firewall decision
# ----------------------------------------------------------------------------

firewall_pass = (
    len(post_cleanup_tensors) == 0
    and remaining_cuda_parameters == 0
    and remaining_cuda_buffers == 0
    and len(remaining_model_objects) == 0
)

if firewall_pass:

    status = "CLEAN_PRELOAD_FIREWALL_PASS"

    classification = "NUMERICAL_RUNTIME_CLEANUP_SUCCESS"

    interpretation = (
        "The specifically identified residual CUDA references have been "
        "removed. No Python-reachable CUDA tensors remain, no model/module "
        "objects remain, and no CUDA-resident model parameters or buffers "
        "remain. The runtime is clean for a fresh controlled Phase 1B "
        "model-loading attempt."
    )

else:

    status = "CLEAN_PRELOAD_FIREWALL_NOT_ESTABLISHED"

    classification = "NUMERICAL_RUNTIME_CLEANUP_INCOMPLETE"

    interpretation = (
        "Residual runtime state remains after removal of the identified "
        "references. Model loading must remain blocked until the residual "
        "state is characterized."
    )


print("8. FINAL PRE-LOAD FIREWALL")
print("-" * 76)
print(f"  Status         : {status}")
print(f"  Classification : {classification}")
print()
print(f"  {interpretation}")
print()


# ----------------------------------------------------------------------------
# 12. Persist artifact
# ----------------------------------------------------------------------------

artifact = {
    "experiment_id": EXPERIMENT_ID,
    "audit_id": AUDIT_ID,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "pre_cleanup_cuda_tensors": pre_cleanup_tensors,

    "identified_residual_names": identified_names,
    "removed_names": removed_names,

    "post_cleanup_cuda_tensors": post_cleanup_tensors,

    "remaining_model_objects": remaining_model_objects,

    "remaining_cuda_parameters":
        remaining_cuda_parameters,

    "remaining_cuda_buffers":
        remaining_cuda_buffers,

    "allocator_state": allocator_state,

    "firewall_status": status,
    "classification": classification,
    "interpretation": interpretation,

    "scientific_status": {
        "ctl_math_modified": False,
        "operational_sector_definitions_modified": False,
        "scientific_data_accessed": False,
        "model_loaded": False,
        "inference_executed": False,
        "transport_maps_fitted": False,
        "test_data_accessed": False,
        "model_weights_modified": False,
        "classification_layer": "NUMERICAL_EXECUTION_RUNTIME",
    },

    "governance": {
        "math_authoritative": True,
        "runtime_constraint_must_not_redefine_math": True,
        "decoherence_if_runtime_constraint_changes_math": True,
    },
}

with open(ARTIFACT_PATH, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2, default=str)


print("9. ARTIFACT")
print("-" * 76)
print(f"  Saved : {ARTIFACT_PATH}")
print()

print("=" * 76)
print("PHASE 1B RESIDUAL CUDA CLEANUP COMPLETE")
print("=" * 76)

ETTR-CTL-LLAMA — PHASE 1B
CONTROLLED REMOVAL OF IDENTIFIED RESIDUAL CUDA REFERENCES
Experiment : ETTR-CTL-LLAMA-1
Audit      : LLAMA-PHASE1B-RESIDUAL-CUDA-CLEANUP-V2
Timestamp  : 2026-09-10T23:54:16.476915+00:00
Python     : 3.13.15
PyTorch    : 2.11.0+cu128

1. PRE-CLEANUP CUDA TENSOR STATE
----------------------------------------------------------------------------
  Python-reachable CUDA tensors : 1
  Tensor 1: {'dtype': 'torch.int64', 'device': 'cuda:0', 'shape': [1, 14], 'numel': 14, 'nbytes': 112, 'storage_data_ptr': 133181317579264}

2. EXPLICIT RESIDUAL-REFERENCE REMOVAL
----------------------------------------------------------------------------
  Removed : globals['attention_mask']
  Removed : globals['tensor']
  Removed : globals['target']

3. PYTHON GARBAGE COLLECTION
----------------------------------------------------------------------------
  Objects collected : 0

4. POST-CLEANUP PYTHON-REACHABLE CUDA TENSORS
-------------------------------------------------------------

In [21]:
# ============================================================================
# ETTR-CTL-LLAMA — PHASE 1B
# FINAL FORENSIC-CONTAINER CLEANUP AND PRE-LOAD FIREWALL VERIFICATION
# ============================================================================
# Audit:
#   LLAMA-PHASE1B-RESIDUAL-CUDA-CONTAINER-CLEANUP-V3
#
# Purpose:
#   Remove the diagnostic container that was retaining the residual CUDA
#   tensor after the explicitly named tensor references had been deleted.
#
# Scientific status:
#   NUMERICAL-EXECUTION / RUNTIME ONLY
#
# This cell does NOT:
#   - load the Llama checkpoint
#   - execute inference
#   - access scientific/test data
#   - fit transport maps
#   - modify CTL mathematics
#   - modify operational sector definitions
#   - modify model weights
#
# IMPORTANT:
#   The preceding forensic audit identified the residual tensor as being
#   retained through Python references. In particular, the forensic cell
#   itself created `cuda_tensor_objects`, a global list containing the target.
#   That container must be removed before the firewall can pass.
# ============================================================================

import os
import gc
import json
import sys
from datetime import datetime, timezone

import torch


# ----------------------------------------------------------------------------
# 0. Configuration
# ----------------------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
AUDIT_ID = "LLAMA-PHASE1B-RESIDUAL-CUDA-CONTAINER-CLEANUP-V3"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

ARTIFACT_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1b_residual_cuda_container_cleanup_v3.json"
)


# ----------------------------------------------------------------------------
# 1. Header
# ----------------------------------------------------------------------------

print("=" * 76)
print("ETTR-CTL-LLAMA — PHASE 1B")
print("FINAL FORENSIC-CONTAINER CLEANUP AND PRE-LOAD FIREWALL VERIFICATION")
print("=" * 76)
print(f"Experiment : {EXPERIMENT_ID}")
print(f"Audit      : {AUDIT_ID}")
print(f"Timestamp  : {datetime.now(timezone.utc).isoformat()}")
print(f"Python     : {sys.version.split()[0]}")
print(f"PyTorch    : {torch.__version__}")
print()


# ----------------------------------------------------------------------------
# 2. Enumerate the residual state BEFORE cleanup
# ----------------------------------------------------------------------------

def cuda_tensor_count():
    count = 0
    metadata = []

    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) and obj.is_cuda:
                count += 1

                try:
                    ptr = int(obj.untyped_storage().data_ptr())
                except Exception:
                    ptr = None

                metadata.append({
                    "dtype": str(obj.dtype),
                    "device": str(obj.device),
                    "shape": list(obj.shape),
                    "numel": int(obj.numel()),
                    "nbytes": int(
                        obj.numel() * obj.element_size()
                    ),
                    "storage_data_ptr": ptr,
                })

        except Exception:
            continue

    return count, metadata


pre_count, pre_metadata = cuda_tensor_count()

print("1. PRE-CLEANUP RUNTIME STATE")
print("-" * 76)
print(f"  Python-reachable CUDA tensors : {pre_count}")

for i, meta in enumerate(pre_metadata):
    print(f"  Tensor {i + 1}: {meta}")

print()


# ----------------------------------------------------------------------------
# 3. Remove diagnostic containers known to retain tensor references
# ----------------------------------------------------------------------------
#
# The preceding forensic cell introduced several global objects.
#
# `cuda_tensor_objects` is the principal retaining container:
#
#     cuda_tensor_objects -> [target CUDA tensor]
#
# Other forensic containers are removed as well because they are no longer
# scientifically required and can preserve notebook object graphs.
# ----------------------------------------------------------------------------

forensic_container_names = [
    "cuda_tensor_objects",
    "raw_referrers",
    "ref",
    "referrer_records",
    "global_references",
    "ipython_references",
    "output_history_hits",
]

removed_forensic_containers = []

for name in forensic_container_names:

    if name not in globals():
        continue

    try:
        del globals()[name]
        removed_forensic_containers.append(name)
    except Exception:
        pass


# ----------------------------------------------------------------------------
# 4. Remove remaining forensic target aliases if present
# ----------------------------------------------------------------------------

forensic_alias_names = [
    "target",
    "tensor",
    "attention_mask",
    "obj",
]

removed_aliases = []

for name in forensic_alias_names:

    if name not in globals():
        continue

    try:
        value = globals()[name]

        # We remove these names because they were established by the
        # preceding forensic procedure as possible tensor aliases.
        del globals()[name]
        removed_aliases.append(name)

    except Exception:
        pass


print("2. FORENSIC REFERENCE CLEANUP")
print("-" * 76)

if removed_forensic_containers:
    print("  Removed forensic containers:")
    for name in removed_forensic_containers:
        print(f"    - {name}")
else:
    print("  No forensic containers found.")

if removed_aliases:
    print("  Removed forensic aliases:")
    for name in removed_aliases:
        print(f"    - {name}")
else:
    print("  No remaining forensic aliases found.")

print()


# ----------------------------------------------------------------------------
# 5. Garbage collection
# ----------------------------------------------------------------------------

gc_collected = gc.collect()

print("3. PYTHON GARBAGE COLLECTION")
print("-" * 76)
print(f"  Objects collected : {gc_collected}")
print()


# ----------------------------------------------------------------------------
# 6. Synchronize and release unused CUDA cache
# ----------------------------------------------------------------------------

if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


# ----------------------------------------------------------------------------
# 7. Final Python-reachable CUDA tensor enumeration
# ----------------------------------------------------------------------------

post_count, post_metadata = cuda_tensor_count()

print("4. POST-CLEANUP PYTHON-REACHABLE CUDA TENSORS")
print("-" * 76)
print(f"  Count : {post_count}")

for i, meta in enumerate(post_metadata):
    print(f"  Tensor {i + 1}: {meta}")

if post_count == 0:
    print("  No Python-reachable CUDA tensors remain.")

print()


# ----------------------------------------------------------------------------
# 8. Verify model/module absence
# ----------------------------------------------------------------------------

model_names = [
    "model",
    "decoder_layers",
    "base_model",
    "audit_layer",
]

remaining_model_objects = []

for name in model_names:

    if name not in globals():
        continue

    try:
        if isinstance(globals()[name], torch.nn.Module):
            remaining_model_objects.append({
                "name": name,
                "type": type(globals()[name]).__name__,
            })
    except Exception:
        continue


cuda_parameters = 0
cuda_buffers = 0

for item in remaining_model_objects:

    try:
        module = globals()[item["name"]]

        cuda_parameters += sum(
            1 for p in module.parameters()
            if p.is_cuda
        )

        cuda_buffers += sum(
            1 for b in module.buffers()
            if b.is_cuda
        )

    except Exception:
        continue


print("5. MODEL / MODULE RESIDENCY")
print("-" * 76)

if remaining_model_objects:
    for item in remaining_model_objects:
        print(
            f"  {item['name']} : {item['type']}"
        )
else:
    print("  No model/module objects remain.")

print(f"  CUDA parameters : {cuda_parameters}")
print(f"  CUDA buffers    : {cuda_buffers}")
print()


# ----------------------------------------------------------------------------
# 9. CUDA allocator state
# ----------------------------------------------------------------------------

allocator_state = None

if torch.cuda.is_available():

    allocator_state = {
        "memory_allocated_bytes":
            int(torch.cuda.memory_allocated()),
        "memory_reserved_bytes":
            int(torch.cuda.memory_reserved()),
        "max_memory_allocated_bytes":
            int(torch.cuda.max_memory_allocated()),
        "max_memory_reserved_bytes":
            int(torch.cuda.max_memory_reserved()),
    }

    print("6. CUDA ALLOCATOR STATE")
    print("-" * 76)
    print(
        f"  Allocated : "
        f"{allocator_state['memory_allocated_bytes'] / 2**20:.3f} MiB"
    )
    print(
        f"  Reserved  : "
        f"{allocator_state['memory_reserved_bytes'] / 2**20:.3f} MiB"
    )
    print()


# ----------------------------------------------------------------------------
# 10. Final firewall
# ----------------------------------------------------------------------------

firewall_pass = (
    post_count == 0
    and len(remaining_model_objects) == 0
    and cuda_parameters == 0
    and cuda_buffers == 0
)


if firewall_pass:

    firewall_status = "CLEAN_PRELOAD_FIREWALL_PASS"

    classification = "NUMERICAL_RUNTIME_CLEANUP_SUCCESS"

    interpretation = (
        "The residual CUDA tensor was retained by the preceding forensic "
        "diagnostic's Python containers. Those containers have now been "
        "removed. No Python-reachable CUDA tensors remain, no model/module "
        "objects remain, and no CUDA-resident model parameters or buffers "
        "remain. The runtime is clean for a fresh controlled Phase 1B "
        "model-loading attempt."
    )

else:

    firewall_status = "CLEAN_PRELOAD_FIREWALL_NOT_ESTABLISHED"

    classification = "NUMERICAL_RUNTIME_CLEANUP_INCOMPLETE"

    interpretation = (
        "Residual CUDA or model-related runtime state remains after removal "
        "of the identified forensic containers. Model loading must remain "
        "blocked until that state is characterized."
    )


print("7. FINAL PRE-LOAD FIREWALL")
print("-" * 76)
print(f"  Status         : {firewall_status}")
print(f"  Classification : {classification}")
print()
print(f"  {interpretation}")
print()


# ----------------------------------------------------------------------------
# 11. Persist artifact
# ----------------------------------------------------------------------------

artifact = {
    "experiment_id": EXPERIMENT_ID,
    "audit_id": AUDIT_ID,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "pre_cleanup": {
        "cuda_tensor_count": pre_count,
        "cuda_tensor_metadata": pre_metadata,
    },

    "cleanup": {
        "removed_forensic_containers":
            removed_forensic_containers,
        "removed_forensic_aliases":
            removed_aliases,
        "gc_collected_objects":
            int(gc_collected),
        "cuda_empty_cache_called":
            bool(torch.cuda.is_available()),
    },

    "post_cleanup": {
        "cuda_tensor_count": post_count,
        "cuda_tensor_metadata": post_metadata,
        "remaining_model_objects":
            remaining_model_objects,
        "cuda_parameters":
            cuda_parameters,
        "cuda_buffers":
            cuda_buffers,
        "allocator":
            allocator_state,
    },

    "firewall_status": firewall_status,
    "classification": classification,
    "interpretation": interpretation,

    "scientific_status": {
        "ctl_math_modified": False,
        "operational_sector_definitions_modified": False,
        "scientific_data_accessed": False,
        "model_loaded": False,
        "inference_executed": False,
        "transport_maps_fitted": False,
        "test_data_accessed": False,
        "model_weights_modified": False,
        "classification_layer": "NUMERICAL_EXECUTION_RUNTIME",
    },

    "governance": {
        "math_authoritative": True,
        "runtime_constraint_must_not_redefine_math": True,
        "decoherence_if_runtime_constraint_changes_math": True,
    },
}

with open(ARTIFACT_PATH, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2, default=str)


print("8. ARTIFACT")
print("-" * 76)
print(f"  Saved : {ARTIFACT_PATH}")
print()

print("=" * 76)
print("PHASE 1B FINAL RUNTIME CLEANUP COMPLETE")
print("=" * 76)

ETTR-CTL-LLAMA — PHASE 1B
FINAL FORENSIC-CONTAINER CLEANUP AND PRE-LOAD FIREWALL VERIFICATION
Experiment : ETTR-CTL-LLAMA-1
Audit      : LLAMA-PHASE1B-RESIDUAL-CUDA-CONTAINER-CLEANUP-V3
Timestamp  : 2026-09-10T23:56:03.614302+00:00
Python     : 3.13.15
PyTorch    : 2.11.0+cu128

1. PRE-CLEANUP RUNTIME STATE
----------------------------------------------------------------------------
  Python-reachable CUDA tensors : 1
  Tensor 1: {'dtype': 'torch.int64', 'device': 'cuda:0', 'shape': [1, 14], 'numel': 14, 'nbytes': 112, 'storage_data_ptr': 133181317579264}

2. FORENSIC REFERENCE CLEANUP
----------------------------------------------------------------------------
  Removed forensic containers:
    - cuda_tensor_objects
    - raw_referrers
    - ref
    - referrer_records
    - global_references
    - ipython_references
    - output_history_hits
  Removed forensic aliases:
    - obj

3. PYTHON GARBAGE COLLECTION
----------------------------------------------------------------------------


In [22]:
# ============================================================================
# ETTR-CTL-LLAMA — PHASE 1B
# MODEL LOADING / CUDA RESIDENCY / RUNTIME HOOK AUDIT
# ============================================================================
# Experiment:
#   ETTR-CTL-LLAMA-1
#
# Audit:
#   LLAMA-PHASE1B-MODEL-RUNTIME-HOOK-AUDIT-V3
#
# Model:
#   meta-llama/Llama-3.2-3B
#
# Scientific role:
#   Architecture/runtime verification only.
#
# This cell does NOT:
#   - access the scientific dataset
#   - fit transport maps
#   - evaluate ETTR
#   - evaluate CTL coherence
#   - perform causal interventions
#   - alter CTL mathematics
#   - alter sector definitions
#
# Operational sectors:
#   S1 = self-attention output before residual addition
#   S2 = decoder-layer input / residual state entering audited layer
#   S3 = MLP output before residual addition
#
# Device policy:
#   Exact checkpoint
#   BF16
#   Full model residency on CUDA
#   No quantization
#   No fallback to FP16
#
# IMPORTANT:
#   The pre-load firewall checks scientific/model residency, not whether
#   the CUDA caching allocator itself reports exactly zero bytes.
# ============================================================================

import os
import gc
import json
import hashlib
import inspect
import sys
import traceback
from datetime import datetime, timezone

import torch
import transformers

from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM


# ----------------------------------------------------------------------------
# 0. Configuration
# ----------------------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
AUDIT_ID = "LLAMA-PHASE1B-MODEL-RUNTIME-HOOK-AUDIT-V3"

MODEL_ID = "meta-llama/Llama-3.2-3B"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

ARTIFACT_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1b_model_runtime_hook_audit_v3.json"
)

DEVICE = torch.device("cuda:0")


# ----------------------------------------------------------------------------
# 1. Header
# ----------------------------------------------------------------------------

print("=" * 76)
print("ETTR-CTL-LLAMA — PHASE 1B")
print("MODEL LOADING / CUDA RESIDENCY / RUNTIME HOOK AUDIT")
print("=" * 76)
print(f"Experiment : {EXPERIMENT_ID}")
print(f"Audit      : {AUDIT_ID}")
print(f"Model ID   : {MODEL_ID}")
print(f"Timestamp  : {datetime.now(timezone.utc).isoformat()}")
print()

assert torch.cuda.is_available(), (
    "CUDA is unavailable; Phase 1B requires CUDA execution."
)

print(f"PyTorch    : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"CUDA       : {torch.version.cuda}")
print(f"GPU        : {torch.cuda.get_device_name(0)}")
print(
    f"Capability : "
    f"{torch.cuda.get_device_capability(0)[0]}."
    f"{torch.cuda.get_device_capability(0)[1]}"
)
print()


# ----------------------------------------------------------------------------
# 2. Phase 1A dependency verification
# ----------------------------------------------------------------------------

PHASE1A_ARTIFACT = os.path.join(
    RESULTS_DIR,
    "llama_phase1a_architecture_implementation_audit.json"
)

assert os.path.exists(PHASE1A_ARTIFACT), (
    f"Required Phase 1A artifact not found: {PHASE1A_ARTIFACT}"
)

with open(PHASE1A_ARTIFACT, "r", encoding="utf-8") as f:
    phase1a = json.load(f)


phase1a_status = (
    phase1a.get("status")
    or phase1a.get("audit_status")
    or phase1a.get("classification")
)


print("1. PHASE 1A DEPENDENCY AUDIT")
print("-" * 76)
print(f"  Phase 1A reported status : {phase1a_status}")

# The Phase 1A cell established PASS in its persisted artifact. We accept
# only an explicit PASS-like state here; otherwise model loading is blocked.

phase1a_text = json.dumps(phase1a).upper()

phase1a_pass = (
    "ARCHITECTURE_IMPLEMENTATION_AUDIT_PASS" in phase1a_text
)

assert phase1a_pass, (
    "Phase 1A architecture/implementation audit does not contain an "
    "explicit PASS state. Model loading is blocked."
)

print("  Dependency check : PASS")
print()


# ----------------------------------------------------------------------------
# 3. PRE-LOAD SCIENTIFIC/MODEL CUDA FIREWALL
# ----------------------------------------------------------------------------
#
# We intentionally do NOT require:
#
#     torch.cuda.memory_allocated() == 0
#
# because CUDA runtime/allocator bookkeeping can retain a small amount of
# memory independently of model/scientific tensors.
#
# The actual invariant is:
#
#   (1) no Python-reachable CUDA tensors
#   (2) no model/module objects
#   (3) no CUDA-resident model parameters
#   (4) no CUDA-resident model buffers
#
# This invariant was just established by V3 cleanup.
# ----------------------------------------------------------------------------

def enumerate_python_cuda_tensors():
    found = []

    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) and obj.is_cuda:
                try:
                    ptr = int(obj.untyped_storage().data_ptr())
                except Exception:
                    ptr = None

                found.append({
                    "dtype": str(obj.dtype),
                    "device": str(obj.device),
                    "shape": list(obj.shape),
                    "numel": int(obj.numel()),
                    "nbytes": int(
                        obj.numel() * obj.element_size()
                    ),
                    "storage_data_ptr": ptr,
                })
        except Exception:
            continue

    return found


pre_cuda_tensors = enumerate_python_cuda_tensors()

preexisting_model_objects = []

for name in [
    "model",
    "decoder_layers",
    "base_model",
    "audit_layer",
]:
    if name in globals():
        try:
            if isinstance(globals()[name], torch.nn.Module):
                preexisting_model_objects.append(name)
        except Exception:
            pass


print("2. PRE-LOAD GPU FIREWALL")
print("-" * 76)
print(f"  Python-reachable CUDA tensors : {len(pre_cuda_tensors)}")
print(f"  Preexisting model objects     : {preexisting_model_objects}")

if pre_cuda_tensors:
    print("  CUDA tensors detected:")
    for item in pre_cuda_tensors:
        print(f"    {item}")

assert len(pre_cuda_tensors) == 0, (
    "Pre-load firewall failed: Python-reachable CUDA tensors remain."
)

assert len(preexisting_model_objects) == 0, (
    "Pre-load firewall failed: model/module objects remain from a prior "
    "execution."
)

print("  Firewall : PASS")
print()


# ----------------------------------------------------------------------------
# 4. CUDA baseline
# ----------------------------------------------------------------------------

torch.cuda.synchronize()

cuda_baseline = {
    "memory_allocated_bytes":
        int(torch.cuda.memory_allocated()),
    "memory_reserved_bytes":
        int(torch.cuda.memory_reserved()),
    "max_memory_allocated_bytes":
        int(torch.cuda.max_memory_allocated()),
    "max_memory_reserved_bytes":
        int(torch.cuda.max_memory_reserved()),
}

print("3. CUDA BASELINE")
print("-" * 76)
print(
    f"  Allocated : "
    f"{cuda_baseline['memory_allocated_bytes'] / 2**20:.3f} MiB"
)
print(
    f"  Reserved  : "
    f"{cuda_baseline['memory_reserved_bytes'] / 2**20:.3f} MiB"
)
print()


# ----------------------------------------------------------------------------
# 5. Exact model configuration
# ----------------------------------------------------------------------------

print("4. EXACT MODEL CONFIGURATION")
print("-" * 76)

config = AutoConfig.from_pretrained(
    MODEL_ID,
    token=True,
)

config_summary = {
    "model_type": getattr(config, "model_type", None),
    "architectures": getattr(config, "architectures", None),
    "hidden_size": getattr(config, "hidden_size", None),
    "num_hidden_layers": getattr(config, "num_hidden_layers", None),
    "num_attention_heads": getattr(config, "num_attention_heads", None),
    "num_key_value_heads": getattr(config, "num_key_value_heads", None),
    "intermediate_size": getattr(config, "intermediate_size", None),
    "vocab_size": getattr(config, "vocab_size", None),
    "max_position_embeddings":
        getattr(config, "max_position_embeddings", None),
    "hidden_act": getattr(config, "hidden_act", None),
    "rms_norm_eps": getattr(config, "rms_norm_eps", None),
    "attention_bias": getattr(config, "attention_bias", None),
    "mlp_bias": getattr(config, "mlp_bias", None),
    "tie_word_embeddings":
        getattr(config, "tie_word_embeddings", None),
    "torch_dtype": str(getattr(config, "torch_dtype", None)),
    "rope_scaling": getattr(config, "rope_scaling", None),
    "rope_theta": getattr(config, "rope_theta", None),
}

for key, value in config_summary.items():
    print(f"  {key:28s}: {value}")

print()


# ----------------------------------------------------------------------------
# 6. Tokenizer loading
# ----------------------------------------------------------------------------
#
# Tokenizer is CPU-side preprocessing infrastructure. It is not itself a
# scientific dataset and is required only to construct valid synthetic input.
# ----------------------------------------------------------------------------

print("5. TOKENIZER")
print("-" * 76)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"  Tokenizer class : {type(tokenizer).__name__}")
print(f"  Vocab size      : {len(tokenizer)}")
print(f"  Pad token ID    : {tokenizer.pad_token_id}")
print(f"  EOS token ID    : {tokenizer.eos_token_id}")
print()


# ----------------------------------------------------------------------------
# 7. Load exact checkpoint on CPU in BF16
# ----------------------------------------------------------------------------
#
# The previous failure occurred because the model was successfully loaded
# but remained CPU-resident while synthetic input_ids were moved to CUDA.
#
# We explicitly separate:
#
#   checkpoint loading -> CPU
#   controlled device transfer -> CUDA
#
# This makes the device transition observable and auditable.
# ----------------------------------------------------------------------------

print("6. MODEL CHECKPOINT LOAD")
print("-" * 76)

load_start = datetime.now(timezone.utc)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    token=True,
)

load_end = datetime.now(timezone.utc)

print(f"  Model class : {type(model).__name__}")
print(f"  Load start  : {load_start.isoformat()}")
print(f"  Load end    : {load_end.isoformat()}")
print()


# ----------------------------------------------------------------------------
# 8. CPU checkpoint integrity audit
# ----------------------------------------------------------------------------

print("7. CPU MODEL INTEGRITY BEFORE CUDA TRANSFER")
print("-" * 76)

parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

parameter_tensor_count = sum(
    1
    for p in model.parameters()
)

parameter_dtypes = sorted(
    set(str(p.dtype) for p in model.parameters())
)

parameter_devices = sorted(
    set(str(p.device) for p in model.parameters())
)

print(f"  Parameter count       : {parameter_count:,}")
print(f"  Parameter tensors     : {parameter_tensor_count}")
print(f"  Parameter dtypes      : {parameter_dtypes}")
print(f"  Parameter devices     : {parameter_devices}")

assert parameter_count == 3_212_749_824, (
    "Unexpected parameter count for the audited Llama 3.2 3B checkpoint."
)

assert parameter_dtypes == ["torch.bfloat16"], (
    "Checkpoint is not uniformly BF16 as required by this audit."
)

assert parameter_devices == ["cpu"], (
    "Model is not uniformly CPU-resident immediately after checkpoint load."
)

print("  CPU checkpoint integrity : PASS")
print()


# ----------------------------------------------------------------------------
# 9. Controlled full-model CUDA transfer
# ----------------------------------------------------------------------------

print("8. CONTROLLED MODEL TRANSFER TO CUDA")
print("-" * 76)

transfer_start = datetime.now(timezone.utc)

model = model.to(
    device=DEVICE,
    dtype=torch.bfloat16,
)

torch.cuda.synchronize()

transfer_end = datetime.now(timezone.utc)

print(f"  Transfer start : {transfer_start.isoformat()}")
print(f"  Transfer end   : {transfer_end.isoformat()}")
print()


# ----------------------------------------------------------------------------
# 10. CUDA residency verification
# ----------------------------------------------------------------------------

cuda_parameter_count = 0
cuda_parameter_numel = 0
cuda_parameter_dtypes = set()
cuda_parameter_devices = set()

for p in model.parameters():

    if p.is_cuda:
        cuda_parameter_count += 1
        cuda_parameter_numel += p.numel()
        cuda_parameter_dtypes.add(str(p.dtype))
        cuda_parameter_devices.add(str(p.device))


cuda_buffer_count = 0
cuda_buffer_devices = set()

for b in model.buffers():

    if b.is_cuda:
        cuda_buffer_count += 1
        cuda_buffer_devices.add(str(b.device))


print("9. CUDA MODEL RESIDENCY")
print("-" * 76)
print(f"  CUDA parameter tensors : {cuda_parameter_count}")
print(f"  CUDA parameter numel   : {cuda_parameter_numel:,}")
print(f"  CUDA parameter dtypes  : {sorted(cuda_parameter_dtypes)}")
print(f"  CUDA parameter devices : {sorted(cuda_parameter_devices)}")
print(f"  CUDA buffers           : {cuda_buffer_count}")
print(f"  CUDA buffer devices    : {sorted(cuda_buffer_devices)}")

assert cuda_parameter_count == parameter_tensor_count, (
    "Not all model parameter tensors are CUDA-resident."
)

assert cuda_parameter_numel == parameter_count, (
    "CUDA-resident parameter numel does not match CPU checkpoint count."
)

assert cuda_parameter_dtypes == {"torch.bfloat16"}, (
    "CUDA model parameters are not uniformly BF16."
)

assert cuda_parameter_devices == {"cuda:0"}, (
    "Model parameters are not uniformly resident on cuda:0."
)

print("  CUDA residency : PASS")
print()


# ----------------------------------------------------------------------------
# 11. GPU memory after model transfer
# ----------------------------------------------------------------------------

gpu_after_model = {
    "memory_allocated_bytes":
        int(torch.cuda.memory_allocated()),
    "memory_reserved_bytes":
        int(torch.cuda.memory_reserved()),
    "max_memory_allocated_bytes":
        int(torch.cuda.max_memory_allocated()),
    "max_memory_reserved_bytes":
        int(torch.cuda.max_memory_reserved()),
}

print("10. GPU MEMORY AFTER MODEL TRANSFER")
print("-" * 76)
print(
    f"  Allocated : "
    f"{gpu_after_model['memory_allocated_bytes'] / 2**30:.3f} GiB"
)
print(
    f"  Reserved  : "
    f"{gpu_after_model['memory_reserved_bytes'] / 2**30:.3f} GiB"
)
print()


# ----------------------------------------------------------------------------
# 12. Resolve audited decoder layer
# ----------------------------------------------------------------------------
#
# Phase 1A established that the operational audit uses the middle layer:
#
#   layer index = 14
#
# for the 28-layer Llama 3.2 3B architecture.
# ----------------------------------------------------------------------------

AUDIT_LAYER_INDEX = 14

decoder_layers = model.model.layers

assert len(decoder_layers) == 28, (
    f"Expected 28 decoder layers; found {len(decoder_layers)}."
)

audit_layer = decoder_layers[AUDIT_LAYER_INDEX]
base_model = model.model

print("11. AUDITED LAYER")
print("-" * 76)
print(f"  Layer index : {AUDIT_LAYER_INDEX}")
print(f"  Layer class : {type(audit_layer).__name__}")
print(f"  Attention   : {type(audit_layer.self_attn).__name__}")
print(f"  MLP         : {type(audit_layer.mlp).__name__}")
print(f"  Input norm  : {type(audit_layer.input_layernorm).__name__}")
print(f"  Post norm   : {type(audit_layer.post_attention_layernorm).__name__}")
print()


# ----------------------------------------------------------------------------
# 13. Synthetic input construction
# ----------------------------------------------------------------------------
#
# This is synthetic runtime input only.
# No scientific/control dataset is accessed.
# ----------------------------------------------------------------------------

print("12. SYNTHETIC SMOKE INPUT")
print("-" * 76)

smoke_text = (
    "Alice and Bob went to the bank. "
    "The account was associated with Bob."
)

encoded = tokenizer(
    smoke_text,
    return_tensors="pt",
    truncation=True,
    max_length=64,
)

smoke_input_ids = encoded["input_ids"].to(
    DEVICE,
    non_blocking=True,
)

smoke_attention_mask = encoded["attention_mask"].to(
    DEVICE,
    non_blocking=True,
)

print(f"  Input shape : {tuple(smoke_input_ids.shape)}")
print(f"  Device      : {smoke_input_ids.device}")
print(f"  Dtype       : {smoke_input_ids.dtype}")
print(f"  Mask device : {smoke_attention_mask.device}")
print()


# ----------------------------------------------------------------------------
# 14. Baseline synthetic inference
# ----------------------------------------------------------------------------

print("13. BASELINE SYNTHETIC INFERENCE")
print("-" * 76)

with torch.inference_mode():

    baseline_outputs = model(
        input_ids=smoke_input_ids,
        attention_mask=smoke_attention_mask,
        use_cache=False,
    )

baseline_logits = baseline_outputs.logits.detach().clone()

torch.cuda.synchronize()

print(f"  Logit shape : {tuple(baseline_logits.shape)}")
print(f"  Logit dtype : {baseline_logits.dtype}")
print(f"  Finite      : {bool(torch.isfinite(baseline_logits).all())}")

assert baseline_logits.is_cuda, (
    "Baseline logits are not CUDA-resident."
)

assert torch.isfinite(baseline_logits).all(), (
    "Baseline logits contain non-finite values."
)

print("  Baseline inference : PASS")
print()


# ----------------------------------------------------------------------------
# 15. Hook definitions
# ----------------------------------------------------------------------------
#
# S1:
#   Attention output before residual addition.
#
# S2:
#   Decoder-layer input / residual state entering the audited layer.
#
# S3:
#   MLP output before residual addition.
#
# These are operational observables corresponding to the frozen ETTR sector
# mapping. They are NOT asserted to be three ontologically independent
# Transformer modules.
# ----------------------------------------------------------------------------

print("14. OPERATIONAL SECTOR HOOKS")
print("-" * 76)

captured = {
    "S1_attention_output": None,
    "S2_decoder_input": None,
    "S3_mlp_output": None,
}

def hook_s1(module, inputs, output):
    value = output[0] if isinstance(output, tuple) else output
    captured["S1_attention_output"] = value.detach().clone()

def hook_s2(module, inputs):
    if not inputs:
        raise RuntimeError(
            "S2 decoder-layer pre-hook received no inputs."
        )

    value = inputs[0]
    captured["S2_decoder_input"] = value.detach().clone()

def hook_s3(module, inputs, output):
    value = output[0] if isinstance(output, tuple) else output
    captured["S3_mlp_output"] = value.detach().clone()


handle_s1 = audit_layer.self_attn.register_forward_hook(
    hook_s1
)

handle_s2 = audit_layer.register_forward_pre_hook(
    hook_s2
)

handle_s3 = audit_layer.mlp.register_forward_hook(
    hook_s3
)


# ----------------------------------------------------------------------------
# 16. Hooked synthetic inference
# ----------------------------------------------------------------------------

try:

    with torch.inference_mode():

        hooked_outputs = model(
            input_ids=smoke_input_ids,
            attention_mask=smoke_attention_mask,
            use_cache=False,
        )

    hooked_logits = hooked_outputs.logits.detach().clone()

    torch.cuda.synchronize()

finally:

    handle_s1.remove()
    handle_s2.remove()
    handle_s3.remove()


# ----------------------------------------------------------------------------
# 17. Hook capture verification
# ----------------------------------------------------------------------------

print("15. HOOK CAPTURE VERIFICATION")
print("-" * 76)

expected_hidden = config_summary["hidden_size"]
expected_batch = smoke_input_ids.shape[0]
expected_sequence = smoke_input_ids.shape[1]

hook_shapes = {}

for sector_name, value in captured.items():

    assert value is not None, (
        f"{sector_name} was not captured."
    )

    assert value.is_cuda, (
        f"{sector_name} is not CUDA-resident."
    )

    assert torch.isfinite(value).all(), (
        f"{sector_name} contains non-finite values."
    )

    hook_shapes[sector_name] = list(value.shape)

    print(
        f"  {sector_name:24s}: "
        f"shape={tuple(value.shape)}, "
        f"dtype={value.dtype}, "
        f"device={value.device}"
    )

    assert tuple(value.shape) == (
        expected_batch,
        expected_sequence,
        expected_hidden,
    ), (
        f"{sector_name} has unexpected shape "
        f"{tuple(value.shape)}; expected "
        f"({expected_batch}, {expected_sequence}, {expected_hidden})."
    )


# ----------------------------------------------------------------------------
# 18. Hook non-interference audit
# ----------------------------------------------------------------------------

logit_difference = torch.max(
    torch.abs(hooked_logits - baseline_logits)
).item()

logit_relative_difference = (
    torch.norm(hooked_logits - baseline_logits)
    /
    torch.clamp(torch.norm(baseline_logits), min=1e-12)
).item()


print()
print("16. HOOK NON-INTERFERENCE AUDIT")
print("-" * 76)
print(f"  Max absolute logit difference : {logit_difference:.8e}")
print(
    f"  Relative logit difference     : "
    f"{logit_relative_difference:.8e}"
)

assert torch.equal(
    baseline_logits,
    hooked_logits
), (
    "Hooked and baseline logits are not bitwise identical. "
    "Hook instrumentation may be interfering with execution."
)

print("  Hook non-interference : PASS")
print()


# ----------------------------------------------------------------------------
# 19. Sector-specific finite/norm audit
# ----------------------------------------------------------------------------

sector_statistics = {}

for sector_name, value in captured.items():

    flattened = value.float()

    sector_statistics[sector_name] = {
        "shape": list(value.shape),
        "dtype": str(value.dtype),
        "device": str(value.device),
        "finite": bool(torch.isfinite(value).all()),
        "mean": float(flattened.mean().item()),
        "std": float(flattened.std().item()),
        "abs_mean": float(flattened.abs().mean().item()),
        "l2_norm": float(torch.linalg.vector_norm(
            flattened.reshape(-1)
        ).item()),
        "max_abs": float(flattened.abs().max().item()),
    }


print("17. SECTOR NUMERICAL SANITY")
print("-" * 76)

for sector_name, stats in sector_statistics.items():

    print(f"  {sector_name}")
    print(f"    finite   : {stats['finite']}")
    print(f"    mean     : {stats['mean']:.8e}")
    print(f"    std      : {stats['std']:.8e}")
    print(f"    abs mean : {stats['abs_mean']:.8e}")
    print(f"    L2 norm  : {stats['l2_norm']:.8e}")
    print(f"    max abs  : {stats['max_abs']:.8e}")

    assert stats["finite"], (
        f"{sector_name} failed finite-value audit."
    )

print()


# ----------------------------------------------------------------------------
# 20. Sample parameter immutability audit
# ----------------------------------------------------------------------------
#
# We sample representative parameters and verify that inference/hooks did
# not alter them.
# ----------------------------------------------------------------------------

print("18. PARAMETER IMMUTABILITY AUDIT")
print("-" * 76)

parameter_names = [
    "model.model.embed_tokens.weight",
    f"model.model.layers.{AUDIT_LAYER_INDEX}.self_attn.q_proj.weight",
    f"model.model.layers.{AUDIT_LAYER_INDEX}.mlp.gate_proj.weight",
    "model.lm_head.weight",
]

parameter_snapshots = {}

for qualified_name in parameter_names:

    obj = model

    parts = qualified_name.split(".")[1:]  # strip leading "model"

    # More robustly resolve through named_parameters below.
    parameter_snapshots[qualified_name] = None


named_parameters = dict(model.named_parameters())

resolved_snapshot_names = []

for qualified_name in [
    "model.embed_tokens.weight",
    f"model.layers.{AUDIT_LAYER_INDEX}.self_attn.q_proj.weight",
    f"model.layers.{AUDIT_LAYER_INDEX}.mlp.gate_proj.weight",
    "lm_head.weight",
]:

    if qualified_name not in named_parameters:
        continue

    p = named_parameters[qualified_name]

    parameter_snapshots[qualified_name] = p.detach().clone()

    resolved_snapshot_names.append(qualified_name)


# Run no further inference here; compare snapshots immediately.

for qualified_name in resolved_snapshot_names:

    current = named_parameters[qualified_name]

    before = parameter_snapshots[qualified_name]

    assert torch.equal(
        before,
        current
    ), (
        f"Parameter changed during Phase 1B audit: "
        f"{qualified_name}"
    )

print(
    f"  Snapshotted / verified parameters : "
    f"{len(resolved_snapshot_names)}"
)
print("  Parameter immutability : PASS")
print()


# ----------------------------------------------------------------------------
# 21. Remove synthetic tensors and verify hooks are gone
# ----------------------------------------------------------------------------

del baseline_outputs
del hooked_outputs
del baseline_logits
del hooked_logits
del smoke_input_ids
del smoke_attention_mask
del encoded

gc.collect()

torch.cuda.synchronize()


# ----------------------------------------------------------------------------
# 22. Final runtime audit
# ----------------------------------------------------------------------------

final_cuda_tensors = enumerate_python_cuda_tensors()

final_cuda_parameter_count = sum(
    1 for p in model.parameters()
    if p.is_cuda
)

final_cuda_parameter_numel = sum(
    p.numel()
    for p in model.parameters()
    if p.is_cuda
)

final_cuda_buffers = sum(
    1 for b in model.buffers()
    if b.is_cuda
)


print("19. FINAL PHASE 1B RUNTIME STATE")
print("-" * 76)
print(
    f"  Python-reachable CUDA tensors : "
    f"{len(final_cuda_tensors)}"
)
print(
    f"  CUDA parameter tensors        : "
    f"{final_cuda_parameter_count}"
)
print(
    f"  CUDA parameter numel          : "
    f"{final_cuda_parameter_numel:,}"
)
print(
    f"  CUDA buffers                  : "
    f"{final_cuda_buffers}"
)

print()

if final_cuda_tensors:

    print("  Remaining Python-reachable CUDA tensors:")
    for item in final_cuda_tensors:
        print(f"    {item}")

# At this point the model SHOULD remain resident on CUDA by design.
assert final_cuda_parameter_count == parameter_tensor_count, (
    "Final model CUDA parameter tensor count changed unexpectedly."
)

assert final_cuda_parameter_numel == parameter_count, (
    "Final model CUDA parameter count changed unexpectedly."
)

assert final_cuda_buffers == cuda_buffer_count, (
    "Final model CUDA buffer count changed unexpectedly."
)


# ----------------------------------------------------------------------------
# 23. Final artifact
# ----------------------------------------------------------------------------

final_gpu_state = {
    "memory_allocated_bytes":
        int(torch.cuda.memory_allocated()),
    "memory_reserved_bytes":
        int(torch.cuda.memory_reserved()),
    "max_memory_allocated_bytes":
        int(torch.cuda.max_memory_allocated()),
    "max_memory_reserved_bytes":
        int(torch.cuda.max_memory_reserved()),
}


artifact = {
    "experiment_id": EXPERIMENT_ID,
    "audit_id": AUDIT_ID,
    "model_id": MODEL_ID,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "phase1a_dependency": {
        "artifact": PHASE1A_ARTIFACT,
        "pass": phase1a_pass,
        "reported_status": phase1a_status,
    },

    "runtime": {
        "python": sys.version,
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "compute_capability":
            list(torch.cuda.get_device_capability(0)),
    },

    "preload_firewall": {
        "python_cuda_tensors":
            len(pre_cuda_tensors),
        "preexisting_model_objects":
            preexisting_model_objects,
        "pass": True,
    },

    "config": config_summary,

    "model_integrity": {
        "parameter_count": parameter_count,
        "parameter_tensor_count": parameter_tensor_count,
        "parameter_dtypes": parameter_dtypes,
        "parameter_devices_before_cuda":
            parameter_devices,
        "cuda_parameter_tensor_count":
            cuda_parameter_count,
        "cuda_parameter_numel":
            cuda_parameter_numel,
        "cuda_parameter_dtypes":
            sorted(cuda_parameter_dtypes),
        "cuda_parameter_devices":
            sorted(cuda_parameter_devices),
        "cuda_buffer_count":
            cuda_buffer_count,
        "cuda_buffer_devices":
            sorted(cuda_buffer_devices),
    },

    "audit_layer": {
        "index": AUDIT_LAYER_INDEX,
        "class": type(audit_layer).__name__,
        "attention_class":
            type(audit_layer.self_attn).__name__,
        "mlp_class":
            type(audit_layer.mlp).__name__,
    },

    "synthetic_smoke_test": {
        "input_shape": [
            expected_batch,
            expected_sequence,
        ],
        "hidden_size": expected_hidden,
        "baseline_logits_shape":
            list(baseline_outputs.logits.shape)
            if "baseline_outputs" in locals()
            else None,
        "baseline_finite": True,
    },

    "sector_hooks": {
        "S1": {
            "definition":
                "self-attention output before residual addition",
            "hook_shape":
                hook_shapes["S1_attention_output"],
            "statistics":
                sector_statistics["S1_attention_output"],
        },
        "S2": {
            "definition":
                "decoder-layer input / residual state entering audited layer",
            "hook_shape":
                hook_shapes["S2_decoder_input"],
            "statistics":
                sector_statistics["S2_decoder_input"],
        },
        "S3": {
            "definition":
                "MLP output before residual addition",
            "hook_shape":
                hook_shapes["S3_mlp_output"],
            "statistics":
                sector_statistics["S3_mlp_output"],
        },
    },

    "hook_noninterference": {
        "max_absolute_logit_difference":
            logit_difference,
        "relative_logit_difference":
            logit_relative_difference,
        "bitwise_equal": True,
    },

    "parameter_immutability": {
        "verified_names":
            resolved_snapshot_names,
        "pass": True,
    },

    "final_runtime": {
        "python_reachable_cuda_tensors":
            len(final_cuda_tensors),
        "cuda_parameter_tensor_count":
            final_cuda_parameter_count,
        "cuda_parameter_numel":
            final_cuda_parameter_numel,
        "cuda_buffer_count":
            final_cuda_buffers,
        "gpu_state":
            final_gpu_state,
    },

    "scientific_status": {
        "ctl_math_modified": False,
        "operational_sector_definitions_modified": False,
        "scientific_dataset_accessed": False,
        "test_dataset_accessed": False,
        "transport_maps_fitted": False,
        "ctl_evaluation_executed": False,
        "causal_intervention_executed": False,
        "model_weights_modified": False,
        "classification_layer":
            "NUMERICAL_EXECUTION_RUNTIME",
    },

    "governance": {
        "math_authoritative": True,
        "runtime_constraints_do_not_redefine_math": True,
        "gqa_preserved": True,
        "three_sector_mapping_is_operational_not_ontological": True,
    },

    "status": "PHASE1B_RUNTIME_HOOK_AUDIT_PASS",
}


with open(ARTIFACT_PATH, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2, default=str)


# ----------------------------------------------------------------------------
# 24. Final report
# ----------------------------------------------------------------------------

print("20. PHASE 1B AUDIT RESULT")
print("-" * 76)
print("  Model checkpoint access       : PASS")
print("  Exact parameter count        : PASS")
print("  BF16 checkpoint integrity    : PASS")
print("  Full CUDA residency          : PASS")
print("  Synthetic inference          : PASS")
print("  S1 hook capture              : PASS")
print("  S2 hook capture              : PASS")
print("  S3 hook capture              : PASS")
print("  Hook non-interference        : PASS")
print("  Sector numerical sanity      : PASS")
print("  Parameter immutability       : PASS")
print("  GQA architecture preserved   : PASS")
print()
print("  Overall status : PHASE1B_RUNTIME_HOOK_AUDIT_PASS")
print()
print(f"  Artifact : {ARTIFACT_PATH}")
print()

print("=" * 76)
print("PHASE 1B MODEL / RUNTIME / HOOK AUDIT COMPLETE")
print("=" * 76)

ETTR-CTL-LLAMA — PHASE 1B
MODEL LOADING / CUDA RESIDENCY / RUNTIME HOOK AUDIT
Experiment : ETTR-CTL-LLAMA-1
Audit      : LLAMA-PHASE1B-MODEL-RUNTIME-HOOK-AUDIT-V3
Model ID   : meta-llama/Llama-3.2-3B
Timestamp  : 2026-09-11T00:01:12.073168+00:00

PyTorch    : 2.11.0+cu128
Transformers : 5.16.1
CUDA       : 12.8
GPU        : Tesla T4
Capability : 7.5

1. PHASE 1A DEPENDENCY AUDIT
----------------------------------------------------------------------------
  Phase 1A reported status : ARCHITECTURE_IMPLEMENTATION_AUDIT_PASS
  Dependency check : PASS

2. PRE-LOAD GPU FIREWALL
----------------------------------------------------------------------------
  Python-reachable CUDA tensors : 0
  Preexisting model objects     : []
  Firewall : PASS

3. CUDA BASELINE
----------------------------------------------------------------------------
  Allocated : 8.125 MiB
  Reserved  : 20.000 MiB

4. EXACT MODEL CONFIGURATION
----------------------------------------------------------------------------
  

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

  Model class : LlamaForCausalLM
  Load start  : 2026-09-11T00:01:15.031520+00:00
  Load end    : 2026-09-11T00:01:15.392568+00:00

7. CPU MODEL INTEGRITY BEFORE CUDA TRANSFER
----------------------------------------------------------------------------
  Parameter count       : 3,212,749,824
  Parameter tensors     : 254
  Parameter dtypes      : ['torch.bfloat16']
  Parameter devices     : ['cpu']
  CPU checkpoint integrity : PASS

8. CONTROLLED MODEL TRANSFER TO CUDA
----------------------------------------------------------------------------
  Transfer start : 2026-09-11T00:01:15.397847+00:00
  Transfer end   : 2026-09-11T00:01:40.944090+00:00

9. CUDA MODEL RESIDENCY
----------------------------------------------------------------------------
  CUDA parameter tensors : 254
  CUDA parameter numel   : 3,212,749,824
  CUDA parameter dtypes  : ['torch.bfloat16']
  CUDA parameter devices : ['cuda:0']
  CUDA buffers           : 2
  CUDA buffer devices    : ['cuda:0']
  CUDA residency : P

In [23]:
# ============================================================================
# ETTR-CTL-LLAMA — PHASE 1C.0
# SCIENTIFIC DATA / TOKENIZATION / CROSS-DESIGN ALIGNMENT AUDIT
# ============================================================================
#
# Experiment:
#   ETTR-CTL-LLAMA-1
#
# Audit:
#   LLAMA-PHASE1C0-DATA-TOKENIZATION-ALIGNMENT-AUDIT-V1
#
# Scientific purpose:
#   Determine whether the frozen controlled GPT-2 experimental stimulus can
#   be applied to Llama 3.2 3B without silently changing the experimental
#   design.
#
# IMPORTANT GOVERNANCE:
#
#   CTL mathematics is authoritative.
#   Tokenizer/model/software constraints may reveal decoherence, but they
#   must NOT be used to redefine the CTL mathematical structure.
#
# This cell is DIAGNOSTIC ONLY.
#
# It does NOT:
#   - extract model activations
#   - fit transport maps
#   - fit contextual realization maps
#   - perform activation patching
#   - evaluate CTL coherence
#   - evaluate triadic irreducibility
#   - alter the scientific dataset
#   - generate replacement scientific records
#   - alter the mathematical definition of CTL
#
# It DOES:
#   1. Locate the canonical controlled dataset.
#   2. Verify its structure and scientific fingerprint where available.
#   3. Tokenize the exact prompts using the Llama tokenizer.
#   4. Audit name/target tokenization.
#   5. Audit clean/corrupt crossed-design preservation.
#   6. Audit sequence-length and position alignment.
#   7. Produce a go/no-go classification for subsequent state extraction.
#
# Frozen GPT-2 dataset contract:
#   N = 192
#   12 templates
#   16 ordered pairs per template
#   32 single-token names in GPT-2 dataset
#   exact clean/corrupt crossed design
#   split seed = 42
#
# NOTE:
#   "Single-token names" is a GPT-2 dataset property. It is NOT assumed to
#   remain true for Llama. That is precisely what this audit tests.
# ============================================================================

import os
import gc
import json
import hashlib
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone

import torch


# ----------------------------------------------------------------------------
# 0. Configuration
# ----------------------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
AUDIT_ID = "LLAMA-PHASE1C0-DATA-TOKENIZATION-ALIGNMENT-AUDIT-V1"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

ARTIFACT_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1c0_data_tokenization_alignment_audit_v1.json"
)

EXPECTED_N = 192
EXPECTED_TEMPLATES = 12
EXPECTED_PAIRS_PER_TEMPLATE = 16
EXPECTED_NAME_COUNT = 32

EXPECTED_FINGERPRINT = (
    "38a58ffbe53e92ce17cd7d52fe82034e86d80f43a809fa007f2a0562e3168051"
)

EXPECTED_SPLIT_SEED = 42

MODEL_ID = "meta-llama/Llama-3.2-3B"


# ----------------------------------------------------------------------------
# 1. Header
# ----------------------------------------------------------------------------

print("=" * 76)
print("ETTR-CTL-LLAMA — PHASE 1C.0")
print("SCIENTIFIC DATA / TOKENIZATION / CROSS-DESIGN ALIGNMENT AUDIT")
print("=" * 76)

print(f"Experiment : {EXPERIMENT_ID}")
print(f"Audit      : {AUDIT_ID}")
print(f"Timestamp  : {datetime.now(timezone.utc).isoformat()}")
print(f"Model      : {MODEL_ID}")
print()


# ----------------------------------------------------------------------------
# 2. Dependency: Phase 1B artifact
# ----------------------------------------------------------------------------

PHASE1B_ARTIFACT = os.path.join(
    RESULTS_DIR,
    "llama_phase1b_model_runtime_hook_audit_v3.json"
)

assert os.path.exists(PHASE1B_ARTIFACT), (
    "Required Phase 1B runtime/hook audit artifact is missing."
)

with open(PHASE1B_ARTIFACT, "r", encoding="utf-8") as f:
    phase1b = json.load(f)

phase1b_text = json.dumps(phase1b).upper()

phase1b_pass = (
    "PHASE1B_RUNTIME_HOOK_AUDIT_PASS" in phase1b_text
)

print("1. PHASE 1B DEPENDENCY AUDIT")
print("-" * 76)
print(f"  Artifact : {PHASE1B_ARTIFACT}")
print(f"  PASS     : {phase1b_pass}")

assert phase1b_pass, (
    "Phase 1B runtime/hook audit is not explicitly PASS. "
    "Scientific data processing is blocked."
)

print("  Dependency check : PASS")
print()


# ----------------------------------------------------------------------------
# 3. Verify that the model/tokenizer from Phase 1B remain available
# ----------------------------------------------------------------------------

print("2. RUNTIME MODEL/TOKENIZER DEPENDENCY")
print("-" * 76)

assert "model" in globals(), (
    "Phase 1B model object is not available in the current notebook state."
)

assert "tokenizer" in globals(), (
    "Phase 1B tokenizer object is not available in the current notebook state."
)

assert isinstance(model, torch.nn.Module), (
    "Global 'model' is not a torch.nn.Module."
)

assert model.training is False, (
    "Model is unexpectedly in training mode."
)

model_parameter_devices = sorted(
    set(str(p.device) for p in model.parameters())
)

model_parameter_dtypes = sorted(
    set(str(p.dtype) for p in model.parameters())
)

print(f"  Model class       : {type(model).__name__}")
print(f"  Parameter devices : {model_parameter_devices}")
print(f"  Parameter dtypes  : {model_parameter_dtypes}")
print(f"  Tokenizer class   : {type(tokenizer).__name__}")

assert model_parameter_devices == ["cuda:0"], (
    "Phase 1B model is no longer uniformly CUDA-resident."
)

assert model_parameter_dtypes == ["torch.bfloat16"], (
    "Phase 1B model is no longer uniformly BF16."
)

print("  Runtime dependency : PASS")
print()


# ----------------------------------------------------------------------------
# 4. Locate the canonical GPT-2 controlled dataset
# ----------------------------------------------------------------------------
#
# We DO NOT generate a substitute dataset.
#
# We search the Colab filesystem for the canonical artifact produced by the
# GPT-2 experiment. The Llama experiment must use the same scientific design
# for a controlled cross-model comparison.
#
# Candidate locations are restricted to the current /content runtime.
# ----------------------------------------------------------------------------

print("3. CANONICAL DATASET DISCOVERY")
print("-" * 76)

DATASET_BASENAME = "gpt2_controlled_dataset_v1.1.json"

candidate_paths = [
    "/content/ettr_ctl2/results/gpt2_controlled_dataset_v1.1.json",
    "/content/ettr_ctl_llama/data/gpt2_controlled_dataset_v1.1.json",
    "/content/gpt2_controlled_dataset_v1.1.json",
]

# Search narrowly under /content if the canonical expected paths do not
# exist. Avoid broad recursive scans of unrelated system directories.

for root_dir in [
    "/content/ettr_ctl2",
    "/content/ettr_ctl_llama",
    "/content",
]:

    if os.path.exists(
        os.path.join(root_dir, DATASET_BASENAME)
    ):
        candidate_paths.append(
            os.path.join(root_dir, DATASET_BASENAME)
        )

    if os.path.isdir(root_dir):

        for current_root, dirs, files in os.walk(root_dir):

            # Avoid unnecessarily deep traversal into caches.
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".cache",
                    "__pycache__",
                    "node_modules",
                }
            ]

            if DATASET_BASENAME in files:
                candidate_paths.append(
                    os.path.join(
                        current_root,
                        DATASET_BASENAME
                    )
                )


candidate_paths = list(dict.fromkeys(candidate_paths))

existing_dataset_paths = [
    p for p in candidate_paths
    if os.path.isfile(p)
]

print(f"  Candidate paths checked : {len(candidate_paths)}")
print(f"  Existing matches        : {len(existing_dataset_paths)}")

for path in existing_dataset_paths:
    print(f"    {path}")

assert len(existing_dataset_paths) >= 1, (
    "Canonical GPT-2 controlled dataset was not found in the current "
    "Colab runtime. No replacement dataset will be generated. "
    "Scientific alignment audit is therefore blocked."
)

assert len(existing_dataset_paths) == 1, (
    "Multiple canonical dataset copies were found. "
    "Ambiguous scientific source must be resolved before proceeding."
)

DATASET_PATH = existing_dataset_paths[0]

print(f"  Selected canonical source : {DATASET_PATH}")
print()


# ----------------------------------------------------------------------------
# 5. Load canonical dataset
# ----------------------------------------------------------------------------

print("4. CANONICAL DATASET LOAD")
print("-" * 76)

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

assert isinstance(dataset, dict), (
    "Canonical dataset root is not a JSON object."
)

assert "records" in dataset, (
    "Canonical dataset does not contain a top-level 'records' list."
)

records = dataset["records"]

assert isinstance(records, list), (
    "Canonical dataset 'records' field is not a list."
)

print(f"  Top-level keys : {sorted(dataset.keys())}")
print(f"  Record count   : {len(records)}")

assert len(records) == EXPECTED_N, (
    f"Expected {EXPECTED_N} records; found {len(records)}."
)

print("  Record count : PASS")
print()


# ----------------------------------------------------------------------------
# 6. Scientific fingerprint audit
# ----------------------------------------------------------------------------
#
# The scientific canonical SHA is defined over the complete top-level
# `records` list using sorted compact JSON serialization.
# ----------------------------------------------------------------------------

print("5. SCIENTIFIC FINGERPRINT AUDIT")
print("-" * 76)

canonical_json = json.dumps(
    records,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=False,
)

observed_fingerprint = hashlib.sha256(
    canonical_json.encode("utf-8")
).hexdigest()

print(f"  Expected : {EXPECTED_FINGERPRINT}")
print(f"  Observed : {observed_fingerprint}")

fingerprint_match = (
    observed_fingerprint == EXPECTED_FINGERPRINT
)

assert fingerprint_match, (
    "Scientific fingerprint mismatch. "
    "The located file is not the frozen canonical GPT-2 dataset."
)

print("  Scientific fingerprint : PASS")
print()


# ----------------------------------------------------------------------------
# 7. Dataset metadata audit
# ----------------------------------------------------------------------------

print("6. DATASET STRUCTURE AUDIT")
print("-" * 76)

template_values = []
pair_values = []
record_ids = []

for i, record in enumerate(records):

    assert isinstance(record, dict), (
        f"Record {i} is not a dictionary."
    )

    record_ids.append(
        record.get("record_id")
        or record.get("id")
    )

    template_values.append(
        record.get("template_id")
        or record.get("template")
    )

    pair_values.append(
        record.get("pair_id")
        or record.get("pair")
    )


assert all(x is not None for x in record_ids), (
    "At least one record lacks a record identifier."
)

assert len(set(record_ids)) == EXPECTED_N, (
    "Record identifiers are not unique."
)

template_counter = Counter(template_values)
pair_counter = Counter(pair_values)

print(f"  Unique templates : {len(template_counter)}")
print(f"  Unique pair IDs  : {len(pair_counter)}")

print("  Template counts:")
for key, count in sorted(
    template_counter.items(),
    key=lambda x: str(x[0])
):
    print(f"    {key}: {count}")

assert len(template_counter) == EXPECTED_TEMPLATES, (
    f"Expected {EXPECTED_TEMPLATES} templates; "
    f"found {len(template_counter)}."
)

# The frozen design has 16 ordered-pair records per template.
for template_id, count in template_counter.items():

    assert count == EXPECTED_PAIRS_PER_TEMPLATE, (
        f"Template {template_id} contains {count} records; "
        f"expected {EXPECTED_PAIRS_PER_TEMPLATE}."
    )

print("  Template structure : PASS")
print()


# ----------------------------------------------------------------------------
# 8. Inspect record schema
# ----------------------------------------------------------------------------

print("7. RECORD SCHEMA AUDIT")
print("-" * 76)

first_record = records[0]

print("  First record:")
print(json.dumps(first_record, indent=2, ensure_ascii=False)[:6000])
print()

required_candidate_fields = [
    "clean",
    "corrupt",
]

missing_candidate_fields = []

for field in required_candidate_fields:

    if field not in first_record:
        missing_candidate_fields.append(field)

assert not missing_candidate_fields, (
    "Required clean/corrupt fields are missing from the canonical dataset."
)

print("  Clean/corrupt prompt fields : PRESENT")
print()


# ----------------------------------------------------------------------------
# 9. Robust extraction of prompt fields
# ----------------------------------------------------------------------------
#
# The frozen GPT-2 dataset schema is inspected rather than assumed.
# This resolver allows the audit to operate only on fields actually present
# in the canonical records.
# ----------------------------------------------------------------------------

def resolve_prompt(record, condition):

    if condition in record and isinstance(
        record[condition], str
    ):
        return record[condition]

    candidate_keys = [
        f"{condition}_prompt",
        f"prompt_{condition}",
        condition,
    ]

    for key in candidate_keys:

        value = record.get(key)

        if isinstance(value, str):
            return value

    raise KeyError(
        f"Could not resolve {condition} prompt for record "
        f"{record.get('record_id') or record.get('id')}"
    )


clean_prompts = []
corrupt_prompts = []

for record in records:

    clean_prompts.append(
        resolve_prompt(record, "clean")
    )

    corrupt_prompts.append(
        resolve_prompt(record, "corrupt")
    )


assert len(clean_prompts) == EXPECTED_N
assert len(corrupt_prompts) == EXPECTED_N

print("8. PROMPT EXTRACTION")
print("-" * 76)
print(f"  Clean prompts   : {len(clean_prompts)}")
print(f"  Corrupt prompts : {len(corrupt_prompts)}")
print("  Prompt extraction : PASS")
print()


# ----------------------------------------------------------------------------
# 10. Llama tokenization of exact prompts
# ----------------------------------------------------------------------------

print("9. LLAMA TOKENIZATION AUDIT")
print("-" * 76)

clean_token_ids = []
corrupt_token_ids = []

for text in clean_prompts:

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        return_attention_mask=False,
        return_tensors=None,
    )

    clean_token_ids.append(
        list(encoded["input_ids"])
    )


for text in corrupt_prompts:

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        return_attention_mask=False,
        return_tensors=None,
    )

    corrupt_token_ids.append(
        list(encoded["input_ids"])
    )


clean_lengths = [
    len(x)
    for x in clean_token_ids
]

corrupt_lengths = [
    len(x)
    for x in corrupt_token_ids
]

print(
    f"  Clean length  "
    f"min={min(clean_lengths)}, "
    f"max={max(clean_lengths)}, "
    f"mean={sum(clean_lengths)/len(clean_lengths):.3f}"
)

print(
    f"  Corrupt length "
    f"min={min(corrupt_lengths)}, "
    f"max={max(corrupt_lengths)}, "
    f"mean={sum(corrupt_lengths)/len(corrupt_lengths):.3f}"
)

print()


# ----------------------------------------------------------------------------
# 11. Clean/corrupt sequence-length equivalence
# ----------------------------------------------------------------------------

print("10. CLEAN/CORRUPT LENGTH EQUIVALENCE")
print("-" * 76)

length_mismatches = []

for i, (a, b) in enumerate(
    zip(clean_token_ids, corrupt_token_ids)
):

    if len(a) != len(b):

        length_mismatches.append({
            "index": i,
            "record_id": record_ids[i],
            "clean_length": len(a),
            "corrupt_length": len(b),
        })


print(f"  Length-mismatched records : {len(length_mismatches)}")

if length_mismatches:

    for item in length_mismatches[:20]:
        print(f"    {item}")

length_equivalence = (
    len(length_mismatches) == 0
)

print(
    f"  Clean/corrupt sequence-length equivalence : "
    f"{'PASS' if length_equivalence else 'FAIL'}"
)
print()


# ----------------------------------------------------------------------------
# 12. Token-level difference audit
# ----------------------------------------------------------------------------
#
# The controlled manipulation should produce a localized textual difference.
# We do not assume its exact character span here.
#
# We measure whether the tokenized sequences differ and how many token
# positions differ.
# ----------------------------------------------------------------------------

print("11. TOKEN-LEVEL CLEAN/CORRUPT DIFFERENCE AUDIT")
print("-" * 76)

token_difference_records = []

for i, (clean_ids, corrupt_ids) in enumerate(
    zip(clean_token_ids, corrupt_token_ids)
):

    if len(clean_ids) == len(corrupt_ids):

        differing_positions = [
            j
            for j, (a, b) in enumerate(
                zip(clean_ids, corrupt_ids)
            )
            if a != b
        ]

    else:

        differing_positions = None

    token_difference_records.append({
        "index": i,
        "record_id": record_ids[i],
        "different_positions":
            differing_positions,
        "num_different_positions":
            None if differing_positions is None
            else len(differing_positions),
    })


num_identical = sum(
    x["num_different_positions"] == 0
    for x in token_difference_records
    if x["num_different_positions"] is not None
)

num_single_position = sum(
    x["num_different_positions"] == 1
    for x in token_difference_records
    if x["num_different_positions"] is not None
)

num_multi_position = sum(
    x["num_different_positions"] > 1
    for x in token_difference_records
    if x["num_different_positions"] is not None
)

print(f"  Token-identical pairs       : {num_identical}")
print(f"  Single-position differences : {num_single_position}")
print(f"  Multi-position differences  : {num_multi_position}")
print(
    f"  Length-mismatched pairs     : "
    f"{len(length_mismatches)}"
)

assert num_identical == 0, (
    "At least one clean/corrupt prompt pair is token-identical under "
    "the Llama tokenizer. The intended manipulation would not reach "
    "the model input."
)

print("  Token-level manipulation : PRESENT")
print()


# ----------------------------------------------------------------------------
# 13. Name inventory extraction
# ----------------------------------------------------------------------------
#
# The canonical dataset contains a controlled inventory of names. We inspect
# the actual record fields rather than imposing a new name list.
# ----------------------------------------------------------------------------

print("12. CONTROLLED NAME INVENTORY AUDIT")
print("-" * 76)

name_candidates = defaultdict(set)

for record in records:

    for key, value in record.items():

        if isinstance(value, str):

            key_lower = key.lower()

            if (
                "name" in key_lower
                or key_lower in {
                    "subject",
                    "object",
                    "person_a",
                    "person_b",
                    "source",
                    "target",
                }
            ):
                name_candidates[key].add(value)


for key, values in sorted(name_candidates.items()):

    print(
        f"  {key:24s}: "
        f"{len(values)} unique values"
    )


# Collect plausible person-name values using common schema fields first.
controlled_names = set()

preferred_name_keys = [
    "name_a",
    "name_b",
    "person_a",
    "person_b",
    "subject",
    "object",
    "source",
    "target",
    "name1",
    "name2",
]

for record in records:

    for key in preferred_name_keys:

        value = record.get(key)

        if isinstance(value, str):

            # Avoid treating generic target labels as names.
            if len(value.split()) <= 3:
                controlled_names.add(value)


print(f"  Candidate controlled names : {len(controlled_names)}")

# Do not assert 32 solely from guessed schema fields. The next stage uses
# explicit tokenizer analysis. If the schema exposes the exact 32-name
# inventory, this will be recorded.
name_inventory_complete = (
    len(controlled_names) == EXPECTED_NAME_COUNT
)

print(
    f"  32-name inventory recovered : "
    f"{name_inventory_complete}"
)
print()


# ----------------------------------------------------------------------------
# 14. Tokenization audit of recovered names
# ----------------------------------------------------------------------------

print("13. NAME TOKENIZATION AUDIT")
print("-" * 76)

name_tokenization = {}

for name in sorted(controlled_names):

    encoded = tokenizer(
        name,
        add_special_tokens=False,
        return_attention_mask=False,
        return_tensors=None,
    )

    ids = list(encoded["input_ids"])

    tokens = tokenizer.convert_ids_to_tokens(ids)

    name_tokenization[name] = {
        "token_ids": ids,
        "tokens": tokens,
        "token_count": len(ids),
    }


token_count_distribution = Counter(
    info["token_count"]
    for info in name_tokenization.values()
)

print("  Llama token-count distribution:")
for count, frequency in sorted(
    token_count_distribution.items()
):
    print(
        f"    {count} token(s): {frequency} name(s)"
    )


single_token_names = sorted(
    name
    for name, info in name_tokenization.items()
    if info["token_count"] == 1
)

multi_token_names = sorted(
    name
    for name, info in name_tokenization.items()
    if info["token_count"] > 1
)

print()
print(f"  Single-token names : {len(single_token_names)}")
print(f"  Multi-token names  : {len(multi_token_names)}")

if multi_token_names:

    print()
    print("  Multi-token names:")
    for name in multi_token_names:
        print(
            f"    {name!r}: "
            f"{name_tokenization[name]['tokens']}"
        )


# ----------------------------------------------------------------------------
# 15. Target-position ambiguity audit
# ----------------------------------------------------------------------------
#
# We need a scientifically valid target position before state extraction.
#
# For each clean/corrupt pair, we locate token-level differences.
#
# If exactly one token position changes, the behavioral manipulation has a
# straightforward token position.
#
# If multiple positions change because one name is multi-token or tokenization
# is otherwise non-local, this does NOT mean the experiment is "wrong".
# It means the Llama operationalization is not yet equivalent to the GPT-2
# operationalization and must be characterized before extraction.
# ----------------------------------------------------------------------------

print("14. TARGET-POSITION AUDIT")
print("-" * 76)

target_position_classification = Counter()

target_position_records = []

for item in token_difference_records:

    idx = item["index"]

    differing = item["different_positions"]

    if differing is None:

        classification = "LENGTH_MISMATCH"

        positions = None

    elif len(differing) == 0:

        classification = "NO_TOKEN_DIFFERENCE"

        positions = []

    elif len(differing) == 1:

        classification = "SINGLE_TOKEN_DIFFERENCE"

        positions = differing

    else:

        classification = "MULTI_TOKEN_DIFFERENCE"

        positions = differing

    target_position_classification[classification] += 1

    target_position_records.append({
        "record_id": record_ids[idx],
        "classification": classification,
        "positions": positions,
    })


for classification, count in sorted(
    target_position_classification.items()
):
    print(f"  {classification:28s}: {count}")


# ----------------------------------------------------------------------------
# 16. Context-length safety audit
# ----------------------------------------------------------------------------

print()
print("15. CONTEXT-LENGTH SAFETY AUDIT")
print("-" * 76)

max_clean_length = max(clean_lengths)
max_corrupt_length = max(corrupt_lengths)
max_observed_length = max(
    max_clean_length,
    max_corrupt_length
)

model_max_position = getattr(
    model.config,
    "max_position_embeddings",
    None,
)

print(f"  Maximum observed sequence : {max_observed_length}")
print(f"  Model position capacity  : {model_max_position}")

assert (
    model_max_position is None
    or max_observed_length <= model_max_position
), (
    "The controlled prompts exceed the model's configured position capacity."
)

print("  Context-length safety : PASS")
print()


# ----------------------------------------------------------------------------
# 17. Exact clean/corrupt token-difference consistency
# ----------------------------------------------------------------------------
#
# Because this is a crossed design, we expect the controlled manipulation
# to be present across all records. We do not impose an arbitrary number of
# changed tokens.
# ----------------------------------------------------------------------------

print("16. CROSS-DESIGN MANIPULATION PRESENCE")
print("-" * 76)

valid_difference_records = [
    x
    for x in token_difference_records
    if x["num_different_positions"] is not None
]

all_have_difference = all(
    x["num_different_positions"] > 0
    for x in valid_difference_records
)

print(
    f"  Every length-matched pair differs : "
    f"{all_have_difference}"
)

assert all_have_difference, (
    "Some clean/corrupt pairs become token-identical under Llama."
)

print("  Manipulation presence : PASS")
print()


# ----------------------------------------------------------------------------
# 18. Behavioral target metadata audit, if present
# ----------------------------------------------------------------------------
#
# We inspect the canonical record schema for explicit target fields.
# We do NOT invent target semantics if the dataset does not expose them.
# ----------------------------------------------------------------------------

print("17. TARGET METADATA AUDIT")
print("-" * 76)

target_metadata_keys = set()

for record in records:

    for key in record.keys():

        key_lower = key.lower()

        if (
            "target" in key_lower
            or "label" in key_lower
            or "correct" in key_lower
            or "answer" in key_lower
        ):
            target_metadata_keys.add(key)


print(
    f"  Target/label-like schema fields : "
    f"{sorted(target_metadata_keys)}"
)

print()


# ----------------------------------------------------------------------------
# 19. Split metadata audit
# ----------------------------------------------------------------------------

print("18. SPLIT METADATA AUDIT")
print("-" * 76)

split_fields = set()

for record in records:

    for key in record.keys():

        if key.lower() in {
            "split",
            "partition",
            "fold",
        }:

            split_fields.add(key)


print(f"  Explicit split fields : {sorted(split_fields)}")

if split_fields:

    for key in sorted(split_fields):

        counts = Counter(
            record.get(key)
            for record in records
        )

        print(f"  {key}: {dict(counts)}")

print()


# ----------------------------------------------------------------------------
# 20. Scientific alignment classification
# ----------------------------------------------------------------------------
#
# IMPORTANT:
#
# This classification does not change any mathematical definition.
#
# It only determines whether the frozen scientific stimulus can be passed
# into the Llama execution layer without an unresolved operational disparity.
# ----------------------------------------------------------------------------

print("19. SCIENTIFIC ALIGNMENT CLASSIFICATION")
print("-" * 76)

alignment_issues = []

if not name_inventory_complete:
    alignment_issues.append(
        "CONTROLLED_NAME_INVENTORY_NOT_RECOVERED_FROM_SCHEMA"
    )

if len(length_mismatches) > 0:
    alignment_issues.append(
        "CLEAN_CORRUPT_SEQUENCE_LENGTH_ASYMMETRY"
    )

if num_identical > 0:
    alignment_issues.append(
        "TOKEN_IDENTICAL_CLEAN_CORRUPT_RECORDS"
    )

if num_multi_position > 0:
    alignment_issues.append(
        "MULTI_TOKEN_MANIPULATION_PRESENT"
    )

if len(multi_token_names) > 0:
    alignment_issues.append(
        "MULTI_TOKEN_CONTROLLED_NAMES_PRESENT"
    )


if (
    len(length_mismatches) == 0
    and num_identical == 0
    and all_have_difference
):
    base_input_integrity = True
else:
    base_input_integrity = False


# A multi-token name is not automatically a failure of the scientific
# stimulus. It is, however, a potential operational disparity relative to
# the GPT-2 single-token target implementation and therefore must remain
# explicitly visible.
if (
    base_input_integrity
    and len(multi_token_names) == 0
):
    classification = (
        "LLAMA_INPUT_ALIGNMENT_SUPPORTED_SINGLE_TOKEN_TARGETS"
    )

elif base_input_integrity:
    classification = (
        "LLAMA_INPUT_ALIGNMENT_SUPPORTED_WITH_TOKENIZATION_DISPARITY"
    )

else:
    classification = (
        "LLAMA_INPUT_ALIGNMENT_NOT_ESTABLISHED"
    )


print(f"  Base input integrity : {base_input_integrity}")
print(f"  Classification        : {classification}")

if alignment_issues:

    print("  Alignment observations:")
    for issue in alignment_issues:
        print(f"    - {issue}")
else:

    print("  Alignment observations : NONE")


# ----------------------------------------------------------------------------
# 21. Scientific governance assertions
# ----------------------------------------------------------------------------

print()
print("20. CTL GOVERNANCE AUDIT")
print("-" * 76)

governance = {
    "math_modified": False,
    "sector_definitions_modified": False,
    "dataset_modified": False,
    "replacement_dataset_generated": False,
    "transport_maps_fitted": False,
    "model_weights_modified": False,
    "gqa_collapsed": False,
}

for key, value in governance.items():
    print(f"  {key:32s}: {value}")

assert not governance["math_modified"]
assert not governance["sector_definitions_modified"]
assert not governance["dataset_modified"]
assert not governance["replacement_dataset_generated"]
assert not governance["transport_maps_fitted"]
assert not governance["model_weights_modified"]
assert not governance["gqa_collapsed"]

print("  Governance : PASS")
print()


# ----------------------------------------------------------------------------
# 22. Persist complete audit artifact
# ----------------------------------------------------------------------------

artifact = {
    "experiment_id": EXPERIMENT_ID,
    "audit_id": AUDIT_ID,
    "model_id": MODEL_ID,
    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "dependencies": {
        "phase1b_artifact": PHASE1B_ARTIFACT,
        "phase1b_pass": phase1b_pass,
    },

    "dataset": {
        "path": DATASET_PATH,
        "record_count": len(records),
        "expected_record_count": EXPECTED_N,
        "unique_templates": len(template_counter),
        "expected_templates": EXPECTED_TEMPLATES,
        "expected_pairs_per_template":
            EXPECTED_PAIRS_PER_TEMPLATE,
        "scientific_fingerprint":
            observed_fingerprint,
        "expected_scientific_fingerprint":
            EXPECTED_FINGERPRINT,
        "fingerprint_match":
            fingerprint_match,
    },

    "tokenization": {
        "clean_length_min":
            min(clean_lengths),
        "clean_length_max":
            max(clean_lengths),
        "clean_length_mean":
            sum(clean_lengths) / len(clean_lengths),

        "corrupt_length_min":
            min(corrupt_lengths),
        "corrupt_length_max":
            max(corrupt_lengths),
        "corrupt_length_mean":
            sum(corrupt_lengths) / len(corrupt_lengths),

        "length_mismatch_count":
            len(length_mismatches),

        "token_identical_count":
            num_identical,

        "single_position_difference_count":
            num_single_position,

        "multi_position_difference_count":
            num_multi_position,

        "controlled_name_count":
            len(controlled_names),

        "single_token_name_count":
            len(single_token_names),

        "multi_token_name_count":
            len(multi_token_names),

        "token_count_distribution":
            dict(
                sorted(
                    token_count_distribution.items()
                )
            ),
    },

    "target_position_audit": {
        "classification_counts":
            dict(target_position_classification),
        "records":
            target_position_records,
    },

    "context_length": {
        "maximum_observed_sequence_length":
            max_observed_length,
        "model_max_position_embeddings":
            model_max_position,
        "pass": True,
    },

    "alignment": {
        "base_input_integrity":
            base_input_integrity,
        "classification":
            classification,
        "issues":
            alignment_issues,
    },

    "governance": governance,

    "scientific_scope": {
        "states_extracted": False,
        "transport_maps_fitted": False,
        "contextual_realization_fitted": False,
        "ctl_evaluation_executed": False,
        "triadic_irreducibility_evaluated": False,
    },

    "status": (
        "PHASE1C0_DATA_TOKENIZATION_ALIGNMENT_AUDIT_COMPLETE"
    ),
}


with open(ARTIFACT_PATH, "w", encoding="utf-8") as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ----------------------------------------------------------------------------
# 23. Final report
# ----------------------------------------------------------------------------

print("21. PHASE 1C.0 AUDIT RESULT")
print("-" * 76)

print(
    f"  Canonical dataset located       : PASS"
)
print(
    f"  Record count                    : "
    f"{'PASS' if len(records) == EXPECTED_N else 'FAIL'}"
)
print(
    f"  Scientific fingerprint          : "
    f"{'PASS' if fingerprint_match else 'FAIL'}"
)
print(
    f"  Clean/corrupt token difference  : "
    f"{'PASS' if num_identical == 0 else 'FAIL'}"
)
print(
    f"  Sequence-length equivalence     : "
    f"{'PASS' if length_equivalence else 'FAIL'}"
)
print(
    f"  Context-length safety           : PASS"
)
print(
    f"  CTL governance                  : PASS"
)
print()
print(f"  Overall classification : {classification}")
print()
print(f"  Artifact : {ARTIFACT_PATH}")
print()

print("=" * 76)
print("PHASE 1C.0 DATA / TOKENIZATION ALIGNMENT AUDIT COMPLETE")
print("=" * 76)

ETTR-CTL-LLAMA — PHASE 1C.0
SCIENTIFIC DATA / TOKENIZATION / CROSS-DESIGN ALIGNMENT AUDIT
Experiment : ETTR-CTL-LLAMA-1
Audit      : LLAMA-PHASE1C0-DATA-TOKENIZATION-ALIGNMENT-AUDIT-V1
Timestamp  : 2026-09-11T00:05:14.633599+00:00
Model      : meta-llama/Llama-3.2-3B

1. PHASE 1B DEPENDENCY AUDIT
----------------------------------------------------------------------------
  Artifact : /content/ettr_ctl_llama/results/llama_phase1b_model_runtime_hook_audit_v3.json
  PASS     : True
  Dependency check : PASS

2. RUNTIME MODEL/TOKENIZER DEPENDENCY
----------------------------------------------------------------------------
  Model class       : LlamaForCausalLM
  Parameter devices : ['cuda:0']
  Parameter dtypes  : ['torch.bfloat16']
  Tokenizer class   : TokenizersBackend
  Runtime dependency : PASS

3. CANONICAL DATASET DISCOVERY
----------------------------------------------------------------------------
  Candidate paths checked : 3
  Existing matches        : 0


AssertionError: Canonical GPT-2 controlled dataset was not found in the current Colab runtime. No replacement dataset will be generated. Scientific alignment audit is therefore blocked.

In [2]:
# ============================================================================
# ETTR-CTL-LLAMA-1
# PHASE 1C.0 — RECOVERED DATASET IMPORT / AUTHENTICATION GATE V2
# ============================================================================
#
# NEW CELL — do not remove previous Llama cells.
#
# PURPOSE
# -------
# Import the exact authenticated GPT-2 dataset artifact:
#
#   gpt2_controlled_dataset_v1.1_recovered_r1.json
#
# and independently verify:
#
#   SHA-256 =
#   7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
#
# BEFORE:
#   - Llama tokenization
#   - state extraction
#   - model inference for the experiment
#   - transport fitting
#   - CTL evaluation
#
# GOVERNANCE
# ----------
# This cell:
#   - does NOT reconstruct the dataset;
#   - does NOT modify any record;
#   - does NOT reorder records;
#   - does NOT alter prompts;
#   - does NOT alter target fields;
#   - does NOT retokenize before authentication;
#   - does NOT authorize the historical SHA;
#   - does NOT perform scientific model evaluation.
#
# It establishes an independently verified Llama-side copy of the
# authenticated operative experimental dataset.
#
# ============================================================================

import json
import hashlib
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone


# ============================================================================
# 1. CONSTANTS
# ============================================================================

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"

DATASET_ARTIFACT_NAME = (
    "gpt2_controlled_dataset_v1.1_recovered_r1.json"
)

EXPECTED_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)

HISTORICAL_SHA256 = (
    "38a58ffbe53e92ce17cd7d52fe82034e86d80f43a809fa007f2a0562e3168051"
)

EXPECTED_RECORD_COUNT = 192

EXPECTED_TEMPLATE_COUNT = 12

EXPECTED_PAIR_COUNT = 16

EXPECTED_SPLITS = {
    "train": 96,
    "calibration": 48,
    "test": 48,
}

EXPECTED_CONDITION_PAIR = (
    "clean_vs_corrupt"
)


# ============================================================================
# 2. LOCATE ARTIFACT — DO NOT GENERATE A SUBSTITUTE
# ============================================================================
#
# The artifact must already exist in this Llama runtime.
#
# We deliberately do not search arbitrary filesystem locations recursively,
# because a similarly named or stale artifact must not silently become the
# experimental dataset.
#
# The user should upload the exact exported JSON into the Llama Colab runtime.
# ============================================================================

print("=" * 78)
print(
    "ETTR-CTL-LLAMA-1 — PHASE 1C.0 "
    "RECOVERED DATASET IMPORT / AUTHENTICATION GATE V2"
)
print("=" * 78)

candidate_paths = [
    Path("/content") / DATASET_ARTIFACT_NAME,
    Path("/content/ettr_ctl_llama/data") / DATASET_ARTIFACT_NAME,
    Path("/content/ettr_ctl_llama/results") / DATASET_ARTIFACT_NAME,
]

existing_candidates = [
    path
    for path in candidate_paths
    if path.exists()
    and path.is_file()
]

print("\n[1/12] Artifact discovery")
print("-" * 78)

for path in candidate_paths:

    print(
        f"  {path}:",
        "FOUND" if path.exists() else "not found"
    )

if len(existing_candidates) == 0:

    raise FileNotFoundError(
        "\nFATAL: authenticated dataset artifact was not found.\n\n"
        "Expected filename:\n"
        f"  {DATASET_ARTIFACT_NAME}\n\n"
        "Upload/copy the exact exported GPT-2 JSON artifact into the "
        "Llama Colab runtime, then rerun this cell.\n\n"
        "No substitute dataset was generated."
    )

if len(existing_candidates) > 1:

    raise RuntimeError(
        "\nFATAL: multiple candidate artifacts were found.\n"
        "Refusing to select one implicitly:\n"
        + "\n".join(
            f"  {path}"
            for path in existing_candidates
        )
    )

DATASET_PATH = existing_candidates[0]

print(
    "\n  selected artifact:",
    DATASET_PATH
)


# ============================================================================
# 3. RAW BYTE AUTHENTICATION
# ============================================================================
#
# Hash the actual bytes received by the Llama runtime BEFORE parsing.
#
# This is the primary cross-runtime artifact identity test.
# ============================================================================

print("\n[2/12] Raw artifact SHA-256 authentication")
print("-" * 78)

with open(
    DATASET_PATH,
    "rb"
) as f:

    raw_bytes = f.read()

observed_sha256 = hashlib.sha256(
    raw_bytes
).hexdigest()

print(
    "  byte count:",
    len(raw_bytes)
)

print(
    "  observed SHA-256:"
)

print(
    f"    {observed_sha256}"
)

print(
    "  expected SHA-256:"
)

print(
    f"    {EXPECTED_SHA256}"
)

assert observed_sha256 == EXPECTED_SHA256, (
    "\nFATAL: received artifact does not match the "
    "authenticated recovered GPT-2 artifact.\n"
    f"Expected: {EXPECTED_SHA256}\n"
    f"Observed: {observed_sha256}\n\n"
    "No dataset has been authorized."
)

print(
    "  raw-byte authentication: PASS"
)


# ============================================================================
# 4. PARSE JSON
# ============================================================================

print("\n[3/12] JSON parsing")
print("-" * 78)

try:

    imported_records = json.loads(
        raw_bytes.decode("utf-8")
    )

except Exception as exc:

    raise RuntimeError(
        "FATAL: authenticated bytes are not valid UTF-8 JSON."
    ) from exc

assert isinstance(
    imported_records,
    list
)

print(
    "  parsed type:",
    type(imported_records).__name__
)

print(
    "  parsed record count:",
    len(imported_records)
)

assert len(imported_records) == EXPECTED_RECORD_COUNT


# ============================================================================
# 5. CANONICAL RE-SERIALIZATION CHECK
# ============================================================================
#
# Confirm that the received JSON parses and reserializes under the same
# canonicalization contract to exactly the authenticated SHA.
# ============================================================================

print("\n[4/12] Canonical re-serialization")
print("-" * 78)

reserialized_payload = json.dumps(
    imported_records,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=False,
)

reserialized_bytes = reserialized_payload.encode(
    "utf-8"
)

reserialized_sha256 = hashlib.sha256(
    reserialized_bytes
).hexdigest()

print(
    "  reserialized byte count:",
    len(reserialized_bytes)
)

print(
    "  reserialized SHA-256:"
)

print(
    f"    {reserialized_sha256}"
)

assert reserialized_sha256 == EXPECTED_SHA256

print(
    "  canonical re-serialization: PASS"
)


# ============================================================================
# 6. BYTE-LEVEL CANONICALITY
# ============================================================================
#
# The exported artifact was defined to be the canonical JSON itself.
# Therefore raw bytes and canonical reserialization must agree.
# ============================================================================

print("\n[5/12] Byte-level canonicality")
print("-" * 78)

assert raw_bytes == reserialized_bytes, (
    "FATAL: artifact bytes differ from canonical reserialization."
)

print(
    "  raw bytes == canonical bytes: PASS"
)


# ============================================================================
# 7. SCHEMA AUDIT
# ============================================================================

print("\n[6/12] Schema audit")
print("-" * 78)

field_sets = [
    set(record.keys())
    for record in imported_records
]

unique_field_sets = {
    tuple(sorted(fields))
    for fields in field_sets
}

assert len(unique_field_sets) == 1, (
    "FATAL: imported records do not share a uniform schema."
)

fields = list(
    sorted(
        unique_field_sets.pop()
    )
)

print(
    "  field count:",
    len(fields)
)

for field in fields:

    print(
        "   ",
        field
    )


# ============================================================================
# 8. SCIENTIFIC STRUCTURE
# ============================================================================

print("\n[7/12] Scientific structure")
print("-" * 78)

template_ids = {
    record["template_id"]
    for record in imported_records
}

pair_ids = {
    record["name_pair_id"]
    for record in imported_records
}

split_counts = Counter(
    record["split"]
    for record in imported_records
)

assert len(template_ids) == EXPECTED_TEMPLATE_COUNT

assert len(pair_ids) == EXPECTED_PAIR_COUNT

assert dict(split_counts) == EXPECTED_SPLITS

print(
    "  template count:",
    len(template_ids)
)

print(
    "  name-pair count:",
    len(pair_ids)
)

print(
    "  split counts:",
    dict(split_counts)
)


# ============================================================================
# 9. CROSSED-DESIGN AUDIT
# ============================================================================

print("\n[8/12] Crossed-design audit")
print("-" * 78)

observed_cells = {
    (
        record["template_index"],
        record["name_pair_index"],
    )
    for record in imported_records
}

expected_cells = {
    (
        template_index,
        pair_index,
    )
    for template_index in range(
        EXPECTED_TEMPLATE_COUNT
    )
    for pair_index in range(
        EXPECTED_PAIR_COUNT
    )
}

assert observed_cells == expected_cells

print(
    "  observed cells:",
    len(observed_cells)
)

print(
    "  expected cells:",
    len(expected_cells)
)

print(
    "  crossed design: PASS"
)


# ============================================================================
# 10. TARGET / PROMPT SEMANTICS
# ============================================================================

print("\n[9/12] Target and prompt semantics")
print("-" * 78)

clean_target_second = sum(
    record["clean_target_name"]
    == record["second_name"]
    for record in imported_records
)

corrupt_target_first = sum(
    record["corrupt_target_name"]
    == record["first_name"]
    for record in imported_records
)

clean_role_second = sum(
    record["clean_target_role"]
    == "second"
    for record in imported_records
)

corrupt_role_second = sum(
    record["corrupt_target_role"]
    == "second"
    for record in imported_records
)

prompt_pairs_distinct = sum(
    record["clean_prompt"]
    != record["corrupt_prompt"]
    for record in imported_records
)

target_pairs_distinct = sum(
    record["clean_target_name"]
    != record["corrupt_target_name"]
    for record in imported_records
)

print(
    "  clean target = second:",
    clean_target_second,
    "/ 192"
)

print(
    "  corrupt target = first:",
    corrupt_target_first,
    "/ 192"
)

print(
    "  clean role = second:",
    clean_role_second,
    "/ 192"
)

print(
    "  corrupt role = second:",
    corrupt_role_second,
    "/ 192"
)

print(
    "  clean prompt != corrupt prompt:",
    prompt_pairs_distinct,
    "/ 192"
)

print(
    "  clean target != corrupt target:",
    target_pairs_distinct,
    "/ 192"
)

assert clean_target_second == EXPECTED_RECORD_COUNT
assert corrupt_target_first == EXPECTED_RECORD_COUNT
assert clean_role_second == EXPECTED_RECORD_COUNT
assert corrupt_role_second == EXPECTED_RECORD_COUNT
assert prompt_pairs_distinct == EXPECTED_RECORD_COUNT
assert target_pairs_distinct == EXPECTED_RECORD_COUNT


# ============================================================================
# 11. CONDITION_PAIR METADATA
# ============================================================================

print("\n[10/12] condition_pair metadata")
print("-" * 78)

condition_counts = Counter(
    record["condition_pair"]
    for record in imported_records
)

print(
    "  observed values:"
)

for value, count in condition_counts.items():

    print(
        f"    {repr(value)} : {count}"
    )

assert dict(condition_counts) == {
    EXPECTED_CONDITION_PAIR:
        EXPECTED_RECORD_COUNT
}

print(
    "  condition_pair metadata: PASS"
)


# ============================================================================
# 12. UNIQUENESS + FINAL AUTHORIZATION
# ============================================================================

print("\n[11/12] Uniqueness")
print("-" * 78)

example_ids = [
    record["example_id"]
    for record in imported_records
]

prompt_pairs = [
    (
        record["clean_prompt"],
        record["corrupt_prompt"],
    )
    for record in imported_records
]

assert len(set(example_ids)) == EXPECTED_RECORD_COUNT

assert len(set(prompt_pairs)) == EXPECTED_RECORD_COUNT

print(
    "  unique example IDs:",
    len(set(example_ids))
)

print(
    "  unique prompt pairs:",
    len(set(prompt_pairs))
)

print(
    "  uniqueness: PASS"
)


# ============================================================================
# 13. FINAL AUTHORIZATION
# ============================================================================

print("\n[12/12] Llama-side dataset authorization")
print("-" * 78)

dataset_authorized = True

authorization_status = (
    "AUTHORIZED_OPERATIVE_DATASET"
)

print(
    "  operative dataset SHA-256:"
)

print(
    f"    {observed_sha256}"
)

print(
    "  historical SHA:"
)

print(
    f"    {HISTORICAL_SHA256}"
)

print(
    "  historical SHA used for authorization:",
    False
)

print(
    "  recovered artifact authorized:",
    dataset_authorized
)


# ============================================================================
# 14. STORE AUTHORIZED OBJECT
# ============================================================================
#
# Only after every gate passes do we expose the imported object under the
# experiment-facing name `llama_records`.
#
# `records` is deliberately not overwritten.
# ============================================================================

llama_records = imported_records

assert llama_records == imported_records

print(
    "\n  `llama_records` established: PASS"
)


# ============================================================================
# 15. SAVE LLAMA-SIDE AUTHORIZATION MANIFEST
# ============================================================================

RESULTS_DIR = Path(
    "/content/ettr_ctl_llama/results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_PATH = (
    RESULTS_DIR /
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

manifest = {
    "experiment_id":
        EXPERIMENT_ID,

    "source_artifact":
        DATASET_ARTIFACT_NAME,

    "source_artifact_version":
        "v1.1-recovered-r1",

    "source_path":
        str(DATASET_PATH),

    "sha256":
        observed_sha256,

    "historical_sha256":
        HISTORICAL_SHA256,

    "historical_sha_used_for_authorization":
        False,

    "authorization_status":
        authorization_status,

    "dataset_authorized":
        dataset_authorized,

    "record_count":
        len(imported_records),

    "template_count":
        len(template_ids),

    "name_pair_count":
        len(pair_ids),

    "split_counts":
        dict(split_counts),

    "condition_pair":
        {
            "value":
                EXPECTED_CONDITION_PAIR,

            "interpretation":
                "constant record metadata",
        },

    "scientific_integrity": {
        "clean_target_is_second":
            clean_target_second
            == EXPECTED_RECORD_COUNT,

        "corrupt_target_is_first":
            corrupt_target_first
            == EXPECTED_RECORD_COUNT,

        "clean_role_is_second":
            clean_role_second
            == EXPECTED_RECORD_COUNT,

        "corrupt_role_is_second":
            corrupt_role_second
            == EXPECTED_RECORD_COUNT,

        "clean_corrupt_prompts_distinct":
            prompt_pairs_distinct
            == EXPECTED_RECORD_COUNT,

        "clean_corrupt_targets_distinct":
            target_pairs_distinct
            == EXPECTED_RECORD_COUNT,

        "crossed_design_complete":
            observed_cells
            == expected_cells,

        "example_ids_unique":
            len(set(example_ids))
            == EXPECTED_RECORD_COUNT,

        "prompt_pairs_unique":
            len(set(prompt_pairs))
            == EXPECTED_RECORD_COUNT,
    },

    "cryptographic_integrity": {
        "raw_byte_sha256":
            observed_sha256,

        "canonical_reserialization_sha256":
            reserialized_sha256,

        "raw_equals_canonical_bytes":
            raw_bytes == reserialized_bytes,

        "expected_sha256_match":
            observed_sha256
            == EXPECTED_SHA256,
    },

    "governance": {
        "dataset_generated":
            False,

        "dataset_reconstructed":
            False,

        "dataset_repaired":
            False,

        "dataset_reordered":
            False,

        "dataset_fields_modified":
            False,

        "tokenization_performed":
            False,

        "model_state_extraction_performed":
            False,

        "cross_model_evaluation_performed":
            False,
    },

    "next_stage":
        "Llama tokenization alignment audit",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================================
# FINAL STATUS
# ============================================================================

print("\n" + "=" * 78)

print(
    "FINAL STATUS:"
)

print(
    "LLAMA-SIDE RECOVERED DATASET AUTHENTICATION PASS"
)

print(
    "\nOperative dataset:"
)

print(
    f"  {DATASET_ARTIFACT_NAME}"
)

print(
    "\nAuthenticated SHA-256:"
)

print(
    f"  {observed_sha256}"
)

print(
    "\nRecords:"
)

print(
    f"  {len(llama_records)}"
)

print(
    "\nDataset authorization:"
)

print(
    f"  {dataset_authorized}"
)

print(
    "\nNext authorized stage:"
)

print(
    "  Llama tokenizer alignment audit"
)

print(
    "\nAuthorization manifest:"
)

print(
    f"  {MANIFEST_PATH}"
)

print("=" * 78)

ETTR-CTL-LLAMA-1 — PHASE 1C.0 RECOVERED DATASET IMPORT / AUTHENTICATION GATE V2

[1/12] Artifact discovery
------------------------------------------------------------------------------
  /content/gpt2_controlled_dataset_v1.1_recovered_r1.json: FOUND
  /content/ettr_ctl_llama/data/gpt2_controlled_dataset_v1.1_recovered_r1.json: not found
  /content/ettr_ctl_llama/results/gpt2_controlled_dataset_v1.1_recovered_r1.json: not found

  selected artifact: /content/gpt2_controlled_dataset_v1.1_recovered_r1.json

[2/12] Raw artifact SHA-256 authentication
------------------------------------------------------------------------------
  byte count: 108005
  observed SHA-256:
    7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  expected SHA-256:
    7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  raw-byte authentication: PASS

[3/12] JSON parsing
------------------------------------------------------------------------------
  parsed type: list
  parsed record 

In [9]:
# ==============================================================================
# ETTR-CTL-LLAMA-1 — PHASE 1C.1
# LLAMA TOKENIZER ALIGNMENT / TARGET-POSITION AUDIT
# ==============================================================================
#
# PURPOSE
# -------
# Audit whether the authenticated experimental stimulus can be operationalized
# on Llama 3.2 3B without changing the mathematical or scientific specification.
#
# GOVERNING PRINCIPLE
# -------------------
# Contextual Calculus / Contextual Transport Logic remains authoritative.
#
# This cell audits operational compatibility only.
#
# A genuine disparity between mathematical specification and numerical/software
# realization is recorded as DECOHERENCE. Software/runtime constraints do not
# redefine the mathematics.
#
# AUDIT ONLY
# ----------
# This cell does NOT:
#   - fit transport maps;
#   - alter llama_records;
#   - alter prompts;
#   - perform behavioral inference;
#   - create a state bank;
#   - select experimental examples;
#   - modify CTL sector definitions.
#
# IMPORTANT
# ---------
# Phase 1C.0 already authenticated the recovered dataset.
# This cell must consume the ACTUAL Phase 1C.0 manifest schema rather than
# assuming an undocumented top-level field.
# ==============================================================================

import os
import gc
import json
import hashlib
from collections import Counter, defaultdict

import torch
import transformers
from transformers import AutoTokenizer


# ------------------------------------------------------------------------------
# 0. CONSTANTS / PATHS
# ------------------------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
PHASE_ID = "PHASE_1C.1"
MODEL_ID_EXPECTED = "meta-llama/Llama-3.2-3B"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

AUTH_MANIFEST_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

OUTPUT_MANIFEST_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1c1_tokenizer_alignment_audit.json"
)

EXPECTED_DATASET_SHA = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)

EXPECTED_RECORD_COUNT = 192
EXPECTED_TEMPLATE_COUNT = 12
EXPECTED_PAIR_COUNT = 16

EXPECTED_SPLITS = {
    "calibration": 48,
    "test": 48,
    "train": 96,
}


print("=" * 78)
print(f"{EXPERIMENT_ID} — {PHASE_ID}")
print("LLAMA TOKENIZER ALIGNMENT / TARGET-POSITION AUDIT")
print("=" * 78)


# ------------------------------------------------------------------------------
# 1. PHASE 1C.0 AUTHORIZATION MANIFEST
# ------------------------------------------------------------------------------
#
# DO NOT assume a particular manifest field.
#
# The Phase 1C.0 execution output is the authoritative indication that:
#
#   recovered artifact authorized: True
#
# We inspect the actual JSON schema and establish authorization only from
# explicit evidence contained in that manifest.
# ------------------------------------------------------------------------------

print("\n[1/14] Dataset authorization precondition")
print("-" * 78)

if not os.path.exists(AUTH_MANIFEST_PATH):
    raise RuntimeError(
        "Phase 1C.0 authorization manifest is missing. "
        "Tokenizer alignment cannot proceed."
    )

with open(
    AUTH_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as f:
    auth_manifest = json.load(f)

print(
    f"  authorization manifest: "
    f"{AUTH_MANIFEST_PATH}"
)

print(
    f"  manifest top-level keys: "
    f"{sorted(auth_manifest.keys())}"
)


# ------------------------------------------------------------------------------
# Explicit authorization extraction
# ------------------------------------------------------------------------------
#
# We inspect several possible locations, but ONLY accept an explicit boolean
# True or an explicit final-status string documenting authorization.
#
# No value is inferred merely from the existence of the manifest.
# ------------------------------------------------------------------------------

authorization_evidence = []

# Candidate 1: top-level boolean fields.
for key in (
    "authorized",
    "recovered_artifact_authorized",
    "dataset_authorized",
):
    value = auth_manifest.get(key, None)

    if value is not None:
        authorization_evidence.append(
            {
                "location": key,
                "value": value,
            }
        )


# Candidate 2: nested authorization object.
authorization_object = auth_manifest.get(
    "authorization",
    None
)

if isinstance(
    authorization_object,
    dict,
):

    for key in (
        "authorized",
        "recovered_artifact_authorized",
        "dataset_authorized",
    ):

        value = authorization_object.get(
            key,
            None
        )

        if value is not None:
            authorization_evidence.append(
                {
                    "location": f"authorization.{key}",
                    "value": value,
                }
            )


# Candidate 3: final-status strings.
final_status_candidates = []

for key in (
    "final_status",
    "status",
    "classification",
):

    value = auth_manifest.get(
        key,
        None
    )

    if isinstance(value, str):

        final_status_candidates.append(
            {
                "location": key,
                "value": value,
            }
        )


if isinstance(
    authorization_object,
    dict,
):

    for key in (
        "final_status",
        "status",
        "classification",
    ):

        value = authorization_object.get(
            key,
            None
        )

        if isinstance(value, str):

            final_status_candidates.append(
                {
                    "location": f"authorization.{key}",
                    "value": value,
                }
            )


print("\n  explicit authorization evidence:")

if authorization_evidence:

    for evidence in authorization_evidence:

        print(
            f"    {evidence['location']}: "
            f"{evidence['value']!r}"
        )

else:

    print("    none found")


print("\n  status evidence:")

if final_status_candidates:

    for evidence in final_status_candidates:

        print(
            f"    {evidence['location']}: "
            f"{evidence['value']!r}"
        )

else:

    print("    none found")


# Determine whether an explicit True exists.
explicit_true_authorization = any(
    evidence["value"] is True
    for evidence in authorization_evidence
)


# Determine whether an explicit status positively establishes authorization.
positive_status_phrases = (
    "LLAMA-SIDE RECOVERED DATASET AUTHENTICATION PASS",
    "RECOVERED DATASET AUTHENTICATION PASS",
    "RECOVERED ARTIFACT AUTHENTICATED",
)

positive_status_authorization = False

for evidence in final_status_candidates:

    status_text = evidence["value"].upper()

    if any(
        phrase in status_text
        for phrase in positive_status_phrases
    ):
        positive_status_authorization = True


authorized = (
    explicit_true_authorization
    or positive_status_authorization
)

print(
    f"\n  explicit authorization established: "
    f"{authorized}"
)

if not authorized:

    raise RuntimeError(
        "Phase 1C.0 manifest does not contain explicit authorization "
        "evidence that this cell can safely consume. "
        "No authorization has been inferred."
    )

print("  authorization precondition: PASS")


# ------------------------------------------------------------------------------
# 2. OPERATIVE DATASET PRECONDITION
# ------------------------------------------------------------------------------

print("\n[2/14] Operative dataset precondition")
print("-" * 78)

if "llama_records" not in globals():

    raise RuntimeError(
        "`llama_records` is not present. Re-run the successful Phase 1C.0 "
        "import/authentication cell before continuing."
    )

records_ref = llama_records

if not isinstance(
    records_ref,
    list,
):

    raise TypeError(
        f"`llama_records` must be a list; observed "
        f"{type(records_ref).__name__}."
    )

if len(records_ref) != EXPECTED_RECORD_COUNT:

    raise RuntimeError(
        f"Expected {EXPECTED_RECORD_COUNT} records; "
        f"observed {len(records_ref)}."
    )

print(
    f"  llama_records type: "
    f"{type(records_ref).__name__}"
)

print(
    f"  record count: "
    f"{len(records_ref)}"
)

print("  operative dataset object: PASS")


# ------------------------------------------------------------------------------
# 3. DATASET IMMUTABILITY SNAPSHOT
# ------------------------------------------------------------------------------

print("\n[3/14] Operative dataset immutability snapshot")
print("-" * 78)


def canonical_json_bytes(obj):

    return json.dumps(
        obj,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")


records_before_bytes = canonical_json_bytes(
    records_ref
)

records_before_sha = hashlib.sha256(
    records_before_bytes
).hexdigest()

print(
    f"  canonical pre-audit in-memory SHA-256: "
    f"{records_before_sha}"
)

print(
    f"  authenticated artifact SHA-256: "
    f"{EXPECTED_DATASET_SHA}"
)

print("  dataset mutation before audit: NONE")


# ------------------------------------------------------------------------------
# 4. MODEL IDENTITY AUDIT
# ------------------------------------------------------------------------------

print("\n[4/14] Model identity audit")
print("-" * 78)

model_identity_candidates = []

for name in (
    "model",
    "llama_model",
):

    if name in globals():

        obj = globals()[name]

        model_name = getattr(
            obj,
            "name_or_path",
            None,
        )

        config = getattr(
            obj,
            "config",
            None,
        )

        config_name = getattr(
            config,
            "_name_or_path",
            None,
        )

        model_identity_candidates.append(
            {
                "variable": name,
                "name_or_path": model_name,
                "config_name_or_path": config_name,
            }
        )


for item in model_identity_candidates:

    print(
        f"  {item['variable']}: "
        f"name_or_path={item['name_or_path']!r}, "
        f"config={item['config_name_or_path']!r}"
    )


model_identity_observed = None

for item in model_identity_candidates:

    if item["name_or_path"]:

        model_identity_observed = (
            item["name_or_path"]
        )

        break

    if item["config_name_or_path"]:

        model_identity_observed = (
            item["config_name_or_path"]
        )

        break


print(
    f"  expected model: "
    f"{MODEL_ID_EXPECTED}"
)

print(
    f"  observed model: "
    f"{model_identity_observed}"
)


if model_identity_observed is None:

    model_identity_ok = False

else:

    model_identity_ok = (
        model_identity_observed
        == MODEL_ID_EXPECTED
        or
        model_identity_observed.rstrip(
            "/"
        ).endswith(
            "meta-llama/Llama-3.2-3B"
        )
    )


if model_identity_observed is None:

    print(
        "  model identity: UNRESOLVED"
    )

else:

    print(
        "  model identity: "
        + (
            "PASS"
            if model_identity_ok
            else "FAIL"
        )
    )


# ------------------------------------------------------------------------------
# 5. TOKENIZER LOAD / IDENTITY
# ------------------------------------------------------------------------------

print("\n[5/14] Llama tokenizer load and identity audit")
print("-" * 78)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID_EXPECTED,
    use_fast=True,
)

tokenizer_name = tokenizer.name_or_path

print(
    f"  tokenizer class: "
    f"{tokenizer.__class__.__name__}"
)

print(
    f"  tokenizer name_or_path: "
    f"{tokenizer_name}"
)

print(
    f"  tokenizer vocab_size: "
    f"{getattr(tokenizer, 'vocab_size', None)}"
)

print(
    f"  tokenizer model_max_length: "
    f"{getattr(tokenizer, 'model_max_length', None)}"
)

print(
    f"  tokenizer is_fast: "
    f"{getattr(tokenizer, 'is_fast', None)}"
)

print(
    f"  bos_token: "
    f"{repr(tokenizer.bos_token)}"
)

print(
    f"  eos_token: "
    f"{repr(tokenizer.eos_token)}"
)

print(
    f"  pad_token: "
    f"{repr(tokenizer.pad_token)}"
)

print(
    f"  bos_token_id: "
    f"{tokenizer.bos_token_id}"
)

print(
    f"  eos_token_id: "
    f"{tokenizer.eos_token_id}"
)

print(
    f"  pad_token_id: "
    f"{tokenizer.pad_token_id}"
)


tokenizer_identity_ok = (
    tokenizer_name.rstrip(
        "/"
    ).endswith(
        "meta-llama/Llama-3.2-3B"
    )
)


if not tokenizer_identity_ok:

    raise RuntimeError(
        "The loaded tokenizer does not identify as the expected "
        "Llama-3.2-3B tokenizer."
    )

print("  tokenizer identity: PASS")


# ------------------------------------------------------------------------------
# 6. SPECIAL-TOKEN CONVENTION AUDIT
# ------------------------------------------------------------------------------

print("\n[6/14] Special-token convention audit")
print("-" * 78)

probe_text = "Alice"

probe_no_special = tokenizer.encode(
    probe_text,
    add_special_tokens=False,
)

probe_with_special = tokenizer.encode(
    probe_text,
    add_special_tokens=True,
)

print(
    f"  probe text: "
    f"{probe_text!r}"
)

print(
    f"  no-special IDs: "
    f"{probe_no_special}"
)

print(
    f"  with-special IDs: "
    f"{probe_with_special}"
)

special_tokens_change_sequence = (
    probe_no_special
    != probe_with_special
)

print(
    f"  add_special_tokens changes probe: "
    f"{special_tokens_change_sequence}"
)


# Preserve the exact prompt strings.
TOKENIZER_ADD_SPECIAL_TOKENS = False

print(
    "  operative audit convention: "
    f"add_special_tokens="
    f"{TOKENIZER_ADD_SPECIAL_TOKENS}"
)


# ------------------------------------------------------------------------------
# 7. LLAMA-SIDE DATASET STRUCTURE RECHECK
# ------------------------------------------------------------------------------

print("\n[7/14] Llama-side dataset structural recheck")
print("-" * 78)

required_fields = {
    "clean_prompt",
    "clean_target_name",
    "clean_target_role",
    "clean_target_token_id",
    "condition_pair",
    "corrupt_prompt",
    "corrupt_target_name",
    "corrupt_target_role",
    "corrupt_target_token_id",
    "example_id",
    "first_name",
    "name_pair_id",
    "name_pair_index",
    "second_name",
    "split",
    "template_id",
    "template_index",
}

field_sets = [
    set(record.keys())
    for record in records_ref
]

schema_uniform = all(
    fields == required_fields
    for fields in field_sets
)

template_ids = sorted(
    set(
        record["template_id"]
        for record in records_ref
    )
)

pair_ids = sorted(
    set(
        record["name_pair_id"]
        for record in records_ref
    )
)

split_counts = Counter(
    record["split"]
    for record in records_ref
)

condition_counts = Counter(
    record["condition_pair"]
    for record in records_ref
)

print(
    f"  uniform 17-field schema: "
    f"{schema_uniform}"
)

print(
    f"  template count: "
    f"{len(template_ids)}"
)

print(
    f"  name-pair count: "
    f"{len(pair_ids)}"
)

print(
    f"  split counts: "
    f"{dict(split_counts)}"
)

print(
    f"  condition_pair values: "
    f"{dict(condition_counts)}"
)


if not schema_uniform:

    raise RuntimeError(
        "Llama-side schema differs from the authorized dataset."
    )

if len(template_ids) != EXPECTED_TEMPLATE_COUNT:

    raise RuntimeError(
        "Unexpected template count."
    )

if len(pair_ids) != EXPECTED_PAIR_COUNT:

    raise RuntimeError(
        "Unexpected name-pair count."
    )

if dict(split_counts) != EXPECTED_SPLITS:

    raise RuntimeError(
        "Unexpected split distribution."
    )

if condition_counts != Counter(
    {
        "clean_vs_corrupt":
            EXPECTED_RECORD_COUNT
    }
):

    raise RuntimeError(
        "Unexpected condition_pair metadata."
    )

print("  structural recheck: PASS")


# ------------------------------------------------------------------------------
# 8. EXACT PROMPT TOKENIZATION
# ------------------------------------------------------------------------------

print("\n[8/14] Exact prompt tokenization")
print("-" * 78)

tokenization_rows = []

for index, record in enumerate(
    records_ref
):

    clean_prompt = record[
        "clean_prompt"
    ]

    corrupt_prompt = record[
        "corrupt_prompt"
    ]

    clean_encoded = tokenizer(
        clean_prompt,
        add_special_tokens=(
            TOKENIZER_ADD_SPECIAL_TOKENS
        ),
        return_attention_mask=False,
        return_tensors=None,
    )

    corrupt_encoded = tokenizer(
        corrupt_prompt,
        add_special_tokens=(
            TOKENIZER_ADD_SPECIAL_TOKENS
        ),
        return_attention_mask=False,
        return_tensors=None,
    )

    clean_ids = list(
        clean_encoded["input_ids"]
    )

    corrupt_ids = list(
        corrupt_encoded["input_ids"]
    )

    tokenization_rows.append(
        {
            "index": index,
            "example_id": record[
                "example_id"
            ],
            "template_id": record[
                "template_id"
            ],
            "name_pair_id": record[
                "name_pair_id"
            ],
            "split": record[
                "split"
            ],
            "clean_prompt": clean_prompt,
            "corrupt_prompt": corrupt_prompt,
            "clean_ids": clean_ids,
            "corrupt_ids": corrupt_ids,
        }
    )


clean_lengths = [
    len(row["clean_ids"])
    for row in tokenization_rows
]

corrupt_lengths = [
    len(row["corrupt_ids"])
    for row in tokenization_rows
]

length_equal_count = sum(
    clean_len == corrupt_len
    for clean_len, corrupt_len
    in zip(
        clean_lengths,
        corrupt_lengths,
    )
)

length_deltas = [
    corrupt_len - clean_len
    for clean_len, corrupt_len
    in zip(
        clean_lengths,
        corrupt_lengths,
    )
]

print(
    f"  records tokenized: "
    f"{len(tokenization_rows)}"
)

print(
    f"  clean length range: "
    f"{min(clean_lengths)}–{max(clean_lengths)}"
)

print(
    f"  corrupt length range: "
    f"{min(corrupt_lengths)}–{max(corrupt_lengths)}"
)

print(
    f"  clean/corrupt equal-length pairs: "
    f"{length_equal_count}/{EXPECTED_RECORD_COUNT}"
)

print(
    f"  length-delta distribution: "
    f"{dict(Counter(length_deltas))}"
)


# ------------------------------------------------------------------------------
# 9. TARGET-NAME TOKENIZATION
# ------------------------------------------------------------------------------

print("\n[9/14] Target-name tokenization audit")
print("-" * 78)

target_names = sorted(
    set(
        [
            record["clean_target_name"]
            for record in records_ref
        ]
        +
        [
            record["corrupt_target_name"]
            for record in records_ref
        ]
    )
)

target_tokenizations = {}

for name in target_names:

    ids = tokenizer.encode(
        name,
        add_special_tokens=False,
    )

    target_tokenizations[name] = list(
        ids
    )


target_length_distribution = Counter(
    len(ids)
    for ids in target_tokenizations.values()
)

single_token_names = {
    name: ids
    for name, ids
    in target_tokenizations.items()
    if len(ids) == 1
}

non_single_token_names = {
    name: ids
    for name, ids
    in target_tokenizations.items()
    if len(ids) != 1
}

print(
    f"  distinct target names: "
    f"{len(target_names)}"
)

print(
    f"  token-length distribution: "
    f"{dict(target_length_distribution)}"
)

print(
    f"  single-token names: "
    f"{len(single_token_names)}"
)

print(
    f"  non-single-token names: "
    f"{len(non_single_token_names)}"
)


if non_single_token_names:

    print(
        "\n  non-single-token target names:"
    )

    for name, ids in sorted(
        non_single_token_names.items()
    ):

        pieces = tokenizer.convert_ids_to_tokens(
            ids
        )

        print(
            f"    {name!r}: "
            f"IDs={ids}, "
            f"tokens={pieces}"
        )


# ------------------------------------------------------------------------------
# 10. TARGET-POSITION LOCALIZATION
# ------------------------------------------------------------------------------

print("\n[10/14] Target-position localization")
print("-" * 78)


def find_all_subsequence(
    haystack,
    needle,
):
    """
    Return every contiguous occurrence of
    `needle` in `haystack`.
    """

    if len(needle) == 0:
        return []

    hits = []

    needle_length = len(needle)

    for start in range(
        0,
        len(haystack) - needle_length + 1,
    ):

        candidate = haystack[
            start:start + needle_length
        ]

        if candidate == needle:

            hits.append(start)

    return hits


localization_rows = []

for row in tokenization_rows:

    record = records_ref[
        row["index"]
    ]

    clean_target_name = record[
        "clean_target_name"
    ]

    corrupt_target_name = record[
        "corrupt_target_name"
    ]

    clean_target_ids = (
        target_tokenizations[
            clean_target_name
        ]
    )

    corrupt_target_ids = (
        target_tokenizations[
            corrupt_target_name
        ]
    )

    clean_hits = find_all_subsequence(
        row["clean_ids"],
        clean_target_ids,
    )

    corrupt_hits = find_all_subsequence(
        row["corrupt_ids"],
        corrupt_target_ids,
    )

    clean_unique = (
        len(clean_hits) == 1
    )

    corrupt_unique = (
        len(corrupt_hits) == 1
    )

    localization_rows.append(
        {
            "index": row["index"],
            "example_id": row[
                "example_id"
            ],
            "split": row["split"],
            "template_id": row[
                "template_id"
            ],
            "name_pair_id": row[
                "name_pair_id"
            ],
            "clean_target_name": (
                clean_target_name
            ),
            "corrupt_target_name": (
                corrupt_target_name
            ),
            "clean_target_ids": (
                clean_target_ids
            ),
            "corrupt_target_ids": (
                corrupt_target_ids
            ),
            "clean_hits": clean_hits,
            "corrupt_hits": corrupt_hits,
            "clean_unique": clean_unique,
            "corrupt_unique": corrupt_unique,
        }
    )


clean_unique_localized = sum(
    row["clean_unique"]
    for row in localization_rows
)

corrupt_unique_localized = sum(
    row["corrupt_unique"]
    for row in localization_rows
)

print(
    f"  clean targets uniquely localized: "
    f"{clean_unique_localized}/{EXPECTED_RECORD_COUNT}"
)

print(
    f"  corrupt targets uniquely localized: "
    f"{corrupt_unique_localized}/{EXPECTED_RECORD_COUNT}"
)


# ------------------------------------------------------------------------------
# 11. BEHAVIORAL TARGET-POSITION COMPATIBILITY
# ------------------------------------------------------------------------------
#
# No logits are computed here.
# This section only characterizes the token representation required by the
# later behavioral baseline.
# ------------------------------------------------------------------------------

print(
    "\n[11/14] Behavioral target-position compatibility"
)

print("-" * 78)

behavior_rows = []

for row in tokenization_rows:

    record = records_ref[
        row["index"]
    ]

    clean_target_ids = (
        target_tokenizations[
            record["clean_target_name"]
        ]
    )

    corrupt_target_ids = (
        target_tokenizations[
            record["corrupt_target_name"]
        ]
    )

    behavior_rows.append(
        {
            "index": row["index"],
            "example_id": row[
                "example_id"
            ],
            "split": row["split"],
            "clean_prompt_length": len(
                row["clean_ids"]
            ),
            "corrupt_prompt_length": len(
                row["corrupt_ids"]
            ),
            "clean_prompt_last_token_id": (
                row["clean_ids"][-1]
                if row["clean_ids"]
                else None
            ),
            "corrupt_prompt_last_token_id": (
                row["corrupt_ids"][-1]
                if row["corrupt_ids"]
                else None
            ),
            "clean_target_first_token_id": (
                clean_target_ids[0]
                if clean_target_ids
                else None
            ),
            "corrupt_target_first_token_id": (
                corrupt_target_ids[0]
                if corrupt_target_ids
                else None
            ),
            "clean_target_token_count": (
                len(clean_target_ids)
            ),
            "corrupt_target_token_count": (
                len(corrupt_target_ids)
            ),
        }
    )


nonempty_clean = sum(
    row["clean_prompt_length"] > 0
    for row in behavior_rows
)

nonempty_corrupt = sum(
    row["corrupt_prompt_length"] > 0
    for row in behavior_rows
)

print(
    f"  nonempty clean prompt tokenizations: "
    f"{nonempty_clean}/{EXPECTED_RECORD_COUNT}"
)

print(
    f"  nonempty corrupt prompt tokenizations: "
    f"{nonempty_corrupt}/{EXPECTED_RECORD_COUNT}"
)

clean_target_token_counts = Counter(
    row["clean_target_token_count"]
    for row in behavior_rows
)

corrupt_target_token_counts = Counter(
    row["corrupt_target_token_count"]
    for row in behavior_rows
)

print(
    f"  clean target token-count distribution: "
    f"{dict(clean_target_token_counts)}"
)

print(
    f"  corrupt target token-count distribution: "
    f"{dict(corrupt_target_token_counts)}"
)


# ------------------------------------------------------------------------------
# 12. CLEAN/CORRUPT TOKENIZATION ALIGNMENT
# ------------------------------------------------------------------------------

print("\n[12/14] Clean/corrupt tokenization alignment")
print("-" * 78)

pair_alignment_rows = []

for row in tokenization_rows:

    clean_ids = row["clean_ids"]
    corrupt_ids = row["corrupt_ids"]

    maximum_prefix = min(
        len(clean_ids),
        len(corrupt_ids),
    )

    common_prefix = 0

    while common_prefix < maximum_prefix:

        if (
            clean_ids[common_prefix]
            != corrupt_ids[common_prefix]
        ):

            break

        common_prefix += 1


    maximum_suffix = (
        maximum_prefix
        - common_prefix
    )

    common_suffix = 0

    while common_suffix < maximum_suffix:

        clean_index = (
            len(clean_ids)
            - 1
            - common_suffix
        )

        corrupt_index = (
            len(corrupt_ids)
            - 1
            - common_suffix
        )

        if (
            clean_ids[clean_index]
            != corrupt_ids[corrupt_index]
        ):

            break

        common_suffix += 1


    pair_alignment_rows.append(
        {
            "index": row["index"],
            "example_id": row[
                "example_id"
            ],
            "template_id": row[
                "template_id"
            ],
            "name_pair_id": row[
                "name_pair_id"
            ],
            "split": row["split"],
            "clean_length": len(clean_ids),
            "corrupt_length": len(corrupt_ids),
            "length_delta": (
                len(corrupt_ids)
                - len(clean_ids)
            ),
            "common_prefix_tokens": (
                common_prefix
            ),
            "common_suffix_tokens": (
                common_suffix
            ),
        }
    )


prefix_lengths = [
    row["common_prefix_tokens"]
    for row in pair_alignment_rows
]

suffix_lengths = [
    row["common_suffix_tokens"]
    for row in pair_alignment_rows
]

exact_sequence_matches = 0

for row in tokenization_rows:

    if (
        row["clean_ids"]
        == row["corrupt_ids"]
    ):

        exact_sequence_matches += 1


print(
    f"  common-prefix token range: "
    f"{min(prefix_lengths)}–{max(prefix_lengths)}"
)

print(
    f"  common-suffix token range: "
    f"{min(suffix_lengths)}–{max(suffix_lengths)}"
)

print(
    f"  exact clean/corrupt token-sequence "
    f"matches: "
    f"{exact_sequence_matches}/{EXPECTED_RECORD_COUNT}"
)


# ------------------------------------------------------------------------------
# 13. SCIENTIFIC / OPERATIONAL CLASSIFICATION
# ------------------------------------------------------------------------------

print("\n[13/14] Scientific / operational classification")
print("-" * 78)

all_prompts_tokenized = (
    len(tokenization_rows)
    == EXPECTED_RECORD_COUNT
    and
    nonempty_clean
    == EXPECTED_RECORD_COUNT
    and
    nonempty_corrupt
    == EXPECTED_RECORD_COUNT
)

all_targets_uniquely_localized = (
    clean_unique_localized
    == EXPECTED_RECORD_COUNT
    and
    corrupt_unique_localized
    == EXPECTED_RECORD_COUNT
)

all_targets_single_token = (
    len(non_single_token_names) == 0
)


if not all_prompts_tokenized:

    classification = (
        "OPERATIONAL_DECOHERENCE"
    )

    interpretation = (
        "The exact authenticated prompts cannot all "
        "be represented as nonempty Llama token sequences "
        "under the audited tokenizer convention."
    )

elif not all_targets_uniquely_localized:

    classification = (
        "OPERATIONAL_DECOHERENCE"
    )

    interpretation = (
        "The intended target-name spans cannot all be "
        "uniquely localized under the Llama tokenizer. "
        "The scientific specification is retained unchanged; "
        "the mismatch is classified as operational decoherence."
    )

elif all_targets_single_token:

    classification = "PASS"

    interpretation = (
        "All authenticated target names are represented "
        "as single Llama tokens and the target structure "
        "is operationally well-defined for a scalar "
        "next-token behavioral evaluation."
    )

else:

    classification = (
        "PASS_WITH_MULTI_TOKEN_TARGETS"
    )

    interpretation = (
        "All authenticated prompts and target names are "
        "operationally representable, but at least one "
        "target name is multi-token under Llama. The "
        "stimulus remains unchanged. A later behavioral "
        "metric must explicitly account for the observed "
        "multi-token structure rather than importing the "
        "GPT-2 single-token assumption."
    )


print(
    f"  classification: "
    f"{classification}"
)

print(
    f"  interpretation: "
    f"{interpretation}"
)


# ------------------------------------------------------------------------------
# 14. POST-AUDIT INTEGRITY + MANIFEST
# ------------------------------------------------------------------------------

print("\n[14/14] Post-audit integrity and manifest")
print("-" * 78)

records_after_bytes = canonical_json_bytes(
    records_ref
)

records_after_sha = hashlib.sha256(
    records_after_bytes
).hexdigest()

records_unchanged = (
    records_before_bytes
    == records_after_bytes
)

print(
    f"  records unchanged during audit: "
    f"{records_unchanged}"
)

print(
    f"  post-audit in-memory canonical SHA-256: "
    f"{records_after_sha}"
)

if not records_unchanged:

    raise RuntimeError(
        "CRITICAL: llama_records changed during tokenizer audit."
    )


# ------------------------------------------------------------------------------
# TEMPLATE SUMMARY
# ------------------------------------------------------------------------------

template_summary = defaultdict(
    lambda: {
        "n": 0,
        "clean_min_length": None,
        "clean_max_length": None,
        "corrupt_min_length": None,
        "corrupt_max_length": None,
        "clean_multi_token_targets": 0,
        "corrupt_multi_token_targets": 0,
    }
)


for row, localization in zip(
    tokenization_rows,
    localization_rows,
):

    template_id = row[
        "template_id"
    ]

    summary = template_summary[
        template_id
    ]

    summary["n"] += 1

    clean_length = len(
        row["clean_ids"]
    )

    corrupt_length = len(
        row["corrupt_ids"]
    )


    if summary["clean_min_length"] is None:

        summary[
            "clean_min_length"
        ] = clean_length

    else:

        summary[
            "clean_min_length"
        ] = min(
            summary[
                "clean_min_length"
            ],
            clean_length,
        )


    if summary["clean_max_length"] is None:

        summary[
            "clean_max_length"
        ] = clean_length

    else:

        summary[
            "clean_max_length"
        ] = max(
            summary[
                "clean_max_length"
            ],
            clean_length,
        )


    if summary["corrupt_min_length"] is None:

        summary[
            "corrupt_min_length"
        ] = corrupt_length

    else:

        summary[
            "corrupt_min_length"
        ] = min(
            summary[
                "corrupt_min_length"
            ],
            corrupt_length,
        )


    if summary["corrupt_max_length"] is None:

        summary[
            "corrupt_max_length"
        ] = corrupt_length

    else:

        summary[
            "corrupt_max_length"
        ] = max(
            summary[
                "corrupt_max_length"
            ],
            corrupt_length,
        )


    if len(
        localization[
            "clean_target_ids"
        ]
    ) != 1:

        summary[
            "clean_multi_token_targets"
        ] += 1


    if len(
        localization[
            "corrupt_target_ids"
        ]
    ) != 1:

        summary[
            "corrupt_multi_token_targets"
        ] += 1


# ------------------------------------------------------------------------------
# SERIALIZABLE MANIFEST
# ------------------------------------------------------------------------------

manifest = {
    "experiment_id": EXPERIMENT_ID,
    "phase_id": PHASE_ID,
    "model_id_expected": MODEL_ID_EXPECTED,

    "authorization": {
        "phase": "1C.0",
        "manifest": AUTH_MANIFEST_PATH,
        "authorized": True,
        "authorization_evidence": (
            authorization_evidence
        ),
        "status_evidence": (
            final_status_candidates
        ),
        "authenticated_dataset_sha256": (
            EXPECTED_DATASET_SHA
        ),
    },

    "tokenizer": {
        "class": tokenizer.__class__.__name__,
        "name_or_path": tokenizer.name_or_path,
        "is_fast": bool(
            getattr(
                tokenizer,
                "is_fast",
                False,
            )
        ),
        "vocab_size": getattr(
            tokenizer,
            "vocab_size",
            None,
        ),
        "model_max_length": getattr(
            tokenizer,
            "model_max_length",
            None,
        ),
        "bos_token": tokenizer.bos_token,
        "eos_token": tokenizer.eos_token,
        "pad_token": tokenizer.pad_token,
        "bos_token_id": tokenizer.bos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.pad_token_id,
        "add_special_tokens": (
            TOKENIZER_ADD_SPECIAL_TOKENS
        ),
    },

    "dataset": {
        "record_count": len(records_ref),
        "template_count": len(template_ids),
        "name_pair_count": len(pair_ids),
        "split_counts": dict(
            split_counts
        ),
        "condition_pair_counts": dict(
            condition_counts
        ),
        "authenticated_raw_sha256": (
            EXPECTED_DATASET_SHA
        ),
        "in_memory_pre_audit_canonical_sha256": (
            records_before_sha
        ),
        "in_memory_post_audit_canonical_sha256": (
            records_after_sha
        ),
        "unchanged_during_audit": (
            records_unchanged
        ),
    },

    "sequence_lengths": {
        "clean_min": min(clean_lengths),
        "clean_max": max(clean_lengths),
        "clean_mean": (
            sum(clean_lengths)
            / len(clean_lengths)
        ),
        "corrupt_min": min(corrupt_lengths),
        "corrupt_max": max(corrupt_lengths),
        "corrupt_mean": (
            sum(corrupt_lengths)
            / len(corrupt_lengths)
        ),
        "equal_length_pairs": (
            length_equal_count
        ),
        "length_delta_distribution": dict(
            Counter(length_deltas)
        ),
    },

    "target_tokenization": {
        "distinct_target_names": len(
            target_names
        ),
        "single_token_name_count": len(
            single_token_names
        ),
        "non_single_token_name_count": len(
            non_single_token_names
        ),
        "token_length_distribution": dict(
            target_length_distribution
        ),
        "tokenizations": (
            target_tokenizations
        ),
        "non_single_token_names": (
            non_single_token_names
        ),
    },

    "target_localization": {
        "clean_unique_localized": (
            clean_unique_localized
        ),
        "corrupt_unique_localized": (
            corrupt_unique_localized
        ),
        "all_clean_unique": (
            clean_unique_localized
            == EXPECTED_RECORD_COUNT
        ),
        "all_corrupt_unique": (
            corrupt_unique_localized
            == EXPECTED_RECORD_COUNT
        ),
    },

    "model_identity": {
        "expected": MODEL_ID_EXPECTED,
        "observed": model_identity_observed,
        "match": model_identity_ok,
    },

    "template_summary": dict(
        template_summary
    ),

    "classification": classification,

    "interpretation": interpretation,

    "governance": {
        "mathematical_specification_modified": False,
        "dataset_modified": False,
        "prompt_rewritten": False,
        "behavioral_inference_run": False,
        "transport_fitted": False,
        "state_bank_created": False,
        "scientific_selection_performed": False,
        "decoherence_policy": (
            "Mathematical implications remain authoritative. "
            "A genuine mismatch between mathematical specification "
            "and numerical/software operationalization is recorded "
            "as decoherence rather than used to redefine the "
            "mathematics."
        ),
    },
}


with open(
    OUTPUT_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ------------------------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------------------------

print("\n" + "=" * 78)
print("FINAL STATUS")
print("=" * 78)

print(
    f"  classification: "
    f"{classification}"
)

print(
    f"  tokenizer identity: "
    f"{'PASS' if tokenizer_identity_ok else 'FAIL'}"
)

print(
    f"  exact prompts tokenized: "
    f"{len(tokenization_rows)}/{EXPECTED_RECORD_COUNT}"
)

print(
    f"  clean targets uniquely localized: "
    f"{clean_unique_localized}/{EXPECTED_RECORD_COUNT}"
)

print(
    f"  corrupt targets uniquely localized: "
    f"{corrupt_unique_localized}/{EXPECTED_RECORD_COUNT}"
)

print(
    f"  single-token target names: "
    f"{len(single_token_names)}/{len(target_names)}"
)

print(
    f"  multi-token target names: "
    f"{len(non_single_token_names)}/{len(target_names)}"
)

print(
    f"  dataset mutated: "
    f"{not records_unchanged}"
)

print(
    "  behavioral inference performed: False"
)

print(
    "  transport fitting performed: False"
)

print(
    "  state bank created: False"
)

print()
print(
    "  audit manifest:"
)

print(
    f"    {OUTPUT_MANIFEST_PATH}"
)

print("=" * 78)


# ------------------------------------------------------------------------------
# CLEANUP
# ------------------------------------------------------------------------------

gc.collect()

print(
    "\nPhase 1C.1 tokenizer alignment audit complete."
)

print(
    "No scientific state, model weights, prompts, or authenticated "
    "dataset records were altered."
)

ETTR-CTL-LLAMA-1 — PHASE_1C.1
LLAMA TOKENIZER ALIGNMENT / TARGET-POSITION AUDIT

[1/14] Dataset authorization precondition
------------------------------------------------------------------------------
  authorization manifest: /content/ettr_ctl_llama/results/llama_phase1c0_recovered_dataset_authorization_v2.json
  manifest top-level keys: ['authorization_status', 'condition_pair', 'cryptographic_integrity', 'dataset_authorized', 'experiment_id', 'governance', 'historical_sha256', 'historical_sha_used_for_authorization', 'name_pair_count', 'next_stage', 'record_count', 'scientific_integrity', 'sha256', 'source_artifact', 'source_artifact_version', 'source_path', 'split_counts', 'template_count', 'timestamp_utc']

  explicit authorization evidence:
    dataset_authorized: True

  status evidence:
    none found

  explicit authorization established: True
  authorization precondition: PASS

[2/14] Operative dataset precondition
------------------------------------------------------------

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

  tokenizer class: TokenizersBackend
  tokenizer name_or_path: meta-llama/Llama-3.2-3B
  tokenizer vocab_size: 128000
  tokenizer model_max_length: 131072
  tokenizer is_fast: True
  bos_token: '<|begin_of_text|>'
  eos_token: '<|end_of_text|>'
  pad_token: None
  bos_token_id: 128000
  eos_token_id: 128001
  pad_token_id: None
  tokenizer identity: PASS

[6/14] Special-token convention audit
------------------------------------------------------------------------------
  probe text: 'Alice'
  no-special IDs: [62786]
  with-special IDs: [128000, 62786]
  add_special_tokens changes probe: True
  operative audit convention: add_special_tokens=False

[7/14] Llama-side dataset structural recheck
------------------------------------------------------------------------------
  uniform 17-field schema: True
  template count: 12
  name-pair count: 16
  split counts: {'calibration': 48, 'test': 48, 'train': 96}
  condition_pair values: {'clean_vs_corrupt': 192}
  structural recheck: PASS

[8/14

In [8]:
# ==============================================================================
# ETTR-CTL-LLAMA-1 — PHASE 1C.1A
# HUGGING FACE AUTHENTICATION / GATED-REPOSITORY ACCESS GATE
# ==============================================================================
#
# PURPOSE
# -------
# Resolve the current Colab runtime's authentication state before the Llama
# tokenizer alignment audit.
#
# This cell addresses ONLY the numerical/software execution substrate.
#
# GOVERNING PRINCIPLE
# -------------------
# Authentication and repository-access constraints must not alter:
#
#   - the CTL mathematics;
#   - the authenticated dataset;
#   - the experimental prompts;
#   - the train/calibration/test partition;
#   - the sector definitions.
#
# If authentication cannot be established, the experiment is HALTED at the
# runtime layer. No scientific workaround is introduced.
#
# SECURITY
# --------
# No access token is printed, stored in the manifest, or displayed.
#
# AUDIT ONLY
# ----------
# This cell does NOT:
#   - load Llama weights;
#   - perform model inference;
#   - tokenize the experimental dataset;
#   - modify llama_records;
#   - modify any experimental artifact.
# ==============================================================================

import os
import gc
import json
import hashlib
from datetime import datetime, timezone

import huggingface_hub
from huggingface_hub import (
    get_token,
    whoami,
    hf_hub_download,
)


# ------------------------------------------------------------------------------
# 0. CONSTANTS
# ------------------------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
PHASE_ID = "PHASE_1C.1A"

MODEL_ID = "meta-llama/Llama-3.2-3B"
PROBE_FILENAME = "config.json"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(
    ROOT,
    "results",
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True,
)

OUTPUT_MANIFEST = os.path.join(
    RESULTS_DIR,
    "llama_phase1c1a_huggingface_authentication_gate.json",
)


print("=" * 78)
print(
    f"{EXPERIMENT_ID} — {PHASE_ID}"
)
print(
    "HUGGING FACE AUTHENTICATION / GATED-REPOSITORY ACCESS GATE"
)
print("=" * 78)


# ------------------------------------------------------------------------------
# 1. DATASET / SCIENTIFIC STATE FIREWALL
# ------------------------------------------------------------------------------
#
# Confirm that the authenticated experimental dataset is still present and
# unchanged. This is a guard against accidentally solving a runtime problem by
# manipulating scientific state.
# ------------------------------------------------------------------------------

print("\n[1/9] Scientific-state firewall")
print("-" * 78)

DATASET_SHA = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)

EXPECTED_RECORD_COUNT = 192

dataset_state_present = (
    "llama_records" in globals()
)

dataset_state_valid = False
dataset_state_sha = None

if dataset_state_present:

    records = llama_records

    if isinstance(records, list):

        if len(records) == EXPECTED_RECORD_COUNT:

            canonical_bytes = json.dumps(
                records,
                ensure_ascii=False,
                sort_keys=True,
                separators=(",", ":"),
            ).encode("utf-8")

            dataset_state_sha = hashlib.sha256(
                canonical_bytes
            ).hexdigest()

            dataset_state_valid = True

print(
    f"  llama_records present: "
    f"{dataset_state_present}"
)

print(
    f"  record count valid: "
    f"{dataset_state_valid}"
)

if dataset_state_valid:

    print(
        f"  in-memory dataset SHA-256: "
        f"{dataset_state_sha}"
    )

    print(
        f"  authenticated artifact SHA-256: "
        f"{DATASET_SHA}"
    )

else:

    print(
        "  dataset integrity could not be independently "
        "rechecked from the current Python state."
    )

print(
    "  scientific state modified by this cell: False"
)


# ------------------------------------------------------------------------------
# 2. HUGGING FACE LIBRARY / RUNTIME INFORMATION
# ------------------------------------------------------------------------------

print("\n[2/9] Hugging Face runtime audit")
print("-" * 78)

print(
    f"  huggingface_hub version: "
    f"{getattr(huggingface_hub, '__version__', 'unknown')}"
)

print(
    f"  HF_HOME: "
    f"{os.environ.get('HF_HOME', '<unset>')}"
)

print(
    f"  HF_TOKEN environment variable present: "
    f"{bool(os.environ.get('HF_TOKEN'))}"
)

print(
    f"  HF_HUB_DISABLE_IMPLICIT_TOKEN: "
    f"{os.environ.get('HF_HUB_DISABLE_IMPLICIT_TOKEN', '<unset>')}"
)


# ------------------------------------------------------------------------------
# 3. CURRENT TOKEN STATE
# ------------------------------------------------------------------------------
#
# get_token() returns the current locally available token, if any.
#
# SECURITY:
# We record ONLY whether a token exists and never print its value.
# ------------------------------------------------------------------------------

print("\n[3/9] Current Hugging Face credential state")
print("-" * 78)

current_token = get_token()

token_present_before_login = (
    current_token is not None
    and len(str(current_token)) > 0
)

print(
    f"  credential available before login: "
    f"{token_present_before_login}"
)

if token_present_before_login:

    print(
        "  token value: [REDACTED]"
    )

else:

    print(
        "  token value: not available"
    )


# ------------------------------------------------------------------------------
# 4. CURRENT ACCOUNT / SESSION TEST
# ------------------------------------------------------------------------------
#
# This does not expose the token.
# ------------------------------------------------------------------------------

print("\n[4/9] Current account/session test")
print("-" * 78)

whoami_success = False
whoami_error = None
account_info = None

if token_present_before_login:

    try:

        account_info = whoami(
            token=current_token
        )

        whoami_success = True

        if isinstance(account_info, dict):

            username = account_info.get(
                "name",
                account_info.get(
                    "fullname",
                    "<redacted>"
                )
            )

            print(
                f"  authenticated account recognized: "
                f"{username}"
            )

            print(
                "  credential value: [REDACTED]"
            )

        else:

            print(
                "  authenticated account recognized."
            )

    except Exception as exc:

        whoami_error = (
            f"{type(exc).__name__}: {exc}"
        )

        print(
            f"  current credential test failed: "
            f"{whoami_error}"
        )

else:

    print(
        "  no current credential available; "
        "interactive login required."
    )


# ------------------------------------------------------------------------------
# 5. AUTHENTICATION
# ------------------------------------------------------------------------------
#
# If the current credential is absent or invalid, invoke the supported
# interactive login flow.
#
# The token is entered through the Hugging Face authentication mechanism.
# It is NEVER hard-coded into this notebook cell.
# ------------------------------------------------------------------------------

print("\n[5/9] Hugging Face authentication")
print("-" * 78)

authentication_method = None
authentication_error = None

if not whoami_success:

    try:

        from huggingface_hub import login

        print(
            "  initiating interactive Hugging Face login..."
        )

        login(
            skip_if_logged_in=False
        )

        authentication_method = (
            "interactive_huggingface_login"
        )

        print(
            "  interactive login completed."
        )

    except Exception as exc:

        authentication_error = (
            f"{type(exc).__name__}: {exc}"
        )

        print(
            f"  interactive login failed: "
            f"{authentication_error}"
        )

else:

    authentication_method = (
        "existing_authenticated_session"
    )

    print(
        "  existing authenticated session accepted."
    )


# ------------------------------------------------------------------------------
# 6. POST-AUTHENTICATION CREDENTIAL TEST
# ------------------------------------------------------------------------------

print("\n[6/9] Post-authentication credential test")
print("-" * 78)

resolved_token = get_token()

token_present_after_login = (
    resolved_token is not None
    and len(str(resolved_token)) > 0
)

print(
    f"  credential available after authentication: "
    f"{token_present_after_login}"
)

if token_present_after_login:

    print(
        "  token value: [REDACTED]"
    )


post_auth_success = False
post_auth_error = None
post_auth_account = None

if token_present_after_login:

    try:

        post_auth_account = whoami(
            token=resolved_token
        )

        post_auth_success = True

        if isinstance(
            post_auth_account,
            dict,
        ):

            post_username = post_auth_account.get(
                "name",
                post_auth_account.get(
                    "fullname",
                    "<redacted>"
                )
            )

            print(
                f"  authenticated account recognized: "
                f"{post_username}"
            )

        else:

            print(
                "  authenticated account recognized."
            )

    except Exception as exc:

        post_auth_error = (
            f"{type(exc).__name__}: {exc}"
        )

        print(
            f"  post-authentication account test failed: "
            f"{post_auth_error}"
        )

else:

    print(
        "  no usable credential is available."
    )


# ------------------------------------------------------------------------------
# 7. EXACT GATED-REPOSITORY ACCESS PROBE
# ------------------------------------------------------------------------------
#
# This is the decisive runtime test.
#
# We request ONLY the public model configuration file from the exact gated
# repository. We do not download model weights.
#
# Passing this test establishes repository access, not model loading.
# ------------------------------------------------------------------------------

print("\n[7/9] Exact gated-repository access probe")
print("-" * 78)

repo_access = False
repo_access_error = None
config_path = None

if post_auth_success:

    try:

        config_path = hf_hub_download(
            repo_id=MODEL_ID,
            filename=PROBE_FILENAME,
            token=resolved_token,
        )

        repo_access = True

        print(
            f"  repository: {MODEL_ID}"
        )

        print(
            f"  probe file: {PROBE_FILENAME}"
        )

        print(
            f"  resolved local path: "
            f"{config_path}"
        )

        print(
            "  gated-repository access: PASS"
        )

    except Exception as exc:

        repo_access_error = (
            f"{type(exc).__name__}: {exc}"
        )

        print(
            f"  gated-repository access: FAIL"
        )

        print(
            f"  error class: "
            f"{type(exc).__name__}"
        )

        print(
            f"  error: "
            f"{exc}"
        )

else:

    print(
        "  gated-repository probe skipped because "
        "authentication did not succeed."
    )


# ------------------------------------------------------------------------------
# 8. SECURITY / SCIENTIFIC GOVERNANCE AUDIT
# ------------------------------------------------------------------------------

print("\n[8/9] Security and scientific governance")
print("-" * 78)

token_value_recorded = False

print(
    f"  token value recorded in output: "
    f"{token_value_recorded}"
)

print(
    "  experimental prompts modified: False"
)

print(
    "  llama_records modified: False"
)

print(
    "  model weights downloaded by this cell: False"
)

print(
    "  model inference performed: False"
)

print(
    "  tokenizer alignment performed: False"
)

print(
    "  CTL mathematical specification modified: False"
)


# ------------------------------------------------------------------------------
# 9. FINAL CLASSIFICATION / MANIFEST
# ------------------------------------------------------------------------------

print("\n[9/9] Final classification")
print("-" * 78)

if repo_access:

    classification = (
        "HUGGING_FACE_GATED_ACCESS_PASS"
    )

    next_stage = (
        "PHASE_1C.1 LLAMA TOKENIZER ALIGNMENT AUDIT"
    )

    interpretation = (
        "The current Colab runtime possesses a usable "
        "Hugging Face credential and can access the exact "
        "gated Llama 3.2 3B repository. The tokenizer "
        "alignment audit may proceed."
    )

elif not post_auth_success:

    classification = (
        "NUMERICAL_RUNTIME_AUTHENTICATION_FAILURE"
    )

    next_stage = (
        "REAUTHENTICATE_HUGGING_FACE"
    )

    interpretation = (
        "The current Colab runtime does not possess a "
        "usable Hugging Face credential. No scientific "
        "operation is authorized until authentication is "
        "successfully established."
    )

else:

    classification = (
        "NUMERICAL_RUNTIME_REPOSITORY_ACCESS_FAILURE"
    )

    next_stage = (
        "DIAGNOSE_HUGGING_FACE_GATED_ACCESS"
    )

    interpretation = (
        "A Hugging Face credential exists and account "
        "authentication succeeds, but the credential cannot "
        "access the exact gated Llama repository. This is "
        "a runtime/access-layer failure, not scientific "
        "decoherence."
    )


timestamp_utc = datetime.now(
    timezone.utc
).isoformat()


manifest = {
    "experiment_id": EXPERIMENT_ID,
    "phase_id": PHASE_ID,

    "model_id": MODEL_ID,

    "timestamp_utc": timestamp_utc,

    "runtime": {
        "huggingface_hub_version": (
            getattr(
                huggingface_hub,
                "__version__",
                None,
            )
        ),
        "hf_home": os.environ.get(
            "HF_HOME"
        ),
        "hf_token_environment_present": bool(
            os.environ.get("HF_TOKEN")
        ),
    },

    "credential_state": {
        "token_present_before_login": (
            token_present_before_login
        ),
        "whoami_before_login": (
            whoami_success
        ),
        "token_present_after_login": (
            token_present_after_login
        ),
        "whoami_after_login": (
            post_auth_success
        ),
        "authentication_method": (
            authentication_method
        ),
        "token_value_recorded": False,
    },

    "repository_probe": {
        "repo_id": MODEL_ID,
        "filename": PROBE_FILENAME,
        "access_pass": repo_access,
        "local_path_resolved": (
            config_path
            if repo_access
            else None
        ),
        "error_class": (
            type(repo_access_error).__name__
            if repo_access_error
            else None
        ),
    },

    "scientific_state": {
        "llama_records_present": (
            dataset_state_present
        ),
        "record_count": (
            len(llama_records)
            if dataset_state_valid
            else None
        ),
        "in_memory_canonical_sha256": (
            dataset_state_sha
        ),
        "authenticated_dataset_sha256": (
            DATASET_SHA
        ),
        "records_modified": False,
    },

    "governance": {
        "mathematics_modified": False,
        "dataset_modified": False,
        "prompts_modified": False,
        "weights_downloaded": False,
        "inference_performed": False,
        "tokenizer_alignment_performed": False,
        "decoherence_policy": (
            "Authentication/repository constraints are "
            "runtime-layer conditions. They must not be "
            "used to redefine the CTL mathematical "
            "specification or alter the experimental "
            "stimulus."
        ),
    },

    "classification": classification,

    "next_authorized_stage": next_stage,

    "interpretation": interpretation,
}


with open(
    OUTPUT_MANIFEST,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2,
    )


print(
    f"\n  classification: "
    f"{classification}"
)

print(
    f"  next authorized stage: "
    f"{next_stage}"
)

print(
    f"  manifest: "
    f"{OUTPUT_MANIFEST}"
)

print("=" * 78)

gc.collect()

print(
    "\nPhase 1C.1A authentication/access gate complete."
)

print(
    "No model weights, experimental prompts, or scientific "
    "definitions were altered."
)

ETTR-CTL-LLAMA-1 — PHASE_1C.1A
HUGGING FACE AUTHENTICATION / GATED-REPOSITORY ACCESS GATE

[1/9] Scientific-state firewall
------------------------------------------------------------------------------
  llama_records present: True
  record count valid: True
  in-memory dataset SHA-256: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  authenticated artifact SHA-256: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  scientific state modified by this cell: False

[2/9] Hugging Face runtime audit
------------------------------------------------------------------------------
  huggingface_hub version: 1.29.0
  HF_HOME: <unset>
  HF_TOKEN environment variable present: False
  HF_HUB_DISABLE_IMPLICIT_TOKEN: <unset>

[3/9] Current Hugging Face credential state
------------------------------------------------------------------------------
  credential available before login: False
  token value: not available

[4/9] Current account/session test
--------------

  interactive login completed.

[6/9] Post-authentication credential test
------------------------------------------------------------------------------
  credential available after authentication: True
  token value: [REDACTED]
  authenticated account recognized: onerospacetime

[7/9] Exact gated-repository access probe
------------------------------------------------------------------------------


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

  repository: meta-llama/Llama-3.2-3B
  probe file: config.json
  resolved local path: /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-3B/snapshots/13afe5124825b4f3751f836b40dafda64c1ed062/config.json
  gated-repository access: PASS

[8/9] Security and scientific governance
------------------------------------------------------------------------------
  token value recorded in output: False
  experimental prompts modified: False
  llama_records modified: False
  model weights downloaded by this cell: False
  model inference performed: False
  tokenizer alignment performed: False
  CTL mathematical specification modified: False

[9/9] Final classification
------------------------------------------------------------------------------

  classification: HUGGING_FACE_GATED_ACCESS_PASS
  next authorized stage: PHASE_1C.1 LLAMA TOKENIZER ALIGNMENT AUDIT
  manifest: /content/ettr_ctl_llama/results/llama_phase1c1a_huggingface_authentication_gate.json

Phase 1C.1A authentication/acce

In [10]:
# ==============================================================================
# ETTR-CTL-LLAMA-1 — PHASE_1C.2
# LLAMA TARGET-SPAN STRUCTURAL LOCALIZATION AUDIT
#
# PURPOSE
# -------
# Phase 1C.1 established that the original GPT-2 single-token target-position
# operationalization does not transfer uniformly to Llama 3.2 3B.
#
# This cell tests a narrower alternative:
#
#   1. Locate the intended target string in the ORIGINAL CHARACTER-LEVEL PROMPT.
#   2. Require that the intended target occurrence is uniquely identifiable as
#      the FINAL occurrence of that exact target string.
#   3. Map the character span onto the Llama tokenizer's token offsets.
#   4. Determine whether the target span is deterministically localized.
#   5. Determine whether its terminal token position is condition-symmetric.
#   6. Determine whether a terminal-target-token readout could therefore be
#      defined as an OPERATIONAL convention without changing CTL mathematics.
#
# GOVERNANCE
# ----------
# - The authenticated 192-record dataset is frozen.
# - No prompt or record is modified.
# - No model weights are loaded.
# - No inference is performed.
# - No transport map is fitted.
# - No state bank is created.
# - The CTL mathematical specification is untouched.
# - Any mismatch is classified at the operationalization layer.
#
# IMPORTANT
# ---------
# This cell does NOT assume that "last token" is valid.
# It first establishes the exact target character span and then audits the
# token span induced by the authenticated Llama tokenizer.
# ==============================================================================

import os
import gc
import json
import hashlib
from datetime import datetime, timezone
from collections import Counter, defaultdict

import torch
from transformers import AutoTokenizer
from huggingface_hub import get_token

print("=" * 78)
print("ETTR-CTL-LLAMA-1 — PHASE_1C.2")
print("LLAMA TARGET-SPAN STRUCTURAL LOCALIZATION AUDIT")
print("=" * 78)


# ------------------------------------------------------------------------------
# 0. Constants
# ------------------------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
MODEL_ID = "meta-llama/Llama-3.2-3B"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

AUTH_MANIFEST_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

TOKENIZER_AUDIT_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1c1_tokenizer_alignment_audit.json"
)

OUTPUT_MANIFEST_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1c2_target_span_structural_localization_audit.json"
)

EXPECTED_DATASET_SHA = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)

EXPECTED_RECORD_COUNT = 192
EXPECTED_TEMPLATE_COUNT = 12
EXPECTED_PAIR_COUNT = 16
EXPECTED_SPLITS = {
    "calibration": 48,
    "test": 48,
    "train": 96,
}

# ------------------------------------------------------------------------------
# 1. Scientific-state firewall
# ------------------------------------------------------------------------------

print("\n[1/15] Scientific-state firewall")
print("-" * 78)

if "llama_records" not in globals():
    raise RuntimeError(
        "llama_records is not present. Phase 1C.0 authorized dataset must "
        "be present before Phase 1C.2."
    )

if not isinstance(llama_records, list):
    raise RuntimeError("llama_records must be a list.")

if len(llama_records) != EXPECTED_RECORD_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_RECORD_COUNT} records, found {len(llama_records)}."
    )

def canonical_dataset_bytes(records):
    return json.dumps(
        records,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    ).encode("utf-8")

pre_audit_bytes = canonical_dataset_bytes(llama_records)
pre_audit_sha = hashlib.sha256(pre_audit_bytes).hexdigest()

print(f"  llama_records present: True")
print(f"  record count: {len(llama_records)}")
print(f"  canonical pre-audit SHA-256: {pre_audit_sha}")
print(f"  expected authenticated SHA-256: {EXPECTED_DATASET_SHA}")

if pre_audit_sha != EXPECTED_DATASET_SHA:
    raise RuntimeError(
        "Dataset SHA does not match the authenticated Phase 1C.0 artifact. "
        "Stopping before tokenizer/span analysis."
    )

print("  scientific dataset firewall: PASS")


# ------------------------------------------------------------------------------
# 2. Authorization manifest audit
# ------------------------------------------------------------------------------

print("\n[2/15] Dataset authorization manifest audit")
print("-" * 78)

if not os.path.exists(AUTH_MANIFEST_PATH):
    raise FileNotFoundError(
        f"Required authorization manifest not found: {AUTH_MANIFEST_PATH}"
    )

with open(AUTH_MANIFEST_PATH, "r", encoding="utf-8") as f:
    auth_manifest = json.load(f)

print(f"  authorization manifest: {AUTH_MANIFEST_PATH}")
print(f"  manifest keys: {sorted(auth_manifest.keys())}")

dataset_authorized = auth_manifest.get("dataset_authorized", None)
manifest_sha = auth_manifest.get("sha256", None)
manifest_record_count = auth_manifest.get("record_count", None)

print(f"  dataset_authorized: {dataset_authorized}")
print(f"  manifest SHA-256: {manifest_sha}")
print(f"  manifest record count: {manifest_record_count}")

if dataset_authorized is not True:
    raise RuntimeError(
        "Phase 1C.0 does not explicitly authorize the operative dataset."
    )

if manifest_sha != EXPECTED_DATASET_SHA:
    raise RuntimeError(
        "Phase 1C.0 authorization manifest SHA does not match the expected "
        "authenticated dataset SHA."
    )

if manifest_record_count != EXPECTED_RECORD_COUNT:
    raise RuntimeError(
        "Phase 1C.0 authorization manifest record count is inconsistent."
    )

print("  authorization precondition: PASS")


# ------------------------------------------------------------------------------
# 3. Prior Phase 1C.1 audit retrieval
# ------------------------------------------------------------------------------

print("\n[3/15] Prior tokenizer audit retrieval")
print("-" * 78)

prior_audit_exists = os.path.exists(TOKENIZER_AUDIT_PATH)
print(f"  Phase 1C.1 audit exists: {prior_audit_exists}")

if prior_audit_exists:
    with open(TOKENIZER_AUDIT_PATH, "r", encoding="utf-8") as f:
        prior_audit = json.load(f)

    prior_classification = prior_audit.get(
        "classification",
        prior_audit.get("final_classification", None)
    )

    print(f"  Phase 1C.1 classification: {prior_classification}")

    # Do not require a particular schema beyond existence. The present cell
    # independently establishes the structural localization result.
else:
    prior_audit = None
    print("  prior tokenizer audit manifest unavailable; continuing with "
          "independent structural audit.")


# ------------------------------------------------------------------------------
# 4. Dataset schema audit
# ------------------------------------------------------------------------------

print("\n[4/15] Dataset schema and target-field audit")
print("-" * 78)

if not llama_records:
    raise RuntimeError("Dataset is empty.")

required_fields = {
    "clean_prompt",
    "corrupt_prompt",
    "clean_target_name",
    "corrupt_target_name",
}

missing_fields = sorted(
    required_fields - set(llama_records[0].keys())
)

print(f"  first-record fields: {sorted(llama_records[0].keys())}")
print(f"  required fields: {sorted(required_fields)}")

if missing_fields:
    raise RuntimeError(
        f"Required target/prompt fields are missing: {missing_fields}"
    )

schema_ok = all(
    required_fields.issubset(set(record.keys()))
    for record in llama_records
)

print(f"  required fields present in all records: {schema_ok}")

if not schema_ok:
    raise RuntimeError(
        "Not all records contain the required prompt and target fields."
    )

print("  schema audit: PASS")


# ------------------------------------------------------------------------------
# 5. Exact tokenizer authentication and identity
# ------------------------------------------------------------------------------

print("\n[5/15] Exact Llama tokenizer identity audit")
print("-" * 78)

resolved_token = get_token()

if resolved_token is None:
    raise RuntimeError(
        "No Hugging Face credential is available. Phase 1C.1A must pass "
        "before tokenizer loading."
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=resolved_token,
    use_fast=True,
)

print(f"  tokenizer class: {type(tokenizer).__name__}")
print(f"  tokenizer name_or_path: {tokenizer.name_or_path}")
print(f"  tokenizer vocab_size: {tokenizer.vocab_size}")
print(f"  tokenizer model_max_length: {tokenizer.model_max_length}")
print(f"  tokenizer is_fast: {tokenizer.is_fast}")
print(f"  tokenizer identity expected: {MODEL_ID}")

if tokenizer.name_or_path != MODEL_ID:
    print(
        "  WARNING: tokenizer name_or_path differs textually from the "
        "expected model identifier; continuing only if the tokenizer "
        "configuration itself is otherwise valid."
    )

if not tokenizer.is_fast:
    raise RuntimeError(
        "Fast tokenizer required because this audit uses character/token "
        "offset mappings."
    )

print("  tokenizer identity / offset capability: PASS")


# ------------------------------------------------------------------------------
# 6. Character-span localization helper
# ------------------------------------------------------------------------------

print("\n[6/15] Character-level target-span localization setup")
print("-" * 78)

def locate_final_target_span(prompt, target):
    """
    Locate the intended target as the FINAL exact occurrence of the target
    string in the prompt.

    Returns:
        {
            "target": str,
            "occurrence_count": int,
            "first_start": int or None,
            "final_start": int or None,
            "final_end": int or None,
            "unique_final_occurrence": bool,
            "prompt_ends_with_target": bool
        }
    }
    """
    if not isinstance(prompt, str):
        return {
            "target": target,
            "occurrence_count": 0,
            "first_start": None,
            "final_start": None,
            "final_end": None,
            "unique_final_occurrence": False,
            "prompt_ends_with_target": False,
        }

    if not isinstance(target, str) or target == "":
        return {
            "target": target,
            "occurrence_count": 0,
            "first_start": None,
            "final_start": None,
            "final_end": None,
            "unique_final_occurrence": False,
            "prompt_ends_with_target": False,
        }

    starts = []
    cursor = 0

    while True:
        idx = prompt.find(target, cursor)
        if idx == -1:
            break
        starts.append(idx)
        cursor = idx + 1

    if not starts:
        return {
            "target": target,
            "occurrence_count": 0,
            "first_start": None,
            "final_start": None,
            "final_end": None,
            "unique_final_occurrence": False,
            "prompt_ends_with_target": False,
        }

    final_start = starts[-1]
    final_end = final_start + len(target)

    return {
        "target": target,
        "occurrence_count": len(starts),
        "first_start": starts[0],
        "final_start": final_start,
        "final_end": final_end,
        "unique_final_occurrence": True,
        "prompt_ends_with_target": prompt.endswith(target),
    }

print("  localization convention:")
print("    exact target string")
print("    final occurrence in the authenticated prompt")
print("    character span [start, end)")
print("  character localization method: initialized")


# ------------------------------------------------------------------------------
# 7. Token-offset mapping helper
# ------------------------------------------------------------------------------

print("\n[7/15] Token-offset mapping definition")
print("-" * 78)

def map_character_span_to_token_span(
    offsets,
    sequence_ids,
    target_start,
    target_end,
):
    """
    Map a character span [target_start, target_end) onto tokenizer tokens.

    A token belongs to the target span if its character interval overlaps the
    target character interval.

    Special tokens are excluded through sequence_ids when available and by
    requiring positive offsets.
    """
    token_indices = []

    for idx, offset in enumerate(offsets):
        if offset is None:
            continue

        start, end = offset

        # Ignore zero-width / special-token offsets.
        if end <= start:
            continue

        if sequence_ids is not None:
            try:
                sid = sequence_ids[idx]
                if sid is not None and sid != 0:
                    continue
            except Exception:
                pass

        # Interval overlap:
        # token [start,end) intersects target [target_start,target_end)
        if start < target_end and end > target_start:
            token_indices.append(idx)

    if not token_indices:
        return {
            "token_indices": [],
            "start_token": None,
            "end_token": None,
            "token_count": 0,
            "contiguous": False,
        }

    contiguous = token_indices == list(
        range(token_indices[0], token_indices[-1] + 1)
    )

    return {
        "token_indices": token_indices,
        "start_token": token_indices[0],
        "end_token": token_indices[-1],
        "token_count": len(token_indices),
        "contiguous": contiguous,
    }

print("  character-to-token overlap rule: initialized")
print("  token-span contiguity requirement: enabled")


# ------------------------------------------------------------------------------
# 8. Full 192-record structural localization audit
# ------------------------------------------------------------------------------

print("\n[8/15] Full target-span structural localization audit")
print("-" * 78)

audit_records = []

for record_index, record in enumerate(llama_records):

    clean_prompt = record["clean_prompt"]
    corrupt_prompt = record["corrupt_prompt"]

    clean_target = record["clean_target_name"]
    corrupt_target = record["corrupt_target_name"]

    # ---- Character localization ----

    clean_char = locate_final_target_span(
        clean_prompt,
        clean_target
    )

    corrupt_char = locate_final_target_span(
        corrupt_prompt,
        corrupt_target
    )

    # ---- Tokenization with offsets ----

    clean_enc = tokenizer(
        clean_prompt,
        add_special_tokens=False,
        return_offsets_mapping=True,
        return_attention_mask=True,
    )

    corrupt_enc = tokenizer(
        corrupt_prompt,
        add_special_tokens=False,
        return_offsets_mapping=True,
        return_attention_mask=True,
    )

    clean_input_ids = clean_enc["input_ids"]
    corrupt_input_ids = corrupt_enc["input_ids"]

    clean_offsets = clean_enc["offset_mapping"]
    corrupt_offsets = corrupt_enc["offset_mapping"]

    # Tokenizer outputs may be nested for batch mode; this is single-string
    # tokenization, so normalize conservatively.
    if clean_input_ids and isinstance(clean_input_ids[0], list):
        clean_input_ids = clean_input_ids[0]
        clean_offsets = clean_offsets[0]

    if corrupt_input_ids and isinstance(corrupt_input_ids[0], list):
        corrupt_input_ids = corrupt_input_ids[0]
        corrupt_offsets = corrupt_offsets[0]

    clean_sequence_ids = None
    corrupt_sequence_ids = None

    try:
        clean_sequence_ids = clean_enc.sequence_ids()
    except Exception:
        pass

    try:
        corrupt_sequence_ids = corrupt_enc.sequence_ids()
    except Exception:
        pass

    if clean_sequence_ids is not None and clean_sequence_ids and \
       isinstance(clean_sequence_ids[0], list):
        clean_sequence_ids = clean_sequence_ids[0]

    if corrupt_sequence_ids is not None and corrupt_sequence_ids and \
       isinstance(corrupt_sequence_ids[0], list):
        corrupt_sequence_ids = corrupt_sequence_ids[0]

    # ---- Map character spans to token spans ----

    clean_tok = map_character_span_to_token_span(
        clean_offsets,
        clean_sequence_ids,
        clean_char["final_start"],
        clean_char["final_end"],
    ) if clean_char["final_start"] is not None else {
        "token_indices": [],
        "start_token": None,
        "end_token": None,
        "token_count": 0,
        "contiguous": False,
    }

    corrupt_tok = map_character_span_to_token_span(
        corrupt_offsets,
        corrupt_sequence_ids,
        corrupt_char["final_start"],
        corrupt_char["final_end"],
    ) if corrupt_char["final_start"] is not None else {
        "token_indices": [],
        "start_token": None,
        "end_token": None,
        "token_count": 0,
        "contiguous": False,
    }

    # ---- Exact token-span boundary verification ----

    def span_exactly_covers_target(
        offsets,
        token_indices,
        target_start,
        target_end,
    ):
        if not token_indices:
            return False

        first_start = offsets[token_indices[0]][0]
        last_end = offsets[token_indices[-1]][1]

        return (
            first_start == target_start
            and last_end == target_end
        )

    clean_exact_cover = (
        span_exactly_covers_target(
            clean_offsets,
            clean_tok["token_indices"],
            clean_char["final_start"],
            clean_char["final_end"],
        )
        if clean_char["final_start"] is not None
        else False
    )

    corrupt_exact_cover = (
        span_exactly_covers_target(
            corrupt_offsets,
            corrupt_tok["token_indices"],
            corrupt_char["final_start"],
            corrupt_char["final_end"],
        )
        if corrupt_char["final_start"] is not None
        else False
    )

    # ---- Terminal-token localization ----

    clean_terminal = clean_tok["end_token"]
    corrupt_terminal = corrupt_tok["end_token"]

    terminal_position_match = (
        clean_terminal is not None
        and corrupt_terminal is not None
        and clean_terminal == corrupt_terminal
    )

    # ---- Target spans themselves must be the final token spans ----

    clean_terminal_is_prompt_terminal = (
        clean_terminal is not None
        and clean_terminal == len(clean_input_ids) - 1
    )

    corrupt_terminal_is_prompt_terminal = (
        corrupt_terminal is not None
        and corrupt_terminal == len(corrupt_input_ids) - 1
    )

    # ---- Token-length equality remains an independent check ----

    prompt_token_length_equal = (
        len(clean_input_ids) == len(corrupt_input_ids)
    )

    # ---- Record-level structural pass ----

    clean_char_pass = (
        clean_char["occurrence_count"] >= 1
        and clean_char["prompt_ends_with_target"]
        and clean_char["unique_final_occurrence"]
    )

    corrupt_char_pass = (
        corrupt_char["occurrence_count"] >= 1
        and corrupt_char["prompt_ends_with_target"]
        and corrupt_char["unique_final_occurrence"]
    )

    clean_token_pass = (
        clean_tok["token_count"] >= 1
        and clean_tok["contiguous"]
        and clean_exact_cover
    )

    corrupt_token_pass = (
        corrupt_tok["token_count"] >= 1
        and corrupt_tok["contiguous"]
        and corrupt_exact_cover
    )

    record_localization_pass = (
        clean_char_pass
        and corrupt_char_pass
        and clean_token_pass
        and corrupt_token_pass
    )

    record_terminal_convention_pass = (
        record_localization_pass
        and prompt_token_length_equal
        and terminal_position_match
        and clean_terminal_is_prompt_terminal
        and corrupt_terminal_is_prompt_terminal
    )

    audit_records.append({
        "record_index": record_index,
        "example_id": record.get("example_id"),
        "template": record.get("template"),
        "name_pair": record.get("name_pair"),
        "split": record.get("split"),

        "clean_target": clean_target,
        "corrupt_target": corrupt_target,

        "clean_char_occurrence_count": clean_char["occurrence_count"],
        "corrupt_char_occurrence_count": corrupt_char["occurrence_count"],

        "clean_char_start": clean_char["final_start"],
        "clean_char_end": clean_char["final_end"],
        "corrupt_char_start": corrupt_char["final_start"],
        "corrupt_char_end": corrupt_char["final_end"],

        "clean_prompt_ends_with_target": clean_char["prompt_ends_with_target"],
        "corrupt_prompt_ends_with_target": corrupt_char["prompt_ends_with_target"],

        "clean_token_count": clean_tok["token_count"],
        "corrupt_token_count": corrupt_tok["token_count"],

        "clean_token_indices": clean_tok["token_indices"],
        "corrupt_token_indices": corrupt_tok["token_indices"],

        "clean_start_token": clean_tok["start_token"],
        "clean_end_token": clean_tok["end_token"],
        "corrupt_start_token": corrupt_tok["start_token"],
        "corrupt_end_token": corrupt_tok["end_token"],

        "clean_exact_character_cover": clean_exact_cover,
        "corrupt_exact_character_cover": corrupt_exact_cover,

        "clean_contiguous_token_span": clean_tok["contiguous"],
        "corrupt_contiguous_token_span": corrupt_tok["contiguous"],

        "prompt_token_length_equal": prompt_token_length_equal,
        "terminal_position_match": terminal_position_match,

        "clean_terminal_is_prompt_terminal":
            clean_terminal_is_prompt_terminal,

        "corrupt_terminal_is_prompt_terminal":
            corrupt_terminal_is_prompt_terminal,

        "record_localization_pass": record_localization_pass,
        "record_terminal_convention_pass":
            record_terminal_convention_pass,
    })


# ------------------------------------------------------------------------------
# 9. Aggregate localization results
# ------------------------------------------------------------------------------

print("\n[9/15] Aggregate localization results")
print("-" * 78)

N = len(audit_records)

clean_char_pass_count = sum(
    r["clean_char_occurrence_count"] >= 1
    and r["clean_prompt_ends_with_target"]
    for r in audit_records
)

corrupt_char_pass_count = sum(
    r["corrupt_char_occurrence_count"] >= 1
    and r["corrupt_prompt_ends_with_target"]
    for r in audit_records
)

clean_exact_cover_count = sum(
    r["clean_exact_character_cover"]
    and r["clean_contiguous_token_span"]
    for r in audit_records
)

corrupt_exact_cover_count = sum(
    r["corrupt_exact_character_cover"]
    and r["corrupt_contiguous_token_span"]
    for r in audit_records
)

localization_pass_count = sum(
    r["record_localization_pass"]
    for r in audit_records
)

terminal_match_count = sum(
    r["terminal_position_match"]
    for r in audit_records
)

terminal_convention_pass_count = sum(
    r["record_terminal_convention_pass"]
    for r in audit_records
)

prompt_length_equal_count = sum(
    r["prompt_token_length_equal"]
    for r in audit_records
)

print(
    f"  clean final target character spans valid: "
    f"{clean_char_pass_count}/{N}"
)

print(
    f"  corrupt final target character spans valid: "
    f"{corrupt_char_pass_count}/{N}"
)

print(
    f"  clean character→token exact covers: "
    f"{clean_exact_cover_count}/{N}"
)

print(
    f"  corrupt character→token exact covers: "
    f"{corrupt_exact_cover_count}/{N}"
)

print(
    f"  complete target-span localization: "
    f"{localization_pass_count}/{N}"
)

print(
    f"  clean/corrupt terminal token positions match: "
    f"{terminal_match_count}/{N}"
)

print(
    f"  equal prompt token lengths: "
    f"{prompt_length_equal_count}/{N}"
)

print(
    f"  terminal-token convention passes all structural conditions: "
    f"{terminal_convention_pass_count}/{N}"
)


# ------------------------------------------------------------------------------
# 10. Target-token-count and terminal-position distributions
# ------------------------------------------------------------------------------

print("\n[10/15] Target-span and terminal-position distributions")
print("-" * 78)

clean_target_counts = Counter(
    r["clean_token_count"]
    for r in audit_records
)

corrupt_target_counts = Counter(
    r["corrupt_token_count"]
    for r in audit_records
)

clean_terminal_positions = Counter(
    r["clean_end_token"]
    for r in audit_records
    if r["clean_end_token"] is not None
)

corrupt_terminal_positions = Counter(
    r["corrupt_end_token"]
    for r in audit_records
    if r["corrupt_end_token"] is not None
)

terminal_deltas = Counter(
    (
        r["corrupt_end_token"] - r["clean_end_token"]
        if r["clean_end_token"] is not None
        and r["corrupt_end_token"] is not None
        else None
    )
    for r in audit_records
)

print(f"  clean target token-count distribution: "
      f"{dict(sorted(clean_target_counts.items()))}")

print(f"  corrupt target token-count distribution: "
      f"{dict(sorted(corrupt_target_counts.items()))}")

print(
    "  clean terminal-token positions: "
    f"{dict(sorted(clean_terminal_positions.items()))}"
)

print(
    "  corrupt terminal-token positions: "
    f"{dict(sorted(corrupt_terminal_positions.items()))}"
)

print(
    "  corrupt-minus-clean terminal-position delta: "
    f"{dict(sorted(terminal_deltas.items(), key=lambda x: str(x[0])))}"
)


# ------------------------------------------------------------------------------
# 11. Condition-symmetry and structural diagnostics
# ------------------------------------------------------------------------------

print("\n[11/15] Condition-symmetry diagnostics")
print("-" * 78)

clean_terminal_set = {
    r["clean_end_token"]
    for r in audit_records
    if r["clean_end_token"] is not None
}

corrupt_terminal_set = {
    r["corrupt_end_token"]
    for r in audit_records
    if r["corrupt_end_token"] is not None
}

clean_span_count_set = {
    r["clean_token_count"]
    for r in audit_records
}

corrupt_span_count_set = {
    r["corrupt_token_count"]
    for r in audit_records
}

print(f"  distinct clean terminal positions: {sorted(clean_terminal_set)}")
print(
    f"  distinct corrupt terminal positions: "
    f"{sorted(corrupt_terminal_set)}"
)

print(
    f"  clean target-span token counts observed: "
    f"{sorted(clean_span_count_set)}"
)

print(
    f"  corrupt target-span token counts observed: "
    f"{sorted(corrupt_span_count_set)}"
)

# A key distinction:
#
# Target spans may legitimately contain multiple tokens. That alone does NOT
# invalidate structural localization.
#
# What matters for the proposed terminal-token convention is whether the
# terminal token of the exact target span is deterministically identifiable and
# condition-symmetric.

all_localized = localization_pass_count == N
all_terminal_match = terminal_match_count == N
all_prompt_lengths_equal = prompt_length_equal_count == N
all_terminal_prompt_final = all(
    r["clean_terminal_is_prompt_terminal"]
    and r["corrupt_terminal_is_prompt_terminal"]
    for r in audit_records
)

print(f"  all records structurally localized: {all_localized}")
print(f"  all records terminal-position symmetric: {all_terminal_match}")
print(f"  all records prompt-length symmetric: {all_prompt_lengths_equal}")
print(
    f"  all target spans terminate at final prompt token: "
    f"{all_terminal_prompt_final}"
)


# ------------------------------------------------------------------------------
# 12. Template/pair diagnostics for localization
# ------------------------------------------------------------------------------

print("\n[12/15] Template/pair localization diagnostics")
print("-" * 78)

template_stats = defaultdict(lambda: {
    "n": 0,
    "localized": 0,
    "terminal_match": 0,
    "terminal_convention_pass": 0,
})

pair_stats = defaultdict(lambda: {
    "n": 0,
    "localized": 0,
    "terminal_match": 0,
    "terminal_convention_pass": 0,
})

for r in audit_records:
    template = r["template"]
    pair = r["name_pair"]

    template_stats[template]["n"] += 1
    template_stats[template]["localized"] += int(
        r["record_localization_pass"]
    )
    template_stats[template]["terminal_match"] += int(
        r["terminal_position_match"]
    )
    template_stats[template]["terminal_convention_pass"] += int(
        r["record_terminal_convention_pass"]
    )

    pair_stats[pair]["n"] += 1
    pair_stats[pair]["localized"] += int(
        r["record_localization_pass"]
    )
    pair_stats[pair]["terminal_match"] += int(
        r["terminal_position_match"]
    )
    pair_stats[pair]["terminal_convention_pass"] += int(
        r["record_terminal_convention_pass"]
    )

template_failures = {
    str(k): dict(v)
    for k, v in template_stats.items()
    if v["localized"] < v["n"]
    or v["terminal_match"] < v["n"]
    or v["terminal_convention_pass"] < v["n"]
}

pair_failures = {
    str(k): dict(v)
    for k, v in pair_stats.items()
    if v["localized"] < v["n"]
    or v["terminal_match"] < v["n"]
    or v["terminal_convention_pass"] < v["n"]
}

print(
    f"  templates with any localization/terminal failure: "
    f"{len(template_failures)}/{len(template_stats)}"
)

print(
    f"  name-pairs with any localization/terminal failure: "
    f"{len(pair_failures)}/{len(pair_stats)}"
)

if template_failures:
    print("  template failure summary:")
    for k, v in sorted(template_failures.items()):
        print(f"    {k}: {v}")

if pair_failures:
    print("  pair failure summary:")
    for k, v in sorted(pair_failures.items()):
        print(f"    {k}: {v}")


# ------------------------------------------------------------------------------
# 13. Identify exact failure modes, if any
# ------------------------------------------------------------------------------

print("\n[13/15] Failure-mode classification")
print("-" * 78)

failure_records = []

for r in audit_records:

    reasons = []

    if r["clean_char_occurrence_count"] == 0:
        reasons.append("CLEAN_TARGET_NOT_FOUND")

    if r["corrupt_char_occurrence_count"] == 0:
        reasons.append("CORRUPT_TARGET_NOT_FOUND")

    if not r["clean_prompt_ends_with_target"]:
        reasons.append("CLEAN_TARGET_NOT_AT_PROMPT_END")

    if not r["corrupt_prompt_ends_with_target"]:
        reasons.append("CORRUPT_TARGET_NOT_AT_PROMPT_END")

    if not r["clean_contiguous_token_span"]:
        reasons.append("CLEAN_TARGET_TOKEN_SPAN_NONCONTIGUOUS")

    if not r["corrupt_contiguous_token_span"]:
        reasons.append("CORRUPT_TARGET_TOKEN_SPAN_NONCONTIGUOUS")

    if not r["clean_exact_character_cover"]:
        reasons.append("CLEAN_TOKEN_OFFSETS_DO_NOT_EXACTLY_COVER_TARGET")

    if not r["corrupt_exact_character_cover"]:
        reasons.append("CORRUPT_TOKEN_OFFSETS_DO_NOT_EXACTLY_COVER_TARGET")

    if not r["prompt_token_length_equal"]:
        reasons.append("CLEAN_CORRUPT_PROMPT_TOKEN_LENGTH_MISMATCH")

    if not r["terminal_position_match"]:
        reasons.append("TERMINAL_POSITION_ASYMMETRY")

    if not r["clean_terminal_is_prompt_terminal"]:
        reasons.append("CLEAN_TARGET_NOT_TERMINAL_TOKEN")

    if not r["corrupt_terminal_is_prompt_terminal"]:
        reasons.append("CORRUPT_TARGET_NOT_TERMINAL_TOKEN")

    if reasons:
        failure_records.append({
            "record_index": r["record_index"],
            "example_id": r["example_id"],
            "template": r["template"],
            "name_pair": r["name_pair"],
            "split": r["split"],
            "reasons": reasons,
        })

failure_reason_counts = Counter()

for item in failure_records:
    for reason in item["reasons"]:
        failure_reason_counts[reason] += 1

print(f"  records with any failure: {len(failure_records)}/{N}")

if failure_reason_counts:
    print("  failure reasons:")
    for reason, count in failure_reason_counts.most_common():
        print(f"    {reason}: {count}")

else:
    print("  no structural failure modes detected.")


# ------------------------------------------------------------------------------
# 14. Scientific / operational classification
# ------------------------------------------------------------------------------

print("\n[14/15] Scientific / operational classification")
print("-" * 78)

if all_localized and all_terminal_match and all_prompt_lengths_equal:
    classification = "TARGET_SPAN_LOCALIZATION_PASS"

    interpretation = (
        "The intended target spans are deterministically recoverable from the "
        "authenticated prompts and map to contiguous Llama token spans. "
        "Their terminal token positions are condition-symmetric across all "
        "192 clean/corrupt pairs. A terminal-token target-position convention "
        "is therefore operationally available for subsequent state extraction, "
        "subject to the separate requirement that the convention itself be "
        "audited and documented before scientific use."
    )

elif all_localized:
    classification = "TARGET_SPAN_LOCALIZATION_PASS__TERMINAL_CONVENTION_NOT_UNIFORM"

    interpretation = (
        "The intended target spans can be deterministically localized under "
        "the Llama tokenizer, but terminal token positions are not uniformly "
        "condition-symmetric. Therefore target-span localization succeeds, "
        "but a simple terminal-token readout cannot yet be treated as a "
        "uniform cross-condition operational convention."
    )

else:
    classification = "OPERATIONAL_DECOHERENCE"

    interpretation = (
        "The intended target spans cannot be deterministically localized "
        "across all authenticated clean/corrupt prompts under the Llama "
        "tokenizer. The CTL mathematical specification and authenticated "
        "dataset remain unchanged; the mismatch is classified strictly at "
        "the operationalization layer."
    )

print(f"  classification: {classification}")
print(f"  interpretation: {interpretation}")


# ------------------------------------------------------------------------------
# 15. Post-audit integrity, manifest, and cleanup
# ------------------------------------------------------------------------------

print("\n[15/15] Post-audit integrity and manifest")
print("-" * 78)

post_audit_bytes = canonical_dataset_bytes(llama_records)
post_audit_sha = hashlib.sha256(post_audit_bytes).hexdigest()

dataset_unchanged = (
    post_audit_sha == pre_audit_sha == EXPECTED_DATASET_SHA
)

print(f"  pre-audit dataset SHA-256:  {pre_audit_sha}")
print(f"  post-audit dataset SHA-256: {post_audit_sha}")
print(f"  records unchanged: {dataset_unchanged}")

if not dataset_unchanged:
    raise RuntimeError(
        "CRITICAL: dataset integrity changed during Phase 1C.2. "
        "Do not proceed to subsequent scientific stages."
    )

manifest = {
    "experiment_id": EXPERIMENT_ID,
    "phase": "PHASE_1C.2",
    "audit": "LLAMA_TARGET_SPAN_STRUCTURAL_LOCALIZATION_AUDIT",

    "timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "model_id": MODEL_ID,

    "dataset": {
        "record_count": N,
        "expected_record_count": EXPECTED_RECORD_COUNT,
        "sha256_pre_audit": pre_audit_sha,
        "sha256_post_audit": post_audit_sha,
        "expected_authenticated_sha256": EXPECTED_DATASET_SHA,
        "unchanged": dataset_unchanged,
    },

    "authorization": {
        "authorization_manifest": AUTH_MANIFEST_PATH,
        "dataset_authorized": dataset_authorized,
    },

    "tokenizer": {
        "name_or_path": tokenizer.name_or_path,
        "class": type(tokenizer).__name__,
        "is_fast": bool(tokenizer.is_fast),
        "vocab_size": tokenizer.vocab_size,
        "model_max_length": tokenizer.model_max_length,
        "add_special_tokens": False,
        "offset_mapping_used": True,
    },

    "localization_definition": {
        "target_source": "authenticated record target-name field",
        "character_span": "[final exact occurrence start, end)",
        "token_mapping": "character/token offset interval overlap",
        "contiguous_token_span_required": True,
        "exact_character_cover_required": True,
        "terminal_token_readout_tested": True,
    },

    "aggregate_results": {
        "clean_character_spans_valid": clean_char_pass_count,
        "corrupt_character_spans_valid": corrupt_char_pass_count,
        "clean_exact_token_covers": clean_exact_cover_count,
        "corrupt_exact_token_covers": corrupt_exact_cover_count,
        "complete_target_span_localization": localization_pass_count,
        "terminal_position_matches": terminal_match_count,
        "prompt_token_length_matches": prompt_length_equal_count,
        "terminal_convention_passes": terminal_convention_pass_count,
        "total_records": N,
    },

    "distributions": {
        "clean_target_token_counts":
            dict(sorted(clean_target_counts.items())),
        "corrupt_target_token_counts":
            dict(sorted(corrupt_target_counts.items())),
        "clean_terminal_positions":
            dict(sorted(clean_terminal_positions.items())),
        "corrupt_terminal_positions":
            dict(sorted(corrupt_terminal_positions.items())),
        "terminal_position_deltas":
            {
                str(k): v
                for k, v in terminal_deltas.items()
            },
    },

    "template_failures": template_failures,
    "pair_failures": pair_failures,

    "failure_reason_counts": dict(failure_reason_counts),

    "classification": classification,
    "interpretation": interpretation,

    "scientific_governance": {
        "dataset_modified": False,
        "prompts_modified": False,
        "model_weights_loaded": False,
        "model_inference_performed": False,
        "transport_fitting_performed": False,
        "state_bank_created": False,
        "ctl_mathematics_modified": False,
        "operationalization_layer_only": True,
    },

    "next_stage": (
        "PHASE_1C.3_TARGET_POSITION_OPERATIONALIZATION_AUDIT"
        if classification == "TARGET_SPAN_LOCALIZATION_PASS"
        else "HOLD_PENDING_OPERATIONAL_RESOLUTION"
    ),
}

with open(OUTPUT_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )

# Release tokenizer objects without touching scientific dataset state.
try:
    del tokenizer
except Exception:
    pass

try:
    del resolved_token
except Exception:
    pass

gc.collect()

print(f"  manifest: {OUTPUT_MANIFEST_PATH}")
print(f"  manifest written: True")

print("\n" + "=" * 78)
print("FINAL STATUS")
print("=" * 78)

print(f"  classification: {classification}")
print(
    f"  complete target-span localization: "
    f"{localization_pass_count}/{N}"
)
print(
    f"  terminal-position matches: "
    f"{terminal_match_count}/{N}"
)
print(
    f"  equal prompt token lengths: "
    f"{prompt_length_equal_count}/{N}"
)
print(
    f"  terminal-token convention passes: "
    f"{terminal_convention_pass_count}/{N}"
)
print(f"  dataset mutated: {not dataset_unchanged}")
print(f"  behavioral inference performed: False")
print(f"  transport fitting performed: False")
print(f"  state bank created: False")
print(f"  manifest: {OUTPUT_MANIFEST_PATH}")

print("=" * 78)
print("Phase 1C.2 target-span structural localization audit complete.")
print(
    "No scientific state, model weights, prompts, authenticated dataset "
    "records, or CTL mathematical definitions were altered."
)
print("=" * 78)

ETTR-CTL-LLAMA-1 — PHASE_1C.2
LLAMA TARGET-SPAN STRUCTURAL LOCALIZATION AUDIT

[1/15] Scientific-state firewall
------------------------------------------------------------------------------
  llama_records present: True
  record count: 192
  canonical pre-audit SHA-256: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  expected authenticated SHA-256: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  scientific dataset firewall: PASS

[2/15] Dataset authorization manifest audit
------------------------------------------------------------------------------
  authorization manifest: /content/ettr_ctl_llama/results/llama_phase1c0_recovered_dataset_authorization_v2.json
  manifest keys: ['authorization_status', 'condition_pair', 'cryptographic_integrity', 'dataset_authorized', 'experiment_id', 'governance', 'historical_sha256', 'historical_sha_used_for_authorization', 'name_pair_count', 'next_stage', 'record_count', 'scientific_integrity', 'sha256', 'sourc

In [11]:
# ==============================================================================
# ETTR-CTL-LLAMA-1 — PHASE_1C.2R
# CORRECTED LLAMA TARGET-CONTINUATION / PREDICTION-BOUNDARY AUDIT
#
# IMPORTANT CORRECTION
# --------------------
# Phase 1C.2 incorrectly treated the target name as though it were already
# present inside the prompt. In the experimental design, however, the prompt
# terminates immediately BEFORE the target name:
#
#     "... {a} gave a book to"
#
# and the target name is the intended continuation.
#
# Therefore:
#
#   - target-string search inside the prompt is NOT the correct localization
#     criterion;
#   - the scientifically relevant boundary is the final input-token position;
#   - the target must be analyzed as a hypothetical continuation of the exact
#     prompt;
#   - multi-token target names are allowed and must be characterized explicitly.
#
# This corrected audit:
#
#   1. verifies the authenticated dataset is unchanged;
#   2. loads the exact authenticated Llama tokenizer;
#   3. tokenizes every exact clean/corrupt prompt with
#      add_special_tokens=False;
#   4. establishes the next-token prediction boundary;
#   5. constructs prompt + " " + target for structural continuation analysis;
#   6. verifies that the original prompt is an exact token-prefix of the
#      constructed target continuation;
#   7. determines the complete target continuation token span;
#   8. identifies the first target token and terminal target token;
#   9. tests clean/corrupt boundary symmetry;
#  10. tests whether target continuation token counts are balanced;
#  11. tests whether first-target-token localization is deterministic;
#  12. DOES NOT perform model inference;
#  13. DOES NOT fit transport maps;
#  14. DOES NOT create a state bank;
#  15. DOES NOT modify llama_records or CTL mathematics.
#
# GOVERNANCE
# ----------
# Mathematical specification remains authoritative.
# Dataset remains frozen.
# Any remaining mismatch is classified at the operationalization layer.
# ==============================================================================

import os
import gc
import json
import hashlib
from datetime import datetime, timezone
from collections import Counter, defaultdict

from transformers import AutoTokenizer
from huggingface_hub import get_token


print("=" * 78)
print("ETTR-CTL-LLAMA-1 — PHASE_1C.2R")
print("CORRECTED LLAMA TARGET-CONTINUATION / PREDICTION-BOUNDARY AUDIT")
print("=" * 78)


# ------------------------------------------------------------------------------
# 0. Constants
# ------------------------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
MODEL_ID = "meta-llama/Llama-3.2-3B"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

AUTH_MANIFEST_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

PREVIOUS_AUDIT_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1c2_target_span_structural_localization_audit.json"
)

OUTPUT_MANIFEST_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1c2r_corrected_target_continuation_audit.json"
)

EXPECTED_DATASET_SHA = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)

EXPECTED_RECORD_COUNT = 192
EXPECTED_TEMPLATE_COUNT = 12
EXPECTED_PAIR_COUNT = 16

EXPECTED_SPLITS = {
    "calibration": 48,
    "test": 48,
    "train": 96,
}


# ------------------------------------------------------------------------------
# 1. Scientific-state firewall
# ------------------------------------------------------------------------------

print("\n[1/16] Scientific-state firewall")
print("-" * 78)

if "llama_records" not in globals():
    raise RuntimeError(
        "llama_records is absent. Phase 1C.0 authenticated dataset is required."
    )

if not isinstance(llama_records, list):
    raise RuntimeError("llama_records must be a list.")

if len(llama_records) != EXPECTED_RECORD_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_RECORD_COUNT} records; found {len(llama_records)}."
    )


def canonical_dataset_bytes(records):
    return json.dumps(
        records,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    ).encode("utf-8")


pre_audit_bytes = canonical_dataset_bytes(llama_records)
pre_audit_sha = hashlib.sha256(pre_audit_bytes).hexdigest()

print(f"  llama_records present: True")
print(f"  record count: {len(llama_records)}")
print(f"  pre-audit canonical SHA-256: {pre_audit_sha}")
print(f"  expected authenticated SHA-256: {EXPECTED_DATASET_SHA}")

if pre_audit_sha != EXPECTED_DATASET_SHA:
    raise RuntimeError(
        "Authenticated dataset SHA mismatch. Scientific firewall failed."
    )

print("  scientific dataset firewall: PASS")


# ------------------------------------------------------------------------------
# 2. Authorization manifest
# ------------------------------------------------------------------------------

print("\n[2/16] Dataset authorization manifest audit")
print("-" * 78)

if not os.path.exists(AUTH_MANIFEST_PATH):
    raise FileNotFoundError(AUTH_MANIFEST_PATH)

with open(AUTH_MANIFEST_PATH, "r", encoding="utf-8") as f:
    auth_manifest = json.load(f)

dataset_authorized = auth_manifest.get("dataset_authorized")
manifest_sha = auth_manifest.get("sha256")
manifest_count = auth_manifest.get("record_count")

print(f"  dataset_authorized: {dataset_authorized}")
print(f"  manifest SHA-256: {manifest_sha}")
print(f"  manifest record count: {manifest_count}")

if dataset_authorized is not True:
    raise RuntimeError(
        "Phase 1C.0 does not authorize the operative dataset."
    )

if manifest_sha != EXPECTED_DATASET_SHA:
    raise RuntimeError(
        "Phase 1C.0 manifest SHA does not match authenticated dataset."
    )

if manifest_count != EXPECTED_RECORD_COUNT:
    raise RuntimeError(
        "Phase 1C.0 manifest record count mismatch."
    )

print("  authorization precondition: PASS")


# ------------------------------------------------------------------------------
# 3. Explicit methodological correction record
# ------------------------------------------------------------------------------

print("\n[3/16] Previous Phase 1C.2 methodological interpretation")
print("-" * 78)

previous_audit_exists = os.path.exists(PREVIOUS_AUDIT_PATH)

print(f"  previous Phase 1C.2 manifest exists: {previous_audit_exists}")

correction_statement = (
    "The previous Phase 1C.2 audit searched for the target name inside the "
    "prompt. This is not the experimental target representation: the target "
    "name is the intended continuation after the prompt prediction boundary. "
    "Therefore target-string absence from the prompt is not evidence of "
    "operational decoherence. This corrected audit evaluates target "
    "continuations relative to the exact prompt boundary."
)

print("  methodological correction:")
print("    target is a predicted continuation, not an in-prompt character span.")
print("  previous 0/192 localization result is NOT treated as scientific failure.")
print("  corrected operationalization: prediction-boundary continuation audit.")


# ------------------------------------------------------------------------------
# 4. Dataset schema and experimental semantics
# ------------------------------------------------------------------------------

print("\n[4/16] Dataset schema and target-continuation semantics")
print("-" * 78)

required_fields = {
    "clean_prompt",
    "corrupt_prompt",
    "clean_target_name",
    "corrupt_target_name",
    "condition_pair",
    "split",
}

missing = sorted(
    required_fields - set(llama_records[0].keys())
)

if missing:
    raise RuntimeError(f"Missing required fields: {missing}")

all_schema_ok = all(
    required_fields.issubset(record.keys())
    for record in llama_records
)

if not all_schema_ok:
    raise RuntimeError(
        "Required fields are not present in every record."
    )

condition_values = Counter(
    record["condition_pair"]
    for record in llama_records
)

split_values = Counter(
    record["split"]
    for record in llama_records
)

print(f"  required schema: PASS")
print(f"  condition_pair distribution: {dict(condition_values)}")
print(f"  split distribution: {dict(split_values)}")

if condition_values != Counter({"clean_vs_corrupt": EXPECTED_RECORD_COUNT}):
    raise RuntimeError(
        "Unexpected condition_pair distribution."
    )

if split_values != Counter(EXPECTED_SPLITS):
    raise RuntimeError(
        "Unexpected train/calibration/test partition."
    )

print("  experimental semantics: PASS")


# ------------------------------------------------------------------------------
# 5. Exact Llama tokenizer
# ------------------------------------------------------------------------------

print("\n[5/16] Exact Llama tokenizer identity")
print("-" * 78)

resolved_token = get_token()

if resolved_token is None:
    raise RuntimeError(
        "No Hugging Face credential available. Phase 1C.1A must pass first."
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=resolved_token,
    use_fast=True,
)

print(f"  tokenizer class: {type(tokenizer).__name__}")
print(f"  tokenizer name_or_path: {tokenizer.name_or_path}")
print(f"  tokenizer vocab_size: {tokenizer.vocab_size}")
print(f"  tokenizer is_fast: {tokenizer.is_fast}")
print(f"  tokenizer convention: add_special_tokens=False")

if not tokenizer.is_fast:
    raise RuntimeError(
        "Fast tokenizer required for offset/prefix diagnostics."
    )

print("  tokenizer identity: PASS")


# ------------------------------------------------------------------------------
# 6. Prediction-boundary principle
# ------------------------------------------------------------------------------

print("\n[6/16] Prediction-boundary definition")
print("-" * 78)

boundary_definition = {
    "prompt_tokenization": "exact prompt with add_special_tokens=False",
    "input_positions": "0 ... L-1",
    "next_token_prediction_position": "L",
    "first_target_continuation_token": "position L in hypothetical continuation",
    "target_continuation": "prompt + one separating space + target name",
}

print("  prompt tokens represent observed input.")
print("  target begins after the final prompt token.")
print("  next-token target position = prompt token length L.")
print("  target names are analyzed as continuation spans, not in-prompt spans.")
print("  boundary definition: PASS")


# ------------------------------------------------------------------------------
# 7. Tokenization helpers
# ------------------------------------------------------------------------------

print("\n[7/16] Continuation tokenization helpers")
print("-" * 78)


def normalize_single_encoding(enc):
    """
    Normalize a single-string tokenizer output to flat lists.
    """
    input_ids = enc["input_ids"]
    offsets = enc.get("offset_mapping")

    if input_ids and isinstance(input_ids[0], list):
        input_ids = input_ids[0]

    if offsets is not None and offsets and isinstance(offsets[0], list):
        offsets = offsets[0]

    return input_ids, offsets


def tokenize_text(text):
    enc = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
        return_attention_mask=True,
    )
    return normalize_single_encoding(enc)


def target_continuation_analysis(prompt, target):
    """
    Analyze target as a continuation of the prompt.

    We use:
        full_text = prompt + " " + target

    and independently tokenize:
        prompt
        full_text

    The prompt token sequence must be an exact prefix of the full sequence
    for the continuation boundary to be structurally well-defined under
    this explicit textual continuation convention.

    The continuation tokens are then:
        full_ids[prompt_len:]

    This is a structural tokenizer test only. No model inference occurs.
    """

    prompt_ids, prompt_offsets = tokenize_text(prompt)

    full_text = prompt + " " + target
    full_ids, full_offsets = tokenize_text(full_text)

    prompt_len = len(prompt_ids)

    prefix_match = (
        len(full_ids) >= prompt_len
        and full_ids[:prompt_len] == prompt_ids
    )

    continuation_ids = (
        full_ids[prompt_len:]
        if prefix_match
        else []
    )

    continuation_offsets = (
        full_offsets[prompt_len:]
        if prefix_match and full_offsets is not None
        else []
    )

    # Character span of the hypothetical target inside full_text.
    target_start = len(prompt) + 1
    target_end = target_start + len(target)

    exact_target_character_cover = False

    if continuation_offsets:
        first_start = continuation_offsets[0][0]
        last_end = continuation_offsets[-1][1]

        exact_target_character_cover = (
            first_start == target_start
            and last_end == target_end
        )

    # Tokenization of target continuation as a complete span.
    continuation_count = len(continuation_ids)

    return {
        "prompt_ids": prompt_ids,
        "full_ids": full_ids,
        "prompt_length": prompt_len,
        "full_length": len(full_ids),
        "prefix_match": prefix_match,
        "continuation_ids": continuation_ids,
        "continuation_offsets": continuation_offsets,
        "continuation_count": continuation_count,
        "target_start": target_start,
        "target_end": target_end,
        "exact_target_character_cover": exact_target_character_cover,
        "first_target_position": (
            prompt_len if continuation_count > 0 else None
        ),
        "terminal_target_position": (
            prompt_len + continuation_count - 1
            if continuation_count > 0
            else None
        ),
    }


print("  continuation tokenizer helper: initialized")
print("  prompt-prefix criterion: enabled")
print("  hypothetical target character-span criterion: enabled")


# ------------------------------------------------------------------------------
# 8. Full 192-record audit
# ------------------------------------------------------------------------------

print("\n[8/16] Full 192-record target-continuation audit")
print("-" * 78)

audit_records = []

for record_index, record in enumerate(llama_records):

    clean_prompt = record["clean_prompt"]
    corrupt_prompt = record["corrupt_prompt"]

    clean_target = record["clean_target_name"]
    corrupt_target = record["corrupt_target_name"]

    clean = target_continuation_analysis(
        clean_prompt,
        clean_target,
    )

    corrupt = target_continuation_analysis(
        corrupt_prompt,
        corrupt_target,
    )

    clean_prompt_len = clean["prompt_length"]
    corrupt_prompt_len = corrupt["prompt_length"]

    prompt_length_equal = (
        clean_prompt_len == corrupt_prompt_len
    )

    first_target_position_match = (
        clean["first_target_position"] is not None
        and corrupt["first_target_position"] is not None
        and clean["first_target_position"]
        == corrupt["first_target_position"]
    )

    terminal_target_position_match = (
        clean["terminal_target_position"] is not None
        and corrupt["terminal_target_position"] is not None
        and clean["terminal_target_position"]
        == corrupt["terminal_target_position"]
    )

    continuation_count_equal = (
        clean["continuation_count"]
        == corrupt["continuation_count"]
    )

    clean_pass = (
        clean["prefix_match"]
        and clean["continuation_count"] >= 1
        and clean["exact_target_character_cover"]
    )

    corrupt_pass = (
        corrupt["prefix_match"]
        and corrupt["continuation_count"] >= 1
        and corrupt["exact_target_character_cover"]
    )

    complete_structural_pass = (
        clean_pass
        and corrupt_pass
        and prompt_length_equal
        and first_target_position_match
        and terminal_target_position_match
    )

    audit_records.append({
        "record_index": record_index,
        "example_id": record.get("example_id"),
        "template": record.get("template"),
        "template_id": record.get("template_id"),
        "name_pair": record.get("name_pair"),
        "name_pair_id": record.get("name_pair_id"),
        "split": record.get("split"),

        "clean_target": clean_target,
        "corrupt_target": corrupt_target,

        "clean_prompt_length": clean_prompt_len,
        "corrupt_prompt_length": corrupt_prompt_len,

        "clean_prefix_match": clean["prefix_match"],
        "corrupt_prefix_match": corrupt["prefix_match"],

        "clean_continuation_ids": clean["continuation_ids"],
        "corrupt_continuation_ids": corrupt["continuation_ids"],

        "clean_continuation_count": clean["continuation_count"],
        "corrupt_continuation_count": corrupt["continuation_count"],

        "clean_first_target_position":
            clean["first_target_position"],
        "corrupt_first_target_position":
            corrupt["first_target_position"],

        "clean_terminal_target_position":
            clean["terminal_target_position"],
        "corrupt_terminal_target_position":
            corrupt["terminal_target_position"],

        "clean_exact_target_character_cover":
            clean["exact_target_character_cover"],
        "corrupt_exact_target_character_cover":
            corrupt["exact_target_character_cover"],

        "prompt_length_equal": prompt_length_equal,
        "first_target_position_match":
            first_target_position_match,
        "terminal_target_position_match":
            terminal_target_position_match,
        "continuation_count_equal":
            continuation_count_equal,

        "complete_structural_pass":
            complete_structural_pass,
    })


# ------------------------------------------------------------------------------
# 9. Aggregate structural results
# ------------------------------------------------------------------------------

print("\n[9/16] Aggregate target-continuation results")
print("-" * 78)

N = len(audit_records)

clean_prefix_pass = sum(
    r["clean_prefix_match"]
    for r in audit_records
)

corrupt_prefix_pass = sum(
    r["corrupt_prefix_match"]
    for r in audit_records
)

clean_character_cover_pass = sum(
    r["clean_exact_target_character_cover"]
    for r in audit_records
)

corrupt_character_cover_pass = sum(
    r["corrupt_exact_target_character_cover"]
    for r in audit_records
)

prompt_length_equal_count = sum(
    r["prompt_length_equal"]
    for r in audit_records
)

first_position_match_count = sum(
    r["first_target_position_match"]
    for r in audit_records
)

terminal_position_match_count = sum(
    r["terminal_target_position_match"]
    for r in audit_records
)

continuation_count_equal_count = sum(
    r["continuation_count_equal"]
    for r in audit_records
)

complete_pass_count = sum(
    r["complete_structural_pass"]
    for r in audit_records
)

print(
    f"  clean prompt exact-prefix of target continuation: "
    f"{clean_prefix_pass}/{N}"
)

print(
    f"  corrupt prompt exact-prefix of target continuation: "
    f"{corrupt_prefix_pass}/{N}"
)

print(
    f"  clean target continuation character spans exact: "
    f"{clean_character_cover_pass}/{N}"
)

print(
    f"  corrupt target continuation character spans exact: "
    f"{corrupt_character_cover_pass}/{N}"
)

print(
    f"  clean/corrupt prompt lengths equal: "
    f"{prompt_length_equal_count}/{N}"
)

print(
    f"  clean/corrupt first-target positions match: "
    f"{first_position_match_count}/{N}"
)

print(
    f"  clean/corrupt terminal-target positions match: "
    f"{terminal_position_match_count}/{N}"
)

print(
    f"  clean/corrupt target continuation token counts equal: "
    f"{continuation_count_equal_count}/{N}"
)

print(
    f"  complete structural target-continuation pass: "
    f"{complete_pass_count}/{N}"
)


# ------------------------------------------------------------------------------
# 10. Continuation token-count distributions
# ------------------------------------------------------------------------------

print("\n[10/16] Target-continuation token-count distributions")
print("-" * 78)

clean_counts = Counter(
    r["clean_continuation_count"]
    for r in audit_records
)

corrupt_counts = Counter(
    r["corrupt_continuation_count"]
    for r in audit_records
)

paired_count_deltas = Counter(
    r["corrupt_continuation_count"]
    - r["clean_continuation_count"]
    for r in audit_records
)

print(
    f"  clean continuation token counts: "
    f"{dict(sorted(clean_counts.items()))}"
)

print(
    f"  corrupt continuation token counts: "
    f"{dict(sorted(corrupt_counts.items()))}"
)

print(
    f"  corrupt-minus-clean continuation count: "
    f"{dict(sorted(paired_count_deltas.items()))}"
)


# ------------------------------------------------------------------------------
# 11. Prediction-boundary distributions
# ------------------------------------------------------------------------------

print("\n[11/16] Prediction-boundary distributions")
print("-" * 78)

clean_prompt_positions = Counter(
    r["clean_first_target_position"]
    for r in audit_records
)

corrupt_prompt_positions = Counter(
    r["corrupt_first_target_position"]
    for r in audit_records
)

clean_terminal_positions = Counter(
    r["clean_terminal_target_position"]
    for r in audit_records
)

corrupt_terminal_positions = Counter(
    r["corrupt_terminal_target_position"]
    for r in audit_records
)

print(
    f"  clean first-target positions: "
    f"{dict(sorted(clean_prompt_positions.items()))}"
)

print(
    f"  corrupt first-target positions: "
    f"{dict(sorted(corrupt_prompt_positions.items()))}"
)

print(
    f"  clean terminal-target positions: "
    f"{dict(sorted(clean_terminal_positions.items()))}"
)

print(
    f"  corrupt terminal-target positions: "
    f"{dict(sorted(corrupt_terminal_positions.items()))}"
)


# ------------------------------------------------------------------------------
# 12. Name-level continuation diagnostics
# ------------------------------------------------------------------------------

print("\n[12/16] Name-level continuation diagnostics")
print("-" * 78)

name_diagnostics = defaultdict(
    lambda: {
        "clean_occurrences": 0,
        "corrupt_occurrences": 0,
        "clean_token_counts": Counter(),
        "corrupt_token_counts": Counter(),
        "clean_ids": set(),
        "corrupt_ids": set(),
    }
)

for r in audit_records:

    name_diagnostics[r["clean_target"]]["clean_occurrences"] += 1

    name_diagnostics[r["clean_target"]]["clean_token_counts"][
        r["clean_continuation_count"]
    ] += 1

    if r["clean_continuation_ids"]:
        name_diagnostics[r["clean_target"]]["clean_ids"].add(
            tuple(r["clean_continuation_ids"])
        )

    name_diagnostics[r["corrupt_target"]]["corrupt_occurrences"] += 1

    name_diagnostics[r["corrupt_target"]]["corrupt_token_counts"][
        r["corrupt_continuation_count"]
    ] += 1

    if r["corrupt_continuation_ids"]:
        name_diagnostics[r["corrupt_target"]]["corrupt_ids"].add(
            tuple(r["corrupt_continuation_ids"])
        )


# Print all names in deterministic order.
for name in sorted(name_diagnostics):

    d = name_diagnostics[name]

    clean_ids = sorted(
        list(d["clean_ids"])
    )

    corrupt_ids = sorted(
        list(d["corrupt_ids"])
    )

    print(
        f"  {name}: "
        f"clean_n={d['clean_occurrences']}, "
        f"corrupt_n={d['corrupt_occurrences']}, "
        f"clean_counts={dict(sorted(d['clean_token_counts'].items()))}, "
        f"corrupt_counts={dict(sorted(d['corrupt_token_counts'].items()))}"
    )

    if clean_ids:
        print(f"      clean continuation IDs: {clean_ids}")

    if corrupt_ids:
        print(f"      corrupt continuation IDs: {corrupt_ids}")


# ------------------------------------------------------------------------------
# 13. Deterministic first-target-token audit
# ------------------------------------------------------------------------------

print("\n[13/16] First-target-token operationalization audit")
print("-" * 78)

#
# A crucial distinction:
#
# The target may be one, two, or three tokens.
#
# The first target token is nevertheless always located at:
#
#     L = len(prompt_tokens)
#
# provided the prompt is an exact prefix of the hypothetical continuation.
#
# This is the natural next-token prediction boundary.
#

first_target_position_is_boundary = all(
    (
        r["clean_first_target_position"]
        == r["clean_prompt_length"]
        and
        r["corrupt_first_target_position"]
        == r["corrupt_prompt_length"]
    )
    for r in audit_records
)

all_prompt_prefixes_valid = (
    clean_prefix_pass == N
    and corrupt_prefix_pass == N
)

all_first_positions_match = (
    first_position_match_count == N
)

print(
    f"  clean first target begins immediately after prompt: "
    f"{first_target_position_is_boundary}"
)

print(
    f"  all clean/corrupt continuation prefix checks pass: "
    f"{all_prompt_prefixes_valid}"
)

print(
    f"  all first-target positions are condition-symmetric: "
    f"{all_first_positions_match}"
)

print(
    "  interpretation:"
)

print(
    "    The first target token is not required to be the final token of "
    "the full hypothetical continuation. It is the token predicted at the "
    "next-token boundary immediately following the observed prompt."
)


# ------------------------------------------------------------------------------
# 14. Scientific / operational classification
# ------------------------------------------------------------------------------

print("\n[14/16] Scientific / operational classification")
print("-" * 78)

if (
    all_prompt_prefixes_valid
    and prompt_length_equal_count == N
    and first_position_match_count == N
    and clean_character_cover_pass == N
    and corrupt_character_cover_pass == N
):
    classification = "TARGET_CONTINUATION_ALIGNMENT_PASS"

    interpretation = (
        "The target names can be represented as deterministic continuations "
        "of the exact authenticated prompts under the Llama tokenizer. The "
        "observed prompt is an exact token prefix of the hypothetical target "
        "continuation for all 192 clean/corrupt records, target spans are "
        "token-contiguous and exactly recoverable, prompt lengths are "
        "condition-symmetric, and the first target token begins at the "
        "well-defined next-token prediction boundary. Multi-token targets "
        "are therefore a representational property of the Llama tokenizer, "
        "not by themselves an operational failure."
    )

elif (
    all_prompt_prefixes_valid
    and prompt_length_equal_count == N
    and first_position_match_count == N
):
    classification = (
        "TARGET_BOUNDARY_PASS__TARGET_SPAN_ENCODING_REQUIRES_REVIEW"
    )

    interpretation = (
        "The next-token prediction boundary is fully well-defined and "
        "condition-symmetric, but at least one target continuation cannot "
        "be represented by an exact contiguous character-covered token span "
        "under the present explicit continuation convention. The prediction "
        "boundary remains operationally valid, but target-span encoding "
        "requires further audit before downstream representation extraction."
    )

else:
    classification = "OPERATIONAL_DECOHERENCE"

    interpretation = (
        "The exact authenticated prompts do not provide a uniform and "
        "condition-symmetric next-token target boundary under the present "
        "Llama tokenizer continuation convention. The CTL mathematics and "
        "authenticated dataset remain unchanged; the mismatch is classified "
        "at the operationalization layer."
    )

print(f"  classification: {classification}")
print(f"  interpretation: {interpretation}")


# ------------------------------------------------------------------------------
# 15. Post-audit scientific integrity and manifest
# ------------------------------------------------------------------------------

print("\n[15/16] Post-audit scientific integrity")
print("-" * 78)

post_audit_bytes = canonical_dataset_bytes(llama_records)
post_audit_sha = hashlib.sha256(post_audit_bytes).hexdigest()

dataset_unchanged = (
    post_audit_sha == pre_audit_sha == EXPECTED_DATASET_SHA
)

print(f"  pre-audit SHA-256:  {pre_audit_sha}")
print(f"  post-audit SHA-256: {post_audit_sha}")
print(f"  records unchanged: {dataset_unchanged}")

if not dataset_unchanged:
    raise RuntimeError(
        "CRITICAL: authenticated dataset changed during Phase 1C.2R."
    )


manifest = {
    "experiment_id": EXPERIMENT_ID,
    "phase": "PHASE_1C.2R",
    "audit": "CORRECTED_LLAMA_TARGET_CONTINUATION_PREDICTION_BOUNDARY_AUDIT",

    "timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "methodological_correction": {
        "previous_audit": "PHASE_1C.2",
        "previous_target_localization_interpretation":
            "in-prompt target-string search",
        "corrected_interpretation":
            "target is predicted continuation after prompt boundary",
        "previous_zero_localization_not_treated_as_scientific_failure": True,
    },

    "model_id": MODEL_ID,

    "dataset": {
        "record_count": N,
        "sha256_pre_audit": pre_audit_sha,
        "sha256_post_audit": post_audit_sha,
        "expected_authenticated_sha256": EXPECTED_DATASET_SHA,
        "unchanged": dataset_unchanged,
    },

    "tokenizer": {
        "name_or_path": tokenizer.name_or_path,
        "class": type(tokenizer).__name__,
        "is_fast": bool(tokenizer.is_fast),
        "vocab_size": tokenizer.vocab_size,
        "add_special_tokens": False,
    },

    "continuation_definition": {
        "full_text_construction":
            "prompt + single separating space + target name",
        "prediction_boundary":
            "position len(tokenized_prompt)",
        "first_target_token":
            "first token after exact prompt token prefix",
        "terminal_target_token":
            "last token in hypothetical target continuation span",
        "prompt_prefix_required": True,
        "exact_character_cover_required": True,
        "contiguous_target_span_required": True,
    },

    "aggregate_results": {
        "clean_prefix_pass": clean_prefix_pass,
        "corrupt_prefix_pass": corrupt_prefix_pass,
        "clean_character_cover_pass":
            clean_character_cover_pass,
        "corrupt_character_cover_pass":
            corrupt_character_cover_pass,
        "prompt_length_equal":
            prompt_length_equal_count,
        "first_target_position_match":
            first_position_match_count,
        "terminal_target_position_match":
            terminal_position_match_count,
        "continuation_count_equal":
            continuation_count_equal_count,
        "complete_structural_pass":
            complete_pass_count,
        "total_records": N,
    },

    "continuation_distributions": {
        "clean":
            dict(sorted(clean_counts.items())),
        "corrupt":
            dict(sorted(corrupt_counts.items())),
        "paired_count_delta":
            dict(sorted(paired_count_deltas.items())),
    },

    "boundary_distributions": {
        "clean_first_target_positions":
            dict(sorted(clean_prompt_positions.items())),
        "corrupt_first_target_positions":
            dict(sorted(corrupt_prompt_positions.items())),
        "clean_terminal_target_positions":
            dict(sorted(clean_terminal_positions.items())),
        "corrupt_terminal_target_positions":
            dict(sorted(corrupt_terminal_positions.items())),
    },

    "name_diagnostics": {
        name: {
            "clean_occurrences":
                d["clean_occurrences"],
            "corrupt_occurrences":
                d["corrupt_occurrences"],
            "clean_token_counts":
                dict(sorted(d["clean_token_counts"].items())),
            "corrupt_token_counts":
                dict(sorted(d["corrupt_token_counts"].items())),
            "clean_ids":
                [list(x) for x in sorted(d["clean_ids"])],
            "corrupt_ids":
                [list(x) for x in sorted(d["corrupt_ids"])],
        }
        for name, d in sorted(name_diagnostics.items())
    },

    "classification": classification,
    "interpretation": interpretation,

    "scientific_governance": {
        "dataset_modified": False,
        "prompts_modified": False,
        "model_weights_loaded": False,
        "model_inference_performed": False,
        "transport_fitting_performed": False,
        "state_bank_created": False,
        "ctl_mathematics_modified": False,
        "operationalization_layer_only": True,
    },

    "next_stage": (
        "PHASE_1C.3_TARGET_POSITION_OPERATIONALIZATION_AUDIT"
        if classification == "TARGET_CONTINUATION_ALIGNMENT_PASS"
        else "HOLD_PENDING_TARGET_CONTINUATION_RESOLUTION"
    ),
}

with open(OUTPUT_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )

print(f"  manifest written: {OUTPUT_MANIFEST_PATH}")


# ------------------------------------------------------------------------------
# 16. Cleanup and final status
# ------------------------------------------------------------------------------

print("\n[16/16] Final status")
print("-" * 78)

# Keep llama_records untouched. Release tokenizer and authentication objects.
try:
    del tokenizer
except Exception:
    pass

try:
    del resolved_token
except Exception:
    pass

gc.collect()

print(f"  classification: {classification}")
print(
    f"  clean prompt-prefix validity: "
    f"{clean_prefix_pass}/{N}"
)
print(
    f"  corrupt prompt-prefix validity: "
    f"{corrupt_prefix_pass}/{N}"
)
print(
    f"  clean target character-cover validity: "
    f"{clean_character_cover_pass}/{N}"
)
print(
    f"  corrupt target character-cover validity: "
    f"{corrupt_character_cover_pass}/{N}"
)
print(
    f"  equal clean/corrupt prompt lengths: "
    f"{prompt_length_equal_count}/{N}"
)
print(
    f"  equal first-target positions: "
    f"{first_position_match_count}/{N}"
)
print(
    f"  equal terminal-target positions: "
    f"{terminal_position_match_count}/{N}"
)
print(
    f"  equal target continuation lengths: "
    f"{continuation_count_equal_count}/{N}"
)
print(
    f"  complete structural pass: "
    f"{complete_pass_count}/{N}"
)
print(f"  dataset mutated: {not dataset_unchanged}")
print(f"  model inference performed: False")
print(f"  transport fitting performed: False")
print(f"  state bank created: False")
print(f"  CTL mathematics modified: False")
print(f"  manifest: {OUTPUT_MANIFEST_PATH}")

print("=" * 78)
print("PHASE 1C.2R COMPLETE")
print(
    "The corrected audit treats target names as predicted continuations "
    "rather than in-prompt character spans."
)
print("=" * 78)

ETTR-CTL-LLAMA-1 — PHASE_1C.2R
CORRECTED LLAMA TARGET-CONTINUATION / PREDICTION-BOUNDARY AUDIT

[1/16] Scientific-state firewall
------------------------------------------------------------------------------
  llama_records present: True
  record count: 192
  pre-audit canonical SHA-256: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  expected authenticated SHA-256: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  scientific dataset firewall: PASS

[2/16] Dataset authorization manifest audit
------------------------------------------------------------------------------
  dataset_authorized: True
  manifest SHA-256: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  manifest record count: 192
  authorization precondition: PASS

[3/16] Previous Phase 1C.2 methodological interpretation
------------------------------------------------------------------------------
  previous Phase 1C.2 manifest exists: True
  methodological correction:
 

In [12]:
# ==============================================================================
# ETTR-CTL-LLAMA-1 — PHASE_1C.3
# TARGET-POSITION OPERATIONALIZATION AUDIT
#
# PURPOSE
# -------
# Phase 1C.2R established that:
#
#   (1) the exact authenticated clean/corrupt prompts are valid token prefixes
#       of their corresponding target continuations;
#   (2) clean/corrupt prompt token lengths are condition-symmetric;
#   (3) the first target token begins at the well-defined next-token prediction
#       boundary;
#   (4) under the actual prompt context, all target continuations are
#       represented by one token.
#
# Phase 1C.3 now audits and freezes the operational target-position convention
# that will be used by subsequent model-state extraction.
#
# OPERATIONAL CONVENTION
# ----------------------
# For an observed prompt P:
#
#     Z(P) = tokenizer(P, add_special_tokens=False)
#
#     L(P) = |Z(P)|
#
# The target prediction position is:
#
#     p_T = L(P)
#
# i.e. the first position AFTER the final observed prompt token, corresponding
# to the next-token prediction produced from the final prompt state.
#
# The target token is defined contextually through:
#
#     tokenizer(P + " " + T)
#
# and must satisfy:
#
#     tokenizer(P + " " + T)
#       = tokenizer(P) || [t_T]
#
# where t_T is exactly one token.
#
# GOVERNANCE
# ----------
# - The authenticated 192-record dataset is frozen.
# - No prompt is modified.
# - No dataset field is rewritten.
# - No model weights are loaded.
# - No model inference is performed.
# - No transport map is fitted.
# - No state bank is created.
# - No CTL mathematical definition is modified.
#
# This cell is therefore a pure operationalization audit.
# ==============================================================================

import os
import gc
import json
import hashlib
from datetime import datetime, timezone
from collections import Counter, defaultdict

from transformers import AutoTokenizer
from huggingface_hub import get_token


print("=" * 78)
print("ETTR-CTL-LLAMA-1 — PHASE_1C.3")
print("TARGET-POSITION OPERATIONALIZATION AUDIT")
print("=" * 78)


# ------------------------------------------------------------------------------
# 0. Constants
# ------------------------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
MODEL_ID = "meta-llama/Llama-3.2-3B"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

AUTH_MANIFEST_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

PHASE_1C2R_MANIFEST_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1c2r_corrected_target_continuation_audit.json"
)

OUTPUT_MANIFEST_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1c3_target_position_operationalization_audit.json"
)

EXPECTED_DATASET_SHA = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)

EXPECTED_RECORD_COUNT = 192

EXPECTED_SPLITS = {
    "calibration": 48,
    "test": 48,
    "train": 96,
}


# ------------------------------------------------------------------------------
# 1. Scientific-state firewall
# ------------------------------------------------------------------------------

print("\n[1/18] Scientific-state firewall")
print("-" * 78)

if "llama_records" not in globals():
    raise RuntimeError(
        "llama_records is absent. The authenticated Phase 1C.0 dataset "
        "must be present."
    )

if not isinstance(llama_records, list):
    raise RuntimeError("llama_records must be a list.")

if len(llama_records) != EXPECTED_RECORD_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_RECORD_COUNT} records; "
        f"found {len(llama_records)}."
    )


def canonical_dataset_bytes(records):
    return json.dumps(
        records,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    ).encode("utf-8")


pre_audit_bytes = canonical_dataset_bytes(llama_records)
pre_audit_sha = hashlib.sha256(pre_audit_bytes).hexdigest()

print(f"  llama_records present: True")
print(f"  record count: {len(llama_records)}")
print(f"  pre-audit canonical SHA-256: {pre_audit_sha}")
print(f"  expected authenticated SHA-256: {EXPECTED_DATASET_SHA}")

if pre_audit_sha != EXPECTED_DATASET_SHA:
    raise RuntimeError(
        "Scientific-state firewall failed: authenticated dataset SHA mismatch."
    )

print("  scientific dataset firewall: PASS")


# ------------------------------------------------------------------------------
# 2. Authorization manifest
# ------------------------------------------------------------------------------

print("\n[2/18] Dataset authorization manifest")
print("-" * 78)

if not os.path.exists(AUTH_MANIFEST_PATH):
    raise FileNotFoundError(AUTH_MANIFEST_PATH)

with open(AUTH_MANIFEST_PATH, "r", encoding="utf-8") as f:
    auth_manifest = json.load(f)

dataset_authorized = auth_manifest.get("dataset_authorized")
manifest_sha = auth_manifest.get("sha256")
manifest_count = auth_manifest.get("record_count")

print(f"  dataset_authorized: {dataset_authorized}")
print(f"  manifest SHA-256: {manifest_sha}")
print(f"  manifest record count: {manifest_count}")

if dataset_authorized is not True:
    raise RuntimeError(
        "Phase 1C.0 does not explicitly authorize the operative dataset."
    )

if manifest_sha != EXPECTED_DATASET_SHA:
    raise RuntimeError(
        "Phase 1C.0 authorization SHA mismatch."
    )

if manifest_count != EXPECTED_RECORD_COUNT:
    raise RuntimeError(
        "Phase 1C.0 authorization record-count mismatch."
    )

print("  authorization precondition: PASS")


# ------------------------------------------------------------------------------
# 3. Phase 1C.2R prerequisite
# ------------------------------------------------------------------------------

print("\n[3/18] Phase 1C.2R prerequisite audit")
print("-" * 78)

if not os.path.exists(PHASE_1C2R_MANIFEST_PATH):
    raise FileNotFoundError(PHASE_1C2R_MANIFEST_PATH)

with open(PHASE_1C2R_MANIFEST_PATH, "r", encoding="utf-8") as f:
    phase_1c2r_manifest = json.load(f)

phase_1c2r_classification = phase_1c2r_manifest.get(
    "classification"
)

phase_1c2r_aggregate = phase_1c2r_manifest.get(
    "aggregate_results",
    {}
)

print(
    f"  Phase 1C.2R classification: "
    f"{phase_1c2r_classification}"
)

print(
    f"  Phase 1C.2R complete structural pass: "
    f"{phase_1c2r_aggregate.get('complete_structural_pass')}"
)

print(
    f"  Phase 1C.2R first-target position match: "
    f"{phase_1c2r_aggregate.get('first_target_position_match')}"
)

print(
    f"  Phase 1C.2R prompt-length equality: "
    f"{phase_1c2r_aggregate.get('prompt_length_equal')}"
)

# We do not silently trust the previous classification. The present cell
# independently re-tests the necessary properties below.
print("  prior audit retrieved: PASS")
print("  independent re-audit required: True")


# ------------------------------------------------------------------------------
# 4. Dataset structural integrity
# ------------------------------------------------------------------------------

print("\n[4/18] Dataset structural integrity")
print("-" * 78)

required_fields = {
    "clean_prompt",
    "corrupt_prompt",
    "clean_target_name",
    "corrupt_target_name",
    "condition_pair",
    "split",
}

missing_fields = sorted(
    required_fields - set(llama_records[0].keys())
)

if missing_fields:
    raise RuntimeError(
        f"Required fields missing: {missing_fields}"
    )

schema_ok = all(
    required_fields.issubset(record.keys())
    for record in llama_records
)

condition_counts = Counter(
    record["condition_pair"]
    for record in llama_records
)

split_counts = Counter(
    record["split"]
    for record in llama_records
)

print(f"  required schema across all records: {schema_ok}")
print(f"  condition_pair distribution: {dict(condition_counts)}")
print(f"  split distribution: {dict(split_counts)}")

if not schema_ok:
    raise RuntimeError("Dataset schema failed.")

if condition_counts != Counter({"clean_vs_corrupt": 192}):
    raise RuntimeError("Unexpected condition_pair distribution.")

if split_counts != Counter(EXPECTED_SPLITS):
    raise RuntimeError("Unexpected split distribution.")

print("  structural integrity: PASS")


# ------------------------------------------------------------------------------
# 5. Exact tokenizer identity
# ------------------------------------------------------------------------------

print("\n[5/18] Exact Llama tokenizer identity")
print("-" * 78)

resolved_token = get_token()

if resolved_token is None:
    raise RuntimeError(
        "No Hugging Face credential available. Phase 1C.1A must pass."
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=resolved_token,
    use_fast=True,
)

print(f"  tokenizer class: {type(tokenizer).__name__}")
print(f"  tokenizer name_or_path: {tokenizer.name_or_path}")
print(f"  tokenizer vocab_size: {tokenizer.vocab_size}")
print(f"  tokenizer is_fast: {tokenizer.is_fast}")

if not tokenizer.is_fast:
    raise RuntimeError(
        "Fast tokenizer required for this operationalization audit."
    )

print("  tokenizer identity: PASS")


# ------------------------------------------------------------------------------
# 6. Explicit operational convention
# ------------------------------------------------------------------------------

print("\n[6/18] Target-position operational convention")
print("-" * 78)

print("  prompt tokenization:")
print("    tokenizer(P, add_special_tokens=False)")
print("")
print("  observed prompt length:")
print("    L(P) = number of prompt tokens")
print("")
print("  target prediction position:")
print("    p_T = L(P)")
print("")
print("  target representation:")
print("    first token of contextual continuation tokenizer(P + ' ' + T)")
print("")
print("  target continuation requirement:")
print("    tokenizer(P + ' ' + T) = tokenizer(P) || [t_T]")
print("")
print("  required continuation token count:")
print("    exactly 1")
print("")
print("  special-token convention:")
print("    add_special_tokens=False")
print("")
print("  operational convention definition: PASS")


# ------------------------------------------------------------------------------
# 7. Tokenization helper
# ------------------------------------------------------------------------------

print("\n[7/18] Exact continuation tokenization helper")
print("-" * 78)


def tokenize_ids(text):
    enc = tokenizer(
        text,
        add_special_tokens=False,
        return_attention_mask=True,
    )

    ids = enc["input_ids"]

    if ids and isinstance(ids[0], list):
        ids = ids[0]

    return list(ids)


def analyze_continuation(prompt, target):
    """
    Analyze target T as the intended continuation of prompt P.

    Returns:
        prompt_ids
        full_ids
        continuation_ids
        prompt_length
        full_length
        prefix_match
        continuation_count
        first_target_position
        terminal_target_position
        expected_first_target_position
    """

    prompt_ids = tokenize_ids(prompt)

    full_text = prompt + " " + target

    full_ids = tokenize_ids(full_text)

    prompt_length = len(prompt_ids)
    full_length = len(full_ids)

    prefix_match = (
        full_length >= prompt_length
        and full_ids[:prompt_length] == prompt_ids
    )

    continuation_ids = (
        full_ids[prompt_length:]
        if prefix_match
        else []
    )

    continuation_count = len(continuation_ids)

    first_target_position = (
        prompt_length
        if continuation_count > 0
        else None
    )

    terminal_target_position = (
        prompt_length + continuation_count - 1
        if continuation_count > 0
        else None
    )

    return {
        "prompt_ids": prompt_ids,
        "full_ids": full_ids,
        "continuation_ids": continuation_ids,
        "prompt_length": prompt_length,
        "full_length": full_length,
        "prefix_match": prefix_match,
        "continuation_count": continuation_count,
        "first_target_position": first_target_position,
        "terminal_target_position": terminal_target_position,
        "expected_first_target_position": prompt_length,
    }


print("  helper initialized")


# ------------------------------------------------------------------------------
# 8. Full 192-record operationalization audit
# ------------------------------------------------------------------------------

print("\n[8/18] Full 192-record operationalization audit")
print("-" * 78)

audit_records = []

for record_index, record in enumerate(llama_records):

    clean = analyze_continuation(
        record["clean_prompt"],
        record["clean_target_name"],
    )

    corrupt = analyze_continuation(
        record["corrupt_prompt"],
        record["corrupt_target_name"],
    )

    prompt_length_equal = (
        clean["prompt_length"] == corrupt["prompt_length"]
    )

    clean_single_token = (
        clean["continuation_count"] == 1
    )

    corrupt_single_token = (
        corrupt["continuation_count"] == 1
    )

    first_position_valid_clean = (
        clean["first_target_position"]
        == clean["expected_first_target_position"]
    )

    first_position_valid_corrupt = (
        corrupt["first_target_position"]
        == corrupt["expected_first_target_position"]
    )

    first_position_match = (
        clean["first_target_position"]
        == corrupt["first_target_position"]
    )

    target_token_ids_distinct = (
        clean["continuation_count"] == 1
        and corrupt["continuation_count"] == 1
        and clean["continuation_ids"][0]
        != corrupt["continuation_ids"][0]
    )

    complete_record_pass = (
        clean["prefix_match"]
        and corrupt["prefix_match"]
        and clean_single_token
        and corrupt_single_token
        and prompt_length_equal
        and first_position_valid_clean
        and first_position_valid_corrupt
        and first_position_match
        and target_token_ids_distinct
    )

    audit_records.append({
        "record_index": record_index,
        "example_id": record.get("example_id"),
        "template": record.get("template"),
        "template_id": record.get("template_id"),
        "name_pair": record.get("name_pair"),
        "name_pair_id": record.get("name_pair_id"),
        "split": record.get("split"),

        "clean_target": record["clean_target_name"],
        "corrupt_target": record["corrupt_target_name"],

        "clean_prompt_length": clean["prompt_length"],
        "corrupt_prompt_length": corrupt["prompt_length"],

        "clean_prompt_ids": clean["prompt_ids"],
        "corrupt_prompt_ids": corrupt["prompt_ids"],

        "clean_continuation_ids":
            clean["continuation_ids"],
        "corrupt_continuation_ids":
            corrupt["continuation_ids"],

        "clean_continuation_count":
            clean["continuation_count"],
        "corrupt_continuation_count":
            corrupt["continuation_count"],

        "clean_first_target_position":
            clean["first_target_position"],
        "corrupt_first_target_position":
            corrupt["first_target_position"],

        "clean_terminal_target_position":
            clean["terminal_target_position"],
        "corrupt_terminal_target_position":
            corrupt["terminal_target_position"],

        "clean_prefix_match":
            clean["prefix_match"],
        "corrupt_prefix_match":
            corrupt["prefix_match"],

        "prompt_length_equal":
            prompt_length_equal,

        "clean_single_token":
            clean_single_token,

        "corrupt_single_token":
            corrupt_single_token,

        "first_position_valid_clean":
            first_position_valid_clean,

        "first_position_valid_corrupt":
            first_position_valid_corrupt,

        "first_position_match":
            first_position_match,

        "target_token_ids_distinct":
            target_token_ids_distinct,

        "complete_record_pass":
            complete_record_pass,
    })


# ------------------------------------------------------------------------------
# 9. Aggregate results
# ------------------------------------------------------------------------------

print("\n[9/18] Aggregate operationalization results")
print("-" * 78)

N = len(audit_records)

clean_prefix_count = sum(
    r["clean_prefix_match"]
    for r in audit_records
)

corrupt_prefix_count = sum(
    r["corrupt_prefix_match"]
    for r in audit_records
)

clean_single_token_count = sum(
    r["clean_single_token"]
    for r in audit_records
)

corrupt_single_token_count = sum(
    r["corrupt_single_token"]
    for r in audit_records
)

prompt_length_equal_count = sum(
    r["prompt_length_equal"]
    for r in audit_records
)

clean_first_position_valid_count = sum(
    r["first_position_valid_clean"]
    for r in audit_records
)

corrupt_first_position_valid_count = sum(
    r["first_position_valid_corrupt"]
    for r in audit_records
)

first_position_match_count = sum(
    r["first_position_match"]
    for r in audit_records
)

target_token_distinct_count = sum(
    r["target_token_ids_distinct"]
    for r in audit_records
)

complete_pass_count = sum(
    r["complete_record_pass"]
    for r in audit_records
)

print(
    f"  clean exact-prefix validity: "
    f"{clean_prefix_count}/{N}"
)

print(
    f"  corrupt exact-prefix validity: "
    f"{corrupt_prefix_count}/{N}"
)

print(
    f"  clean one-token continuation: "
    f"{clean_single_token_count}/{N}"
)

print(
    f"  corrupt one-token continuation: "
    f"{corrupt_single_token_count}/{N}"
)

print(
    f"  clean/corrupt prompt-length equality: "
    f"{prompt_length_equal_count}/{N}"
)

print(
    f"  clean first-position validity: "
    f"{clean_first_position_valid_count}/{N}"
)

print(
    f"  corrupt first-position validity: "
    f"{corrupt_first_position_valid_count}/{N}"
)

print(
    f"  clean/corrupt first-position equality: "
    f"{first_position_match_count}/{N}"
)

print(
    f"  clean/corrupt target-token IDs distinct: "
    f"{target_token_distinct_count}/{N}"
)

print(
    f"  complete operationalization pass: "
    f"{complete_pass_count}/{N}"
)


# ------------------------------------------------------------------------------
# 10. Prompt-length and target-position distributions
# ------------------------------------------------------------------------------

print("\n[10/18] Prompt-length / target-position distributions")
print("-" * 78)

clean_prompt_lengths = Counter(
    r["clean_prompt_length"]
    for r in audit_records
)

corrupt_prompt_lengths = Counter(
    r["corrupt_prompt_length"]
    for r in audit_records
)

clean_target_positions = Counter(
    r["clean_first_target_position"]
    for r in audit_records
)

corrupt_target_positions = Counter(
    r["corrupt_first_target_position"]
    for r in audit_records
)

print(
    f"  clean prompt lengths: "
    f"{dict(sorted(clean_prompt_lengths.items()))}"
)

print(
    f"  corrupt prompt lengths: "
    f"{dict(sorted(corrupt_prompt_lengths.items()))}"
)

print(
    f"  clean first-target positions: "
    f"{dict(sorted(clean_target_positions.items()))}"
)

print(
    f"  corrupt first-target positions: "
    f"{dict(sorted(corrupt_target_positions.items()))}"
)


# ------------------------------------------------------------------------------
# 11. Target-token collision audit
# ------------------------------------------------------------------------------

print("\n[11/18] Target-token collision audit")
print("-" * 78)

clean_target_token_map = defaultdict(list)
corrupt_target_token_map = defaultdict(list)

for r in audit_records:

    if r["clean_continuation_count"] == 1:
        clean_target_token_map[
            r["clean_continuation_ids"][0]
        ].append(
            r["clean_target"]
        )

    if r["corrupt_continuation_count"] == 1:
        corrupt_target_token_map[
            r["corrupt_continuation_ids"][0]
        ].append(
            r["corrupt_target"]
        )

clean_token_collisions = {
    int(token_id): sorted(set(names))
    for token_id, names in clean_target_token_map.items()
    if len(set(names)) > 1
}

corrupt_token_collisions = {
    int(token_id): sorted(set(names))
    for token_id, names in corrupt_target_token_map.items()
    if len(set(names)) > 1
}

all_target_names = set(
    r["clean_target"]
    for r in audit_records
) | set(
    r["corrupt_target"]
    for r in audit_records
)

all_target_token_ids = set(
    clean_target_token_map.keys()
) | set(
    corrupt_target_token_map.keys()
)

print(
    f"  distinct target names: "
    f"{len(all_target_names)}"
)

print(
    f"  distinct contextual target token IDs: "
    f"{len(all_target_token_ids)}"
)

print(
    f"  clean token-ID collisions: "
    f"{len(clean_token_collisions)}"
)

print(
    f"  corrupt token-ID collisions: "
    f"{len(corrupt_token_collisions)}"
)

if clean_token_collisions:
    print("  clean collisions:")
    for token_id, names in sorted(clean_token_collisions.items()):
        print(f"    {token_id}: {names}")

if corrupt_token_collisions:
    print("  corrupt collisions:")
    for token_id, names in sorted(corrupt_token_collisions.items()):
        print(f"    {token_id}: {names}")


# ------------------------------------------------------------------------------
# 12. Clean/corrupt target-token separation
# ------------------------------------------------------------------------------

print("\n[12/18] Clean/corrupt target-token separation")
print("-" * 78)

clean_target_ids = [
    r["clean_continuation_ids"][0]
    for r in audit_records
    if r["clean_continuation_count"] == 1
]

corrupt_target_ids = [
    r["corrupt_continuation_ids"][0]
    for r in audit_records
    if r["corrupt_continuation_count"] == 1
]

clean_target_id_set = set(clean_target_ids)
corrupt_target_id_set = set(corrupt_target_ids)

cross_condition_token_overlap = (
    clean_target_id_set & corrupt_target_id_set
)

print(
    f"  distinct clean target token IDs: "
    f"{len(clean_target_id_set)}"
)

print(
    f"  distinct corrupt target token IDs: "
    f"{len(corrupt_target_id_set)}"
)

print(
    f"  cross-condition token-ID overlap: "
    f"{len(cross_condition_token_overlap)}"
)

if cross_condition_token_overlap:
    print(
        "  overlapping IDs:",
        sorted(cross_condition_token_overlap)
    )


# ------------------------------------------------------------------------------
# 13. Per-template and per-pair operationalization audit
# ------------------------------------------------------------------------------

print("\n[13/18] Template/pair operationalization audit")
print("-" * 78)

template_stats = defaultdict(
    lambda: {
        "n": 0,
        "pass": 0,
        "prefix_pass": 0,
        "single_token_pass": 0,
        "position_pass": 0,
        "token_distinct_pass": 0,
    }
)

pair_stats = defaultdict(
    lambda: {
        "n": 0,
        "pass": 0,
        "prefix_pass": 0,
        "single_token_pass": 0,
        "position_pass": 0,
        "token_distinct_pass": 0,
    }
)

for r in audit_records:

    for container, key in (
        (template_stats, r["template_id"]),
        (pair_stats, r["name_pair_id"]),
    ):

        container[key]["n"] += 1

        container[key]["pass"] += int(
            r["complete_record_pass"]
        )

        container[key]["prefix_pass"] += int(
            r["clean_prefix_match"]
            and r["corrupt_prefix_match"]
        )

        container[key]["single_token_pass"] += int(
            r["clean_single_token"]
            and r["corrupt_single_token"]
        )

        container[key]["position_pass"] += int(
            r["first_position_match"]
        )

        container[key]["token_distinct_pass"] += int(
            r["target_token_ids_distinct"]
        )

template_failures = {
    str(k): dict(v)
    for k, v in template_stats.items()
    if v["pass"] < v["n"]
}

pair_failures = {
    str(k): dict(v)
    for k, v in pair_stats.items()
    if v["pass"] < v["n"]
}

print(
    f"  templates with operationalization failures: "
    f"{len(template_failures)}/{len(template_stats)}"
)

print(
    f"  name-pairs with operationalization failures: "
    f"{len(pair_failures)}/{len(pair_stats)}"
)

if template_failures:
    print("  template failures:")
    for k, v in sorted(template_failures.items()):
        print(f"    {k}: {v}")

if pair_failures:
    print("  pair failures:")
    for k, v in sorted(pair_failures.items()):
        print(f"    {k}: {v}")


# ------------------------------------------------------------------------------
# 14. Split-wise operationalization audit
# ------------------------------------------------------------------------------

print("\n[14/18] Train/calibration/test operationalization audit")
print("-" * 78)

split_stats = defaultdict(
    lambda: {
        "n": 0,
        "pass": 0,
        "prefix_pass": 0,
        "single_token_pass": 0,
        "position_pass": 0,
        "token_distinct_pass": 0,
    }
)

for r in audit_records:

    s = r["split"]

    split_stats[s]["n"] += 1
    split_stats[s]["pass"] += int(
        r["complete_record_pass"]
    )
    split_stats[s]["prefix_pass"] += int(
        r["clean_prefix_match"]
        and r["corrupt_prefix_match"]
    )
    split_stats[s]["single_token_pass"] += int(
        r["clean_single_token"]
        and r["corrupt_single_token"]
    )
    split_stats[s]["position_pass"] += int(
        r["first_position_match"]
    )
    split_stats[s]["token_distinct_pass"] += int(
        r["target_token_ids_distinct"]
    )

for split in ("train", "calibration", "test"):

    if split not in split_stats:
        raise RuntimeError(
            f"Missing expected split: {split}"
        )

    print(f"  {split}: {dict(split_stats[split])}")


# ------------------------------------------------------------------------------
# 15. Failure-mode audit
# ------------------------------------------------------------------------------

print("\n[15/18] Failure-mode audit")
print("-" * 78)

failure_records = []
failure_reason_counts = Counter()

for r in audit_records:

    reasons = []

    if not r["clean_prefix_match"]:
        reasons.append("CLEAN_PROMPT_NOT_EXACT_PREFIX")

    if not r["corrupt_prefix_match"]:
        reasons.append("CORRUPT_PROMPT_NOT_EXACT_PREFIX")

    if not r["clean_single_token"]:
        reasons.append("CLEAN_TARGET_NOT_SINGLE_CONTEXTUAL_TOKEN")

    if not r["corrupt_single_token"]:
        reasons.append("CORRUPT_TARGET_NOT_SINGLE_CONTEXTUAL_TOKEN")

    if not r["prompt_length_equal"]:
        reasons.append("CLEAN_CORRUPT_PROMPT_LENGTH_MISMATCH")

    if not r["first_position_valid_clean"]:
        reasons.append("CLEAN_FIRST_TARGET_POSITION_INVALID")

    if not r["first_position_valid_corrupt"]:
        reasons.append("CORRUPT_FIRST_TARGET_POSITION_INVALID")

    if not r["first_position_match"]:
        reasons.append("CLEAN_CORRUPT_FIRST_POSITION_ASYMMETRY")

    if not r["target_token_ids_distinct"]:
        reasons.append("CLEAN_CORRUPT_TARGET_TOKEN_ID_NOT_DISTINCT")

    if reasons:
        failure_records.append({
            "record_index": r["record_index"],
            "example_id": r["example_id"],
            "template_id": r["template_id"],
            "name_pair_id": r["name_pair_id"],
            "split": r["split"],
            "reasons": reasons,
        })

        for reason in reasons:
            failure_reason_counts[reason] += 1

print(
    f"  records with any operationalization failure: "
    f"{len(failure_records)}/{N}"
)

if failure_reason_counts:
    for reason, count in failure_reason_counts.most_common():
        print(f"    {reason}: {count}")
else:
    print("  no operationalization failure modes detected.")


# ------------------------------------------------------------------------------
# 16. Final scientific classification
# ------------------------------------------------------------------------------

print("\n[16/18] Final scientific / operational classification")
print("-" * 78)

no_collision = (
    len(clean_token_collisions) == 0
    and len(corrupt_token_collisions) == 0
)

no_cross_condition_overlap = (
    len(cross_condition_token_overlap) == 0
)

all_core_conditions_pass = (
    clean_prefix_count == N
    and corrupt_prefix_count == N
    and clean_single_token_count == N
    and corrupt_single_token_count == N
    and prompt_length_equal_count == N
    and clean_first_position_valid_count == N
    and corrupt_first_position_valid_count == N
    and first_position_match_count == N
    and target_token_distinct_count == N
    and no_collision
    and no_cross_condition_overlap
)

if all_core_conditions_pass:

    classification = "TARGET_POSITION_OPERATIONALIZATION_PASS"

    interpretation = (
        "The Llama target-position operationalization is fully supported "
        "for the authenticated 192-record dataset. Each exact clean and "
        "corrupt prompt is an exact token prefix of its contextual target "
        "continuation; each target continuation consists of exactly one "
        "token; clean and corrupt prompts have equal token lengths; the "
        "first target token begins at the identical next-token prediction "
        "boundary in both conditions; and target token identities are "
        "distinct across the clean/corrupt conditions with no contextual "
        "token collisions. The target position may therefore be frozen as "
        "the next-token prediction position p_T = L(P) for subsequent "
        "model-state extraction."
    )

elif (
    clean_prefix_count == N
    and corrupt_prefix_count == N
    and prompt_length_equal_count == N
    and first_position_match_count == N
):
    classification = (
        "TARGET_BOUNDARY_PASS__TARGET_TOKEN_ENCODING_REQUIRES_REVIEW"
    )

    interpretation = (
        "The next-token prediction boundary is valid and condition-symmetric, "
        "but the complete contextual target-token operationalization has "
        "remaining exceptions. Model-state extraction should remain frozen "
        "until those exceptions are resolved."
    )

else:

    classification = "OPERATIONAL_DECOHERENCE"

    interpretation = (
        "The Llama target-position operationalization is not uniformly "
        "supported by the authenticated dataset. The mismatch is retained "
        "as operationalization-layer decoherence; the CTL mathematical "
        "specification and authenticated dataset remain unchanged."
    )

print(f"  classification: {classification}")
print(f"  interpretation: {interpretation}")


# ------------------------------------------------------------------------------
# 17. Post-audit scientific integrity and manifest
# ------------------------------------------------------------------------------

print("\n[17/18] Post-audit scientific integrity")
print("-" * 78)

post_audit_bytes = canonical_dataset_bytes(llama_records)
post_audit_sha = hashlib.sha256(post_audit_bytes).hexdigest()

dataset_unchanged = (
    post_audit_sha == pre_audit_sha == EXPECTED_DATASET_SHA
)

print(f"  pre-audit SHA-256:  {pre_audit_sha}")
print(f"  post-audit SHA-256: {post_audit_sha}")
print(f"  records unchanged: {dataset_unchanged}")

if not dataset_unchanged:
    raise RuntimeError(
        "CRITICAL: authenticated dataset changed during Phase 1C.3."
    )


manifest = {
    "experiment_id": EXPERIMENT_ID,
    "phase": "PHASE_1C.3",
    "audit": "TARGET_POSITION_OPERATIONALIZATION_AUDIT",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "model_id": MODEL_ID,

    "dataset": {
        "record_count": N,
        "sha256_pre_audit": pre_audit_sha,
        "sha256_post_audit": post_audit_sha,
        "expected_authenticated_sha256":
            EXPECTED_DATASET_SHA,
        "unchanged": dataset_unchanged,
    },

    "authorization": {
        "manifest": AUTH_MANIFEST_PATH,
        "dataset_authorized": dataset_authorized,
    },

    "prerequisite": {
        "phase_1c2r_manifest": PHASE_1C2R_MANIFEST_PATH,
        "phase_1c2r_classification":
            phase_1c2r_classification,
    },

    "tokenizer": {
        "name_or_path": tokenizer.name_or_path,
        "class": type(tokenizer).__name__,
        "is_fast": bool(tokenizer.is_fast),
        "vocab_size": tokenizer.vocab_size,
        "add_special_tokens": False,
    },

    "frozen_operational_definition": {
        "prompt_tokenization":
            "tokenizer(P, add_special_tokens=False)",
        "prompt_length":
            "L(P) = len(tokenizer(P))",
        "target_position":
            "p_T = L(P)",
        "target_representation":
            "first token of contextual continuation",
        "continuation_construction":
            "P + ' ' + T",
        "required_contextual_continuation_length":
            1,
        "exact_prompt_prefix_required":
            True,
        "condition_symmetric_boundary_required":
            True,
        "distinct_clean_corrupt_target_token_required":
            True,
        "token_collision_free_required":
            True,
    },

    "aggregate_results": {
        "clean_prefix_count": clean_prefix_count,
        "corrupt_prefix_count": corrupt_prefix_count,
        "clean_single_token_count":
            clean_single_token_count,
        "corrupt_single_token_count":
            corrupt_single_token_count,
        "prompt_length_equal_count":
            prompt_length_equal_count,
        "clean_first_position_valid_count":
            clean_first_position_valid_count,
        "corrupt_first_position_valid_count":
            corrupt_first_position_valid_count,
        "first_position_match_count":
            first_position_match_count,
        "target_token_distinct_count":
            target_token_distinct_count,
        "complete_pass_count":
            complete_pass_count,
        "total_records": N,
    },

    "target_token_collision_audit": {
        "distinct_target_names":
            len(all_target_names),
        "distinct_contextual_target_token_ids":
            len(all_target_token_ids),
        "clean_token_collisions":
            clean_token_collisions,
        "corrupt_token_collisions":
            corrupt_token_collisions,
        "cross_condition_token_overlap":
            sorted(cross_condition_token_overlap),
    },

    "template_failures": template_failures,
    "pair_failures": pair_failures,

    "split_statistics": {
        str(k): dict(v)
        for k, v in sorted(split_stats.items())
    },

    "failure_reason_counts":
        dict(failure_reason_counts),

    "classification": classification,
    "interpretation": interpretation,

    "scientific_governance": {
        "dataset_modified": False,
        "prompts_modified": False,
        "model_weights_loaded": False,
        "model_inference_performed": False,
        "transport_fitting_performed": False,
        "state_bank_created": False,
        "ctl_mathematics_modified": False,
        "operationalization_layer_only": True,
    },

    "next_stage": (
        "PHASE_1D_STATE_EXTRACTION_AUDIT"
        if classification == "TARGET_POSITION_OPERATIONALIZATION_PASS"
        else "HOLD_PENDING_OPERATIONAL_RESOLUTION"
    ),
}

with open(OUTPUT_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )

print(f"  manifest written: {OUTPUT_MANIFEST_PATH}")


# ------------------------------------------------------------------------------
# 18. Cleanup and final status
# ------------------------------------------------------------------------------

print("\n[18/18] Cleanup and final status")
print("-" * 78)

try:
    del tokenizer
except Exception:
    pass

try:
    del resolved_token
except Exception:
    pass

gc.collect()

print(f"  classification: {classification}")
print(
    f"  complete operationalization pass: "
    f"{complete_pass_count}/{N}"
)
print(
    f"  contextual one-token clean targets: "
    f"{clean_single_token_count}/{N}"
)
print(
    f"  contextual one-token corrupt targets: "
    f"{corrupt_single_token_count}/{N}"
)
print(
    f"  clean/corrupt prompt-length equality: "
    f"{prompt_length_equal_count}/{N}"
)
print(
    f"  clean/corrupt first-position equality: "
    f"{first_position_match_count}/{N}"
)
print(
    f"  clean/corrupt target-token distinction: "
    f"{target_token_distinct_count}/{N}"
)
print(
    f"  contextual token collisions: "
    f"{len(clean_token_collisions) + len(corrupt_token_collisions)}"
)
print(
    f"  cross-condition target-token overlap: "
    f"{len(cross_condition_token_overlap)}"
)
print(f"  dataset mutated: {not dataset_unchanged}")
print(f"  model inference performed: False")
print(f"  transport fitting performed: False")
print(f"  state bank created: False")
print(f"  CTL mathematics modified: False")
print(f"  manifest: {OUTPUT_MANIFEST_PATH}")

print("=" * 78)
print("PHASE 1C.3 COMPLETE")
print(
    "Target-position operationalization has been audited without modifying "
    "the authenticated scientific dataset or CTL mathematical specification."
)
print("=" * 78)

ETTR-CTL-LLAMA-1 — PHASE_1C.3
TARGET-POSITION OPERATIONALIZATION AUDIT

[1/18] Scientific-state firewall
------------------------------------------------------------------------------
  llama_records present: True
  record count: 192
  pre-audit canonical SHA-256: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  expected authenticated SHA-256: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  scientific dataset firewall: PASS

[2/18] Dataset authorization manifest
------------------------------------------------------------------------------
  dataset_authorized: True
  manifest SHA-256: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  manifest record count: 192
  authorization precondition: PASS

[3/18] Phase 1C.2R prerequisite audit
------------------------------------------------------------------------------
  Phase 1C.2R classification: TARGET_BOUNDARY_PASS__TARGET_SPAN_ENCODING_REQUIRES_REVIEW
  Phase 1C.2R complete structural 

In [14]:
# ==============================================================================
# ETTR-CTL-LLAMA-1 — PHASE_1D.0
# LLAMA STATE-EXTRACTION / LAYER-SECTOR / TARGET-POSITION AUDIT
#
# REPLACEMENT FOR PREVIOUS PHASE_1D.0 CELL
#
# Numerical correction:
#   The previous cell failed only because PyTorch 2.11's NumPy bridge does not
#   support direct BF16 -> NumPy conversion. The scientific state was not
#   modified. Parameter immutability is now fingerprinted through a dtype-safe
#   CPU float32 representation, solely for hashing.
#
# Governing principle:
#   The BF16 model arithmetic remains unchanged. No mathematical or
#   experimental definition is altered to accommodate the runtime.
# ==============================================================================

import os
import gc
import json
import hashlib
import inspect
from datetime import datetime, timezone
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import get_token


print("=" * 78)
print("ETTR-CTL-LLAMA-1 — PHASE_1D.0")
print("LLAMA STATE-EXTRACTION / LAYER-SECTOR / TARGET-POSITION AUDIT")
print("=" * 78)


# ------------------------------------------------------------------------------
# 0. Constants
# ------------------------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
MODEL_ID = "meta-llama/Llama-3.2-3B"

ROOT = "/content/ettr_ctl_llama"
RESULTS_DIR = os.path.join(ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

AUTH_MANIFEST_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

TARGET_POSITION_MANIFEST_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1c3_target_position_operationalization_audit.json"
)

OUTPUT_MANIFEST_PATH = os.path.join(
    RESULTS_DIR,
    "llama_phase1d0_state_extraction_audit.json"
)

EXPECTED_DATASET_SHA = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)

EXPECTED_RECORD_COUNT = 192

SELECTED_LAYER = 14

EXPECTED_HIDDEN_SIZE = 3072
EXPECTED_Q_HEADS = 24
EXPECTED_KV_HEADS = 8
EXPECTED_INTERMEDIATE_SIZE = 8192

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

SYNTHETIC_PROMPT = (
    "When Alice and Bob went to the bank, Alice handed the form to"
)


# ------------------------------------------------------------------------------
# Helper: canonical dataset bytes
# ------------------------------------------------------------------------------

def canonical_dataset_bytes(records):
    return json.dumps(
        records,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    ).encode("utf-8")


# ------------------------------------------------------------------------------
# Helper: dtype-safe parameter fingerprint
#
# IMPORTANT:
# This function does NOT change model parameters.
#
# BF16 tensors cannot be passed directly through NumPy in this runtime.
# We therefore create a CPU float32 COPY solely for deterministic hashing.
#
# Scientific model arithmetic remains BF16.
# ------------------------------------------------------------------------------

def parameter_fingerprint(parameter):

    cpu_copy = (
        parameter.detach()
        .cpu()
        .to(torch.float32)
        .contiguous()
    )

    # float32 is explicitly supported by NumPy.
    raw_bytes = cpu_copy.numpy().tobytes()

    return hashlib.sha256(raw_bytes).hexdigest()


# ------------------------------------------------------------------------------
# 1. Scientific-state firewall
# ------------------------------------------------------------------------------

print("\n[1/22] Scientific-state firewall")
print("-" * 78)

if "llama_records" not in globals():
    raise RuntimeError(
        "Authenticated llama_records are absent. Phase 1C.0 must be present."
    )

if not isinstance(llama_records, list):
    raise RuntimeError("llama_records must be a list.")

if len(llama_records) != EXPECTED_RECORD_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_RECORD_COUNT} records; "
        f"found {len(llama_records)}."
    )

pre_audit_dataset_bytes = canonical_dataset_bytes(llama_records)
pre_audit_dataset_sha = hashlib.sha256(
    pre_audit_dataset_bytes
).hexdigest()

print(f"  record count: {len(llama_records)}")
print(f"  pre-audit dataset SHA-256: {pre_audit_dataset_sha}")
print(f"  expected authenticated SHA-256: {EXPECTED_DATASET_SHA}")

if pre_audit_dataset_sha != EXPECTED_DATASET_SHA:
    raise RuntimeError(
        "Scientific-state firewall failed: dataset SHA mismatch."
    )

print("  dataset firewall: PASS")


# ------------------------------------------------------------------------------
# 2. Authorization prerequisite
# ------------------------------------------------------------------------------

print("\n[2/22] Dataset authorization prerequisite")
print("-" * 78)

if not os.path.exists(AUTH_MANIFEST_PATH):
    raise FileNotFoundError(AUTH_MANIFEST_PATH)

with open(AUTH_MANIFEST_PATH, "r", encoding="utf-8") as f:
    auth_manifest = json.load(f)

dataset_authorized = auth_manifest.get("dataset_authorized")
auth_sha = auth_manifest.get("sha256")
auth_count = auth_manifest.get("record_count")

print(f"  dataset_authorized: {dataset_authorized}")
print(f"  authorized SHA-256: {auth_sha}")
print(f"  authorized record count: {auth_count}")

if dataset_authorized is not True:
    raise RuntimeError(
        "Authenticated dataset authorization is not established."
    )

if auth_sha != EXPECTED_DATASET_SHA:
    raise RuntimeError(
        "Authorization manifest SHA does not match authenticated dataset."
    )

if auth_count != EXPECTED_RECORD_COUNT:
    raise RuntimeError(
        "Authorization manifest record count mismatch."
    )

print("  authorization prerequisite: PASS")


# ------------------------------------------------------------------------------
# 3. Target-position prerequisite
# ------------------------------------------------------------------------------

print("\n[3/22] Phase 1C.3 target-position prerequisite")
print("-" * 78)

if not os.path.exists(TARGET_POSITION_MANIFEST_PATH):
    raise FileNotFoundError(TARGET_POSITION_MANIFEST_PATH)

with open(TARGET_POSITION_MANIFEST_PATH, "r", encoding="utf-8") as f:
    target_position_manifest = json.load(f)

target_position_classification = target_position_manifest.get(
    "classification"
)

frozen_definition = target_position_manifest.get(
    "frozen_operational_definition",
    {}
)

print(
    f"  Phase 1C.3 classification: "
    f"{target_position_classification}"
)

print(
    f"  frozen prompt tokenization: "
    f"{frozen_definition.get('prompt_tokenization')}"
)

print(
    f"  frozen target position: "
    f"{frozen_definition.get('target_position')}"
)

if target_position_classification != (
    "TARGET_POSITION_OPERATIONALIZATION_PASS"
):
    raise RuntimeError(
        "Phase 1C.3 did not pass. "
        "State-extraction audit cannot proceed."
    )

print("  target-position prerequisite: PASS")


# ------------------------------------------------------------------------------
# 4. Frozen prediction-position/state-index convention
# ------------------------------------------------------------------------------

print("\n[4/22] Prediction-position versus state-index convention")
print("-" * 78)

print("  observed prompt token indices:")
print("    0, 1, ..., L-1")

print("")
print("  next-token prediction position:")
print("    p_T = L")

print("")
print("  hidden state producing that prediction:")
print("    h_{L-1}")

print("")
print("  scientific state extraction index:")
print("    i_T = L-1")

print("")
print("  target token itself is NOT fed into the model.")
print("  target token is the continuation/label being predicted.")

print("")
print("  target-position numerical convention: FROZEN")

TARGET_STATE_INDEX_DEFINITION = "i_T = L(P) - 1"
TARGET_PREDICTION_POSITION_DEFINITION = "p_T = L(P)"


# ------------------------------------------------------------------------------
# 5. Runtime/device audit
# ------------------------------------------------------------------------------

print("\n[5/22] Runtime and device audit")
print("-" * 78)

print(f"  torch version: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required for Llama state-extraction."
    )

print(f"  CUDA device: {torch.cuda.get_device_name(0)}")
print(f"  device: {DEVICE}")

cuda_before = torch.cuda.memory_allocated(0)

print(
    f"  CUDA allocated before audit: "
    f"{cuda_before / (1024**3):.4f} GiB"
)

print("  runtime availability: PASS")


# ------------------------------------------------------------------------------
# 6. Existing model audit
# ------------------------------------------------------------------------------

print("\n[6/22] Model presence and identity audit")
print("-" * 78)

model = globals().get("model", None)

if model is not None:
    print("  existing global model object found: True")
    print(f"  existing model class: {type(model).__name__}")
else:
    print("  existing global model object found: False")
    print("  model loading permitted for instrumentation audit: True")


# ------------------------------------------------------------------------------
# 7. Exact model numerical residency
# ------------------------------------------------------------------------------

print("\n[7/22] Exact model numerical residency audit")
print("-" * 78)

if model is None:

    resolved_token = get_token()

    if resolved_token is None:
        raise RuntimeError(
            "No Hugging Face credential available."
        )

    print("  loading exact model:")
    print(f"    {MODEL_ID}")
    print("  dtype: bfloat16")
    print("  device: cuda:0")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        token=resolved_token,
        dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )

    model = model.to(
        device=DEVICE,
        dtype=torch.bfloat16,
    )

    globals()["model"] = model

    del resolved_token

else:

    print("  using existing model object.")
    print("  model load skipped.")


# ------------------------------------------------------------------------------
# 8. Model architecture identity
# ------------------------------------------------------------------------------

print("\n[8/22] Exact model architecture audit")
print("-" * 78)

model_config = model.config

observed_hidden = getattr(
    model_config,
    "hidden_size",
    None
)

observed_layers = getattr(
    model_config,
    "num_hidden_layers",
    None
)

observed_q_heads = getattr(
    model_config,
    "num_attention_heads",
    None
)

observed_kv_heads = getattr(
    model_config,
    "num_key_value_heads",
    None
)

observed_intermediate = getattr(
    model_config,
    "intermediate_size",
    None
)

observed_model_type = getattr(
    model_config,
    "model_type",
    None
)

print(f"  model_type: {observed_model_type}")
print(f"  hidden_size: {observed_hidden}")
print(f"  num_hidden_layers: {observed_layers}")
print(f"  num_attention_heads: {observed_q_heads}")
print(f"  num_key_value_heads: {observed_kv_heads}")
print(f"  intermediate_size: {observed_intermediate}")

architecture_identity_pass = (
    observed_model_type == "llama"
    and observed_hidden == EXPECTED_HIDDEN_SIZE
    and observed_layers == 28
    and observed_q_heads == EXPECTED_Q_HEADS
    and observed_kv_heads == EXPECTED_KV_HEADS
    and observed_intermediate == EXPECTED_INTERMEDIATE_SIZE
)

print(
    f"  expected Llama 3.2 3B architecture identity: "
    f"{architecture_identity_pass}"
)

if not architecture_identity_pass:
    raise RuntimeError(
        "Observed model architecture does not match "
        "the audited Llama 3.2 3B configuration."
    )

print("  architecture identity: PASS")


# ------------------------------------------------------------------------------
# 9. Parameter residency / dtype
# ------------------------------------------------------------------------------

print("\n[9/22] Parameter residency / dtype audit")
print("-" * 78)

parameters = list(model.parameters())
buffers = list(model.buffers())

parameter_count = sum(
    p.numel()
    for p in parameters
)

parameter_devices = Counter(
    str(p.device)
    for p in parameters
)

parameter_dtypes = Counter(
    str(p.dtype)
    for p in parameters
)

buffer_devices = Counter(
    str(b.device)
    for b in buffers
)

print(f"  parameter tensor count: {len(parameters)}")
print(f"  parameter count: {parameter_count:,}")
print(f"  parameter devices: {dict(parameter_devices)}")
print(f"  parameter dtypes: {dict(parameter_dtypes)}")
print(f"  buffer devices: {dict(buffer_devices)}")

all_parameters_cuda = all(
    p.device.type == "cuda"
    for p in parameters
)

all_parameters_bf16 = all(
    p.dtype == torch.bfloat16
    for p in parameters
)

if not all_parameters_cuda:
    raise RuntimeError(
        "Model parameters are not fully resident on CUDA."
    )

if not all_parameters_bf16:
    raise RuntimeError(
        "Model parameters are not uniformly BF16."
    )

print("  model parameter residency: PASS")
print("  model parameter dtype: PASS")


# ------------------------------------------------------------------------------
# 10. Selected layer implementation audit
# ------------------------------------------------------------------------------

print("\n[10/22] Selected-layer implementation audit")
print("-" * 78)

layers = getattr(
    getattr(model, "model", None),
    "layers",
    None
)

if layers is None:
    raise RuntimeError(
        "Could not locate model.model.layers."
    )

if SELECTED_LAYER < 0 or SELECTED_LAYER >= len(layers):
    raise RuntimeError(
        f"Selected layer {SELECTED_LAYER} outside layer range."
    )

decoder_layer = layers[SELECTED_LAYER]

attention_module = getattr(
    decoder_layer,
    "self_attn",
    None
)

mlp_module = getattr(
    decoder_layer,
    "mlp",
    None
)

input_norm_module = getattr(
    decoder_layer,
    "input_layernorm",
    None
)

post_attention_norm_module = getattr(
    decoder_layer,
    "post_attention_layernorm",
    None
)

print(f"  selected layer: {SELECTED_LAYER}")
print(
    f"  decoder layer class: "
    f"{type(decoder_layer).__name__}"
)
print(
    f"  attention module: "
    f"{type(attention_module).__name__}"
)
print(
    f"  MLP module: "
    f"{type(mlp_module).__name__}"
)
print(
    f"  input RMSNorm: "
    f"{type(input_norm_module).__name__}"
)
print(
    f"  post-attention RMSNorm: "
    f"{type(post_attention_norm_module).__name__}"
)

layer_structure_pass = (
    attention_module is not None
    and mlp_module is not None
    and input_norm_module is not None
    and post_attention_norm_module is not None
)

if not layer_structure_pass:
    raise RuntimeError(
        "Selected layer lacks expected decoder components."
    )

print("  selected-layer structure: PASS")


# ------------------------------------------------------------------------------
# 11. Actual installed source audit
# ------------------------------------------------------------------------------

print("\n[11/22] Actual installed Llama forward-source audit")
print("-" * 78)

decoder_forward_source = inspect.getsource(
    decoder_layer.__class__.forward
)

attention_forward_source = inspect.getsource(
    attention_module.__class__.forward
)

mlp_forward_source = inspect.getsource(
    mlp_module.__class__.forward
)

source_checks = {
    "decoder_has_input_layernorm":
        "input_layernorm" in decoder_forward_source,

    "decoder_has_self_attention":
        "self_attn" in decoder_forward_source,

    "decoder_has_post_attention_layernorm":
        "post_attention_layernorm" in decoder_forward_source,

    "decoder_has_mlp":
        ".mlp" in decoder_forward_source
        or "self.mlp" in decoder_forward_source,

    "decoder_has_attention_residual_add":
        "residual + hidden_states" in decoder_forward_source
        or "hidden_states = residual + hidden_states"
        in decoder_forward_source,

    "decoder_has_mlp_residual_add":
        "residual + hidden_states" in decoder_forward_source,

    "attention_has_q_proj":
        "q_proj" in attention_forward_source,

    "attention_has_k_proj":
        "k_proj" in attention_forward_source,

    "attention_has_v_proj":
        "v_proj" in attention_forward_source,

    "attention_has_o_proj":
        "o_proj" in attention_forward_source,

    "mlp_has_gate_proj":
        "gate_proj" in mlp_forward_source,

    "mlp_has_up_proj":
        "up_proj" in mlp_forward_source,

    "mlp_has_down_proj":
        "down_proj" in mlp_forward_source,
}

for key, value in source_checks.items():
    print(f"  {key}: {value}")

source_structure_pass = all(
    source_checks.values()
)

print(
    f"  source-level structural audit: "
    f"{source_structure_pass}"
)

if not source_structure_pass:
    raise RuntimeError(
        "Installed Llama source failed structural audit."
    )


# ------------------------------------------------------------------------------
# 12. Operational sector definitions
# ------------------------------------------------------------------------------

print("\n[12/22] Operational-sector definition audit")
print("-" * 78)

sector_definitions = {

    "S1": {
        "name": "contextual_interaction",
        "module":
            "model.model.layers[14].self_attn",
        "capture":
            "attention module output[0]",
        "residual_inclusion": False,
        "state_role":
            "attention output before decoder residual addition",
    },

    "S2": {
        "name": "state_propagation",
        "module":
            "model.model.layers[14]",
        "capture":
            "decoder-layer input hidden_states",
        "residual_inclusion": True,
        "state_role":
            "residual state entering selected decoder layer",
    },

    "S3": {
        "name": "feature_transformation",
        "module":
            "model.model.layers[14].mlp",
        "capture":
            "MLP module output",
        "residual_inclusion": False,
        "state_role":
            "MLP output before decoder residual addition",
    },
}

for sector, definition in sector_definitions.items():

    print(f"  {sector}: {definition['name']}")
    print(f"    module: {definition['module']}")
    print(f"    capture: {definition['capture']}")
    print(f"    role: {definition['state_role']}")

print("")
print(
    "  governance: S1/S2/S3 are operational measurement sectors, "
    "not ontological Transformer modules."
)

print("  sector definition: PASS")


# ------------------------------------------------------------------------------
# 13. Synthetic prediction-boundary audit
# ------------------------------------------------------------------------------

print("\n[13/22] Synthetic prediction-boundary instrumentation audit")
print("-" * 78)

resolved_token_for_tokenizer = get_token()

if resolved_token_for_tokenizer is None:
    raise RuntimeError(
        "No Hugging Face credential available for tokenizer loading."
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=resolved_token_for_tokenizer,
    use_fast=True,
)

del resolved_token_for_tokenizer

encoding = tokenizer(
    SYNTHETIC_PROMPT,
    add_special_tokens=False,
    return_tensors="pt",
    return_attention_mask=True,
)

input_ids = encoding["input_ids"].to(DEVICE)
attention_mask = encoding["attention_mask"].to(DEVICE)

prompt_length = int(
    input_ids.shape[1]
)

if prompt_length < 2:
    raise RuntimeError(
        "Synthetic prompt must contain at least two tokens."
    )

prediction_position = prompt_length
state_index = prompt_length - 1

print(
    f"  synthetic prompt: {SYNTHETIC_PROMPT}"
)
print(
    f"  prompt token length L: {prompt_length}"
)
print(
    f"  next-token prediction position p_T: "
    f"{prediction_position}"
)
print(
    f"  state extraction index i_T: "
    f"{state_index}"
)

if prediction_position != state_index + 1:
    raise RuntimeError(
        "Prediction-position/state-index relationship failed."
    )

print("  p_T = L(P): PASS")
print("  i_T = L(P)-1: PASS")


# ------------------------------------------------------------------------------
# 14. Synthetic baseline inference
# ------------------------------------------------------------------------------

print("\n[14/22] Synthetic baseline inference")
print("-" * 78)

model.eval()

with torch.inference_mode():

    baseline_outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )

baseline_logits = baseline_outputs.logits

print(
    f"  logits shape: "
    f"{tuple(baseline_logits.shape)}"
)

print(
    f"  logits dtype: "
    f"{baseline_logits.dtype}"
)

print(
    f"  logits device: "
    f"{baseline_logits.device}"
)

expected_logits_shape = (
    1,
    prompt_length,
    getattr(model.config, "vocab_size", None),
)

baseline_shape_pass = (
    tuple(baseline_logits.shape)
    == expected_logits_shape
)

baseline_finite_pass = bool(
    torch.isfinite(
        baseline_logits
    ).all().item()
)

print(
    f"  logits shape correct: "
    f"{baseline_shape_pass}"
)

print(
    f"  logits finite: "
    f"{baseline_finite_pass}"
)

if not baseline_shape_pass or not baseline_finite_pass:
    raise RuntimeError(
        "Synthetic baseline inference failed."
    )

next_token_logits = baseline_logits[
    :,
    state_index,
    :
]

print(
    f"  next-token prediction logits slice: "
    f"{tuple(next_token_logits.shape)}"
)

if next_token_logits.shape != (
    1,
    getattr(model.config, "vocab_size", None)
):
    raise RuntimeError(
        "Next-token prediction slice has unexpected shape."
    )

print("  baseline inference: PASS")


# ------------------------------------------------------------------------------
# 15. Sector hook registration
# ------------------------------------------------------------------------------

print("\n[15/22] Sector-hook instrumentation definition")
print("-" * 78)

captured = {
    "S1": None,
    "S2": None,
    "S3": None,
}

hook_errors = []


def make_sector_hook(sector_name):

    def hook(module, module_inputs, module_output):

        try:

            if sector_name == "S1":

                if isinstance(module_output, tuple):
                    value = module_output[0]
                else:
                    value = module_output

            elif sector_name == "S2":

                if not module_inputs:
                    raise RuntimeError(
                        "Decoder-layer hook received no positional inputs."
                    )

                value = module_inputs[0]

            elif sector_name == "S3":

                value = module_output

            else:

                raise RuntimeError(
                    f"Unknown sector: {sector_name}"
                )

            if not torch.is_tensor(value):
                raise RuntimeError(
                    f"{sector_name} capture is not a tensor: "
                    f"{type(value).__name__}"
                )

            captured[sector_name] = value.detach()

        except Exception as exc:

            hook_errors.append(
                {
                    "sector": sector_name,
                    "error": repr(exc),
                }
            )

    return hook


hook_s1 = attention_module.register_forward_hook(
    make_sector_hook("S1")
)

hook_s2 = decoder_layer.register_forward_hook(
    make_sector_hook("S2")
)

hook_s3 = mlp_module.register_forward_hook(
    make_sector_hook("S3")
)

print("  S1 hook registered: attention output")
print("  S2 hook registered: decoder-layer input")
print("  S3 hook registered: MLP output")
print("  hook registration: PASS")


# ------------------------------------------------------------------------------
# 16. Hooked inference / non-interference
# ------------------------------------------------------------------------------

print("\n[16/22] Hooked inference / non-interference audit")
print("-" * 78)

captured = {
    "S1": None,
    "S2": None,
    "S3": None,
}

hook_errors.clear()

with torch.inference_mode():

    hooked_outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )

hooked_logits = hooked_outputs.logits

print(
    f"  hooked logits shape: "
    f"{tuple(hooked_logits.shape)}"
)

if hook_errors:
    raise RuntimeError(
        f"Hook execution errors: {hook_errors}"
    )

if not torch.equal(
    baseline_logits,
    hooked_logits
):

    max_abs_logit_diff = float(
        (
            baseline_logits - hooked_logits
        ).abs().max().item()
    )

    relative_logit_diff = float(
        (
            (
                baseline_logits - hooked_logits
            ).abs().max()
            /
            baseline_logits.abs()
            .max()
            .clamp_min(1e-12)
        ).item()
    )

    print(
        f"  hook non-interference max abs diff: "
        f"{max_abs_logit_diff:.8e}"
    )

    print(
        f"  hook non-interference relative diff: "
        f"{relative_logit_diff:.8e}"
    )

    raise RuntimeError(
        "Hooked inference changed model logits."
    )

print("  hook non-interference: PASS")


# ------------------------------------------------------------------------------
# 17. Sector tensor / target-position audit
# ------------------------------------------------------------------------------

print("\n[17/22] Sector tensor / target-position audit")
print("-" * 78)

sector_results = {}

expected_sector_shape = (
    1,
    prompt_length,
    EXPECTED_HIDDEN_SIZE,
)

for sector in ("S1", "S2", "S3"):

    value = captured[sector]

    if value is None:
        raise RuntimeError(
            f"{sector} hook did not capture a tensor."
        )

    shape = tuple(value.shape)
    dtype = value.dtype
    device = value.device

    print(f"  {sector}:")
    print(f"    shape: {shape}")
    print(f"    dtype: {dtype}")
    print(f"    device: {device}")

    shape_pass = (
        shape == expected_sector_shape
    )

    dtype_pass = (
        dtype == torch.bfloat16
    )

    device_pass = (
        device.type == "cuda"
    )

    finite_pass = bool(
        torch.isfinite(value).all().item()
    )

    target_position_in_bounds = (
        0 <= state_index < value.shape[1]
    )

    target_state = value[
        :,
        state_index,
        :
    ]

    target_state_shape_pass = (
        tuple(target_state.shape)
        == (1, EXPECTED_HIDDEN_SIZE)
    )

    target_state_finite = bool(
        torch.isfinite(target_state).all().item()
    )

    target_norm = float(
        torch.linalg.vector_norm(
            target_state.float(),
            dim=-1
        ).mean().item()
    )

    print(
        f"    expected sequence shape: "
        f"{expected_sector_shape}"
    )

    print(
        f"    shape pass: {shape_pass}"
    )

    print(
        f"    BF16 dtype pass: {dtype_pass}"
    )

    print(
        f"    CUDA device pass: {device_pass}"
    )

    print(
        f"    all finite: {finite_pass}"
    )

    print(
        f"    target state index in bounds: "
        f"{target_position_in_bounds}"
    )

    print(
        f"    target state shape: "
        f"{tuple(target_state.shape)}"
    )

    print(
        f"    target state shape pass: "
        f"{target_state_shape_pass}"
    )

    print(
        f"    target state finite: "
        f"{target_state_finite}"
    )

    print(
        f"    target state L2 norm: "
        f"{target_norm:.8f}"
    )

    if not (
        shape_pass
        and dtype_pass
        and device_pass
        and finite_pass
        and target_position_in_bounds
        and target_state_shape_pass
        and target_state_finite
        and target_norm > 0
    ):
        raise RuntimeError(
            f"{sector} failed numerical tensor/target-position audit."
        )

    sector_results[sector] = {
        "shape": list(shape),
        "dtype": str(dtype),
        "device": str(device),
        "shape_pass": shape_pass,
        "dtype_pass": dtype_pass,
        "device_pass": device_pass,
        "finite_pass": finite_pass,
        "target_state_index": state_index,
        "target_state_shape":
            list(target_state.shape),
        "target_state_shape_pass":
            target_state_shape_pass,
        "target_state_finite":
            target_state_finite,
        "target_state_norm":
            target_norm,
    }

print("  all three sector captures: PASS")


# ------------------------------------------------------------------------------
# 18. Sector distinction / aliasing audit
# ------------------------------------------------------------------------------

print("\n[18/22] Sector distinction / aliasing audit")
print("-" * 78)

sector_pair_diagnostics = {}

for sector_a, sector_b in (
    ("S1", "S2"),
    ("S1", "S3"),
    ("S2", "S3"),
):

    a = captured[
        sector_a
    ][:, state_index, :].float()

    b = captured[
        sector_b
    ][:, state_index, :].float()

    cosine = float(
        torch.nn.functional.cosine_similarity(
            a,
            b,
            dim=-1
        ).mean().item()
    )

    difference_norm = float(
        torch.linalg.vector_norm(
            a - b,
            dim=-1
        ).mean().item()
    )

    exact_equal = bool(
        torch.equal(a, b)
    )

    sector_pair_diagnostics[
        f"{sector_a}_vs_{sector_b}"
    ] = {
        "cosine": cosine,
        "difference_norm": difference_norm,
        "exact_equal": exact_equal,
    }

    print(
        f"  {sector_a} vs {sector_b}: "
        f"cosine={cosine:.8f}, "
        f"difference_norm={difference_norm:.8f}, "
        f"exact_equal={exact_equal}"
    )

all_sector_pairs_nonidentical = all(
    not d["exact_equal"]
    for d in sector_pair_diagnostics.values()
)

if not all_sector_pairs_nonidentical:
    raise RuntimeError(
        "At least two operational sectors are numerically identical "
        "on the synthetic audit prompt."
    )

print("  sector non-aliasing sanity: PASS")


# ------------------------------------------------------------------------------
# 19. Parameter immutability audit
# ------------------------------------------------------------------------------

print("\n[19/22] Parameter immutability audit")
print("-" * 78)

print(
    "  fingerprint representation: CPU float32 copy"
)
print(
    "  scientific model representation remains: BF16 CUDA"
)
print(
    "  fingerprint conversion modifies model parameters: False"
)

parameter_fingerprints_before = [
    parameter_fingerprint(p)
    for p in model.parameters()
]

parameter_fingerprints_after = [
    parameter_fingerprint(p)
    for p in model.parameters()
]

parameter_immutability = (
    parameter_fingerprints_before
    == parameter_fingerprints_after
)

print(
    f"  parameter tensors fingerprinted: "
    f"{len(parameter_fingerprints_before)}"
)

print(
    f"  parameters unchanged: "
    f"{parameter_immutability}"
)

if not parameter_immutability:
    raise RuntimeError(
        "Model parameters changed during instrumentation."
    )

print("  parameter immutability: PASS")


# ------------------------------------------------------------------------------
# 20. Hook cleanup
# ------------------------------------------------------------------------------

print("\n[20/22] Hook cleanup")
print("-" * 78)

hook_s1.remove()
hook_s2.remove()
hook_s3.remove()

print("  S1 hook removed: True")
print("  S2 hook removed: True")
print("  S3 hook removed: True")
print("  hook cleanup: PASS")


# ------------------------------------------------------------------------------
# 21. Final scientific integrity / classification
# ------------------------------------------------------------------------------

print("\n[21/22] Final scientific integrity / classification")
print("-" * 78)

post_audit_dataset_bytes = canonical_dataset_bytes(
    llama_records
)

post_audit_dataset_sha = hashlib.sha256(
    post_audit_dataset_bytes
).hexdigest()

dataset_unchanged = (
    post_audit_dataset_sha
    == pre_audit_dataset_sha
    == EXPECTED_DATASET_SHA
)

print(
    f"  pre-audit dataset SHA-256: "
    f"{pre_audit_dataset_sha}"
)

print(
    f"  post-audit dataset SHA-256: "
    f"{post_audit_dataset_sha}"
)

print(
    f"  dataset unchanged: "
    f"{dataset_unchanged}"
)

if not dataset_unchanged:
    raise RuntimeError(
        "CRITICAL: authenticated dataset changed during Phase 1D.0."
    )

core_runtime_pass = (
    architecture_identity_pass
    and layer_structure_pass
    and source_structure_pass
    and baseline_shape_pass
    and baseline_finite_pass
    and all_sector_pairs_nonidentical
    and parameter_immutability
)

all_sector_pass = all(
    result["shape_pass"]
    and result["dtype_pass"]
    and result["device_pass"]
    and result["finite_pass"]
    and result["target_state_shape_pass"]
    and result["target_state_finite"]
    for result in sector_results.values()
)

if (
    core_runtime_pass
    and all_sector_pass
    and dataset_unchanged
):

    classification = (
        "STATE_EXTRACTION_OPERATIONALIZATION_PASS"
    )

    interpretation = (
        "The audited Llama implementation supports the frozen "
        "target-position and three-sector state-extraction convention. "
        "The next-token prediction position is p_T=L(P), while the "
        "corresponding observed hidden-state extraction index is "
        "i_T=L(P)-1. At the selected layer, attention output, "
        "decoder-layer input residual state, and MLP output are "
        "independently capturable at that observed final-token index "
        "with the expected sequence and hidden dimensions, BF16 CUDA "
        "residency, finite values, hook non-interference, and parameter "
        "immutability. The authenticated dataset and CTL mathematical "
        "specification remain unchanged."
    )

else:

    classification = (
        "NUMERICAL_OR_OPERATIONAL_DECOHERENCE"
    )

    interpretation = (
        "The Llama state-extraction convention could not be fully "
        "validated at the numerical/operational layer. The CTL "
        "mathematical specification remains authoritative and unchanged, "
        "and the authenticated dataset remains frozen. No scientific "
        "state bank should be created until the failed instrumentation "
        "condition is resolved."
    )

print(f"  classification: {classification}")
print(f"  interpretation: {interpretation}")


# ------------------------------------------------------------------------------
# 22. Manifest / final state
# ------------------------------------------------------------------------------

print("\n[22/22] Manifest / final state")
print("-" * 78)

manifest = {
    "experiment_id": EXPERIMENT_ID,
    "phase": "PHASE_1D.0",
    "audit":
        "LLAMA_STATE_EXTRACTION_LAYER_SECTOR_TARGET_POSITION_AUDIT",
    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "model_id": MODEL_ID,
    "selected_layer": SELECTED_LAYER,

    "dataset": {
        "record_count": len(llama_records),
        "sha256_pre_audit": pre_audit_dataset_sha,
        "sha256_post_audit": post_audit_dataset_sha,
        "expected_authenticated_sha256":
            EXPECTED_DATASET_SHA,
        "unchanged": dataset_unchanged,
    },

    "frozen_target_position": {
        "prediction_position":
            TARGET_PREDICTION_POSITION_DEFINITION,
        "state_extraction_index":
            TARGET_STATE_INDEX_DEFINITION,
        "interpretation":
            "final observed prompt token state produces next-token prediction",
        "target_token_not_fed_as_input": True,
    },

    "model_architecture": {
        "model_type": observed_model_type,
        "hidden_size": observed_hidden,
        "num_hidden_layers": observed_layers,
        "num_attention_heads": observed_q_heads,
        "num_key_value_heads": observed_kv_heads,
        "intermediate_size": observed_intermediate,
        "architecture_identity_pass":
            architecture_identity_pass,
    },

    "model_runtime": {
        "device": str(DEVICE),
        "parameter_count": parameter_count,
        "parameter_tensor_count": len(parameters),
        "parameter_devices": dict(parameter_devices),
        "parameter_dtypes": dict(parameter_dtypes),
        "buffer_devices": dict(buffer_devices),
    },

    "sector_definitions": sector_definitions,

    "sector_results": sector_results,

    "sector_pair_diagnostics":
        sector_pair_diagnostics,

    "source_audit": {
        "checks": source_checks,
        "pass": source_structure_pass,
    },

    "baseline_inference": {
        "prompt": SYNTHETIC_PROMPT,
        "prompt_length": prompt_length,
        "prediction_position": prediction_position,
        "state_index": state_index,
        "logits_shape":
            list(baseline_logits.shape),
        "logits_dtype":
            str(baseline_logits.dtype),
        "logits_device":
            str(baseline_logits.device),
        "shape_pass":
            baseline_shape_pass,
        "finite_pass":
            baseline_finite_pass,
    },

    "hook_audit": {
        "hook_noninterference": True,
        "hook_errors": hook_errors,
        "cleanup_complete": True,
    },

    "parameter_immutability": {
        "pass": parameter_immutability,
        "fingerprint_count":
            len(parameter_fingerprints_before),
        "fingerprint_representation":
            "CPU float32 copy used solely for hashing",
        "model_parameter_dtype_unchanged":
            True,
    },

    "scientific_governance": {
        "dataset_modified": False,
        "prompts_modified": False,
        "transport_fitting_performed": False,
        "pca_performed": False,
        "learned_maps_fitted": False,
        "state_bank_created": False,
        "ctl_mathematics_modified": False,
        "synthetic_prompt_used_as_scientific_data":
            False,
        "operationalization_layer_only": True,
    },

    "classification": classification,
    "interpretation": interpretation,

    "next_stage": (
        "PHASE_1D.1_STATE_EXTRACTION_PRECURSOR_AUDIT"
        if classification
        == "STATE_EXTRACTION_OPERATIONALIZATION_PASS"
        else
        "HOLD_PENDING_NUMERICAL_OR_OPERATIONAL_RESOLUTION"
    ),
}

with open(
    OUTPUT_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )

print(
    f"  manifest: {OUTPUT_MANIFEST_PATH}"
)

print("  manifest written: True")


# ------------------------------------------------------------------------------
# Cleanup temporary diagnostic tensors.
#
# IMPORTANT:
# Keep the authenticated llama_records untouched.
# Keep the model resident for the next authorized stage.
# ------------------------------------------------------------------------------

for name in (
    "baseline_outputs",
    "hooked_outputs",
    "baseline_logits",
    "hooked_logits",
    "next_token_logits",
    "input_ids",
    "attention_mask",
    "encoding",
    "captured",
    "tokenizer",
):
    try:
        del globals()[name]
    except Exception:
        pass

gc.collect()


# ------------------------------------------------------------------------------
# Final status
# ------------------------------------------------------------------------------

print("\n" + "=" * 78)
print("FINAL STATUS")
print("=" * 78)

print(
    f"  classification: {classification}"
)

print(
    "  selected layer: "
    f"{SELECTED_LAYER}"
)

print(
    "  frozen prediction position: "
    "p_T = L(P)"
)

print(
    "  frozen state extraction index: "
    "i_T = L(P)-1"
)

print(
    "  S1 attention output capture: "
    f"{'PASS' if 'S1' in sector_results else 'FAIL'}"
)

print(
    "  S2 decoder-input capture: "
    f"{'PASS' if 'S2' in sector_results else 'FAIL'}"
)

print(
    "  S3 MLP-output capture: "
    f"{'PASS' if 'S3' in sector_results else 'FAIL'}"
)

print(
    "  hook non-interference: PASS"
)

print(
    "  parameter immutability: "
    f"{parameter_immutability}"
)

print(
    "  parameter scientific dtype: "
    "BF16"
)

print(
    "  fingerprint conversion altered model: False"
)

print(
    "  dataset mutated: "
    f"{not dataset_unchanged}"
)

print(
    "  transport fitting performed: False"
)

print(
    "  PCA performed: False"
)

print(
    "  state bank created: False"
)

print(
    "  CTL mathematics modified: False"
)

print(
    f"  manifest: {OUTPUT_MANIFEST_PATH}"
)

print("=" * 78)
print("PHASE 1D.0 COMPLETE")
print(
    "State-extraction instrumentation has been audited without "
    "changing the authenticated dataset or CTL mathematical specification."
)
print("=" * 78)

ETTR-CTL-LLAMA-1 — PHASE_1D.0
LLAMA STATE-EXTRACTION / LAYER-SECTOR / TARGET-POSITION AUDIT

[1/22] Scientific-state firewall
------------------------------------------------------------------------------
  record count: 192
  pre-audit dataset SHA-256: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  expected authenticated SHA-256: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  dataset firewall: PASS

[2/22] Dataset authorization prerequisite
------------------------------------------------------------------------------
  dataset_authorized: True
  authorized SHA-256: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  authorized record count: 192
  authorization prerequisite: PASS

[3/22] Phase 1C.3 target-position prerequisite
------------------------------------------------------------------------------
  Phase 1C.3 classification: TARGET_POSITION_OPERATIONALIZATION_PASS
  frozen prompt tokenization: tokenizer(P, add_special_toke

In [20]:
# ==============================================================================
# ETTR-CTL-LLAMA-1
# Phase 1D.2 — Full Llama State Bank Extraction
#
# NEW CELL — do not remove previous completed cells/artifacts.
#
# PURPOSE
# -------
# Construct the full empirical state bank for the frozen Llama experiment:
#
#     192 records
#     × 2 conditions (clean/corrupt)
#     × 3 operational sectors (S1/S2/S3)
#     × 3072 hidden dimensions
#
# This cell establishes the empirical substrate for subsequent transport
# analysis. It does NOT perform:
#
#     - PCA
#     - dimensionality reduction
#     - transport fitting
#     - contextual realization
#     - triadic irreducibility analysis
#     - geometric reconstruction
#     - CTL mathematical modification
#
# FROZEN TARGET CONVENTION
# ------------------------
#
#     p_T = L(P)
#     i_T = p_T - 1 = L(P) - 1
#
# The model receives ONLY the prompt P.
# The target token is never appended to the model input.
#
# OPERATIONAL SECTORS
# -------------------
#
#     S1 = attention output before residual addition
#     S2 = decoder-layer input hidden_states
#     S3 = MLP output before residual addition
#
# These are operational measurement sectors, NOT claims that the Transformer
# possesses three ontologically independent modules.
#
# GOVERNING PRINCIPLE
# -------------------
#
# The CTL mathematics is authoritative. Software, CUDA, PyTorch, tokenizer,
# or hardware constraints may not redefine the mathematical object being
# tested. Any genuine disparity is recorded as decoherence rather than used
# to alter the theory.
#
# ==============================================================================

import os
import gc
import json
import hashlib
import random
from pathlib import Path

import numpy as np
import torch


# ------------------------------------------------------------------------------
# 0. EXPERIMENT CONSTANTS
# ------------------------------------------------------------------------------

SEED = 42

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
MODEL_ID = "meta-llama/Llama-3.2-3B"

SELECTED_LAYER = 14
HIDDEN_SIZE = 3072

DEVICE = torch.device("cuda:0")
DTYPE = torch.bfloat16

ROOT = Path("/content/ettr_ctl_llama")
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

AUTH_PATH = (
    RESULTS_DIR
    / "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

PHASE1C3_PATH = (
    RESULTS_DIR
    / "llama_phase1c3_target_position_operationalization_audit.json"
)

PHASE1D0_PATH = (
    RESULTS_DIR
    / "llama_phase1d0_state_extraction_audit.json"
)

PHASE1D1_PATH = (
    RESULTS_DIR
    / "llama_phase1d1_state_extraction_precursor_audit_v5.json"
)

EXPECTED_DATASET_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)

STATE_BANK_PATH = (
    RESULTS_DIR
    / "llama_phase1d2_full_state_bank.npz"
)

MANIFEST_PATH = (
    RESULTS_DIR
    / "llama_phase1d2_full_state_bank_manifest.json"
)


# ------------------------------------------------------------------------------
# 1. DETERMINISTIC SEEDS
# ------------------------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


print("=" * 78)
print("ETTR-CTL-LLAMA-1 | Phase 1D.2 Full State Bank Extraction")
print("=" * 78)

print(f"Experiment: {EXPERIMENT_ID}")
print(f"Model: {MODEL_ID}")
print(f"Selected layer: {SELECTED_LAYER}")
print(f"Hidden size: {HIDDEN_SIZE}")
print("Frozen target rule: p_T = L(P)")
print("Frozen state rule: i_T = p_T - 1 = L(P) - 1")
print(f"Seed: {SEED}")
print()


# ------------------------------------------------------------------------------
# 2. RUNTIME OBJECT FIREWALL
# ------------------------------------------------------------------------------

required_runtime_objects = [
    "model",
    "tokenizer",
]

missing_objects = [
    name
    for name in required_runtime_objects
    if name not in globals() or globals()[name] is None
]

assert not missing_objects, (
    "Required runtime objects missing: "
    + ", ".join(missing_objects)
)

assert torch.cuda.is_available(), "CUDA is required."

print("RUNTIME OBJECT FIREWALL")
print("  Model object: PASS")
print("  Tokenizer object: PASS")
print()


# ------------------------------------------------------------------------------
# 3. DATASET AUTHORIZATION AND IMMUTABILITY
# ------------------------------------------------------------------------------

assert AUTH_PATH.exists(), (
    f"Missing authorization artifact: {AUTH_PATH}"
)

with open(AUTH_PATH, "r", encoding="utf-8") as f:
    auth = json.load(f)

assert bool(auth.get("dataset_authorized", False)), (
    "Dataset authorization flag is not PASS."
)

source_artifact = auth.get("source_artifact")
source_path = auth.get("source_path")

assert source_artifact is not None
assert source_path is not None

candidate_dataset_paths = [
    Path(source_path),
    Path("/content") / source_artifact,
    ROOT / source_artifact,
    RESULTS_DIR / source_artifact,
]

resolved_dataset_path = None

for candidate in candidate_dataset_paths:
    if candidate.exists():
        resolved_dataset_path = candidate
        break

assert resolved_dataset_path is not None, (
    "Authorized operative dataset could not be resolved.\n"
    f"source_artifact={source_artifact}\n"
    f"source_path={source_path}"
)

with open(resolved_dataset_path, "rb") as f:
    dataset_bytes_before = f.read()

dataset_sha_before = hashlib.sha256(
    dataset_bytes_before
).hexdigest()

assert dataset_sha_before == EXPECTED_DATASET_SHA256, (
    "Dataset SHA mismatch before state extraction.\n"
    f"Expected: {EXPECTED_DATASET_SHA256}\n"
    f"Observed: {dataset_sha_before}"
)

with open(resolved_dataset_path, "r", encoding="utf-8") as f:
    llama_records = json.load(f)

assert isinstance(llama_records, list)
assert len(llama_records) == 192


print("DATASET AUTHORIZATION")
print("  Authorization: PASS")
print(f"  Source artifact: {source_artifact}")
print(f"  Resolved path: {resolved_dataset_path}")
print(f"  Records: {len(llama_records)}")
print(f"  SHA256: {dataset_sha_before}")
print("  Frozen dataset identity: PASS")
print()


# ------------------------------------------------------------------------------
# 4. DATASET STRUCTURAL FIREWALL
# ------------------------------------------------------------------------------

required_fields = {
    "clean_prompt",
    "clean_target_name",
    "clean_target_role",
    "clean_target_token_id",
    "condition_pair",
    "corrupt_prompt",
    "corrupt_target_name",
    "corrupt_target_role",
    "corrupt_target_token_id",
    "example_id",
    "first_name",
    "name_pair_id",
    "name_pair_index",
    "second_name",
    "split",
    "template_id",
    "template_index",
}

for idx, record in enumerate(llama_records):

    missing = required_fields - set(record.keys())

    assert not missing, (
        f"Record {idx} missing fields: {sorted(missing)}"
    )


split_counts = {
    split: sum(
        1
        for r in llama_records
        if r["split"] == split
    )
    for split in ["train", "calibration", "test"]
}

assert split_counts == {
    "train": 96,
    "calibration": 48,
    "test": 48,
}

assert len(
    {
        r["example_id"]
        for r in llama_records
    }
) == 192


print("DATASET STRUCTURAL FIREWALL")
print("  Required fields: PASS")
print("  Unique example IDs: PASS")
print("  Train: 96")
print("  Calibration: 48")
print("  Test: 48")
print()


# ------------------------------------------------------------------------------
# 5. PREREQUISITE AUDIT FIREWALL
# ------------------------------------------------------------------------------

for path in [
    PHASE1C3_PATH,
    PHASE1D0_PATH,
    PHASE1D1_PATH,
]:
    assert path.exists(), f"Missing prerequisite artifact: {path}"


with open(PHASE1C3_PATH, "r", encoding="utf-8") as f:
    phase1c3 = json.load(f)

with open(PHASE1D0_PATH, "r", encoding="utf-8") as f:
    phase1d0 = json.load(f)

with open(PHASE1D1_PATH, "r", encoding="utf-8") as f:
    phase1d1 = json.load(f)


assert (
    phase1c3.get("classification")
    == "TARGET_POSITION_OPERATIONALIZATION_PASS"
)

assert (
    phase1d0.get("classification")
    == "STATE_EXTRACTION_OPERATIONALIZATION_PASS"
)

assert (
    phase1d1.get("classification")
    == "STATE_EXTRACTION_PRECURSOR_OPERATIONALIZATION_PASS"
)


# Verify that Phase 1D.1 used the same frozen dataset.
assert (
    phase1d1["dataset"]["sha256"]
    == EXPECTED_DATASET_SHA256
)

assert phase1d1["dataset"]["records"] == 192

print("PREREQUISITE AUDITS")
print("  Phase 1C.3: PASS")
print("  Phase 1D.0: PASS")
print("  Phase 1D.1: PASS")
print("  Dataset SHA agreement across prerequisites: PASS")
print()


# ------------------------------------------------------------------------------
# 6. MODEL ARCHITECTURE FIREWALL
# ------------------------------------------------------------------------------

assert str(next(model.parameters()).device) == "cuda:0"

parameter_dtypes = {
    str(p.dtype)
    for p in model.parameters()
}

assert parameter_dtypes == {"torch.bfloat16"}, parameter_dtypes

config = model.config

assert int(config.num_hidden_layers) == 28
assert int(config.hidden_size) == 3072
assert int(config.num_attention_heads) == 24
assert int(config.num_key_value_heads) == 8
assert int(config.intermediate_size) == 8192

assert len(model.model.layers) == 28

layer = model.model.layers[SELECTED_LAYER]

assert layer.__class__.__name__ == "LlamaDecoderLayer"
assert layer.self_attn.__class__.__name__ == "LlamaAttention"
assert layer.mlp.__class__.__name__ == "LlamaMLP"

print("MODEL ARCHITECTURE FIREWALL")
print("  28 decoder layers: PASS")
print("  Hidden size 3072: PASS")
print("  Attention heads 24: PASS")
print("  KV heads 8: PASS")
print("  Intermediate size 8192: PASS")
print("  Selected layer 14: PASS")
print("  Parameter dtype BF16: PASS")
print("  Parameter device CUDA: PASS")
print()


# ------------------------------------------------------------------------------
# 7. TOKENIZER FIREWALL
# ------------------------------------------------------------------------------

assert tokenizer is not None

tokenizer_name = getattr(
    tokenizer,
    "name_or_path",
    None,
)

assert tokenizer_name == MODEL_ID, (
    f"Unexpected tokenizer: {tokenizer_name}"
)

tokenizer_vocab_size = len(tokenizer)

assert tokenizer_vocab_size == 128256

print("TOKENIZER FIREWALL")
print(f"  Tokenizer: {tokenizer_name}")
print(f"  Vocabulary entries: {tokenizer_vocab_size}")
print("  add_special_tokens=False convention: PASS")
print()


# ------------------------------------------------------------------------------
# 8. PRE-EXTRACTION MODEL FINGERPRINT
# ------------------------------------------------------------------------------

def parameter_fingerprint(model):
    """
    BF16-safe immutable parameter fingerprint.

    A temporary CPU float32 copy is used only for hashing.
    The scientific model remains BF16 on CUDA.
    """
    h = hashlib.sha256()

    for name, parameter in model.named_parameters():

        h.update(name.encode("utf-8"))
        h.update(
            str(tuple(parameter.shape)).encode("utf-8")
        )
        h.update(
            str(parameter.dtype).encode("utf-8")
        )

        cpu_fp32 = (
            parameter.detach()
            .cpu()
            .float()
            .contiguous()
            .numpy()
        )

        h.update(cpu_fp32.tobytes())

    return h.hexdigest()


model_fingerprint_before = parameter_fingerprint(model)

print("MODEL FINGERPRINT")
print("  Pre-extraction fingerprint: recorded")
print()


# ------------------------------------------------------------------------------
# 9. TARGET POSITION / TARGET ID DERIVATION
#
# This audit is performed again at the point of full extraction so the state
# bank has a self-contained target-position record.
#
# GPT-2 IDs in the recovered dataset remain source metadata.
# Llama IDs are derived from the Llama tokenizer.
# ------------------------------------------------------------------------------

target_metadata = {}

target_position_checks = 0

for record in llama_records:

    example_id = record["example_id"]

    target_metadata[example_id] = {}

    for condition in ["clean", "corrupt"]:

        prompt = record[f"{condition}_prompt"]
        target_name = record[f"{condition}_target_name"]

        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=False,
            return_tensors=None,
        )["input_ids"]

        full_ids = tokenizer(
            prompt + " " + target_name,
            add_special_tokens=False,
            return_tensors=None,
        )["input_ids"]

        L = len(prompt_ids)

        assert L > 0

        assert full_ids[:L] == prompt_ids, (
            f"Prefix mismatch: {example_id}/{condition}"
        )

        continuation = full_ids[L:]

        assert len(continuation) == 1, (
            f"Non-single-token continuation: "
            f"{example_id}/{condition}: {continuation}"
        )

        llama_target_id = int(continuation[0])

        p_T = L
        i_T = L - 1

        # Critical invariant:
        # The actual input to the model later will be exactly prompt_ids.
        # No target token is appended.
        inference_ids = list(prompt_ids)

        assert len(inference_ids) == p_T
        assert inference_ids == prompt_ids

        target_metadata[example_id][condition] = {
            "prompt_length": int(L),
            "p_T": int(p_T),
            "i_T": int(i_T),
            "llama_target_id": llama_target_id,
            "gpt2_reference_target_id": int(
                record[f"{condition}_target_token_id"]
            ),
            "target_continuation_length": 1,
            "target_appended_to_inference_input": False,
            "target_token_occurs_inside_prompt": bool(
                llama_target_id in prompt_ids
            ),
        }

        target_position_checks += 1


assert target_position_checks == 384


# Equal clean/corrupt prompt lengths.
for record in llama_records:

    clean_L = target_metadata[
        record["example_id"]
    ]["clean"]["prompt_length"]

    corrupt_L = target_metadata[
        record["example_id"]
    ]["corrupt"]["prompt_length"]

    assert clean_L == corrupt_L, (
        f"Clean/corrupt length mismatch: "
        f"{record['example_id']}"
    )


print("TARGET POSITION FIREWALL")
print("  384 condition target derivations: PASS")
print("  Exact prompt prefixes: PASS")
print("  One-token target continuations: PASS")
print("  Frozen p_T = L(P): PASS")
print("  Frozen i_T = L(P)-1: PASS")
print("  Clean/corrupt prompt lengths: PASS")
print("  Target appended to inference input: 0/384")
print("  GPT-2 source target IDs preserved: PASS")
print("  Llama target IDs independently derived: PASS")
print()


# ------------------------------------------------------------------------------
# 10. HOOK CAPTURE STORAGE
#
# Full bank is stored as float32 on CPU.
#
# Rationale:
#   - model computation remains BF16 on CUDA;
#   - stored scientific state bank has deterministic numeric representation;
#   - CPU float32 avoids repeated BF16 interpretation during later analysis;
#   - no dimensionality reduction is performed.
#
# This is numerical storage, not a mathematical transformation of the
# underlying state definition.
# ------------------------------------------------------------------------------

N = len(llama_records)
N_CONDITIONS = N * 2

assert N == 192
assert N_CONDITIONS == 384

S1_bank = np.empty(
    (N_CONDITIONS, HIDDEN_SIZE),
    dtype=np.float32,
)

S2_bank = np.empty(
    (N_CONDITIONS, HIDDEN_SIZE),
    dtype=np.float32,
)

S3_bank = np.empty(
    (N_CONDITIONS, HIDDEN_SIZE),
    dtype=np.float32,
)

target_logits = np.empty(
    (N_CONDITIONS,),
    dtype=np.float32,
)

target_logit_ranks = np.empty(
    (N_CONDITIONS,),
    dtype=np.int32,
)

condition_example_ids = []
condition_splits = []
condition_labels = []
condition_template_ids = []
condition_name_pair_ids = []
condition_prompt_lengths = np.empty(
    (N_CONDITIONS,),
    dtype=np.int32,
)

condition_p_T = np.empty(
    (N_CONDITIONS,),
    dtype=np.int32,
)

condition_i_T = np.empty(
    (N_CONDITIONS,),
    dtype=np.int32,
)

condition_llama_target_ids = np.empty(
    (N_CONDITIONS,),
    dtype=np.int32,
)

condition_gpt2_target_ids = np.empty(
    (N_CONDITIONS,),
    dtype=np.int32,
)


# ------------------------------------------------------------------------------
# 11. HOOK DEFINITIONS
# ------------------------------------------------------------------------------

captured = {
    "S1": None,
    "S2": None,
    "S3": None,
}


def s1_hook(module, inputs, output):

    hidden = (
        output[0]
        if isinstance(output, tuple)
        else output
    )

    captured["S1"] = hidden.detach()


def s2_pre_hook(module, inputs):

    hidden = (
        inputs[0]
        if isinstance(inputs, tuple)
        else inputs
    )

    captured["S2"] = hidden.detach()


def s3_hook(module, inputs, output):

    hidden = (
        output[0]
        if isinstance(output, tuple)
        else output
    )

    captured["S3"] = hidden.detach()


h_s1 = layer.self_attn.register_forward_hook(s1_hook)
h_s2 = layer.register_forward_pre_hook(s2_pre_hook)
h_s3 = layer.mlp.register_forward_hook(s3_hook)

print("HOOKS")
print("  S1 attention-output hook: installed")
print("  S2 decoder-input pre-hook: installed")
print("  S3 MLP-output hook: installed")
print()


# ------------------------------------------------------------------------------
# 12. SINGLE CONDITION EXTRACTION
# ------------------------------------------------------------------------------

@torch.no_grad()
def extract_condition(
    prompt,
    llama_target_id,
    expected_p_T,
    expected_i_T,
):
    """
    Extract one condition.

    The model receives ONLY tokenizer(prompt).

    Target token is used solely to select the prediction logit.
    """

    captured["S1"] = None
    captured["S2"] = None
    captured["S3"] = None

    encoding = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    )

    input_ids = encoding["input_ids"].to(
        DEVICE,
        dtype=torch.long,
    )

    attention_mask = encoding.get("attention_mask")

    if attention_mask is not None:
        attention_mask = attention_mask.to(DEVICE)

    actual_L = int(input_ids.shape[1])

    assert actual_L == int(expected_p_T), (
        "Inference input length violates frozen p_T."
    )

    # ------------------------------------------------------------------
    # Explicit no-target-input invariant.
    #
    # Reconstruct prompt-only IDs independently and verify exact identity.
    # ------------------------------------------------------------------

    prompt_only_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors=None,
    )["input_ids"]

    assert list(input_ids[0].detach().cpu().tolist()) == (
        list(prompt_only_ids)
    )

    assert input_ids.shape[1] == int(expected_p_T)

    # ------------------------------------------------------------------
    # Forward pass.
    # ------------------------------------------------------------------

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )

    logits = outputs.logits

    assert logits.shape[0] == 1
    assert logits.shape[1] == actual_L
    assert logits.shape[2] == int(config.vocab_size)

    assert logits.dtype == DTYPE
    assert logits.device.type == "cuda"

    assert torch.isfinite(logits).all()

    # ------------------------------------------------------------------
    # Frozen prediction state index.
    # ------------------------------------------------------------------

    state_index = int(expected_i_T)

    assert state_index == actual_L - 1

    target_vector = logits[
        0,
        state_index,
    ]

    target_logit = target_vector[
        int(llama_target_id)
    ]

    # ------------------------------------------------------------------
    # State capture.
    # ------------------------------------------------------------------

    assert captured["S1"] is not None
    assert captured["S2"] is not None
    assert captured["S3"] is not None

    S1 = captured["S1"][0, state_index]
    S2 = captured["S2"][0, state_index]
    S3 = captured["S3"][0, state_index]

    for name, state in [
        ("S1", S1),
        ("S2", S2),
        ("S3", S3),
    ]:

        assert tuple(state.shape) == (HIDDEN_SIZE,), (
            f"{name} shape failure: {state.shape}"
        )

        assert state.dtype == DTYPE, (
            f"{name} dtype failure: {state.dtype}"
        )

        assert state.device.type == "cuda"

        assert torch.isfinite(state).all(), (
            f"{name} contains non-finite values."
        )

    # ------------------------------------------------------------------
    # Norm sanity.
    # ------------------------------------------------------------------

    norms = {}

    for name, state in [
        ("S1", S1),
        ("S2", S2),
        ("S3", S3),
    ]:

        norm = float(
            torch.linalg.vector_norm(
                state.float()
            ).item()
        )

        assert np.isfinite(norm)
        assert norm > 0.0

        norms[name] = norm

    # ------------------------------------------------------------------
    # Target-logit rank.
    #
    # Rank 1 means the target token has the highest logit.
    # This is descriptive metadata only; it is not used for state fitting.
    # ------------------------------------------------------------------

    rank = int(
        1
        + torch.sum(
            target_vector > target_logit
        ).item()
    )

    return {
        "S1": S1.float().cpu().numpy(),
        "S2": S2.float().cpu().numpy(),
        "S3": S3.float().cpu().numpy(),
        "target_logit": float(
            target_logit.float().item()
        ),
        "target_rank": rank,
        "norms": norms,
    }


# ------------------------------------------------------------------------------
# 13. FULL 384-CONDITION EXTRACTION
# ------------------------------------------------------------------------------

condition_index = 0

record_order = []

for record_index, record in enumerate(llama_records):

    example_id = record["example_id"]

    record_order.append(example_id)

    assert example_id in target_metadata

    for condition in ["clean", "corrupt"]:

        metadata = target_metadata[
            example_id
        ][condition]

        prompt = record[
            f"{condition}_prompt"
        ]

        llama_target_id = metadata[
            "llama_target_id"
        ]

        p_T = metadata["p_T"]
        i_T = metadata["i_T"]

        extraction = extract_condition(
            prompt=prompt,
            llama_target_id=llama_target_id,
            expected_p_T=p_T,
            expected_i_T=i_T,
        )

        S1_bank[
            condition_index
        ] = extraction["S1"]

        S2_bank[
            condition_index
        ] = extraction["S2"]

        S3_bank[
            condition_index
        ] = extraction["S3"]

        target_logits[
            condition_index
        ] = extraction["target_logit"]

        target_logit_ranks[
            condition_index
        ] = extraction["target_rank"]

        condition_example_ids.append(
            example_id
        )

        condition_splits.append(
            record["split"]
        )

        condition_labels.append(
            condition
        )

        condition_template_ids.append(
            record["template_id"]
        )

        condition_name_pair_ids.append(
            record["name_pair_id"]
        )

        condition_prompt_lengths[
            condition_index
        ] = metadata["prompt_length"]

        condition_p_T[
            condition_index
        ] = p_T

        condition_i_T[
            condition_index
        ] = i_T

        condition_llama_target_ids[
            condition_index
        ] = llama_target_id

        condition_gpt2_target_ids[
            condition_index
        ] = metadata[
            "gpt2_reference_target_id"
        ]

        condition_index += 1

        if condition_index % 48 == 0:
            print(
                f"  Extracted conditions: "
                f"{condition_index}/384"
            )


assert condition_index == 384

assert S1_bank.shape == (
    384,
    HIDDEN_SIZE,
)

assert S2_bank.shape == (
    384,
    HIDDEN_SIZE,
)

assert S3_bank.shape == (
    384,
    HIDDEN_SIZE,
)


print()
print("FULL EXTRACTION")
print("  Records: 192")
print("  Clean conditions: 192")
print("  Corrupt conditions: 192")
print("  Total conditions: 384")
print("  S1 states: 384 × 3072")
print("  S2 states: 384 × 3072")
print("  S3 states: 384 × 3072")
print()


# ------------------------------------------------------------------------------
# 14. FINITE-VALUE AND BASIC NUMERICAL AUDIT
# ------------------------------------------------------------------------------

assert np.isfinite(S1_bank).all()
assert np.isfinite(S2_bank).all()
assert np.isfinite(S3_bank).all()
assert np.isfinite(target_logits).all()

assert np.isfinite(
    target_logit_ranks
).all()

S1_norms = np.linalg.norm(
    S1_bank,
    axis=1,
)

S2_norms = np.linalg.norm(
    S2_bank,
    axis=1,
)

S3_norms = np.linalg.norm(
    S3_bank,
    axis=1,
)

assert np.isfinite(S1_norms).all()
assert np.isfinite(S2_norms).all()
assert np.isfinite(S3_norms).all()

assert np.all(S1_norms > 0)
assert np.all(S2_norms > 0)
assert np.all(S3_norms > 0)


print("NUMERICAL FINITENESS")
print("  S1 finite: PASS")
print("  S2 finite: PASS")
print("  S3 finite: PASS")
print("  Target logits finite: PASS")
print("  State norms finite/nonzero: PASS")
print()


# ------------------------------------------------------------------------------
# 15. CONDITION-ORDER / RECORD CORRESPONDENCE AUDIT
# ------------------------------------------------------------------------------

assert len(condition_example_ids) == 384
assert len(condition_splits) == 384
assert len(condition_labels) == 384

for record_index, record in enumerate(llama_records):

    base = 2 * record_index

    assert condition_example_ids[
        base
    ] == record["example_id"]

    assert condition_example_ids[
        base + 1
    ] == record["example_id"]

    assert condition_labels[
        base
    ] == "clean"

    assert condition_labels[
        base + 1
    ] == "corrupt"

    assert condition_splits[
        base
    ] == record["split"]

    assert condition_splits[
        base + 1
    ] == record["split"]


assert len(set(condition_example_ids)) == 192


print("RECORD / CONDITION CORRESPONDENCE")
print("  192 unique example IDs: PASS")
print("  Clean/corrupt pairing: PASS")
print("  Fixed condition ordering: PASS")
print("  Split correspondence: PASS")
print()


# ------------------------------------------------------------------------------
# 16. TARGET-POSITION CONSISTENCY AUDIT
# ------------------------------------------------------------------------------

for k in range(384):

    assert (
        condition_p_T[k]
        == condition_prompt_lengths[k]
    )

    assert (
        condition_i_T[k]
        == condition_p_T[k] - 1
    )

    assert (
        condition_i_T[k] >= 0
    )


assert np.all(
    condition_prompt_lengths[
        0::2
    ]
    ==
    condition_prompt_lengths[
        1::2
    ]
)

print("TARGET POSITION CONSISTENCY")
print("  p_T = L(P): PASS")
print("  i_T = p_T - 1: PASS")
print("  Clean/corrupt lengths equal: PASS")
print()


# ------------------------------------------------------------------------------
# 17. CLEAN/CORRUPT STATE-DIFFERENCE AUDIT
#
# This does not claim causal sufficiency.
# It verifies that the controlled condition produces measurable paired
# representation differences in the extracted sectors.
# ------------------------------------------------------------------------------

paired_delta_norms = {
    "S1": np.empty(192, dtype=np.float64),
    "S2": np.empty(192, dtype=np.float64),
    "S3": np.empty(192, dtype=np.float64),
}

for record_index in range(192):

    clean_idx = 2 * record_index
    corrupt_idx = clean_idx + 1

    assert condition_labels[
        clean_idx
    ] == "clean"

    assert condition_labels[
        corrupt_idx
    ] == "corrupt"

    for sector_name, bank, output in [
        ("S1", S1_bank, paired_delta_norms["S1"]),
        ("S2", S2_bank, paired_delta_norms["S2"]),
        ("S3", S3_bank, paired_delta_norms["S3"]),
    ]:

        delta = (
            bank[corrupt_idx]
            - bank[clean_idx]
        )

        delta_norm = float(
            np.linalg.norm(delta)
        )

        assert np.isfinite(delta_norm)

        output[record_index] = delta_norm


print("CLEAN/CORRUPT STATE DIFFERENCE AUDIT")

for sector_name in ["S1", "S2", "S3"]:

    values = paired_delta_norms[
        sector_name
    ]

    print(
        f"  {sector_name}: "
        f"mean={values.mean():.6f}, "
        f"median={np.median(values):.6f}, "
        f"min={values.min():.6f}, "
        f"max={values.max():.6f}"
    )

    assert np.isfinite(values).all()
    assert np.all(values >= 0)


print("  Paired finite differences: PASS")
print()


# ------------------------------------------------------------------------------
# 18. SECTOR DISTINCTNESS AUDIT
#
# Descriptive only.
#
# We do NOT interpret this as proof that S1/S2/S3 are ontologically
# independent components. It simply checks that the extracted operational
# sectors are numerically non-identical over the full bank.
# ------------------------------------------------------------------------------

sector_equal_counts = {
    "S1_equals_S2": 0,
    "S1_equals_S3": 0,
    "S2_equals_S3": 0,
}

for k in range(384):

    if np.array_equal(
        S1_bank[k],
        S2_bank[k],
    ):
        sector_equal_counts[
            "S1_equals_S2"
        ] += 1

    if np.array_equal(
        S1_bank[k],
        S3_bank[k],
    ):
        sector_equal_counts[
            "S1_equals_S3"
        ] += 1

    if np.array_equal(
        S2_bank[k],
        S3_bank[k],
    ):
        sector_equal_counts[
            "S2_equals_S3"
        ] += 1


assert sector_equal_counts[
    "S1_equals_S2"
] == 0

assert sector_equal_counts[
    "S1_equals_S3"
] == 0

assert sector_equal_counts[
    "S2_equals_S3"
] == 0


print("SECTOR DISTINCTNESS")
print("  S1/S2 exact equality: 0/384")
print("  S1/S3 exact equality: 0/384")
print("  S2/S3 exact equality: 0/384")
print("  Operational-sector distinction: PASS")
print()


# ------------------------------------------------------------------------------
# 19. REPRODUCIBILITY AUDIT
#
# Repeat extraction of one clean and one corrupt condition from the full bank.
# Exact equality is expected under the frozen deterministic runtime.
# ------------------------------------------------------------------------------

repro_indices = [
    0,   # first clean
    1,   # first corrupt
    190, # final clean
    191, # final corrupt
]

for k in repro_indices:

    record_index = k // 2
    condition = (
        "clean"
        if k % 2 == 0
        else "corrupt"
    )

    record = llama_records[
        record_index
    ]

    prompt = record[
        f"{condition}_prompt"
    ]

    metadata = target_metadata[
        record["example_id"]
    ][condition]

    repeated = extract_condition(
        prompt=prompt,
        llama_target_id=metadata[
            "llama_target_id"
        ],
        expected_p_T=metadata["p_T"],
        expected_i_T=metadata["i_T"],
    )

    assert np.array_equal(
        repeated["S1"],
        S1_bank[k],
    )

    assert np.array_equal(
        repeated["S2"],
        S2_bank[k],
    )

    assert np.array_equal(
        repeated["S3"],
        S3_bank[k],
    )

    assert (
        repeated["target_logit"]
        == float(target_logits[k])
    )

    assert (
        repeated["target_rank"]
        == int(target_logit_ranks[k])
    )


print("REPRODUCIBILITY")
print("  First clean: exact PASS")
print("  First corrupt: exact PASS")
print("  Final clean: exact PASS")
print("  Final corrupt: exact PASS")
print("  State and target-logit repeatability: PASS")
print()


# ------------------------------------------------------------------------------
# 20. HOOK NONINTERFERENCE AUDIT
#
# Remove hooks and compare the full logits of one representative condition.
# The hook-enabled and hook-free model outputs must be exactly equal.
# ------------------------------------------------------------------------------

reference_record = llama_records[0]

reference_prompt = reference_record[
    "clean_prompt"
]

reference_metadata = target_metadata[
    reference_record["example_id"]
]["clean"]


# Remove hooks.
h_s1.remove()
h_s2.remove()
h_s3.remove()

captured["S1"] = None
captured["S2"] = None
captured["S3"] = None


@torch.no_grad()
def forward_without_hooks(prompt):

    encoding = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    )

    ids = encoding["input_ids"].to(
        DEVICE,
        dtype=torch.long,
    )

    mask = encoding.get("attention_mask")

    if mask is not None:
        mask = mask.to(DEVICE)

    output = model(
        input_ids=ids,
        attention_mask=mask,
        use_cache=False,
    )

    return output.logits.detach().clone()


baseline_logits = forward_without_hooks(
    reference_prompt
)


# Reinstall hooks.
h_s1 = layer.self_attn.register_forward_hook(s1_hook)
h_s2 = layer.register_forward_pre_hook(s2_pre_hook)
h_s3 = layer.mlp.register_forward_hook(s3_hook)


hooked_reference = extract_condition(
    prompt=reference_prompt,
    llama_target_id=reference_metadata[
        "llama_target_id"
    ],
    expected_p_T=reference_metadata["p_T"],
    expected_i_T=reference_metadata["i_T"],
)

# Reconstruct full hooked logits through an independent forward pass.
captured["S1"] = None
captured["S2"] = None
captured["S3"] = None

encoding = tokenizer(
    reference_prompt,
    add_special_tokens=False,
    return_tensors="pt",
)

reference_ids = encoding["input_ids"].to(
    DEVICE,
    dtype=torch.long,
)

reference_mask = encoding.get(
    "attention_mask"
)

if reference_mask is not None:
    reference_mask = reference_mask.to(DEVICE)

with torch.no_grad():

    hooked_outputs = model(
        input_ids=reference_ids,
        attention_mask=reference_mask,
        use_cache=False,
    )

hooked_logits = hooked_outputs.logits.detach().clone()

assert torch.equal(
    baseline_logits,
    hooked_logits,
), "Hook noninterference failure: logits differ."


print("HOOK NONINTERFERENCE")
print("  Full logits exact equality: PASS")
print("  Hook-induced model-output change: NONE")
print()


# ------------------------------------------------------------------------------
# 21. MODEL PARAMETER IMMUTABILITY
# ------------------------------------------------------------------------------

model_fingerprint_after = parameter_fingerprint(model)

assert (
    model_fingerprint_before
    == model_fingerprint_after
), (
    "Model parameter fingerprint changed "
    "during Phase 1D.2."
)

print("MODEL PARAMETER IMMUTABILITY")
print("  Fingerprint unchanged: PASS")
print()


# ------------------------------------------------------------------------------
# 22. DATASET IMMUTABILITY AFTER EXTRACTION
# ------------------------------------------------------------------------------

with open(resolved_dataset_path, "rb") as f:
    dataset_bytes_after = f.read()

dataset_sha_after = hashlib.sha256(
    dataset_bytes_after
).hexdigest()

assert (
    dataset_sha_after
    == EXPECTED_DATASET_SHA256
)

assert (
    dataset_sha_before
    == dataset_sha_after
)

print("DATASET IMMUTABILITY")
print(f"  SHA256 after extraction: {dataset_sha_after}")
print("  Dataset mutation: NONE")
print()


# ------------------------------------------------------------------------------
# 23. STATE-BANK INTERNAL HASHES
#
# Hash each full state array before serialization. These hashes become part
# of the manifest and permit later detection of accidental state-bank
# mutation.
# ------------------------------------------------------------------------------

def array_sha256(array):
    contiguous = np.ascontiguousarray(array)
    return hashlib.sha256(
        contiguous.tobytes()
    ).hexdigest()


state_hashes = {
    "S1": array_sha256(S1_bank),
    "S2": array_sha256(S2_bank),
    "S3": array_sha256(S3_bank),
    "target_logits": array_sha256(target_logits),
    "target_logit_ranks": array_sha256(
        target_logit_ranks
    ),
}


print("STATE-BANK INTERNAL HASHES")
print(f"  S1: {state_hashes['S1']}")
print(f"  S2: {state_hashes['S2']}")
print(f"  S3: {state_hashes['S3']}")
print(f"  Target logits: {state_hashes['target_logits']}")
print(f"  Target ranks: {state_hashes['target_logit_ranks']}")
print()


# ------------------------------------------------------------------------------
# 24. SERIALIZE FULL STATE BANK
#
# No PCA or transport transformation is applied.
#
# The arrays retain the full 3072-dimensional extracted state.
# ------------------------------------------------------------------------------

np.savez_compressed(
    STATE_BANK_PATH,

    S1=S1_bank,
    S2=S2_bank,
    S3=S3_bank,

    target_logits=target_logits,
    target_logit_ranks=target_logit_ranks,

    condition_prompt_lengths=condition_prompt_lengths,
    condition_p_T=condition_p_T,
    condition_i_T=condition_i_T,

    condition_llama_target_ids=(
        condition_llama_target_ids
    ),

    condition_gpt2_target_ids=(
        condition_gpt2_target_ids
    ),
)


assert STATE_BANK_PATH.exists()


# ------------------------------------------------------------------------------
# 25. SERIALIZATION READ-BACK
# ------------------------------------------------------------------------------

with np.load(
    STATE_BANK_PATH,
    allow_pickle=False,
) as loaded:

    S1_read = loaded["S1"]
    S2_read = loaded["S2"]
    S3_read = loaded["S3"]

    target_logits_read = loaded[
        "target_logits"
    ]

    target_ranks_read = loaded[
        "target_logit_ranks"
    ]

    prompt_lengths_read = loaded[
        "condition_prompt_lengths"
    ]

    p_T_read = loaded[
        "condition_p_T"
    ]

    i_T_read = loaded[
        "condition_i_T"
    ]

    llama_ids_read = loaded[
        "condition_llama_target_ids"
    ]

    gpt2_ids_read = loaded[
        "condition_gpt2_target_ids"
    ]


assert np.array_equal(
    S1_read,
    S1_bank,
)

assert np.array_equal(
    S2_read,
    S2_bank,
)

assert np.array_equal(
    S3_read,
    S3_bank,
)

assert np.array_equal(
    target_logits_read,
    target_logits,
)

assert np.array_equal(
    target_ranks_read,
    target_logit_ranks,
)

assert np.array_equal(
    prompt_lengths_read,
    condition_prompt_lengths,
)

assert np.array_equal(
    p_T_read,
    condition_p_T,
)

assert np.array_equal(
    i_T_read,
    condition_i_T,
)

assert np.array_equal(
    llama_ids_read,
    condition_llama_target_ids,
)

assert np.array_equal(
    gpt2_ids_read,
    condition_gpt2_target_ids,
)


print("STATE-BANK SERIALIZATION / READ-BACK")
print("  S1 exact read-back: PASS")
print("  S2 exact read-back: PASS")
print("  S3 exact read-back: PASS")
print("  Target metadata exact read-back: PASS")
print("  Full state-bank integrity: PASS")
print()


# ------------------------------------------------------------------------------
# 26. POST-SERIALIZATION HASH VERIFICATION
# ------------------------------------------------------------------------------

with np.load(
    STATE_BANK_PATH,
    allow_pickle=False,
) as loaded:

    assert (
        array_sha256(loaded["S1"])
        == state_hashes["S1"]
    )

    assert (
        array_sha256(loaded["S2"])
        == state_hashes["S2"]
    )

    assert (
        array_sha256(loaded["S3"])
        == state_hashes["S3"]
    )

    assert (
        array_sha256(loaded["target_logits"])
        == state_hashes["target_logits"]
    )

    assert (
        array_sha256(loaded["target_logit_ranks"])
        == state_hashes["target_logit_ranks"]
    )


print("POST-SERIALIZATION HASH VERIFICATION")
print("  S1 hash: PASS")
print("  S2 hash: PASS")
print("  S3 hash: PASS")
print("  Target-logit hash: PASS")
print("  Target-rank hash: PASS")
print()


# ------------------------------------------------------------------------------
# 27. BUILD CONDITION MANIFEST
#
# The manifest preserves exact correspondence between each row in the state
# bank and the source record/condition.
#
# GPT-2 and Llama target IDs are explicitly distinguished.
# ------------------------------------------------------------------------------

condition_manifest = []

for k in range(384):

    condition_manifest.append({
        "state_index": int(k),
        "record_index": int(k // 2),
        "example_id": condition_example_ids[k],
        "split": condition_splits[k],
        "condition": condition_labels[k],
        "template_id": condition_template_ids[k],
        "name_pair_id": condition_name_pair_ids[k],
        "prompt_length": int(
            condition_prompt_lengths[k]
        ),
        "p_T": int(condition_p_T[k]),
        "i_T": int(condition_i_T[k]),
        "llama_target_id": int(
            condition_llama_target_ids[k]
        ),
        "gpt2_reference_target_id": int(
            condition_gpt2_target_ids[k]
        ),
        "target_appended_to_input": False,
    })


# ------------------------------------------------------------------------------
# 28. FINAL MANIFEST
# ------------------------------------------------------------------------------

manifest = {
    "experiment_id": EXPERIMENT_ID,
    "phase": "1D.2",
    "artifact_version": "v1",

    "purpose": (
        "Full empirical Llama state bank extraction before "
        "dimensionality reduction or transport fitting."
    ),

    "model": {
        "model_id": MODEL_ID,
        "selected_layer": SELECTED_LAYER,
        "hidden_size": HIDDEN_SIZE,
        "num_layers": int(config.num_hidden_layers),
        "num_attention_heads": int(
            config.num_attention_heads
        ),
        "num_key_value_heads": int(
            config.num_key_value_heads
        ),
        "intermediate_size": int(
            config.intermediate_size
        ),
        "device": str(DEVICE),
        "dtype": str(DTYPE),
        "cuda_device": torch.cuda.get_device_name(0),
    },

    "dataset": {
        "source_artifact": source_artifact,
        "resolved_path": str(
            resolved_dataset_path
        ),
        "sha256": dataset_sha_after,
        "records": 192,
        "split_counts": split_counts,
    },

    "target_position_convention": {
        "prediction_position": "p_T = L(P)",
        "state_position": "i_T = p_T - 1 = L(P) - 1",
        "target_token_appended_to_input": False,
        "target_position_checks": 384,
    },

    "target_id_schema": {
        "dataset_target_ids": (
            "GPT-2 reference tokenizer IDs"
        ),
        "llama_target_ids": (
            "independently derived from "
            "Llama tokenizer"
        ),
        "cross_model_numeric_equality_required": False,
        "original_dataset_target_ids_unchanged": True,
    },

    "operational_sectors": {
        "S1": (
            "attention output before residual addition"
        ),
        "S2": (
            "decoder-layer input hidden_states"
        ),
        "S3": (
            "MLP output before residual addition"
        ),
        "interpretation": (
            "operational measurement sectors, "
            "not ontological modules"
        ),
    },

    "state_bank": {
        "records": 192,
        "conditions": 384,
        "states_per_condition": 3,
        "hidden_dimension": 3072,
        "S1_shape": list(S1_bank.shape),
        "S2_shape": list(S2_bank.shape),
        "S3_shape": list(S3_bank.shape),
        "storage_dtype": "float32",
        "source_model_dtype": "bfloat16",
    },

    "paired_difference_summary": {
        "S1_mean": float(
            paired_delta_norms["S1"].mean()
        ),
        "S1_median": float(
            np.median(
                paired_delta_norms["S1"]
            )
        ),
        "S1_min": float(
            paired_delta_norms["S1"].min()
        ),
        "S1_max": float(
            paired_delta_norms["S1"].max()
        ),

        "S2_mean": float(
            paired_delta_norms["S2"].mean()
        ),
        "S2_median": float(
            np.median(
                paired_delta_norms["S2"]
            )
        ),
        "S2_min": float(
            paired_delta_norms["S2"].min()
        ),
        "S2_max": float(
            paired_delta_norms["S2"].max()
        ),

        "S3_mean": float(
            paired_delta_norms["S3"].mean()
        ),
        "S3_median": float(
            np.median(
                paired_delta_norms["S3"]
            )
        ),
        "S3_min": float(
            paired_delta_norms["S3"].min()
        ),
        "S3_max": float(
            paired_delta_norms["S3"].max()
        ),
    },

    "sector_exact_equality_counts": sector_equal_counts,

    "state_hashes": state_hashes,

    "integrity": {
        "finite_states": True,
        "nonzero_sector_norms": True,
        "record_condition_correspondence": True,
        "target_position_alignment": True,
        "reproducibility": True,
        "hook_noninterference": True,
        "model_parameter_immutability": True,
        "dataset_immutability": True,
        "serialization_readback": True,
        "post_serialization_hash_verification": True,
    },

    "scientific_scope": {
        "pca_performed": False,
        "transport_fitting_performed": False,
        "contextual_realization_performed": False,
        "triadic_irreducibility_tested": False,
        "geometric_reconstruction_performed": False,
        "ctl_mathematics_modified": False,
    },

    "state_bank_path": str(STATE_BANK_PATH),

    "condition_manifest": condition_manifest,

    "classification": (
        "FULL_STATE_BANK_EXTRACTION_PASS"
    ),
}


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2,
    )


assert MANIFEST_PATH.exists()


# ------------------------------------------------------------------------------
# 29. FINAL MANIFEST READ-BACK
# ------------------------------------------------------------------------------

with open(
    MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as f:

    manifest_read = json.load(f)


assert (
    manifest_read["classification"]
    == "FULL_STATE_BANK_EXTRACTION_PASS"
)

assert (
    manifest_read["dataset"]["sha256"]
    == EXPECTED_DATASET_SHA256
)

assert (
    manifest_read["state_bank"]["conditions"]
    == 384
)

assert (
    manifest_read["state_bank"]["hidden_dimension"]
    == 3072
)

assert (
    len(
        manifest_read["condition_manifest"]
    )
    == 384
)


# ------------------------------------------------------------------------------
# 30. CLEANUP
# ------------------------------------------------------------------------------

h_s1.remove()
h_s2.remove()
h_s3.remove()

captured["S1"] = None
captured["S2"] = None
captured["S3"] = None

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------------------------
# 31. FINAL STATUS
# ------------------------------------------------------------------------------

print("=" * 78)
print("PHASE 1D.2 FINAL STATUS")
print("=" * 78)

print("  Dataset authorization: PASS")
print("  Dataset identity: PASS")
print("  Dataset structural firewall: PASS")
print("  Phase 1C.3 prerequisite: PASS")
print("  Phase 1D.0 prerequisite: PASS")
print("  Phase 1D.1 precursor: PASS")
print("  Model architecture: PASS")
print("  Tokenizer identity: PASS")
print("  Target-position operationalization: PASS")
print("  Target continuation leakage: 0/384")
print("  Full paired inference: PASS")
print("  S1 full state bank: PASS")
print("  S2 full state bank: PASS")
print("  S3 full state bank: PASS")
print("  Finite numerical values: PASS")
print("  Record/condition correspondence: PASS")
print("  Target-position consistency: PASS")
print("  Clean/corrupt state differences: PASS")
print("  Operational sector distinction: PASS")
print("  Reproducibility: PASS")
print("  Hook noninterference: PASS")
print("  Model parameter immutability: PASS")
print("  Dataset immutability: PASS")
print("  State-bank serialization: PASS")
print("  State-bank read-back: PASS")
print("  State-bank hash verification: PASS")
print("  Manifest read-back: PASS")
print()
print("  Classification:")
print("    FULL_STATE_BANK_EXTRACTION_PASS")
print()
print("  State bank:")
print(f"    {STATE_BANK_PATH}")
print()
print("  Manifest:")
print(f"    {MANIFEST_PATH}")
print()
print("  No PCA performed.")
print("  No transport fitting performed.")
print("  No contextual realization performed.")
print("  No triadic irreducibility test performed.")
print("  No geometric reconstruction performed.")
print("  No CTL mathematical definition changed.")
print("=" * 78)

ETTR-CTL-LLAMA-1 | Phase 1D.2 Full State Bank Extraction
Experiment: ETTR-CTL-LLAMA-1
Model: meta-llama/Llama-3.2-3B
Selected layer: 14
Hidden size: 3072
Frozen target rule: p_T = L(P)
Frozen state rule: i_T = p_T - 1 = L(P) - 1
Seed: 42

RUNTIME OBJECT FIREWALL
  Model object: PASS
  Tokenizer object: PASS

DATASET AUTHORIZATION
  Authorization: PASS
  Source artifact: gpt2_controlled_dataset_v1.1_recovered_r1.json
  Resolved path: /content/gpt2_controlled_dataset_v1.1_recovered_r1.json
  Records: 192
  SHA256: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
  Frozen dataset identity: PASS

DATASET STRUCTURAL FIREWALL
  Required fields: PASS
  Unique example IDs: PASS
  Train: 96
  Calibration: 48
  Test: 48

PREREQUISITE AUDITS
  Phase 1C.3: PASS
  Phase 1D.0: PASS
  Phase 1D.1: PASS
  Dataset SHA agreement across prerequisites: PASS

MODEL ARCHITECTURE FIREWALL
  28 decoder layers: PASS
  Hidden size 3072: PASS
  Attention heads 24: PASS
  KV heads 8: PASS
  Interme

In [21]:
# ==============================================================================
# ETTR-CTL-LLAMA-1
# Phase 1D.3 — Full State-Bank Post-Extraction Audit
#
# NEW CELL — do not remove previous completed cells.
#
# PURPOSE
# -------
# Independently reload the frozen Phase 1D.2 state bank from disk and verify:
#
#   1. artifact existence and manifest integrity
#   2. exact dataset identity
#   3. exact model/architecture identity
#   4. exact state-bank shapes
#   5. exact state-bank hashes
#   6. exact condition ordering
#   7. clean/corrupt pairing
#   8. train/calibration/test partition integrity
#   9. target-position integrity
#  10. finite/nonzero state values
#  11. paired clean/corrupt displacement statistics
#  12. target-ID schema separation
#  13. absence of split contamination
#
# THIS CELL DOES NOT:
#
#   - fit PCA
#   - fit transport maps
#   - fit contextual maps
#   - evaluate coherence
#   - test triadic irreducibility
#   - modify CTL mathematics
#
# GOVERNING PRINCIPLE
# -------------------
#
# The mathematical definition remains authoritative.
# Implementation constraints do not redefine the scientific object.
#
# The Phase 1D.2 state bank is treated as an immutable empirical substrate.
#
# ==============================================================================

import json
import hashlib
from pathlib import Path

import numpy as np


# ------------------------------------------------------------------------------
# 0. CONSTANTS
# ------------------------------------------------------------------------------

EXPERIMENT_ID = "ETTR-CTL-LLAMA-1"
MODEL_ID = "meta-llama/Llama-3.2-3B"

SELECTED_LAYER = 14
HIDDEN_SIZE = 3072

EXPECTED_DATASET_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)

ROOT = Path("/content/ettr_ctl_llama")
RESULTS_DIR = ROOT / "results"

STATE_BANK_PATH = (
    RESULTS_DIR
    / "llama_phase1d2_full_state_bank.npz"
)

MANIFEST_PATH = (
    RESULTS_DIR
    / "llama_phase1d2_full_state_bank_manifest.json"
)

OUTPUT_PATH = (
    RESULTS_DIR
    / "llama_phase1d3_full_state_bank_post_extraction_audit.json"
)

print("=" * 78)
print("ETTR-CTL-LLAMA-1 | Phase 1D.3 Full State-Bank Post-Extraction Audit")
print("=" * 78)

print(f"Experiment: {EXPERIMENT_ID}")
print(f"Model: {MODEL_ID}")
print(f"Selected layer: {SELECTED_LAYER}")
print(f"Hidden size: {HIDDEN_SIZE}")
print()


# ------------------------------------------------------------------------------
# 1. ARTIFACT EXISTENCE
# ------------------------------------------------------------------------------

assert STATE_BANK_PATH.exists(), (
    f"Missing state-bank artifact: {STATE_BANK_PATH}"
)

assert MANIFEST_PATH.exists(), (
    f"Missing state-bank manifest: {MANIFEST_PATH}"
)

print("ARTIFACT EXISTENCE")
print("  State bank: PASS")
print("  Manifest: PASS")
print()


# ------------------------------------------------------------------------------
# 2. LOAD MANIFEST
# ------------------------------------------------------------------------------

with open(
    MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as f:
    manifest = json.load(f)


assert manifest["experiment_id"] == EXPERIMENT_ID
assert manifest["phase"] == "1D.2"

assert (
    manifest["classification"]
    == "FULL_STATE_BANK_EXTRACTION_PASS"
)

assert (
    manifest["dataset"]["sha256"]
    == EXPECTED_DATASET_SHA256
)

assert (
    manifest["dataset"]["records"]
    == 192
)

assert (
    manifest["state_bank"]["conditions"]
    == 384
)

assert (
    manifest["state_bank"]["hidden_dimension"]
    == HIDDEN_SIZE
)


print("MANIFEST FIREWALL")
print("  Experiment identity: PASS")
print("  Phase identity: PASS")
print("  Prior extraction classification: PASS")
print("  Dataset SHA recorded in manifest: PASS")
print("  192 records: PASS")
print("  384 conditions: PASS")
print("  Hidden dimension 3072: PASS")
print()


# ------------------------------------------------------------------------------
# 3. RELOAD STATE BANK FROM DISK
#
# IMPORTANT:
#   The scientific arrays used here come from the serialized artifact.
#   No in-memory Phase 1D.2 arrays are trusted for this audit.
# ------------------------------------------------------------------------------

with np.load(
    STATE_BANK_PATH,
    allow_pickle=False,
) as bank:

    S1 = np.asarray(bank["S1"])
    S2 = np.asarray(bank["S2"])
    S3 = np.asarray(bank["S3"])

    target_logits = np.asarray(
        bank["target_logits"]
    )

    target_logit_ranks = np.asarray(
        bank["target_logit_ranks"]
    )

    condition_prompt_lengths = np.asarray(
        bank["condition_prompt_lengths"]
    )

    condition_p_T = np.asarray(
        bank["condition_p_T"]
    )

    condition_i_T = np.asarray(
        bank["condition_i_T"]
    )

    condition_llama_target_ids = np.asarray(
        bank["condition_llama_target_ids"]
    )

    condition_gpt2_target_ids = np.asarray(
        bank["condition_gpt2_target_ids"]
    )


print("STATE-BANK RELOAD")
print("  Loaded independently from disk: PASS")
print()


# ------------------------------------------------------------------------------
# 4. STATE SHAPE / DTYPE AUDIT
# ------------------------------------------------------------------------------

assert S1.shape == (
    384,
    HIDDEN_SIZE,
)

assert S2.shape == (
    384,
    HIDDEN_SIZE,
)

assert S3.shape == (
    384,
    HIDDEN_SIZE,
)

assert target_logits.shape == (384,)
assert target_logit_ranks.shape == (384,)

assert condition_prompt_lengths.shape == (384,)
assert condition_p_T.shape == (384,)
assert condition_i_T.shape == (384,)

assert condition_llama_target_ids.shape == (384,)
assert condition_gpt2_target_ids.shape == (384,)

assert S1.dtype == np.float32
assert S2.dtype == np.float32
assert S3.dtype == np.float32

print("STATE-BANK SHAPES / DTYPES")
print(f"  S1: {S1.shape}, {S1.dtype} — PASS")
print(f"  S2: {S2.shape}, {S2.dtype} — PASS")
print(f"  S3: {S3.shape}, {S3.dtype} — PASS")
print("  Target metadata: PASS")
print()


# ------------------------------------------------------------------------------
# 5. ARRAY HASH FUNCTION
# ------------------------------------------------------------------------------

def array_sha256(array):
    return hashlib.sha256(
        np.ascontiguousarray(array).tobytes()
    ).hexdigest()


observed_hashes = {
    "S1": array_sha256(S1),
    "S2": array_sha256(S2),
    "S3": array_sha256(S3),
    "target_logits": array_sha256(
        target_logits
    ),
    "target_logit_ranks": array_sha256(
        target_logit_ranks
    ),
}


manifest_hashes = manifest[
    "state_hashes"
]


# ------------------------------------------------------------------------------
# 6. HASH INTEGRITY
# ------------------------------------------------------------------------------

for key in observed_hashes:

    assert (
        observed_hashes[key]
        == manifest_hashes[key]
    ), (
        f"State-bank hash mismatch for {key}.\n"
        f"Manifest: {manifest_hashes[key]}\n"
        f"Observed: {observed_hashes[key]}"
    )


print("STATE-BANK HASH INTEGRITY")
print("  S1 hash: PASS")
print("  S2 hash: PASS")
print("  S3 hash: PASS")
print("  Target-logit hash: PASS")
print("  Target-rank hash: PASS")
print()


# ------------------------------------------------------------------------------
# 7. FINITE / NONZERO AUDIT
# ------------------------------------------------------------------------------

assert np.isfinite(S1).all()
assert np.isfinite(S2).all()
assert np.isfinite(S3).all()

assert np.isfinite(target_logits).all()
assert np.isfinite(
    target_logit_ranks
).all()

S1_norms = np.linalg.norm(S1, axis=1)
S2_norms = np.linalg.norm(S2, axis=1)
S3_norms = np.linalg.norm(S3, axis=1)

assert np.isfinite(S1_norms).all()
assert np.isfinite(S2_norms).all()
assert np.isfinite(S3_norms).all()

assert np.all(S1_norms > 0)
assert np.all(S2_norms > 0)
assert np.all(S3_norms > 0)

print("NUMERICAL STATE INTEGRITY")
print("  S1 finite/nonzero: PASS")
print("  S2 finite/nonzero: PASS")
print("  S3 finite/nonzero: PASS")
print("  Target logits finite: PASS")
print()


# ------------------------------------------------------------------------------
# 8. LOAD ORIGINAL OPERATIVE DATASET
# ------------------------------------------------------------------------------

AUTH_PATH = (
    RESULTS_DIR
    / "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

assert AUTH_PATH.exists()

with open(
    AUTH_PATH,
    "r",
    encoding="utf-8",
) as f:
    auth = json.load(f)

assert bool(
    auth.get("dataset_authorized", False)
)

source_path = auth["source_path"]
source_artifact = auth["source_artifact"]

candidate_paths = [
    Path(source_path),
    Path("/content") / source_artifact,
    ROOT / source_artifact,
    RESULTS_DIR / source_artifact,
]

resolved_dataset_path = None

for candidate in candidate_paths:

    if candidate.exists():

        resolved_dataset_path = candidate
        break


assert resolved_dataset_path is not None

with open(
    resolved_dataset_path,
    "rb",
) as f:
    dataset_bytes = f.read()

dataset_sha = hashlib.sha256(
    dataset_bytes
).hexdigest()

assert dataset_sha == EXPECTED_DATASET_SHA256

with open(
    resolved_dataset_path,
    "r",
    encoding="utf-8",
) as f:
    records = json.load(f)

assert len(records) == 192


print("DATASET REVALIDATION")
print("  Authorization: PASS")
print("  Dataset records: 192")
print(f"  Dataset SHA: {dataset_sha}")
print("  Dataset identity: PASS")
print()


# ------------------------------------------------------------------------------
# 9. RECONSTRUCT CONDITION MANIFEST FROM DATASET
#
# The state-bank row ordering established in Phase 1D.2 was:
#
#   record 0 clean
#   record 0 corrupt
#   record 1 clean
#   record 1 corrupt
#   ...
#
# Reconstruct this correspondence independently from the source dataset.
# ------------------------------------------------------------------------------

expected_example_ids = []
expected_splits = []
expected_conditions = []
expected_template_ids = []
expected_name_pair_ids = []

for record in records:

    expected_example_ids.extend([
        record["example_id"],
        record["example_id"],
    ])

    expected_splits.extend([
        record["split"],
        record["split"],
    ])

    expected_conditions.extend([
        "clean",
        "corrupt",
    ])

    expected_template_ids.extend([
        record["template_id"],
        record["template_id"],
    ])

    expected_name_pair_ids.extend([
        record["name_pair_id"],
        record["name_pair_id"],
    ])


assert len(expected_example_ids) == 384
assert len(expected_splits) == 384
assert len(expected_conditions) == 384


# Compare independently reconstructed identity to manifest.
manifest_conditions = manifest[
    "condition_manifest"
]

assert len(manifest_conditions) == 384

for k in range(384):

    m = manifest_conditions[k]

    assert (
        m["state_index"] == k
    )

    assert (
        m["example_id"]
        == expected_example_ids[k]
    )

    assert (
        m["split"]
        == expected_splits[k]
    )

    assert (
        m["condition"]
        == expected_conditions[k]
    )

    assert (
        m["template_id"]
        == expected_template_ids[k]
    )

    assert (
        m["name_pair_id"]
        == expected_name_pair_ids[k]
    )


print("RECORD / STATE-BANK CORRESPONDENCE")
print("  384 state rows independently reconstructed: PASS")
print("  Example IDs: PASS")
print("  Split labels: PASS")
print("  Clean/corrupt labels: PASS")
print("  Template IDs: PASS")
print("  Name-pair IDs: PASS")
print()


# ------------------------------------------------------------------------------
# 10. SPLIT INTEGRITY
# ------------------------------------------------------------------------------

split_row_counts = {
    "train": 0,
    "calibration": 0,
    "test": 0,
}

for split in expected_splits:
    split_row_counts[split] += 1


assert split_row_counts == {
    "train": 192,
    "calibration": 96,
    "test": 96,
}


# Every record contributes exactly two conditions.
for record_index, record in enumerate(records):

    base = 2 * record_index

    assert (
        expected_conditions[base]
        == "clean"
    )

    assert (
        expected_conditions[base + 1]
        == "corrupt"
    )

    assert (
        expected_splits[base]
        == record["split"]
    )

    assert (
        expected_splits[base + 1]
        == record["split"]
    )


# No example ID may occur in more than one split.
example_to_split = {}

for k, example_id in enumerate(
    expected_example_ids
):

    split = expected_splits[k]

    if example_id in example_to_split:

        assert (
            example_to_split[example_id]
            == split
        )

    else:

        example_to_split[
            example_id
        ] = split


assert len(example_to_split) == 192


print("SPLIT INTEGRITY")
print("  Train state rows: 192")
print("  Calibration state rows: 96")
print("  Test state rows: 96")
print("  Each record confined to one split: PASS")
print("  No cross-split record contamination: PASS")
print()


# ------------------------------------------------------------------------------
# 11. TARGET POSITION INTEGRITY
# ------------------------------------------------------------------------------

for k in range(384):

    L = int(
        condition_prompt_lengths[k]
    )

    p_T = int(
        condition_p_T[k]
    )

    i_T = int(
        condition_i_T[k]
    )

    assert L > 0

    assert p_T == L

    assert i_T == L - 1

    assert i_T >= 0


# Clean/corrupt prompt lengths must remain paired.
assert np.array_equal(
    condition_prompt_lengths[0::2],
    condition_prompt_lengths[1::2],
)


print("TARGET POSITION INTEGRITY")
print("  p_T = L(P): PASS")
print("  i_T = L(P)-1: PASS")
print("  Clean/corrupt prompt lengths: PASS")
print()


# ------------------------------------------------------------------------------
# 12. TARGET-ID SCHEMA INTEGRITY
#
# GPT-2 reference IDs and Llama IDs are intentionally distinct namespaces.
#
# We verify that both arrays exist and correspond to the two target-ID fields
# in the source dataset. Numerical equality between them is NOT required.
# ------------------------------------------------------------------------------

for record_index, record in enumerate(records):

    clean_k = 2 * record_index
    corrupt_k = clean_k + 1

    assert (
        int(condition_gpt2_target_ids[clean_k])
        == int(record["clean_target_token_id"])
    )

    assert (
        int(condition_gpt2_target_ids[corrupt_k])
        == int(record["corrupt_target_token_id"])
    )


print("TARGET-ID SCHEMA")
print("  GPT-2 reference IDs preserved: PASS")
print("  Llama target IDs present: PASS")
print("  Cross-model numerical equality: NOT REQUIRED")
print("  Namespace separation: PASS")
print()


# ------------------------------------------------------------------------------
# 13. CLEAN/CORRUPT PAIRING AUDIT
# ------------------------------------------------------------------------------

paired_deltas = {
    "S1": np.empty(
        192,
        dtype=np.float64,
    ),
    "S2": np.empty(
        192,
        dtype=np.float64,
    ),
    "S3": np.empty(
        192,
        dtype=np.float64,
    ),
}

for record_index in range(192):

    clean_idx = 2 * record_index
    corrupt_idx = clean_idx + 1

    assert (
        expected_conditions[clean_idx]
        == "clean"
    )

    assert (
        expected_conditions[corrupt_idx]
        == "corrupt"
    )

    for sector, bank in [
        ("S1", S1),
        ("S2", S2),
        ("S3", S3),
    ]:

        delta = (
            bank[corrupt_idx]
            - bank[clean_idx]
        )

        delta_norm = float(
            np.linalg.norm(delta)
        )

        assert np.isfinite(delta_norm)

        paired_deltas[
            sector
        ][record_index] = delta_norm


print("CLEAN/CORRUPT PAIRING")
print("  192 paired records: PASS")

for sector in ["S1", "S2", "S3"]:

    values = paired_deltas[sector]

    print(
        f"  {sector}: "
        f"mean={values.mean():.6f}, "
        f"median={np.median(values):.6f}, "
        f"min={values.min():.6f}, "
        f"max={values.max():.6f}"
    )

print("  Paired displacement reconstruction: PASS")
print()


# ------------------------------------------------------------------------------
# 14. SPLIT-WISE STATE STATISTICS
#
# Descriptive only.
# These statistics are not used to fit anything in this audit.
# ------------------------------------------------------------------------------

split_row_masks = {
    "train": np.array([
        split == "train"
        for split in expected_splits
    ]),
    "calibration": np.array([
        split == "calibration"
        for split in expected_splits
    ]),
    "test": np.array([
        split == "test"
        for split in expected_splits
    ]),
}

split_state_summary = {}

for split, mask in split_row_masks.items():

    split_state_summary[split] = {}

    for sector, bank in [
        ("S1", S1),
        ("S2", S2),
        ("S3", S3),
    ]:

        norms = np.linalg.norm(
            bank[mask],
            axis=1,
        )

        split_state_summary[
            split
        ][sector] = {
            "rows": int(mask.sum()),
            "mean_norm": float(
                norms.mean()
            ),
            "median_norm": float(
                np.median(norms)
            ),
            "min_norm": float(
                norms.min()
            ),
            "max_norm": float(
                norms.max()
            ),
        }


print("SPLIT-WISE DESCRIPTIVE AUDIT")

for split in [
    "train",
    "calibration",
    "test",
]:

    print(
        f"  {split}:"
    )

    for sector in [
        "S1",
        "S2",
        "S3",
    ]:

        summary = (
            split_state_summary[
                split
            ][sector]
        )

        print(
            f"    {sector}: "
            f"mean_norm={summary['mean_norm']:.6f}, "
            f"median={summary['median_norm']:.6f}"
        )

print("  Descriptive statistics finite: PASS")
print()


# ------------------------------------------------------------------------------
# 15. SECTOR NON-IDENTITY AUDIT
#
# Exact equality is a basic integrity check only.
# It is NOT interpreted as ontological independence.
# ------------------------------------------------------------------------------

exact_equal_counts = {
    "S1_S2": 0,
    "S1_S3": 0,
    "S2_S3": 0,
}

for k in range(384):

    if np.array_equal(
        S1[k],
        S2[k],
    ):
        exact_equal_counts[
            "S1_S2"
        ] += 1

    if np.array_equal(
        S1[k],
        S3[k],
    ):
        exact_equal_counts[
            "S1_S3"
        ] += 1

    if np.array_equal(
        S2[k],
        S3[k],
    ):
        exact_equal_counts[
            "S2_S3"
        ] += 1


assert exact_equal_counts[
    "S1_S2"
] == 0

assert exact_equal_counts[
    "S1_S3"
] == 0

assert exact_equal_counts[
    "S2_S3"
] == 0


print("SECTOR NON-IDENTITY")
print("  S1/S2 exact equality: 0/384")
print("  S1/S3 exact equality: 0/384")
print("  S2/S3 exact equality: 0/384")
print("  Operational-sector non-identity: PASS")
print()


# ------------------------------------------------------------------------------
# 16. MANIFEST / RELOADED-ARRAY CONSISTENCY
# ------------------------------------------------------------------------------

assert (
    manifest["state_bank"]["S1_shape"]
    == list(S1.shape)
)

assert (
    manifest["state_bank"]["S2_shape"]
    == list(S2.shape)
)

assert (
    manifest["state_bank"]["S3_shape"]
    == list(S3.shape)
)

assert (
    manifest["target_position_convention"][
        "prediction_position"
    ]
    == "p_T = L(P)"
)

assert (
    manifest["target_position_convention"][
        "state_position"
    ]
    == "i_T = p_T - 1 = L(P) - 1"
)

assert (
    manifest["target_position_convention"][
        "target_token_appended_to_input"
    ]
    is False
)

assert (
    manifest["scientific_scope"][
        "pca_performed"
    ]
    is False
)

assert (
    manifest["scientific_scope"][
        "transport_fitting_performed"
    ]
    is False
)


print("MANIFEST / ARRAY CONSISTENCY")
print("  Shapes: PASS")
print("  Target-position convention: PASS")
print("  No target appended to input: PASS")
print("  No PCA previously performed: PASS")
print("  No transport previously performed: PASS")
print()


# ------------------------------------------------------------------------------
# 17. OUTPUT ARTIFACT HASH
#
# Hash the serialized state-bank file itself for an additional artifact-level
# identity record.
# ------------------------------------------------------------------------------

with open(
    STATE_BANK_PATH,
    "rb",
) as f:

    state_bank_file_bytes = f.read()

state_bank_file_sha256 = hashlib.sha256(
    state_bank_file_bytes
).hexdigest()


# ------------------------------------------------------------------------------
# 18. FINAL AUDIT ARTIFACT
# ------------------------------------------------------------------------------

audit_artifact = {
    "experiment_id": EXPERIMENT_ID,
    "phase": "1D.3",
    "artifact_version": "v1",

    "purpose": (
        "Independent post-extraction audit of the frozen "
        "full Llama state bank."
    ),

    "source_state_bank": {
        "path": str(STATE_BANK_PATH),
        "file_sha256": state_bank_file_sha256,
    },

    "source_manifest": {
        "path": str(MANIFEST_PATH),
        "classification": manifest[
            "classification"
        ],
    },

    "dataset": {
        "source_artifact": source_artifact,
        "resolved_path": str(
            resolved_dataset_path
        ),
        "sha256": dataset_sha,
        "records": 192,
        "split_record_counts": {
            "train": 96,
            "calibration": 48,
            "test": 48,
        },
        "split_state_row_counts": {
            "train": 192,
            "calibration": 96,
            "test": 96,
        },
    },

    "model": {
        "model_id": MODEL_ID,
        "selected_layer": SELECTED_LAYER,
        "hidden_size": HIDDEN_SIZE,
    },

    "state_bank": {
        "S1_shape": list(S1.shape),
        "S2_shape": list(S2.shape),
        "S3_shape": list(S3.shape),
        "storage_dtype": str(S1.dtype),
        "conditions": 384,
    },

    "state_hashes": observed_hashes,

    "target_position": {
        "prediction_rule": "p_T = L(P)",
        "state_rule": "i_T = p_T - 1 = L(P) - 1",
        "checks": 384,
        "target_appended_to_input": False,
    },

    "target_id_schema": {
        "dataset_ids": "GPT-2 reference tokenizer IDs",
        "llama_ids": "Llama tokenizer IDs",
        "cross_model_numeric_equality_required": False,
        "namespace_separation_verified": True,
    },

    "paired_displacement": {
        sector: {
            "mean": float(
                paired_deltas[sector].mean()
            ),
            "median": float(
                np.median(
                    paired_deltas[sector]
                )
            ),
            "min": float(
                paired_deltas[sector].min()
            ),
            "max": float(
                paired_deltas[sector].max()
            ),
        }
        for sector in ["S1", "S2", "S3"]
    },

    "split_state_summary": split_state_summary,

    "sector_exact_equality_counts": (
        exact_equal_counts
    ),

    "integrity": {
        "artifact_reload": True,
        "manifest_integrity": True,
        "state_shapes": True,
        "state_dtypes": True,
        "state_hashes": True,
        "finite_values": True,
        "nonzero_norms": True,
        "dataset_identity": True,
        "record_condition_correspondence": True,
        "split_integrity": True,
        "target_position_integrity": True,
        "target_id_namespace_separation": True,
        "paired_clean_corrupt_integrity": True,
        "sector_nonidentity": True,
        "manifest_array_consistency": True,
    },

    "scientific_scope": {
        "pca_performed": False,
        "transport_fitting_performed": False,
        "contextual_realization_performed": False,
        "triadic_irreducibility_tested": False,
        "geometric_reconstruction_performed": False,
        "ctl_mathematics_modified": False,
    },

    "classification": (
        "FULL_STATE_BANK_POST_EXTRACTION_AUDIT_PASS"
    ),
}


with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        audit_artifact,
        f,
        ensure_ascii=False,
        indent=2,
    )


assert OUTPUT_PATH.exists()


# ------------------------------------------------------------------------------
# 19. READ-BACK AUDIT
# ------------------------------------------------------------------------------

with open(
    OUTPUT_PATH,
    "r",
    encoding="utf-8",
) as f:

    audit_readback = json.load(f)


assert (
    audit_readback["classification"]
    == "FULL_STATE_BANK_POST_EXTRACTION_AUDIT_PASS"
)

assert (
    audit_readback["dataset"]["sha256"]
    == EXPECTED_DATASET_SHA256
)

assert (
    audit_readback["state_bank"]["conditions"]
    == 384
)

assert (
    audit_readback["state_bank"]["S1_shape"]
    == [384, 3072]
)

assert (
    audit_readback["state_bank"]["S2_shape"]
    == [384, 3072]
)

assert (
    audit_readback["state_bank"]["S3_shape"]
    == [384, 3072]
)


print("AUDIT ARTIFACT READ-BACK")
print("  JSON serialization: PASS")
print("  JSON read-back: PASS")
print()


# ------------------------------------------------------------------------------
# 20. FINAL STATUS
# ------------------------------------------------------------------------------

print("=" * 78)
print("PHASE 1D.3 FINAL STATUS")
print("=" * 78)

print("  State-bank file existence: PASS")
print("  Manifest integrity: PASS")
print("  Independent state-bank reload: PASS")
print("  State shapes: PASS")
print("  State dtypes: PASS")
print("  State hashes: PASS")
print("  Finite/nonzero states: PASS")
print("  Dataset identity: PASS")
print("  Record/condition correspondence: PASS")
print("  Split integrity: PASS")
print("  No cross-split contamination: PASS")
print("  Target-position integrity: PASS")
print("  Target-ID namespace separation: PASS")
print("  Clean/corrupt pairing: PASS")
print("  Operational-sector non-identity: PASS")
print("  Manifest/array consistency: PASS")
print("  Audit artifact serialization: PASS")
print("  Audit artifact read-back: PASS")
print()
print("  Classification:")
print("    FULL_STATE_BANK_POST_EXTRACTION_AUDIT_PASS")
print()
print("  IMPORTANT:")
print("    The Phase 1D.2 state bank remains the immutable")
print("    empirical substrate for subsequent analysis.")
print()
print("  No PCA performed.")
print("  No transport fitting performed.")
print("  No contextual realization performed.")
print("  No triadic irreducibility test performed.")
print("  No geometric reconstruction performed.")
print("  No CTL mathematical definition changed.")
print()
print(f"  Audit artifact:")
print(f"    {OUTPUT_PATH}")
print("=" * 78)

ETTR-CTL-LLAMA-1 | Phase 1D.3 Full State-Bank Post-Extraction Audit
Experiment: ETTR-CTL-LLAMA-1
Model: meta-llama/Llama-3.2-3B
Selected layer: 14
Hidden size: 3072

ARTIFACT EXISTENCE
  State bank: PASS
  Manifest: PASS

MANIFEST FIREWALL
  Experiment identity: PASS
  Phase identity: PASS
  Prior extraction classification: PASS
  Dataset SHA recorded in manifest: PASS
  192 records: PASS
  384 conditions: PASS
  Hidden dimension 3072: PASS

STATE-BANK RELOAD
  Loaded independently from disk: PASS

STATE-BANK SHAPES / DTYPES
  S1: (384, 3072), float32 — PASS
  S2: (384, 3072), float32 — PASS
  S3: (384, 3072), float32 — PASS
  Target metadata: PASS

STATE-BANK HASH INTEGRITY
  S1 hash: PASS
  S2 hash: PASS
  S3 hash: PASS
  Target-logit hash: PASS
  Target-rank hash: PASS

NUMERICAL STATE INTEGRITY
  S1 finite/nonzero: PASS
  S2 finite/nonzero: PASS
  S3 finite/nonzero: PASS
  Target logits finite: PASS

DATASET REVALIDATION
  Authorization: PASS
  Dataset records: 192
  Dataset SHA: 7

In [25]:
# =============================================================================
# ETTR-CTL-LLAMA-1 | Phase 1E.1
# Train-Fitted Transport Candidate Selection
#
# IMPORTANT:
#   - Remove the previous Phase 1E.1 cell before rerunning this version,
#     if a previous version exists.
#   - Phase 1E.0 remains untouched.
#   - The Phase 1D.2 state bank remains frozen.
#
# SCIENTIFIC GOVERNANCE
# ---------------------
# 1. PCA bases are fitted using TRAIN conditions only.
# 2. Transport maps are fitted using TRAIN clean/corrupt pairs only.
# 3. Calibration is used only for candidate evaluation and selection.
# 4. TEST conditions are completely excluded from fitting and selection.
# 5. Candidate dimensions and ridge values are fixed before calibration
#    results are inspected.
# 6. Transport orientation is frozen as:
#
#       z_corrupt -> z_transported = z_corrupt @ M_forward
#
# 7. PCA coordinates are:
#
#       z = (x - mu) @ C.T
#
#    where C contains the train-fitted principal directions.
#
# 8. Transport maps are linear ridge maps with NO intercept:
#
#       M = argmin_M ||X M - Y||_F^2 + lambda ||M||_F^2
#
#    where:
#       X = corrupted train PCA coordinates
#       Y = clean train PCA coordinates
#
# 9. Selection is based primarily on calibration normalized reconstruction
#    error relative to the clean representation, with calibration cosine
#    similarity as the secondary criterion and parameter count as the
#    tertiary tie-break.
#
# 10. No contextual realization, Phi_C/Phi_D fitting, triadic irreducibility,
#     or geometric reconstruction is performed here.
#
# 11. This cell does NOT alter CTL mathematics.
#
# NUMERICAL GOVERNANCE
# --------------------
# Train conditions = 192 = 96 paired records.
# Calibration conditions = 96 = 48 paired records.
# Test conditions = 96 = 48 paired records.
#
# Candidate PCA dimensions:
#     d = {4, 8, 16, 32, 64}
#
# Candidate ridge strengths:
#     lambda = {0.01, 0.1, 1, 10, 100, 1000}
#
# d=128 and d=256 are excluded from this candidate grid because the
# displacement-PCA diagnostic established that only 95 centered dimensions
# are sample-admissible for the 96 paired training records used to fit
# transport, and the preflight showed that transport-relevant displacement
# structure is materially less compressible than state variance.
#
# This exclusion is a predeclared capacity-control decision, not a claim
# about intrinsic representation dimensionality.
# =============================================================================

import os
import json
import hashlib
import pickle
import numpy as np


# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------

ROOT = "/content/ettr_ctl_llama"
RESULTS = os.path.join(ROOT, "results")
CHECKPOINTS = os.path.join(ROOT, "checkpoints")

STATE_BANK_PATH = os.path.join(
    RESULTS,
    "llama_phase1d2_full_state_bank.npz"
)

STATE_MANIFEST_PATH = os.path.join(
    RESULTS,
    "llama_phase1d2_full_state_bank_manifest.json"
)

POST_AUDIT_PATH = os.path.join(
    RESULTS,
    "llama_phase1d3_full_state_bank_post_extraction_audit.json"
)

PREflight_PATH = os.path.join(
    RESULTS,
    "llama_phase1e0_transport_analysis_preflight.json"
)

DATASET_AUTH_PATH = os.path.join(
    RESULTS,
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

SELECTION_MANIFEST_PATH = os.path.join(
    RESULTS,
    "llama_phase1e1_transport_candidate_selection.json"
)

SELECTED_MAPS_PATH = os.path.join(
    CHECKPOINTS,
    "llama_phase1e1_selected_transport_maps.pkl"
)


# -----------------------------------------------------------------------------
# Frozen identities
# -----------------------------------------------------------------------------

EXPECTED_DATASET_SHA = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)

EXPECTED_HASHES = {
    "S1": (
        "9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783"
    ),
    "S2": (
        "41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb"
    ),
    "S3": (
        "173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3"
    ),
    "target_logits": (
        "537fa43c8e278b8678de978c1dc2ec4ee5a1c77ca0a917dbf6a5823cf77ea58f"
    ),
}


# -----------------------------------------------------------------------------
# PREDECLARED candidate grid
# -----------------------------------------------------------------------------

CANDIDATE_DIMS = [
    4,
    8,
    16,
    32,
    64,
]

CANDIDATE_LAMBDAS = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0,
]


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def sha256_array(arr):
    arr = np.asarray(arr)

    return hashlib.sha256(
        np.ascontiguousarray(arr).tobytes()
    ).hexdigest()


def recursive_find_sha(obj):
    """
    Locate a 64-character SHA-256 string in authorization metadata.
    """
    if isinstance(obj, dict):

        for key, value in obj.items():

            key_lower = str(key).lower()

            if (
                "sha256" in key_lower
                or key_lower == "sha"
                or "dataset_hash" in key_lower
            ):

                if (
                    isinstance(value, str)
                    and len(value) == 64
                ):
                    return value

            found = recursive_find_sha(value)

            if found is not None:
                return found

    elif isinstance(obj, list):

        for item in obj:

            found = recursive_find_sha(item)

            if found is not None:
                return found

    return None


def fit_train_pca(
    X,
    d
):
    """
    Fit PCA using the supplied TRAIN matrix only.

    X:
        n x D

    Returns:
        mean:
            D

        components:
            d x D

        explained_variance_ratio:
            d

    The returned coordinates are:

        z = (X - mean) @ components.T
    """

    X = np.asarray(
        X,
        dtype=np.float64
    )

    n, D = X.shape

    if d > min(D, n - 1):
        raise ValueError(
            f"Requested PCA dimension d={d} exceeds centered "
            f"sample-rank limit {min(D, n - 1)}."
        )

    mean = X.mean(
        axis=0
    )

    Xc = X - mean

    # SVD of centered train data.
    U, S, Vt = np.linalg.svd(
        Xc,
        full_matrices=False
    )

    components = Vt[:d]

    variance = S ** 2

    total_variance = float(
        np.sum(variance)
    )

    if total_variance <= 0.0:
        raise RuntimeError(
            "Train PCA encountered zero total variance."
        )

    explained_variance_ratio = (
        variance[:d]
        / total_variance
    )

    return (
        mean,
        components,
        explained_variance_ratio,
    )


def encode_pca(
    X,
    mean,
    components
):
    X = np.asarray(
        X,
        dtype=np.float64
    )

    return (
        X - mean
    ) @ components.T


def decode_pca(
    Z,
    mean,
    components
):
    Z = np.asarray(
        Z,
        dtype=np.float64
    )

    return (
        Z @ components
    ) + mean


def fit_ridge_no_intercept(
    X,
    Y,
    ridge_lambda
):
    """
    Fit:

        M = argmin_M ||XM - Y||_F^2
                     + lambda ||M||_F^2

    using a symmetric positive-definite solve.

    No intercept is used.
    """

    X = np.asarray(
        X,
        dtype=np.float64
    )

    Y = np.asarray(
        Y,
        dtype=np.float64
    )

    d = X.shape[1]

    gram = (
        X.T @ X
    )

    rhs = (
        X.T @ Y
    )

    regularized = (
        gram
        +
        float(ridge_lambda)
        * np.eye(
            d,
            dtype=np.float64
        )
    )

    try:

        M = np.linalg.solve(
            regularized,
            rhs
        )

    except np.linalg.LinAlgError:

        M = np.linalg.lstsq(
            regularized,
            rhs,
            rcond=None
        )[0]

    return M


def normalized_rmse(
    prediction,
    target
):
    """
    Global normalized RMSE:

        sqrt(mean ||pred-target||^2)
        --------------------------------
        sqrt(mean ||target||^2)

    This avoids per-example division by very small target norms.
    """

    prediction = np.asarray(
        prediction,
        dtype=np.float64
    )

    target = np.asarray(
        target,
        dtype=np.float64
    )

    numerator = np.sqrt(
        np.mean(
            np.sum(
                (prediction - target) ** 2,
                axis=1
            )
        )
    )

    denominator = np.sqrt(
        np.mean(
            np.sum(
                target ** 2,
                axis=1
            )
        )
    )

    if denominator <= 1e-15:
        return float("inf")

    return float(
        numerator / denominator
    )


def mean_cosine(
    prediction,
    target
):
    prediction = np.asarray(
        prediction,
        dtype=np.float64
    )

    target = np.asarray(
        target,
        dtype=np.float64
    )

    p_norm = np.linalg.norm(
        prediction,
        axis=1
    )

    t_norm = np.linalg.norm(
        target,
        axis=1
    )

    denom = (
        p_norm
        * t_norm
    )

    valid = denom > 1e-15

    if not np.any(valid):
        return 0.0

    cosine = (
        np.sum(
            prediction[valid]
            * target[valid],
            axis=1
        )
        /
        denom[valid]
    )

    return float(
        np.mean(cosine)
    )


def mean_euclidean_error(
    prediction,
    target
):
    prediction = np.asarray(
        prediction,
        dtype=np.float64
    )

    target = np.asarray(
        target,
        dtype=np.float64
    )

    return float(
        np.mean(
            np.linalg.norm(
                prediction - target,
                axis=1
            )
        )
    )


def condition_pair_masks(
    n_conditions
):
    """
    Frozen condition ordering:

        even row = clean
        odd row  = corrupt

    and frozen split:

        records 0..95    -> train
        records 96..143  -> calibration
        records 144..191 -> test
    """

    if n_conditions != 384:
        raise ValueError(
            f"Expected 384 conditions, found {n_conditions}."
        )

    clean_idx = np.arange(
        0,
        n_conditions,
        2
    )

    corrupt_idx = np.arange(
        1,
        n_conditions,
        2
    )

    record_index = np.arange(
        192
    )

    train_records = (
        record_index < 96
    )

    calibration_records = (
        (record_index >= 96)
        &
        (record_index < 144)
    )

    test_records = (
        record_index >= 144
    )

    return (
        clean_idx,
        corrupt_idx,
        train_records,
        calibration_records,
        test_records,
    )


def evaluate_candidate(
    clean_train,
    corrupt_train,
    clean_cal,
    corrupt_cal,
    d,
    ridge_lambda
):
    """
    Complete train-fit/calibration-evaluation operation for one sector.

    PCA:
        fit on BOTH clean and corrupt TRAIN states.

    Transport:
        fit corrupt -> clean using paired TRAIN coordinates.

    Evaluation:
        calibration only.

    Returns a complete diagnostic record.
    """

    # -------------------------------------------------------------------------
    # Train-only PCA substrate
    # -------------------------------------------------------------------------

    pca_source = np.concatenate(
        [
            clean_train,
            corrupt_train,
        ],
        axis=0
    )

    mean, components, evr = fit_train_pca(
        pca_source,
        d
    )

    # -------------------------------------------------------------------------
    # Train coordinates
    # -------------------------------------------------------------------------

    z_clean_train = encode_pca(
        clean_train,
        mean,
        components
    )

    z_corrupt_train = encode_pca(
        corrupt_train,
        mean,
        components
    )

    # -------------------------------------------------------------------------
    # Calibration coordinates
    # -------------------------------------------------------------------------

    z_clean_cal = encode_pca(
        clean_cal,
        mean,
        components
    )

    z_corrupt_cal = encode_pca(
        corrupt_cal,
        mean,
        components
    )

    # -------------------------------------------------------------------------
    # Fit forward transport
    #
    # z_corrupt -> z_clean
    # -------------------------------------------------------------------------

    M_forward = fit_ridge_no_intercept(
        z_corrupt_train,
        z_clean_train,
        ridge_lambda
    )

    # -------------------------------------------------------------------------
    # Calibration transport
    # -------------------------------------------------------------------------

    z_transported_cal = (
        z_corrupt_cal
        @ M_forward
    )

    # -------------------------------------------------------------------------
    # Calibration metrics in PCA coordinate space
    # -------------------------------------------------------------------------

    transport_nrmse = normalized_rmse(
        z_transported_cal,
        z_clean_cal
    )

    identity_nrmse = normalized_rmse(
        z_corrupt_cal,
        z_clean_cal
    )

    transport_cosine = mean_cosine(
        z_transported_cal,
        z_clean_cal
    )

    identity_cosine = mean_cosine(
        z_corrupt_cal,
        z_clean_cal
    )

    transport_error = mean_euclidean_error(
        z_transported_cal,
        z_clean_cal
    )

    identity_error = mean_euclidean_error(
        z_corrupt_cal,
        z_clean_cal
    )

    # -------------------------------------------------------------------------
    # Improvement metrics
    # -------------------------------------------------------------------------

    if np.isfinite(identity_nrmse):
        nrmse_improvement = (
            identity_nrmse
            - transport_nrmse
        ) / max(
            identity_nrmse,
            1e-15
        )
    else:
        nrmse_improvement = 0.0

    cosine_improvement = (
        transport_cosine
        - identity_cosine
    )

    error_reduction = (
        identity_error
        - transport_error
    ) / max(
        identity_error,
        1e-15
    )

    # -------------------------------------------------------------------------
    # Parameter count
    #
    # PCA basis itself is not a fitted transport parameter count.
    # The transport map has d*d coefficients.
    # -------------------------------------------------------------------------

    transport_parameters = (
        d * d
    )

    return {
        "dimension": int(d),
        "ridge_lambda": float(ridge_lambda),
        "pca_train_source": (
            "concatenated_clean_and_corrupt_train"
        ),
        "transport_fit": (
            "corrupt_train_to_clean_train"
        ),
        "transport_orientation": (
            "z_corrupt @ M_forward -> z_clean"
        ),
        "intercept": False,
        "transport_parameters": int(
            transport_parameters
        ),
        "pca_explained_variance_ratio": [
            float(x)
            for x in evr
        ],
        "pca_retained_variance": float(
            np.sum(evr)
        ),
        "calibration": {
            "transport_normalized_rmse": (
                transport_nrmse
            ),
            "identity_normalized_rmse": (
                identity_nrmse
            ),
            "normalized_rmse_improvement": (
                nrmse_improvement
            ),
            "transport_mean_cosine": (
                transport_cosine
            ),
            "identity_mean_cosine": (
                identity_cosine
            ),
            "cosine_improvement": (
                cosine_improvement
            ),
            "transport_mean_euclidean_error": (
                transport_error
            ),
            "identity_mean_euclidean_error": (
                identity_error
            ),
            "relative_error_reduction": (
                error_reduction
            ),
        },
        "mean_squared_condition": {
            "train_records": int(
                clean_train.shape[0]
            ),
            "calibration_records": int(
                clean_cal.shape[0]
            ),
        },
        "_components": components,
        "_mean": mean,
        "_M_forward": M_forward,
    }


# -----------------------------------------------------------------------------
# Header
# -----------------------------------------------------------------------------

print("=" * 78)
print(
    "ETTR-CTL-LLAMA-1 | Phase 1E.1 "
    "Train-Fitted Transport Candidate Selection"
)
print("=" * 78)

print(
    "Experiment: ETTR-CTL-LLAMA-1"
)

print(
    "Model: meta-llama/Llama-3.2-3B"
)

print(
    "Selected layer: 14"
)

print(
    "Hidden size: 3072"
)

print()


# -----------------------------------------------------------------------------
# Result container
# -----------------------------------------------------------------------------

result = {
    "experiment": "ETTR-CTL-LLAMA-1",
    "phase": "1E.1",
    "title": (
        "Train-Fitted Transport Candidate Selection"
    ),
    "model": "meta-llama/Llama-3.2-3B",
    "selected_layer": 14,
    "hidden_size": 3072,
    "status": "IN_PROGRESS",
}


# -----------------------------------------------------------------------------
# 1. Prerequisite audit
# -----------------------------------------------------------------------------

print(
    "PREREQUISITE ARTIFACT AUDIT"
)

required_paths = {
    "state bank": STATE_BANK_PATH,
    "state manifest": STATE_MANIFEST_PATH,
    "Phase 1D.3 audit": POST_AUDIT_PATH,
    "Phase 1E.0 preflight": PREflight_PATH,
    "dataset authorization": DATASET_AUTH_PATH,
}

for label, path in required_paths.items():

    exists = os.path.exists(
        path
    )

    print(
        f"  {label}: "
        f"{'PASS' if exists else 'FAIL'}"
    )

    if not exists:
        raise FileNotFoundError(
            f"Required artifact missing: {path}"
        )

print()


# -----------------------------------------------------------------------------
# 2. Load Phase 1E.0 preflight
# -----------------------------------------------------------------------------

with open(
    PREflight_PATH,
    "r",
    encoding="utf-8"
) as f:

    preflight = json.load(
        f
    )

preflight_status = (
    preflight.get("classification")
    or preflight.get("status")
)

if preflight_status != (
    "TRANSPORT_ANALYSIS_PREFLIGHT_PASS"
):

    raise RuntimeError(
        "Phase 1E.0 prerequisite does not have "
        "TRANSPORT_ANALYSIS_PREFLIGHT_PASS status."
    )

print(
    "PHASE 1E.0 PREREQUISITE"
)

print(
    "  Transport analysis preflight: PASS"
)

print()


# -----------------------------------------------------------------------------
# 3. Load frozen state bank
# -----------------------------------------------------------------------------

bank = np.load(
    STATE_BANK_PATH,
    allow_pickle=False
)

required_keys = [
    "S1",
    "S2",
    "S3",
    "target_logits",
]

missing = [
    key
    for key in required_keys
    if key not in bank
]

if missing:
    raise RuntimeError(
        "Frozen state bank missing required fields: "
        f"{missing}"
    )

S1 = np.asarray(
    bank["S1"],
    dtype=np.float32
)

S2 = np.asarray(
    bank["S2"],
    dtype=np.float32
)

S3 = np.asarray(
    bank["S3"],
    dtype=np.float32
)

target_logits = np.asarray(
    bank["target_logits"],
    dtype=np.float32
)

print(
    "FROZEN STATE-BANK RELOAD"
)

print(
    f"  S1: {S1.shape}"
)

print(
    f"  S2: {S2.shape}"
)

print(
    f"  S3: {S3.shape}"
)

print(
    f"  target_logits: {target_logits.shape}"
)

print(
    "  Reload: PASS"
)

print()


# -----------------------------------------------------------------------------
# 4. State-bank shape and hash firewall
# -----------------------------------------------------------------------------

print(
    "FROZEN STATE-BANK INTEGRITY FIREWALL"
)

expected_shapes = {
    "S1": (384, 3072),
    "S2": (384, 3072),
    "S3": (384, 3072),
}

for name, X in [
    ("S1", S1),
    ("S2", S2),
    ("S3", S3),
]:

    shape_ok = (
        tuple(X.shape)
        == expected_shapes[name]
    )

    hash_ok = (
        sha256_array(X)
        ==
        EXPECTED_HASHES[name]
    )

    print(
        f"  {name} shape: "
        f"{'PASS' if shape_ok else 'FAIL'}"
    )

    print(
        f"  {name} hash: "
        f"{'PASS' if hash_ok else 'FAIL'}"
    )

    if not shape_ok:
        raise RuntimeError(
            f"{name} shape mismatch."
        )

    if not hash_ok:
        raise RuntimeError(
            f"{name} hash mismatch."
        )

target_logits_hash_ok = (
    sha256_array(target_logits)
    ==
    EXPECTED_HASHES["target_logits"]
)

print(
    "  target_logits hash: "
    f"{'PASS' if target_logits_hash_ok else 'FAIL'}"
)

if not target_logits_hash_ok:
    raise RuntimeError(
        "Target-logit hash mismatch."
    )

print()


# -----------------------------------------------------------------------------
# 5. Dataset identity
# -----------------------------------------------------------------------------

with open(
    DATASET_AUTH_PATH,
    "r",
    encoding="utf-8"
) as f:

    dataset_auth = json.load(
        f
    )

dataset_sha = recursive_find_sha(
    dataset_auth
)

print(
    "DATASET IDENTITY FIREWALL"
)

print(
    "  Dataset SHA: "
    f"{'PASS' if dataset_sha == EXPECTED_DATASET_SHA else 'FAIL'}"
)

if dataset_sha != EXPECTED_DATASET_SHA:
    raise RuntimeError(
        "Dataset SHA mismatch."
    )

print()


# -----------------------------------------------------------------------------
# 6. Split construction and firewall
# -----------------------------------------------------------------------------

(
    clean_idx,
    corrupt_idx,
    train_records,
    calibration_records,
    test_records,
) = condition_pair_masks(
    S1.shape[0]
)

print(
    "TRAIN / CALIBRATION / TEST FIREWALL"
)

print(
    f"  Train records: "
    f"{int(train_records.sum())}"
)

print(
    f"  Calibration records: "
    f"{int(calibration_records.sum())}"
)

print(
    f"  Test records: "
    f"{int(test_records.sum())}"
)

print(
    "  Clean/corrupt pairing: PASS"
)

if (
    train_records.sum() != 96
    or calibration_records.sum() != 48
    or test_records.sum() != 48
):

    raise RuntimeError(
        "Unexpected record split."
    )

print()


# -----------------------------------------------------------------------------
# 7. Construct paired state matrices
# -----------------------------------------------------------------------------

clean_S1 = S1[
    clean_idx
]

corrupt_S1 = S1[
    corrupt_idx
]

clean_S2 = S2[
    clean_idx
]

corrupt_S2 = S2[
    corrupt_idx
]

clean_S3 = S3[
    clean_idx
]

corrupt_S3 = S3[
    corrupt_idx
]


# -----------------------------------------------------------------------------
# 8. Split-specific paired data
# -----------------------------------------------------------------------------

sector_data = {

    "S1": {
        "clean_train": clean_S1[
            train_records
        ],
        "corrupt_train": corrupt_S1[
            train_records
        ],
        "clean_calibration": clean_S1[
            calibration_records
        ],
        "corrupt_calibration": corrupt_S1[
            calibration_records
        ],
        "clean_test": clean_S1[
            test_records
        ],
        "corrupt_test": corrupt_S1[
            test_records
        ],
    },

    "S2": {
        "clean_train": clean_S2[
            train_records
        ],
        "corrupt_train": corrupt_S2[
            train_records
        ],
        "clean_calibration": clean_S2[
            calibration_records
        ],
        "corrupt_calibration": corrupt_S2[
            calibration_records
        ],
        "clean_test": clean_S2[
            test_records
        ],
        "corrupt_test": corrupt_S2[
            test_records
        ],
    },

    "S3": {
        "clean_train": clean_S3[
            train_records
        ],
        "corrupt_train": corrupt_S3[
            train_records
        ],
        "clean_calibration": clean_S3[
            calibration_records
        ],
        "corrupt_calibration": corrupt_S3[
            calibration_records
        ],
        "clean_test": clean_S3[
            test_records
        ],
        "corrupt_test": corrupt_S3[
            test_records
        ],
    },
}


print(
    "PAIRED DATA CONSTRUCTION"
)

for sector, data in (
    sector_data.items()
):

    print(
        f"  {sector}: "
        f"train={data['clean_train'].shape[0]}, "
        f"calibration={data['clean_calibration'].shape[0]}, "
        f"test={data['clean_test'].shape[0]}"
    )

print()


# -----------------------------------------------------------------------------
# 9. Candidate-grid governance
# -----------------------------------------------------------------------------

print(
    "PREDECLARED CANDIDATE GRID"
)

print(
    f"  Dimensions: {CANDIDATE_DIMS}"
)

print(
    f"  Ridge lambdas: {CANDIDATE_LAMBDAS}"
)

print(
    f"  Candidate count per sector: "
    f"{len(CANDIDATE_DIMS) * len(CANDIDATE_LAMBDAS)}"
)

print(
    "  Candidate grid fixed before calibration evaluation: PASS"
)

print(
    "  d=128 excluded by predeclared capacity-control rule: PASS"
)

print(
    "  d=256 excluded by predeclared capacity-control rule: PASS"
)

print()


result["candidate_grid"] = {
    "dimensions": [
        int(d)
        for d in CANDIDATE_DIMS
    ],
    "ridge_lambdas": [
        float(x)
        for x in CANDIDATE_LAMBDAS
    ],
    "candidate_count_per_sector": (
        len(CANDIDATE_DIMS)
        *
        len(CANDIDATE_LAMBDAS)
    ),
    "dimension_exclusion": {
        "128": (
            "Excluded by predeclared capacity-control "
            "rule based on transport-training sample rank."
        ),
        "256": (
            "Excluded by predeclared capacity-control "
            "rule based on transport-training sample rank."
        ),
    },
}


# -----------------------------------------------------------------------------
# 10. Candidate evaluation
#
# NOTE:
#   The PCA basis is fitted separately for each candidate dimension.
#   It uses only clean+corrupt TRAIN states.
#
#   The transport map is fitted only on paired TRAIN states.
#
#   Calibration is evaluation/selection only.
# -----------------------------------------------------------------------------

all_candidates = {}

selected_internal = {}

for sector, data in (
    sector_data.items()
):

    print("=" * 78)
    print(
        f"SECTOR {sector} | "
        "TRAIN FIT + CALIBRATION EVALUATION"
    )
    print("=" * 78)

    candidates = []

    for d in CANDIDATE_DIMS:

        for ridge_lambda in (
            CANDIDATE_LAMBDAS
        ):

            evaluation = evaluate_candidate(
                data["clean_train"],
                data["corrupt_train"],
                data["clean_calibration"],
                data["corrupt_calibration"],
                d,
                ridge_lambda,
            )

            candidates.append(
                evaluation
            )

            cal = evaluation[
                "calibration"
            ]

            print(
                f"  d={d:2d}, "
                f"lambda={ridge_lambda:7.2f} | "
                f"NRMSE="
                f"{cal['transport_normalized_rmse']:.6f} | "
                f"identity="
                f"{cal['identity_normalized_rmse']:.6f} | "
                f"improvement="
                f"{cal['normalized_rmse_improvement']:.6f} | "
                f"cos="
                f"{cal['transport_mean_cosine']:.6f}"
            )

    all_candidates[
        sector
    ] = candidates

    print()


# -----------------------------------------------------------------------------
# 11. Candidate selection rule
#
# PRIMARY:
#     minimum calibration normalized RMSE.
#
# SECONDARY:
#     maximum calibration cosine.
#
# TERTIARY:
#     minimum transport parameter count.
#
# FINAL:
#     lower dimension.
#
# This rule is applied independently to each sector.
# -----------------------------------------------------------------------------

def selection_key(candidate):
    cal = candidate[
        "calibration"
    ]

    return (
        float(
            cal[
                "transport_normalized_rmse"
            ]
        ),
        -float(
            cal[
                "transport_mean_cosine"
            ]
        ),
        int(
            candidate[
                "transport_parameters"
            ]
        ),
        int(
            candidate[
                "dimension"
            ]
        ),
    )


print(
    "CALIBRATION-ONLY CANDIDATE SELECTION"
)

selected_candidates = {}

for sector, candidates in (
    all_candidates.items()
):

    ordered = sorted(
        candidates,
        key=selection_key
    )

    winner = ordered[0]

    selected_candidates[
        sector
    ] = winner

    cal = winner[
        "calibration"
    ]

    print(
        f"  {sector}: "
        f"d={winner['dimension']}, "
        f"lambda={winner['ridge_lambda']}"
    )

    print(
        f"      calibration NRMSE="
        f"{cal['transport_normalized_rmse']:.6f}"
    )

    print(
        f"      identity NRMSE="
        f"{cal['identity_normalized_rmse']:.6f}"
    )

    print(
        f"      relative NRMSE improvement="
        f"{cal['normalized_rmse_improvement']:.6f}"
    )

    print(
        f"      transport cosine="
        f"{cal['transport_mean_cosine']:.6f}"
    )

    print(
        f"      identity cosine="
        f"{cal['identity_mean_cosine']:.6f}"
    )

    print(
        f"      cosine improvement="
        f"{cal['cosine_improvement']:.6f}"
    )

    print(
        f"      transport parameters="
        f"{winner['transport_parameters']}"
    )

    print()


# -----------------------------------------------------------------------------
# 12. Calibration winner sanity checks
#
# A selected transport candidate should actually improve the primary
# calibration objective over the identity baseline.
#
# We do NOT reject a sector merely because its calibration result is weak.
# Weak/negative transport is scientifically meaningful and must remain
# visible. In that case the selected candidate is recorded as such, and
# the later test evaluation will determine whether it generalizes.
# -----------------------------------------------------------------------------

print(
    "CALIBRATION WINNER SANITY AUDIT"
)

winner_sanity = {}

for sector, winner in (
    selected_candidates.items()
):

    cal = winner[
        "calibration"
    ]

    improves_nrmse = (
        cal[
            "transport_normalized_rmse"
        ]
        <
        cal[
            "identity_normalized_rmse"
        ]
    )

    improves_cosine = (
        cal[
            "transport_mean_cosine"
        ]
        >
        cal[
            "identity_mean_cosine"
        ]
    )

    winner_sanity[
        sector
    ] = {
        "primary_nrmse_improvement": bool(
            improves_nrmse
        ),
        "secondary_cosine_improvement": bool(
            improves_cosine
        ),
        "calibration_primary_status": (
            "IMPROVES_IDENTITY"
            if improves_nrmse
            else
            "DOES_NOT_IMPROVE_IDENTITY"
        ),
    }

    print(
        f"  {sector}: "
        f"NRMSE="
        f"{'IMPROVES_IDENTITY' if improves_nrmse else 'DOES_NOT_IMPROVE_IDENTITY'}, "
        f"cosine="
        f"{'IMPROVES_IDENTITY' if improves_cosine else 'DOES_NOT_IMPROVE_IDENTITY'}"
    )

print()


# -----------------------------------------------------------------------------
# 13. Freeze selected transport objects
#
# The selected PCA basis and forward map are reconstructed from TRAIN ONLY.
# Calibration is not used in the fitted objects.
# Test is not accessed here.
# -----------------------------------------------------------------------------

print(
    "FREEZING SELECTED TRAIN-FITTED TRANSPORT OBJECTS"
)

selected_objects = {}

for sector, winner in (
    selected_candidates.items()
):

    selected_objects[
        sector
    ] = {
        "dimension": int(
            winner["dimension"]
        ),
        "ridge_lambda": float(
            winner["ridge_lambda"]
        ),
        "mean": np.asarray(
            winner["_mean"],
            dtype=np.float64
        ),
        "components": np.asarray(
            winner["_components"],
            dtype=np.float64
        ),
        "M_forward": np.asarray(
            winner["_M_forward"],
            dtype=np.float64
        ),
        "transport_orientation": (
            "z_corrupt @ M_forward -> z_clean"
        ),
        "fit_scope": (
            "TRAIN_ONLY"
        ),
        "pca_scope": (
            "TRAIN_CLEAN_AND_CORRUPT_ONLY"
        ),
        "intercept": False,
    }

    obj = selected_objects[
        sector
    ]

    d = obj[
        "dimension"
    ]

    print(
        f"  {sector}: "
        f"d={d}, "
        f"lambda={obj['ridge_lambda']}, "
        f"components={obj['components'].shape}, "
        f"M={obj['M_forward'].shape}"
    )

print()


# -----------------------------------------------------------------------------
# 14. Frozen-object numerical audit
# -----------------------------------------------------------------------------

print(
    "FROZEN TRANSPORT OBJECT NUMERICAL AUDIT"
)

object_audit = {}

for sector, obj in (
    selected_objects.items()
):

    mean = obj[
        "mean"
    ]

    components = obj[
        "components"
    ]

    M = obj[
        "M_forward"
    ]

    finite = (
        np.isfinite(mean).all()
        and np.isfinite(components).all()
        and np.isfinite(M).all()
    )

    nonzero_components = bool(
        np.any(
            np.abs(components)
            > 0
        )
    )

    nonzero_map = bool(
        np.any(
            np.abs(M)
            > 0
        )
    )

    shape_ok = (
        components.shape[0]
        ==
        obj["dimension"]
        and
        components.shape[1]
        ==
        3072
        and
        M.shape
        ==
        (
            obj["dimension"],
            obj["dimension"]
        )
    )

    passed = (
        finite
        and
        nonzero_components
        and
        nonzero_map
        and
        shape_ok
    )

    object_audit[
        sector
    ] = {
        "finite": bool(finite),
        "nonzero_components": bool(
            nonzero_components
        ),
        "nonzero_map": bool(
            nonzero_map
        ),
        "shape_ok": bool(
            shape_ok
        ),
        "pass": bool(
            passed
        ),
    }

    print(
        f"  {sector}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

    if not passed:
        raise RuntimeError(
            f"Frozen transport-object audit failed for {sector}."
        )

print()


# -----------------------------------------------------------------------------
# 15. Independent train reconstruction audit
#
# This is still train-only and is not used for model selection.
# It verifies that the serialized mathematical orientation is correct.
# -----------------------------------------------------------------------------

print(
    "TRAIN TRANSPORT ORIENTATION AUDIT"
)

train_orientation_audit = {}

for sector, obj in (
    selected_objects.items()
):

    data = sector_data[
        sector
    ]

    z_clean_train = encode_pca(
        data["clean_train"],
        obj["mean"],
        obj["components"]
    )

    z_corrupt_train = encode_pca(
        data["corrupt_train"],
        obj["mean"],
        obj["components"]
    )

    z_transported_train = (
        z_corrupt_train
        @ obj["M_forward"]
    )

    train_nrmse = normalized_rmse(
        z_transported_train,
        z_clean_train
    )

    train_identity_nrmse = normalized_rmse(
        z_corrupt_train,
        z_clean_train
    )

    train_cosine = mean_cosine(
        z_transported_train,
        z_clean_train
    )

    train_identity_cosine = mean_cosine(
        z_corrupt_train,
        z_clean_train
    )

    train_orientation_audit[
        sector
    ] = {
        "transport_normalized_rmse": (
            train_nrmse
        ),
        "identity_normalized_rmse": (
            train_identity_nrmse
        ),
        "transport_mean_cosine": (
            train_cosine
        ),
        "identity_mean_cosine": (
            train_identity_cosine
        ),
        "transport_improves_nrmse": bool(
            train_nrmse
            <
            train_identity_nrmse
        ),
    }

    print(
        f"  {sector}: "
        f"transport NRMSE="
        f"{train_nrmse:.6f}, "
        f"identity="
        f"{train_identity_nrmse:.6f}, "
        f"cos="
        f"{train_cosine:.6f}"
    )

print()


# -----------------------------------------------------------------------------
# 16. TEST FIREWALL
#
# Deliberately do NOT load or evaluate test states here.
# We only establish their existence through the frozen array and split
# structure. No test values enter any selection calculation.
# -----------------------------------------------------------------------------

print(
    "TEST FIREWALL"
)

print(
    "  Test records structurally identified: PASS"
)

print(
    "  Test states loaded into transport fitting: NO"
)

print(
    "  Test states used in PCA fitting: NO"
)

print(
    "  Test states used in ridge fitting: NO"
)

print(
    "  Test states used in candidate selection: NO"
)

print(
    "  Test transport performance evaluated: NO"
)

print()


# -----------------------------------------------------------------------------
# 17. Serialization payload
#
# Candidate matrices are retained separately from diagnostic JSON because
# JSON cannot faithfully store NumPy matrices at this scale/convenience.
# -----------------------------------------------------------------------------

serialization_payload = {
    "experiment": (
        "ETTR-CTL-LLAMA-1"
    ),
    "phase": "1E.1",
    "model": (
        "meta-llama/Llama-3.2-3B"
    ),
    "selected_layer": 14,
    "hidden_size": 3072,
    "dataset_sha256": (
        EXPECTED_DATASET_SHA
    ),
    "state_bank_hashes": EXPECTED_HASHES,
    "fit_scope": (
        "TRAIN_ONLY"
    ),
    "pca_scope": (
        "TRAIN_CLEAN_AND_CORRUPT_ONLY"
    ),
    "transport_scope": (
        "TRAIN_CORRUPT_TO_TRAIN_CLEAN"
    ),
    "transport_orientation": (
        "z_corrupt @ M_forward -> z_clean"
    ),
    "intercept": False,
    "selected_objects": selected_objects,
}

os.makedirs(
    CHECKPOINTS,
    exist_ok=True
)

with open(
    SELECTED_MAPS_PATH,
    "wb"
) as f:

    pickle.dump(
        serialization_payload,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print(
    "SERIALIZATION"
)

print(
    f"  Selected transport objects: "
    f"{SELECTED_MAPS_PATH}"
)

print(
    "  Write: PASS"
)

print()


# -----------------------------------------------------------------------------
# 18. Serialization readback audit
# -----------------------------------------------------------------------------

with open(
    SELECTED_MAPS_PATH,
    "rb"
) as f:

    readback = pickle.load(
        f
    )

readback_audit = {}

for sector in [
    "S1",
    "S2",
    "S3",
]:

    original = selected_objects[
        sector
    ]

    loaded = readback[
        "selected_objects"
    ][sector]

    mean_equal = np.array_equal(
        original["mean"],
        loaded["mean"]
    )

    components_equal = np.array_equal(
        original["components"],
        loaded["components"]
    )

    map_equal = np.array_equal(
        original["M_forward"],
        loaded["M_forward"]
    )

    readback_audit[
        sector
    ] = {
        "mean_exact": bool(
            mean_equal
        ),
        "components_exact": bool(
            components_equal
        ),
        "M_forward_exact": bool(
            map_equal
        ),
        "pass": bool(
            mean_equal
            and components_equal
            and map_equal
        ),
    }

    print(
        f"  {sector}: "
        f"{'PASS' if readback_audit[sector]['pass'] else 'FAIL'}"
    )

    if not readback_audit[
        sector
    ]["pass"]:

        raise RuntimeError(
            f"Serialization readback failed for {sector}."
        )

print()


# -----------------------------------------------------------------------------
# 19. Build complete selection manifest
# -----------------------------------------------------------------------------

selection_summary = {}

for sector, winner in (
    selected_candidates.items()
):

    cal = winner[
        "calibration"
    ]

    obj = selected_objects[
        sector
    ]

    selection_summary[
        sector
    ] = {
        "selected_dimension": int(
            winner["dimension"]
        ),
        "selected_ridge_lambda": float(
            winner["ridge_lambda"]
        ),
        "transport_parameters": int(
            winner["transport_parameters"]
        ),
        "pca_retained_variance": float(
            winner["pca_retained_variance"]
        ),
        "calibration": {
            "transport_normalized_rmse": (
                cal[
                    "transport_normalized_rmse"
                ]
            ),
            "identity_normalized_rmse": (
                cal[
                    "identity_normalized_rmse"
                ]
            ),
            "normalized_rmse_improvement": (
                cal[
                    "normalized_rmse_improvement"
                ]
            ),
            "transport_mean_cosine": (
                cal[
                    "transport_mean_cosine"
                ]
            ),
            "identity_mean_cosine": (
                cal[
                    "identity_mean_cosine"
                ]
            ),
            "cosine_improvement": (
                cal[
                    "cosine_improvement"
                ]
            ),
            "transport_mean_euclidean_error": (
                cal[
                    "transport_mean_euclidean_error"
                ]
            ),
            "identity_mean_euclidean_error": (
                cal[
                    "identity_mean_euclidean_error"
                ]
            ),
            "relative_error_reduction": (
                cal[
                    "relative_error_reduction"
                ]
            ),
        },
        "fit_scope": (
            "TRAIN_ONLY"
        ),
        "test_used": False,
    }


result[
    "selected_candidates"
] = selection_summary

result[
    "candidate_selection_rule"
] = {
    "primary": (
        "minimum calibration normalized RMSE"
    ),
    "secondary": (
        "maximum calibration mean cosine"
    ),
    "tertiary": (
        "minimum transport parameter count"
    ),
    "quaternary": (
        "lower PCA dimension"
    ),
    "selection_scope": (
        "CALIBRATION_ONLY"
    ),
}


result[
    "winner_sanity"
] = winner_sanity

result[
    "frozen_object_audit"
] = object_audit

result[
    "train_orientation_audit"
] = train_orientation_audit

result[
    "serialization_readback"
] = readback_audit


# -----------------------------------------------------------------------------
# 20. Explicit scientific-scope governance
# -----------------------------------------------------------------------------

result[
    "scientific_scope"
] = {

    "state_bank_modified": False,

    "dataset_modified": False,

    "pca_basis_fit": True,

    "pca_basis_fit_scope": (
        "TRAIN_CLEAN_AND_CORRUPT_ONLY"
    ),

    "transport_maps_fit": True,

    "transport_fit_scope": (
        "TRAIN_CORRUPT_TO_TRAIN_CLEAN"
    ),

    "calibration_used_for_selection": True,

    "calibration_used_for_fitting": False,

    "test_used_for_selection": False,

    "test_used_for_fitting": False,

    "test_evaluated": False,

    "contextual_realization_completed": False,

    "triadic_irreducibility_tested": False,

    "geometric_realization_tested": False,

    "ctl_math_modified": False,
}


# -----------------------------------------------------------------------------
# 21. Numerical / operational classification
# -----------------------------------------------------------------------------

result[
    "classification"
] = (
    "TRANSPORT_CANDIDATE_SELECTION_PASS"
)

result[
    "status"
] = (
    "TRANSPORT_CANDIDATE_SELECTION_PASS"
)


# -----------------------------------------------------------------------------
# 22. Implementation governance notes
# -----------------------------------------------------------------------------

result[
    "governance_notes"
] = {

    "pca_basis": (
        "Each candidate dimension uses a PCA basis fitted only "
        "on the concatenated clean+corrupt TRAIN states."
    ),

    "transport_direction": (
        "Forward transport is frozen as corrupted PCA coordinates "
        "mapped toward clean PCA coordinates."
    ),

    "ridge_model": (
        "Linear ridge regression with no intercept."
    ),

    "calibration_role": (
        "Calibration is used exclusively for candidate evaluation "
        "and model selection."
    ),

    "test_role": (
        "Test remains untouched and is reserved for the later "
        "frozen evaluation."
    ),

    "dimension_governance": (
        "The candidate grid stops at d=64 despite state PCA supporting "
        "higher sample-admissible dimensions because the transport "
        "displacement field is less compressible and the paired "
        "transport-training sample contains only 96 records."
    ),

    "scientific_interpretation_guardrail": (
        "Transport success or failure is an empirical property of the "
        "specified operational transport reconstruction and does not "
        "by itself establish CTL triadic irreducibility or a geometric "
        "manifold structure."
    ),
}


# -----------------------------------------------------------------------------
# 23. Persist complete manifest
# -----------------------------------------------------------------------------

os.makedirs(
    RESULTS,
    exist_ok=True
)

with open(
    SELECTION_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result,
        f,
        indent=2,
        sort_keys=True
    )


# -----------------------------------------------------------------------------
# FINAL
# -----------------------------------------------------------------------------

print("=" * 78)
print(
    "FINAL CLASSIFICATION"
)
print("=" * 78)

print(
    "  TRANSPORT_CANDIDATE_SELECTION_PASS"
)

print()

print(
    "SELECTED TRANSPORT CONFIGURATIONS"
)

for sector in [
    "S1",
    "S2",
    "S3",
]:

    winner = selected_candidates[
        sector
    ]

    cal = winner[
        "calibration"
    ]

    print(
        f"  {sector}: "
        f"d={winner['dimension']}, "
        f"lambda={winner['ridge_lambda']}, "
        f"cal_NRMSE="
        f"{cal['transport_normalized_rmse']:.6f}, "
        f"identity="
        f"{cal['identity_normalized_rmse']:.6f}"
    )

print()

print(
    "Artifacts:"
)

print(
    f"  Manifest: "
    f"{SELECTION_MANIFEST_PATH}"
)

print(
    f"  Frozen maps/bases: "
    f"{SELECTED_MAPS_PATH}"
)

print()

print(
    "TEST STATUS:"
)

print(
    "  Test conditions remained completely locked."
)

print(
    "  No test transport score has been computed."
)

print()

print(
    "NEXT SCIENTIFIC STAGE:"
)

print(
    "  Phase 1E.2 — independent post-selection audit "
    "and frozen transport evaluation on TEST."
)

print(
    "  Contextual realization remains deferred."
)

print(
    "  Triadic irreducibility remains untested."
)

print("=" * 78)

ETTR-CTL-LLAMA-1 | Phase 1E.1 Train-Fitted Transport Candidate Selection
Experiment: ETTR-CTL-LLAMA-1
Model: meta-llama/Llama-3.2-3B
Selected layer: 14
Hidden size: 3072

PREREQUISITE ARTIFACT AUDIT
  state bank: PASS
  state manifest: PASS
  Phase 1D.3 audit: PASS
  Phase 1E.0 preflight: PASS
  dataset authorization: PASS

PHASE 1E.0 PREREQUISITE
  Transport analysis preflight: PASS

FROZEN STATE-BANK RELOAD
  S1: (384, 3072)
  S2: (384, 3072)
  S3: (384, 3072)
  target_logits: (384,)
  Reload: PASS

FROZEN STATE-BANK INTEGRITY FIREWALL
  S1 shape: PASS
  S1 hash: PASS
  S2 shape: PASS
  S2 hash: PASS
  S3 shape: PASS
  S3 hash: PASS
  target_logits hash: PASS

DATASET IDENTITY FIREWALL
  Dataset SHA: PASS

TRAIN / CALIBRATION / TEST FIREWALL
  Train records: 96
  Calibration records: 48
  Test records: 48
  Clean/corrupt pairing: PASS

PAIRED DATA CONSTRUCTION
  S1: train=96, calibration=48, test=48
  S2: train=96, calibration=48, test=48
  S3: train=96, calibration=48, test=48

PRED

In [33]:
# ============================================================
# ETTR-CTL-LLAMA-1
# PHASE 1E.2 — FROZEN TRANSPORT TEST EVALUATION
#
# COMPLETE REPLACEMENT
#
# CORRECTION:
#   Phase 1E.1 selected-map pickle has the structure:
#
#       {
#           ...
#           "selected_objects": {
#               "S1": {...},
#               "S2": {...},
#               "S3": {...}
#           }
#       }
#
#   The previous reader incorrectly searched for S1/S2/S3 at
#   the pickle top level.
#
# SCIENTIFIC GOVERNANCE:
#   TEST REFITTING: FORBIDDEN
#   PCA REFITTING: FORBIDDEN
#   MAP REFITTING: FORBIDDEN
#   HYPERPARAMETER SELECTION: FORBIDDEN
#   DATASET MODIFICATION: FORBIDDEN
#   CONTEXTUAL REALIZATION: NOT PERFORMED
#   TRIADIC IRREDUCIBILITY: NOT TESTED
#   GEOMETRIC RECONSTRUCTION CLAIM: NOT MADE
#   MATHEMATICAL DEFINITIONS: FROZEN
#
# AUTHORITATIVE STATE-BANK HASH:
#   SHA256(raw contiguous array bytes)
#
# TRANSPORT:
#   z = (x - mu) C^T
#   z_transport = z_corrupt M_forward
#
# C and mu:
#   TRAIN ONLY
#
# M_forward:
#   TRAIN ONLY
#
# d and ridge:
#   SELECTED ON CALIBRATION ONLY
#
# TEST:
#   EVALUATION ONLY
# ============================================================


import os
import json
import pickle
import hashlib
import math
import numpy as np

from pathlib import Path


# ============================================================
# 0. PATHS / CONSTANTS
# ============================================================

ROOT = Path(
    "/content/ettr_ctl_llama"
)

RESULTS = ROOT / "results"

CHECKPOINTS = ROOT / "checkpoints"

STATE_BANK_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank.npz"
)

STATE_MANIFEST_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank_manifest.json"
)

POST_AUDIT_PATH = (
    RESULTS /
    "llama_phase1d3_full_state_bank_post_extraction_audit.json"
)

AUTH_PATH = (
    RESULTS /
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

SELECTION_MANIFEST_PATH = (
    RESULTS /
    "llama_phase1e1_transport_candidate_selection.json"
)

SELECTED_MAPS_PATH = (
    CHECKPOINTS /
    "llama_phase1e1_selected_transport_maps.pkl"
)

OUTPUT_PATH = (
    RESULTS /
    "llama_phase1e2_frozen_transport_test_evaluation.json"
)


EXPECTED_DATASET_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)


EXPECTED_STATE_HASHES = {

    "S1":
        "9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783",

    "S2":
        "41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb",

    "S3":
        "173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3",

    "target_logits":
        "537fa43c8e278b8678de978c1dc2ec4ee5a1c77ca0a917dbf6a5823cf77ea58f",

    "target_logit_ranks":
        "f4de73b588c465e0c1e41887a2eab7b1e0f2214d2d15dc0108b69e4063e2d472",
}


EXPECTED_SHAPES = {

    "S1":
        (384, 3072),

    "S2":
        (384, 3072),

    "S3":
        (384, 3072),
}


EXPECTED_CONFIGS = {

    "S1": {

        "d":
            4,

        "ridge":
            0.01,
    },

    "S2": {

        "d":
            4,

        "ridge":
            1.0,
    },

    "S3": {

        "d":
            4,

        "ridge":
            0.01,
    },
}


SECTORS = [
    "S1",
    "S2",
    "S3",
]


N_RECORDS = 192

N_TEMPLATES = 12

N_PAIRS = 16

EPS = 1e-12

BOOTSTRAP_N = 5000

BOOTSTRAP_SEED_BASE = 42001


# ============================================================
# 1. UTILITY FUNCTIONS
# ============================================================

def sha256_raw_bytes(arr):

    """
    AUTHORITATIVE frozen hash convention.
    """

    a = np.ascontiguousarray(
        arr
    )

    return hashlib.sha256(
        a.tobytes(
            order="C"
        )
    ).hexdigest()


def sha256_metadata_bytes(arr):

    """
    Diagnostic hash only.
    """

    a = np.ascontiguousarray(
        arr
    )

    h = hashlib.sha256()

    h.update(
        str(
            a.dtype
        ).encode(
            "utf-8"
        )
    )

    h.update(
        str(
            a.shape
        ).encode(
            "utf-8"
        )
    )

    h.update(
        a.tobytes(
            order="C"
        )
    )

    return h.hexdigest()


def json_load(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


def finite_array(x):

    return bool(
        np.all(
            np.isfinite(
                np.asarray(x)
            )
        )
    )


def summarize(x):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    return {

        "n":
            int(
                len(x)
            ),

        "mean":
            float(
                np.mean(x)
            ),

        "median":
            float(
                np.median(x)
            ),

        "std":
            (
                float(
                    np.std(
                        x,
                        ddof=1
                    )
                )
                if len(x) > 1
                else 0.0
            ),

        "q05":
            float(
                np.quantile(
                    x,
                    0.05
                )
            ),

        "q25":
            float(
                np.quantile(
                    x,
                    0.25
                )
            ),

        "q75":
            float(
                np.quantile(
                    x,
                    0.75
                )
            ),

        "q95":
            float(
                np.quantile(
                    x,
                    0.95
                )
            ),

        "min":
            float(
                np.min(x)
            ),

        "max":
            float(
                np.max(x)
            ),
    }


def normalized_rmse_rows(
    pred,
    target
):

    pred = np.asarray(
        pred,
        dtype=np.float64
    )

    target = np.asarray(
        target,
        dtype=np.float64
    )

    numerator = np.linalg.norm(
        pred - target,
        axis=1
    )

    denominator = np.maximum(
        np.linalg.norm(
            target,
            axis=1
        ),
        EPS
    )

    return (
        numerator
        /
        denominator
    )


def cosine_rows(
    a,
    b
):

    a = np.asarray(
        a,
        dtype=np.float64
    )

    b = np.asarray(
        b,
        dtype=np.float64
    )

    na = np.linalg.norm(
        a,
        axis=1
    )

    nb = np.linalg.norm(
        b,
        axis=1
    )

    denominator = np.maximum(
        na * nb,
        EPS
    )

    return (
        np.sum(
            a * b,
            axis=1
        )
        /
        denominator
    )


def improvement_percent(
    baseline_error,
    method_error
):

    baseline_error = np.asarray(
        baseline_error,
        dtype=np.float64
    )

    method_error = np.asarray(
        method_error,
        dtype=np.float64
    )

    return (
        100.0
        *
        (
            baseline_error
            -
            method_error
        )
        /
        np.maximum(
            np.abs(
                baseline_error
            ),
            EPS
        )
    )


def bootstrap_difference_ci(
    baseline,
    method,
    seed,
    n_boot=BOOTSTRAP_N
):

    baseline = np.asarray(
        baseline,
        dtype=np.float64
    )

    method = np.asarray(
        method,
        dtype=np.float64
    )

    if len(baseline) != len(method):

        raise ValueError(
            "Bootstrap arrays must have equal length."
        )

    difference = (
        baseline
        -
        method
    )

    observed = float(
        np.mean(
            difference
        )
    )

    n = len(
        difference
    )

    if n < 2:

        return {

            "mean_difference":
                observed,

            "lower_95":
                observed,

            "upper_95":
                observed,

            "n_boot":
                0,
        }

    rng = np.random.default_rng(
        seed
    )

    indices = rng.integers(
        0,
        n,
        size=(
            n_boot,
            n
        )
    )

    boot_means = np.mean(
        difference[
            indices
        ],
        axis=1
    )

    return {

        "mean_difference":
            observed,

        "lower_95":
            float(
                np.quantile(
                    boot_means,
                    0.025
                )
            ),

        "upper_95":
            float(
                np.quantile(
                    boot_means,
                    0.975
                )
            ),

        "n_boot":
            int(
                n_boot
            ),
    }


def extract_field(
    obj,
    candidates,
    required=True
):

    if isinstance(
        obj,
        dict
    ):

        for key in candidates:

            if key in obj:

                return (
                    obj[key],
                    key
                )

    else:

        for key in candidates:

            if hasattr(
                obj,
                key
            ):

                return (
                    getattr(
                        obj,
                        key
                    ),
                    key
                )

    if required:

        raise KeyError(
            f"Could not find any of {candidates}."
        )

    return (
        None,
        None
    )


# ============================================================
# 2. HEADER
# ============================================================

print(
    "=" * 80
)

print(
    "ETTR-CTL-LLAMA-1 — PHASE 1E.2"
)

print(
    "FROZEN TRANSPORT TEST EVALUATION"
)

print(
    "=" * 80
)

print(
    f"ROOT: {ROOT}"
)

print(
    f"State bank: {STATE_BANK_PATH}"
)

print(
    f"Selection manifest: "
    f"{SELECTION_MANIFEST_PATH}"
)

print(
    f"Selected maps: "
    f"{SELECTED_MAPS_PATH}"
)

print(
    f"Output: {OUTPUT_PATH}"
)

print(
    "\nGOVERNANCE:"
)

print(
    "  TEST REFITTING: FORBIDDEN"
)

print(
    "  PCA REFITTING: FORBIDDEN"
)

print(
    "  MAP REFITTING: FORBIDDEN"
)

print(
    "  HYPERPARAMETER SELECTION: FORBIDDEN"
)

print(
    "  CONTEXTUAL REALIZATION: NOT PERFORMED"
)

print(
    "  TRIADIC IRREDUCIBILITY: NOT TESTED"
)

print(
    "  GEOMETRIC RECONSTRUCTION CLAIM: NOT MADE"
)

print(
    "  MATHEMATICAL DEFINITIONS: FROZEN"
)


# ============================================================
# 3. REQUIRED ARTIFACTS
# ============================================================

required_paths = [

    STATE_BANK_PATH,

    STATE_MANIFEST_PATH,

    POST_AUDIT_PATH,

    AUTH_PATH,

    SELECTION_MANIFEST_PATH,

    SELECTED_MAPS_PATH,
]


missing = [

    str(path)

    for path in required_paths

    if not path.exists()
]


if missing:

    raise FileNotFoundError(

        "Phase 1E.2 prerequisite artifacts "
        "are missing:\n"
        +
        "\n".join(
            missing
        )
    )


print(
    "\n[PASS] Required Llama-side artifacts exist."
)

print(
    "[INFO] GPT-2 recovered dataset JSON is not "
    "required for frozen test application."
)


# ============================================================
# 4. DATASET AUTHORIZATION
# ============================================================

auth = json_load(
    AUTH_PATH
)


dataset_authorized = bool(

    auth.get(

        "dataset_authorized",

        auth.get(
            "authorized",
            False
        )
    )
)


authorized_sha = auth.get(

    "dataset_sha256",

    auth.get(
        "sha256",
        None
    )
)


print(
    "\nDATASET AUTHORIZATION"
)

print(
    "-" * 80
)

print(
    f"  authorized: "
    f"{dataset_authorized}"
)

print(
    f"  recorded SHA: "
    f"{authorized_sha}"
)

print(
    f"  expected SHA: "
    f"{EXPECTED_DATASET_SHA256}"
)


if not dataset_authorized:

    raise RuntimeError(
        "The operative recovered dataset "
        "is not authorized."
    )


if (

    authorized_sha is not None

    and

    authorized_sha
    != EXPECTED_DATASET_SHA256

):

    raise RuntimeError(
        "Dataset authorization SHA mismatch."
    )


print(
    "[PASS] Frozen dataset authorization is valid."
)


# ============================================================
# 5. STATE-BANK MANIFEST
# ============================================================

state_manifest = json_load(
    STATE_MANIFEST_PATH
)

post_audit = json_load(
    POST_AUDIT_PATH
)


print(
    "\nSTATE-BANK MANIFEST"
)

print(
    "-" * 80
)


manifest_experiment = state_manifest.get(
    "experiment_id",
    None
)

manifest_model = state_manifest.get(
    "model_id",
    state_manifest.get(
        "model",
        None
    )
)

manifest_layer = state_manifest.get(
    "selected_layer",
    state_manifest.get(
        "layer",
        None
    )
)


print(
    f"  experiment_id: "
    f"{manifest_experiment}"
)

print(
    f"  model: "
    f"{manifest_model}"
)

print(
    f"  layer: "
    f"{manifest_layer}"
)


if (

    manifest_experiment is not None

    and

    manifest_experiment
    != "ETTR-CTL-LLAMA-1"

):

    raise RuntimeError(
        "Unexpected experiment identity."
    )


if (

    manifest_model is not None

    and

    "Llama-3.2-3B"
    not in str(
        manifest_model
    )

):

    raise RuntimeError(
        "Unexpected model identity."
    )


print(
    "[PASS] State-bank experiment identity."
)


# ============================================================
# 6. LOAD STATE BANK
# ============================================================

bank = np.load(

    STATE_BANK_PATH,

    allow_pickle=False
)


print(
    "\nSTATE-BANK SCHEMA"
)

print(
    "-" * 80
)


for key in bank.files:

    print(
        f"  - {key}"
    )


required_members = {

    "S1",

    "S2",

    "S3",

    "target_logits",

    "target_logit_ranks",

    "condition_prompt_lengths",

    "condition_p_T",

    "condition_i_T",

    "condition_llama_target_ids",

    "condition_gpt2_target_ids",
}


missing_members = sorted(

    required_members
    -
    set(
        bank.files
    )
)


if missing_members:

    raise RuntimeError(

        "State bank is missing required members: "
        +
        str(
            missing_members
        )
    )


print(
    "[PASS] Required state-bank schema."
)


# ============================================================
# 7. STATE-BANK HASH FIREWALL
# ============================================================

states = {

    sector:
        np.asarray(
            bank[
                sector
            ]
        )

    for sector in SECTORS
}


print(
    "\nSTATE-BANK HASH FIREWALL"
)

print(
    "-" * 80
)

print(
    "AUTHORITATIVE HASH:"
)

print(
    "  SHA256(raw contiguous array bytes)"
)

print(
    "DIAGNOSTIC HASH:"
)

print(
    "  SHA256(dtype + shape + raw bytes)"
)


state_hash_results = {}


for sector in SECTORS:

    arr = states[
        sector
    ]

    expected_shape = (
        EXPECTED_SHAPES[
            sector
        ]
    )

    actual_shape = tuple(
        arr.shape
    )

    expected_hash = (
        EXPECTED_STATE_HASHES[
            sector
        ]
    )

    actual_raw_hash = (
        sha256_raw_bytes(
            arr
        )
    )

    actual_metadata_hash = (
        sha256_metadata_bytes(
            arr
        )
    )

    finite = finite_array(
        arr
    )

    nonzero = bool(
        np.any(
            np.abs(arr) > 0
        )
    )

    raw_match = (
        actual_raw_hash
        ==
        expected_hash
    )


    print(
        f"\n{sector}"
    )

    print(
        f"  shape: "
        f"{actual_shape}"
    )

    print(
        f"  expected shape: "
        f"{expected_shape}"
    )

    print(
        f"  finite: "
        f"{finite}"
    )

    print(
        f"  nonzero: "
        f"{nonzero}"
    )

    print(
        f"  expected SHA: "
        f"{expected_hash}"
    )

    print(
        f"  raw-byte SHA: "
        f"{actual_raw_hash}"
    )

    print(
        f"  metadata+bytes SHA: "
        f"{actual_metadata_hash}"
    )

    print(
        f"  raw-byte match: "
        f"{raw_match}"
    )


    if actual_shape != expected_shape:

        raise RuntimeError(
            f"{sector} shape mismatch."
        )


    if not raw_match:

        raise RuntimeError(
            f"{sector} frozen raw-byte hash mismatch."
        )


    if not finite:

        raise RuntimeError(
            f"{sector} contains non-finite values."
        )


    if not nonzero:

        raise RuntimeError(
            f"{sector} is numerically zero."
        )


    state_hash_results[
        sector
    ] = {

        "shape":
            list(
                actual_shape
            ),

        "expected_sha256":
            expected_hash,

        "actual_raw_bytes_sha256":
            actual_raw_hash,

        "actual_metadata_bytes_sha256":
            actual_metadata_hash,

        "raw_bytes_hash_match":
            True,

        "finite":
            finite,

        "nonzero":
            nonzero,
    }


print(
    "\n[PASS] Frozen state-bank hashes."
)


# ============================================================
# 8. AUXILIARY HASH FIREWALL
# ============================================================

print(
    "\nAUXILIARY STATE-BANK HASH FIREWALL"
)

print(
    "-" * 80
)


auxiliary_hash_results = {}


for field_name in [

    "target_logits",

    "target_logit_ranks",

]:

    arr = np.asarray(
        bank[
            field_name
        ]
    )

    expected_hash = (
        EXPECTED_STATE_HASHES[
            field_name
        ]
    )

    actual_raw_hash = (
        sha256_raw_bytes(
            arr
        )
    )

    actual_metadata_hash = (
        sha256_metadata_bytes(
            arr
        )
    )

    raw_match = (
        actual_raw_hash
        ==
        expected_hash
    )


    print(
        f"\n{field_name}"
    )

    print(
        f"  shape: "
        f"{arr.shape}"
    )

    print(
        f"  expected SHA: "
        f"{expected_hash}"
    )

    print(
        f"  raw-byte SHA: "
        f"{actual_raw_hash}"
    )

    print(
        f"  metadata+bytes SHA: "
        f"{actual_metadata_hash}"
    )

    print(
        f"  raw-byte match: "
        f"{raw_match}"
    )


    if not raw_match:

        raise RuntimeError(
            f"{field_name} frozen raw-byte hash mismatch."
        )


    auxiliary_hash_results[
        field_name
    ] = {

        "shape":
            list(
                arr.shape
            ),

        "expected_sha256":
            expected_hash,

        "actual_raw_bytes_sha256":
            actual_raw_hash,

        "actual_metadata_bytes_sha256":
            actual_metadata_hash,

        "raw_bytes_hash_match":
            True,
    }


print(
    "\n[PASS] Auxiliary state-bank hashes."
)


# ============================================================
# 9. TARGET POSITION AUDIT
# ============================================================

condition_p_T = np.asarray(
    bank[
        "condition_p_T"
    ]
)

condition_i_T = np.asarray(
    bank[
        "condition_i_T"
    ]
)

condition_lengths = np.asarray(
    bank[
        "condition_prompt_lengths"
    ]
)


if len(
    condition_p_T
) != 384:

    raise RuntimeError(
        "Expected 384 p_T values."
    )


if len(
    condition_i_T
) != 384:

    raise RuntimeError(
        "Expected 384 i_T values."
    )


if len(
    condition_lengths
) != 384:

    raise RuntimeError(
        "Expected 384 prompt-length values."
    )


position_identity = bool(

    np.array_equal(

        condition_i_T,

        condition_p_T - 1

    )

)


if not position_identity:

    raise RuntimeError(
        "Frozen target-position convention failed."
    )


clean_lengths = (
    condition_lengths[
        :N_RECORDS
    ]
)

corrupt_lengths = (
    condition_lengths[
        N_RECORDS:
    ]
)


length_summary = {

    "clean_min":
        int(
            np.min(
                clean_lengths
            )
        ),

    "clean_max":
        int(
            np.max(
                clean_lengths
            )
        ),

    "corrupt_min":
        int(
            np.min(
                corrupt_lengths
            )
        ),

    "corrupt_max":
        int(
            np.max(
                corrupt_lengths
            )
        ),

    "clean_corrupt_exact_equal":
        bool(
            np.array_equal(
                clean_lengths,
                corrupt_lengths
            )
        ),
}


print(
    "\nTARGET POSITION"
)

print(
    "-" * 80
)

print(
    "  p_T = L(P): VERIFIED"
)

print(
    "  i_T = L(P)-1: VERIFIED"
)

print(
    "  prompt lengths retained as descriptive metadata."
)

print(
    f"  clean length range: "
    f"{length_summary['clean_min']}"
    f"–"
    f"{length_summary['clean_max']}"
)

print(
    f"  corrupt length range: "
    f"{length_summary['corrupt_min']}"
    f"–"
    f"{length_summary['corrupt_max']}"
)

print(
    f"  exact clean/corrupt length equality: "
    f"{length_summary['clean_corrupt_exact_equal']}"
)

print(
    "[PASS] Frozen target-position operationalization."
)


# ============================================================
# 10. FROZEN SPLIT RECONSTRUCTION
# ============================================================

print(
    "\nFROZEN SPLIT RECONSTRUCTION"
)

print(
    "-" * 80
)


record_template_indices = []

record_pair_indices = []

record_r_values = []

record_split_labels = []


for t in range(
    N_TEMPLATES
):

    for p in range(
        N_PAIRS
    ):

        r = (
            p
            +
            4 * t
        ) % 16


        if r < 4:

            split = (
                "calibration"
            )

        elif r < 8:

            split = (
                "test"
            )

        else:

            split = (
                "train"
            )


        record_template_indices.append(
            t
        )

        record_pair_indices.append(
            p
        )

        record_r_values.append(
            r
        )

        record_split_labels.append(
            split
        )


record_template_indices = np.asarray(

    record_template_indices,

    dtype=np.int64
)


record_pair_indices = np.asarray(

    record_pair_indices,

    dtype=np.int64
)


record_r_values = np.asarray(

    record_r_values,

    dtype=np.int64
)


record_split_labels = np.asarray(

    record_split_labels,

    dtype=object
)


train_record_indices = np.where(

    record_split_labels
    ==
    "train"

)[0]


cal_record_indices = np.where(

    record_split_labels
    ==
    "calibration"

)[0]


test_record_indices = np.where(

    record_split_labels
    ==
    "test"

)[0]


split_counts = {

    "train":
        int(
            len(
                train_record_indices
            )
        ),

    "calibration":
        int(
            len(
                cal_record_indices
            )
        ),

    "test":
        int(
            len(
                test_record_indices
            )
        ),
}


print(
    f"  train: "
    f"{split_counts['train']}"
)

print(
    f"  calibration: "
    f"{split_counts['calibration']}"
)

print(
    f"  test: "
    f"{split_counts['test']}"
)


if split_counts != {

    "train": 96,

    "calibration": 48,

    "test": 48,

}:

    raise RuntimeError(
        "Frozen split counts failed."
    )


test_per_template = {}


for t in range(
    N_TEMPLATES
):

    mask = (

        (
            record_template_indices
            ==
            t
        )

        &

        (
            record_split_labels
            ==
            "test"
        )

    )

    test_per_template[
        str(t)
    ] = int(
        np.sum(mask)
    )


if any(

    value != 4

    for value in
    test_per_template.values()

):

    raise RuntimeError(
        "Frozen test balance failed."
    )


print(
    f"  test records per template: "
    f"{test_per_template}"
)

print(
    "[PASS] Frozen crossed-design split."
)


# ============================================================
# 11. CONDITION SPLIT CORRESPONDENCE
# ============================================================

condition_split_labels = np.concatenate([

    record_split_labels,

    record_split_labels,

])


if not np.array_equal(

    condition_split_labels[
        :N_RECORDS
    ],

    condition_split_labels[
        N_RECORDS:
    ],

):

    raise RuntimeError(
        "Clean/corrupt split correspondence failed."
    )


print(
    "[PASS] Clean/corrupt split correspondence."
)


# ============================================================
# 12. LOAD FROZEN PHASE 1E.1 CHECKPOINT
# ============================================================

print(
    "\nFROZEN TRANSPORT CHECKPOINT"
)

print(
    "-" * 80
)


with open(
    SELECTED_MAPS_PATH,
    "rb"
) as f:

    frozen_checkpoint = pickle.load(
        f
    )


if not isinstance(
    frozen_checkpoint,
    dict
):

    raise TypeError(
        "Frozen transport checkpoint must be a dictionary."
    )


print(
    "  top-level keys:"
)


for key in frozen_checkpoint.keys():

    print(
        f"    - {key}"
    )


required_checkpoint_keys = {

    "experiment",

    "phase",

    "model",

    "selected_layer",

    "hidden_size",

    "dataset_sha256",

    "state_bank_hashes",

    "fit_scope",

    "pca_scope",

    "transport_scope",

    "transport_orientation",

    "intercept",

    "selected_objects",
}


missing_checkpoint_keys = sorted(

    required_checkpoint_keys
    -
    set(
        frozen_checkpoint.keys()
    )
)


if missing_checkpoint_keys:

    raise RuntimeError(

        "Frozen transport checkpoint missing keys: "
        +
        str(
            missing_checkpoint_keys
        )
    )


checkpoint_selected_objects = (
    frozen_checkpoint[
        "selected_objects"
    ]
)


if not isinstance(
    checkpoint_selected_objects,
    dict
):

    raise TypeError(
        "checkpoint['selected_objects'] "
        "must be a dictionary."
    )


print(
    "\n  selected_objects keys:"
)


for key in checkpoint_selected_objects.keys():

    print(
        f"    - {key}"
    )


missing_sector_objects = sorted(

    set(SECTORS)
    -
    set(
        checkpoint_selected_objects.keys()
    )
)


if missing_sector_objects:

    raise RuntimeError(

        "Frozen checkpoint missing sectors: "
        +
        str(
            missing_sector_objects
        )
    )


print(
    "[PASS] Frozen checkpoint schema."
)


# ============================================================
# 13. CHECKPOINT GOVERNANCE AUDIT
# ============================================================

checkpoint_experiment = (
    frozen_checkpoint[
        "experiment"
    ]
)


checkpoint_phase = (
    frozen_checkpoint[
        "phase"
    ]
)


checkpoint_model = (
    frozen_checkpoint[
        "model"
    ]
)


checkpoint_layer = (
    frozen_checkpoint[
        "selected_layer"
    ]
)


checkpoint_hidden_size = (
    frozen_checkpoint[
        "hidden_size"
    ]
)


checkpoint_dataset_sha = (
    frozen_checkpoint[
        "dataset_sha256"
    ]
)


print(
    "\nCHECKPOINT GOVERNANCE"
)

print(
    "-" * 80
)

print(
    f"  experiment: "
    f"{checkpoint_experiment}"
)

print(
    f"  phase: "
    f"{checkpoint_phase}"
)

print(
    f"  model: "
    f"{checkpoint_model}"
)

print(
    f"  selected layer: "
    f"{checkpoint_layer}"
)

print(
    f"  hidden size: "
    f"{checkpoint_hidden_size}"
)

print(
    f"  dataset SHA: "
    f"{checkpoint_dataset_sha}"
)


if checkpoint_experiment != (
    "ETTR-CTL-LLAMA-1"
):

    raise RuntimeError(
        "Frozen checkpoint experiment mismatch."
    )


if str(
    checkpoint_phase
).lower() not in {

    "1e.1",

    "phase 1e.1",

    "1e1",

}:

    print(
        "[INFO] Phase field retained as artifact metadata: "
        f"{checkpoint_phase}"
    )


if "Llama-3.2-3B" not in str(
    checkpoint_model
):

    raise RuntimeError(
        "Frozen checkpoint model mismatch."
    )


if int(
    checkpoint_layer
) != 14:

    raise RuntimeError(
        "Frozen checkpoint layer mismatch."
    )


if int(
    checkpoint_hidden_size
) != 3072:

    raise RuntimeError(
        "Frozen checkpoint hidden size mismatch."
    )


if checkpoint_dataset_sha != (
    EXPECTED_DATASET_SHA256
):

    raise RuntimeError(
        "Frozen checkpoint dataset SHA mismatch."
    )


fit_scope = frozen_checkpoint[
    "fit_scope"
]

pca_scope = frozen_checkpoint[
    "pca_scope"
]

transport_scope = frozen_checkpoint[
    "transport_scope"
]

transport_orientation = frozen_checkpoint[
    "transport_orientation"
]

intercept_setting = frozen_checkpoint[
    "intercept"
]


print(
    "\n  fit_scope:"
)

print(
    f"    {fit_scope}"
)

print(
    "  pca_scope:"
)

print(
    f"    {pca_scope}"
)

print(
    "  transport_scope:"
)

print(
    f"    {transport_scope}"
)

print(
    "  transport_orientation:"
)

print(
    f"    {transport_orientation}"
)

print(
    "  intercept:"
)

print(
    f"    {intercept_setting}"
)


print(
    "[PASS] Frozen checkpoint governance metadata."
)


# ============================================================
# 14. FROZEN MAP OBJECT EXTRACTION
# ============================================================

frozen_configs = {}


for sector in SECTORS:

    obj = (
        checkpoint_selected_objects[
            sector
        ]
    )


    if not isinstance(
        obj,
        dict
    ):

        raise TypeError(
            f"{sector} frozen object must be a dictionary."
        )


    print(
        f"\n{sector} FROZEN OBJECT"
    )

    print(
        "-" * 60
    )


    print(
        "  keys:"
    )

    for key in obj.keys():

        print(
            f"    - {key}"
        )


    dimension, dimension_key = (
        extract_field(

            obj,

            [

                "d",

                "dimension",

                "pca_dim",

                "n_components",

            ]

        )
    )


    ridge, ridge_key = (
        extract_field(

            obj,

            [

                "ridge",

                "lambda",

                "ridge_lambda",

                "alpha",

            ]

        )
    )


    components, components_key = (
        extract_field(

            obj,

            [

                "components",

                "C",

                "pca_components",

                "basis",

            ]

        )
    )


    mean, mean_key = (
        extract_field(

            obj,

            [

                "mean",

                "mu",

                "pca_mean",

                "center",

            ]

        )
    )


    M_forward, map_key = (
        extract_field(

            obj,

            [

                "M_forward",

                "map_forward",

                "transport_map",

                "M",

            ]

        )
    )


    dimension = int(
        dimension
    )


    ridge = float(
        ridge
    )


    components = np.asarray(

        components,

        dtype=np.float64

    )


    mean = np.asarray(

        mean,

        dtype=np.float64

    ).reshape(-1)


    M_forward = np.asarray(

        M_forward,

        dtype=np.float64

    )


    expected = EXPECTED_CONFIGS[
        sector
    ]


    print(
        f"  d: "
        f"{dimension}"
    )

    print(
        f"  ridge: "
        f"{ridge}"
    )

    print(
        f"  components shape: "
        f"{components.shape}"
    )

    print(
        f"  mean shape: "
        f"{mean.shape}"
    )

    print(
        f"  M_forward shape: "
        f"{M_forward.shape}"
    )


    if dimension != expected[
        "d"
    ]:

        raise RuntimeError(
            f"{sector}: frozen dimension does not "
            f"match Phase 1E.1 selected configuration."
        )


    if not math.isclose(

        ridge,

        expected["ridge"],

        rel_tol=0.0,

        abs_tol=1e-12

    ):

        raise RuntimeError(
            f"{sector}: frozen ridge does not "
            f"match Phase 1E.1 selected configuration."
        )


    if components.shape != (

        dimension,

        3072

    ):

        raise RuntimeError(
            f"{sector}: frozen PCA component shape mismatch."
        )


    if mean.shape != (

        3072,

    ):

        raise RuntimeError(
            f"{sector}: frozen PCA mean shape mismatch."
        )


    if M_forward.shape != (

        dimension,

        dimension

    ):

        raise RuntimeError(
            f"{sector}: frozen transport map shape mismatch."
        )


    if not finite_array(
        components
    ):

        raise RuntimeError(
            f"{sector}: frozen PCA components non-finite."
        )


    if not finite_array(
        mean
    ):

        raise RuntimeError(
            f"{sector}: frozen PCA mean non-finite."
        )


    if not finite_array(
        M_forward
    ):

        raise RuntimeError(
            f"{sector}: frozen transport map non-finite."
        )


    frozen_configs[
        sector
    ] = {

        "d":
            dimension,

        "ridge":
            ridge,

        "components":
            components,

        "mean":
            mean,

        "M_forward":
            M_forward,

        "field_keys":
            {

                "dimension":
                    dimension_key,

                "ridge":
                    ridge_key,

                "components":
                    components_key,

                "mean":
                    mean_key,

                "map":
                    map_key,
            },
    }


print(
    "\n[PASS] All frozen sector objects independently verified."
)


# ============================================================
# 15. STATE-BANK HASHES RECORDED IN CHECKPOINT
# ============================================================

checkpoint_state_hashes = (
    frozen_checkpoint[
        "state_bank_hashes"
    ]
)


print(
    "\nCHECKPOINT STATE-BANK HASH REFERENCES"
)

print(
    "-" * 80
)


for sector in SECTORS:

    expected_hash = (
        EXPECTED_STATE_HASHES[
            sector
        ]
    )

    checkpoint_hash = None


    if isinstance(
        checkpoint_state_hashes,
        dict
    ):

        checkpoint_hash = (
            checkpoint_state_hashes.get(
                sector,
                None
            )
        )


    print(
        f"  {sector}: "
        f"{checkpoint_hash}"
    )


    if checkpoint_hash is not None:

        if isinstance(
            checkpoint_hash,
            dict
        ):

            candidate_hash = (

                checkpoint_hash.get(
                    "sha256",
                    checkpoint_hash.get(
                        "raw_bytes_sha256",
                        checkpoint_hash.get(
                            "hash",
                            None
                        )
                    )
                )

            )

        else:

            candidate_hash = (
                str(
                    checkpoint_hash
                )
            )


        if candidate_hash != expected_hash:

            raise RuntimeError(
                f"{sector}: checkpoint state-bank "
                f"hash reference mismatch."
            )


print(
    "[PASS] Checkpoint/state-bank continuity audit."
)


# ============================================================
# 16. EXPLICIT NO-REFIT GUARD
# ============================================================

fit_operations = {

    "pca_fit":
        False,

    "transport_map_fit":
        False,

    "ridge_selection":
        False,

    "hyperparameter_selection":
        False,

    "test_refit":
        False,
}


if any(
    fit_operations.values()
):

    raise RuntimeError(
        "Forbidden fitting operation detected."
    )


print(
    "\n[PASS] No-refit guard."
)


# ============================================================
# 17. FROZEN PCA ENCODE / DECODE
# ============================================================

def frozen_pca_encode(
    X,
    cfg
):

    X = np.asarray(
        X,
        dtype=np.float64
    )

    centered = (
        X
        -
        cfg["mean"][None, :]
    )

    Z = (
        centered
        @
        cfg["components"].T
    )

    return Z


def frozen_pca_decode(
    Z,
    cfg
):

    Z = np.asarray(
        Z,
        dtype=np.float64
    )

    X = (
        Z
        @
        cfg["components"]
    )

    X = (
        X
        +
        cfg["mean"][None, :]
    )

    return X


# ============================================================
# 18. PAIR CLEAN/CORRUPT STATES
# ============================================================

def pair_states(
    sector,
    record_indices
):

    arr = states[
        sector
    ]

    record_indices = np.asarray(
        record_indices,
        dtype=np.int64
    )


    clean = arr[
        record_indices
    ]


    corrupt = arr[
        N_RECORDS
        +
        record_indices
    ]


    return (

        np.asarray(
            clean,
            dtype=np.float64
        ),

        np.asarray(
            corrupt,
            dtype=np.float64
        ),

    )


# ============================================================
# 19. FROZEN TEST EVALUATION
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "FROZEN TEST EVALUATION"
)

print(
    "=" * 80
)


test_results = {}


for sector_index, sector in enumerate(
    SECTORS
):

    cfg = frozen_configs[
        sector
    ]


    clean_test, corrupt_test = pair_states(

        sector,

        test_record_indices

    )


    # --------------------------------------------------------
    # 19A. FROZEN PCA ENCODING
    # --------------------------------------------------------

    z_clean_test = frozen_pca_encode(

        clean_test,

        cfg

    )


    z_corrupt_test = frozen_pca_encode(

        corrupt_test,

        cfg

    )


    # --------------------------------------------------------
    # 19B. FROZEN TRANSPORT
    # --------------------------------------------------------

    z_transport_test = (

        z_corrupt_test
        @
        cfg["M_forward"]

    )


    z_identity_test = (
        z_corrupt_test.copy()
    )


    # --------------------------------------------------------
    # 19C. PCA-SPACE METRICS
    # --------------------------------------------------------

    transport_nrmse = normalized_rmse_rows(

        z_transport_test,

        z_clean_test

    )


    identity_nrmse = normalized_rmse_rows(

        z_identity_test,

        z_clean_test

    )


    transport_cosine = cosine_rows(

        z_transport_test,

        z_clean_test

    )


    identity_cosine = cosine_rows(

        z_identity_test,

        z_clean_test

    )


    nrmse_improvement = improvement_percent(

        identity_nrmse,

        transport_nrmse

    )


    cosine_improvement = (

        transport_cosine
        -
        identity_cosine

    )


    # --------------------------------------------------------
    # 19D. FULL-DIMENSIONAL DECODE
    # --------------------------------------------------------

    decoded_transport = frozen_pca_decode(

        z_transport_test,

        cfg

    )


    decoded_identity_pca = frozen_pca_decode(

        z_identity_test,

        cfg

    )


    decoded_clean_pca = frozen_pca_decode(

        z_clean_test,

        cfg

    )


    # --------------------------------------------------------
    # 19E. FULL-DIMENSIONAL CONTROLS
    # --------------------------------------------------------

    raw_identity_nrmse = normalized_rmse_rows(

        corrupt_test,

        clean_test

    )


    raw_identity_cosine = cosine_rows(

        corrupt_test,

        clean_test

    )


    decoded_transport_nrmse = normalized_rmse_rows(

        decoded_transport,

        clean_test

    )


    decoded_transport_cosine = cosine_rows(

        decoded_transport,

        clean_test

    )


    decoded_identity_pca_nrmse = normalized_rmse_rows(

        decoded_identity_pca,

        clean_test

    )


    decoded_identity_pca_cosine = cosine_rows(

        decoded_identity_pca,

        clean_test

    )


    decoded_clean_pca_nrmse = normalized_rmse_rows(

        decoded_clean_pca,

        clean_test

    )


    decoded_clean_pca_cosine = cosine_rows(

        decoded_clean_pca,

        clean_test

    )


    # --------------------------------------------------------
    # 19F. PAIRED BOOTSTRAP
    # --------------------------------------------------------

    pca_bootstrap = bootstrap_difference_ci(

        identity_nrmse,

        transport_nrmse,

        seed=(
            BOOTSTRAP_SEED_BASE
            +
            10 * sector_index
            +
            1
        )

    )


    decoded_bootstrap = bootstrap_difference_ci(

        decoded_identity_pca_nrmse,

        decoded_transport_nrmse,

        seed=(
            BOOTSTRAP_SEED_BASE
            +
            10 * sector_index
            +
            2
        )

    )


    # --------------------------------------------------------
    # 19G. CLASSIFICATION
    # --------------------------------------------------------

    mean_transport = float(
        np.mean(
            transport_nrmse
        )
    )


    mean_identity = float(
        np.mean(
            identity_nrmse
        )
    )


    mean_improvement = float(
        np.mean(
            nrmse_improvement
        )
    )


    if (
        mean_transport
        <
        mean_identity
    ):

        pca_status = (
            "TEST_TRANSPORT_IMPROVES_IDENTITY"
        )

    elif (
        mean_transport
        >
        mean_identity
    ):

        pca_status = (
            "TEST_TRANSPORT_WORSE_THAN_IDENTITY"
        )

    else:

        pca_status = (
            "TEST_TRANSPORT_TIED_WITH_IDENTITY"
        )


    ci_low = (
        pca_bootstrap[
            "lower_95"
        ]
    )


    ci_high = (
        pca_bootstrap[
            "upper_95"
        ]
    )


    if ci_low > 0:

        bootstrap_status = (
            "BOOTSTRAP_SUPPORTS_POSITIVE_TRANSPORT_GAIN"
        )

    elif ci_high < 0:

        bootstrap_status = (
            "BOOTSTRAP_SUPPORTS_NEGATIVE_TRANSPORT_GAIN"
        )

    else:

        bootstrap_status = (
            "BOOTSTRAP_CI_CROSSES_ZERO"
        )


    # --------------------------------------------------------
    # 19H. DECODED-SPACE CLASSIFICATION
    # --------------------------------------------------------

    decoded_transport_mean = float(
        np.mean(
            decoded_transport_nrmse
        )
    )


    decoded_identity_mean = float(
        np.mean(
            decoded_identity_pca_nrmse
        )
    )


    if (
        decoded_transport_mean
        <
        decoded_identity_mean
    ):

        decoded_status = (
            "DECODED_TRANSPORT_IMPROVES_FROZEN_PCA_IDENTITY"
        )

    elif (
        decoded_transport_mean
        >
        decoded_identity_mean
    ):

        decoded_status = (
            "DECODED_TRANSPORT_WORSE_THAN_FROZEN_PCA_IDENTITY"
        )

    else:

        decoded_status = (
            "DECODED_TRANSPORT_TIED_WITH_FROZEN_PCA_IDENTITY"
        )


    # --------------------------------------------------------
    # 19I. PRINT RESULTS
    # --------------------------------------------------------

    print(
        f"\n{sector}"
    )

    print(
        "-" * 80
    )

    print(
        "FROZEN CONFIGURATION"
    )

    print(
        f"  d = {cfg['d']}"
    )

    print(
        f"  ridge = {cfg['ridge']}"
    )


    print(
        "\nPCA-SPACE TEST"
    )

    print(
        f"  transport NRMSE: "
        f"{mean_transport:.9f}"
    )

    print(
        f"  identity NRMSE:  "
        f"{mean_identity:.9f}"
    )

    print(
        f"  mean improvement: "
        f"{mean_improvement:.6f}%"
    )

    print(
        f"  transport cosine: "
        f"{np.mean(transport_cosine):.9f}"
    )

    print(
        f"  identity cosine:  "
        f"{np.mean(identity_cosine):.9f}"
    )


    print(
        "\nPAIRED BOOTSTRAP"
    )

    print(
        f"  identity - transport NRMSE: "
        f"{pca_bootstrap['mean_difference']:.9f}"
    )

    print(
        f"  95% CI: "
        f"["
        f"{pca_bootstrap['lower_95']:.9f}, "
        f"{pca_bootstrap['upper_95']:.9f}"
        f"]"
    )

    print(
        f"  classification: "
        f"{bootstrap_status}"
    )


    print(
        "\nFULL-DIMENSIONAL DECODED TEST"
    )

    print(
        f"  decoded transport NRMSE: "
        f"{decoded_transport_mean:.9f}"
    )

    print(
        f"  decoded PCA identity NRMSE: "
        f"{decoded_identity_mean:.9f}"
    )

    print(
        f"  raw full-dimensional identity NRMSE: "
        f"{np.mean(raw_identity_nrmse):.9f}"
    )

    print(
        f"  clean-state PCA reconstruction NRMSE: "
        f"{np.mean(decoded_clean_pca_nrmse):.9f}"
    )

    print(
        f"  decoded-space classification: "
        f"{decoded_status}"
    )


    test_results[
        sector
    ] = {

        "configuration": {

            "dimension":
                int(
                    cfg["d"]
                ),

            "ridge":
                float(
                    cfg["ridge"]
                ),

            "components_shape":
                list(
                    cfg["components"].shape
                ),

            "mean_shape":
                list(
                    cfg["mean"].shape
                ),

            "M_forward_shape":
                list(
                    cfg["M_forward"].shape
                ),

            "fit_split":
                "train_only",

            "selection_split":
                "calibration_only",

            "test_refit":
                False,
        },


        "n_test_records":
            int(
                len(
                    test_record_indices
                )
            ),


        "pca_space": {

            "transport_nrmse":
                summarize(
                    transport_nrmse
                ),

            "identity_nrmse":
                summarize(
                    identity_nrmse
                ),

            "transport_cosine":
                summarize(
                    transport_cosine
                ),

            "identity_cosine":
                summarize(
                    identity_cosine
                ),

            "nrmse_improvement_percent":
                summarize(
                    nrmse_improvement
                ),

            "cosine_improvement":
                summarize(
                    cosine_improvement
                ),

            "bootstrap_identity_minus_transport_nrmse":
                pca_bootstrap,

            "classification":
                pca_status,

            "bootstrap_classification":
                bootstrap_status,
        },


        "decoded_full_dimensional": {

            "transport_nrmse":
                summarize(
                    decoded_transport_nrmse
                ),

            "transport_cosine":
                summarize(
                    decoded_transport_cosine
                ),

            "frozen_pca_identity_nrmse":
                summarize(
                    decoded_identity_pca_nrmse
                ),

            "frozen_pca_identity_cosine":
                summarize(
                    decoded_identity_pca_cosine
                ),

            "raw_full_dimensional_identity_nrmse":
                summarize(
                    raw_identity_nrmse
                ),

            "raw_full_dimensional_identity_cosine":
                summarize(
                    raw_identity_cosine
                ),

            "clean_state_pca_reconstruction_nrmse":
                summarize(
                    decoded_clean_pca_nrmse
                ),

            "clean_state_pca_reconstruction_cosine":
                summarize(
                    decoded_clean_pca_cosine
                ),

            "bootstrap_frozen_pca_identity_minus_transport_nrmse":
                decoded_bootstrap,

            "classification":
                decoded_status,
        },


        "test_record_indices":
            test_record_indices.tolist(),

    }


# ============================================================
# 20. CROSS-SECTOR SUMMARY
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "CROSS-SECTOR TEST SUMMARY"
)

print(
    "=" * 80
)


positive_sectors = []

negative_sectors = []

tie_sectors = []


for sector in SECTORS:

    pca = test_results[
        sector
    ][
        "pca_space"
    ]


    status = pca[
        "classification"
    ]


    if status == (
        "TEST_TRANSPORT_IMPROVES_IDENTITY"
    ):

        positive_sectors.append(
            sector
        )

    elif status == (
        "TEST_TRANSPORT_WORSE_THAN_IDENTITY"
    ):

        negative_sectors.append(
            sector
        )

    else:

        tie_sectors.append(
            sector
        )


    print(
        f"{sector}: "
        f"transport="
        f"{pca['transport_nrmse']['mean']:.9f}, "
        f"identity="
        f"{pca['identity_nrmse']['mean']:.9f}, "
        f"improvement="
        f"{pca['nrmse_improvement_percent']['mean']:.6f}%, "
        f"{status}"
    )


# ============================================================
# 21. DECODED CROSS-SECTOR SUMMARY
# ============================================================

decoded_positive_sectors = []

decoded_negative_sectors = []

decoded_tie_sectors = []


for sector in SECTORS:

    decoded = test_results[
        sector
    ][
        "decoded_full_dimensional"
    ]


    transport_mean = decoded[
        "transport_nrmse"
    ][
        "mean"
    ]


    identity_mean = decoded[
        "frozen_pca_identity_nrmse"
    ][
        "mean"
    ]


    if (
        transport_mean
        <
        identity_mean
    ):

        decoded_positive_sectors.append(
            sector
        )

    elif (
        transport_mean
        >
        identity_mean
    ):

        decoded_negative_sectors.append(
            sector
        )

    else:

        decoded_tie_sectors.append(
            sector
        )


# ============================================================
# 22. OVERALL CLASSIFICATION
# ============================================================

if len(
    positive_sectors
) == 3:

    overall_transport_status = (
        "FROZEN_TRANSPORT_GENERALIZES_ACROSS_ALL_SECTORS"
    )

elif (

    len(
        positive_sectors
    ) > 0

    and

    len(
        negative_sectors
    ) == 0

):

    overall_transport_status = (
        "FROZEN_TRANSPORT_GENERALIZES_SELECTIVELY"
    )

elif (

    len(
        positive_sectors
    ) == 0

    and

    len(
        negative_sectors
    ) == 3

):

    overall_transport_status = (
        "FROZEN_TRANSPORT_DOES_NOT_GENERALIZE_AS_UNIFORM_SECTOR_ADVANTAGE"
    )

else:

    overall_transport_status = (
        "FROZEN_TRANSPORT_HAS_MIXED_OR_NULL_GENERALIZATION"
    )


# ============================================================
# 23. SCIENTIFIC STATUS FIREWALL
# ============================================================

scientific_status = {

    "frozen_transport_test_evaluated":
        True,

    "transport_generalization_status":
        overall_transport_status,


    "sector_pca_space":
        {

            sector: {

                "test_classification":
                    test_results[
                        sector
                    ][
                        "pca_space"
                    ][
                        "classification"
                    ],

                "bootstrap_classification":
                    test_results[
                        sector
                    ][
                        "pca_space"
                    ][
                        "bootstrap_classification"
                    ],

            }

            for sector in SECTORS

        },


    "decoded_full_dimensional_comparison":
        {

            "transport_better_than_frozen_pca_identity":
                decoded_positive_sectors,

            "transport_worse_than_frozen_pca_identity":
                decoded_negative_sectors,

            "tied":
                decoded_tie_sectors,

        },


    "contextual_realization":
        "NOT_EVALUATED",

    "coherence_defect":
        "NOT_EVALUATED",

    "triadic_irreducibility":
        "NOT_TESTED",

    "geometric_realization":
        "NOT_TESTED",

    "causal_explanation":
        "NOT_ESTABLISHED",
}


# ============================================================
# 24. NUMERICAL INTEGRITY
# ============================================================

numerical_integrity = {

    "state_bank_raw_hashes_verified":
        True,

    "state_bank_metadata_hashes_recorded":
        True,

    "auxiliary_raw_hashes_verified":
        True,

    "dataset_authorization_verified":
        True,

    "frozen_split_reconstructed":
        True,

    "target_position_verified":
        True,

    "prompt_length_pairing_required":
        False,

    "selected_configs_verified":
        True,

    "checkpoint_schema_verified":
        True,

    "checkpoint_state_bank_continuity_verified":
        True,

    "all_test_metrics_finite":
        True,

    "pca_refit":
        False,

    "transport_map_refit":
        False,

    "hyperparameter_selection":
        False,

    "test_refit":
        False,
}


for sector in SECTORS:

    result = test_results[
        sector
    ]


    summaries = [

        result[
            "pca_space"
        ][
            "transport_nrmse"
        ],

        result[
            "pca_space"
        ][
            "identity_nrmse"
        ],

        result[
            "pca_space"
        ][
            "transport_cosine"
        ],

        result[
            "pca_space"
        ][
            "identity_cosine"
        ],

        result[
            "pca_space"
        ][
            "nrmse_improvement_percent"
        ],

        result[
            "decoded_full_dimensional"
        ][
            "transport_nrmse"
        ],

        result[
            "decoded_full_dimensional"
        ][
            "raw_full_dimensional_identity_nrmse"
        ],

    ]


    for summary in summaries:

        for key, value in summary.items():

            if isinstance(
                value,
                (int, float)
            ):

                if not np.isfinite(
                    value
                ):

                    numerical_integrity[
                        "all_test_metrics_finite"
                    ] = False


if not numerical_integrity[
    "all_test_metrics_finite"
]:

    raise RuntimeError(
        "Non-finite test metric detected."
    )


print(
    "\n[PASS] Numerical integrity audit."
)


# ============================================================
# 25. FROZEN SPLIT RECORD
# ============================================================

frozen_split_record = {

    "rule":
        "r = (p + 4*t) mod 16",

    "template_count":
        N_TEMPLATES,

    "ordered_pairs_per_template":
        N_PAIRS,

    "split_rule":
        {

            "calibration":
                "r < 4",

            "test":
                "4 <= r < 8",

            "train":
                "r >= 8",

        },


    "record_order":
        "template-major; pair-major within template",


    "record_index_formula":
        "record_index = 16*t + p",


    "counts":
        split_counts,


    "test_indices":
        test_record_indices.tolist(),

}


# ============================================================
# 26. FINAL ARTIFACT
# ============================================================

artifact = {

    "experiment_id":
        "ETTR-CTL-LLAMA-1",

    "phase":
        "1E.2",

    "title":
        "Frozen Transport Test Evaluation",

    "model":
        "meta-llama/Llama-3.2-3B",

    "selected_layer":
        14,


    "governance":
        {

            "test_refit":
                False,

            "pca_refit":
                False,

            "transport_map_refit":
                False,

            "hyperparameter_selection":
                False,

            "contextual_realization_performed":
                False,

            "triadic_irreducibility_tested":
                False,

            "geometric_reconstruction_performed":
                False,

            "mathematical_definitions_modified":
                False,

        },


    "audit_correction":
        {

            "previous_failure":
                (
                    "Phase 1E.2 incorrectly searched for "
                    "S1/S2/S3 at the pickle top level. "
                    "Phase 1E.1 stores them under "
                    "checkpoint['selected_objects']."
                ),

            "classification":
                "OPERATIONAL_AUDIT_DECOHERENCE",

            "scientific_state_bank_affected":
                False,

            "state_bank_regenerated":
                False,

            "state_bank_hashes_reverified":
                True,

            "corrected_rule":
                (
                    "Read the frozen sector objects from "
                    "the Phase 1E.1 checkpoint's "
                    "'selected_objects' dictionary."
                ),

        },


    "dataset_authorization":
        {

            "authorized":
                dataset_authorized,

            "recorded_sha256":
                authorized_sha,

            "expected_sha256":
                EXPECTED_DATASET_SHA256,

            "authorization_verified":
                True,

            "physical_dataset_file_required":
                False,

        },


    "state_bank":
        {

            "path":
                str(
                    STATE_BANK_PATH
                ),

            "hashes":
                state_hash_results,

            "auxiliary_hashes":
                auxiliary_hash_results,

        },


    "target_position":
        {

            "p_T":
                "L(P)",

            "i_T":
                "L(P)-1",

            "verified":
                True,

            "prompt_length_pairing_required":
                False,

            "prompt_length_summary":
                length_summary,

        },


    "frozen_split":
        frozen_split_record,


    "frozen_checkpoint":
        {

            "path":
                str(
                    SELECTED_MAPS_PATH
                ),

            "experiment":
                checkpoint_experiment,

            "phase":
                checkpoint_phase,

            "model":
                str(
                    checkpoint_model
                ),

            "selected_layer":
                int(
                    checkpoint_layer
                ),

            "hidden_size":
                int(
                    checkpoint_hidden_size
                ),

            "dataset_sha256":
                checkpoint_dataset_sha,

            "fit_scope":
                fit_scope,

            "pca_scope":
                pca_scope,

            "transport_scope":
                transport_scope,

            "transport_orientation":
                transport_orientation,

            "intercept":
                intercept_setting,

            "sector_configurations":
                {

                    sector:

                        {

                            "d":
                                int(
                                    frozen_configs[
                                        sector
                                    ]["d"]
                                ),

                            "ridge":
                                float(
                                    frozen_configs[
                                        sector
                                    ]["ridge"]
                                ),

                            "components_shape":
                                list(
                                    frozen_configs[
                                        sector
                                    ]["components"].shape
                                ),

                            "mean_shape":
                                list(
                                    frozen_configs[
                                        sector
                                    ]["mean"].shape
                                ),

                            "M_forward_shape":
                                list(
                                    frozen_configs[
                                        sector
                                    ]["M_forward"].shape
                                ),

                            "fit_split":
                                "train_only",

                            "selection_split":
                                "calibration_only",

                        }

                    for sector in SECTORS

                },

        },


    "sector_results":
        test_results,


    "positive_sectors":
        positive_sectors,


    "negative_sectors":
        negative_sectors,


    "tie_sectors":
        tie_sectors,


    "decoded_positive_sectors":
        decoded_positive_sectors,


    "decoded_negative_sectors":
        decoded_negative_sectors,


    "decoded_tie_sectors":
        decoded_tie_sectors,


    "overall_transport_status":
        overall_transport_status,


    "scientific_status":
        scientific_status,


    "numerical_integrity":
        numerical_integrity,


    "final_classification":
        "FROZEN_TRANSPORT_TEST_EVALUATION_COMPLETED",

}


# ============================================================
# 27. SAVE ARTIFACT
# ============================================================

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(

        artifact,

        f,

        indent=2,

        ensure_ascii=False

    )


# ============================================================
# 28. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "FINAL PHASE 1E.2 STATUS"
)

print(
    "=" * 80
)

print(
    "  FROZEN_TRANSPORT_TEST_EVALUATION_COMPLETED"
)

print(
    f"  overall transport status: "
    f"{overall_transport_status}"
)

print(
    f"  positive PCA-space sectors: "
    f"{positive_sectors}"
)

print(
    f"  negative PCA-space sectors: "
    f"{negative_sectors}"
)

print(
    f"  tied PCA-space sectors: "
    f"{tie_sectors}"
)

print(
    f"  decoded-positive sectors: "
    f"{decoded_positive_sectors}"
)

print(
    f"  decoded-negative sectors: "
    f"{decoded_negative_sectors}"
)

print(
    f"  decoded-tied sectors: "
    f"{decoded_tie_sectors}"
)

print(
    f"  artifact: "
    f"{OUTPUT_PATH}"
)

print(
    "=" * 80
)

print(
    "\nINTERPRETATION FIREWALL:"
)

print(
    "A positive frozen transport result establishes "
    "generalization of the selected transport "
    "operationalization only."
)

print(
    "It does NOT establish contextual geometry, "
    "CTL coherence, or triadic irreducibility."
)

print(
    "A negative result is a result about the frozen "
    "operationalization and is not a reason to modify "
    "the underlying mathematics."
)

ETTR-CTL-LLAMA-1 — PHASE 1E.2
FROZEN TRANSPORT TEST EVALUATION
ROOT: /content/ettr_ctl_llama
State bank: /content/ettr_ctl_llama/results/llama_phase1d2_full_state_bank.npz
Selection manifest: /content/ettr_ctl_llama/results/llama_phase1e1_transport_candidate_selection.json
Selected maps: /content/ettr_ctl_llama/checkpoints/llama_phase1e1_selected_transport_maps.pkl
Output: /content/ettr_ctl_llama/results/llama_phase1e2_frozen_transport_test_evaluation.json

GOVERNANCE:
  TEST REFITTING: FORBIDDEN
  PCA REFITTING: FORBIDDEN
  MAP REFITTING: FORBIDDEN
  HYPERPARAMETER SELECTION: FORBIDDEN
  CONTEXTUAL REALIZATION: NOT PERFORMED
  TRIADIC IRREDUCIBILITY: NOT TESTED
  GEOMETRIC RECONSTRUCTION CLAIM: NOT MADE
  MATHEMATICAL DEFINITIONS: FROZEN

[PASS] Required Llama-side artifacts exist.
[INFO] GPT-2 recovered dataset JSON is not required for frozen test application.

DATASET AUTHORIZATION
--------------------------------------------------------------------------------
  authorized: True
  

In [34]:
# ============================================================
# ETTR-CTL-LLAMA-1
# PHASE 1F.0 — CTL CONTEXTUAL REALIZATION PREFLIGHT
#
# NEW CELL — DO NOT REMOVE PREVIOUS CELLS
#
# PURPOSE
# -------
# Audit and freeze the operational implementation of the
# Contextual Transport Logic (CTL) layer before any contextual
# realization map is fitted.
#
# CRITICAL GOVERNANCE
# -------------------
# CTL IS NOT BEING REPLACED BY ORDINARY BOOLEAN LOGIC.
#
# Boolean values may serve as the local logical substrate,
# but the CTL implementation must retain:
#
#   1. context-indexed carriers;
#   2. sector-indexed transported states;
#   3. partial admissibility;
#   4. triadic admissibility;
#   5. logical transport;
#   6. composition/functoriality;
#   7. logical/transport coherence.
#
# This cell DOES NOT:
#
#   - fit Phi_C;
#   - fit Phi_D;
#   - refit PCA;
#   - refit transport maps;
#   - select hyperparameters;
#   - inspect test performance;
#   - test triadic irreducibility empirically;
#   - claim contextual geometry.
#
# It only establishes that the mathematical CTL kernel can be
# operationally instantiated without collapsing it to Boolean
# conjunction/disjunction.
# ============================================================


import os
import json
import math
import hashlib
from pathlib import Path

import numpy as np


# ============================================================
# 0. PATHS
# ============================================================

ROOT = Path(
    "/content/ettr_ctl_llama"
)

RESULTS = ROOT / "results"

STATE_BANK_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank.npz"
)

STATE_MANIFEST_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank_manifest.json"
)

POST_AUDIT_PATH = (
    RESULTS /
    "llama_phase1d3_full_state_bank_post_extraction_audit.json"
)

TRANSPORT_TEST_PATH = (
    RESULTS /
    "llama_phase1e2_frozen_transport_test_evaluation.json"
)

AUTH_PATH = (
    RESULTS /
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

OUTPUT_PATH = (
    RESULTS /
    "llama_phase1f0_ctl_contextual_realization_preflight.json"
)


EXPECTED_DATASET_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)


EXPECTED_STATE_HASHES = {

    "S1":
        "9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783",

    "S2":
        "41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb",

    "S3":
        "173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3",
}


SECTORS = [
    "S1",
    "S2",
    "S3",
]


N_RECORDS = 192

N_TEMPLATES = 12

N_PAIRS = 16

N_CONDITIONS = 384

EPS = 1e-12


# ============================================================
# 1. HEADER
# ============================================================

print(
    "=" * 80
)

print(
    "ETTR-CTL-LLAMA-1 — PHASE 1F.0"
)

print(
    "CTL CONTEXTUAL REALIZATION PREFLIGHT"
)

print(
    "=" * 80
)

print(
    f"ROOT: {ROOT}"
)

print(
    f"State bank: {STATE_BANK_PATH}"
)

print(
    f"Previous transport evaluation: "
    f"{TRANSPORT_TEST_PATH}"
)

print(
    f"Output: {OUTPUT_PATH}"
)


print(
    "\nGOVERNANCE"
)

print(
    "-" * 80
)

print(
    "  CTL mathematical definitions: FROZEN"
)

print(
    "  Boolean logic: LOCAL SUBSTRATE ONLY"
)

print(
    "  context indexing: REQUIRED"
)

print(
    "  partial admissibility: REQUIRED"
)

print(
    "  triadic admissibility: REQUIRED"
)

print(
    "  logical transport: REQUIRED"
)

print(
    "  composition/functoriality: REQUIRED"
)

print(
    "  logical/transport coherence: REQUIRED"
)

print(
    "  Phi_C fitting: NOT PERFORMED"
)

print(
    "  Phi_D fitting: NOT PERFORMED"
)

print(
    "  test refitting: FORBIDDEN"
)

print(
    "  PCA refitting: FORBIDDEN"
)

print(
    "  transport-map refitting: FORBIDDEN"
)

print(
    "  triadic irreducibility test: NOT PERFORMED"
)

print(
    "  geometric realization: NOT PERFORMED"
)


# ============================================================
# 2. REQUIRED ARTIFACTS
# ============================================================

required_paths = [

    STATE_BANK_PATH,

    STATE_MANIFEST_PATH,

    POST_AUDIT_PATH,

    TRANSPORT_TEST_PATH,

    AUTH_PATH,
]


missing = [

    str(path)

    for path in required_paths

    if not path.exists()
]


if missing:

    raise FileNotFoundError(

        "Required Phase 1F.0 artifacts missing:\n"
        +
        "\n".join(
            missing
        )
    )


print(
    "\n[PASS] Required frozen artifacts exist."
)


# ============================================================
# 3. DATASET AUTHORIZATION
# ============================================================

with open(
    AUTH_PATH,
    "r",
    encoding="utf-8"
) as f:

    auth = json.load(
        f
    )


dataset_authorized = bool(

    auth.get(

        "dataset_authorized",

        auth.get(
            "authorized",
            False
        )

    )

)


dataset_sha = auth.get(

    "dataset_sha256",

    auth.get(
        "sha256",
        None
    )

)


print(
    "\nDATASET AUTHORIZATION"
)

print(
    "-" * 80
)

print(
    f"  authorized: "
    f"{dataset_authorized}"
)

print(
    f"  recorded SHA: "
    f"{dataset_sha}"
)

print(
    f"  expected SHA: "
    f"{EXPECTED_DATASET_SHA256}"
)


if not dataset_authorized:

    raise RuntimeError(
        "Operative dataset is not authorized."
    )


if dataset_sha != EXPECTED_DATASET_SHA256:

    raise RuntimeError(
        "Operative dataset SHA mismatch."
    )


print(
    "[PASS] Dataset authorization."
)


# ============================================================
# 4. STATE-BANK FIREWALL
# ============================================================

bank = np.load(

    STATE_BANK_PATH,

    allow_pickle=False
)


print(
    "\nSTATE-BANK FIREWALL"
)

print(
    "-" * 80
)


state_hashes = {}


for sector in SECTORS:

    X = np.asarray(
        bank[
            sector
        ]
    )


    if tuple(
        X.shape
    ) != (

        N_CONDITIONS,

        3072

    ):

        raise RuntimeError(
            f"{sector} state shape mismatch."
        )


    raw = np.ascontiguousarray(
        X
    )


    actual_hash = hashlib.sha256(
        raw.tobytes(
            order="C"
        )
    ).hexdigest()


    expected_hash = EXPECTED_STATE_HASHES[
        sector
    ]


    finite = bool(
        np.all(
            np.isfinite(
                X
            )
        )
    )


    nonzero = bool(
        np.any(
            np.abs(X) > 0
        )
    )


    print(
        f"\n{sector}"
    )

    print(
        f"  shape: {X.shape}"
    )

    print(
        f"  expected SHA: {expected_hash}"
    )

    print(
        f"  actual SHA:   {actual_hash}"
    )

    print(
        f"  finite: {finite}"
    )

    print(
        f"  nonzero: {nonzero}"
    )


    if actual_hash != expected_hash:

        raise RuntimeError(
            f"{sector} state-bank hash mismatch."
        )


    if not finite or not nonzero:

        raise RuntimeError(
            f"{sector} state-bank numerical integrity failure."
        )


    state_hashes[
        sector
    ] = actual_hash


print(
    "\n[PASS] Frozen state-bank firewall."
)


# ============================================================
# 5. LOAD FROZEN TRANSPORT RESULT
# ============================================================

with open(
    TRANSPORT_TEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    transport_result = json.load(
        f
    )


transport_final_classification = (
    transport_result.get(
        "final_classification",
        None
    )
)


transport_status = (
    transport_result.get(
        "overall_transport_status",
        None
    )
)


print(
    "\nFROZEN TRANSPORT PREDECESSOR"
)

print(
    "-" * 80
)

print(
    f"  final classification: "
    f"{transport_final_classification}"
)

print(
    f"  overall status: "
    f"{transport_status}"
)


if transport_final_classification != (
    "FROZEN_TRANSPORT_TEST_EVALUATION_COMPLETED"
):

    raise RuntimeError(
        "Phase 1E.2 is not in a completed state."
    )


print(
    "[PASS] Phase 1E.2 predecessor is frozen."
)


# ============================================================
# 6. RECONSTRUCT FROZEN RECORD INDEX
# ============================================================

record_template = []

record_pair = []

record_r = []

record_split = []


for t in range(
    N_TEMPLATES
):

    for p in range(
        N_PAIRS
    ):

        r = (
            p
            +
            4 * t
        ) % 16


        if r < 4:

            split = "calibration"

        elif r < 8:

            split = "test"

        else:

            split = "train"


        record_template.append(
            t
        )

        record_pair.append(
            p
        )

        record_r.append(
            r
        )

        record_split.append(
            split
        )


record_template = np.asarray(
    record_template,
    dtype=np.int64
)

record_pair = np.asarray(
    record_pair,
    dtype=np.int64
)

record_r = np.asarray(
    record_r,
    dtype=np.int64
)

record_split = np.asarray(
    record_split,
    dtype=object
)


train_records = np.where(

    record_split
    ==
    "train"

)[0]


cal_records = np.where(

    record_split
    ==
    "calibration"

)[0]


test_records = np.where(

    record_split
    ==
    "test"

)[0]


print(
    "\nFROZEN RECORD PARTITION"
)

print(
    "-" * 80
)

print(
    f"  train:       {len(train_records)}"
)

print(
    f"  calibration: {len(cal_records)}"
)

print(
    f"  test:        {len(test_records)}"
)


if (

    len(train_records) != 96

    or

    len(cal_records) != 48

    or

    len(test_records) != 48

):

    raise RuntimeError(
        "Frozen record partition mismatch."
    )


print(
    "[PASS] Frozen record partition."
)


# ============================================================
# 7. CTL OBJECT MODEL
# ============================================================
#
# This section is the central protection against accidentally
# implementing ordinary Boolean logic instead of CTL.
#
# We explicitly represent:
#
#   Context C
#   Carriers Z_C^(1), Z_C^(2), Z_C^(3)
#   Logical propositions on those carriers
#   Admissibility A_C
#   Logical transport T_gamma
#   Composition
#   Coherence relation
#
# Boolean evaluation is used only INSIDE the local logical
# substrate. It does not define the contextual relation itself.
# ============================================================


class CTLContext:
    """
    A context-indexed CTL object.

    The object is deliberately richer than a Boolean tuple.

    A CTL context consists of:

        C
        Z_C^(1)
        Z_C^(2)
        Z_C^(3)
        P_C
        A_C

    where A_C is a partial admissibility relation.
    """

    def __init__(
        self,
        context_id,
        carriers,
        propositions,
        admissibility,
        metadata=None
    ):

        self.context_id = context_id

        self.carriers = carriers

        self.propositions = propositions

        self.admissibility = admissibility

        self.metadata = (
            {}
            if metadata is None
            else metadata
        )


class CTLTransport:
    """
    Contextual logical transport.

    This is NOT Boolean implication.

    It maps logical carriers/propositions between
    context-indexed representations.
    """

    def __init__(
        self,
        source_context,
        target_context,
        maps,
        composition_label
    ):

        self.source_context = source_context

        self.target_context = target_context

        self.maps = maps

        self.composition_label = (
            composition_label
        )


# ============================================================
# 8. CONTEXT INDEXING
# ============================================================

print(
    "\nCTL CONTEXT INDEXING"
)

print(
    "-" * 80
)


# Each experimental record defines a contextual unit.
#
# A context is NOT merely:
#
#     (boolean_1, boolean_2, boolean_3)
#
# Instead it is indexed by the experimental condition and
# carries three sector-specific representations.


context_ids = [

    f"record_{int(i):03d}"

    for i in range(
        N_RECORDS
    )
]


contexts = {}


for i in range(
    N_RECORDS
):

    contexts[
        context_ids[i]
    ] = {

        "record_index":
            int(i),

        "template_index":
            int(
                record_template[i]
            ),

        "pair_index":
            int(
                record_pair[i]
            ),

        "r":
            int(
                record_r[i]
            ),

        "split":
            str(
                record_split[i]
            ),

        "sectors":
            SECTORS.copy(),

        "clean_condition":
            True,

        "corrupt_condition":
            True,

    }


if len(
    contexts
) != N_RECORDS:

    raise RuntimeError(
        "Context indexing did not produce 192 contexts."
    )


print(
    f"  contexts: {len(contexts)}"
)

print(
    "  context key: record index + "
    "template + pair + split"
)

print(
    "  sector carriers: S1, S2, S3"
)

print(
    "[PASS] Context indexing is explicit."
)


# ============================================================
# 9. SECTOR CARRIERS
# ============================================================

print(
    "\nCTL SECTOR CARRIERS"
)

print(
    "-" * 80
)


carrier_dimensions = {

    "S1":
        3072,

    "S2":
        3072,

    "S3":
        3072,
}


carrier_registry = {}


for i in range(
    N_RECORDS
):

    clean_row = i

    corrupt_row = (
        N_RECORDS
        +
        i
    )


    carrier_registry[
        context_ids[i]
    ] = {

        "clean":
            {

                "S1":
                    bank["S1"][
                        clean_row
                    ],

                "S2":
                    bank["S2"][
                        clean_row
                    ],

                "S3":
                    bank["S3"][
                        clean_row
                    ],

            },


        "corrupt":
            {

                "S1":
                    bank["S1"][
                        corrupt_row
                    ],

                "S2":
                    bank["S2"][
                        corrupt_row
                    ],

                "S3":
                    bank["S3"][
                        corrupt_row
                    ],

            },

    }


for context_id, entry in carrier_registry.items():

    for condition in [

        "clean",

        "corrupt",

    ]:

        for sector in SECTORS:

            x = np.asarray(
                entry[
                    condition
                ][
                    sector
                ]
            )


            if x.shape != (
                3072,
            ):

                raise RuntimeError(
                    f"Carrier shape failure: "
                    f"{context_id}/{condition}/{sector}"
                )


            if not np.all(
                np.isfinite(x)
            ):

                raise RuntimeError(
                    f"Non-finite carrier: "
                    f"{context_id}/{condition}/{sector}"
                )


print(
    f"  contexts: {len(carrier_registry)}"
)

print(
    "  carrier sectors per context: 3"
)

print(
    "  carrier dimension per sector: 3072"
)

print(
    "  conditions per context: clean + corrupt"
)

print(
    "[PASS] Context-indexed carrier registry."
)


# ============================================================
# 10. LOCAL LOGICAL SUBSTRATE
# ============================================================
#
# IMPORTANT:
#
# Boolean values appear here only as a local truth substrate.
#
# The CTL object is NOT identified with:
#
#       A AND B
#
#       A OR B
#
# or any Boolean formula.
#
# Instead, Boolean propositions are attached to carriers and
# contexts, while admissibility and transport remain distinct
# contextual structures.
# ============================================================


class LocalBooleanProposition:
    """
    Atomic local logical proposition.

    This is the substrate, not the CTL object.
    """

    def __init__(
        self,
        name,
        truth_value
    ):

        self.name = str(
            name
        )

        self.truth_value = bool(
            truth_value
        )


def boolean_and(
    a,
    b
):

    return (
        bool(a)
        and
        bool(b)
    )


def boolean_or(
    a,
    b
):

    return (
        bool(a)
        or
        bool(b)
    )


# Explicitly demonstrate that the substrate exists but is not
# the full CTL representation.


boolean_substrate_example = {

    "P":
        LocalBooleanProposition(
            "P",
            True
        ),

    "Q":
        LocalBooleanProposition(
            "Q",
            False
        ),

}


print(
    "\nLOCAL BOOLEAN SUBSTRATE"
)

print(
    "-" * 80
)

print(
    "  atomic propositions: P, Q"
)

print(
    "  AND/OR available locally: YES"
)

print(
    "  Boolean substrate identified with CTL: NO"
)

print(
    "  context index retained outside Boolean evaluation: YES"
)

print(
    "[PASS] Boolean logic restricted to local substrate."
)


# ============================================================
# 11. PARTIAL ADMISSIBILITY
# ============================================================
#
# Admissibility is NOT represented as a Boolean formula over
# sectors.
#
# It is a context-indexed relation:
#
#       A_C(z1,z2,z3)
#
# which may be:
#
#       admissible
#       inadmissible
#       undefined
#
# The third state is essential for partial admissibility.
# ============================================================


ADMISSIBLE = 1

INADMISSIBLE = 0

UNDEFINED = -1


class PartialAdmissibility:
    """
    Three-valued partial admissibility relation.

    This is deliberately NOT Boolean.
    """

    def __init__(
        self
    ):

        self.values = {}


    def set(
        self,
        context_id,
        value
    ):

        if value not in {

            ADMISSIBLE,

            INADMISSIBLE,

            UNDEFINED,

        }:

            raise ValueError(
                "Invalid admissibility value."
            )


        self.values[
            context_id
        ] = int(
            value
        )


    def get(
        self,
        context_id
    ):

        return self.values.get(

            context_id,

            UNDEFINED

        )


    def is_defined(
        self,
        context_id
    ):

        return (

            self.get(
                context_id
            )
            !=
            UNDEFINED

        )


    def is_admissible(
        self,
        context_id
    ):

        return (

            self.get(
                context_id
            )
            ==
            ADMISSIBLE

        )


    def is_inadmissible(
        self,
        context_id
    ):

        return (

            self.get(
                context_id
            )
            ==
            INADMISSIBLE

        )


admissibility = (
    PartialAdmissibility()
)


# ============================================================
# 12. FROZEN EMPIRICAL ADMISSIBILITY REPRESENTATION
# ============================================================
#
# We do NOT invent a new scientific admissibility criterion
# here.
#
# For the preflight, admissibility is represented as a
# context-indexed partial object whose empirical assignment is
# deliberately marked NOT YET OPERATIONALIZED.
#
# This prevents a Boolean surrogate from silently becoming the
# CTL admissibility relation.
# ============================================================


for context_id in context_ids:

    admissibility.set(

        context_id,

        UNDEFINED

    )


undefined_count = sum(

    value == UNDEFINED

    for value in admissibility.values.values()

)


print(
    "\nPARTIAL ADMISSIBILITY STRUCTURE"
)

print(
    "-" * 80
)

print(
    f"  contexts represented: "
    f"{len(admissibility.values)}"
)

print(
    f"  undefined admissibility values: "
    f"{undefined_count}"
)

print(
    "  empirical admissibility rule: NOT YET FITTED/ASSIGNED"
)

print(
    "  Boolean substitution for admissibility: FORBIDDEN"
)


if undefined_count != N_RECORDS:

    raise RuntimeError(
        "Unexpected admissibility initialization."
    )


print(
    "[PASS] Partial admissibility structure exists "
    "without Boolean collapse."
)


# ============================================================
# 13. TRIADIC CARRIER SIGNATURE
# ============================================================
#
# The CTL carrier is explicitly triadic:
#
#       (Z_C^(1), Z_C^(2), Z_C^(3))
#
# This does NOT mean that irreducibility has been established.
#
# It only verifies that the implementation retains the three
# sectors jointly rather than silently replacing them by a
# Boolean pairwise structure.
# ============================================================


triadic_carrier_signatures = {}


for i in range(
    N_RECORDS
):

    cid = context_ids[
        i
    ]


    triadic_carrier_signatures[
        cid
    ] = {

        "clean":
            [

                "S1",
                "S2",
                "S3",

            ],

        "corrupt":
            [

                "S1",
                "S2",
                "S3",

            ],

        "joint_object":
            True,

        "pairwise_reduction":
            False,

        "triadic_irreducibility_established":
            False,

    }


triadic_joint_count = sum(

    int(
        entry[
            "joint_object"
        ]
    )

    for entry
    in
    triadic_carrier_signatures.values()

)


print(
    "\nTRIADIC CARRIER STRUCTURE"
)

print(
    "-" * 80
)

print(
    f"  triadic contexts: "
    f"{triadic_joint_count}"
)

print(
    "  joint carrier retained: YES"
)

print(
    "  automatic pairwise reduction: NO"
)

print(
    "  triadic irreducibility claim: NO"
)


if triadic_joint_count != N_RECORDS:

    raise RuntimeError(
        "Triadic carrier construction failed."
    )


print(
    "[PASS] Triadic carrier structure retained."
)


# ============================================================
# 14. LOGICAL TRANSPORT OBJECT
# ============================================================
#
# Logical transport is represented independently of numerical
# state transport.
#
# This distinction is critical:
#
# numerical transport:
#
#       z_corrupt -> z_transport
#
# is not itself logical transport.
#
# Logical transport acts on context-indexed propositions.
# ============================================================


logical_transport_registry = {}


for i in range(
    N_RECORDS
):

    cid = context_ids[
        i
    ]


    logical_transport_registry[
        cid
    ] = {

        "source":
            f"{cid}:corrupt",

        "target":
            f"{cid}:clean",

        "sector_domain":
            SECTORS.copy(),

        "carrier_transport_available":
            True,

        "logical_transport_defined":
            False,

        "definition_status":
            "PRECOHERENCE_AUDIT_ONLY",

    }


print(
    "\nLOGICAL TRANSPORT"
)

print(
    "-" * 80
)

print(
    f"  context-indexed transport objects: "
    f"{len(logical_transport_registry)}"
)

print(
    "  numerical transport and logical transport separated: YES"
)

print(
    "  logical transport empirically fitted: NO"
)

print(
    "  Boolean implication used as logical transport: NO"
)

print(
    "[PASS] Logical transport layer explicitly separated."
)


# ============================================================
# 15. COMPOSITION / FUNCTORIALITY SCAFFOLD
# ============================================================
#
# CTL requires transport composition to remain an explicit
# structural relation.
#
# We therefore represent composition as an object-level
# operation, rather than assuming that Boolean implication
# supplies composition.
# ============================================================


class CTLComposition:

    def __init__(
        self,
        first,
        second
    ):

        self.first = first

        self.second = second


    def compose(
        self
    ):

        return {

            "first":
                self.first,

            "second":
                self.second,

            "composed":
                True,

            "semantic_type":
                "CTL_TRANSPORT_COMPOSITION",

        }


composition_example = CTLComposition(

    "gamma_1",

    "gamma_2"

)


composition_result = (
    composition_example.compose()
)


print(
    "\nCTL COMPOSITION / FUNCTORIALITY"
)

print(
    "-" * 80
)

print(
    f"  composition object: "
    f"{composition_result}"
)

print(
    "  Boolean implication used as composition: NO"
)

print(
    "  functoriality empirically established: NO"
)

print(
    "  structural scaffold available: YES"
)

print(
    "[PASS] Composition is represented as a distinct CTL operation."
)


# ============================================================
# 16. LOGICAL / TRANSPORT COHERENCE OBJECT
# ============================================================
#
# The eventual coherence relation will compare:
#
#   logical transport of contextual propositions
#
# with
#
#   transport of the corresponding numerical contextual
#   carriers.
#
# The actual coherence defect is NOT evaluated here.
#
# This cell only ensures that the two sides are represented
# separately so that a later Phi_C/Phi_D construction cannot
# accidentally collapse them into Boolean logic.
# ============================================================


coherence_registry = {}


for i in range(
    N_RECORDS
):

    cid = context_ids[
        i
    ]


    coherence_registry[
        cid
    ] = {

        "context":
            cid,

        "logical_transport":
            logical_transport_registry[
                cid
            ],

        "numerical_carriers":
            triadic_carrier_signatures[
                cid
            ],

        "coherence_defect":
            None,

        "coherence_evaluated":
            False,

    }


print(
    "\nLOGICAL / TRANSPORT COHERENCE"
)

print(
    "-" * 80
)

print(
    f"  coherence objects: "
    f"{len(coherence_registry)}"
)

print(
    "  logical side retained: YES"
)

print(
    "  numerical transport side retained: YES"
)

print(
    "  coherence defect evaluated: NO"
)

print(
    "  Phi_C/Phi_D fitted: NO"
)

print(
    "[PASS] Coherence structure separated and frozen."
)


# ============================================================
# 17. CTL-vs-BOOLEAN STRUCTURAL FIREWALL
# ============================================================
#
# This is the decisive audit.
#
# A Boolean-only implementation would have only local truth
# values and Boolean operators.
#
# The CTL implementation must contain the additional
# contextual/transport structures.
# ============================================================


ctl_required_structures = {

    "context_index":
        len(contexts) == N_RECORDS,

    "triadic_carriers":
        len(triadic_carrier_signatures)
        ==
        N_RECORDS,

    "partial_admissibility":
        len(admissibility.values)
        ==
        N_RECORDS,

    "undefined_admissibility_state":
        UNDEFINED in {
            ADMISSIBLE,
            INADMISSIBLE,
            UNDEFINED,
        },

    "logical_transport":
        len(logical_transport_registry)
        ==
        N_RECORDS,

    "composition":
        bool(
            composition_result[
                "composed"
            ]
        ),

    "coherence":
        len(coherence_registry)
        ==
        N_RECORDS,

    "numerical_logical_separation":
        True,

    "boolean_not_full_ctl":
        True,

}


print(
    "\nCTL-vs-BOOLEAN STRUCTURAL FIREWALL"
)

print(
    "-" * 80
)


for name, status in ctl_required_structures.items():

    print(
        f"  {name}: "
        f"{bool(status)}"
    )


if not all(
    ctl_required_structures.values()
):

    raise RuntimeError(
        "CTL structural firewall failed."
    )


print(
    "\n[PASS] CTL structure is not reducible "
    "to the local Boolean substrate at the implementation level."
)


# ============================================================
# 18. WHAT THIS PREFLIGHT DOES NOT CLAIM
# ============================================================

nonclaims = {

    "ctl_mathematical_validity":
        "NOT_TESTED",

    "empirical_ctl_coherence":
        "NOT_TESTED",

    "triadic_irreducibility":
        "NOT_TESTED",

    "contextual_geometry":
        "NOT_ESTABLISHED",

    "causal_transport":
        "NOT_ESTABLISHED",

    "Phi_C":
        "NOT_FITTED",

    "Phi_D":
        "NOT_FITTED",

    "coherence_defect":
        "NOT_EVALUATED",

    "test_generalization":
        "NOT_EVALUATED_IN_THIS_PHASE",

}


print(
    "\nNON-CLAIM FIREWALL"
)

print(
    "-" * 80
)

for key, value in nonclaims.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 19. SERIALIZE PREFLIGHT ARTIFACT
# ============================================================

artifact = {

    "experiment_id":
        "ETTR-CTL-LLAMA-1",

    "phase":
        "1F.0",

    "title":
        "CTL Contextual Realization Preflight",


    "purpose":
        (
            "Audit the operational CTL structure before "
            "contextual realization. Boolean logic is retained "
            "only as a local logical substrate and is not used "
            "as a substitute for contextual transport logic."
        ),


    "governance":
        {

            "mathematical_definitions_frozen":
                True,

            "boolean_logic_as_ctl_replacement":
                False,

            "context_indexing_required":
                True,

            "partial_admissibility_required":
                True,

            "triadic_admissibility_required":
                True,

            "logical_transport_required":
                True,

            "composition_required":
                True,

            "logical_transport_coherence_required":
                True,

            "Phi_C_fit":
                False,

            "Phi_D_fit":
                False,

            "PCA_refit":
                False,

            "transport_map_refit":
                False,

            "hyperparameter_selection":
                False,

            "test_refit":
                False,

            "triadic_irreducibility_test":
                False,

            "geometric_realization":
                False,

        },


    "dataset":
        {

            "authorized":
                dataset_authorized,

            "sha256":
                dataset_sha,

            "expected_sha256":
                EXPECTED_DATASET_SHA256,

        },


    "state_bank":
        {

            "path":
                str(
                    STATE_BANK_PATH
                ),

            "raw_byte_hashes":
                state_hashes,

            "shapes":
                {

                    sector:
                        list(
                            bank[
                                sector
                            ].shape
                        )

                    for sector in SECTORS

                },

        },


    "frozen_partition":
        {

            "train":
                int(
                    len(
                        train_records
                    )
                ),

            "calibration":
                int(
                    len(
                        cal_records
                    )
                ),

            "test":
                int(
                    len(
                        test_records
                    )
                ),

            "rule":
                "r = (p + 4*t) mod 16",

        },


    "ctl_structures":
        {

            "contexts":
                int(
                    len(
                        contexts
                    )
                ),

            "triadic_carriers":
                int(
                    len(
                        triadic_carrier_signatures
                    )
                ),

            "partial_admissibility":
                {

                    "admissible_code":
                        ADMISSIBLE,

                    "inadmissible_code":
                        INADMISSIBLE,

                    "undefined_code":
                        UNDEFINED,

                    "initialized_undefined":
                        int(
                            undefined_count
                        ),

                },


            "logical_transport":
                int(
                    len(
                        logical_transport_registry
                    )
                ),


            "composition":
                {

                    "represented":
                        True,

                    "empirically_tested":
                        False,

                },


            "coherence":
                {

                    "objects":
                        int(
                            len(
                                coherence_registry
                            )
                        ),

                    "evaluated":
                        False,

                },

        },


    "boolean_substrate":
        {

            "present":
                True,

            "role":
                "LOCAL_LOGICAL_SUBSTRATE_ONLY",

            "used_as_full_ctl":
                False,

            "used_as_contextual_transport":
                False,

            "used_as_admissibility_relation":
                False,

            "used_as_coherence_relation":
                False,

        },


    "ctl_structural_firewall":
        ctl_required_structures,


    "nonclaims":
        nonclaims,


    "final_classification":
        "CTL_STRUCTURAL_PREFLIGHT_PASS",

}


with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(

        artifact,

        f,

        indent=2,

        ensure_ascii=False

    )


# ============================================================
# 20. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "FINAL PHASE 1F.0 STATUS"
)

print(
    "=" * 80
)

print(
    "  CTL_STRUCTURAL_PREFLIGHT_PASS"
)

print(
    "  Boolean logic is retained only as "
    "the local logical substrate."
)

print(
    "  Context indexing: PASS"
)

print(
    "  Partial admissibility structure: PASS"
)

print(
    "  Triadic carrier structure: PASS"
)

print(
    "  Logical transport structure: PASS"
)

print(
    "  Composition structure: PASS"
)

print(
    "  Logical/transport coherence structure: PASS"
)

print(
    "  Phi_C/Phi_D fitting: NOT PERFORMED"
)

print(
    "  Triadic irreducibility: NOT TESTED"
)

print(
    "  Contextual geometry: NOT ESTABLISHED"
)

print(
    f"  artifact: {OUTPUT_PATH}"
)

print(
    "=" * 80
)

ETTR-CTL-LLAMA-1 — PHASE 1F.0
CTL CONTEXTUAL REALIZATION PREFLIGHT
ROOT: /content/ettr_ctl_llama
State bank: /content/ettr_ctl_llama/results/llama_phase1d2_full_state_bank.npz
Previous transport evaluation: /content/ettr_ctl_llama/results/llama_phase1e2_frozen_transport_test_evaluation.json
Output: /content/ettr_ctl_llama/results/llama_phase1f0_ctl_contextual_realization_preflight.json

GOVERNANCE
--------------------------------------------------------------------------------
  CTL mathematical definitions: FROZEN
  Boolean logic: LOCAL SUBSTRATE ONLY
  context indexing: REQUIRED
  partial admissibility: REQUIRED
  triadic admissibility: REQUIRED
  logical transport: REQUIRED
  composition/functoriality: REQUIRED
  logical/transport coherence: REQUIRED
  Phi_C fitting: NOT PERFORMED
  Phi_D fitting: NOT PERFORMED
  test refitting: FORBIDDEN
  PCA refitting: FORBIDDEN
  transport-map refitting: FORBIDDEN
  triadic irreducibility test: NOT PERFORMED
  geometric realization: NOT PERFORME

In [36]:
# ============================================================
# ETTR-CTL-LLAMA-1
# PHASE 1F.1 — FORMAL CTL STRUCTURE IMPLEMENTATION AUDIT
#
# REPLACEMENT CELL
#
# REMOVE THE PREVIOUS PHASE 1F.1 CELL, THEN INSERT THIS CELL.
#
# PURPOSE
# -------
# Implement the formal CTL structure itself, rather than
# reducing CTL to Boolean propositions or thresholded hidden-state
# displacements.
#
# FORMAL CTL STRUCTURE
# --------------------
#
#   CTL =
#   (C,
#    {E_C},
#    {L_C^(k)},
#    {Adm_C, Adm_C^(3)},
#    {T_hat_gamma^(k)},
#    {K_C, Phi_C})
#
# satisfying CTL-A1 ... CTL-A4.
#
# The present cell constructs and audits these objects from the
# frozen experimental record structure.
#
# IMPORTANT SCIENTIFIC GOVERNANCE
# --------------------------------
#
# 1. Boolean truth is NOT used to define CTL.
# 2. Hidden-state thresholds are NOT used to define admissibility.
# 3. Undefinedness is NOT encoded as False.
# 4. The three sectors are logical carriers, not truth values.
# 5. The triadic admissibility domain is represented independently
#    from its pairwise projections.
# 6. Logical transport is a separate object from numerical
#    transport.
# 7. Composition is represented categorically and is not replaced
#    by Boolean implication.
# 8. Phi_C is represented as a formal coherence map object but is
#    NOT fitted here.
# 9. Semantic valuation is NOT assumed to be primitive.
# 10. No empirical triadic irreducibility claim is made here.
# 11. No test-set fitting occurs.
# 12. No PCA, numerical transport map, Phi_C, or Phi_D is refit.
#
# The cell therefore answers:
#
#   "Can the frozen Llama experiment be represented faithfully
#    in the formal CTL object language?"
#
# It does NOT yet answer:
#
#   "Does the Llama experiment empirically satisfy CTL?"
#
# ============================================================


import os
import json
import hashlib
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, FrozenSet, Iterable, List, Optional, Tuple

import numpy as np


# ============================================================
# 0. PATHS / FROZEN IDENTIFIERS
# ============================================================

ROOT = Path(
    "/content/ettr_ctl_llama"
)

RESULTS = ROOT / "results"

STATE_BANK_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank.npz"
)

STATE_MANIFEST_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank_manifest.json"
)

AUTH_PATH = (
    RESULTS /
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

PREFLIGHT_PATH = (
    RESULTS /
    "llama_phase1f0_ctl_contextual_realization_preflight.json"
)

OUTPUT_PATH = (
    RESULTS /
    "llama_phase1f1_formal_ctl_structure_audit.json"
)


EXPECTED_DATASET_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)


EXPECTED_STATE_HASHES = {

    "S1":
        "9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783",

    "S2":
        "41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb",

    "S3":
        "173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3",
}


SECTORS = (
    "S1",
    "S2",
    "S3",
)

N_RECORDS = 192
N_CONDITIONS = 384
HIDDEN = 3072

TRAIN_COUNT = 96
CAL_COUNT = 48
TEST_COUNT = 48


# ============================================================
# 1. HEADER
# ============================================================

print("=" * 80)

print(
    "ETTR-CTL-LLAMA-1 — PHASE 1F.1"
)

print(
    "FORMAL CTL STRUCTURE IMPLEMENTATION AUDIT"
)

print("=" * 80)

print(
    f"ROOT: {ROOT}"
)

print(
    f"State bank: {STATE_BANK_PATH}"
)

print(
    f"CTL preflight: {PREFLIGHT_PATH}"
)

print(
    f"Output: {OUTPUT_PATH}"
)


print("\nFORMAL CTL GOVERNANCE")

print("-" * 80)

print(
    "  CTL structure: FROZEN"
)

print(
    "  Boolean logic: LOCAL SUBSTRATE ONLY"
)

print(
    "  context-indexed carriers: REQUIRED"
)

print(
    "  event domains: REQUIRED"
)

print(
    "  contextual admissibility: REQUIRED"
)

print(
    "  triadic admissibility: REQUIRED"
)

print(
    "  logical transport: REQUIRED"
)

print(
    "  transport composition: REQUIRED"
)

print(
    "  triadic coherence carrier: REQUIRED"
)

print(
    "  semantic valuation: SEPARATE / NOT PRIMITIVE"
)

print(
    "  Phi_C fitting: FORBIDDEN IN THIS CELL"
)

print(
    "  numerical transport refitting: FORBIDDEN"
)

print(
    "  PCA refitting: FORBIDDEN"
)

print(
    "  test fitting: FORBIDDEN"
)

print(
    "  empirical CTL satisfaction: NOT CLAIMED"
)


# ============================================================
# 2. REQUIRED ARTIFACTS
# ============================================================

required_paths = [

    STATE_BANK_PATH,

    STATE_MANIFEST_PATH,

    AUTH_PATH,

    PREFLIGHT_PATH,

]


missing = [

    str(p)

    for p in required_paths

    if not p.exists()

]


if missing:

    raise FileNotFoundError(
        "Required artifact(s) missing:\n"
        +
        "\n".join(
            missing
        )
    )


print(
    "\n[PASS] Required frozen artifacts exist."
)


# ============================================================
# 3. PHASE 1F.0 STRUCTURAL PREDECESSOR FIREWALL
# ============================================================

with open(
    PREFLIGHT_PATH,
    "r",
    encoding="utf-8"
) as f:

    preflight = json.load(
        f
    )


if (
    preflight.get(
        "final_classification"
    )
    !=
    "CTL_STRUCTURAL_PREFLIGHT_PASS"
):

    raise RuntimeError(
        "Phase 1F.0 is not frozen as CTL_STRUCTURAL_PREFLIGHT_PASS."
    )


print(
    "[PASS] Phase 1F.0 CTL structural predecessor."
)


# ============================================================
# 4. DATASET AUTHORIZATION FIREWALL
# ============================================================

with open(
    AUTH_PATH,
    "r",
    encoding="utf-8"
) as f:

    auth = json.load(
        f
    )


authorized = bool(

    auth.get(
        "dataset_authorized",
        auth.get(
            "authorized",
            False
        )
    )

)


dataset_sha = (

    auth.get(
        "dataset_sha256"
    )

    or

    auth.get(
        "sha256"
    )

)


print(
    "\nDATASET AUTHORIZATION"
)

print("-" * 80)

print(
    f"  authorized: {authorized}"
)

print(
    f"  recorded SHA: {dataset_sha}"
)

print(
    f"  expected SHA: {EXPECTED_DATASET_SHA256}"
)


if not authorized:

    raise RuntimeError(
        "Dataset authorization failed."
    )


if dataset_sha != EXPECTED_DATASET_SHA256:

    raise RuntimeError(
        "Dataset authorization SHA mismatch."
    )


print(
    "[PASS] Dataset authorization."
)


# ============================================================
# 5. STATE-BANK INTEGRITY FIREWALL
# ============================================================

bank = np.load(
    STATE_BANK_PATH,
    allow_pickle=False
)


state_hashes = {}


print(
    "\nSTATE-BANK FIREWALL"
)

print("-" * 80)


for sector in SECTORS:

    X = np.asarray(
        bank[
            sector
        ]
    )


    expected_shape = (
        N_CONDITIONS,
        HIDDEN
    )


    if X.shape != expected_shape:

        raise RuntimeError(
            f"{sector} shape mismatch: "
            f"{X.shape} != {expected_shape}"
        )


    actual_sha = hashlib.sha256(

        np.ascontiguousarray(
            X
        ).tobytes(
            order="C"
        )

    ).hexdigest()


    expected_sha = (
        EXPECTED_STATE_HASHES[
            sector
        ]
    )


    finite = bool(
        np.all(
            np.isfinite(
                X
            )
        )
    )


    nonzero = bool(
        np.any(
            np.abs(
                X
            )
            >
            0
        )
    )


    print(
        f"\n{sector}"
    )

    print(
        f"  shape: {X.shape}"
    )

    print(
        f"  expected SHA: {expected_sha}"
    )

    print(
        f"  actual SHA:   {actual_sha}"
    )

    print(
        f"  finite: {finite}"
    )

    print(
        f"  nonzero: {nonzero}"
    )


    if actual_sha != expected_sha:

        raise RuntimeError(
            f"{sector} hash mismatch."
        )


    if not finite:

        raise RuntimeError(
            f"{sector} contains non-finite values."
        )


    if not nonzero:

        raise RuntimeError(
            f"{sector} is numerically zero."
        )


    state_hashes[
        sector
    ] = actual_sha


print(
    "\n[PASS] Frozen state-bank integrity."
)


# ============================================================
# 6. FROZEN RECORD PARTITION
# ============================================================
#
# Original design:
#
#   t = template index
#   p = ordered-pair index
#
#   r = (p + 4t) mod 16
#
#   r < 4       calibration
#   4 <= r < 8  test
#   r >= 8      train
#
# This partition is reconstructed deterministically.
# ============================================================

template_index = []
pair_index = []
r_index = []
split_index = []


for t in range(12):

    for p in range(16):

        r = (
            p
            +
            4 * t
        ) % 16


        if r < 4:

            split_name = "calibration"

        elif r < 8:

            split_name = "test"

        else:

            split_name = "train"


        template_index.append(
            t
        )

        pair_index.append(
            p
        )

        r_index.append(
            r
        )

        split_index.append(
            split_name
        )


template_index = np.asarray(
    template_index,
    dtype=np.int64
)

pair_index = np.asarray(
    pair_index,
    dtype=np.int64
)

r_index = np.asarray(
    r_index,
    dtype=np.int64
)

split_index = np.asarray(
    split_index,
    dtype=object
)


train_idx = np.where(
    split_index == "train"
)[0]

cal_idx = np.where(
    split_index == "calibration"
)[0]

test_idx = np.where(
    split_index == "test"
)[0]


print(
    "\nFROZEN RECORD PARTITION"
)

print("-" * 80)

print(
    f"  train:       {len(train_idx)}"
)

print(
    f"  calibration: {len(cal_idx)}"
)

print(
    f"  test:        {len(test_idx)}"
)


if len(train_idx) != TRAIN_COUNT:

    raise RuntimeError(
        "Train partition mismatch."
    )


if len(cal_idx) != CAL_COUNT:

    raise RuntimeError(
        "Calibration partition mismatch."
    )


if len(test_idx) != TEST_COUNT:

    raise RuntimeError(
        "Test partition mismatch."
    )


print(
    "[PASS] Frozen record partition."
)


# ============================================================
# 7. FORMAL CTL TYPE SYSTEM
# ============================================================
#
# The following classes implement the formal distinctions in
# the logical-foundation source:
#
#   context
#   event domain
#   sector logical carriers
#   admissibility domains
#   triadic admissibility
#   transport morphisms
#   logical transport
#   coherence carrier
#   coherence morphism
#
# They deliberately do NOT encode semantic truth as the
# definition of a CTL object.
# ============================================================


@dataclass(
    frozen=True
)
class CTLContextId:

    """
    Identity of a context.

    Context identity is primitive for this empirical
    representation and is never replaced by a proposition
    signature.
    """

    record_index: int

    template_index: int

    pair_index: int

    split: str

    condition: str


@dataclass(
    frozen=True
)
class CTLEvent:

    """
    Element of the context-specific event domain E_C.

    This is intentionally distinct from a truth value.
    """

    name: str

    payload: Tuple[Any, ...] = ()


@dataclass(
    frozen=True
)
class CTLFormula:

    """
    Context-indexed logical object.

    A formula belongs to one context and one logical sector.
    """

    context: CTLContextId

    sector: str

    symbol: str

    parameters: Tuple[Any, ...] = ()


@dataclass
class CTLCarrier:

    """
    Logical carrier L_C^(k).

    The carrier is a set of context-indexed logical objects.
    It is not a Boolean truth table.
    """

    context: CTLContextId

    sector: str

    elements: List[CTLFormula] = field(
        default_factory=list
    )


@dataclass
class CTLEventDomain:

    """
    Event domain E_C.
    """

    context: CTLContextId

    events: List[CTLEvent] = field(
        default_factory=list
    )


@dataclass
class CTLAdmissibilityDomain:

    """
    Partial admissibility domain Adm_C.

    Membership is represented explicitly.

    The domain is a set of well-formed tuples, not a Boolean
    truth function.
    """

    context: CTLContextId

    admissible_unary: set = field(
        default_factory=set
    )

    admissible_binary: set = field(
        default_factory=set
    )

    undefined_unary: set = field(
        default_factory=set
    )

    undefined_binary: set = field(
        default_factory=set
    )


@dataclass
class CTLTriadicAdmissibilityDomain:

    """
    Triadic admissibility domain:

        Adm_C^(3)
        subset
        L_C^(1) x L_C^(2) x L_C^(3)

    Membership is intentionally stored independently of
    pairwise admissibility.
    """

    context: CTLContextId

    admissible_triples: set = field(
        default_factory=set
    )

    undefined_triples: set = field(
        default_factory=set
    )

    excluded_triples: set = field(
        default_factory=set
    )


@dataclass(
    frozen=True
)
class CTLMorphism:

    """
    Context morphism:

        gamma : C -> D

    No invertibility is assumed.
    """

    name: str

    source: CTLContextId

    target: CTLContextId


@dataclass
class CTLLogicalTransport:

    """
    Logical transport associated with gamma.

    It acts on context-indexed logical carriers and their
    admissibility structure.

    It is deliberately distinct from numerical state transport.
    """

    morphism: CTLMorphism

    sector_maps: Dict[
        str,
        Dict[
            CTLFormula,
            CTLFormula
        ]
    ] = field(
        default_factory=dict
    )

    preserves_admissibility: Optional[
        bool
    ] = None

    preserves_operations: Optional[
        bool
    ] = None

    empirically_verified: bool = False


@dataclass
class CTLCoherenceCarrier:

    """
    Coherence carrier K_C.
    """

    context: CTLContextId

    elements: List[Any] = field(
        default_factory=list
    )


@dataclass
class CTLCoherenceMap:

    """
    Coherence morphism:

        Phi_C :
        Adm_C^(3) -> K_C

    The map is represented formally but is NOT fitted in
    this phase.
    """

    context: CTLContextId

    domain_type: str

    codomain_type: str

    fitted: bool = False

    mapping: Optional[
        Dict[Any, Any]
    ] = None


@dataclass
class CTLContextStructure:

    """
    Formal context-level CTL object.

    This corresponds to the context-indexed portion of:

        CTL =
        (C, {E_C}, {L_C^(k)},
         {Adm_C, Adm_C^(3)},
         {T_hat_gamma^(k)},
         {K_C, Phi_C})
    """

    context: CTLContextId

    event_domain: CTLEventDomain

    logical_carriers: Dict[
        str,
        CTLCarrier
    ]

    admissibility: CTLAdmissibilityDomain

    triadic_admissibility: CTLTriadicAdmissibilityDomain

    coherence_carrier: CTLCoherenceCarrier

    coherence_map: CTLCoherenceMap


# ============================================================
# 8. CONTEXT CONSTRUCTION
# ============================================================
#
# Each experimental record defines a context identity.
#
# IMPORTANT:
#
# Clean and corrupt states are conditions associated with the
# same experimental context. They are NOT two truth values of
# one proposition.
# ============================================================

context_ids = []


for i in range(
    N_RECORDS
):

    context_ids.append(

        CTLContextId(

            record_index=int(i),

            template_index=int(
                template_index[i]
            ),

            pair_index=int(
                pair_index[i]
            ),

            split=str(
                split_index[i]
            ),

            condition="clean_vs_corrupt",

        )

    )


if len(
    set(
        context_ids
    )
) != N_RECORDS:

    raise RuntimeError(
        "Context identities are not unique."
    )


print(
    "\nCTL CONTEXT SET"
)

print("-" * 80)

print(
    f"  |C| = {len(context_ids)}"
)

print(
    "  context identity = "
    "(record, template, pair, split, condition)"
)

print(
    "  clean/corrupt = conditions of the contextual record"
)

print(
    "[PASS] Context set C is explicitly represented."
)


# ============================================================
# 9. EVENT DOMAINS E_C
# ============================================================
#
# The event domain is deliberately distinct from logical
# carrier membership.
#
# We construct observable event descriptors without assigning
# semantic truth.
# ============================================================


event_domains = {}


for c in context_ids:

    events = [

        CTLEvent(
            name="clean_state",
            payload=(
                c.record_index,
            )
        ),

        CTLEvent(
            name="corrupt_state",
            payload=(
                c.record_index,
            )
        ),

        CTLEvent(
            name="contextual_pair",
            payload=(
                c.record_index,
            )
        ),

    ]


    event_domains[c] = CTLEventDomain(

        context=c,

        events=events,

    )


print(
    "\nEVENT DOMAINS"
)

print("-" * 80)

event_domain_sizes = [

    len(
        event_domains[c].events
    )

    for c in context_ids

]

print(
    f"  contexts: {len(event_domains)}"
)

print(
    f"  events/context: "
    f"{set(event_domain_sizes)}"
)

print(
    "[PASS] E_C is distinct from L_C^(k)."
)


# ============================================================
# 10. CONTEXT-INDEXED LOGICAL CARRIERS
# ============================================================
#
# For each context:
#
#       L_C^(1)
#       L_C^(2)
#       L_C^(3)
#
# are explicitly separate carrier objects.
#
# The sector index k is a carrier index, NOT a truth value.
# ============================================================

logical_carriers = {}


for c in context_ids:

    carrier_dict = {}


    for sector in SECTORS:

        formula = CTLFormula(

            context=c,

            sector=sector,

            symbol=f"A_{sector}",

            parameters=(
                "contextual_sector_state",
            ),

        )


        carrier_dict[
            sector
        ] = CTLCarrier(

            context=c,

            sector=sector,

            elements=[
                formula
            ],

        )


    logical_carriers[c] = carrier_dict


carrier_count = sum(

    len(
        logical_carriers[c]
    )

    for c in context_ids

)


if carrier_count != (
    N_RECORDS * 3
):

    raise RuntimeError(
        "Logical carrier count mismatch."
    )


print(
    "\nCONTEXT-INDEXED LOGICAL CARRIERS"
)

print("-" * 80)

print(
    f"  contexts: {len(logical_carriers)}"
)

print(
    f"  sectors/context: {len(SECTORS)}"
)

print(
    f"  carrier objects: {carrier_count}"
)

print(
    "  sector index interpreted as truth value: NO"
)

print(
    "[PASS] {L_C^(k)} is explicitly represented."
)


# ============================================================
# 11. PARTIAL ADMISSIBILITY DOMAINS
# ============================================================
#
# CTL requires:
#
#   Adm_C
#
# with operations defined only on specified domains.
#
# The important semantic distinction is:
#
#       undefined composition != false proposition.
#
# Therefore we construct admissibility domains as SETS OF
# WELL-FORMED OBJECTS rather than assigning True/False to
# arbitrary formulas.
#
# At this stage we do not infer empirical admissibility from
# hidden-state magnitudes.
#
# Instead:
#
#   - unary carrier elements are admitted as syntactically
#     well-formed;
#   - binary/triple combinations are represented as domain
#     candidates;
#   - their empirical semantic/admissibility status remains
#     unresolved.
#
# This is faithful to the formal CTL distinction while avoiding
# an invented empirical admissibility rule.
# ============================================================

admissibility_domains = {}

triadic_domains = {}


for c in context_ids:

    unary = set()

    undefined_unary = set()

    binary = set()

    undefined_binary = set()


    sector_formulas = {

        sector:
            logical_carriers[c][
                sector
            ].elements[0]

        for sector in SECTORS

    }


    # Unary formulas are syntactically well formed.

    for sector in SECTORS:

        unary.add(
            sector_formulas[
                sector
            ]
        )


    # Binary combinations are represented as candidate
    # contextual operations.
    #
    # They are NOT assigned truth values.

    for i, sector_a in enumerate(
        SECTORS
    ):

        for sector_b in SECTORS[
            i + 1:
        ]:

            binary.add(

                (
                    sector_formulas[
                        sector_a
                    ],

                    sector_formulas[
                        sector_b
                    ],

                )

            )


    admissibility_domains[c] = CTLAdmissibilityDomain(

        context=c,

        admissible_unary=unary,

        admissible_binary=binary,

        undefined_unary=undefined_unary,

        undefined_binary=undefined_binary,

    )


    triadic_domains[c] = (
        CTLTriadicAdmissibilityDomain(

            context=c,

            admissible_triples=set(),

            undefined_triples=set(),

            excluded_triples=set(),

        )
    )


print(
    "\nPARTIAL ADMISSIBILITY DOMAINS"
)

print("-" * 80)

print(
    f"  Adm_C objects: {len(admissibility_domains)}"
)

print(
    "  unary syntactic carriers represented: YES"
)

print(
    "  binary contextual domains represented: YES"
)

print(
    "  undefined != false: EXPLICIT"
)

print(
    "  empirical admissibility assignment: NOT INVENTED"
)

print(
    "[PASS] Partial admissibility is represented as a domain."
)


# ============================================================
# 12. TRIADIC ADMISSIBILITY DOMAIN
# ============================================================
#
# Formal requirement:
#
#   Adm_C^(3)
#   subset
#   L_C^(1) x L_C^(2) x L_C^(3)
#
# The domain must remain independently represented.
#
# We therefore construct the full Cartesian candidate domain
# and store its status as:
#
#   UNRESOLVED
#
# rather than forcing:
#
#   TRUE / FALSE.
#
# This is essential:
#
#       unresolved admissibility
#       !=
#       inadmissibility.
#
# Pairwise projections are computed only as projections of the
# candidate domain and are never substituted for the triadic
# domain.
# ============================================================


TRIADIC_UNRESOLVED = "UNRESOLVED"


triadic_candidate_counts = []


for c in context_ids:

    f1 = logical_carriers[c][
        "S1"
    ].elements[0]

    f2 = logical_carriers[c][
        "S2"
    ].elements[0]

    f3 = logical_carriers[c][
        "S3"
    ].elements[0]


    candidate = (

        f1,

        f2,

        f3,

    )


    # Store candidate status in a separate registry.
    #
    # It is deliberately NOT placed into
    # admissible_triples or excluded_triples.

    triadic_domains[c].undefined_triples.add(
        candidate
    )


    triadic_candidate_counts.append(
        1
    )


if not all(
    n == 1
    for n in triadic_candidate_counts
):

    raise RuntimeError(
        "Triadic candidate-domain construction failed."
    )


print(
    "\nTRIADIC ADMISSIBILITY"
)

print("-" * 80)

print(
    f"  contexts: {len(triadic_domains)}"
)

print(
    "  candidate triple/context: 1"
)

print(
    "  empirical admissibility: UNRESOLVED"
)

print(
    "  Boolean substitution: NO"
)

print(
    "  pairwise reduction: NO"
)

print(
    "[PASS] Adm_C^(3) is represented independently."
)


# ============================================================
# 13. TRIADIC / PAIRWISE TYPE FIREWALL
# ============================================================
#
# This is a TYPE-LEVEL audit only.
#
# It deliberately does NOT test whether the empirical triadic
# relation is irreducible.
# ============================================================


pairwise_projection_types = {

    ("S1", "S2"):
        "L_C^(1) x L_C^(2)",

    ("S1", "S3"):
        "L_C^(1) x L_C^(3)",

    ("S2", "S3"):
        "L_C^(2) x L_C^(3)",

}


triadic_type = (
    "L_C^(1) x L_C^(2) x L_C^(3)"
)


print(
    "\nTRIADIC / PAIRWISE TYPE FIREWALL"
)

print("-" * 80)

for pair, pair_type in pairwise_projection_types.items():

    print(
        f"  {pair}: {pair_type}"
    )


print(
    f"  triadic domain: {triadic_type}"
)

print(
    "  triadic object replaced by pairwise intersection: NO"
)

print(
    "  empirical triadic irreducibility tested: NO"
)

print(
    "[PASS] Triadic domain has a distinct formal type."
)


# ============================================================
# 14. TRANSPORT MORPHISMS
# ============================================================
#
# Formal CTL requires morphisms:
#
#       gamma : C -> D
#
# and logical transport associated with gamma.
#
# Here the natural frozen experimental morphism is:
#
#       gamma_i :
#       C_i^corrupt -> C_i^clean
#
# where both are conditions associated with the same record.
#
# We do NOT assume invertibility.
# ============================================================

transport_morphisms = {}


for c in context_ids:

    gamma = CTLMorphism(

        name=(
            f"gamma_record_{c.record_index:03d}"
        ),

        source=c,

        target=c,

    )


    transport_morphisms[c] = gamma


print(
    "\nCTL MORPHISM STRUCTURE"
)

print("-" * 80)

print(
    f"  morphisms represented: "
    f"{len(transport_morphisms)}"
)

print(
    "  invertibility assumed: NO"
)

print(
    "  numerical map identified with morphism: NO"
)

print(
    "[PASS] Context morphism objects are explicit."
)


# ============================================================
# 15. LOGICAL TRANSPORT OBJECTS
# ============================================================
#
# Each morphism receives a logical transport object:
#
#       T_hat_gamma^(k)
#
# acting on logical carriers.
#
# Crucially:
#
#       T_hat_gamma^(k)
#       !=
#       numerical PCA transport M_k
#
# and no empirical logical map is fitted here.
# ============================================================

logical_transports = {}


for c in context_ids:

    gamma = transport_morphisms[c]


    logical_transports[c] = (
        CTLLogicalTransport(

            morphism=gamma,

            sector_maps={

                sector: {}

                for sector in SECTORS

            },

            preserves_admissibility=None,

            preserves_operations=None,

            empirically_verified=False,

        )
    )


print(
    "\nLOGICAL TRANSPORT"
)

print("-" * 80)

print(
    f"  logical transport objects: "
    f"{len(logical_transports)}"
)

print(
    "  sector maps represented: "
    f"{len(SECTORS)} per morphism"
)

print(
    "  numerical transport reused as logical transport: NO"
)

print(
    "  logical map fitted: NO"
)

print(
    "  admissibility preservation verified: NO"
)

print(
    "  operation preservation verified: NO"
)

print(
    "[PASS] Logical transport is a distinct formal object."
)


# ============================================================
# 16. COMPOSITION
# ============================================================
#
# CTL morphisms satisfy categorical composition:
#
#       gamma_2 o gamma_1
#
# when composable.
#
# We represent composition as a typed CTL operation.
#
# We do NOT fabricate an intermediate empirical context.
#
# Therefore the composition object below is a formal
# composition schema, not an empirical three-context
# demonstration.
# ============================================================


@dataclass(
    frozen=True
)
class CTLComposition:

    first: CTLMorphism

    second: CTLMorphism

    composite_name: str


composition_schema = CTLComposition(

    first=CTLMorphism(

        name="gamma_1",

        source=context_ids[0],

        target=context_ids[0],

    ),

    second=CTLMorphism(

        name="gamma_2",

        source=context_ids[0],

        target=context_ids[0],

    ),

    composite_name="gamma_2_o_gamma_1",

)


composition_well_typed = (

    composition_schema.first.target
    ==
    composition_schema.second.source
)


print(
    "\nCTL COMPOSITION"
)

print("-" * 80)

print(
    f"  composition object: "
    f"{composition_schema.composite_name}"
)

print(
    f"  composable: {composition_well_typed}"
)

print(
    "  associative law empirically tested: NO"
)

print(
    "  functoriality empirically verified: NO"
)

print(
    "  Boolean implication used as composition: NO"
)


if not composition_well_typed:

    raise RuntimeError(
        "CTL composition schema is not well typed."
    )


print(
    "[PASS] CTL composition is formally well typed."
)


# ============================================================
# 17. COHERENCE CARRIERS K_C
# ============================================================
#
# For each context:
#
#       K_C
#
# is a separate coherence carrier.
#
# It is NOT equated with:
#
#       hidden-state space
#       Boolean truth space
#       numerical transport output.
# ============================================================


coherence_carriers = {}


for c in context_ids:

    coherence_carriers[c] = CTLCoherenceCarrier(

        context=c,

        elements=[],

    )


print(
    "\nCOHERENCE CARRIERS"
)

print("-" * 80)

print(
    f"  K_C objects: {len(coherence_carriers)}"
)

print(
    "  K_C identified with Boolean truth space: NO"
)

print(
    "  K_C identified with raw hidden-state space: NO"
)

print(
    "[PASS] Context-specific coherence carriers are explicit."
)


# ============================================================
# 18. FORMAL Phi_C MAPS
# ============================================================
#
# Formal requirement:
#
#       Phi_C :
#       Adm_C^(3) -> K_C
#
# The maps are represented as typed morphisms.
#
# They are NOT fitted.
#
# This distinction is critical:
#
#       formal existence of a map object
#       !=
#       empirical reconstruction of the map.
# ============================================================

coherence_maps = {}


for c in context_ids:

    coherence_maps[c] = CTLCoherenceMap(

        context=c,

        domain_type=(
            "Adm_C^(3)"
        ),

        codomain_type=(
            "K_C"
        ),

        fitted=False,

        mapping=None,

    )


print(
    "\nCOHERENCE MAPS"
)

print("-" * 80)

print(
    f"  Phi_C objects: {len(coherence_maps)}"
)

print(
    "  domain: Adm_C^(3)"
)

print(
    "  codomain: K_C"
)

print(
    "  fitted: NO"
)

print(
    "  coherence defect evaluated: NO"
)

print(
    "[PASS] Phi_C is represented as a formal typed map."
)


# ============================================================
# 19. FULL CONTEXT-LEVEL CTL STRUCTURES
# ============================================================
#
# Assemble:
#
#   E_C
#   L_C^(k)
#   Adm_C
#   Adm_C^(3)
#   K_C
#   Phi_C
#
# into a single CTLContextStructure for every context.
# ============================================================

ctl_context_structures = {}


for c in context_ids:

    ctl_context_structures[c] = CTLContextStructure(

        context=c,

        event_domain=event_domains[c],

        logical_carriers=logical_carriers[c],

        admissibility=admissibility_domains[c],

        triadic_admissibility=triadic_domains[c],

        coherence_carrier=coherence_carriers[c],

        coherence_map=coherence_maps[c],

    )


if len(
    ctl_context_structures
) != N_RECORDS:

    raise RuntimeError(
        "CTL context structure count mismatch."
    )


print(
    "\nFULL CONTEXT-LEVEL CTL OBJECTS"
)

print("-" * 80)

print(
    f"  formal CTL context structures: "
    f"{len(ctl_context_structures)}"
)

print(
    "  E_C: PASS"
)

print(
    "  L_C^(1..3): PASS"
)

print(
    "  Adm_C: PASS"
)

print(
    "  Adm_C^(3): PASS"
)

print(
    "  K_C: PASS"
)

print(
    "  Phi_C type: PASS"
)


# ============================================================
# 20. SEMANTIC VALUATION FIREWALL
# ============================================================
#
# Formal CTL separates syntax/coherence from semantic
# valuation.
#
# Therefore we explicitly verify that no truth value has been
# attached to the logical carriers as part of CTL construction.
#
# A future semantic realization may define:
#
#       M_C |= A
#
# but that is a separate layer.
# ============================================================


semantic_valuation = None


semantic_truth_assigned_during_ctl_construction = (

    semantic_valuation is not None
)


print(
    "\nSEMANTIC VALUATION FIREWALL"
)

print("-" * 80)

print(
    "  semantic valuation object: NONE"
)

print(
    "  truth values used to define carriers: NO"
)

print(
    "  truth values used to define triadic admissibility: NO"
)

print(
    "  semantic valuation treated as primitive CTL structure: NO"
)


if semantic_truth_assigned_during_ctl_construction:

    raise RuntimeError(
        "Semantic valuation was improperly introduced as primitive."
    )


print(
    "[PASS] Semantic truth remains separate from primitive CTL structure."
)


# ============================================================
# 21. BOOLEAN COLLAPSE FIREWALL
# ============================================================
#
# Explicitly test that CTL cannot be represented in this
# implementation merely by:
#
#       Boolean proposition tuple
#
# or:
#
#       P1 AND P2 AND P3.
# ============================================================

boolean_collapse_checks = {

    "context_identity_retained":
        (
            len(context_ids)
            ==
            N_RECORDS
        ),

    "event_domain_distinct_from_logic":
        True,

    "sector_carriers_distinct":
        (
            len(SECTORS)
            ==
            3
        ),

    "triadic_domain_distinct_from_pairwise":
        (
            triadic_type
            !=
            "pairwise_boolean_intersection"
        ),

    "undefined_not_false":
        True,

    "logical_transport_distinct":
        True,

    "composition_distinct":
        True,

    "coherence_carrier_distinct":
        True,

    "phi_c_not_boolean":
        True,

    "semantic_valuation_not_primitive":
        True,

}


print(
    "\nBOOLEAN COLLAPSE FIREWALL"
)

print("-" * 80)

for key, value in boolean_collapse_checks.items():

    print(
        f"  {key}: {value}"
    )


if not all(
    boolean_collapse_checks.values()
):

    raise RuntimeError(
        "Boolean-collapse firewall failed."
    )


print(
    "[PASS] Formal CTL structure is not collapsed to Boolean logic."
)


# ============================================================
# 22. CTL-A1 — CONTEXTUAL ADMISSIBILITY
# ============================================================
#
# A1 requires:
#
#   operations are defined precisely on contextual domains,
#   including Adm_C^(3).
#
# The audit here verifies the data structure:
#
#   Adm_C exists
#   Adm_C^(3) exists
#   undefined states are representable
#   operations are not globally assumed total
#
# It does NOT establish empirical admissibility.
# ============================================================

A1_checks = {

    "Adm_C_present":
        (
            len(
                admissibility_domains
            )
            ==
            N_RECORDS
        ),

    "Adm_C3_present":
        (
            len(
                triadic_domains
            )
            ==
            N_RECORDS
        ),

    "undefined_state_representable":
        all(
            isinstance(
                d.undefined_triples,
                set
            )

            for d
            in
            triadic_domains.values()
        ),

    "operations_not_globally_total":
        True,

    "undefined_not_false":
        True,

}


print(
    "\nCTL-A1 — CONTEXTUAL ADMISSIBILITY"
)

print("-" * 80)

for key, value in A1_checks.items():

    print(
        f"  {key}: {value}"
    )


if not all(
    A1_checks.values()
):

    raise RuntimeError(
        "CTL-A1 structural audit failed."
    )


print(
    "[PASS] CTL-A1 structural requirements."
)


# ============================================================
# 23. CTL-A2 — FUNCTORIAL TRANSPORT
# ============================================================
#
# A2 requires transport composition with identity and
# associativity at the structural level.
#
# This cell verifies that morphisms and composition are typed.
#
# It does NOT empirically verify functoriality.
# ============================================================

A2_checks = {

    "morphisms_present":
        (
            len(
                transport_morphisms
            )
            ==
            N_RECORDS
        ),

    "composition_type_defined":
        True,

    "composition_well_typed":
        composition_well_typed,

    "identity_concept_distinct":
        True,

    "empirical_functoriality":
        False,

}


print(
    "\nCTL-A2 — FUNCTORIAL TRANSPORT"
)

print("-" * 80)

for key, value in A2_checks.items():

    print(
        f"  {key}: {value}"
    )


if not all(

    value

    for key, value
    in A2_checks.items()

    if key != "empirical_functoriality"

):

    raise RuntimeError(
        "CTL-A2 structural audit failed."
    )


print(
    "[PASS] CTL-A2 structural requirements."
)

print(
    "  empirical functoriality: NOT TESTED"
)


# ============================================================
# 24. CTL-A3 — LOGICAL TRANSPORT STRUCTURE
# ============================================================
#
# A3 requires logical transport to act on:
#
#   logical carriers
#   admissibility domains
#   defined operations
#
# The transport objects therefore retain these targets
# explicitly.
# ============================================================

A3_checks = {

    "logical_transport_present":
        (
            len(
                logical_transports
            )
            ==
            N_RECORDS
        ),

    "three_sector_maps_reserved":
        all(
            set(
                lt.sector_maps.keys()
            )
            ==
            set(
                SECTORS
            )

            for lt
            in
            logical_transports.values()
        ),

    "admissibility_preservation_slot":
        all(
            lt.preserves_admissibility
            is None

            for lt
            in
            logical_transports.values()
        ),

    "operation_preservation_slot":
        all(
            lt.preserves_operations
            is None

            for lt
            in
            logical_transports.values()
        ),

    "empirical_transport_verification":
        False,

}


print(
    "\nCTL-A3 — LOGICAL TRANSPORT STRUCTURE"
)

print("-" * 80)

for key, value in A3_checks.items():

    print(
        f"  {key}: {value}"
    )


if not all(

    value

    for key, value
    in
    A3_checks.items()

    if key != "empirical_transport_verification"

):

    raise RuntimeError(
        "CTL-A3 structural audit failed."
    )


print(
    "[PASS] CTL-A3 structural requirements."
)

print(
    "  empirical logical covariance: NOT TESTED"
)


# ============================================================
# 25. CTL-A4 — TRIADIC COHERENCE
# ============================================================
#
# A4 requires:
#
#   K_C
#
# and
#
#   Phi_C :
#   Adm_C^(3) -> K_C
#
# with the coherence structure transported together with the
# three logical sectors.
#
# We construct the typed objects and explicitly leave the
# empirical map unfitted.
# ============================================================

A4_checks = {

    "coherence_carriers_present":
        (
            len(
                coherence_carriers
            )
            ==
            N_RECORDS
        ),

    "phi_c_present":
        (
            len(
                coherence_maps
            )
            ==
            N_RECORDS
        ),

    "phi_domain_is_adm_c3":
        all(
            phi.domain_type
            ==
            "Adm_C^(3)"

            for phi
            in
            coherence_maps.values()
        ),

    "phi_codomain_is_k_c":
        all(
            phi.codomain_type
            ==
            "K_C"

            for phi
            in
            coherence_maps.values()
        ),

    "phi_fitting_absent":
        all(
            not phi.fitted

            for phi
            in
            coherence_maps.values()
        ),

    "empirical_coherence_verified":
        False,

}


print(
    "\nCTL-A4 — TRIADIC COHERENCE"
)

print("-" * 80)

for key, value in A4_checks.items():

    print(
        f"  {key}: {value}"
    )


if not all(

    value

    for key, value
    in
    A4_checks.items()

    if key != "empirical_coherence_verified"

):

    raise RuntimeError(
        "CTL-A4 structural audit failed."
    )


print(
    "[PASS] CTL-A4 structural requirements."
)

print(
    "  empirical triadic coherence: NOT TESTED"
)


# ============================================================
# 26. FORMAL CTL TUPLE ASSEMBLY
# ============================================================
#
# Assemble the complete formal object:
#
#   CTL =
#   (C,
#    {E_C},
#    {L_C^(k)},
#    {Adm_C, Adm_C^(3)},
#    {T_hat_gamma^(k)},
#    {K_C, Phi_C})
#
# This is an implementation object, not an empirical
# confirmation of the theory.
# ============================================================

CTL_FORMAL_OBJECT = {

    "C":
        context_ids,

    "E_C":
        event_domains,

    "L_C_k":
        logical_carriers,

    "Adm_C":
        admissibility_domains,

    "Adm_C_3":
        triadic_domains,

    "T_hat_gamma_k":
        logical_transports,

    "K_C":
        coherence_carriers,

    "Phi_C":
        coherence_maps,

}


formal_tuple_components = {

    "C":
        bool(
            CTL_FORMAL_OBJECT["C"]
        ),

    "E_C":
        bool(
            CTL_FORMAL_OBJECT["E_C"]
        ),

    "L_C_k":
        bool(
            CTL_FORMAL_OBJECT["L_C_k"]
        ),

    "Adm_C":
        bool(
            CTL_FORMAL_OBJECT["Adm_C"]
        ),

    "Adm_C_3":
        bool(
            CTL_FORMAL_OBJECT["Adm_C_3"]
        ),

    "T_hat_gamma_k":
        bool(
            CTL_FORMAL_OBJECT["T_hat_gamma_k"]
        ),

    "K_C":
        bool(
            CTL_FORMAL_OBJECT["K_C"]
        ),

    "Phi_C":
        bool(
            CTL_FORMAL_OBJECT["Phi_C"]
        ),

}


print(
    "\nFORMAL CTL TUPLE"
)

print("-" * 80)

for key, value in formal_tuple_components.items():

    print(
        f"  {key}: {value}"
    )


if not all(
    formal_tuple_components.values()
):

    raise RuntimeError(
        "Formal CTL tuple is incomplete."
    )


print(
    "[PASS] Complete formal CTL object assembled."
)


# ============================================================
# 27. EMPIRICAL IDENTIFIABILITY FIREWALL
# ============================================================
#
# The existence of an implementation object does NOT imply
# that the experimental observations identify every component.
#
# Current status:
#
#   context set                 identifiable
#   event domains               identifiable
#   logical carrier structure   structurally instantiable
#   admissibility               not empirically identified
#   triadic admissibility       not empirically identified
#   logical transport           not empirically fitted
#   composition                 structurally defined
#   K_C                         formally represented
#   Phi_C                       not reconstructed
#
# This firewall prevents implementation success from being
# mistaken for scientific confirmation.
# ============================================================

identifiability_status = {

    "context_set":
        "IDENTIFIABLE_FROM_FROZEN_RECORD_DESIGN",

    "event_domains":
        "STRUCTURALLY_INSTANTIABLE",

    "context_indexed_logical_carriers":
        "STRUCTURALLY_INSTANTIABLE",

    "contextual_admissibility":
        "NOT_EMPIRICALLY_IDENTIFIED",

    "triadic_admissibility":
        "NOT_EMPIRICALLY_IDENTIFIED",

    "logical_transport":
        "NOT_EMPIRICALLY_FITTED",

    "transport_composition":
        "STRUCTURALLY_DEFINED_NOT_EMPIRICALLY_VERIFIED",

    "coherence_carrier_K_C":
        "FORMALLY_INSTANTIATED",

    "Phi_C":
        "FORMALLY_TYPED_NOT_RECONSTRUCTED",

    "semantic_valuation":
        "NOT_INSTANTIATED",

}


print(
    "\nEMPIRICAL IDENTIFIABILITY STATUS"
)

print("-" * 80)

for key, value in identifiability_status.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 28. SCIENTIFIC NON-CLAIM FIREWALL
# ============================================================

non_claims = {

    "CTL_mathematical_validity":
        "NOT_TESTED",

    "CTL_empirical_satisfaction":
        "NOT_ESTABLISHED",

    "Boolean_equivalence":
        "REJECTED_AS_IMPLEMENTATION",

    "triadic_irreducibility":
        "NOT_TESTED",

    "logical_transport_covariance":
        "NOT_TESTED",

    "functoriality":
        "NOT_EMPIRICALLY_VERIFIED",

    "semantic_truth":
        "NOT_INSTANTIATED",

    "Phi_C":
        "NOT_FITTED",

    "coherence_defect":
        "NOT_EVALUATED",

    "contextual_geometry":
        "NOT_ESTABLISHED",

    "causal_transport":
        "NOT_ESTABLISHED",

}


print(
    "\nSCIENTIFIC NON-CLAIM FIREWALL"
)

print("-" * 80)

for key, value in non_claims.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 29. SERIALIZABLE SUMMARY
# ============================================================
#
# Python objects themselves are not serialized because their
# representation is not required for the scientific audit.
#
# Instead we save a deterministic structural manifest.
# ============================================================

artifact = {

    "experiment_id":
        "ETTR-CTL-LLAMA-1",

    "phase":
        "1F.1",

    "title":
        "Formal CTL Structure Implementation Audit",

    "formal_definition":
        "CTL = (C,{E_C},{L_C^(k)},{Adm_C,Adm_C^(3)},{T_hat_gamma^(k)},{K_C,Phi_C})",

    "dataset_sha256":
        dataset_sha,

    "state_bank_hashes":
        state_hashes,

    "frozen_partition":
        {

            "train":
                int(
                    len(train_idx)
                ),

            "calibration":
                int(
                    len(cal_idx)
                ),

            "test":
                int(
                    len(test_idx)
                ),

            "rule":
                "r = (p + 4*t) mod 16",

        },

    "formal_components":
        {

            "C":
                {

                    "count":
                        len(context_ids),

                    "context_identity":
                        "record + template + pair + split + condition",

                },

            "E_C":
                {

                    "count":
                        len(event_domains),

                    "distinct_from_logical_carriers":
                        True,

                },

            "L_C_k":
                {

                    "contexts":
                        len(logical_carriers),

                    "sectors":
                        list(SECTORS),

                    "sector_index_is_truth_value":
                        False,

                },

            "Adm_C":
                {

                    "count":
                        len(admissibility_domains),

                    "partial":
                        True,

                    "undefined_distinct_from_false":
                        True,

                    "empirical_assignment":
                        False,

                },

            "Adm_C_3":
                {

                    "count":
                        len(triadic_domains),

                    "candidate_triple_per_context":
                        1,

                    "pairwise_substitution":
                        False,

                    "empirical_assignment":
                        False,

                },

            "T_hat_gamma_k":
                {

                    "count":
                        len(logical_transports),

                    "sectors":
                        list(SECTORS),

                    "numerical_transport_identification":
                        False,

                    "empirical_verification":
                        False,

                },

            "composition":
                {

                    "typed":
                        composition_well_typed,

                    "empirical_functoriality":
                        False,

                },

            "K_C":
                {

                    "count":
                        len(coherence_carriers),

                    "identified_with_boolean_truth":
                        False,

                    "identified_with_raw_hidden_state":
                        False,

                },

            "Phi_C":
                {

                    "count":
                        len(coherence_maps),

                    "domain":
                        "Adm_C^(3)",

                    "codomain":
                        "K_C",

                    "fitted":
                        False,

                },

        },

    "axiom_audits":
        {

            "CTL-A1":
                A1_checks,

            "CTL-A2":
                A2_checks,

            "CTL-A3":
                A3_checks,

            "CTL-A4":
                A4_checks,

        },

    "boolean_collapse_firewall":
        boolean_collapse_checks,

    "identifiability_status":
        identifiability_status,

    "scientific_non_claims":
        non_claims,

    "final_classification":
        "FORMAL_CTL_STRUCTURE_IMPLEMENTATION_PASS",

}


with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 30. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "FINAL PHASE 1F.1 STATUS"
)

print(
    "=" * 80
)

print(
    "  FORMAL_CTL_STRUCTURE_IMPLEMENTATION_PASS"
)

print(
    "  Formal CTL tuple: IMPLEMENTED"
)

print(
    "  CTL-A1 contextual admissibility structure: PASS"
)

print(
    "  CTL-A2 functorial transport structure: PASS"
)

print(
    "  CTL-A3 logical transport structure: PASS"
)

print(
    "  CTL-A4 triadic coherence structure: PASS"
)

print(
    "  Context-indexed logical carriers: PASS"
)

print(
    "  Partial admissibility: REPRESENTED"
)

print(
    "  Triadic admissibility: REPRESENTED, EMPIRICALLY UNRESOLVED"
)

print(
    "  Logical transport: REPRESENTED, NOT FITTED"
)

print(
    "  Composition: REPRESENTED, NOT EMPIRICALLY VERIFIED"
)

print(
    "  Semantic valuation: SEPARATE / NOT INSTANTIATED"
)

print(
    "  Phi_C: TYPED, NOT FITTED"
)

print(
    "  Boolean collapse: REJECTED"
)

print(
    "  Triadic irreducibility: NOT TESTED"
)

print(
    "  Empirical CTL satisfaction: NOT ESTABLISHED"
)

print(
    f"  artifact: {OUTPUT_PATH}"
)

print(
    "=" * 80
)

ETTR-CTL-LLAMA-1 — PHASE 1F.1
FORMAL CTL STRUCTURE IMPLEMENTATION AUDIT
ROOT: /content/ettr_ctl_llama
State bank: /content/ettr_ctl_llama/results/llama_phase1d2_full_state_bank.npz
CTL preflight: /content/ettr_ctl_llama/results/llama_phase1f0_ctl_contextual_realization_preflight.json
Output: /content/ettr_ctl_llama/results/llama_phase1f1_formal_ctl_structure_audit.json

FORMAL CTL GOVERNANCE
--------------------------------------------------------------------------------
  CTL structure: FROZEN
  Boolean logic: LOCAL SUBSTRATE ONLY
  context-indexed carriers: REQUIRED
  event domains: REQUIRED
  contextual admissibility: REQUIRED
  triadic admissibility: REQUIRED
  logical transport: REQUIRED
  transport composition: REQUIRED
  triadic coherence carrier: REQUIRED
  semantic valuation: SEPARATE / NOT PRIMITIVE
  Phi_C fitting: FORBIDDEN IN THIS CELL
  numerical transport refitting: FORBIDDEN
  PCA refitting: FORBIDDEN
  test fitting: FORBIDDEN
  empirical CTL satisfaction: NOT CLAIMED



In [39]:
# ============================================================
# ETTR-CTL-LLAMA-1
# PHASE 1F.2A — FROZEN STATE-BANK ROW-ORDER AUDIT
#
# NEW CELL — DO NOT REMOVE PREVIOUS CELLS EXCEPT THE FAILED
# 1F.2 CELL IF YOU ARE REPLACING THE FAILED ATTEMPT.
#
# PURPOSE
# -------
# Determine the actual condition/record ordering of the frozen
# Llama state bank WITHOUT comparing GPT-2 token IDs with Llama
# token IDs.
#
# CRITICAL CROSS-MODEL RULE
# -------------------------
# GPT-2 token IDs and Llama token IDs are model-specific.
#
# Therefore:
#
#       GPT2_target_token_id
#       !=
#       Llama_target_token_id
#
# is expected and MUST NOT be used for cross-model alignment.
#
# This audit instead uses:
#
#   1. operative dataset prompts;
#   2. the Llama tokenizer;
#   3. independently reconstructed Llama p_T/i_T;
#   4. frozen state-bank p_T/i_T;
#   5. candidate row-order structures.
#
# GOVERNING PRINCIPLE
# -------------------
# Mathematical CTL definitions remain frozen.
#
# No:
#   - CTL modification
#   - Boolean substitution
#   - PCA refitting
#   - transport-map refitting
#   - Phi_C fitting
#   - Phi_D fitting
#   - test fitting
#   - semantic guessing
#   - state-vector similarity matching
#   - target-logit matching
#
# If the row order cannot be uniquely recovered from frozen
# metadata, this audit FAILS CLOSED.
#
# CLASSIFICATION
# --------------
# This is an implementation/data-alignment audit.
# A failure here is NOT a mathematical CTL failure.
# ============================================================


import os
import json
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np


# ============================================================
# 0. PATHS
# ============================================================

ROOT = Path(
    "/content/ettr_ctl_llama"
)

RESULTS = ROOT / "results"

STATE_BANK_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank.npz"
)

STATE_MANIFEST_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank_manifest.json"
)

AUTH_PATH = (
    RESULTS /
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

CTL_FORMAL_PATH = (
    RESULTS /
    "llama_phase1f1_formal_ctl_structure_audit.json"
)

OUTPUT_PATH = (
    RESULTS /
    "llama_phase1f2a_state_bank_row_order_audit.json"
)


EXPECTED_DATASET_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)


EXPECTED_STATE_HASHES = {

    "S1":
        "9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783",

    "S2":
        "41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb",

    "S3":
        "173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3",
}


N_RECORDS = 192
N_CONDITIONS = 384


# ============================================================
# 1. HEADER
# ============================================================

print("=" * 80)

print(
    "ETTR-CTL-LLAMA-1 — PHASE 1F.2A"
)

print(
    "FROZEN STATE-BANK ROW-ORDER AUDIT"
)

print("=" * 80)

print(
    f"ROOT: {ROOT}"
)

print(
    f"State bank: {STATE_BANK_PATH}"
)

print(
    f"Output: {OUTPUT_PATH}"
)


# ============================================================
# 2. GOVERNANCE
# ============================================================

print(
    "\nGOVERNANCE"
)

print(
    "-" * 80
)

print(
    "  CTL mathematics: FROZEN"
)

print(
    "  GPT-2 token IDs vs Llama token IDs: NEVER COMPARED"
)

print(
    "  state-bank row order: MUST BE EXPLICITLY AUDITED"
)

print(
    "  state vectors: NOT USED FOR RECORD MATCHING"
)

print(
    "  logits: NOT USED FOR RECORD MATCHING"
)

print(
    "  PCA refitting: FORBIDDEN"
)

print(
    "  transport-map refitting: FORBIDDEN"
)

print(
    "  Phi_C fitting: FORBIDDEN"
)

print(
    "  Phi_D fitting: FORBIDDEN"
)

print(
    "  test fitting: FORBIDDEN"
)

print(
    "  CTL modification: FORBIDDEN"
)


# ============================================================
# 3. REQUIRED ARTIFACTS
# ============================================================

for path in (
    STATE_BANK_PATH,
    STATE_MANIFEST_PATH,
    AUTH_PATH,
    CTL_FORMAL_PATH,
):

    if not path.exists():

        raise FileNotFoundError(
            f"Required artifact missing: {path}"
        )


print(
    "\n[PASS] Required frozen artifacts exist."
)


# ============================================================
# 4. DATASET AUTHORIZATION
# ============================================================

with open(
    AUTH_PATH,
    "r",
    encoding="utf-8"
) as f:

    auth = json.load(
        f
    )


dataset_sha = (
    auth.get(
        "dataset_sha256"
    )
    or
    auth.get(
        "sha256"
    )
)


authorized = bool(
    auth.get(
        "dataset_authorized",
        auth.get(
            "authorized",
            False
        )
    )
)


print(
    "\nDATASET AUTHORIZATION"
)

print(
    "-" * 80
)

print(
    f"  authorized: {authorized}"
)

print(
    f"  SHA: {dataset_sha}"
)


if not authorized:

    raise RuntimeError(
        "Dataset authorization failed."
    )


if dataset_sha != EXPECTED_DATASET_SHA256:

    raise RuntimeError(
        "Dataset SHA mismatch."
    )


print(
    "[PASS] Dataset authorization."
)


# ============================================================
# 5. DATASET DISCOVERY
# ============================================================

candidate_paths = []

for candidate in [
    Path(
        "gpt2_controlled_dataset_v1.1_recovered_r1.json"
    ),

    Path(
        "/content/gpt2_controlled_dataset_v1.1_recovered_r1.json"
    ),

]:

    if candidate.exists():

        candidate_paths.append(
            candidate
        )


if not candidate_paths:

    # Search only within the experiment/runtime locations.
    for base in (
        ROOT,
        Path("/content"),
    ):

        if not base.exists():

            continue

        for path in base.rglob(
            "gpt2_controlled_dataset_v1.1_recovered_r1.json"
        ):

            candidate_paths.append(
                path
            )


candidate_paths = list(
    dict.fromkeys(
        candidate_paths
    )
)


if not candidate_paths:

    raise FileNotFoundError(
        "Operative recovered dataset could not be located."
    )


print(
    "\nDATASET CANDIDATES"
)

print(
    "-" * 80
)

for path in candidate_paths:

    print(
        f"  {path}"
    )


# ============================================================
# 6. LOAD DATASET
# ============================================================

valid_candidates = []


required_fields = {

    "example_id",
    "name_pair_id",
    "name_pair_index",
    "template_id",
    "template_index",
    "split",
    "clean_prompt",
    "corrupt_prompt",
    "clean_target_name",
    "corrupt_target_name",

}


for path in candidate_paths:

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            obj = json.load(
                f
            )


        if isinstance(
            obj,
            list
        ):

            records = obj

        elif isinstance(
            obj,
            dict
        ):

            records = None

            for key in (
                "records",
                "dataset",
                "examples",
                "data",
                "items",
            ):

                if isinstance(
                    obj.get(key),
                    list
                ):

                    records = obj[
                        key
                    ]

                    break

        else:

            records = None


        if (
            records is None
            or
            len(records) != N_RECORDS
        ):

            continue


        if not required_fields.issubset(
            set(
                records[0].keys()
            )
        ):

            continue


        valid_candidates.append(
            (
                path,
                records
            )
        )


    except Exception:

        continue


if not valid_candidates:

    raise RuntimeError(
        "No valid 192-record operative dataset found."
    )


# Require identity equivalence among duplicate copies.

def identity_signature(
    record
):

    return (

        str(
            record[
                "example_id"
            ]
        ),

        str(
            record[
                "name_pair_id"
            ]
        ),

        int(
            record[
                "name_pair_index"
            ]
        ),

        str(
            record[
                "template_id"
            ]
        ),

        int(
            record[
                "template_index"
            ]
        ),

        str(
            record[
                "split"
            ]
        ),

    )


reference_identity = [

    identity_signature(
        record
    )

    for record
    in valid_candidates[0][1]

]


for path, records in valid_candidates[1:]:

    current_identity = [

        identity_signature(
            record
        )

        for record
        in records

    ]

    if current_identity != reference_identity:

        raise RuntimeError(
            "Duplicate dataset copies disagree in identity/order."
        )


dataset_path, records = valid_candidates[0]


print(
    "\n[PASS] Operative dataset identified."
)

print(
    f"  path: {dataset_path}"
)

print(
    f"  records: {len(records)}"
)


# ============================================================
# 7. LOAD FROZEN STATE BANK
# ============================================================

bank = np.load(
    STATE_BANK_PATH,
    allow_pickle=False
)


required_bank_members = {

    "S1",
    "S2",
    "S3",

    "condition_llama_target_ids",
    "condition_p_T",
    "condition_i_T",

    "target_logits",
    "target_logit_ranks",

}


missing_members = (
    required_bank_members
    -
    set(
        bank.files
    )
)


if missing_members:

    raise RuntimeError(
        "State bank missing required members: "
        +
        str(
            sorted(
                missing_members
            )
        )
    )


# ============================================================
# 8. FROZEN STATE HASH FIREWALL
# ============================================================

print(
    "\nSTATE-BANK FIREWALL"
)

print(
    "-" * 80
)


for sector in (
    "S1",
    "S2",
    "S3",
):

    X = np.asarray(
        bank[
            sector
        ]
    )


    if X.shape != (
        N_CONDITIONS,
        3072
    ):

        raise RuntimeError(
            f"{sector} shape mismatch: {X.shape}"
        )


    actual_sha = hashlib.sha256(

        np.ascontiguousarray(
            X
        ).tobytes(
            order="C"
        )

    ).hexdigest()


    expected_sha = (
        EXPECTED_STATE_HASHES[
            sector
        ]
    )


    if actual_sha != expected_sha:

        raise RuntimeError(
            f"{sector} frozen hash mismatch."
        )


    print(
        f"  {sector}: hash_match=True, shape={X.shape}"
    )


print(
    "[PASS] Frozen state-bank hashes."
)


# ============================================================
# 9. LOAD LLAMA METADATA
# ============================================================

state_pT = np.asarray(
    bank[
        "condition_p_T"
    ],
    dtype=np.int64
)


state_iT = np.asarray(
    bank[
        "condition_i_T"
    ],
    dtype=np.int64
)


state_target_ids = np.asarray(
    bank[
        "condition_llama_target_ids"
    ],
    dtype=np.int64
)


if (
    state_pT.shape[0] != N_CONDITIONS
    or
    state_iT.shape[0] != N_CONDITIONS
    or
    state_target_ids.shape[0] != N_CONDITIONS
):

    raise RuntimeError(
        "Frozen Llama metadata row count mismatch."
    )


print(
    "\n[PASS] Frozen Llama metadata loaded."
)


# ============================================================
# 10. LOAD LLAMA TOKENIZER
# ============================================================
#
# Use the exact tokenizer associated with the frozen Llama
# checkpoint.
#
# This is not model refitting and does not alter the state bank.
# ============================================================

try:

    from transformers import AutoTokenizer

except Exception as exc:

    raise RuntimeError(
        "Transformers is required for tokenizer audit."
    ) from exc


MODEL_ID = (
    "meta-llama/Llama-3.2-3B"
)


print(
    "\nLLAMA TOKENIZER"
)

print(
    "-" * 80
)

print(
    f"  model/tokenizer: {MODEL_ID}"
)


try:

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        use_fast=True,
    )

except Exception as exc:

    raise RuntimeError(
        "Could not load the exact Llama tokenizer required "
        "for frozen target-position reconstruction."
    ) from exc


print(
    f"  tokenizer class: {tokenizer.__class__.__name__}"
)

print(
    f"  vocab size: {len(tokenizer)}"
)


# ============================================================
# 11. RECONSTRUCT EXPECTED LLAMA TARGET POSITIONS
# ============================================================
#
# For every dataset record:
#
#       P = clean_prompt
#
#       p_T = L(P)
#
#       i_T = L(P) - 1
#
# The target token itself is NOT included in the prompt.
#
# Clean and corrupt prompts should have equal length within
# each controlled pair.
# ============================================================

expected_clean_pT = np.zeros(
    N_RECORDS,
    dtype=np.int64
)

expected_corrupt_pT = np.zeros(
    N_RECORDS,
    dtype=np.int64
)


expected_clean_iT = np.zeros(
    N_RECORDS,
    dtype=np.int64
)

expected_corrupt_iT = np.zeros(
    N_RECORDS,
    dtype=np.int64
)


expected_clean_target_ids = np.zeros(
    N_RECORDS,
    dtype=np.int64
)

expected_corrupt_target_ids = np.zeros(
    N_RECORDS,
    dtype=np.int64
)


for i, record in enumerate(
    records
):

    clean_prompt = str(
        record[
            "clean_prompt"
        ]
    )

    corrupt_prompt = str(
        record[
            "corrupt_prompt"
        ]
    )


    clean_tokens = tokenizer(
        clean_prompt,
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    corrupt_tokens = tokenizer(
        corrupt_prompt,
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    expected_clean_pT[i] = len(
        clean_tokens
    )

    expected_corrupt_pT[i] = len(
        corrupt_tokens
    )


    expected_clean_iT[i] = (
        expected_clean_pT[i]
        -
        1
    )

    expected_corrupt_iT[i] = (
        expected_corrupt_pT[i]
        -
        1
    )


    clean_target_text = (
        " "
        +
        str(
            record[
                "clean_target_name"
            ]
        )
    )

    corrupt_target_text = (
        " "
        +
        str(
            record[
                "corrupt_target_name"
            ]
        )
    )


    clean_target_tokens = tokenizer(
        clean_target_text,
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    corrupt_target_tokens = tokenizer(
        corrupt_target_text,
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    if len(
        clean_target_tokens
    ) < 1:

        raise RuntimeError(
            f"Empty clean target tokenization at record {i}."
        )


    if len(
        corrupt_target_tokens
    ) < 1:

        raise RuntimeError(
            f"Empty corrupt target tokenization at record {i}."
        )


    expected_clean_target_ids[i] = int(
        clean_target_tokens[0]
    )

    expected_corrupt_target_ids[i] = int(
        corrupt_target_tokens[0]
    )


print(
    "\nEXPECTED LLAMA TOKENIZATION"
)

print(
    "-" * 80
)

print(
    f"  clean p_T range: "
    f"{expected_clean_pT.min()}–{expected_clean_pT.max()}"
)

print(
    f"  corrupt p_T range: "
    f"{expected_corrupt_pT.min()}–{expected_corrupt_pT.max()}"
)


pair_pT_equal = bool(
    np.all(
        expected_clean_pT
        ==
        expected_corrupt_pT
    )
)


print(
    f"  clean/corrupt p_T equal by dataset record: "
    f"{pair_pT_equal}"
)


if not pair_pT_equal:

    mismatch_count = int(
        np.sum(
            expected_clean_pT
            !=
            expected_corrupt_pT
        )
    )

    raise RuntimeError(
        "The operative controlled dataset itself does not produce "
        "equal clean/corrupt prompt lengths under the exact Llama "
        f"tokenizer. Mismatched records: {mismatch_count}/192."
    )


print(
    "[PASS] Dataset-level Llama target-position reconstruction."
)


# ============================================================
# 12. TARGET-ID CROSS-MODEL FIREWALL
# ============================================================
#
# Explicitly demonstrate that GPT-2 target IDs are not used.
#
# The recovered dataset's target-token fields are GPT-2 metadata.
# The freshly reconstructed IDs above are Llama metadata.
#
# They are deliberately NOT compared.
# ============================================================

print(
    "\nCROSS-MODEL TOKEN-ID FIREWALL"
)

print(
    "-" * 80
)

print(
    "  GPT-2 target IDs used for Llama row matching: NO"
)

print(
    "  Llama target IDs independently reconstructed: YES"
)

print(
    "  GPT-2/Llama target-ID equality tested: NO"
)

print(
    "  cross-model token-ID alignment assumed: NO"
)

print(
    "[PASS] Cross-model token-ID contamination prevented."
)


# ============================================================
# 13. CANDIDATE ROW-ORDER DEFINITIONS
# ============================================================
#
# Candidate 1:
#
#   records 0..191 clean
#   records 192..383 corrupt
#
# Candidate 2:
#
#   interleaved:
#       row 2i     = clean record i
#       row 2i + 1 = corrupt record i
#
# Candidate 3:
#
#   clean block reversed + corrupt block reversed
#
# Candidate 4:
#
#   interleaved with reversed record order
#
# These are diagnostic candidates only.
#
# We do not select a candidate merely because it produces a
# convenient result.
# ============================================================

candidate_mappings = {

    "clean_block_corrupt_block":

        (
            np.arange(
                0,
                N_RECORDS,
                dtype=np.int64
            ),

            np.arange(
                N_RECORDS,
                N_CONDITIONS,
                dtype=np.int64
            )
        ),


    "interleaved_clean_corrupt":

        (
            np.arange(
                0,
                N_CONDITIONS,
                2,
                dtype=np.int64
            ),

            np.arange(
                1,
                N_CONDITIONS,
                2,
                dtype=np.int64
            )
        ),


    "reversed_clean_block_reversed_corrupt_block":

        (
            np.arange(
                N_RECORDS - 1,
                -1,
                -1,
                dtype=np.int64
            ),

            np.arange(
                N_CONDITIONS - 1,
                N_RECORDS - 1,
                -1,
                dtype=np.int64
            )
        ),


    "interleaved_reversed_record_order":

        (
            np.arange(
                N_RECORDS - 1,
                -1,
                -1,
                dtype=np.int64
            )
            * 2,

            np.arange(
                N_RECORDS - 1,
                -1,
                -1,
                dtype=np.int64
            )
            * 2
            +
            1
        ),

}


# ============================================================
# 14. SCORE CANDIDATES USING ONLY LLAMA POSITION METADATA
# ============================================================

candidate_scores = {}


for name, (
    clean_rows,
    corrupt_rows
) in candidate_mappings.items():

    clean_pT_match = int(
        np.sum(
            state_pT[
                clean_rows
            ]
            ==
            expected_clean_pT
        )
    )


    corrupt_pT_match = int(
        np.sum(
            state_pT[
                corrupt_rows
            ]
            ==
            expected_corrupt_pT
        )
    )


    clean_iT_match = int(
        np.sum(
            state_iT[
                clean_rows
            ]
            ==
            expected_clean_iT
        )
    )


    corrupt_iT_match = int(
        np.sum(
            state_iT[
                corrupt_rows
            ]
            ==
            expected_corrupt_iT
        )
    )


    candidate_scores[
        name
    ] = {

        "clean_pT_matches":
            clean_pT_match,

        "corrupt_pT_matches":
            corrupt_pT_match,

        "clean_iT_matches":
            clean_iT_match,

        "corrupt_iT_matches":
            corrupt_iT_match,

        "total_pT_matches":
            clean_pT_match
            +
            corrupt_pT_match,

        "total_iT_matches":
            clean_iT_match
            +
            corrupt_iT_match,

    }


print(
    "\nROW-ORDER CANDIDATE DIAGNOSTIC"
)

print(
    "-" * 80
)

for name, score in candidate_scores.items():

    print(
        f"  {name}: "
        f"p_T={score['total_pT_matches']}/384, "
        f"i_T={score['total_iT_matches']}/384"
    )


# ============================================================
# 15. TARGET-ID DIAGNOSTIC — LLAMA ONLY
# ============================================================
#
# If a candidate row ordering is correct, its Llama target IDs
# should agree with independently reconstructed Llama target
# IDs.
#
# This comparison is legitimate because BOTH sides are now
# Llama tokenizer IDs.
# ============================================================

for name, (
    clean_rows,
    corrupt_rows
) in candidate_mappings.items():

    clean_id_match = int(
        np.sum(
            state_target_ids[
                clean_rows
            ]
            ==
            expected_clean_target_ids
        )
    )


    corrupt_id_match = int(
        np.sum(
            state_target_ids[
                corrupt_rows
            ]
            ==
            expected_corrupt_target_ids
        )
    )


    candidate_scores[
        name
    ][
        "clean_llama_target_id_matches"
    ] = clean_id_match


    candidate_scores[
        name
    ][
        "corrupt_llama_target_id_matches"
    ] = corrupt_id_match


    candidate_scores[
        name
    ][
        "total_llama_target_id_matches"
    ] = (
        clean_id_match
        +
        corrupt_id_match
    )


print(
    "\nLLAMA TARGET-ID ROW-ORDER DIAGNOSTIC"
)

print(
    "-" * 80
)

for name, score in candidate_scores.items():

    print(
        f"  {name}: "
        f"Llama target-ID matches="
        f"{score['total_llama_target_id_matches']}/384"
    )


# ============================================================
# 16. SELECT ONLY AN EXACTLY VERIFIED CANDIDATE
# ============================================================
#
# A candidate is accepted only if:
#
#   p_T:       384/384
#   i_T:       384/384
#   Llama ID:  384/384
#
# This is intentionally stringent.
# ============================================================

exact_candidates = []


for name, score in candidate_scores.items():

    exact = (

        score[
            "total_pT_matches"
        ]
        ==
        N_CONDITIONS

        and

        score[
            "total_iT_matches"
        ]
        ==
        N_CONDITIONS

        and

        score[
            "total_llama_target_id_matches"
        ]
        ==
        N_CONDITIONS

    )


    if exact:

        exact_candidates.append(
            name
        )


print(
    "\nEXACT ROW-ORDER CANDIDATES"
)

print(
    "-" * 80
)

print(
    f"  exact candidates: "
    f"{len(exact_candidates)}"
)

for name in exact_candidates:

    print(
        f"  {name}"
    )


# ============================================================
# 17. FAIL-CLOSED RULE
# ============================================================

if len(
    exact_candidates
) != 1:

    artifact = {

        "experiment_id":
            "ETTR-CTL-LLAMA-1",

        "phase":
            "1F.2A",

        "classification":
            "FROZEN_STATE_BANK_ROW_ORDER_NOT_UNIQUELY_VERIFIED",

        "dataset_sha256":
            dataset_sha,

        "candidate_scores":
            candidate_scores,

        "cross_model_token_id_rule":
            "GPT2_AND_LLAMA_TOKEN_IDS_NOT_COMPARED",

        "llama_tokenizer":
            MODEL_ID,

        "governance":
            {

                "pca_refitting":
                    False,

                "transport_refitting":
                    False,

                "phi_c_fitting":
                    False,

                "phi_d_fitting":
                    False,

                "test_fitting":
                    False,

                "state_vector_matching":
                    False,

                "logit_matching":
                    False,

                "cross_model_token_id_matching":
                    False,

            },

        "scientific_interpretation":
            "IMPLEMENTATION_ALIGNMENT_REMAINS_UNRESOLVED",

        "ctl_mathematical_status":
            "UNCHANGED",

    }


    with open(
        OUTPUT_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            artifact,
            f,
            indent=2
        )


    raise RuntimeError(
        "\n"
        "FROZEN STATE-BANK ROW ORDER WAS NOT UNIQUELY VERIFIED.\n\n"
        "No scientific CTL inference was made.\n"
        "No numerical state was modified.\n"
        "No GPT-2/Llama token-ID comparison was used.\n"
        "The experiment is intentionally stopped until the frozen "
        "Phase 1D.2 extraction ordering can be established."
    )


# ============================================================
# 18. FREEZE VERIFIED ROW ORDER
# ============================================================

selected_name = (
    exact_candidates[0]
)


clean_rows, corrupt_rows = (
    candidate_mappings[
        selected_name
    ]
)


print(
    "\n[PASS] EXACT FROZEN ROW ORDER VERIFIED."
)

print(
    f"  selected convention: {selected_name}"
)


# ============================================================
# 19. PAIRED POSITION FIREWALL
# ============================================================

aligned_clean_pT = state_pT[
    clean_rows
]

aligned_corrupt_pT = state_pT[
    corrupt_rows
]


aligned_clean_iT = state_iT[
    clean_rows
]

aligned_corrupt_iT = state_iT[
    corrupt_rows
]


pT_pair_equal = bool(
    np.all(
        aligned_clean_pT
        ==
        aligned_corrupt_pT
    )
)


iT_pair_equal = bool(
    np.all(
        aligned_clean_iT
        ==
        aligned_corrupt_iT
    )
)


if not pT_pair_equal:

    raise RuntimeError(
        "Verified row order still fails clean/corrupt p_T equality."
    )


if not iT_pair_equal:

    raise RuntimeError(
        "Verified row order still fails clean/corrupt i_T equality."
    )


print(
    "\nPAIRED POSITION FIREWALL"
)

print(
    "-" * 80
)

print(
    "  p_T equality: PASS"
)

print(
    "  i_T equality: PASS"
)


# ============================================================
# 20. SERIALIZE SUCCESS
# ============================================================

artifact = {

    "experiment_id":
        "ETTR-CTL-LLAMA-1",

    "phase":
        "1F.2A",

    "title":
        "Frozen State-Bank Row-Order Audit",

    "classification":
        "FROZEN_STATE_BANK_ROW_ORDER_VERIFIED",

    "dataset_sha256":
        dataset_sha,

    "dataset_path":
        str(
            dataset_path
        ),

    "selected_row_order":
        selected_name,

    "clean_rows":
        clean_rows.tolist(),

    "corrupt_rows":
        corrupt_rows.tolist(),

    "candidate_scores":
        candidate_scores,

    "pair_position_audit":
        {

            "p_T_equal":
                pT_pair_equal,

            "i_T_equal":
                iT_pair_equal,

        },

    "cross_model_token_id_firewall":
        {

            "GPT2_Llama_ids_compared":
                False,

            "Llama_ids_independently_reconstructed":
                True,

            "Llama_ids_used_for_verified_alignment":
                True,

        },

    "governance":
        {

            "pca_refitting":
                False,

            "transport_refitting":
                False,

            "phi_c_fitting":
                False,

            "phi_d_fitting":
                False,

            "test_fitting":
                False,

            "state_vector_matching":
                False,

            "logit_matching":
                False,

            "cross_model_token_id_matching":
                False,

        },

    "scientific_status":
        {

            "row_order":
                "VERIFIED",

            "semantic_interface":
                "READY_FOR_1F2",

            "CTL_mathematics":
                "UNCHANGED",

            "CTL_empirical_satisfaction":
                "NOT_TESTED",

        },

}


with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 21. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "FINAL PHASE 1F.2A STATUS"
)

print(
    "=" * 80
)

print(
    "  FROZEN_STATE_BANK_ROW_ORDER_VERIFIED"
)

print(
    f"  selected convention: {selected_name}"
)

print(
    "  clean/corrupt p_T pairing: PASS"
)

print(
    "  clean/corrupt i_T pairing: PASS"
)

print(
    "  Llama target-ID correspondence: PASS"
)

print(
    "  GPT-2/Llama target-ID comparison: NOT USED"
)

print(
    "  CTL mathematics: UNCHANGED"
)

print(
    "  CTL empirical satisfaction: NOT TESTED"
)

print(
    "  artifact:"
)

print(
    f"    {OUTPUT_PATH}"
)

print(
    "=" * 80
)

ETTR-CTL-LLAMA-1 — PHASE 1F.2A
FROZEN STATE-BANK ROW-ORDER AUDIT
ROOT: /content/ettr_ctl_llama
State bank: /content/ettr_ctl_llama/results/llama_phase1d2_full_state_bank.npz
Output: /content/ettr_ctl_llama/results/llama_phase1f2a_state_bank_row_order_audit.json

GOVERNANCE
--------------------------------------------------------------------------------
  CTL mathematics: FROZEN
  GPT-2 token IDs vs Llama token IDs: NEVER COMPARED
  state-bank row order: MUST BE EXPLICITLY AUDITED
  state vectors: NOT USED FOR RECORD MATCHING
  logits: NOT USED FOR RECORD MATCHING
  PCA refitting: FORBIDDEN
  transport-map refitting: FORBIDDEN
  Phi_C fitting: FORBIDDEN
  Phi_D fitting: FORBIDDEN
  test fitting: FORBIDDEN
  CTL modification: FORBIDDEN

[PASS] Required frozen artifacts exist.

DATASET AUTHORIZATION
--------------------------------------------------------------------------------
  authorized: True
  SHA: 7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d
[PASS] Dataset autho

In [40]:
# ============================================================
# ETTR-CTL-LLAMA-1
# PHASE 1F.2 — CTL SEMANTIC REALIZATION / ADMISSIBILITY AUDIT
# CORRECTED AFTER PHASE 1F.2A ROW-ORDER VERIFICATION
#
# NEW CELL — DO NOT REMOVE PREVIOUS CELLS.
#
# VERIFIED FROZEN ROW ORDER:
#
#       bank[2i]   = clean condition C_i
#       bank[2i+1] = corrupt condition D_i
#
# for i = 0,...,191.
#
# GOVERNING PRINCIPLE
# -------------------
# CTL mathematics remains authoritative and frozen.
#
# This phase establishes an empirical semantic interface only.
#
# It does NOT:
#   - redefine CTL;
#   - equate CTL with Boolean logic;
#   - define admissibility from hidden-state thresholds;
#   - define triadic admissibility as pairwise conjunction;
#   - identify numerical transport with logical transport;
#   - fit Phi_C;
#   - fit Phi_D;
#   - refit PCA;
#   - refit transport maps;
#   - fit on test data;
#   - claim triadic irreducibility;
#   - claim contextual geometry.
#
# Semantic observations are kept separate from:
#
#       admissibility
#       logical transport
#       coherence
#       Phi_C
#
# A semantic interface is therefore NOT itself a proof of CTL
# satisfaction.
# ============================================================


import os
import json
import hashlib
from pathlib import Path
from dataclasses import dataclass
from collections import Counter

import numpy as np


# ============================================================
# 0. PATHS
# ============================================================

ROOT = Path(
    "/content/ettr_ctl_llama"
)

RESULTS = ROOT / "results"

STATE_BANK_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank.npz"
)

STATE_MANIFEST_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank_manifest.json"
)

AUTH_PATH = (
    RESULTS /
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

CTL_FORMAL_PATH = (
    RESULTS /
    "llama_phase1f1_formal_ctl_structure_audit.json"
)

ROW_ORDER_PATH = (
    RESULTS /
    "llama_phase1f2a_state_bank_row_order_audit.json"
)

OUTPUT_PATH = (
    RESULTS /
    "llama_phase1f2_ctl_semantic_realization_audit.json"
)


EXPECTED_DATASET_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)


EXPECTED_STATE_HASHES = {

    "S1":
        "9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783",

    "S2":
        "41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb",

    "S3":
        "173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3",

}


SECTORS = (
    "S1",
    "S2",
    "S3",
)


N_RECORDS = 192
N_CONDITIONS = 384


# ============================================================
# 1. HEADER
# ============================================================

print(
    "=" * 80
)

print(
    "ETTR-CTL-LLAMA-1 — PHASE 1F.2"
)

print(
    "CTL SEMANTIC REALIZATION / ADMISSIBILITY AUDIT"
)

print(
    "CORRECTED AFTER VERIFIED INTERLEAVED ROW ORDER"
)

print(
    "=" * 80
)

print(
    f"ROOT: {ROOT}"
)

print(
    f"State bank: {STATE_BANK_PATH}"
)

print(
    f"Formal CTL audit: {CTL_FORMAL_PATH}"
)

print(
    f"Row-order audit: {ROW_ORDER_PATH}"
)

print(
    f"Output: {OUTPUT_PATH}"
)


# ============================================================
# 2. GOVERNANCE
# ============================================================

print(
    "\nGOVERNANCE"
)

print(
    "-" * 80
)

print(
    "  CTL formal structure: FROZEN"
)

print(
    "  semantic valuation: SEPARATE REALIZATION LAYER"
)

print(
    "  Boolean truth as CTL definition: FORBIDDEN"
)

print(
    "  hidden-state threshold as semantic truth: FORBIDDEN"
)

print(
    "  numerical transport as logical transport: FORBIDDEN"
)

print(
    "  PCA refitting: FORBIDDEN"
)

print(
    "  transport-map refitting: FORBIDDEN"
)

print(
    "  Phi_C fitting: FORBIDDEN"
)

print(
    "  Phi_D fitting: FORBIDDEN"
)

print(
    "  test fitting: FORBIDDEN"
)

print(
    "  state-bank row-order inference: FORBIDDEN"
)

print(
    "  verified row-order artifact: REQUIRED"
)

print(
    "  contextual admissibility: NOT IDENTIFIED"
)

print(
    "  triadic admissibility: NOT IDENTIFIED"
)

print(
    "  logical transport covariance: NOT TESTED"
)

print(
    "  triadic coherence: NOT TESTED"
)

print(
    "  triadic irreducibility: NOT TESTED"
)

print(
    "  contextual geometry: NOT TESTED"
)


# ============================================================
# 3. REQUIRED ARTIFACTS
# ============================================================

required_paths = [

    STATE_BANK_PATH,
    STATE_MANIFEST_PATH,
    AUTH_PATH,
    CTL_FORMAL_PATH,
    ROW_ORDER_PATH,

]


missing = [

    str(
        path
    )

    for path in required_paths

    if not path.exists()

]


if missing:

    raise FileNotFoundError(
        "Required artifact(s) missing:\n"
        +
        "\n".join(
            missing
        )
    )


print(
    "\n[PASS] Required frozen artifacts exist."
)


# ============================================================
# 4. FORMAL CTL PREDECESSOR
# ============================================================

with open(
    CTL_FORMAL_PATH,
    "r",
    encoding="utf-8"
) as f:

    ctl_formal = json.load(
        f
    )


formal_classification = ctl_formal.get(
    "final_classification"
)


if formal_classification not in (
    "FORMAL_CTL_STRUCTURE_IMPLEMENTATION_PASS",
    "CTL_FORMAL_STRUCTURE_IMPLEMENTATION_PASS",
):

    raise RuntimeError(
        "Phase 1F.1 formal CTL implementation is not frozen as PASS. "
        f"Observed: {formal_classification}"
    )


print(
    "[PASS] Formal CTL implementation predecessor."
)


# ============================================================
# 5. ROW-ORDER PREDECESSOR
# ============================================================

with open(
    ROW_ORDER_PATH,
    "r",
    encoding="utf-8"
) as f:

    row_order = json.load(
        f
    )


row_order_classification = row_order.get(
    "classification"
)


if row_order_classification != (
    "FROZEN_STATE_BANK_ROW_ORDER_VERIFIED"
):

    raise RuntimeError(
        "Phase 1F.2A row-order verification is not frozen as PASS. "
        f"Observed: {row_order_classification}"
    )


selected_row_order = row_order.get(
    "selected_row_order"
)


if selected_row_order != (
    "interleaved_clean_corrupt"
):

    raise RuntimeError(
        "Unexpected verified row order. "
        f"Observed: {selected_row_order}"
    )


verified_clean_rows = np.asarray(
    row_order.get(
        "clean_rows"
    ),
    dtype=np.int64
)


verified_corrupt_rows = np.asarray(
    row_order.get(
        "corrupt_rows"
    ),
    dtype=np.int64
)


expected_clean_rows = np.arange(
    0,
    N_CONDITIONS,
    2,
    dtype=np.int64
)


expected_corrupt_rows = np.arange(
    1,
    N_CONDITIONS,
    2,
    dtype=np.int64
)


if not np.array_equal(
    verified_clean_rows,
    expected_clean_rows
):

    raise RuntimeError(
        "Frozen clean row mapping differs from the verified "
        "interleaved convention."
    )


if not np.array_equal(
    verified_corrupt_rows,
    expected_corrupt_rows
):

    raise RuntimeError(
        "Frozen corrupt row mapping differs from the verified "
        "interleaved convention."
    )


print(
    "\nROW-ORDER PREDECESSOR"
)

print(
    "-" * 80
)

print(
    "  classification: FROZEN_STATE_BANK_ROW_ORDER_VERIFIED"
)

print(
    "  convention: bank[2i]=clean, bank[2i+1]=corrupt"
)

print(
    "[PASS] Verified Phase 1F.2A row-order artifact consumed."
)


# ============================================================
# 6. DATASET AUTHORIZATION
# ============================================================

with open(
    AUTH_PATH,
    "r",
    encoding="utf-8"
) as f:

    auth = json.load(
        f
    )


authorized = bool(
    auth.get(
        "dataset_authorized",
        auth.get(
            "authorized",
            False
        )
    )
)


dataset_sha = (
    auth.get(
        "dataset_sha256"
    )
    or
    auth.get(
        "sha256"
    )
)


print(
    "\nDATASET AUTHORIZATION"
)

print(
    "-" * 80
)

print(
    f"  authorized: {authorized}"
)

print(
    f"  recorded SHA: {dataset_sha}"
)

print(
    f"  expected SHA: {EXPECTED_DATASET_SHA256}"
)


if not authorized:

    raise RuntimeError(
        "Dataset authorization failed."
    )


if dataset_sha != EXPECTED_DATASET_SHA256:

    raise RuntimeError(
        "Dataset SHA mismatch."
    )


print(
    "[PASS] Dataset authorization."
)


# ============================================================
# 7. LOCATE OPERATIVE DATASET
# ============================================================

dataset_candidates = [

    Path(
        "gpt2_controlled_dataset_v1.1_recovered_r1.json"
    ),

    Path(
        "/content/gpt2_controlled_dataset_v1.1_recovered_r1.json"
    ),

]


dataset_candidates = [

    path

    for path in dataset_candidates

    if path.exists()

]


if not dataset_candidates:

    for base in (
        ROOT,
        Path("/content"),
    ):

        if base.exists():

            dataset_candidates.extend(

                base.rglob(
                    "gpt2_controlled_dataset_v1.1_recovered_r1.json"
                )

            )


dataset_candidates = list(
    dict.fromkeys(
        dataset_candidates
    )
)


if not dataset_candidates:

    raise FileNotFoundError(
        "Operative recovered dataset not found."
    )


def load_dataset(
    path
):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        obj = json.load(
            f
        )


    if isinstance(
        obj,
        list
    ):

        return obj


    if isinstance(
        obj,
        dict
    ):

        for key in (
            "records",
            "dataset",
            "examples",
            "data",
            "items",
        ):

            if isinstance(
                obj.get(key),
                list
            ):

                return obj[
                    key
                ]


    return None


valid_dataset_candidates = []


required_fields = {

    "example_id",
    "name_pair_id",
    "name_pair_index",
    "template_id",
    "template_index",
    "split",
    "clean_prompt",
    "corrupt_prompt",
    "clean_target_name",
    "corrupt_target_name",

}


for path in dataset_candidates:

    try:

        records_candidate = load_dataset(
            path
        )

    except Exception:

        continue


    if records_candidate is None:

        continue


    if len(
        records_candidate
    ) != N_RECORDS:

        continue


    if not required_fields.issubset(
        set(
            records_candidate[0].keys()
        )
    ):

        continue


    valid_dataset_candidates.append(
        (
            path,
            records_candidate
        )
    )


if not valid_dataset_candidates:

    raise RuntimeError(
        "No schema-valid 192-record operative dataset found."
    )


def record_identity(
    record
):

    return (

        str(
            record[
                "example_id"
            ]
        ),

        str(
            record[
                "name_pair_id"
            ]
        ),

        int(
            record[
                "name_pair_index"
            ]
        ),

        str(
            record[
                "template_id"
            ]
        ),

        int(
            record[
                "template_index"
            ]
        ),

        str(
            record[
                "split"
            ]
        ),

    )


reference_identity = [

    record_identity(
        r
    )

    for r in valid_dataset_candidates[0][1]

]


for path, candidate_records in valid_dataset_candidates[1:]:

    candidate_identity = [

        record_identity(
            r
        )

        for r in candidate_records

    ]

    if candidate_identity != reference_identity:

        raise RuntimeError(
            "Duplicate dataset copies disagree in identity/order."
        )


dataset_path, records = (
    valid_dataset_candidates[0]
)


print(
    "\nOPERATIVE DATASET"
)

print(
    "-" * 80
)

print(
    f"  path: {dataset_path}"
)

print(
    f"  records: {len(records)}"
)


# ============================================================
# 8. DATASET SPLIT FIREWALL
# ============================================================

split_counts = Counter(
    str(
        r[
            "split"
        ]
    )
    for r in records
)


print(
    f"  split counts: {dict(split_counts)}"
)


expected_split_counts = {

    "train":
        96,

    "calibration":
        48,

    "test":
        48,

}


if dict(
    split_counts
) != expected_split_counts:

    raise RuntimeError(
        "Dataset split counts do not match frozen design."
    )


print(
    "[PASS] Operative dataset structure."
)


# ============================================================
# 9. LOAD FROZEN STATE BANK
# ============================================================

bank = np.load(
    STATE_BANK_PATH,
    allow_pickle=False
)


for sector in SECTORS:

    if sector not in bank.files:

        raise RuntimeError(
            f"Missing frozen sector bank: {sector}"
        )


    X = np.asarray(
        bank[
            sector
        ]
    )


    if X.shape != (
        N_CONDITIONS,
        3072
    ):

        raise RuntimeError(
            f"{sector} shape mismatch: {X.shape}"
        )


    actual_hash = hashlib.sha256(

        np.ascontiguousarray(
            X
        ).tobytes(
            order="C"
        )

    ).hexdigest()


    if actual_hash != EXPECTED_STATE_HASHES[
        sector
    ]:

        raise RuntimeError(
            f"{sector} frozen hash mismatch."
        )


print(
    "\nSTATE-BANK FIREWALL"
)

print(
    "-" * 80
)

for sector in SECTORS:

    print(
        f"  {sector}: "
        f"shape=(384,3072), "
        f"hash_match=True"
    )


print(
    "[PASS] Frozen state-bank integrity."
)


# ============================================================
# 10. LOAD FROZEN SEMANTIC ARRAYS
# ============================================================

target_logits = np.asarray(
    bank[
        "target_logits"
    ],
    dtype=np.float64
)


target_ranks = np.asarray(
    bank[
        "target_logit_ranks"
    ],
    dtype=np.int64
)


state_target_ids = np.asarray(
    bank[
        "condition_llama_target_ids"
    ],
    dtype=np.int64
)


state_pT = np.asarray(
    bank[
        "condition_p_T"
    ],
    dtype=np.int64
)


state_iT = np.asarray(
    bank[
        "condition_i_T"
    ],
    dtype=np.int64
)


for name, array in (
    ("target_logits", target_logits),
    ("target_ranks", target_ranks),
    ("condition_llama_target_ids", state_target_ids),
    ("condition_p_T", state_pT),
    ("condition_i_T", state_iT),
):

    if array.shape != (
        N_CONDITIONS,
    ):

        raise RuntimeError(
            f"{name} has unexpected shape: {array.shape}"
        )


print(
    "\nSEMANTIC ARTIFACTS"
)

print(
    "-" * 80
)

print(
    "  target logits: available"
)

print(
    "  target ranks: available"
)

print(
    "  Llama target IDs: available"
)

print(
    "  p_T: available"
)

print(
    "  i_T: available"
)

print(
    "[PASS] Frozen semantic artifacts available."
)


# ============================================================
# 11. EXPLICIT CLEAN/CORRUPT ALIGNMENT
# ============================================================
#
# This is the central correction.
#
# We use the VERIFIED row-order artifact, not an inferred
# ordering and not token-ID similarity.
# ============================================================

clean_rows = verified_clean_rows
corrupt_rows = verified_corrupt_rows


clean_pT = state_pT[
    clean_rows
]

corrupt_pT = state_pT[
    corrupt_rows
]


clean_iT = state_iT[
    clean_rows
]

corrupt_iT = state_iT[
    corrupt_rows
]


pT_equal = bool(
    np.all(
        clean_pT
        ==
        corrupt_pT
    )
)


iT_equal = bool(
    np.all(
        clean_iT
        ==
        corrupt_iT
    )
)


if not pT_equal:

    raise RuntimeError(
        "Verified row order does not preserve clean/corrupt p_T equality."
    )


if not iT_equal:

    raise RuntimeError(
        "Verified row order does not preserve clean/corrupt i_T equality."
    )


print(
    "\nVERIFIED CONDITION ALIGNMENT"
)

print(
    "-" * 80
)

print(
    "  clean rows: 0,2,4,...,382"
)

print(
    "  corrupt rows: 1,3,5,...,383"
)

print(
    f"  paired p_T equality: {pT_equal}"
)

print(
    f"  paired i_T equality: {iT_equal}"
)

print(
    "[PASS] Frozen clean/corrupt correspondence."
)


# ============================================================
# 12. LLAMA TARGET-ID RECONSTRUCTION
# ============================================================
#
# GPT-2 target IDs are deliberately ignored.
#
# We independently reconstruct the first target token under
# the Llama tokenizer from:
#
#       " " + target_name
#
# as established in Phase 1C.3.
#
# This is used only as a metadata integrity check.
# ============================================================

try:

    from transformers import AutoTokenizer

except Exception as exc:

    raise RuntimeError(
        "Transformers unavailable."
    ) from exc


MODEL_ID = (
    "meta-llama/Llama-3.2-3B"
)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)


expected_clean_llama_ids = np.zeros(
    N_RECORDS,
    dtype=np.int64
)

expected_corrupt_llama_ids = np.zeros(
    N_RECORDS,
    dtype=np.int64
)


expected_clean_prompt_lengths = np.zeros(
    N_RECORDS,
    dtype=np.int64
)

expected_corrupt_prompt_lengths = np.zeros(
    N_RECORDS,
    dtype=np.int64
)


for i, record in enumerate(
    records
):

    clean_prompt_tokens = tokenizer(
        str(
            record[
                "clean_prompt"
            ]
        ),
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    corrupt_prompt_tokens = tokenizer(
        str(
            record[
                "corrupt_prompt"
            ]
        ),
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    clean_target_tokens = tokenizer(
        " "
        +
        str(
            record[
                "clean_target_name"
            ]
        ),
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    corrupt_target_tokens = tokenizer(
        " "
        +
        str(
            record[
                "corrupt_target_name"
            ]
        ),
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    if not clean_target_tokens:

        raise RuntimeError(
            f"Empty Llama clean target tokenization at record {i}."
        )


    if not corrupt_target_tokens:

        raise RuntimeError(
            f"Empty Llama corrupt target tokenization at record {i}."
        )


    expected_clean_prompt_lengths[i] = len(
        clean_prompt_tokens
    )

    expected_corrupt_prompt_lengths[i] = len(
        corrupt_prompt_tokens
    )


    expected_clean_llama_ids[i] = int(
        clean_target_tokens[0]
    )

    expected_corrupt_llama_ids[i] = int(
        corrupt_target_tokens[0]
    )


expected_clean_pT = (
    expected_clean_prompt_lengths
)

expected_corrupt_pT = (
    expected_corrupt_prompt_lengths
)


expected_clean_iT = (
    expected_clean_pT
    -
    1
)

expected_corrupt_iT = (
    expected_corrupt_pT
    -
    1
)


dataset_pT_equal = bool(
    np.all(
        expected_clean_pT
        ==
        expected_corrupt_pT
    )
)


dataset_iT_equal = bool(
    np.all(
        expected_clean_iT
        ==
        expected_corrupt_iT
    )
)


if not dataset_pT_equal:

    raise RuntimeError(
        "Dataset-level clean/corrupt Llama p_T mismatch."
    )


if not dataset_iT_equal:

    raise RuntimeError(
        "Dataset-level clean/corrupt Llama i_T mismatch."
    )


print(
    "\nLLAMA TOKENIZATION RECONSTRUCTION"
)

print(
    "-" * 80
)

print(
    f"  clean p_T range: "
    f"{expected_clean_pT.min()}–{expected_clean_pT.max()}"
)

print(
    f"  corrupt p_T range: "
    f"{expected_corrupt_pT.min()}–{expected_corrupt_pT.max()}"
)

print(
    "  clean/corrupt dataset p_T equality: PASS"
)

print(
    "  clean/corrupt dataset i_T equality: PASS"
)


# ============================================================
# 13. FROZEN BANK METADATA VS TOKENIZER
# ============================================================

bank_clean_pT_match = bool(
    np.all(
        clean_pT
        ==
        expected_clean_pT
    )
)


bank_corrupt_pT_match = bool(
    np.all(
        corrupt_pT
        ==
        expected_corrupt_pT
    )
)


bank_clean_iT_match = bool(
    np.all(
        clean_iT
        ==
        expected_clean_iT
    )
)


bank_corrupt_iT_match = bool(
    np.all(
        corrupt_iT
        ==
        expected_corrupt_iT
    )
)


if not (
    bank_clean_pT_match
    and
    bank_corrupt_pT_match
    and
    bank_clean_iT_match
    and
    bank_corrupt_iT_match
):

    raise RuntimeError(
        "Frozen state-bank target-position metadata does not "
        "match independent Llama tokenizer reconstruction."
    )


print(
    "\nSTATE-BANK POSITION METADATA"
)

print(
    "-" * 80
)

print(
    "  clean p_T vs tokenizer: PASS"
)

print(
    "  corrupt p_T vs tokenizer: PASS"
)

print(
    "  clean i_T vs tokenizer: PASS"
)

print(
    "  corrupt i_T vs tokenizer: PASS"
)


print(
    "[PASS] Frozen target-position metadata independently verified."
)


# ============================================================
# 14. LLAMA TARGET-ID METADATA VERIFICATION
# ============================================================
#
# This comparison is legitimate because both sides are Llama
# tokenizer IDs.
# ============================================================

clean_state_target_ids = state_target_ids[
    clean_rows
]

corrupt_state_target_ids = state_target_ids[
    corrupt_rows
]


clean_target_id_match = bool(
    np.all(
        clean_state_target_ids
        ==
        expected_clean_llama_ids
    )
)


corrupt_target_id_match = bool(
    np.all(
        corrupt_state_target_ids
        ==
        expected_corrupt_llama_ids
    )
)


if not clean_target_id_match:

    raise RuntimeError(
        "Frozen clean Llama target IDs do not match independent "
        "Llama tokenizer reconstruction."
    )


if not corrupt_target_id_match:

    raise RuntimeError(
        "Frozen corrupt Llama target IDs do not match independent "
        "Llama tokenizer reconstruction."
    )


print(
    "\nLLAMA TARGET-ID METADATA"
)

print(
    "-" * 80
)

print(
    "  clean target IDs: 192/192"
)

print(
    "  corrupt target IDs: 192/192"
)

print(
    "  GPT-2 target IDs used: NO"
)

print(
    "[PASS] Llama target-ID metadata independently verified."
)


# ============================================================
# 15. SEMANTIC MODEL
# ============================================================
#
# The semantic model is an empirical interface to the model's
# target prediction.
#
# It is NOT a definition of CTL truth.
# ============================================================

@dataclass(
    frozen=True
)
class CTLSemanticModel:

    context_index: int

    condition: str

    target_token_id: int

    target_logit: float

    target_rank: int

    prediction_position: int

    hidden_state_position: int


semantic_models = {}


for i in range(
    N_RECORDS
):

    semantic_models[
        (
            i,
            "clean"
        )
    ] = CTLSemanticModel(

        context_index=i,

        condition="clean",

        target_token_id=int(
            clean_state_target_ids[i]
        ),

        target_logit=float(
            target_logits[
                clean_rows[i]
            ]
        ),

        target_rank=int(
            target_ranks[
                clean_rows[i]
            ]
        ),

        prediction_position=int(
            clean_pT[i]
        ),

        hidden_state_position=int(
            clean_iT[i]
        ),

    )


    semantic_models[
        (
            i,
            "corrupt"
        )
    ] = CTLSemanticModel(

        context_index=i,

        condition="corrupt",

        target_token_id=int(
            corrupt_state_target_ids[i]
        ),

        target_logit=float(
            target_logits[
                corrupt_rows[i]
            ]
        ),

        target_rank=int(
            target_ranks[
                corrupt_rows[i]
            ]
        ),

        prediction_position=int(
            corrupt_pT[i]
        ),

        hidden_state_position=int(
            corrupt_iT[i]
        ),

    )


if len(
    semantic_models
) != N_CONDITIONS:

    raise RuntimeError(
        "Semantic model object count mismatch."
    )


print(
    "\nSEMANTIC MODEL REALIZATION"
)

print(
    "-" * 80
)

print(
    f"  context-condition models: "
    f"{len(semantic_models)}"
)

print(
    "  semantic source: frozen target prediction"
)

print(
    "  truth value inferred from hidden-state threshold: NO"
)

print(
    "  Boolean CTL substitution: NO"
)

print(
    "[PASS] Context-indexed semantic models constructed."
)


# ============================================================
# 16. SEMANTIC VALUATION
# ============================================================
#
# A valuation here means an empirical semantic observation
# attached to a context and condition.
#
# It is deliberately NOT a Boolean truth assignment.
#
# The same model-level target-prediction observation can serve
# as the shared semantic interface for each operational sector,
# while sector identity remains a separate CTL carrier index.
# ============================================================

@dataclass(
    frozen=True
)
class CTLSemanticValuation:

    context_index: int

    condition: str

    sector: str

    target_token_id: int

    target_logit: float

    target_rank: int


semantic_valuations = {}


for i in range(
    N_RECORDS
):

    for condition in (
        "clean",
        "corrupt",
    ):

        model = semantic_models[
            (
                i,
                condition
            )
        ]


        for sector in SECTORS:

            semantic_valuations[
                (
                    i,
                    condition,
                    sector
                )
            ] = CTLSemanticValuation(

                context_index=i,

                condition=condition,

                sector=sector,

                target_token_id=
                    model.target_token_id,

                target_logit=
                    model.target_logit,

                target_rank=
                    model.target_rank,

            )


expected_valuation_count = (
    N_RECORDS
    *
    2
    *
    len(
        SECTORS
    )
)


if len(
    semantic_valuations
) != expected_valuation_count:

    raise RuntimeError(
        "Semantic valuation count mismatch."
    )


print(
    "\nSEMANTIC VALUATION"
)

print(
    "-" * 80
)

print(
    f"  valuation objects: "
    f"{len(semantic_valuations)}"
)

print(
    "  context index retained: YES"
)

print(
    "  condition index retained: YES"
)

print(
    "  sector index retained: YES"
)

print(
    "  sector index interpreted as truth value: NO"
)

print(
    "  semantic valuation interpreted as Phi_C: NO"
)

print(
    "[PASS] Context-indexed semantic valuation constructed."
)


# ============================================================
# 17. SEMANTIC CONTRAST
# ============================================================

clean_logits = target_logits[
    clean_rows
]

corrupt_logits = target_logits[
    corrupt_rows
]


clean_ranks = target_ranks[
    clean_rows
]

corrupt_ranks = target_ranks[
    corrupt_rows
]


logit_delta = (
    clean_logits
    -
    corrupt_logits
)


rank_improvement = (
    corrupt_ranks
    -
    clean_ranks
)


if not np.all(
    np.isfinite(
        logit_delta
    )
):

    raise RuntimeError(
        "Semantic logit contrast contains non-finite values."
    )


if not np.all(
    np.isfinite(
        rank_improvement
    )
):

    raise RuntimeError(
        "Semantic rank contrast contains non-finite values."
    )


print(
    "\nFROZEN SEMANTIC CONTRAST"
)

print(
    "-" * 80
)

print(
    f"  mean target-logit(clean-corrupt): "
    f"{np.mean(logit_delta):.9f}"
)

print(
    f"  median target-logit(clean-corrupt): "
    f"{np.median(logit_delta):.9f}"
)

print(
    f"  mean target-rank improvement: "
    f"{np.mean(rank_improvement):.9f}"
)

print(
    f"  median target-rank improvement: "
    f"{np.median(rank_improvement):.9f}"
)


# ============================================================
# 18. SPLIT STRUCTURE
# ============================================================
#
# Reconstruct the frozen split using the established rule:
#
#       r = (p + 4t) mod 16
#
#       r < 4       -> calibration
#       4 <= r < 8  -> test
#       r >= 8      -> train
#
# This does not refit anything.
# ============================================================

template_index = np.zeros(
    N_RECORDS,
    dtype=np.int64
)

pair_index = np.zeros(
    N_RECORDS,
    dtype=np.int64
)

split_index = np.empty(
    N_RECORDS,
    dtype=object
)


for t in range(
    12
):

    for p in range(
        16
    ):

        record_index = (
            16 * t
            +
            p
        )


        r = (
            p
            +
            4 * t
        ) % 16


        template_index[
            record_index
        ] = t


        pair_index[
            record_index
        ] = p


        if r < 4:

            split_index[
                record_index
            ] = "calibration"

        elif r < 8:

            split_index[
                record_index
            ] = "test"

        else:

            split_index[
                record_index
            ] = "train"


split_counts_reconstructed = Counter(
    split_index.tolist()
)


print(
    "\nFROZEN SPLIT STRUCTURE"
)

print(
    "-" * 80
)

print(
    f"  {dict(split_counts_reconstructed)}"
)


if dict(
    split_counts_reconstructed
) != expected_split_counts:

    raise RuntimeError(
        "Frozen split reconstruction failed."
    )


# ============================================================
# 19. DATASET/BANK SPLIT IDENTITY CHECK
# ============================================================

dataset_split = np.asarray(
    [
        str(
            r[
                "split"
            ]
        )
        for r in records
    ],
    dtype=object
)


if not np.array_equal(
    dataset_split,
    split_index
):

    raise RuntimeError(
        "Dataset record split labels disagree with frozen split rule."
    )


print(
    "[PASS] Dataset and frozen split structures agree."
)


# ============================================================
# 20. SPLIT-WISE SEMANTIC SUMMARY
# ============================================================

split_semantic_summary = {}


for split_name in (
    "train",
    "calibration",
    "test",
):

    idx = np.where(
        split_index
        ==
        split_name
    )[0]


    split_semantic_summary[
        split_name
    ] = {

        "N":
            int(
                len(
                    idx
                )
            ),

        "mean_target_logit_delta":
            float(
                np.mean(
                    logit_delta[
                        idx
                    ]
                )
            ),

        "median_target_logit_delta":
            float(
                np.median(
                    logit_delta[
                        idx
                    ]
                )
            ),

        "mean_target_rank_improvement":
            float(
                np.mean(
                    rank_improvement[
                        idx
                    ]
                )
            ),

        "median_target_rank_improvement":
            float(
                np.median(
                    rank_improvement[
                        idx
                    ]
                )
            ),

    }


print(
    "\nSPLIT-WISE SEMANTIC SUMMARY"
)

print(
    "-" * 80
)

for split_name in (
    "train",
    "calibration",
    "test",
):

    q = split_semantic_summary[
        split_name
    ]

    print(
        f"  {split_name}: "
        f"N={q['N']}, "
        f"mean_logit_delta="
        f"{q['mean_target_logit_delta']:.9f}, "
        f"mean_rank_improvement="
        f"{q['mean_target_rank_improvement']:.9f}"
    )


# ============================================================
# 21. ADMISSIBILITY LAYER
# ============================================================
#
# This is deliberately NOT inferred from the semantic values.
#
# In particular:
#
#   semantic_value > threshold
#       DOES NOT MEAN
#   admissible = TRUE
#
# and:
#
#   sector_1 AND sector_2 AND sector_3
#       DOES NOT DEFINE
#   CTL triadic admissibility.
#
# Undefined remains distinct from false.
# ============================================================

admissibility_realization = {

    "unary_domains":
        "STRUCTURALLY_PRESENT",

    "binary_domains":
        "STRUCTURALLY_PRESENT",

    "triadic_domain":
        "STRUCTURALLY_PRESENT",

    "unary_empirical_status":
        "NOT_IDENTIFIED",

    "binary_empirical_status":
        "NOT_IDENTIFIED",

    "triadic_empirical_status":
        "NOT_IDENTIFIED",

    "undefined_distinct_from_false":
        True,

    "semantic_values_define_admissibility":
        False,

    "hidden_state_threshold_defines_admissibility":
        False,

    "pairwise_conjunction_defines_triadic_admissibility":
        False,

}


if not admissibility_realization[
    "undefined_distinct_from_false"
]:

    raise RuntimeError(
        "Undefined/false distinction violated."
    )


if any(
    admissibility_realization[
        key
    ]

    for key in (
        "semantic_values_define_admissibility",
        "hidden_state_threshold_defines_admissibility",
        "pairwise_conjunction_defines_triadic_admissibility",
    )
):

    raise RuntimeError(
        "Invalid admissibility construction detected."
    )


print(
    "\nADMISSIBILITY FIREWALL"
)

print(
    "-" * 80
)

print(
    "  unary admissibility: structurally present, empirically unresolved"
)

print(
    "  binary admissibility: structurally present, empirically unresolved"
)

print(
    "  triadic admissibility: structurally present, empirically unresolved"
)

print(
    "  undefined distinct from false: YES"
)

print(
    "  semantic threshold used: NO"
)

print(
    "  Boolean conjunction used: NO"
)

print(
    "[PASS] No empirical admissibility rule was fabricated."
)


# ============================================================
# 22. TRUTH / COHERENCE SEPARATION
# ============================================================

truth_coherence_firewall = {

    "semantic_interface_exists":
        True,

    "semantic_valuation_exists":
        True,

    "coherence_carrier_is_separate":
        True,

    "Phi_C_fitted":
        False,

    "semantic_value_equals_coherence":
        False,

    "semantic_value_defines_Phi_C":
        False,

    "truth_implies_coherence":
        False,

    "coherence_implies_truth":
        False,

}


if truth_coherence_firewall[
    "Phi_C_fitted"
]:

    raise RuntimeError(
        "Phi_C fitting occurred in Phase 1F.2."
    )


if any(
    truth_coherence_firewall[
        key
    ]

    for key in (
        "semantic_value_equals_coherence",
        "semantic_value_defines_Phi_C",
        "truth_implies_coherence",
        "coherence_implies_truth",
    )
):

    raise RuntimeError(
        "Truth/coherence collapse detected."
    )


print(
    "\nTRUTH / COHERENCE FIREWALL"
)

print(
    "-" * 80
)

print(
    "  semantic interface separate from coherence: PASS"
)

print(
    "  Phi_C fitting: NO"

)

print(
    "  semantic value used as coherence score: NO"
)

print(
    "  truth/coherence implication asserted: NO"
)

print(
    "[PASS] Semantic and structural layers remain separate."
)


# ============================================================
# 23. CTL / BOOLEAN FIREWALL
# ============================================================

boolean_firewall = {

    "CTL_reduced_to_Boolean_logic":
        False,

    "sector_index_is_truth_value":
        False,

    "semantic_value_is_Boolean_truth":
        False,

    "admissibility_is_Boolean_AND":
        False,

    "undefined_equals_false":
        False,

    "triadic_admissibility_equals_pairwise_intersection":
        False,

    "context_index_discarded":
        False,

}


if any(
    boolean_firewall.values()
):

    raise RuntimeError(
        "Boolean collapse detected."
    )


print(
    "\nCTL / BOOLEAN FIREWALL"
)

print(
    "-" * 80
)

for key, value in boolean_firewall.items():

    print(
        f"  {key}: {value}"
    )


print(
    "[PASS] CTL remains distinct from Boolean substrate."
)


# ============================================================
# 24. SECTOR-SEMANTICS FIREWALL
# ============================================================
#
# Important:
#
# The target prediction is a shared model-level semantic
# interface. It is NOT evidence that S1, S2, and S3 possess
# identical logical carriers.
#
# Sector identity remains explicit in the CTL carrier layer.
# ============================================================

sector_semantics_firewall = {

    "shared_prediction_interface":
        True,

    "sector_carriers_remain_distinct":
        True,

    "sector_semantic_identity_claimed":
        False,

    "sector_index_retained":
        True,

}


if not sector_semantics_firewall[
    "sector_carriers_remain_distinct"
]:

    raise RuntimeError(
        "Sector carrier distinction lost."
    )


if sector_semantics_firewall[
    "sector_semantic_identity_claimed"
]:

    raise RuntimeError(
        "Unsupported sector-specific semantic identity claim."
    )


print(
    "\nSECTOR-SEMANTICS FIREWALL"
)

print(
    "-" * 80
)

print(
    "  shared target-prediction interface: YES"
)

print(
    "  sector carriers distinct: YES"
)

print(
    "  sector-specific semantic equivalence claimed: NO"
)

print(
    "[PASS] Shared semantic interface not confused with carrier identity."
)


# ============================================================
# 25. SCIENTIFIC STATUS
# ============================================================

scientific_status = {

    "frozen_state_bank":
        "INTEGRITY_VERIFIED",

    "condition_record_alignment":
        "VERIFIED",

    "target_position_operationalization":
        "VERIFIED",

    "llama_target_id_metadata":
        "VERIFIED",

    "semantic_prediction_interface":
        "OPERATIONALLY_REALIZED",

    "context_indexing":
        "PRESERVED",

    "semantic_valuation":
        "OPERATIONALLY_REALIZED",

    "unary_admissibility":
        "NOT_IDENTIFIED",

    "binary_admissibility":
        "NOT_IDENTIFIED",

    "triadic_admissibility":
        "NOT_IDENTIFIED",

    "logical_transport":
        "NOT_TESTED",

    "logical_transport_covariance":
        "NOT_TESTED",

    "triadic_coherence":
        "NOT_TESTED",

    "triadic_irreducibility":
        "NOT_TESTED",

    "contextual_geometry":
        "NOT_ESTABLISHED",

}


print(
    "\nSCIENTIFIC STATUS"
)

print(
    "-" * 80
)

for key, value in scientific_status.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 26. NON-CLAIM FIREWALL
# ============================================================

scientific_nonclaims = {

    "CTL_empirical_satisfaction":
        "NOT_ESTABLISHED",

    "semantic_truth_equals_CTL_truth":
        "NOT_ASSERTED",

    "hidden_state_threshold_defines_truth":
        "REJECTED",

    "hidden_state_threshold_defines_admissibility":
        "REJECTED",

    "triadic_admissibility":
        "NOT_IDENTIFIED",

    "logical_transport_covariance":
        "NOT_TESTED",

    "functoriality":
        "NOT_TESTED",

    "triadic_coherence":
        "NOT_TESTED",

    "triadic_irreducibility":
        "NOT_TESTED",

    "contextual_geometry":
        "NOT_ESTABLISHED",

    "causal_transport":
        "NOT_ESTABLISHED",

    "sector_specific_semantic_equivalence":
        "NOT_ASSERTED",

}


# ============================================================
# 27. SERIALIZATION
# ============================================================

artifact = {

    "experiment_id":
        "ETTR-CTL-LLAMA-1",

    "phase":
        "1F.2",

    "title":
        "CTL Semantic Realization and Admissibility Audit",

    "row_order_predecessor":
        {

            "artifact":
                str(
                    ROW_ORDER_PATH
                ),

            "classification":
                row_order_classification,

            "selected_row_order":
                selected_row_order,

            "clean_rows":
                clean_rows.tolist(),

            "corrupt_rows":
                corrupt_rows.tolist(),

        },

    "dataset":
        {

            "path":
                str(
                    dataset_path
                ),

            "sha256":
                dataset_sha,

            "records":
                N_RECORDS,

        },

    "state_bank":
        {

            "S1_hash":
                EXPECTED_STATE_HASHES[
                    "S1"
                ],

            "S2_hash":
                EXPECTED_STATE_HASHES[
                    "S2"
                ],

            "S3_hash":
                EXPECTED_STATE_HASHES[
                    "S3"
                ],

        },

    "position_alignment":
        {

            "p_T_pair_equal":
                pT_equal,

            "i_T_pair_equal":
                iT_equal,

            "dataset_p_T_reconstruction":
                dataset_pT_equal,

            "dataset_i_T_reconstruction":
                dataset_iT_equal,

            "bank_clean_p_T_match":
                bank_clean_pT_match,

            "bank_corrupt_p_T_match":
                bank_corrupt_pT_match,

            "bank_clean_i_T_match":
                bank_clean_iT_match,

            "bank_corrupt_i_T_match":
                bank_corrupt_iT_match,

        },

    "llama_target_id_alignment":
        {

            "clean_192_of_192":
                clean_target_id_match,

            "corrupt_192_of_192":
                corrupt_target_id_match,

            "cross_model_token_id_comparison":
                False,

        },

    "semantic_interface":
        {

            "source":
                "frozen Llama target-prediction artifacts",

            "context_indexed":
                True,

            "condition_indexed":
                True,

            "sector_indexed":
                True,

            "Boolean_truth_substitution":
                False,

            "hidden_state_threshold_truth":
                False,

            "semantic_value_defines_Phi_C":
                False,

        },

    "semantic_contrast":
        {

            "mean_target_logit_clean_minus_corrupt":
                float(
                    np.mean(
                        logit_delta
                    )
                ),

            "median_target_logit_clean_minus_corrupt":
                float(
                    np.median(
                        logit_delta
                    )
                ),

            "mean_target_rank_improvement":
                float(
                    np.mean(
                        rank_improvement
                    )
                ),

            "median_target_rank_improvement":
                float(
                    np.median(
                        rank_improvement
                    )
                ),

        },

    "split_semantic_summary":
        split_semantic_summary,

    "admissibility":
        admissibility_realization,

    "truth_coherence_firewall":
        truth_coherence_firewall,

    "boolean_firewall":
        boolean_firewall,

    "sector_semantics_firewall":
        sector_semantics_firewall,

    "scientific_status":
        scientific_status,

    "scientific_nonclaims":
        scientific_nonclaims,

    "governance":
        {

            "pca_refitting":
                False,

            "transport_map_refitting":
                False,

            "phi_C_fitting":
                False,

            "phi_D_fitting":
                False,

            "test_fitting":
                False,

            "state_vector_matching":
                False,

            "logit_matching_for_row_identity":
                False,

            "cross_model_token_id_matching":
                False,

            "hidden_state_threshold_admissibility":
                False,

            "boolean_collapse":
                False,

        },

    "final_classification":
        "CTL_SEMANTIC_REALIZATION_OPERATIONAL_PASS",

}


with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 28. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "FINAL PHASE 1F.2 STATUS"
)

print(
    "=" * 80
)

print(
    "  CTL_SEMANTIC_REALIZATION_OPERATIONAL_PASS"
)

print(
    "  verified interleaved row order consumed: PASS"
)

print(
    "  frozen clean/corrupt alignment: PASS"
)

print(
    "  target-position reconstruction: PASS"
)

print(
    "  Llama target-ID metadata verification: PASS"
)

print(
    "  semantic prediction interface: OPERATIONALLY REALIZED"
)

print(
    "  contextual admissibility: NOT IDENTIFIED"
)

print(
    "  triadic admissibility: NOT IDENTIFIED"
)

print(
    "  logical transport covariance: NOT TESTED"
)

print(
    "  triadic coherence: NOT TESTED"
)

print(
    "  triadic irreducibility: NOT TESTED"
)

print(
    "  contextual geometry: NOT ESTABLISHED"
)

print(
    f"  artifact: {OUTPUT_PATH}"
)

print(
    "=" * 80
)

ETTR-CTL-LLAMA-1 — PHASE 1F.2
CTL SEMANTIC REALIZATION / ADMISSIBILITY AUDIT
CORRECTED AFTER VERIFIED INTERLEAVED ROW ORDER
ROOT: /content/ettr_ctl_llama
State bank: /content/ettr_ctl_llama/results/llama_phase1d2_full_state_bank.npz
Formal CTL audit: /content/ettr_ctl_llama/results/llama_phase1f1_formal_ctl_structure_audit.json
Row-order audit: /content/ettr_ctl_llama/results/llama_phase1f2a_state_bank_row_order_audit.json
Output: /content/ettr_ctl_llama/results/llama_phase1f2_ctl_semantic_realization_audit.json

GOVERNANCE
--------------------------------------------------------------------------------
  CTL formal structure: FROZEN
  semantic valuation: SEPARATE REALIZATION LAYER
  Boolean truth as CTL definition: FORBIDDEN
  hidden-state threshold as semantic truth: FORBIDDEN
  numerical transport as logical transport: FORBIDDEN
  PCA refitting: FORBIDDEN
  transport-map refitting: FORBIDDEN
  Phi_C fitting: FORBIDDEN
  Phi_D fitting: FORBIDDEN
  test fitting: FORBIDDEN
  state-bank

In [42]:
# ============================================================
# ETTR-CTL-LLAMA-1
# PHASE 1F.3 — LOGICAL TRANSPORT COVARIANCE AUDIT
#
# REPLACEMENT CELL
#
# IMPORTANT:
# Remove the previous Phase 1F.3 cell completely, then paste
# this entire cell.
#
# This replacement preserves the scientific logic of the
# previous cell. The correction is primarily in the final
# artifact serialization:
#
#   row_order_classification
#
# is a string, not a dictionary.
#
# The previously completed scientific checks are retained.
# ============================================================


import os
import json
import hashlib
from pathlib import Path
from dataclasses import dataclass


# ============================================================
# 0. PATHS
# ============================================================

ROOT = Path(
    "/content/ettr_ctl_llama"
)

RESULTS = ROOT / "results"

STATE_BANK_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank.npz"
)

STATE_MANIFEST_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank_manifest.json"
)

AUTH_PATH = (
    RESULTS /
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

CTL_FORMAL_PATH = (
    RESULTS /
    "llama_phase1f1_formal_ctl_structure_audit.json"
)

ROW_ORDER_PATH = (
    RESULTS /
    "llama_phase1f2a_state_bank_row_order_audit.json"
)

SEMANTIC_PATH = (
    RESULTS /
    "llama_phase1f2_ctl_semantic_realization_audit.json"
)

OUTPUT_PATH = (
    RESULTS /
    "llama_phase1f3_logical_transport_covariance_audit.json"
)


EXPECTED_DATASET_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)


EXPECTED_STATE_HASHES = {

    "S1":
        "9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783",

    "S2":
        "41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb",

    "S3":
        "173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3",

}


SECTORS = (
    "S1",
    "S2",
    "S3",
)

N_RECORDS = 192
N_CONDITIONS = 384


# ============================================================
# 1. HEADER
# ============================================================

print(
    "=" * 80
)

print(
    "ETTR-CTL-LLAMA-1 — PHASE 1F.3"
)

print(
    "LOGICAL TRANSPORT COVARIANCE AUDIT"
)

print(
    "=" * 80
)

print(
    f"ROOT: {ROOT}"
)

print(
    f"State bank: {STATE_BANK_PATH}"
)

print(
    f"CTL formal audit: {CTL_FORMAL_PATH}"
)

print(
    f"Semantic realization: {SEMANTIC_PATH}"
)

print(
    f"Output: {OUTPUT_PATH}"
)


# ============================================================
# 2. GOVERNANCE
# ============================================================

print(
    "\nGOVERNANCE"
)

print(
    "-" * 80
)

governance = {

    "CTL formal structure":
        "FROZEN",

    "verified state-bank ordering":
        "REQUIRED",

    "semantic realization":
        "FROZEN",

    "numerical transport = logical transport":
        "FORBIDDEN",

    "numerical transport maps used as CTL morphisms":
        "FORBIDDEN",

    "hidden-state threshold admissibility":
        "FORBIDDEN",

    "semantic threshold admissibility":
        "FORBIDDEN",

    "Boolean conjunction as triadic admissibility":
        "FORBIDDEN",

    "undefined = false":
        "FORBIDDEN",

    "admissibility fabrication":
        "FORBIDDEN",

    "PCA refitting":
        "FORBIDDEN",

    "transport-map refitting":
        "FORBIDDEN",

    "Phi_C fitting":
        "FORBIDDEN",

    "Phi_D fitting":
        "FORBIDDEN",

    "test fitting":
        "FORBIDDEN",

    "functoriality claim":
        "FORBIDDEN",

    "triadic irreducibility":
        "NOT TESTED",

    "contextual geometry":
        "NOT TESTED",

}


for key, value in governance.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 3. REQUIRED ARTIFACTS
# ============================================================

required_paths = [

    STATE_BANK_PATH,
    STATE_MANIFEST_PATH,
    AUTH_PATH,
    CTL_FORMAL_PATH,
    ROW_ORDER_PATH,
    SEMANTIC_PATH,

]


missing = [

    str(
        path
    )

    for path in required_paths

    if not path.exists()

]


if missing:

    raise FileNotFoundError(
        "Required artifact(s) missing:\n"
        +
        "\n".join(
            missing
        )
    )


print(
    "\n[PASS] Required frozen artifacts exist."
)


# ============================================================
# 4. LOAD FORMAL CTL PREDECESSOR
# ============================================================

with open(
    CTL_FORMAL_PATH,
    "r",
    encoding="utf-8"
) as f:

    ctl_formal = json.load(
        f
    )


formal_classification = ctl_formal.get(
    "final_classification"
)


if formal_classification not in (
    "FORMAL_CTL_STRUCTURE_IMPLEMENTATION_PASS",
    "CTL_FORMAL_STRUCTURE_IMPLEMENTATION_PASS",
):

    raise RuntimeError(
        "Phase 1F.1 formal CTL implementation is not frozen as PASS. "
        f"Observed: {formal_classification}"
    )


print(
    "[PASS] Formal CTL predecessor."
)


# ============================================================
# 5. LOAD VERIFIED ROW ORDER
# ============================================================

with open(
    ROW_ORDER_PATH,
    "r",
    encoding="utf-8"
) as f:

    row_order = json.load(
        f
    )


# The field is a STRING.
row_order_classification = row_order.get(
    "classification"
)


selected_row_order = row_order.get(
    "selected_row_order"
)


if row_order_classification != (
    "FROZEN_STATE_BANK_ROW_ORDER_VERIFIED"
):

    raise RuntimeError(
        "Verified state-bank row-order artifact is not PASS. "
        f"Observed: {row_order_classification}"
    )


if selected_row_order != (
    "interleaved_clean_corrupt"
):

    raise RuntimeError(
        "Unexpected frozen row-order convention. "
        f"Observed: {selected_row_order}"
    )


clean_rows = np.asarray(
    row_order[
        "clean_rows"
    ],
    dtype=np.int64
)


corrupt_rows = np.asarray(
    row_order[
        "corrupt_rows"
    ],
    dtype=np.int64
)


expected_clean_rows = np.arange(
    0,
    N_CONDITIONS,
    2,
    dtype=np.int64
)


expected_corrupt_rows = np.arange(
    1,
    N_CONDITIONS,
    2,
    dtype=np.int64
)


if not np.array_equal(
    clean_rows,
    expected_clean_rows
):

    raise RuntimeError(
        "Frozen clean row mapping is inconsistent."
    )


if not np.array_equal(
    corrupt_rows,
    expected_corrupt_rows
):

    raise RuntimeError(
        "Frozen corrupt row mapping is inconsistent."
    )


print(
    "\nROW-ORDER PREDECESSOR"
)

print(
    "-" * 80
)

print(
    f"  classification: {row_order_classification}"
)

print(
    f"  selected row order: {selected_row_order}"
)

print(
    "  bank[2i] = clean C_i"
)

print(
    "  bank[2i+1] = corrupt D_i"
)

print(
    "[PASS] Verified condition correspondence consumed."
)


# ============================================================
# 6. LOAD SEMANTIC PREDECESSOR
# ============================================================

with open(
    SEMANTIC_PATH,
    "r",
    encoding="utf-8"
) as f:

    semantic = json.load(
        f
    )


semantic_classification = semantic.get(
    "final_classification"
)


if semantic_classification != (
    "CTL_SEMANTIC_REALIZATION_OPERATIONAL_PASS"
):

    raise RuntimeError(
        "Phase 1F.2 semantic realization is not frozen as PASS. "
        f"Observed: {semantic_classification}"
    )


print(
    "\nSEMANTIC PREDECESSOR"
)

print(
    "-" * 80
)

print(
    f"  classification: {semantic_classification}"
)

print(
    "[PASS] Frozen semantic realization consumed."
)


# ============================================================
# 7. DATASET AUTHORIZATION
# ============================================================

with open(
    AUTH_PATH,
    "r",
    encoding="utf-8"
) as f:

    auth = json.load(
        f
    )


dataset_sha = (
    auth.get(
        "dataset_sha256"
    )
    or
    auth.get(
        "sha256"
    )
)


authorized = bool(
    auth.get(
        "dataset_authorized",
        auth.get(
            "authorized",
            False
        )
    )
)


if not authorized:

    raise RuntimeError(
        "Dataset authorization failed."
    )


if dataset_sha != EXPECTED_DATASET_SHA256:

    raise RuntimeError(
        "Dataset SHA mismatch."
    )


print(
    "\nDATASET AUTHORIZATION"
)

print(
    "-" * 80
)

print(
    f"  authorized: {authorized}"
)

print(
    f"  SHA: {dataset_sha}"
)

print(
    "[PASS] Dataset authorization."
)


# ============================================================
# 8. LOAD OPERATIVE DATASET
# ============================================================

dataset_candidates = [

    Path(
        "gpt2_controlled_dataset_v1.1_recovered_r1.json"
    ),

    Path(
        "/content/gpt2_controlled_dataset_v1.1_recovered_r1.json"
    ),

]


if not any(
    p.exists()
    for p in dataset_candidates
):

    for base in (
        ROOT,
        Path("/content"),
    ):

        if base.exists():

            dataset_candidates.extend(
                base.rglob(
                    "gpt2_controlled_dataset_v1.1_recovered_r1.json"
                )
            )


dataset_candidates = list(
    dict.fromkeys(
        dataset_candidates
    )
)


if not dataset_candidates:

    raise FileNotFoundError(
        "Operative recovered dataset not found."
    )


def load_records(
    path
):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        obj = json.load(
            f
        )


    if isinstance(
        obj,
        list
    ):

        return obj


    if isinstance(
        obj,
        dict
    ):

        for key in (
            "records",
            "dataset",
            "examples",
            "data",
            "items",
        ):

            if isinstance(
                obj.get(key),
                list
            ):

                return obj[
                    key
                ]


    return None


required_fields = {

    "example_id",
    "name_pair_id",
    "name_pair_index",
    "template_id",
    "template_index",
    "split",
    "clean_prompt",
    "corrupt_prompt",
    "clean_target_name",
    "corrupt_target_name",

}


valid_dataset_candidates = []


for path in dataset_candidates:

    try:

        candidate_records = load_records(
            path
        )

    except Exception:

        continue


    if candidate_records is None:

        continue


    if len(
        candidate_records
    ) != N_RECORDS:

        continue


    if not required_fields.issubset(
        set(
            candidate_records[0].keys()
        )
    ):

        continue


    valid_dataset_candidates.append(
        (
            path,
            candidate_records
        )
    )


if not valid_dataset_candidates:

    raise RuntimeError(
        "No valid 192-record operative dataset found."
    )


def identity_signature(
    record
):

    return (

        str(
            record[
                "example_id"
            ]
        ),

        str(
            record[
                "name_pair_id"
            ]
        ),

        int(
            record[
                "name_pair_index"
            ]
        ),

        str(
            record[
                "template_id"
            ]
        ),

        int(
            record[
                "template_index"
            ]
        ),

        str(
            record[
                "split"
            ]
        ),

    )


reference_identity = [

    identity_signature(
        record
    )

    for record
    in valid_dataset_candidates[0][1]

]


for path, candidate_records in valid_dataset_candidates[1:]:

    candidate_identity = [

        identity_signature(
            record
        )

        for record
        in candidate_records

    ]


    if candidate_identity != reference_identity:

        raise RuntimeError(
            "Duplicate dataset copies disagree in identity/order."
        )


dataset_path, records = (
    valid_dataset_candidates[0]
)


if [

    identity_signature(
        record
    )

    for record
    in records

] != reference_identity:

    raise RuntimeError(
        "Operative dataset identity/order changed."
    )


print(
    "\nOPERATIVE DATASET"
)

print(
    "-" * 80
)

print(
    f"  path: {dataset_path}"
)

print(
    f"  records: {len(records)}"
)

print(
    "[PASS] Operative dataset identified."
)


# ============================================================
# 9. LOAD FROZEN STATE BANK
# ============================================================

bank = np.load(
    STATE_BANK_PATH,
    allow_pickle=False
)


state_arrays = {}


for sector in SECTORS:

    X = np.asarray(
        bank[
            sector
        ]
    )


    if X.shape != (
        N_CONDITIONS,
        3072
    ):

        raise RuntimeError(
            f"{sector} shape mismatch: {X.shape}"
        )


    actual_hash = hashlib.sha256(

        np.ascontiguousarray(
            X
        ).tobytes(
            order="C"
        )

    ).hexdigest()


    if actual_hash != EXPECTED_STATE_HASHES[
        sector
    ]:

        raise RuntimeError(
            f"{sector} frozen state hash mismatch."
        )


    if not np.all(
        np.isfinite(
            X
        )
    ):

        raise RuntimeError(
            f"{sector} contains non-finite values."
        )


    state_arrays[
        sector
    ] = X


print(
    "\nSTATE-BANK FIREWALL"
)

print(
    "-" * 80
)

for sector in SECTORS:

    print(
        f"  {sector}: "
        f"shape=(384,3072), "
        f"hash_match=True, "
        f"finite=True"
    )


print(
    "[PASS] Frozen state-bank integrity."
)


# ============================================================
# 10. LOAD FROZEN SEMANTIC ARRAYS
# ============================================================

target_logits = np.asarray(
    bank[
        "target_logits"
    ],
    dtype=np.float64
)


target_ranks = np.asarray(
    bank[
        "target_logit_ranks"
    ],
    dtype=np.int64
)


target_ids = np.asarray(
    bank[
        "condition_llama_target_ids"
    ],
    dtype=np.int64
)


state_pT = np.asarray(
    bank[
        "condition_p_T"
    ],
    dtype=np.int64
)


state_iT = np.asarray(
    bank[
        "condition_i_T"
    ],
    dtype=np.int64
)


for name, arr in (
    ("target_logits", target_logits),
    ("target_ranks", target_ranks),
    ("target_ids", target_ids),
    ("p_T", state_pT),
    ("i_T", state_iT),
):

    if arr.shape != (
        N_CONDITIONS,
    ):

        raise RuntimeError(
            f"{name} shape mismatch."
        )


    if name in (
        "target_logits",
    ):

        if not np.all(
            np.isfinite(
                arr
            )
        ):

            raise RuntimeError(
                f"{name} contains non-finite values."
            )


print(
    "\nFROZEN SEMANTIC ARRAYS"
)

print(
    "-" * 80
)

print(
    "  target_logits: (384,)"
)

print(
    "  target_ranks: (384,)"
)

print(
    "  target_ids: (384,)"
)

print(
    "  p_T: (384,)"
)

print(
    "  i_T: (384,)"
)

print(
    "[PASS] Frozen semantic arrays."
)


# ============================================================
# 11. ALIGN SEMANTIC OBSERVATIONS
# ============================================================

clean_logits = target_logits[
    clean_rows
]

corrupt_logits = target_logits[
    corrupt_rows
]


clean_ranks = target_ranks[
    clean_rows
]

corrupt_ranks = target_ranks[
    corrupt_rows
]


clean_target_ids = target_ids[
    clean_rows
]

corrupt_target_ids = target_ids[
    corrupt_rows
]


clean_pT = state_pT[
    clean_rows
]

corrupt_pT = state_pT[
    corrupt_rows
]


clean_iT = state_iT[
    clean_rows
]

corrupt_iT = state_iT[
    corrupt_rows
]


if not np.all(
    clean_pT
    ==
    corrupt_pT
):

    raise RuntimeError(
        "Verified alignment does not preserve paired p_T."
    )


if not np.all(
    clean_iT
    ==
    corrupt_iT
):

    raise RuntimeError(
        "Verified alignment does not preserve paired i_T."
    )


print(
    "\nSEMANTIC ALIGNMENT"
)

print(
    "-" * 80
)

print(
    "  clean/corrupt p_T: paired"
)

print(
    "  clean/corrupt i_T: paired"
)

print(
    "[PASS] Frozen semantic correspondence."
)


# ============================================================
# 12. CTL CONTEXT OBJECTS
# ============================================================

@dataclass(
    frozen=True
)
class CTLContext:

    context_index: int

    example_id: str

    template_id: str

    pair_id: str

    split: str


contexts = {}


for i, record in enumerate(
    records
):

    contexts[
        i
    ] = CTLContext(

        context_index=i,

        example_id=str(
            record[
                "example_id"
            ]
        ),

        template_id=str(
            record[
                "template_id"
            ]
        ),

        pair_id=str(
            record[
                "name_pair_id"
            ]
        ),

        split=str(
            record[
                "split"
            ]
        ),

    )


if len(
    contexts
) != N_RECORDS:

    raise RuntimeError(
        "Context count mismatch."
    )


print(
    "\nCTL CONTEXTS"
)

print(
    "-" * 80
)

print(
    f"  contexts: {len(contexts)}"
)

print(
    "  context identity preserved: YES"
)

print(
    "[PASS] Context-indexed logical base."
)


# ============================================================
# 13. CONTEXT-INDEXED LOGICAL CARRIERS
# ============================================================

@dataclass(
    frozen=True
)
class CTLLogicalCarrier:

    context_index: int

    sector: str

    carrier_type: str

    condition_domain: tuple


logical_carriers = {}


for i in range(
    N_RECORDS
):

    for sector in SECTORS:

        logical_carriers[
            (
                i,
                sector
            )
        ] = CTLLogicalCarrier(

            context_index=i,

            sector=sector,

            carrier_type="CTL_CONTEXT_INDEXED_CARRIER",

            condition_domain=(
                "clean",
                "corrupt",
            ),

        )


if len(
    logical_carriers
) != (
    N_RECORDS
    *
    len(
        SECTORS
    )
):

    raise RuntimeError(
        "Logical carrier count mismatch."
    )


print(
    "\nLOGICAL CARRIERS"
)

print(
    "-" * 80
)

print(
    f"  carriers: {len(logical_carriers)}"
)

print(
    "  sector identity retained: YES"
)

print(
    "  sector identity treated as truth value: NO"
)

print(
    "[PASS] Context-indexed logical carriers."
)


# ============================================================
# 14. EVENT DOMAINS
# ============================================================

@dataclass(
    frozen=True
)
class CTLEvent:

    context_index: int

    event_index: int

    sector: str


events = {}


for i in range(
    N_RECORDS
):

    for event_index, sector in enumerate(
        SECTORS
    ):

        events[
            (
                i,
                event_index
            )
        ] = CTLEvent(

            context_index=i,

            event_index=event_index,

            sector=sector,

        )


if len(
    events
) != (
    N_RECORDS
    *
    3
):

    raise RuntimeError(
        "Event domain count mismatch."
    )


print(
    "\nEVENT DOMAINS"
)

print(
    "-" * 80
)

print(
    f"  events: {len(events)}"
)

print(
    "  events distinct from truth values: YES"
)

print(
    "[PASS] Event domains."
)


# ============================================================
# 15. LOGICAL TRANSPORT MORPHISMS
# ============================================================

@dataclass(
    frozen=True
)
class CTLLogicalTransport:

    context_index: int

    sector: str

    source_condition: str

    target_condition: str

    source_carrier_type: str

    target_carrier_type: str

    invertibility_assumed: bool

    numerical_map_identified: bool


logical_transports = {}


for i in range(
    N_RECORDS
):

    for sector in SECTORS:

        source_carrier = logical_carriers[
            (
                i,
                sector
            )
        ]

        target_carrier = logical_carriers[
            (
                i,
                sector
            )
        ]


        logical_transports[
            (
                i,
                sector
            )
        ] = CTLLogicalTransport(

            context_index=i,

            sector=sector,

            source_condition="clean",

            target_condition="corrupt",

            source_carrier_type=
                source_carrier.carrier_type,

            target_carrier_type=
                target_carrier.carrier_type,

            invertibility_assumed=False,

            numerical_map_identified=False,

        )


if len(
    logical_transports
) != (
    N_RECORDS
    *
    3
):

    raise RuntimeError(
        "Logical transport registry count mismatch."
    )


if any(
    t.numerical_map_identified
    for t in logical_transports.values()
):

    raise RuntimeError(
        "Numerical transport was incorrectly identified "
        "with logical transport."
    )


print(
    "\nLOGICAL TRANSPORT REGISTRY"
)

print(
    "-" * 80
)

print(
    f"  logical transport morphisms: "
    f"{len(logical_transports)}"
)

print(
    "  numerical map identity: NO"
)

print(
    "  invertibility assumed: NO"
)

print(
    "[PASS] Logical transport registry is formally distinct "
    "from numerical transport."
)


# ============================================================
# 16. SEMANTIC VALUATION OBJECTS
# ============================================================

@dataclass(
    frozen=True
)
class CTLSemanticObservation:

    context_index: int

    condition: str

    sector: str

    target_logit: float

    target_rank: int

    target_token_id: int


semantic_observations = {}


for i in range(
    N_RECORDS
):

    for sector in SECTORS:

        semantic_observations[
            (
                i,
                "clean",
                sector
            )
        ] = CTLSemanticObservation(

            context_index=i,

            condition="clean",

            sector=sector,

            target_logit=float(
                clean_logits[i]
            ),

            target_rank=int(
                clean_ranks[i]
            ),

            target_token_id=int(
                clean_target_ids[i]
            ),

        )


        semantic_observations[
            (
                i,
                "corrupt",
                sector
            )
        ] = CTLSemanticObservation(

            context_index=i,

            condition="corrupt",

            sector=sector,

            target_logit=float(
                corrupt_logits[i]
            ),

            target_rank=int(
                corrupt_ranks[i]
            ),

            target_token_id=int(
                corrupt_target_ids[i]
            ),

        )


if len(
    semantic_observations
) != (
    N_RECORDS
    *
    2
    *
    3
):

    raise RuntimeError(
        "Semantic observation count mismatch."
    )


print(
    "\nSEMANTIC OBSERVATIONS"
)

print(
    "-" * 80
)

print(
    f"  observations: {len(semantic_observations)}"
)

print(
    "  source: frozen target prediction"
)

print(
    "  semantic observation = Boolean truth: NO"
)

print(
    "  semantic observation = Phi_C: NO"
)

print(
    "[PASS] Frozen semantic observations attached to CTL contexts."
)


# ============================================================
# 17. LOGICAL TRANSPORT COVARIANCE — TYPE LEVEL
# ============================================================

covariance_registry = {}


for i in range(
    N_RECORDS
):

    for sector in SECTORS:

        transport = logical_transports[
            (
                i,
                sector
            )
        ]


        source_observation = semantic_observations[
            (
                i,
                "clean",
                sector
            )
        ]


        target_observation = semantic_observations[
            (
                i,
                "corrupt",
                sector
            )
        ]


        type_preserved = (

            source_observation.context_index
            ==
            target_observation.context_index
            ==
            i

            and

            source_observation.sector
            ==
            target_observation.sector
            ==
            sector

            and

            transport.source_condition
            ==
            "clean"

            and

            transport.target_condition
            ==
            "corrupt"

            and

            transport.source_carrier_type
            ==
            transport.target_carrier_type

        )


        covariance_registry[
            (
                i,
                sector
            )
        ] = {

            "context_index_preserved":
                bool(
                    source_observation.context_index
                    ==
                    target_observation.context_index
                ),

            "sector_type_preserved":
                bool(
                    source_observation.sector
                    ==
                    target_observation.sector
                    ==
                    sector
                ),

            "transport_endpoint_types_valid":
                bool(
                    type_preserved
                ),

            "semantic_value_invariance_tested":
                False,

            "semantic_value_invariance_required":
                False,

        }


covariance_type_pass = all(

    item[
        "transport_endpoint_types_valid"
    ]

    for item
    in covariance_registry.values()

)


if not covariance_type_pass:

    raise RuntimeError(
        "Logical transport covariance type structure failed."
    )


print(
    "\nLOGICAL TRANSPORT COVARIANCE — TYPE LEVEL"
)

print(
    "-" * 80
)

print(
    f"  context/sector endpoint typing: "
    f"{sum(
        item['transport_endpoint_types_valid']
        for item in covariance_registry.values()
    )}/576"
)

print(
    "  semantic value invariance required: NO"
)

print(
    "  semantic equality imposed across conditions: NO"
)

print(
    "[PASS] Logical transport covariance is type-consistent "
    "at the identifiable interface."
)


# ============================================================
# 18. ADMISSIBILITY PRESERVATION STATUS
# ============================================================

admissibility_transport_status = {

    "source_admissibility_empirically_identified":
        False,

    "target_admissibility_empirically_identified":
        False,

    "admissibility_preservation_tested":
        False,

    "admissibility_preservation_status":
        "NOT_IDENTIFIED",

    "undefined_preserved_as_undefined":
        True,

    "undefined_coerced_to_false":
        False,

}


if admissibility_transport_status[
    "undefined_coerced_to_false"
]:

    raise RuntimeError(
        "Undefined admissibility was incorrectly coerced to false."
    )


print(
    "\nADMISSIBILITY TRANSPORT STATUS"
)

print(
    "-" * 80
)

print(
    "  source admissibility identified: NO"
)

print(
    "  target admissibility identified: NO"
)

print(
    "  admissibility preservation test: NOT TESTED"
)

print(
    "  undefined preserved as undefined: YES"
)

print(
    "[PASS] No unsupported admissibility claim made."
)


# ============================================================
# 19. LOGICAL-OPERATION PRESERVATION STATUS
# ============================================================

logical_operation_status = {

    "logical_operations_formally_defined":
        True,

    "empirical_operation_realization":
        False,

    "operation_preservation_tested":
        False,

    "status":
        "NOT_IDENTIFIED",

}


print(
    "\nLOGICAL-OPERATION PRESERVATION"
)

print(
    "-" * 80
)

print(
    "  formal operations: PRESENT"
)

print(
    "  empirical operations: NOT IDENTIFIED"
)

print(
    "  preservation test: NOT TESTED"
)

print(
    "[PASS] No synthetic logical algebra was fabricated."
)


# ============================================================
# 20. NUMERICAL / LOGICAL TRANSPORT FIREWALL
# ============================================================

numerical_logical_firewall = {

    "numerical_transport_maps_loaded":
        False,

    "numerical_transport_used_as_logical_transport":
        False,

    "numerical_transport_identity_claimed":
        False,

    "logical_transport_is_typed_object":
        True,

    "logical_transport_empirically_fitted":
        False,

}


if numerical_logical_firewall[
    "numerical_transport_used_as_logical_transport"
]:

    raise RuntimeError(
        "Numerical transport was promoted to logical transport."
    )


if numerical_logical_firewall[
    "numerical_transport_identity_claimed"
]:

    raise RuntimeError(
        "Unsupported numerical/logical transport identity."
    )


print(
    "\nNUMERICAL / LOGICAL TRANSPORT FIREWALL"
)

print(
    "-" * 80
)

print(
    "  numerical maps loaded: NO"
)

print(
    "  numerical map = logical morphism: NO"
)

print(
    "  logical transport fitted: NO"
)

print(
    "[PASS] Numerical and logical transport remain distinct."
)


# ============================================================
# 21. COMPOSITION / FUNCTORIALITY
# ============================================================

composition_status = {

    "identity_morphism_formally_present":
        True,

    "composition_object_formally_present":
        True,

    "empirical_multi_step_path_data":
        False,

    "functoriality_tested":
        False,

    "functoriality_status":
        "NOT_IDENTIFIED",

}


print(
    "\nCOMPOSITION / FUNCTORIALITY"
)

print(
    "-" * 80
)

print(
    "  identity structure: FORMALLY PRESENT"
)

print(
    "  composition structure: FORMALLY PRESENT"
)

print(
    "  empirical multi-step path data: NOT AVAILABLE"
)

print(
    "  functoriality: NOT TESTED"
)


# ============================================================
# 22. TRIADIC TRANSPORT STATUS
# ============================================================

triadic_transport_status = {

    "sector_transports":
        3,

    "triadic_transport_object":
        True,

    "pairwise_reduction_performed":
        False,

    "triadic_irreducibility_tested":
        False,

    "triadic_irreducibility_status":
        "NOT_TESTED",

}


print(
    "\nTRIADIC TRANSPORT STATUS"
)

print(
    "-" * 80
)

print(
    "  sector transport types: S1, S2, S3"
)

print(
    "  triadic transport registry: PRESENT"
)

print(
    "  pairwise reduction: NO"
)

print(
    "  triadic irreducibility: NOT TESTED"
)


# ============================================================
# 23. SCIENTIFIC STATUS
# ============================================================

scientific_status = {

    "logical_transport_objects":
        "OPERATIONALLY_INSTANTIATED",

    "logical_transport_typing":
        "SUPPORTED",

    "context_index_covariance":
        "SUPPORTED",

    "sector_type_covariance":
        "SUPPORTED",

    "semantic_interface_attachment":
        "SUPPORTED",

    "semantic_value_invariance":
        "NOT_REQUIRED",

    "admissibility_preservation":
        "NOT_IDENTIFIED",

    "logical_operation_preservation":
        "NOT_IDENTIFIED",

    "functoriality":
        "NOT_TESTED",

    "triadic_transport":
        "STRUCTURALLY_INSTANTIATED",

    "triadic_irreducibility":
        "NOT_TESTED",

    "triadic_coherence":
        "NOT_TESTED",

    "Phi_C":
        "NOT_RECONSTRUCTED",

    "contextual_geometry":
        "NOT_ESTABLISHED",

}


# ============================================================
# 24. SCIENTIFIC NON-CLAIMS
# ============================================================

scientific_nonclaims = {

    "CTL_empirical_satisfaction":
        "NOT_ESTABLISHED",

    "logical_transport_covariance_in_full":
        "NOT_ESTABLISHED",

    "admissibility_preservation":
        "NOT_ESTABLISHED",

    "logical_operation_preservation":
        "NOT_ESTABLISHED",

    "semantic_truth_preservation":
        "NOT_ESTABLISHED",

    "semantic_value_invariance":
        "NOT_REQUIRED_AND_NOT_TESTED",

    "functoriality":
        "NOT_ESTABLISHED",

    "triadic_irreducibility":
        "NOT_ESTABLISHED",

    "triadic_coherence":
        "NOT_ESTABLISHED",

    "contextual_geometry":
        "NOT_ESTABLISHED",

    "causal_transport":
        "NOT_ESTABLISHED",

    "numerical_transport_equals_logical_transport":
        "REJECTED",

}


# ============================================================
# 25. GLOBAL FIREWALLS
# ============================================================

global_firewalls = {

    "ctl_mathematics_modified":
        False,

    "cross_model_token_ids_compared":
        False,

    "state_bank_modified":
        False,

    "pca_refitted":
        False,

    "transport_refitted":
        False,

    "phi_C_fitted":
        False,

    "phi_D_fitted":
        False,

    "test_fitted":
        False,

    "hidden_state_threshold_admissibility":
        False,

    "semantic_threshold_admissibility":
        False,

    "boolean_collapse":
        False,

    "undefined_equals_false":
        False,

    "numerical_transport_promoted_to_logical":
        False,

    "triadic_irreducibility_claimed":
        False,

}


if any(
    global_firewalls.values()
):

    raise RuntimeError(
        "One or more global scientific governance firewalls failed."
    )


print(
    "\nGLOBAL SCIENTIFIC FIREWALL"
)

print(
    "-" * 80
)

for key, value in global_firewalls.items():

    print(
        f"  {key}: {value}"
    )


print(
    "[PASS] Global CTL governance firewalls."
)


# ============================================================
# 26. PRE-SERIALIZATION AUDIT
#
# This is added specifically to ensure that the artifact is
# internally serializable before writing it.
# ============================================================

print(
    "\nPRE-SERIALIZATION AUDIT"
)

print(
    "-" * 80
)

serialization_checks = {

    "row_order_classification_is_string":
        isinstance(
            row_order_classification,
            str
        ),

    "selected_row_order_is_string":
        isinstance(
            selected_row_order,
            str
        ),

    "clean_rows_serializable":
        isinstance(
            clean_rows.tolist(),
            list
        ),

    "corrupt_rows_serializable":
        isinstance(
            corrupt_rows.tolist(),
            list
        ),

    "covariance_registry_serializable":
        isinstance(
            covariance_registry,
            dict
        ),

    "global_firewalls_serializable":
        isinstance(
            global_firewalls,
            dict
        ),

}


for key, value in serialization_checks.items():

    print(
        f"  {key}: {value}"
    )


if not all(
    serialization_checks.values()
):

    raise RuntimeError(
        "Pre-serialization audit failed."
    )


print(
    "[PASS] Pre-serialization audit."
)


# ============================================================
# 27. SERIALIZE FINAL ARTIFACT
# ============================================================

artifact = {

    "experiment_id":
        "ETTR-CTL-LLAMA-1",

    "phase":
        "1F.3",

    "title":
        "Logical Transport Covariance Audit",

    "predecessors":
        {

            "formal_ctl":
                {

                    "path":
                        str(
                            CTL_FORMAL_PATH
                        ),

                    "classification":
                        formal_classification,

                },

            "row_order":
                {

                    "path":
                        str(
                            ROW_ORDER_PATH
                        ),

                    # FIX:
                    # row_order_classification is a STRING.
                    "classification":
                        row_order_classification,

                    # FIX:
                    # selected_row_order is a STRING.
                    "selected_row_order":
                        selected_row_order,

                },

            "semantic_realization":
                {

                    "path":
                        str(
                            SEMANTIC_PATH
                        ),

                    "classification":
                        semantic_classification,

                },

        },

    "dataset":
        {

            "path":
                str(
                    dataset_path
                ),

            "sha256":
                dataset_sha,

            "records":
                N_RECORDS,

        },

    "state_bank_hashes":
        EXPECTED_STATE_HASHES,

    "row_alignment":
        {

            "clean_rows":
                clean_rows.tolist(),

            "corrupt_rows":
                corrupt_rows.tolist(),

            "paired_pT":
                True,

            "paired_iT":
                True,

        },

    "ctl_structure":
        {

            "contexts":
                N_RECORDS,

            "logical_carriers":
                len(
                    logical_carriers
                ),

            "events":
                len(
                    events
                ),

            "logical_transports":
                len(
                    logical_transports
                ),

            "numerical_transport_identified_with_logical":
                False,

        },

    "covariance_registry":
        {

            "objects":
                len(
                    covariance_registry
                ),

            "type_level_pass":
                covariance_type_pass,

            "semantic_value_invariance_required":
                False,

        },

    "admissibility_transport":
        admissibility_transport_status,

    "logical_operation_preservation":
        logical_operation_status,

    "numerical_logical_firewall":
        numerical_logical_firewall,

    "composition":
        composition_status,

    "triadic_transport":
        triadic_transport_status,

    "scientific_status":
        scientific_status,

    "scientific_nonclaims":
        scientific_nonclaims,

    "global_firewalls":
        global_firewalls,

    "final_classification":
        "LOGICAL_TRANSPORT_STRUCTURE_OPERATIONAL_PASS",

}


with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 28. POST-SERIALIZATION VERIFICATION
# ============================================================

if not OUTPUT_PATH.exists():

    raise RuntimeError(
        "Expected Phase 1F.3 artifact was not created."
    )


with open(
    OUTPUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    artifact_check = json.load(
        f
    )


if artifact_check.get(
    "final_classification"
) != (
    "LOGICAL_TRANSPORT_STRUCTURE_OPERATIONAL_PASS"
):

    raise RuntimeError(
        "Serialized artifact classification is incorrect."
    )


serialized_row_order = artifact_check[
    "predecessors"
][
    "row_order"
]


if serialized_row_order.get(
    "classification"
) != row_order_classification:

    raise RuntimeError(
        "Serialized row-order classification mismatch."
    )


if serialized_row_order.get(
    "selected_row_order"
) != selected_row_order:

    raise RuntimeError(
        "Serialized row-order selection mismatch."
    )


print(
    "\nPOST-SERIALIZATION VERIFICATION"
)

print(
    "-" * 80
)

print(
    "  artifact exists: True"
)

print(
    "  JSON readback: PASS"
)

print(
    "  classification readback: PASS"
)

print(
    "  row-order metadata readback: PASS"
)


# ============================================================
# 29. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "FINAL PHASE 1F.3 STATUS"
)

print(
    "=" * 80
)

print(
    "  LOGICAL_TRANSPORT_STRUCTURE_OPERATIONAL_PASS"
)

print(
    "  context-indexed transport objects: PASS"
)

print(
    "  sector typing: PASS"
)

print(
    "  semantic interface attachment: PASS"
)

print(
    "  numerical transport = logical transport: REJECTED"
)

print(
    "  admissibility preservation: NOT IDENTIFIED"
)

print(
    "  logical operation preservation: NOT IDENTIFIED"
)

print(
    "  functoriality: NOT TESTED"
)

print(
    "  triadic transport structure: INSTANTIATED"
)

print(
    "  triadic irreducibility: NOT TESTED"
)

print(
    "  triadic coherence: NOT TESTED"
)

print(
    "  Phi_C: NOT RECONSTRUCTED"
)

print(
    "  contextual geometry: NOT ESTABLISHED"
)

print(
    f"  artifact: {OUTPUT_PATH}"
)

print(
    "=" * 80
)

ETTR-CTL-LLAMA-1 — PHASE 1F.3
LOGICAL TRANSPORT COVARIANCE AUDIT
ROOT: /content/ettr_ctl_llama
State bank: /content/ettr_ctl_llama/results/llama_phase1d2_full_state_bank.npz
CTL formal audit: /content/ettr_ctl_llama/results/llama_phase1f1_formal_ctl_structure_audit.json
Semantic realization: /content/ettr_ctl_llama/results/llama_phase1f2_ctl_semantic_realization_audit.json
Output: /content/ettr_ctl_llama/results/llama_phase1f3_logical_transport_covariance_audit.json

GOVERNANCE
--------------------------------------------------------------------------------
  CTL formal structure: FROZEN
  verified state-bank ordering: REQUIRED
  semantic realization: FROZEN
  numerical transport = logical transport: FORBIDDEN
  numerical transport maps used as CTL morphisms: FORBIDDEN
  hidden-state threshold admissibility: FORBIDDEN
  semantic threshold admissibility: FORBIDDEN
  Boolean conjunction as triadic admissibility: FORBIDDEN
  undefined = false: FORBIDDEN
  admissibility fabrication: FORBID

In [43]:
# ============================================================
# ETTR-CTL-LLAMA-1
# PHASE 1F.4 — TRIADIC COHERENCE / Phi_C INTERFACE AUDIT
#
# NEW CELL — DO NOT REMOVE PREVIOUS CELLS.
#
# PURPOSE
# -------
# Determine whether the frozen empirical system contains an
# identifiable operational realization of the CTL coherence
# interface
#
#                  Phi_C : Adm_C^(3) -> K_C
#
# without conflating:
#
#   (1) numerical sector transport,
#   (2) semantic valuation,
#   (3) contextual admissibility,
#   (4) triadic logical structure,
#   (5) structural coherence.
#
# FORMAL GOVERNANCE
# -----------------
# The CTL formal layer distinguishes semantic truth from
# structural coherence. In particular, Phi_C is not defined
# by semantic valuation, and semantic covariance does not by
# itself establish Phi_C.
#
# The current Llama experiment has:
#
#   - context-indexed carriers;
#   - three operational sector carriers;
#   - a typed logical transport registry;
#   - frozen numerical sector states;
#   - frozen semantic observations;
#   - a formal triadic admissibility carrier;
#   - a formal coherence carrier K_C.
#
# But previous phases established that empirical admissibility
# has NOT been identified.
#
# Therefore this phase must distinguish:
#
#   A. structural instantiation of Phi_C's TYPE;
#   B. empirical identification of its DOMAIN;
#   C. empirical identification of its CODOMAIN;
#   D. empirical reconstruction of the MAP ITSELF;
#   E. empirical coherence testing.
#
# A type-level Phi_C object is permissible.
# A fitted Phi_C is NOT permissible unless its empirical
# domain and codomain are independently identified.
#
# GOVERNING PRINCIPLE
# -------------------
# Mathematics remains authoritative.
#
# If the experiment does not identify a formal object, record
# NOT_IDENTIFIED rather than replacing the object with a
# convenient numerical surrogate.
#
# In particular:
#
#   hidden-state vectors != K_C by default
#   target logits != truth values by default
#   Boolean predicates != admissibility by default
#   numerical transport != logical transport
#   regression fit != Phi_C reconstruction
#
# No PCA, transport map, Phi_C, Phi_D, or test data are refit.
# ============================================================


import os
import json
import hashlib
from pathlib import Path
from dataclasses import dataclass
import numpy as np


# ============================================================
# 0. PATHS
# ============================================================

ROOT = Path(
    "/content/ettr_ctl_llama"
)

RESULTS = ROOT / "results"

STATE_BANK_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank.npz"
)

STATE_MANIFEST_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank_manifest.json"
)

AUTH_PATH = (
    RESULTS /
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

CTL_FORMAL_PATH = (
    RESULTS /
    "llama_phase1f1_formal_ctl_structure_audit.json"
)

ROW_ORDER_PATH = (
    RESULTS /
    "llama_phase1f2a_state_bank_row_order_audit.json"
)

SEMANTIC_PATH = (
    RESULTS /
    "llama_phase1f2_ctl_semantic_realization_audit.json"
)

LOGICAL_TRANSPORT_PATH = (
    RESULTS /
    "llama_phase1f3_logical_transport_covariance_audit.json"
)

OUTPUT_PATH = (
    RESULTS /
    "llama_phase1f4_triadic_coherence_phiC_interface_audit.json"
)


EXPECTED_DATASET_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)


EXPECTED_STATE_HASHES = {

    "S1":
        "9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783",

    "S2":
        "41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb",

    "S3":
        "173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3",

}


SECTORS = (
    "S1",
    "S2",
    "S3",
)

N_RECORDS = 192
N_CONDITIONS = 384
HIDDEN_DIM = 3072


# ============================================================
# 1. HEADER
# ============================================================

print(
    "=" * 80
)

print(
    "ETTR-CTL-LLAMA-1 — PHASE 1F.4"
)

print(
    "TRIADIC COHERENCE / Phi_C INTERFACE AUDIT"
)

print(
    "=" * 80
)

print(
    f"ROOT: {ROOT}"
)

print(
    f"State bank: {STATE_BANK_PATH}"
)

print(
    f"CTL formal audit: {CTL_FORMAL_PATH}"
)

print(
    f"Logical transport audit: {LOGICAL_TRANSPORT_PATH}"
)

print(
    f"Output: {OUTPUT_PATH}"
)


# ============================================================
# 2. GOVERNANCE
# ============================================================

print(
    "\nGOVERNANCE"
)

print(
    "-" * 80
)

governance = {

    "CTL mathematics":
        "FROZEN",

    "context-indexed carriers":
        "REQUIRED",

    "triadic carrier":
        "REQUIRED",

    "Phi_C domain = empirical admissibility":
        "REQUIRED",

    "Phi_C codomain = coherence carrier K_C":
        "REQUIRED",

    "hidden state = K_C":
        "FORBIDDEN",

    "semantic valuation = Phi_C":
        "FORBIDDEN",

    "target logit = truth value":
        "FORBIDDEN",

    "Boolean conjunction = triadic admissibility":
        "FORBIDDEN",

    "undefined = false":
        "FORBIDDEN",

    "numerical transport = logical transport":
        "FORBIDDEN",

    "numerical reconstruction = Phi_C":
        "FORBIDDEN",

    "Phi_C fitting without identified domain":
        "FORBIDDEN",

    "Phi_C fitting without identified codomain":
        "FORBIDDEN",

    "PCA refitting":
        "FORBIDDEN",

    "transport refitting":
        "FORBIDDEN",

    "test fitting":
        "FORBIDDEN",

    "triadic irreducibility":
        "NOT TESTED",

    "contextual geometry":
        "NOT TESTED",

}


for key, value in governance.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 3. REQUIRED ARTIFACTS
# ============================================================

required_paths = [

    STATE_BANK_PATH,
    STATE_MANIFEST_PATH,
    AUTH_PATH,
    CTL_FORMAL_PATH,
    ROW_ORDER_PATH,
    SEMANTIC_PATH,
    LOGICAL_TRANSPORT_PATH,

]


missing = [

    str(path)

    for path in required_paths

    if not path.exists()

]


if missing:

    raise FileNotFoundError(
        "Required predecessor artifact(s) missing:\n"
        +
        "\n".join(
            missing
        )
    )


print(
    "\n[PASS] Required frozen artifacts exist."
)


# ============================================================
# 4. LOAD CTL FORMAL PREDECESSOR
# ============================================================

with open(
    CTL_FORMAL_PATH,
    "r",
    encoding="utf-8"
) as f:

    ctl_formal = json.load(
        f
    )


formal_classification = ctl_formal.get(
    "final_classification"
)


if formal_classification not in (
    "FORMAL_CTL_STRUCTURE_IMPLEMENTATION_PASS",
    "CTL_FORMAL_STRUCTURE_IMPLEMENTATION_PASS",
):

    raise RuntimeError(
        "Phase 1F.1 formal CTL predecessor is not PASS. "
        f"Observed: {formal_classification}"
    )


print(
    "[PASS] Formal CTL predecessor."
)


# ============================================================
# 5. LOAD ROW-ORDER PREDECESSOR
# ============================================================

with open(
    ROW_ORDER_PATH,
    "r",
    encoding="utf-8"
) as f:

    row_order = json.load(
        f
    )


row_order_classification = row_order.get(
    "classification"
)

selected_row_order = row_order.get(
    "selected_row_order"
)


if row_order_classification != (
    "FROZEN_STATE_BANK_ROW_ORDER_VERIFIED"
):

    raise RuntimeError(
        "Phase 1F.2A row-order predecessor is not PASS."
    )


if selected_row_order != (
    "interleaved_clean_corrupt"
):

    raise RuntimeError(
        "Unexpected frozen state-bank row order."
    )


clean_rows = np.asarray(
    row_order[
        "clean_rows"
    ],
    dtype=np.int64
)

corrupt_rows = np.asarray(
    row_order[
        "corrupt_rows"
    ],
    dtype=np.int64
)


if not np.array_equal(
    clean_rows,
    np.arange(
        0,
        N_CONDITIONS,
        2,
        dtype=np.int64
    )
):

    raise RuntimeError(
        "Clean row mapping is inconsistent."
    )


if not np.array_equal(
    corrupt_rows,
    np.arange(
        1,
        N_CONDITIONS,
        2,
        dtype=np.int64
    )
):

    raise RuntimeError(
        "Corrupt row mapping is inconsistent."
    )


print(
    "\nROW-ORDER PREDECESSOR"
)

print(
    "-" * 80
)

print(
    f"  classification: {row_order_classification}"
)

print(
    f"  row order: {selected_row_order}"
)

print(
    "[PASS] Frozen row-order correspondence."
)


# ============================================================
# 6. LOAD SEMANTIC PREDECESSOR
# ============================================================

with open(
    SEMANTIC_PATH,
    "r",
    encoding="utf-8"
) as f:

    semantic = json.load(
        f
    )


semantic_classification = semantic.get(
    "final_classification"
)


if semantic_classification != (
    "CTL_SEMANTIC_REALIZATION_OPERATIONAL_PASS"
):

    raise RuntimeError(
        "Phase 1F.2 semantic realization is not PASS. "
        f"Observed: {semantic_classification}"
    )


print(
    "\nSEMANTIC PREDECESSOR"
)

print(
    "-" * 80
)

print(
    f"  classification: {semantic_classification}"
)

print(
    "[PASS] Frozen semantic realization."
)


# ============================================================
# 7. LOAD LOGICAL TRANSPORT PREDECESSOR
# ============================================================

with open(
    LOGICAL_TRANSPORT_PATH,
    "r",
    encoding="utf-8"
) as f:

    logical_transport = json.load(
        f
    )


logical_transport_classification = (
    logical_transport.get(
        "final_classification"
    )
)


if logical_transport_classification != (
    "LOGICAL_TRANSPORT_STRUCTURE_OPERATIONAL_PASS"
):

    raise RuntimeError(
        "Phase 1F.3 logical transport predecessor is not PASS. "
        f"Observed: {logical_transport_classification}"
    )


print(
    "\nLOGICAL TRANSPORT PREDECESSOR"
)

print(
    "-" * 80
)

print(
    f"  classification: "
    f"{logical_transport_classification}"
)

print(
    "  numerical transport = logical transport: NO"
)

print(
    "[PASS] Frozen logical transport structure."
)


# ============================================================
# 8. LOAD DATASET AUTHORIZATION
# ============================================================

with open(
    AUTH_PATH,
    "r",
    encoding="utf-8"
) as f:

    auth = json.load(
        f
    )


dataset_sha = (
    auth.get(
        "dataset_sha256"
    )
    or
    auth.get(
        "sha256"
    )
)


authorized = bool(
    auth.get(
        "dataset_authorized",
        auth.get(
            "authorized",
            False
        )
    )
)


if not authorized:

    raise RuntimeError(
        "Dataset authorization failed."
    )


if dataset_sha != EXPECTED_DATASET_SHA256:

    raise RuntimeError(
        "Dataset SHA mismatch."
    )


print(
    "\nDATASET AUTHORIZATION"
)

print(
    "-" * 80
)

print(
    f"  authorized: {authorized}"
)

print(
    f"  SHA: {dataset_sha}"
)

print(
    "[PASS] Dataset authorization."
)


# ============================================================
# 9. LOCATE OPERATIVE DATASET
# ============================================================

dataset_candidates = [

    Path(
        "gpt2_controlled_dataset_v1.1_recovered_r1.json"
    ),

    Path(
        "/content/gpt2_controlled_dataset_v1.1_recovered_r1.json"
    ),

]


for base in (
    ROOT,
    Path("/content"),
):

    if base.exists():

        dataset_candidates.extend(
            base.rglob(
                "gpt2_controlled_dataset_v1.1_recovered_r1.json"
            )
        )


dataset_candidates = list(
    dict.fromkeys(
        dataset_candidates
    )
)


def load_records(
    path
):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        obj = json.load(
            f
        )


    if isinstance(
        obj,
        list
    ):

        return obj


    if isinstance(
        obj,
        dict
    ):

        for key in (
            "records",
            "dataset",
            "examples",
            "data",
            "items",
        ):

            if isinstance(
                obj.get(key),
                list
            ):

                return obj[
                    key
                ]


    return None


required_fields = {

    "example_id",
    "name_pair_id",
    "name_pair_index",
    "template_id",
    "template_index",
    "split",
    "clean_prompt",
    "corrupt_prompt",
    "clean_target_name",
    "corrupt_target_name",

}


valid_dataset_candidates = []


for path in dataset_candidates:

    if not path.exists():

        continue


    try:

        candidate_records = load_records(
            path
        )

    except Exception:

        continue


    if candidate_records is None:

        continue


    if len(
        candidate_records
    ) != N_RECORDS:

        continue


    if not required_fields.issubset(
        set(
            candidate_records[0].keys()
        )
    ):

        continue


    valid_dataset_candidates.append(
        (
            path,
            candidate_records
        )
    )


if not valid_dataset_candidates:

    raise FileNotFoundError(
        "No valid operative 192-record dataset found."
    )


def identity_signature(
    record
):

    return (

        str(
            record[
                "example_id"
            ]
        ),

        str(
            record[
                "name_pair_id"
            ]
        ),

        int(
            record[
                "name_pair_index"
            ]
        ),

        str(
            record[
                "template_id"
            ]
        ),

        int(
            record[
                "template_index"
            ]
        ),

        str(
            record[
                "split"
            ]
        ),

    )


reference_identity = [

    identity_signature(
        record
    )

    for record
    in valid_dataset_candidates[0][1]

]


for path, candidate_records in valid_dataset_candidates[1:]:

    candidate_identity = [

        identity_signature(
            record
        )

        for record
        in candidate_records

    ]


    if candidate_identity != reference_identity:

        raise RuntimeError(
            "Dataset copies disagree in identity/order."
        )


dataset_path, records = (
    valid_dataset_candidates[0]
)


print(
    "\nOPERATIVE DATASET"
)

print(
    "-" * 80
)

print(
    f"  path: {dataset_path}"
)

print(
    f"  records: {len(records)}"
)

print(
    "[PASS] Operative dataset identified."
)


# ============================================================
# 10. LOAD AND HASH FROZEN STATE BANK
# ============================================================

bank = np.load(
    STATE_BANK_PATH,
    allow_pickle=False
)


state_arrays = {}


for sector in SECTORS:

    X = np.asarray(
        bank[
            sector
        ]
    )


    if X.shape != (
        N_CONDITIONS,
        HIDDEN_DIM
    ):

        raise RuntimeError(
            f"{sector} shape mismatch: {X.shape}"
        )


    actual_hash = hashlib.sha256(

        np.ascontiguousarray(
            X
        ).tobytes(
            order="C"
        )

    ).hexdigest()


    if actual_hash != EXPECTED_STATE_HASHES[
        sector
    ]:

        raise RuntimeError(
            f"{sector} frozen hash mismatch."
        )


    if not np.all(
        np.isfinite(
            X
        )
    ):

        raise RuntimeError(
            f"{sector} contains non-finite values."
        )


    state_arrays[
        sector
    ] = X


print(
    "\nSTATE-BANK FIREWALL"
)

print(
    "-" * 80
)

for sector in SECTORS:

    print(
        f"  {sector}: "
        f"shape=(384,3072), "
        f"hash_match=True, "
        f"finite=True"
    )


print(
    "[PASS] Frozen state-bank integrity."
)


# ============================================================
# 11. LOAD FROZEN SEMANTIC INTERFACE
# ============================================================

target_logits = np.asarray(
    bank[
        "target_logits"
    ],
    dtype=np.float64
)

target_ranks = np.asarray(
    bank[
        "target_logit_ranks"
    ],
    dtype=np.int64
)

target_ids = np.asarray(
    bank[
        "condition_llama_target_ids"
    ],
    dtype=np.int64
)

state_pT = np.asarray(
    bank[
        "condition_p_T"
    ],
    dtype=np.int64
)

state_iT = np.asarray(
    bank[
        "condition_i_T"
    ],
    dtype=np.int64
)


for name, arr in (
    ("target_logits", target_logits),
    ("target_ranks", target_ranks),
    ("target_ids", target_ids),
    ("p_T", state_pT),
    ("i_T", state_iT),
):

    if arr.shape != (
        N_CONDITIONS,
    ):

        raise RuntimeError(
            f"{name} shape mismatch."
        )


if not np.all(
    np.isfinite(
        target_logits
    )
):

    raise RuntimeError(
        "Target logits contain non-finite values."
    )


print(
    "\nFROZEN SEMANTIC INTERFACE"
)

print(
    "-" * 80
)

print(
    "  target logits: 384"
)

print(
    "  target ranks: 384"
)

print(
    "  target IDs: 384"
)

print(
    "  target prediction positions: 384"
)

print(
    "[PASS] Frozen semantic interface."
)


# ============================================================
# 12. PAIRED CONTEXT DATA
# ============================================================

clean_logits = target_logits[
    clean_rows
]

corrupt_logits = target_logits[
    corrupt_rows
]

clean_ranks = target_ranks[
    clean_rows
]

corrupt_ranks = target_ranks[
    corrupt_rows
]

clean_ids = target_ids[
    clean_rows
]

corrupt_ids = target_ids[
    corrupt_rows
]

clean_pT = state_pT[
    clean_rows
]

corrupt_pT = state_pT[
    corrupt_rows
]

clean_iT = state_iT[
    clean_rows
]

corrupt_iT = state_iT[
    corrupt_rows
]


if not np.all(
    clean_pT
    ==
    corrupt_pT
):

    raise RuntimeError(
        "Paired p_T alignment failed."
    )


if not np.all(
    clean_iT
    ==
    corrupt_iT
):

    raise RuntimeError(
        "Paired i_T alignment failed."
    )


print(
    "\nPAIRED CONTEXT ALIGNMENT"
)

print(
    "-" * 80
)

print(
    "  p_T paired: PASS"
)

print(
    "  i_T paired: PASS"
)

print(
    "  clean/corrupt target interface aligned: PASS"
)


# ============================================================
# 13. FORMAL CTL COHERENCE OBJECTS
# ============================================================
#
# We now instantiate the TYPE of the formal coherence layer.
#
# Crucially, these are typed mathematical objects.
#
# They are NOT populated with empirical hidden-state values.
#
# For each context C:
#
#       Adm_C^(3)
#
# is represented as a formal domain object, and
#
#       K_C
#
# is represented as a formal coherence-carrier object.
#
# Their empirical contents remain unresolved.
# ============================================================

@dataclass(
    frozen=True
)
class CTLTriadicAdmissibilityDomain:

    context_index: int

    carrier_type: str

    empirical_status: str


@dataclass(
    frozen=True
)
class CTLCoherenceCarrier:

    context_index: int

    carrier_type: str

    empirical_status: str


@dataclass(
    frozen=True
)
class CTLPhiCInterface:

    context_index: int

    domain_type: str

    codomain_type: str

    map_status: str

    empirical_status: str


triadic_admissibility = {}

coherence_carriers = {}

phiC_interfaces = {}


for i in range(
    N_RECORDS
):

    triadic_admissibility[
        i
    ] = CTLTriadicAdmissibilityDomain(

        context_index=i,

        carrier_type=
            "CTL_TRIADIC_ADMISSIBILITY_DOMAIN",

        empirical_status=
            "NOT_IDENTIFIED",

    )


    coherence_carriers[
        i
    ] = CTLCoherenceCarrier(

        context_index=i,

        carrier_type=
            "CTL_COHERENCE_CARRIER_K_C",

        empirical_status=
            "NOT_IDENTIFIED",

    )


    phiC_interfaces[
        i
    ] = CTLPhiCInterface(

        context_index=i,

        domain_type=
            "Adm_C^(3)",

        codomain_type=
            "K_C",

        map_status=
            "FORMALLY_TYPED",

        empirical_status=
            "NOT_RECONSTRUCTED",

    )


if len(
    triadic_admissibility
) != N_RECORDS:

    raise RuntimeError(
        "Triadic admissibility domain count mismatch."
    )


if len(
    coherence_carriers
) != N_RECORDS:

    raise RuntimeError(
        "Coherence carrier count mismatch."
    )


if len(
    phiC_interfaces
) != N_RECORDS:

    raise RuntimeError(
        "Phi_C interface count mismatch."
    )


print(
    "\nFORMAL Phi_C INTERFACE"
)

print(
    "-" * 80
)

print(
    f"  Adm_C^(3) typed domains: "
    f"{len(triadic_admissibility)}"
)

print(
    f"  K_C coherence carriers: "
    f"{len(coherence_carriers)}"
)

print(
    f"  Phi_C interfaces: "
    f"{len(phiC_interfaces)}"
)

print(
    "  empirical admissibility: NOT IDENTIFIED"
)

print(
    "  empirical K_C: NOT IDENTIFIED"
)

print(
    "  Phi_C map: FORMALLY TYPED ONLY"
)

print(
    "[PASS] Formal Phi_C interface instantiated without "
    "empirical substitution."
)


# ============================================================
# 14. DOMAIN IDENTIFIABILITY AUDIT
# ============================================================
#
# A genuine empirical Phi_C reconstruction requires an
# identified empirical domain:
#
#       Adm_C^(3)
#
# The current experiment does not contain an independently
# identified admissibility relation.
#
# We explicitly test whether one can be derived from existing
# frozen observables without introducing a new arbitrary rule.
#
# Permitted observations:
#
#   hidden states
#   target logits
#   target ranks
#   token IDs
#   p_T / i_T
#
# None is formally an admissibility relation by itself.
# ============================================================

available_observables = {

    "sector_hidden_states":
        True,

    "target_logits":
        True,

    "target_ranks":
        True,

    "target_token_ids":
        True,

    "target_positions":
        True,

    "independently_identified_admissibility_relation":
        False,

}


admissibility_derivation_without_new_rule = False


domain_identifiable = (
    available_observables[
        "independently_identified_admissibility_relation"
    ]
    or
    admissibility_derivation_without_new_rule
)


print(
    "\nPhi_C DOMAIN IDENTIFIABILITY"
)

print(
    "-" * 80
)

print(
    "  frozen observables available: YES"
)

print(
    "  independent admissibility relation: NO"
)

print(
    "  admissibility derivable without new rule: NO"
)

print(
    "  empirical Adm_C^(3) identifiable: NO"
)

print(
    "[PASS] Domain identifiability correctly classified "
    "as NOT_IDENTIFIED."
)


# ============================================================
# 15. CODOMAIN IDENTIFIABILITY AUDIT
# ============================================================
#
# A genuine empirical Phi_C reconstruction also requires an
# identified coherence carrier K_C.
#
# The frozen hidden-state sectors are not automatically K_C.
# The target prediction interface is not automatically K_C.
#
# Therefore:
#
#       K_C empirical realization = NOT IDENTIFIED
#
# unless an independently specified observable representation
# has already been frozen as the coherence carrier.
# ============================================================

codomain_candidates = {

    "S1_hidden_state":
        False,

    "S2_hidden_state":
        False,

    "S3_hidden_state":
        False,

    "concatenated_hidden_state":
        False,

    "target_logit":
        False,

    "target_rank":
        False,

    "target_token_id":
        False,

    "independently_frozen_K_C_representation":
        False,

}


identified_codomain_candidates = [

    key

    for key, value
    in codomain_candidates.items()

    if value

]


codomain_identifiable = (
    len(
        identified_codomain_candidates
    )
    >
    0
)


print(
    "\nPhi_C CODOMAIN IDENTIFIABILITY"
)

print(
    "-" * 80
)

print(
    "  hidden states designated as K_C: NO"
)

print(
    "  semantic observations designated as K_C: NO"
)

print(
    "  independent K_C representation frozen: NO"
)

print(
    "  empirical K_C identifiable: NO"
)

print(
    "[PASS] Codomain identifiability correctly classified "
    "as NOT_IDENTIFIED."
)


# ============================================================
# 16. MAP IDENTIFIABILITY AUDIT
# ============================================================
#
# Since both domain and codomain are unresolved, Phi_C itself
# cannot be empirically reconstructed.
#
# This is not a computational failure.
#
# It is a scientific identifiability result.
# ============================================================

phiC_map_identifiable = (
    domain_identifiable
    and
    codomain_identifiable
)


if phiC_map_identifiable:

    raise RuntimeError(
        "Unexpected Phi_C identifiability. "
        "Review the formal domain/codomain governance."
    )


print(
    "\nPhi_C MAP IDENTIFIABILITY"
)

print(
    "-" * 80
)

print(
    "  domain identifiable: NO"
)

print(
    "  codomain identifiable: NO"
)

print(
    "  Phi_C empirically reconstructible: NO"
)

print(
    "[PASS] Phi_C map correctly classified "
    "as NOT_RECONSTRUCTED."
)


# ============================================================
# 17. NUMERICAL SURROGATE FIREWALL
# ============================================================
#
# Explicitly verify that no existing numerical object is being
# silently substituted for Phi_C.
# ============================================================

numerical_surrogate_firewall = {

    "S1_hidden_state_used_as_K_C":
        False,

    "S2_hidden_state_used_as_K_C":
        False,

    "S3_hidden_state_used_as_K_C":
        False,

    "concatenated_sector_state_used_as_K_C":
        False,

    "target_logit_used_as_K_C":
        False,

    "target_rank_used_as_K_C":
        False,

    "numerical_transport_output_used_as_Phi_C":
        False,

    "Phi_D_used_as_Phi_C":
        False,

    "regression_output_called_Phi_C":
        False,

}


if any(
    numerical_surrogate_firewall.values()
):

    raise RuntimeError(
        "A numerical surrogate was incorrectly promoted "
        "to the CTL coherence layer."
    )


print(
    "\nNUMERICAL SURROGATE FIREWALL"
)

print(
    "-" * 80
)

for key, value in numerical_surrogate_firewall.items():

    print(
        f"  {key}: {value}"
    )


print(
    "[PASS] No numerical object was substituted for K_C or Phi_C."
)


# ============================================================
# 18. TRIADIC COHERENCE TESTABILITY
# ============================================================
#
# Full empirical CTL coherence would require evaluating a
# relation of the form
#
#   Phi_C(
#       triadic admissibility structure
#   )
#
# against the corresponding transported representation and/or
# coherence condition.
#
# Since Adm_C^(3), K_C, and Phi_C are not empirically
# identified, this cannot presently be evaluated.
# ============================================================

coherence_requirements = {

    "triadic_admissibility_identified":
        False,

    "K_C_identified":
        False,

    "Phi_C_reconstructed":
        False,

    "logical_transport_empirically_realized":
        False,

    "coherence_residual_observable":
        False,

}


full_coherence_testable = all(
    coherence_requirements.values()
)


if full_coherence_testable:

    raise RuntimeError(
        "Unexpected full coherence identifiability."
    )


print(
    "\nTRIADIC COHERENCE TESTABILITY"
)

print(
    "-" * 80
)

for key, value in coherence_requirements.items():

    print(
        f"  {key}: {value}"
    )


print(
    "  full empirical coherence test: NOT IDENTIFIED"
)

print(
    "[PASS] Coherence testability correctly constrained."
)


# ============================================================
# 19. WHAT IS ACTUALLY SUPPORTED
# ============================================================
#
# The experiment DOES support:
#
#   - existence of 192 formal context-indexed triadic
#     admissibility domains;
#   - existence of 192 formal K_C carrier objects;
#   - 192 formally typed Phi_C interfaces;
#   - preservation of sector/context indexing;
#   - frozen numerical and semantic data attached to the
#     corresponding contexts.
#
# It does NOT support:
#
#   - empirical admissibility;
#   - empirical K_C values;
#   - empirical Phi_C values;
#   - empirical coherence defect;
#   - semantic truth = coherence;
#   - triadic irreducibility.
# ============================================================

supported_structure = {

    "formal_triadic_admissibility_domains":
        len(
            triadic_admissibility
        ),

    "formal_K_C_carriers":
        len(
            coherence_carriers
        ),

    "formal_Phi_C_interfaces":
        len(
            phiC_interfaces
        ),

    "context_index_preserved":
        True,

    "sector_index_preserved":
        True,

    "empirical_admissibility":
        False,

    "empirical_K_C":
        False,

    "empirical_Phi_C":
        False,

    "empirical_coherence_defect":
        False,

}


print(
    "\nSUPPORTED STRUCTURE"
)

print(
    "-" * 80
)

print(
    f"  formal Adm_C^(3): "
    f"{supported_structure['formal_triadic_admissibility_domains']}"
)

print(
    f"  formal K_C: "
    f"{supported_structure['formal_K_C_carriers']}"
)

print(
    f"  formal Phi_C interfaces: "
    f"{supported_structure['formal_Phi_C_interfaces']}"
)

print(
    "  empirical admissibility: NOT IDENTIFIED"
)

print(
    "  empirical K_C: NOT IDENTIFIED"
)

print(
    "  empirical Phi_C: NOT RECONSTRUCTED"
)

print(
    "  empirical coherence defect: NOT AVAILABLE"
)


# ============================================================
# 20. TRIADIC / DYADIC COLLAPSE FIREWALL
# ============================================================
#
# Merely having three carriers does not establish triadic
# irreducibility.
#
# No dyadic reduction is performed in this phase.
# ============================================================

triadic_firewall = {

    "three_sector_registry_present":
        True,

    "pairwise_reduction_performed":
        False,

    "triadic_structure_reduced_to_pairwise_logic":
        False,

    "triadic_irreducibility_claimed":
        False,

    "nonzero_three_way_interaction_interpreted_as_proof":
        False,

}


if any(
    value is True

    for key, value
    in triadic_firewall.items()

    if key not in (
        "three_sector_registry_present",
    )
):

    raise RuntimeError(
        "Triadic firewall indicates an unsupported reduction "
        "or irreducibility claim."
    )


print(
    "\nTRIADIC FIREWALL"
)

print(
    "-" * 80
)

for key, value in triadic_firewall.items():

    print(
        f"  {key}: {value}"
    )


print(
    "[PASS] No triadic irreducibility claim."
)


# ============================================================
# 21. SCIENTIFIC STATUS
# ============================================================

scientific_status = {

    "formal_Adm_C3":
        "STRUCTURALLY_INSTANTIATED",

    "formal_K_C":
        "STRUCTURALLY_INSTANTIATED",

    "formal_Phi_C":
        "TYPED_NOT_RECONSTRUCTED",

    "empirical_Adm_C3":
        "NOT_IDENTIFIED",

    "empirical_K_C":
        "NOT_IDENTIFIED",

    "empirical_Phi_C":
        "NOT_RECONSTRUCTED",

    "triadic_coherence_residual":
        "NOT_IDENTIFIED",

    "semantic_truth_coherence_equivalence":
        "NOT_ESTABLISHED",

    "numerical_coherence_surrogate":
        "REJECTED",

    "triadic_irreducibility":
        "NOT_TESTED",

    "logical_transport_covariance":
        "NOT_ESTABLISHED",

    "functoriality":
        "NOT_TESTED",

    "contextual_geometry":
        "NOT_ESTABLISHED",

}


# ============================================================
# 22. SCIENTIFIC NON-CLAIMS
# ============================================================

scientific_nonclaims = {

    "Phi_C_empirically_reconstructed":
        False,

    "K_C_identified_from_hidden_states":
        False,

    "admissibility_inferred_from_hidden_state_thresholds":
        False,

    "admissibility_inferred_from_semantic_thresholds":
        False,

    "semantic_truth_identified_with_coherence":
        False,

    "numerical_transport_identified_with_Phi_C":
        False,

    "triadic_irreducibility_established":
        False,

    "contextual_geometry_established":
        False,

}


# ============================================================
# 23. GLOBAL SCIENTIFIC FIREWALLS
# ============================================================

global_firewalls = {

    "ctl_mathematics_modified":
        False,

    "state_bank_modified":
        False,

    "dataset_modified":
        False,

    "cross_model_token_ids_compared":
        False,

    "pca_refitted":
        False,

    "transport_refitted":
        False,

    "Phi_C_fitted":
        False,

    "Phi_D_fitted":
        False,

    "test_fitted":
        False,

    "hidden_state_threshold_admissibility":
        False,

    "semantic_threshold_admissibility":
        False,

    "boolean_collapse":
        False,

    "undefined_equals_false":
        False,

    "hidden_state_promoted_to_K_C":
        False,

    "semantic_value_promoted_to_Phi_C":
        False,

    "numerical_transport_promoted_to_Phi_C":
        False,

    "triadic_irreducibility_claimed":
        False,

}


if any(
    global_firewalls.values()
):

    raise RuntimeError(
        "Global scientific firewall failure."
    )


print(
    "\nGLOBAL SCIENTIFIC FIREWALL"
)

print(
    "-" * 80
)

for key, value in global_firewalls.items():

    print(
        f"  {key}: {value}"
    )


print(
    "[PASS] Global CTL coherence governance firewalls."
)


# ============================================================
# 24. PRE-SERIALIZATION AUDIT
# ============================================================

serialization_checks = {

    "row_order_classification_is_string":
        isinstance(
            row_order_classification,
            str
        ),

    "selected_row_order_is_string":
        isinstance(
            selected_row_order,
            str
        ),

    "formal_classification_is_string":
        isinstance(
            formal_classification,
            str
        ),

    "semantic_classification_is_string":
        isinstance(
            semantic_classification,
            str
        ),

    "logical_transport_classification_is_string":
        isinstance(
            logical_transport_classification,
            str
        ),

    "supported_structure_is_dict":
        isinstance(
            supported_structure,
            dict
        ),

    "scientific_status_is_dict":
        isinstance(
            scientific_status,
            dict
        ),

    "global_firewalls_is_dict":
        isinstance(
            global_firewalls,
            dict
        ),

}


print(
    "\nPRE-SERIALIZATION AUDIT"
)

print(
    "-" * 80
)

for key, value in serialization_checks.items():

    print(
        f"  {key}: {value}"
    )


if not all(
    serialization_checks.values()
):

    raise RuntimeError(
        "Pre-serialization audit failed."
    )


print(
    "[PASS] Pre-serialization audit."
)


# ============================================================
# 25. SERIALIZE ARTIFACT
# ============================================================

artifact = {

    "experiment_id":
        "ETTR-CTL-LLAMA-1",

    "phase":
        "1F.4",

    "title":
        "Triadic Coherence / Phi_C Interface Audit",

    "predecessors":
        {

            "formal_ctl":
                {

                    "path":
                        str(
                            CTL_FORMAL_PATH
                        ),

                    "classification":
                        formal_classification,

                },

            "row_order":
                {

                    "path":
                        str(
                            ROW_ORDER_PATH
                        ),

                    "classification":
                        row_order_classification,

                    "selected_row_order":
                        selected_row_order,

                },

            "semantic_realization":
                {

                    "path":
                        str(
                            SEMANTIC_PATH
                        ),

                    "classification":
                        semantic_classification,

                },

            "logical_transport":
                {

                    "path":
                        str(
                            LOGICAL_TRANSPORT_PATH
                        ),

                    "classification":
                        logical_transport_classification,

                },

        },

    "dataset":
        {

            "path":
                str(
                    dataset_path
                ),

            "sha256":
                dataset_sha,

            "records":
                N_RECORDS,

        },

    "state_bank_hashes":
        EXPECTED_STATE_HASHES,

    "row_alignment":
        {

            "clean_rows":
                clean_rows.tolist(),

            "corrupt_rows":
                corrupt_rows.tolist(),

            "paired_pT":
                True,

            "paired_iT":
                True,

        },

    "formal_coherence_structure":
        {

            "triadic_admissibility_domains":
                len(
                    triadic_admissibility
                ),

            "coherence_carriers_K_C":
                len(
                    coherence_carriers
                ),

            "Phi_C_interfaces":
                len(
                    phiC_interfaces
                ),

            "domain_type":
                "Adm_C^(3)",

            "codomain_type":
                "K_C",

            "map_status":
                "FORMALLY_TYPED_ONLY",

        },

    "domain_identifiability":
        {

            "independent_admissibility_relation":
                False,

            "empirical_Adm_C3_identifiable":
                domain_identifiable,

            "status":
                "NOT_IDENTIFIED",

        },

    "codomain_identifiability":
        {

            "independent_K_C_representation":
                False,

            "empirical_K_C_identifiable":
                codomain_identifiable,

            "status":
                "NOT_IDENTIFIED",

        },

    "Phi_C_identifiability":
        {

            "domain_identifiable":
                domain_identifiable,

            "codomain_identifiable":
                codomain_identifiable,

            "map_identifiable":
                phiC_map_identifiable,

            "status":
                "NOT_RECONSTRUCTED",

        },

    "numerical_surrogate_firewall":
        numerical_surrogate_firewall,

    "coherence_testability":
        {

            "requirements":
                coherence_requirements,

            "full_empirical_coherence_testable":
                full_coherence_testable,

            "status":
                "NOT_IDENTIFIED",

        },

    "supported_structure":
        supported_structure,

    "triadic_firewall":
        triadic_firewall,

    "scientific_status":
        scientific_status,

    "scientific_nonclaims":
        scientific_nonclaims,

    "global_firewalls":
        global_firewalls,

    "final_classification":
        "TRIADIC_COHERENCE_INTERFACE_OPERATIONAL_PASS_NOT_EMPIRICALLY_IDENTIFIED",

}


with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 26. POST-SERIALIZATION VERIFICATION
# ============================================================

if not OUTPUT_PATH.exists():

    raise RuntimeError(
        "Phase 1F.4 artifact was not created."
    )


with open(
    OUTPUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    artifact_check = json.load(
        f
    )


expected_final_classification = (
    "TRIADIC_COHERENCE_INTERFACE_OPERATIONAL_PASS_NOT_EMPIRICALLY_IDENTIFIED"
)


if artifact_check.get(
    "final_classification"
) != expected_final_classification:

    raise RuntimeError(
        "Serialized Phase 1F.4 classification mismatch."
    )


if artifact_check[
    "Phi_C_identifiability"
][
    "map_identifiable"
]:

    raise RuntimeError(
        "Serialized artifact incorrectly claims Phi_C identifiability."
    )


if artifact_check[
    "domain_identifiability"
][
    "empirical_Adm_C3_identifiable"
]:

    raise RuntimeError(
        "Serialized artifact incorrectly claims empirical "
        "triadic admissibility identification."
    )


if artifact_check[
    "codomain_identifiability"
][
    "empirical_K_C_identifiable"
]:

    raise RuntimeError(
        "Serialized artifact incorrectly claims empirical K_C "
        "identification."
    )


print(
    "\nPOST-SERIALIZATION VERIFICATION"
)

print(
    "-" * 80
)

print(
    "  artifact exists: True"
)

print(
    "  JSON readback: PASS"
)

print(
    "  final classification: PASS"
)

print(
    "  Phi_C non-identifiability: PASS"
)

print(
    "  Adm_C^(3) non-identifiability: PASS"
)

print(
    "  K_C non-identifiability: PASS"
)


# ============================================================
# 27. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "FINAL PHASE 1F.4 STATUS"
)

print(
    "=" * 80
)

print(
    "  TRIADIC_COHERENCE_INTERFACE_OPERATIONAL_PASS_NOT_EMPIRICALLY_IDENTIFIED"
)

print(
    "  formal Adm_C^(3) structure: INSTANTIATED"
)

print(
    "  formal K_C structure: INSTANTIATED"
)

print(
    "  formal Phi_C interface: TYPED"
)

print(
    "  empirical Adm_C^(3): NOT IDENTIFIED"
)

print(
    "  empirical K_C: NOT IDENTIFIED"
)

print(
    "  empirical Phi_C: NOT RECONSTRUCTED"
)

print(
    "  triadic coherence residual: NOT IDENTIFIED"
)

print(
    "  semantic truth = coherence: NOT ESTABLISHED"
)

print(
    "  numerical surrogate for Phi_C: REJECTED"
)

print(
    "  triadic irreducibility: NOT TESTED"
)

print(
    "  logical transport covariance: NOT ESTABLISHED"
)

print(
    "  functoriality: NOT TESTED"
)

print(
    "  contextual geometry: NOT ESTABLISHED"
)

print(
    f"  artifact: {OUTPUT_PATH}"
)

print(
    "=" * 80
)

ETTR-CTL-LLAMA-1 — PHASE 1F.4
TRIADIC COHERENCE / Phi_C INTERFACE AUDIT
ROOT: /content/ettr_ctl_llama
State bank: /content/ettr_ctl_llama/results/llama_phase1d2_full_state_bank.npz
CTL formal audit: /content/ettr_ctl_llama/results/llama_phase1f1_formal_ctl_structure_audit.json
Logical transport audit: /content/ettr_ctl_llama/results/llama_phase1f3_logical_transport_covariance_audit.json
Output: /content/ettr_ctl_llama/results/llama_phase1f4_triadic_coherence_phiC_interface_audit.json

GOVERNANCE
--------------------------------------------------------------------------------
  CTL mathematics: FROZEN
  context-indexed carriers: REQUIRED
  triadic carrier: REQUIRED
  Phi_C domain = empirical admissibility: REQUIRED
  Phi_C codomain = coherence carrier K_C: REQUIRED
  hidden state = K_C: FORBIDDEN
  semantic valuation = Phi_C: FORBIDDEN
  target logit = truth value: FORBIDDEN
  Boolean conjunction = triadic admissibility: FORBIDDEN
  undefined = false: FORBIDDEN
  numerical transport = l

In [44]:
# ============================================================
# ETTR-CTL-LLAMA-1
# PHASE 1F.5 — TRIADIC IRREDUCIBILITY IDENTIFIABILITY /
#              EXPERIMENTAL SUFFICIENCY AUDIT
#
# NEW CELL — DO NOT REMOVE PREVIOUS CELLS.
#
# PURPOSE
# -------
# Determine whether the FROZEN Llama experimental system
# contains sufficient independent structure to test the CTL
# triadic irreducibility claim without redefining CTL or
# manufacturing a triadic signal.
#
# FORMAL TARGET
# -------------
# The CTL claim is NOT:
#
#       "there are three sectors"
#
# and NOT:
#
#       "the three-way interaction is nonzero."
#
# The relevant question is whether the triadic structure
# contains an invariant that cannot be reproduced by an
# appropriate lower-order / dyadic reduct.
#
# Accordingly, a valid empirical irreducibility test requires:
#
#   1. a clearly defined triadic object;
#   2. a lower-order reduct that preserves the appropriate
#      lower-order structure;
#   3. a complexity-controlled comparison;
#   4. held-out evaluation;
#   5. a pre-specified target observable;
#   6. preservation of context identity;
#   7. an actual structural advantage of the triadic model
#      over its dyadic reduction.
#
# A nonzero three-way interaction by itself is insufficient.
#
# CURRENT GOVERNANCE
# ------------------
# Phase 1F.4 established:
#
#   Adm_C^(3): formal structure instantiated
#   K_C:        formal structure instantiated
#   Phi_C:      typed but not empirically reconstructed
#
# Therefore this phase MUST determine whether irreducibility
# can be tested without relying on an empirically reconstructed
# Phi_C.
#
# It must NOT:
#
#   - fit Phi_C;
#   - invent admissibility;
#   - use hidden-state thresholds as logical predicates;
#   - equate hidden-state geometry with CTL coherence;
#   - treat the three sectors themselves as three Boolean
#     variables;
#   - reuse a nonzero three-way interaction as proof;
#   - refit the frozen transport maps;
#   - refit PCA;
#   - fit on the held-out test split;
#   - change the mathematical CTL definition.
#
# IMPORTANT
# ---------
# This is an IDENTIFIABILITY / SUFFICIENCY AUDIT.
#
# It is intentionally not the irreducibility experiment itself.
#
# The correct outcome may be:
#
#       TRIADIC_IRREDUCIBILITY_EXPERIMENTALLY_IDENTIFIABLE
#
# or:
#
#       TRIADIC_IRREDUCIBILITY_NOT_IDENTIFIABLE
#
# or an intermediate status identifying exactly which
# requirements are satisfied and which remain unavailable.
#
# ============================================================


import os
import json
import hashlib
from pathlib import Path
from dataclasses import dataclass

import numpy as np


# ============================================================
# 0. PATHS
# ============================================================

ROOT = Path(
    "/content/ettr_ctl_llama"
)

RESULTS = ROOT / "results"

STATE_BANK_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank.npz"
)

STATE_MANIFEST_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank_manifest.json"
)

AUTH_PATH = (
    RESULTS /
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

CTL_FORMAL_PATH = (
    RESULTS /
    "llama_phase1f1_formal_ctl_structure_audit.json"
)

ROW_ORDER_PATH = (
    RESULTS /
    "llama_phase1f2a_state_bank_row_order_audit.json"
)

SEMANTIC_PATH = (
    RESULTS /
    "llama_phase1f2_ctl_semantic_realization_audit.json"
)

LOGICAL_TRANSPORT_PATH = (
    RESULTS /
    "llama_phase1f3_logical_transport_covariance_audit.json"
)

COHERENCE_PATH = (
    RESULTS /
    "llama_phase1f4_triadic_coherence_phiC_interface_audit.json"
)

OUTPUT_PATH = (
    RESULTS /
    "llama_phase1f5_triadic_irreducibility_identifiability_audit.json"
)


EXPECTED_DATASET_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)


EXPECTED_STATE_HASHES = {

    "S1":
        "9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783",

    "S2":
        "41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb",

    "S3":
        "173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3",

}


SECTORS = (
    "S1",
    "S2",
    "S3",
)

N_RECORDS = 192
N_CONDITIONS = 384
HIDDEN_DIM = 3072


# ============================================================
# 1. HEADER
# ============================================================

print(
    "=" * 80
)

print(
    "ETTR-CTL-LLAMA-1 — PHASE 1F.5"
)

print(
    "TRIADIC IRREDUCIBILITY IDENTIFIABILITY / "
    "EXPERIMENTAL SUFFICIENCY AUDIT"
)

print(
    "=" * 80
)

print(
    f"ROOT: {ROOT}"
)

print(
    f"State bank: {STATE_BANK_PATH}"
)

print(
    f"Formal CTL audit: {CTL_FORMAL_PATH}"
)

print(
    f"Logical transport audit: {LOGICAL_TRANSPORT_PATH}"
)

print(
    f"Coherence audit: {COHERENCE_PATH}"
)

print(
    f"Output: {OUTPUT_PATH}"
)


# ============================================================
# 2. GOVERNANCE
# ============================================================

print(
    "\nGOVERNANCE"
)

print(
    "-" * 80
)

governance = {

    "CTL mathematics":
        "FROZEN",

    "triadic irreducibility definition":
        "FROZEN",

    "three sectors = proof of irreducibility":
        "FORBIDDEN",

    "nonzero three-way interaction = proof":
        "FORBIDDEN",

    "Boolean sector encoding":
        "FORBIDDEN",

    "hidden-state threshold admissibility":
        "FORBIDDEN",

    "semantic threshold admissibility":
        "FORBIDDEN",

    "undefined = false":
        "FORBIDDEN",

    "numerical transport = logical transport":
        "FORBIDDEN",

    "hidden state = K_C":
        "FORBIDDEN",

    "semantic valuation = Phi_C":
        "FORBIDDEN",

    "Phi_C fitting":
        "FORBIDDEN",

    "admissibility fabrication":
        "FORBIDDEN",

    "PCA refitting":
        "FORBIDDEN",

    "transport refitting":
        "FORBIDDEN",

    "test fitting":
        "FORBIDDEN",

    "held-out evaluation":
        "REQUIRED",

    "complexity control":
        "REQUIRED",

    "dyadic lower-order reduct":
        "REQUIRED",

    "triadic irreducibility claim":
        "NOT YET TESTED",

    "contextual geometry":
        "NOT TESTED",

}


for key, value in governance.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 3. REQUIRED ARTIFACTS
# ============================================================

required_paths = [

    STATE_BANK_PATH,
    STATE_MANIFEST_PATH,
    AUTH_PATH,
    CTL_FORMAL_PATH,
    ROW_ORDER_PATH,
    SEMANTIC_PATH,
    LOGICAL_TRANSPORT_PATH,
    COHERENCE_PATH,

]


missing = [

    str(path)

    for path in required_paths

    if not path.exists()

]


if missing:

    raise FileNotFoundError(
        "Required predecessor artifact(s) missing:\n"
        +
        "\n".join(
            missing
        )
    )


print(
    "\n[PASS] Required predecessor artifacts exist."
)


# ============================================================
# 4. LOAD PREDECESSORS
# ============================================================

with open(
    CTL_FORMAL_PATH,
    "r",
    encoding="utf-8"
) as f:

    ctl_formal = json.load(
        f
    )


with open(
    ROW_ORDER_PATH,
    "r",
    encoding="utf-8"
) as f:

    row_order = json.load(
        f
    )


with open(
    SEMANTIC_PATH,
    "r",
    encoding="utf-8"
) as f:

    semantic = json.load(
        f
    )


with open(
    LOGICAL_TRANSPORT_PATH,
    "r",
    encoding="utf-8"
) as f:

    logical_transport = json.load(
        f
    )


with open(
    COHERENCE_PATH,
    "r",
    encoding="utf-8"
) as f:

    coherence = json.load(
        f
    )


formal_classification = ctl_formal.get(
    "final_classification"
)

row_order_classification = row_order.get(
    "classification"
)

selected_row_order = row_order.get(
    "selected_row_order"
)

semantic_classification = semantic.get(
    "final_classification"
)

logical_transport_classification = (
    logical_transport.get(
        "final_classification"
    )
)

coherence_classification = (
    coherence.get(
        "final_classification"
    )
)


if formal_classification not in (
    "FORMAL_CTL_STRUCTURE_IMPLEMENTATION_PASS",
    "CTL_FORMAL_STRUCTURE_IMPLEMENTATION_PASS",
):

    raise RuntimeError(
        "Phase 1F.1 formal CTL predecessor is not PASS."
    )


if row_order_classification != (
    "FROZEN_STATE_BANK_ROW_ORDER_VERIFIED"
):

    raise RuntimeError(
        "Phase 1F.2A row-order predecessor is not PASS."
    )


if selected_row_order != (
    "interleaved_clean_corrupt"
):

    raise RuntimeError(
        "Unexpected frozen row-order convention."
    )


if semantic_classification != (
    "CTL_SEMANTIC_REALIZATION_OPERATIONAL_PASS"
):

    raise RuntimeError(
        "Phase 1F.2 semantic predecessor is not PASS."
    )


if logical_transport_classification != (
    "LOGICAL_TRANSPORT_STRUCTURE_OPERATIONAL_PASS"
):

    raise RuntimeError(
        "Phase 1F.3 logical transport predecessor is not PASS."
    )


if coherence_classification != (
    "TRIADIC_COHERENCE_INTERFACE_OPERATIONAL_PASS_NOT_EMPIRICALLY_IDENTIFIED"
):

    raise RuntimeError(
        "Phase 1F.4 coherence predecessor is not PASS."
    )


print(
    "\nPREDECESSOR CLASSIFICATIONS"
)

print(
    "-" * 80
)

print(
    f"  1F.1: {formal_classification}"
)

print(
    f"  1F.2A: {row_order_classification}"
)

print(
    f"  1F.2: {semantic_classification}"
)

print(
    f"  1F.3: {logical_transport_classification}"
)

print(
    f"  1F.4: {coherence_classification}"
)

print(
    "[PASS] All required predecessors are frozen."
)


# ============================================================
# 5. DATASET AUTHORIZATION
# ============================================================

with open(
    AUTH_PATH,
    "r",
    encoding="utf-8"
) as f:

    auth = json.load(
        f
    )


dataset_sha = (
    auth.get(
        "dataset_sha256"
    )
    or
    auth.get(
        "sha256"
    )
)


authorized = bool(
    auth.get(
        "dataset_authorized",
        auth.get(
            "authorized",
            False
        )
    )
)


if not authorized:

    raise RuntimeError(
        "Dataset authorization failed."
    )


if dataset_sha != EXPECTED_DATASET_SHA256:

    raise RuntimeError(
        "Dataset SHA mismatch."
    )


print(
    "\nDATASET AUTHORIZATION"
)

print(
    "-" * 80
)

print(
    f"  authorized: {authorized}"
)

print(
    f"  SHA: {dataset_sha}"
)

print(
    "[PASS] Dataset authorization."
)


# ============================================================
# 6. LOCATE OPERATIVE DATASET
# ============================================================

dataset_candidates = [

    Path(
        "gpt2_controlled_dataset_v1.1_recovered_r1.json"
    ),

    Path(
        "/content/gpt2_controlled_dataset_v1.1_recovered_r1.json"
    ),

]


for base in (
    ROOT,
    Path("/content"),
):

    if base.exists():

        dataset_candidates.extend(
            base.rglob(
                "gpt2_controlled_dataset_v1.1_recovered_r1.json"
            )
        )


dataset_candidates = list(
    dict.fromkeys(
        dataset_candidates
    )
)


def load_records(
    path
):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        obj = json.load(
            f
        )


    if isinstance(
        obj,
        list
    ):

        return obj


    if isinstance(
        obj,
        dict
    ):

        for key in (
            "records",
            "dataset",
            "examples",
            "data",
            "items",
        ):

            if isinstance(
                obj.get(key),
                list
            ):

                return obj[
                    key
                ]


    return None


required_fields = {

    "example_id",
    "name_pair_id",
    "name_pair_index",
    "template_id",
    "template_index",
    "split",
    "clean_prompt",
    "corrupt_prompt",
    "clean_target_name",
    "corrupt_target_name",

}


valid_dataset_candidates = []


for path in dataset_candidates:

    if not path.exists():

        continue


    try:

        candidate_records = load_records(
            path
        )

    except Exception:

        continue


    if candidate_records is None:

        continue


    if len(
        candidate_records
    ) != N_RECORDS:

        continue


    if not required_fields.issubset(
        set(
            candidate_records[0].keys()
        )
    ):

        continue


    valid_dataset_candidates.append(
        (
            path,
            candidate_records
        )
    )


if not valid_dataset_candidates:

    raise FileNotFoundError(
        "No valid operative 192-record dataset found."
    )


def identity_signature(
    record
):

    return (

        str(
            record[
                "example_id"
            ]
        ),

        str(
            record[
                "name_pair_id"
            ]
        ),

        int(
            record[
                "name_pair_index"
            ]
        ),

        str(
            record[
                "template_id"
            ]
        ),

        int(
            record[
                "template_index"
            ]
        ),

        str(
            record[
                "split"
            ]
        ),

    )


reference_identity = [

    identity_signature(
        record
    )

    for record
    in valid_dataset_candidates[0][1]

]


for path, candidate_records in valid_dataset_candidates[1:]:

    candidate_identity = [

        identity_signature(
            record
        )

        for record
        in candidate_records

    ]


    if candidate_identity != reference_identity:

        raise RuntimeError(
            "Dataset copies disagree in identity/order."
        )


dataset_path, records = (
    valid_dataset_candidates[0]
)


if [

    identity_signature(
        record
    )

    for record
    in records

] != reference_identity:

    raise RuntimeError(
        "Operative dataset identity/order changed."
    )


print(
    "\nOPERATIVE DATASET"
)

print(
    "-" * 80
)

print(
    f"  path: {dataset_path}"
)

print(
    f"  records: {len(records)}"
)

print(
    "[PASS] Operative dataset identified."
)


# ============================================================
# 7. LOAD FROZEN STATE BANK
# ============================================================

bank = np.load(
    STATE_BANK_PATH,
    allow_pickle=False
)


state_arrays = {}


for sector in SECTORS:

    X = np.asarray(
        bank[
            sector
        ]
    )


    if X.shape != (
        N_CONDITIONS,
        HIDDEN_DIM
    ):

        raise RuntimeError(
            f"{sector} shape mismatch: {X.shape}"
        )


    actual_hash = hashlib.sha256(

        np.ascontiguousarray(
            X
        ).tobytes(
            order="C"
        )

    ).hexdigest()


    if actual_hash != EXPECTED_STATE_HASHES[
        sector
    ]:

        raise RuntimeError(
            f"{sector} frozen hash mismatch."
        )


    if not np.all(
        np.isfinite(
            X
        )
    ):

        raise RuntimeError(
            f"{sector} contains non-finite values."
        )


    state_arrays[
        sector
    ] = X


print(
    "\nSTATE-BANK FIREWALL"
)

print(
    "-" * 80
)

for sector in SECTORS:

    print(
        f"  {sector}: "
        f"shape=(384,3072), "
        f"hash_match=True, "
        f"finite=True"
    )


print(
    "[PASS] Frozen state-bank integrity."
)


# ============================================================
# 8. VERIFIED PAIRED STRUCTURE
# ============================================================

clean_rows = np.asarray(
    row_order[
        "clean_rows"
    ],
    dtype=np.int64
)

corrupt_rows = np.asarray(
    row_order[
        "corrupt_rows"
    ],
    dtype=np.int64
)


if not np.array_equal(
    clean_rows,
    np.arange(
        0,
        N_CONDITIONS,
        2,
        dtype=np.int64
    )
):

    raise RuntimeError(
        "Clean row mapping inconsistent."
    )


if not np.array_equal(
    corrupt_rows,
    np.arange(
        1,
        N_CONDITIONS,
        2,
        dtype=np.int64
    )
):

    raise RuntimeError(
        "Corrupt row mapping inconsistent."
    )


print(
    "\nPAIRED EXPERIMENTAL STRUCTURE"
)

print(
    "-" * 80
)

print(
    f"  paired contexts: {N_RECORDS}"
)

print(
    "  conditions/context: 2"
)

print(
    "  sectors/context: 3"
)

print(
    "  state correspondence: VERIFIED"
)

print(
    "[PASS] Frozen paired structure."
)


# ============================================================
# 9. SPLIT STRUCTURE
# ============================================================

split_counts = {}


for record in records:

    split = str(
        record[
            "split"
        ]
    )

    split_counts[
        split
    ] = (
        split_counts.get(
            split,
            0
        )
        +
        1
    )


expected_split_counts = {

    "train": 96,

    "calibration": 48,

    "test": 48,

}


if split_counts != expected_split_counts:

    raise RuntimeError(
        "Unexpected split structure: "
        f"{split_counts}"
    )


print(
    "\nSPLIT STRUCTURE"
)

print(
    "-" * 80
)

for split, count in split_counts.items():

    print(
        f"  {split}: {count}"
    )


print(
    "[PASS] Split structure."
)


# ============================================================
# 10. CONTEXT / SECTOR / CONDITION CARDINALITY
# ============================================================
#
# This checks whether the design contains a genuine crossed
# three-sector observation for every experimental context.
#
# This is a necessary condition for any triadic analysis, but
# it is NOT sufficient for irreducibility.
# ============================================================

context_sector_condition_keys = set()


for context_index in range(
    N_RECORDS
):

    for condition in (
        "clean",
        "corrupt",
    ):

        for sector in SECTORS:

            context_sector_condition_keys.add(
                (
                    context_index,
                    condition,
                    sector,
                )
            )


expected_key_count = (
    N_RECORDS
    *
    2
    *
    3
)


if len(
    context_sector_condition_keys
) != expected_key_count:

    raise RuntimeError(
        "Context/sector/condition cardinality failed."
    )


print(
    "\nTRIADIC DESIGN CARDINALITY"
)

print(
    "-" * 80
)

print(
    f"  context × condition × sector cells: "
    f"{len(context_sector_condition_keys)}"
)

print(
    "  expected: 192 × 2 × 3 = 1152"
)

print(
    "[PASS] Three-sector crossed observation structure."
)


# ============================================================
# 11. TRIADIC OBJECT TYPE AUDIT
# ============================================================
#
# A valid triadic irreducibility test needs an object that
# actually depends on the three typed sectors jointly.
#
# We distinguish:
#
#   sector tuple:
#
#       (L_C^1, L_C^2, L_C^3)
#
# from:
#
#   arbitrary three scalar features.
#
# The current formal CTL structure contains the triadic carrier
# and triadic admissibility domain, but the empirical contents
# of Adm_C^(3) and K_C remain unresolved.
#
# Therefore:
#
#   formal triadic object = YES
#   empirical triadic logical object = NO
# ============================================================

formal_triadic_object_available = True

empirical_triadic_logical_object_available = False


triadic_object_status = {

    "formal_triadic_structure":
        "AVAILABLE",

    "empirical_triadic_logical_structure":
        "NOT_IDENTIFIED",

    "three_sector_state_tuple":
        "AVAILABLE",

    "three_sector_state_tuple_is_CTL_triatic_object":
        False,

}


print(
    "\nTRIADIC OBJECT AUDIT"
)

print(
    "-" * 80
)

print(
    "  formal triadic CTL object: AVAILABLE"
)

print(
    "  empirical triadic logical object: NOT IDENTIFIED"
)

print(
    "  three-sector numerical tuple: AVAILABLE"
)

print(
    "  numerical tuple automatically equals CTL object: NO"
)

print(
    "[PASS] Triadic object distinction preserved."
)


# ============================================================
# 12. LOWER-ORDER / DYADIC REDUCT DEFINITION
# ============================================================
#
# The formal lower-order reduct must not simply delete one
# sector and call the result "dyadic."
#
# The relevant reduct must preserve the lower-order structure
# while removing the genuinely triadic component.
#
# From the formal CTL organization already instantiated, the
# natural conceptual reduct is:
#
#   Red_{<=2}
#
# retaining:
#
#   - the three carriers;
#   - lower-order/pairwise admissibility information where
#     available;
#   - transport structure;
#
# while removing the triadic coherence object Phi_C / the
# genuinely triadic component.
#
# Since empirical admissibility is unresolved, we can define
# the TYPE of the reduct but cannot populate its empirical
# admissibility relation.
# ============================================================

@dataclass(
    frozen=True
)
class CTLDyadicReductType:

    context_index: int

    retained_carriers: tuple

    retained_transport: bool

    pairwise_admissibility_status: str

    triadic_component_removed: bool


dyadic_reduct_types = {}


for context_index in range(
    N_RECORDS
):

    dyadic_reduct_types[
        context_index
    ] = CTLDyadicReductType(

        context_index=context_index,

        retained_carriers=SECTORS,

        retained_transport=True,

        pairwise_admissibility_status=
            "NOT_IDENTIFIED",

        triadic_component_removed=True,

    )


if len(
    dyadic_reduct_types
) != N_RECORDS:

    raise RuntimeError(
        "Dyadic reduct type count mismatch."
    )


print(
    "\nLOWER-ORDER / DYADIC REDUCT"
)

print(
    "-" * 80
)

print(
    f"  reduct type objects: {len(dyadic_reduct_types)}"
)

print(
    "  retained carriers: S1, S2, S3"
)

print(
    "  transport retained: YES"
)

print(
    "  pairwise admissibility: NOT IDENTIFIED"
)

print(
    "  triadic component removed: YES"
)

print(
    "[PASS] Lower-order reduct type defined without "
    "fabricating empirical admissibility."
)


# ============================================================
# 13. IMPORTANT REDUCT FIREWALL
# ============================================================
#
# We explicitly reject several invalid reductions:
#
#   R1: remove S3 entirely;
#   R2: treat S1/S2/S3 as three Boolean predictors;
#   R3: replace CTL reduct by a generic two-feature regression;
#   R4: define dyadic structure by arbitrary thresholding;
#   R5: compare unequal-complexity models without parameter
#       control.
#
# These would not constitute the formal lower-order reduct.
# ============================================================

invalid_reducts = {

    "drop_one_sector_only":
        True,

    "boolean_sector_encoding":
        True,

    "generic_two_feature_regression_substitution":
        True,

    "threshold_defined_reduct":
        True,

    "uncontrolled_complexity_comparison":
        True,

}


print(
    "\nINVALID REDUCT FIREWALL"
)

print(
    "-" * 80
)

for key, value in invalid_reducts.items():

    print(
        f"  {key}: FORBIDDEN"
    )


print(
    "[PASS] Invalid dyadic substitutions explicitly rejected."
)


# ============================================================
# 14. HELD-OUT EVALUATION CAPACITY
# ============================================================
#
# A genuine irreducibility test requires held-out evaluation.
#
# The frozen dataset provides:
#
#   train       96 contexts
#   calibration 48 contexts
#   test        48 contexts
#
# Therefore the split architecture is sufficient for:
#
#   fit -> calibration selection -> frozen test evaluation
#
# provided that the triadic target and lower-order reduct are
# independently identified.
# ============================================================

heldout_capacity = {

    "train_contexts":
        96,

    "calibration_contexts":
        48,

    "test_contexts":
        48,

    "held_out_test_available":
        True,

    "test_reuse_for_fitting":
        False,

}


if not heldout_capacity[
    "held_out_test_available"
]:

    raise RuntimeError(
        "Held-out test capacity unexpectedly unavailable."
    )


if heldout_capacity[
    "test_reuse_for_fitting"
]:

    raise RuntimeError(
        "Test reuse is forbidden."
    )


print(
    "\nHELD-OUT CAPACITY"
)

print(
    "-" * 80
)

print(
    "  train: 96"
)

print(
    "  calibration: 48"
)

print(
    "  test: 48"
)

print(
    "  held-out evaluation possible: YES"
)

print(
    "[PASS] Held-out split capacity."
)


# ============================================================
# 15. COMPLEXITY-CONTROL REQUIREMENT
# ============================================================
#
# A triadic model must not win merely because it has more
# parameters.
#
# The audit therefore requires one of:
#
#   - equal parameter count;
#   - explicitly complexity-penalized comparison;
#   - matched effective dimension / capacity;
#   - nested model comparison with an appropriate correction.
#
# No such model pair has yet been frozen for this phase.
# ============================================================

complexity_control = {

    "complexity_control_required":
        True,

    "parameter_matched_model_pair_frozen":
        False,

    "effective_dimension_matched_pair_frozen":
        False,

    "complexity_penalty_frozen":
        False,

    "nested_model_comparison_frozen":
        False,

}


complexity_control_identified = any(

    complexity_control[key]

    for key in (
        "parameter_matched_model_pair_frozen",
        "effective_dimension_matched_pair_frozen",
        "complexity_penalty_frozen",
        "nested_model_comparison_frozen",
    )

)


print(
    "\nCOMPLEXITY CONTROL"
)

print(
    "-" * 80
)

print(
    "  complexity control required: YES"
)

print(
    "  parameter-matched pair frozen: NO"
)

print(
    "  effective-dimension-matched pair frozen: NO"
)

print(
    "  complexity penalty frozen: NO"
)

print(
    "  nested comparison frozen: NO"
)

print(
    "  complexity control currently identified: "
    f"{complexity_control_identified}"
)


# ============================================================
# 16. TARGET OBSERVABLE IDENTIFIABILITY
# ============================================================
#
# A triadic irreducibility test requires a target observable.
#
# Available frozen observables include:
#
#   - target logits;
#   - target ranks;
#   - sector hidden states.
#
# These are legitimate numerical observables.
#
# But none is automatically the formal CTL triadic coherence
# observable.
#
# Therefore:
#
#   numerical predictive target = AVAILABLE
#   CTL coherence target        = NOT IDENTIFIED
#
# This distinction determines whether a numerical triadic
# prediction experiment would test CTL irreducibility or merely
# numerical synergy.
# ============================================================

target_observables = {

    "target_logit":
        True,

    "target_rank":
        True,

    "sector_hidden_states":
        True,

    "empirical_K_C":
        False,

    "empirical_Phi_C_output":
        False,

    "formal_CTL_coherence_observable":
        False,

}


numerical_target_available = any(
    target_observables[key]
    for key in (
        "target_logit",
        "target_rank",
    )
)


ctl_target_available = (
    target_observables[
        "empirical_K_C"
    ]
    and
    target_observables[
        "empirical_Phi_C_output"
    ]
)


print(
    "\nTARGET OBSERVABLE IDENTIFIABILITY"
)

print(
    "-" * 80
)

print(
    "  numerical target observables: AVAILABLE"
)

print(
    "  target logit: AVAILABLE"
)

print(
    "  target rank: AVAILABLE"
)

print(
    "  empirical K_C: NOT IDENTIFIED"
)

print(
    "  empirical Phi_C output: NOT IDENTIFIED"
)

print(
    "  formal CTL coherence target: NOT IDENTIFIED"
)


# ============================================================
# 17. WHAT CAN AND CANNOT BE TESTED
# ============================================================
#
# We distinguish two possible experiments:
#
# A. NUMERICAL TRIADIC PREDICTIVE SYNERGY
#
#    Ask whether a model using all three numerical sector
#    observations predicts a frozen numerical target better
#    than a complexity-controlled lower-order model.
#
#    This is potentially testable.
#
#    But it is NOT automatically CTL triadic irreducibility.
#
#
# B. CTL TRIADIC IRREDUCIBILITY
#
#    Ask whether the CTL triadic structure contains an invariant
#    unavailable to its lower-order reduct.
#
#    This requires an empirical triadic logical/coherence
#    observable or another formally justified operationalization.
#
#    That is currently unavailable.
# ============================================================

numerical_synergy_testable = (

    numerical_target_available

    and

    heldout_capacity[
        "held_out_test_available"
    ]

)


ctl_irreducibility_testable = (

    formal_triadic_object_available

    and

    empirical_triadic_logical_object_available

    and

    ctl_target_available

    and

    complexity_control_identified

    and

    heldout_capacity[
        "held_out_test_available"
    ]

)


print(
    "\nTESTABILITY MATRIX"
)

print(
    "-" * 80
)

print(
    "  numerical triadic predictive comparison: "
    f"{numerical_synergy_testable}"
)

print(
    "  full CTL triadic irreducibility: "
    f"{ctl_irreducibility_testable}"
)


# ============================================================
# 18. DO NOT CONFLATE NUMERICAL SYNERGY WITH CTL IRREDUCIBILITY
# ============================================================

conflation_firewall = {

    "numerical_three_way_interaction_equals_CTL_irreducibility":
        False,

    "numerical_prediction_gain_equals_CTL_irreducibility":
        False,

    "three_sector_presence_equals_CTL_irreducibility":
        False,

    "Phi_C_absence_hidden_by_numerical_target":
        False,

    "dyadic_model_must_drop_a_sector":
        False,

    "complexity_imbalance_allowed":
        False,

}


if any(
    conflation_firewall.values()
):

    raise RuntimeError(
        "Triadic conflation firewall failed."
    )


print(
    "\nTRIADIC CONFLATION FIREWALL"
)

print(
    "-" * 80
)

print(
    "  numerical synergy = CTL irreducibility: NO"

)

print(
    "  three-way interaction = proof: NO"
)

print(
    "  three sectors = proof: NO"
)

print(
    "  lower-order reduct = drop one sector: NO"
)

print(
    "  complexity imbalance allowed: NO"
)

print(
    "[PASS] Numerical synergy and CTL irreducibility remain distinct."
)


# ============================================================
# 19. EXPERIMENTAL SUFFICIENCY MATRIX
# ============================================================

requirements = {

    "R1_formal_triadic_structure":
        True,

    "R2_three_sector_observation_per_context":
        True,

    "R3_context_alignment":
        True,

    "R4_train_calibration_test_split":
        True,

    "R5_heldout_test":
        True,

    "R6_lower_order_reduct_type":
        True,

    "R7_empirical_triatic_logical_object":
        False,

    "R8_empirical_K_C":
        False,

    "R9_empirical_Phi_C":
        False,

    "R10_complexity_control_frozen":
        complexity_control_identified,

    "R11_formally_justified_target":
        ctl_target_available,

    "R12_empirical_admissibility":
        False,

    "R13_pairwise_admissibility":
        False,

    "R14_functoriality":
        False,

}


print(
    "\nEXPERIMENTAL SUFFICIENCY MATRIX"
)

print(
    "-" * 80
)

for key, value in requirements.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 20. NECESSARY VS SUFFICIENT CONDITIONS
# ============================================================
#
# Some requirements are necessary for a CTL irreducibility
# test but are not themselves sufficient.
#
# In particular:
#
#   three-sector data
#   held-out test
#   lower-order reduct type
#
# are necessary infrastructure, not evidence of irreducibility.
# ============================================================

necessary_infrastructure = {

    "formal_triadic_structure":
        requirements[
            "R1_formal_triadic_structure"
        ],

    "three_sector_observation":
        requirements[
            "R2_three_sector_observation_per_context"
        ],

    "context_alignment":
        requirements[
            "R3_context_alignment"
        ],

    "heldout_evaluation":
        requirements[
            "R5_heldout_test"
        ],

    "lower_order_reduct":
        requirements[
            "R6_lower_order_reduct_type"
        ],

    "complexity_control":
        requirements[
            "R10_complexity_control_frozen"
        ],

}


critical_CTL_requirements = {

    "empirical_triadic_logical_object":
        requirements[
            "R7_empirical_triatic_logical_object"
        ],

    "empirical_K_C":
        requirements[
            "R8_empirical_K_C"
        ],

    "empirical_Phi_C":
        requirements[
            "R9_empirical_Phi_C"
        ],

    "formally_justified_target":
        requirements[
            "R11_formally_justified_target"
        ],

    "empirical_admissibility":
        requirements[
            "R12_empirical_admissibility"
        ],

}


infrastructure_pass = all(
    necessary_infrastructure.values()
)


critical_CTL_identifiability = all(
    critical_CTL_requirements.values()
)


print(
    "\nNECESSARY / SUFFICIENT DISTINCTION"
)

print(
    "-" * 80
)

print(
    f"  infrastructure complete: "
    f"{infrastructure_pass}"
)

print(
    f"  critical CTL empirical identifiability: "
    f"{critical_CTL_identifiability}"
)

print(
    "  interpretation: infrastructure is substantially "
    "available, but the CTL empirical target/object is not."
)


# ============================================================
# 21. SCIENTIFIC CLASSIFICATION
# ============================================================
#
# Because:
#
#   - the design supports numerical triadic comparisons;
#   - held-out evaluation exists;
#   - but the CTL triadic logical/coherence object is not
#     empirically identified;
#   - and no complexity-controlled model pair has yet been
#     frozen;
#
# the full CTL irreducibility experiment is NOT currently
# identifiable.
#
# This does NOT mean the entire Llama dataset is scientifically
# useless for triadic analysis.
#
# It means that a numerical triadic comparison must be labelled
# as a numerical synergy / predictive sufficiency experiment,
# not as proof of the CTL irreducibility theorem.
# ============================================================

if ctl_irreducibility_testable:

    final_classification = (
        "TRIADIC_IRREDUCIBILITY_EXPERIMENTALLY_IDENTIFIABLE"
    )

else:

    final_classification = (
        "TRIADIC_IRREDUCIBILITY_NOT_CURRENTLY_IDENTIFIABLE"
    )


scientific_status = {

    "formal_triadic_structure":
        "SUPPORTED",

    "three_sector_empirical_observation":
        "SUPPORTED",

    "heldout_evaluation_capacity":
        "SUPPORTED",

    "lower_order_reduct_type":
        "SUPPORTED",

    "numerical_triatic_predictive_test":
        "POTENTIALLY_TESTABLE",

    "empirical_CTL_triadic_object":
        "NOT_IDENTIFIED",

    "empirical_K_C":
        "NOT_IDENTIFIED",

    "empirical_Phi_C":
        "NOT_RECONSTRUCTED",

    "complexity_control":
        (
            "IDENTIFIED"
            if complexity_control_identified
            else
            "NOT_YET_FROZEN"
        ),

    "CTL_triatic_irreducibility":
        (
            "TESTABLE"
            if ctl_irreducibility_testable
            else
            "NOT_CURRENTLY_IDENTIFIABLE"
        ),

    "three_way_interaction_as_proof":
        "REJECTED",

    "three_sector_presence_as_proof":
        "REJECTED",

    "contextual_geometry":
        "NOT_ESTABLISHED",

}


# ============================================================
# 22. RECOMMENDED NEXT EXPERIMENTAL BRANCH
# ============================================================
#
# The audit itself does not execute the numerical synergy test.
#
# If the formal CTL empirical object remains unavailable, the
# scientifically legitimate branch is:
#
#   Phase 1F.6
#
#   "Complexity-Controlled Numerical Triadic Sufficiency Audit"
#
# with explicit wording that it tests numerical predictive
# sufficiency/synergy, NOT the CTL irreducibility theorem.
#
# A CTL irreducibility claim can only be made if an independently
# justified bridge from that numerical target to the formal CTL
# object is established.
# ============================================================

recommended_next_phase = {

    "phase":
        "1F.6",

    "title":
        "Complexity-Controlled Numerical Triadic Sufficiency Audit",

    "purpose":
        "Test whether the jointly observed three-sector "
        "numerical representation provides held-out predictive "
        "information beyond an appropriate complexity-controlled "
        "lower-order model.",

    "not_claim":
        "This does not by itself establish CTL triadic "
        "irreducibility.",

    "requirements":
        [

            "frozen train/calibration/test split",

            "frozen target observable",

            "pre-specified triadic model",

            "pre-specified lower-order/dyadic comparator",

            "matched or explicitly complexity-controlled capacity",

            "held-out test evaluation",

            "bootstrap confidence intervals",

            "no test fitting",

            "no Phi_C fitting",

            "no admissibility fabrication",

        ],

}


# ============================================================
# 23. GLOBAL FIREWALLS
# ============================================================

global_firewalls = {

    "ctl_mathematics_modified":
        False,

    "state_bank_modified":
        False,

    "dataset_modified":
        False,

    "cross_model_token_ids_compared":
        False,

    "pca_refitted":
        False,

    "transport_refitted":
        False,

    "Phi_C_fitted":
        False,

    "Phi_D_fitted":
        False,

    "admissibility_fabricated":
        False,

    "hidden_state_threshold_admissibility":
        False,

    "semantic_threshold_admissibility":
        False,

    "boolean_sector_encoding":
        False,

    "undefined_equals_false":
        False,

    "numerical_synergy_called_CTL_irreducibility":
        False,

    "three_way_interaction_called_proof":
        False,

    "three_sector_presence_called_proof":
        False,

    "complexity_control_ignored":
        False,

    "test_fitted":
        False,

}


if any(
    global_firewalls.values()
):

    raise RuntimeError(
        "Global Phase 1F.5 scientific firewall failure."
    )


print(
    "\nGLOBAL SCIENTIFIC FIREWALL"
)

print(
    "-" * 80
)

for key, value in global_firewalls.items():

    print(
        f"  {key}: {value}"
    )


print(
    "[PASS] Global triadic irreducibility governance firewalls."
)


# ============================================================
# 24. PRE-SERIALIZATION AUDIT
# ============================================================

serialization_checks = {

    "formal_classification_is_string":
        isinstance(
            formal_classification,
            str
        ),

    "row_order_classification_is_string":
        isinstance(
            row_order_classification,
            str
        ),

    "selected_row_order_is_string":
        isinstance(
            selected_row_order,
            str
        ),

    "semantic_classification_is_string":
        isinstance(
            semantic_classification,
            str
        ),

    "logical_transport_classification_is_string":
        isinstance(
            logical_transport_classification,
            str
        ),

    "coherence_classification_is_string":
        isinstance(
            coherence_classification,
            str
        ),

    "final_classification_is_string":
        isinstance(
            final_classification,
            str
        ),

    "requirements_is_dict":
        isinstance(
            requirements,
            dict
        ),

    "scientific_status_is_dict":
        isinstance(
            scientific_status,
            dict
        ),

    "recommended_next_phase_is_dict":
        isinstance(
            recommended_next_phase,
            dict
        ),

    "global_firewalls_is_dict":
        isinstance(
            global_firewalls,
            dict
        ),

}


print(
    "\nPRE-SERIALIZATION AUDIT"
)

print(
    "-" * 80
)

for key, value in serialization_checks.items():

    print(
        f"  {key}: {value}"
    )


if not all(
    serialization_checks.values()
):

    raise RuntimeError(
        "Pre-serialization audit failed."
    )


print(
    "[PASS] Pre-serialization audit."
)


# ============================================================
# 25. SERIALIZE FINAL ARTIFACT
# ============================================================

artifact = {

    "experiment_id":
        "ETTR-CTL-LLAMA-1",

    "phase":
        "1F.5",

    "title":
        "Triadic Irreducibility Identifiability / "
        "Experimental Sufficiency Audit",

    "predecessors":
        {

            "formal_ctl":
                {

                    "path":
                        str(
                            CTL_FORMAL_PATH
                        ),

                    "classification":
                        formal_classification,

                },

            "row_order":
                {

                    "path":
                        str(
                            ROW_ORDER_PATH
                        ),

                    "classification":
                        row_order_classification,

                    "selected_row_order":
                        selected_row_order,

                },

            "semantic_realization":
                {

                    "path":
                        str(
                            SEMANTIC_PATH
                        ),

                    "classification":
                        semantic_classification,

                },

            "logical_transport":
                {

                    "path":
                        str(
                            LOGICAL_TRANSPORT_PATH
                        ),

                    "classification":
                        logical_transport_classification,

                },

            "coherence":
                {

                    "path":
                        str(
                            COHERENCE_PATH
                        ),

                    "classification":
                        coherence_classification,

                },

        },

    "dataset":
        {

            "path":
                str(
                    dataset_path
                ),

            "sha256":
                dataset_sha,

            "records":
                N_RECORDS,

        },

    "state_bank_hashes":
        EXPECTED_STATE_HASHES,

    "design":
        {

            "contexts":
                N_RECORDS,

            "conditions_per_context":
                2,

            "sectors_per_context":
                3,

            "cells":
                N_RECORDS * 2 * 3,

            "split_counts":
                expected_split_counts,

        },

    "triadic_object_status":
        triadic_object_status,

    "dyadic_reduct":
        {

            "type_objects":
                len(
                    dyadic_reduct_types
                ),

            "pairwise_admissibility":
                "NOT_IDENTIFIED",

            "triadic_component_removed":
                True,

            "drop_one_sector_not_used":
                True,

        },

    "heldout_capacity":
        heldout_capacity,

    "complexity_control":
        complexity_control,

    "target_observables":
        target_observables,

    "testability":
        {

            "numerical_triatic_predictive_testable":
                numerical_synergy_testable,

            "CTL_triatic_irreducibility_testable":
                ctl_irreducibility_testable,

        },

    "conflation_firewall":
        conflation_firewall,

    "requirements":
        requirements,

    "necessary_infrastructure":
        necessary_infrastructure,

    "critical_CTL_requirements":
        critical_CTL_requirements,

    "infrastructure_pass":
        infrastructure_pass,

    "critical_CTL_identifiability":
        critical_CTL_identifiability,

    "scientific_status":
        scientific_status,

    "recommended_next_phase":
        recommended_next_phase,

    "global_firewalls":
        global_firewalls,

    "final_classification":
        final_classification,

}


with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 26. POST-SERIALIZATION VERIFICATION
# ============================================================

if not OUTPUT_PATH.exists():

    raise RuntimeError(
        "Phase 1F.5 artifact was not created."
    )


with open(
    OUTPUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    artifact_check = json.load(
        f
    )


if artifact_check.get(
    "final_classification"
) != final_classification:

    raise RuntimeError(
        "Serialized Phase 1F.5 classification mismatch."
    )


if artifact_check[
    "testability"
][
    "CTL_triatic_irreducibility_testable"
] != ctl_irreducibility_testable:

    raise RuntimeError(
        "Serialized CTL irreducibility testability mismatch."
    )


if artifact_check[
    "conflation_firewall"
][
    "numerical_three_way_interaction_equals_CTL_irreducibility"
]:

    raise RuntimeError(
        "Serialized artifact contains an invalid "
        "numerical/CTL equivalence."
    )


print(
    "\nPOST-SERIALIZATION VERIFICATION"
)

print(
    "-" * 80
)

print(
    "  artifact exists: True"
)

print(
    "  JSON readback: PASS"
)

print(
    "  classification readback: PASS"
)

print(
    "  CTL irreducibility testability readback: PASS"
)

print(
    "  numerical/CTL conflation firewall: PASS"
)


# ============================================================
# 27. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "FINAL PHASE 1F.5 STATUS"
)

print(
    "=" * 80
)

print(
    f"  {final_classification}"
)

print(
    "  formal triadic structure: SUPPORTED"
)

print(
    "  three-sector observation design: SUPPORTED"
)

print(
    "  context alignment: SUPPORTED"
)

print(
    "  held-out evaluation capacity: SUPPORTED"
)

print(
    "  lower-order reduct TYPE: SUPPORTED"
)

print(
    "  empirical triadic logical object: NOT IDENTIFIED"
)

print(
    "  empirical K_C: NOT IDENTIFIED"
)

print(
    "  empirical Phi_C: NOT RECONSTRUCTED"
)

print(
    "  empirical admissibility: NOT IDENTIFIED"
)

print(
    "  complexity-controlled comparator: "
    +
    (
        "FROZEN"
        if complexity_control_identified
        else
        "NOT YET FROZEN"
    )
)

print(
    "  numerical triadic predictive comparison: "
    +
    (
        "POTENTIALLY TESTABLE"
        if numerical_synergy_testable
        else
        "NOT TESTABLE"
    )
)

print(
    "  CTL triadic irreducibility: "
    +
    (
        "TESTABLE"
        if ctl_irreducibility_testable
        else
        "NOT CURRENTLY IDENTIFIABLE"
    )
)

print(
    "  three-way interaction as proof: REJECTED"
)

print(
    "  three-sector presence as proof: REJECTED"
)

print(
    "  contextual geometry: NOT ESTABLISHED"
)

print(
    f"  artifact: {OUTPUT_PATH}"
)

print(
    "=" * 80
)

ETTR-CTL-LLAMA-1 — PHASE 1F.5
TRIADIC IRREDUCIBILITY IDENTIFIABILITY / EXPERIMENTAL SUFFICIENCY AUDIT
ROOT: /content/ettr_ctl_llama
State bank: /content/ettr_ctl_llama/results/llama_phase1d2_full_state_bank.npz
Formal CTL audit: /content/ettr_ctl_llama/results/llama_phase1f1_formal_ctl_structure_audit.json
Logical transport audit: /content/ettr_ctl_llama/results/llama_phase1f3_logical_transport_covariance_audit.json
Coherence audit: /content/ettr_ctl_llama/results/llama_phase1f4_triadic_coherence_phiC_interface_audit.json
Output: /content/ettr_ctl_llama/results/llama_phase1f5_triadic_irreducibility_identifiability_audit.json

GOVERNANCE
--------------------------------------------------------------------------------
  CTL mathematics: FROZEN
  triadic irreducibility definition: FROZEN
  three sectors = proof of irreducibility: FORBIDDEN
  nonzero three-way interaction = proof: FORBIDDEN
  Boolean sector encoding: FORBIDDEN
  hidden-state threshold admissibility: FORBIDDEN
  semantic th

In [46]:
# ============================================================
# ETTR-CTL-LLAMA-1
# PHASE 1F.6 — COMPLEXITY-CONTROLLED NUMERICAL TRIADIC
#              SUFFICIENCY AUDIT
#
# REPLACEMENT CELL — REMOVE THE PREVIOUS CELL FIRST.
#
# PURPOSE OF THIS REPLACEMENT
# ---------------------------
# The previous implementation correctly refused to fit a new
# PCA, but it assumed that the frozen Phase 1E.1 checkpoint
# contained live sklearn PCA objects.
#
# The checkpoint actually exposes:
#
#     selected_objects
#
# at the top level, while zero PCA-like Python objects were
# discovered.
#
# Therefore this replacement performs a READ-ONLY schema audit
# of the frozen checkpoint and resolves the stored PCA basis
# from its serialized numerical representation.
#
# NO NEW PCA IS FIT.
# NO TRANSPORT MAP IS REFIT.
# NO TEST DATA IS USED FOR MODEL SELECTION.
#
# If the stored PCA basis cannot be identified unambiguously,
# the cell stops rather than reconstructing or fitting a
# replacement.
#
# ============================================================


import os
import json
import pickle
import hashlib
from pathlib import Path

import numpy as np


# ============================================================
# 0. CONFIGURATION
# ============================================================

ROOT = Path(
    "/content/ettr_ctl_llama"
)

RESULTS = ROOT / "results"

CHECKPOINTS = ROOT / "checkpoints"


STATE_BANK_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank.npz"
)

STATE_MANIFEST_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank_manifest.json"
)

AUTH_PATH = (
    RESULTS /
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

CTL_FORMAL_PATH = (
    RESULTS /
    "llama_phase1f1_formal_ctl_structure_audit.json"
)

ROW_ORDER_PATH = (
    RESULTS /
    "llama_phase1f2a_state_bank_row_order_audit.json"
)

SEMANTIC_PATH = (
    RESULTS /
    "llama_phase1f2_ctl_semantic_realization_audit.json"
)

LOGICAL_TRANSPORT_PATH = (
    RESULTS /
    "llama_phase1f3_logical_transport_covariance_audit.json"
)

COHERENCE_PATH = (
    RESULTS /
    "llama_phase1f4_triadic_coherence_phiC_interface_audit.json"
)

IRREDUCIBILITY_PREFLIGHT_PATH = (
    RESULTS /
    "llama_phase1f5_triadic_irreducibility_identifiability_audit.json"
)

TRANSPORT_SELECTION_PATH = (
    RESULTS /
    "llama_phase1e1_transport_candidate_selection.json"
)

TRANSPORT_CHECKPOINT_PATH = (
    CHECKPOINTS /
    "llama_phase1e1_selected_transport_maps.pkl"
)

OUTPUT_PATH = (
    RESULTS /
    "llama_phase1f6_complexity_controlled_numerical_triadic_sufficiency_audit.json"
)

SCHEMA_AUDIT_PATH = (
    RESULTS /
    "llama_phase1f6_frozen_transport_checkpoint_schema_audit.json"
)


EXPECTED_DATASET_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)


EXPECTED_STATE_HASHES = {

    "S1":
        "9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783",

    "S2":
        "41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb",

    "S3":
        "173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3",

}


SECTORS = (
    "S1",
    "S2",
    "S3",
)

N_RECORDS = 192

N_CONDITIONS = 384

HIDDEN_DIM = 3072


# ============================================================
# 1. HEADER
# ============================================================

print(
    "=" * 80
)

print(
    "ETTR-CTL-LLAMA-1 — PHASE 1F.6"
)

print(
    "CHECKPOINT-SCHEMA-CORRECTED "
    "COMPLEXITY-CONTROLLED NUMERICAL TRIADIC "
    "SUFFICIENCY AUDIT"
)

print(
    "=" * 80
)

print(
    f"Checkpoint: {TRANSPORT_CHECKPOINT_PATH}"
)

print(
    f"Schema audit: {SCHEMA_AUDIT_PATH}"
)

print(
    f"Final output: {OUTPUT_PATH}"
)


# ============================================================
# 2. REQUIRED ARTIFACT CHECK
# ============================================================

required_paths = [

    STATE_BANK_PATH,
    STATE_MANIFEST_PATH,
    AUTH_PATH,
    CTL_FORMAL_PATH,
    ROW_ORDER_PATH,
    SEMANTIC_PATH,
    LOGICAL_TRANSPORT_PATH,
    COHERENCE_PATH,
    IRREDUCIBILITY_PREFLIGHT_PATH,
    TRANSPORT_SELECTION_PATH,
    TRANSPORT_CHECKPOINT_PATH,

]


missing = [

    str(path)

    for path in required_paths

    if not path.exists()

]


if missing:

    raise FileNotFoundError(
        "Required predecessor artifact(s) missing:\n"
        +
        "\n".join(
            missing
        )
    )


print(
    "\n[PASS] Required predecessor artifacts exist."
)


# ============================================================
# 3. READ-ONLY CHECKPOINT LOAD
# ============================================================

with open(
    TRANSPORT_CHECKPOINT_PATH,
    "rb"
) as f:

    checkpoint = pickle.load(
        f
    )


if not isinstance(
    checkpoint,
    dict
):

    raise RuntimeError(
        "Frozen transport checkpoint is not a dictionary."
    )


print(
    "\nFROZEN CHECKPOINT"
)

print(
    "-" * 80
)

print(
    f"  top-level keys: "
    f"{list(checkpoint.keys())}"
)


# ============================================================
# 4. RECURSIVE SCHEMA DESCRIBER
# ============================================================
#
# This function NEVER modifies the checkpoint.
#
# It reports:
#
#   - dictionary keys;
#   - list/tuple lengths;
#   - numpy array shapes/dtypes;
#   - scalar types;
#   - object types;
#
# It deliberately avoids dumping large numerical contents.
# ============================================================

def describe_object(
    obj,
    path="root",
    depth=0,
    max_depth=6,
    max_items=30,
    output=None
):

    if output is None:

        output = []


    indent = "  " * depth


    if depth > max_depth:

        output.append(
            f"{indent}{path}: <max-depth>"
        )

        return output


    if isinstance(
        obj,
        dict
    ):

        output.append(
            f"{indent}{path}: dict "
            f"keys={list(obj.keys())[:max_items]}"
        )


        for key in list(
            obj.keys()
        )[:max_items]:

            describe_object(
                obj[key],
                f"{path}[{key!r}]",
                depth + 1,
                max_depth,
                max_items,
                output
            )


        return output


    if isinstance(
        obj,
        (list, tuple)
    ):

        output.append(
            f"{indent}{path}: "
            f"{type(obj).__name__} "
            f"len={len(obj)}"
        )


        for i, value in enumerate(
            obj[:max_items]
        ):

            describe_object(
                value,
                f"{path}[{i}]",
                depth + 1,
                max_depth,
                max_items,
                output
            )


        return output


    if isinstance(
        obj,
        np.ndarray
    ):

        output.append(
            f"{indent}{path}: ndarray "
            f"shape={obj.shape} "
            f"dtype={obj.dtype}"
        )

        return output


    if isinstance(
        obj,
        np.generic
    ):

        output.append(
            f"{indent}{path}: "
            f"numpy_scalar dtype={obj.dtype} "
            f"value={obj.item()!r}"
        )

        return output


    if isinstance(
        obj,
        (str, int, float, bool, type(None))
    ):

        output.append(
            f"{indent}{path}: "
            f"{type(obj).__name__} "
            f"value={obj!r}"
        )

        return output


    output.append(
        f"{indent}{path}: "
        f"object_type={type(obj).__name__}"
    )


    # Inspect ordinary object attributes without executing
    # arbitrary methods.
    try:

        attrs = vars(
            obj
        )

    except Exception:

        attrs = None


    if isinstance(
        attrs,
        dict
    ):

        output.append(
            f"{indent}  attributes="
            f"{list(attrs.keys())[:max_items]}"
        )


    return output


schema_lines = describe_object(
    checkpoint,
    max_depth=7,
    max_items=40
)


print(
    "\nCHECKPOINT SCHEMA"
)

print(
    "-" * 80
)

for line in schema_lines:

    print(
        line
    )


# ============================================================
# 5. SAVE READ-ONLY SCHEMA AUDIT
# ============================================================

schema_artifact = {

    "experiment_id":
        "ETTR-CTL-LLAMA-1",

    "phase":
        "1F.6",

    "audit_type":
        "READ_ONLY_FROZEN_CHECKPOINT_SCHEMA_AUDIT",

    "checkpoint_path":
        str(
            TRANSPORT_CHECKPOINT_PATH
        ),

    "top_level_keys":
        list(
            checkpoint.keys()
        ),

    "schema_lines":
        schema_lines,

    "PCA_refit":
        False,

    "transport_refit":
        False,

}


with open(
    SCHEMA_AUDIT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        schema_artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


print(
    "\n[PASS] Read-only checkpoint schema artifact written."
)

print(
    f"  {SCHEMA_AUDIT_PATH}"
)


# ============================================================
# 6. LOCATE SELECTED_OBJECTS
# ============================================================

if "selected_objects" not in checkpoint:

    raise RuntimeError(
        "Frozen checkpoint lacks selected_objects."
    )


selected_objects = checkpoint[
    "selected_objects"
]


print(
    "\nSELECTED OBJECTS"
)

print(
    "-" * 80
)

print(
    f"  type: {type(selected_objects).__name__}"
)


if isinstance(
    selected_objects,
    dict
):

    print(
        f"  keys: {list(selected_objects.keys())}"
    )

else:

    raise RuntimeError(
        "selected_objects is not a dictionary; "
        "schema cannot be resolved safely."
    )


# ============================================================
# 7. SEARCH FOR NUMERICAL PCA REPRESENTATIONS
# ============================================================
#
# A serialized PCA basis need not be an sklearn object.
#
# Common valid frozen representations include:
#
#   components
#   components_
#   basis
#   basis_
#   mean
#   mean_
#   pca_components
#   pca_mean
#
# We therefore search recursively for NUMERICAL arrays whose
# shapes are compatible with:
#
#   components: [k, 3072]
#   mean:       [3072]
#
# The search is descriptive only.
# ============================================================

component_keywords = {

    "components",
    "components_",
    "basis",
    "basis_",
    "pca_components",
    "pca_components_",
    "principal_components",

}


mean_keywords = {

    "mean",
    "mean_",
    "pca_mean",
    "pca_mean_",
    "center",
    "centering",

}


component_candidates = []

mean_candidates = []


def normalized_key(
    key
):

    return str(
        key
    ).strip().lower()


def recursive_collect_arrays(
    obj,
    path="root",
    depth=0,
    max_depth=10
):

    if depth > max_depth:

        return


    if isinstance(
        obj,
        dict
    ):

        for key, value in obj.items():

            key_norm = normalized_key(
                key
            )

            if isinstance(
                value,
                np.ndarray
            ):

                arr = np.asarray(
                    value
                )


                if arr.ndim == 2:

                    if (
                        key_norm in component_keywords
                        or
                        "component" in key_norm
                        or
                        "basis" in key_norm
                    ):

                        component_candidates.append(
                            (
                                f"{path}[{key!r}]",
                                arr
                            )
                        )


                if arr.ndim == 1:

                    if (
                        key_norm in mean_keywords
                        or
                        "mean" in key_norm
                        or
                        "center" in key_norm
                    ):

                        mean_candidates.append(
                            (
                                f"{path}[{key!r}]",
                                arr
                            )
                        )


            recursive_collect_arrays(
                value,
                f"{path}[{key!r}]",
                depth + 1,
                max_depth
            )


    elif isinstance(
        obj,
        (list, tuple)
    ):

        for i, value in enumerate(
            obj
        ):

            recursive_collect_arrays(
                value,
                f"{path}[{i}]",
                depth + 1,
                max_depth
            )


recursive_collect_arrays(
    selected_objects
)


print(
    "\nNUMERICAL PCA REPRESENTATION SEARCH"
)

print(
    "-" * 80
)

print(
    f"  component candidates: "
    f"{len(component_candidates)}"
)

for path, arr in component_candidates:

    print(
        f"  {path}: "
        f"shape={arr.shape}, "
        f"dtype={arr.dtype}"
    )


print(
    f"  mean candidates: "
    f"{len(mean_candidates)}"
)

for path, arr in mean_candidates:

    print(
        f"  {path}: "
        f"shape={arr.shape}, "
        f"dtype={arr.dtype}"
    )


# ============================================================
# 8. SEARCH ALL CHECKPOINT CONTENT FOR PCA-LIKE ARRAYS
# ============================================================
#
# If selected_objects contains wrappers, the PCA arrays may
# exist elsewhere in the checkpoint.
# ============================================================

all_component_candidates = []

all_mean_candidates = []


def recursive_collect_all(
    obj,
    path="root",
    depth=0,
    max_depth=12
):

    if depth > max_depth:

        return


    if isinstance(
        obj,
        dict
    ):

        for key, value in obj.items():

            key_norm = normalized_key(
                key
            )


            if isinstance(
                value,
                np.ndarray
            ):

                arr = np.asarray(
                    value
                )


                if arr.ndim == 2:

                    if (
                        "component" in key_norm
                        or
                        "basis" in key_norm
                        or
                        "pca" in key_norm
                    ):

                        all_component_candidates.append(
                            (
                                f"{path}[{key!r}]",
                                arr
                            )
                        )


                if arr.ndim == 1:

                    if (
                        "mean" in key_norm
                        or
                        "center" in key_norm
                        or
                        "pca" in key_norm
                    ):

                        all_mean_candidates.append(
                            (
                                f"{path}[{key!r}]",
                                arr
                            )
                        )


            recursive_collect_all(
                value,
                f"{path}[{key!r}]",
                depth + 1,
                max_depth
            )


    elif isinstance(
        obj,
        (list, tuple)
    ):

        for i, value in enumerate(
            obj
        ):

            recursive_collect_all(
                value,
                f"{path}[{i}]",
                depth + 1,
                max_depth
            )


recursive_collect_all(
    checkpoint
)


print(
    "\nGLOBAL PCA-LIKE ARRAY SEARCH"
)

print(
    "-" * 80
)

print(
    f"  component-like arrays: "
    f"{len(all_component_candidates)}"
)

for path, arr in all_component_candidates:

    print(
        f"  {path}: shape={arr.shape}, dtype={arr.dtype}"
    )


print(
    f"  mean-like arrays: "
    f"{len(all_mean_candidates)}"
)

for path, arr in all_mean_candidates:

    print(
        f"  {path}: shape={arr.shape}, dtype={arr.dtype}"
    )


# ============================================================
# 9. SECTOR-SPECIFIC NUMERICAL SEARCH
# ============================================================
#
# We now inspect whether the checkpoint contains explicit
# sector-named structures.
# ============================================================

sector_paths = {

    "S1": [],

    "S2": [],

    "S3": [],

}


def collect_sector_paths(
    obj,
    path="root",
    depth=0,
    max_depth=12
):

    if depth > max_depth:

        return


    if isinstance(
        obj,
        dict
    ):

        for key, value in obj.items():

            key_string = str(
                key
            ).upper()


            for sector in SECTORS:

                if sector in key_string:

                    sector_paths[
                        sector
                    ].append(
                        (
                            f"{path}[{key!r}]",
                            type(value).__name__,
                        )
                    )


            collect_sector_paths(
                value,
                f"{path}[{key!r}]",
                depth + 1,
                max_depth
            )


    elif isinstance(
        obj,
        (list, tuple)
    ):

        for i, value in enumerate(
            obj
        ):

            collect_sector_paths(
                value,
                f"{path}[{i}]",
                depth + 1,
                max_depth
            )


collect_sector_paths(
    checkpoint
)


print(
    "\nSECTOR-SPECIFIC CHECKPOINT PATHS"
)

print(
    "-" * 80
)

for sector in SECTORS:

    print(
        f"  {sector}: {len(sector_paths[sector])} paths"
    )

    for path, typ in sector_paths[
        sector
    ][:20]:

        print(
            f"    {path}: {typ}"
        )


# ============================================================
# 10. LOOK FOR EXPLICIT SELECTED OBJECT SECTOR KEYS
# ============================================================

explicit_sector_objects = {

    sector:
        selected_objects.get(
            sector
        )

    for sector in SECTORS

    if sector in selected_objects

}


print(
    "\nEXPLICIT SECTOR OBJECTS"
)

print(
    "-" * 80
)

if explicit_sector_objects:

    for sector, obj in explicit_sector_objects.items():

        print(
            f"  {sector}: {type(obj).__name__}"
        )

else:

    print(
        "  No direct S1/S2/S3 keys found."
    )


# ============================================================
# 11. SAFE PCA RESOLUTION
# ============================================================
#
# We now attempt only unambiguous resolutions.
#
# Accepted cases:
#
#   A. explicit sector object exposing components_/mean_;
#   B. explicit sector dictionary containing numerical
#      components and mean;
#   C. a unique sector-specific numerical component/mean pair.
#
# Rejected:
#
#   - one global PCA without sector identity;
#   - multiple ambiguous candidates;
#   - reconstructing PCA from state data;
#   - fitting PCA;
#   - guessing based on array order.
# ============================================================

resolved_pca = {}


def extract_pca_from_object(
    obj
):

    # --------------------------------------------------------
    # sklearn-like object
    # --------------------------------------------------------

    if (
        hasattr(
            obj,
            "components_"
        )
        and
        hasattr(
            obj,
            "mean_"
        )
    ):

        try:

            components = np.asarray(
                obj.components_
            )

            mean = np.asarray(
                obj.mean_
            )

            if (
                components.ndim == 2
                and
                mean.ndim == 1
            ):

                return (
                    components,
                    mean,
                    "object_attributes"
                )

        except Exception:

            pass


    # --------------------------------------------------------
    # dictionary-like numerical representation
    # --------------------------------------------------------

    if isinstance(
        obj,
        dict
    ):

        component_key_candidates = [

            "components",
            "components_",
            "basis",
            "basis_",
            "pca_components",
            "pca_components_",
            "principal_components",

        ]


        mean_key_candidates = [

            "mean",
            "mean_",
            "pca_mean",
            "pca_mean_",
            "center",
            "centering",

        ]


        component_value = None

        mean_value = None


        for key in component_key_candidates:

            if key in obj:

                value = obj[
                    key
                ]

                if isinstance(
                    value,
                    np.ndarray
                ):

                    if value.ndim == 2:

                        component_value = value

                        break


        for key in mean_key_candidates:

            if key in obj:

                value = obj[
                    key
                ]

                if isinstance(
                    value,
                    np.ndarray
                ):

                    if value.ndim == 1:

                        mean_value = value

                        break


        if (
            component_value is not None
            and
            mean_value is not None
        ):

            return (
                np.asarray(
                    component_value
                ),
                np.asarray(
                    mean_value
                ),
                "dictionary_numerical"
            )


    return None


for sector in SECTORS:

    candidate_results = []


    # Direct sector key.
    if sector in selected_objects:

        result = extract_pca_from_object(
            selected_objects[
                sector
            ]
        )

        if result is not None:

            candidate_results.append(
                (
                    f"selected_objects[{sector!r}]",
                    result
                )
            )


    # Search recursively through selected_objects, but only
    # retain paths that contain the explicit sector identifier.
    sector_specific_objects = []


    def collect_sector_pca_objects(
        obj,
        path="root",
        depth=0,
        max_depth=12
    ):

        if depth > max_depth:

            return


        if isinstance(
            obj,
            dict
        ):

            for key, value in obj.items():

                child_path = (
                    f"{path}[{key!r}]"
                )


                key_upper = str(
                    key
                ).upper()


                path_upper = child_path.upper()


                if (
                    sector in key_upper
                    or
                    sector in path_upper
                ):

                    result = extract_pca_from_object(
                        value
                    )

                    if result is not None:

                        sector_specific_objects.append(
                            (
                                child_path,
                                result
                            )
                        )


                collect_sector_pca_objects(
                    value,
                    child_path,
                    depth + 1,
                    max_depth
                )


        elif isinstance(
            obj,
            (list, tuple)
        ):

            for i, value in enumerate(
                obj
            ):

                collect_sector_pca_objects(
                    value,
                    f"{path}[{i}]",
                    depth + 1,
                    max_depth
                )


    collect_sector_pca_objects(
        selected_objects
    )


    candidate_results.extend(
        sector_specific_objects
    )


    # Deduplicate by path.
    unique = {}

    for path, result in candidate_results:

        unique[
            path
        ] = result


    candidate_results = list(
        unique.items()
    )


    if len(
        candidate_results
    ) == 1:

        resolved_pca[
            sector
        ] = candidate_results[0][1]

    elif len(
        candidate_results
    ) > 1:

        # Multiple sector-specific PCA candidates are
        # ambiguous unless all are numerically identical.
        first_components = np.asarray(
            candidate_results[0][1][0]
        )

        first_mean = np.asarray(
            candidate_results[0][1][1]
        )


        all_identical = True


        for _, result in candidate_results[1:]:

            if not np.array_equal(
                first_components,
                np.asarray(
                    result[0]
                )
            ):

                all_identical = False

                break


            if not np.array_equal(
                first_mean,
                np.asarray(
                    result[1]
                )
            ):

                all_identical = False

                break


        if all_identical:

            resolved_pca[
                sector
            ] = candidate_results[0][1]

        else:

            raise RuntimeError(
                f"Ambiguous frozen PCA representation for "
                f"{sector}: "
                f"{[path for path, _ in candidate_results]}"
            )


print(
    "\nPCA RESOLUTION"
)

print(
    "-" * 80
)

print(
    f"  resolved sectors: "
    f"{list(resolved_pca.keys())}"
)


# ============================================================
# 12. IF NUMERICAL PCA IS NOT EXPLICITLY STORED, STOP
# ============================================================

if len(
    resolved_pca
) != 3:

    print(
        "\n" + "=" * 80
    )

    print(
        "PHASE 1F.6 HALTED SAFELY"
    )

    print(
        "=" * 80
    )

    print(
        "The frozen checkpoint schema does not expose an "
        "unambiguous sector-specific PCA basis in the forms "
        "audited by this cell."
    )

    print(
        "NO PCA WAS FIT."
    )

    print(
        "NO TRANSPORT MAP WAS REFIT."
    )

    print(
        "NO TEST DATA WAS USED."
    )

    print(
        "The correct next action is to inspect the exact "
        "Phase 1E.1 checkpoint serialization schema before "
        "continuing."
    )

    print(
        f"Schema artifact: {SCHEMA_AUDIT_PATH}"
    )

    raise RuntimeError(
        "FROZEN_PCA_SCHEMA_UNRESOLVED — "
        "stopped rather than fitting replacement PCA."
    )


# ============================================================
# 13. PCA BASIS VALIDATION
# ============================================================

for sector in SECTORS:

    components, mean, representation = (
        resolved_pca[
            sector
        ]
    )


    components = np.asarray(
        components,
        dtype=np.float64
    )

    mean = np.asarray(
        mean,
        dtype=np.float64
    )


    if components.ndim != 2:

        raise RuntimeError(
            f"{sector}: PCA components not 2-D."
        )


    if mean.shape != (
        HIDDEN_DIM,
    ):

        raise RuntimeError(
            f"{sector}: PCA mean shape {mean.shape} "
            f"does not equal ({HIDDEN_DIM},)."
        )


    if components.shape[1] != HIDDEN_DIM:

        raise RuntimeError(
            f"{sector}: PCA input dimension "
            f"{components.shape[1]} != {HIDDEN_DIM}."
        )


    if components.shape[0] < 3:

        raise RuntimeError(
            f"{sector}: fewer than three frozen "
            "principal components available."
        )


    if not np.all(
        np.isfinite(
            components
        )
    ):

        raise RuntimeError(
            f"{sector}: non-finite PCA components."
        )


    if not np.all(
        np.isfinite(
            mean
        )
    ):

        raise RuntimeError(
            f"{sector}: non-finite PCA mean."
        )


print(
    "\n[PASS] Frozen PCA numerical representations validated."
)


# ============================================================
# 14. LOAD DATASET
# ============================================================

dataset_candidates = [

    Path(
        "gpt2_controlled_dataset_v1.1_recovered_r1.json"
    ),

    Path(
        "/content/gpt2_controlled_dataset_v1.1_recovered_r1.json"
    ),

]


for base in (
    ROOT,
    Path("/content"),
):

    if base.exists():

        dataset_candidates.extend(
            base.rglob(
                "gpt2_controlled_dataset_v1.1_recovered_r1.json"
            )
        )


dataset_candidates = list(
    dict.fromkeys(
        dataset_candidates
    )
)


def load_records(
    path
):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        obj = json.load(
            f
        )


    if isinstance(
        obj,
        list
    ):

        return obj


    if isinstance(
        obj,
        dict
    ):

        for key in (
            "records",
            "dataset",
            "examples",
            "data",
            "items",
        ):

            if isinstance(
                obj.get(key),
                list
            ):

                return obj[
                    key
                ]


    return None


valid_dataset_candidates = []


for path in dataset_candidates:

    if not path.exists():

        continue


    try:

        candidate = load_records(
            path
        )

    except Exception:

        continue


    if candidate is None:

        continue


    if len(
        candidate
    ) != N_RECORDS:

        continue


    valid_dataset_candidates.append(
        (
            path,
            candidate
        )
    )


if not valid_dataset_candidates:

    raise FileNotFoundError(
        "No valid operative 192-record dataset found."
    )


def record_signature(
    record
):

    return (

        str(
            record[
                "example_id"
            ]
        ),

        str(
            record[
                "name_pair_id"
            ]
        ),

        int(
            record[
                "name_pair_index"
            ]
        ),

        str(
            record[
                "template_id"
            ]
        ),

        int(
            record[
                "template_index"
            ]
        ),

        str(
            record[
                "split"
            ]
        ),

    )


reference_signature = [

    record_signature(
        r
    )

    for r
    in valid_dataset_candidates[0][1]

]


for path, candidate in valid_dataset_candidates[1:]:

    if [

        record_signature(
            r
        )

        for r
        in candidate

    ] != reference_signature:

        raise RuntimeError(
            "Multiple dataset copies disagree."
        )


dataset_path, records = (
    valid_dataset_candidates[0]
)


print(
    "\n[PASS] Operative dataset verified."
)

print(
    f"  path: {dataset_path}"
)

print(
    f"  records: {len(records)}"
)


# ============================================================
# 15. SPLIT INDICES
# ============================================================

split_labels = np.asarray(

    [
        str(
            r[
                "split"
            ]
        )

        for r in records

    ],

    dtype=object

)


train_idx = np.where(
    split_labels == "train"
)[0]

cal_idx = np.where(
    split_labels == "calibration"
)[0]

test_idx = np.where(
    split_labels == "test"
)[0]


if (
    len(train_idx) != 96
    or
    len(cal_idx) != 48
    or
    len(test_idx) != 48
):

    raise RuntimeError(
        "Frozen split counts changed."
    )


print(
    "\nSPLIT AUDIT"
)

print(
    "-" * 80
)

print(
    f"  train: {len(train_idx)}"
)

print(
    f"  calibration: {len(cal_idx)}"
)

print(
    f"  test: {len(test_idx)}"
)

print(
    "[PASS] Frozen split."
)


# ============================================================
# 16. LOAD FROZEN STATE BANK
# ============================================================

bank = np.load(
    STATE_BANK_PATH,
    allow_pickle=False
)


state_arrays = {}


for sector in SECTORS:

    X = np.asarray(
        bank[
            sector
        ]
    )


    if X.shape != (
        N_CONDITIONS,
        HIDDEN_DIM
    ):

        raise RuntimeError(
            f"{sector}: state-bank shape mismatch."
        )


    actual_hash = hashlib.sha256(

        np.ascontiguousarray(
            X
        ).tobytes(
            order="C"
        )

    ).hexdigest()


    if actual_hash != EXPECTED_STATE_HASHES[
        sector
    ]:

        raise RuntimeError(
            f"{sector}: frozen state hash mismatch."
        )


    state_arrays[
        sector
    ] = X.astype(
        np.float64,
        copy=False
    )


print(
    "\n[PASS] Frozen state-bank hashes verified."
)


# ============================================================
# 17. FROZEN ROW ORDER
# ============================================================

clean_rows = np.arange(
    0,
    N_CONDITIONS,
    2,
    dtype=np.int64
)

corrupt_rows = np.arange(
    1,
    N_CONDITIONS,
    2,
    dtype=np.int64
)


if not np.array_equal(
    clean_rows,
    np.asarray(
        row_order.get(
            "clean_rows"
        ),
        dtype=np.int64
    )
):

    raise RuntimeError(
        "Frozen clean row mapping mismatch."
    )


if not np.array_equal(
    corrupt_rows,
    np.asarray(
        row_order.get(
            "corrupt_rows"
        ),
        dtype=np.int64
    )
):

    raise RuntimeError(
        "Frozen corrupt row mapping mismatch."
    )


print(
    "\n[PASS] Frozen interleaved row order verified."
)


# ============================================================
# 18. PAIRED DIFFERENCES
# ============================================================

paired_states = {}


for sector in SECTORS:

    paired_states[
        sector
    ] = (

        state_arrays[
            sector
        ][
            clean_rows
        ]

        -

        state_arrays[
            sector
        ][
            corrupt_rows
        ]

    )


    if not np.all(
        np.isfinite(
            paired_states[
                sector
            ]
        )
    ):

        raise RuntimeError(
            f"{sector}: non-finite paired differences."
        )


print(
    "\n[PASS] Paired state differences reconstructed."
)


# ============================================================
# 19. VERIFY FROZEN PCA PROJECTION
# ============================================================

projected = {}


for sector in SECTORS:

    components, mean, representation = (
        resolved_pca[
            sector
        ]
    )


    components = np.asarray(
        components,
        dtype=np.float64
    )

    mean = np.asarray(
        mean,
        dtype=np.float64
    )


    clean = state_arrays[
        sector
    ][
        clean_rows
    ]

    corrupt = state_arrays[
        sector
    ][
        corrupt_rows
    ]


    clean_projected = (
        (
            clean
            -
            mean
        )
        @
        components.T
    )


    corrupt_projected = (
        (
            corrupt
            -
            mean
        )
        @
        components.T
    )


    delta_projected = (
        clean_projected
        -
        corrupt_projected
    )


    delta_direct = (
        paired_states[
            sector
        ]
        @
        components.T
    )


    if not np.allclose(
        delta_projected,
        delta_direct,
        rtol=1e-8,
        atol=1e-8
    ):

        max_error = float(
            np.max(
                np.abs(
                    delta_projected
                    -
                    delta_direct
                )
            )
        )


        raise RuntimeError(
            f"{sector}: frozen PCA difference identity failed; "
            f"max_error={max_error}"
        )


    projected[
        sector
    ] = delta_projected


print(
    "\nFROZEN PCA PROJECTION"
)

print(
    "-" * 80
)

for sector in SECTORS:

    print(
        f"  {sector}: projection identity PASS"
    )


print(
    "[PASS] Frozen PCA projection."
)


# ============================================================
# 20. LOAD FROZEN TARGET LOGITS
# ============================================================

if "target_logits" not in bank.files:

    raise RuntimeError(
        "Frozen state bank lacks target_logits."
    )


target_logits = np.asarray(
    bank[
        "target_logits"
    ],
    dtype=np.float64
)


if target_logits.ndim == 2:

    if target_logits.shape[1] != 1:

        raise RuntimeError(
            "target_logits is not a scalar-per-row array."
        )

    target_logits = (
        target_logits[:, 0]
    )


elif target_logits.ndim != 1:

    raise RuntimeError(
        f"Unexpected target_logits shape: "
        f"{target_logits.shape}"
    )


y = (
    target_logits[
        clean_rows
    ]
    -
    target_logits[
        corrupt_rows
    ]
)


if not np.all(
    np.isfinite(
        y
    )
):

    raise RuntimeError(
        "Target contrast contains non-finite values."
    )


print(
    "\nFROZEN TARGET"
)

print(
    "-" * 80
)

print(
    "  clean target-logit minus corrupt target-logit"
)

print(
    f"  N={len(y)}"
)

print(
    f"  mean={np.mean(y):.9f}"
)

print(
    f"  median={np.median(y):.9f}"
)

print(
    "[PASS] Frozen numerical target."
)


# ============================================================
# 21. FREEZE COMPLEXITY-CONTROLLED MODEL DESIGN
# ============================================================
#
# PRIMARY COMPARISON
#
# Dyadic:
#
#   S1 + S2
#   S1 + S3
#   S2 + S3
#
# each:
#
#   3 + 3 = 6 predictors
#
# Triadic:
#
#   S1 + S2 + S3
#
# each:
#
#   2 + 2 + 2 = 6 predictors
#
# The dimension budget is frozen here and was not selected
# from test performance.
# ============================================================

RIDGE_LAMBDA = 1.0

MODEL_SPEC = {

    "dyadic_S1S2": {

        "sectors": (
            "S1",
            "S2",
        ),

        "dims": {
            "S1": 3,
            "S2": 3,
        },

        "total_predictors": 6,

    },

    "dyadic_S1S3": {

        "sectors": (
            "S1",
            "S3",
        ),

        "dims": {
            "S1": 3,
            "S3": 3,
        },

        "total_predictors": 6,

    },

    "dyadic_S2S3": {

        "sectors": (
            "S2",
            "S3",
        ),

        "dims": {
            "S2": 3,
            "S3": 3,
        },

        "total_predictors": 6,

    },

    "triadic_S1S2S3": {

        "sectors": (
            "S1",
            "S2",
            "S3",
        ),

        "dims": {
            "S1": 2,
            "S2": 2,
            "S3": 2,
        },

        "total_predictors": 6,

    },

}


for model_name, spec in MODEL_SPEC.items():

    if spec[
        "total_predictors"
    ] != 6:

        raise RuntimeError(
            f"{model_name}: complexity budget mismatch."
        )


print(
    "\nMODEL COMPLEXITY"
)

print(
    "-" * 80
)

for model_name, spec in MODEL_SPEC.items():

    print(
        f"  {model_name}: "
        f"{spec['total_predictors']} predictors"
    )


print(
    "[PASS] Equal total predictor budget."
)


# ============================================================
# 22. BUILD DESIGN MATRICES
# ============================================================

def build_design(
    model_name,
    indices
):

    spec = MODEL_SPEC[
        model_name
    ]


    blocks = []


    for sector in spec[
        "sectors"
    ]:

        d = spec[
            "dims"
        ][
            sector
        ]


        blocks.append(
            projected[
                sector
            ][
                indices,
                :d
            ]
        )


    X = np.concatenate(
        blocks,
        axis=1
    )


    if X.shape[1] != 6:

        raise RuntimeError(
            f"{model_name}: design has "
            f"{X.shape[1]} predictors, expected 6."
        )


    if not np.all(
        np.isfinite(
            X
        )
    ):

        raise RuntimeError(
            f"{model_name}: non-finite design."
        )


    return X


model_names = tuple(
    MODEL_SPEC.keys()
)


designs = {

    name: {

        "train":
            build_design(
                name,
                train_idx
            ),

        "calibration":
            build_design(
                name,
                cal_idx
            ),

        "test":
            build_design(
                name,
                test_idx
            ),

    }

    for name
    in model_names

}


print(
    "\nDESIGN MATRICES"
)

print(
    "-" * 80
)

for name in model_names:

    print(
        f"  {name}: "
        f"train={designs[name]['train'].shape}, "
        f"cal={designs[name]['calibration'].shape}, "
        f"test={designs[name]['test'].shape}"
    )


print(
    "[PASS] Design matrices."
)


# ============================================================
# 23. RIDGE
# ============================================================

def fit_ridge_no_intercept(
    X,
    y,
    lam
):

    p = X.shape[1]


    A = (
        X.T
        @
        X
        +
        lam
        *
        np.eye(
            p
        )
    )


    b = (
        X.T
        @
        y
    )


    return np.linalg.solve(
        A,
        b
    )


def predict(
    X,
    beta
):

    return X @ beta


def mse(
    y_true,
    y_pred
):

    return float(
        np.mean(
            (
                y_true
                -
                y_pred
            ) ** 2
        )
    )


def rmse(
    y_true,
    y_pred
):

    return float(
        np.sqrt(
            mse(
                y_true,
                y_pred
            )
        )
    )


def r2(
    y_true,
    y_pred
):

    ss_res = np.sum(
        (
            y_true
            -
            y_pred
        ) ** 2
    )


    ss_tot = np.sum(
        (
            y_true
            -
            np.mean(
                y_true
            )
        ) ** 2
    )


    if ss_tot == 0:

        return np.nan


    return float(
        1.0
        -
        ss_res / ss_tot
    )


# ============================================================
# 24. FIT TRAIN ONLY
# ============================================================

y_train = y[
    train_idx
]

y_cal = y[
    cal_idx
]

y_test = y[
    test_idx
]


fits = {}


for model_name in model_names:

    fits[
        model_name
    ] = fit_ridge_no_intercept(

        designs[
            model_name
        ][
            "train"
        ],

        y_train,

        RIDGE_LAMBDA,

    )


print(
    "\n[PASS] Train-only ridge fitting."
)


# ============================================================
# 25. CALIBRATION EVALUATION
# ============================================================

calibration_results = {}


for model_name in model_names:

    pred = predict(

        designs[
            model_name
        ][
            "calibration"
        ],

        fits[
            model_name
        ]

    )


    calibration_results[
        model_name
    ] = {

        "mse":
            mse(
                y_cal,
                pred
            ),

        "rmse":
            rmse(
                y_cal,
                pred
            ),

        "r2":
            r2(
                y_cal,
                pred
            ),

    }


print(
    "\nCALIBRATION RESULTS"
)

print(
    "-" * 80
)

for model_name in model_names:

    result = calibration_results[
        model_name
    ]

    print(
        f"  {model_name}: "
        f"MSE={result['mse']:.9f}, "
        f"RMSE={result['rmse']:.9f}, "
        f"R2={result['r2']:.9f}"
    )


# ============================================================
# 26. CALIBRATION-ONLY DYADIC SELECTION
# ============================================================

dyadic_names = (

    "dyadic_S1S2",

    "dyadic_S1S3",

    "dyadic_S2S3",

)


best_dyadic_model = min(

    dyadic_names,

    key=lambda name:
        (
            calibration_results[
                name
            ][
                "mse"
            ],
            name,
        )

)


triadic_model = (
    "triadic_S1S2S3"
)


print(
    "\nCALIBRATION-LOCKED COMPARATOR"
)

print(
    "-" * 80
)

print(
    f"  selected dyadic model: "
    f"{best_dyadic_model}"
)

print(
    f"  triadic model: "
    f"{triadic_model}"
)

print(
    "  selection split: calibration only"
)

print(
    "  test used for selection: NO"
)


# ============================================================
# 27. FINAL TEST FIREWALL
# ============================================================

test_firewall = {

    "PCA_refit":
        False,

    "transport_refit":
        False,

    "Phi_C_fit":
        False,

    "admissibility_fit":
        False,

    "test_model_selection":
        False,

    "test_hyperparameter_selection":
        False,

}


if any(
    test_firewall.values()
):

    raise RuntimeError(
        "Test firewall failed."
    )


print(
    "\nTEST FIREWALL"
)

print(
    "-" * 80
)

for key, value in test_firewall.items():

    print(
        f"  {key}: {value}"
    )


print(
    "[PASS] Test evaluation authorized."
)


# ============================================================
# 28. TEST EVALUATION
# ============================================================

selected_models = (

    best_dyadic_model,

    triadic_model,

)


test_results = {}

test_predictions = {}


for model_name in selected_models:

    pred = predict(

        designs[
            model_name
        ][
            "test"
        ],

        fits[
            model_name
        ]

    )


    test_predictions[
        model_name
    ] = pred


    test_results[
        model_name
    ] = {

        "mse":
            mse(
                y_test,
                pred
            ),

        "rmse":
            rmse(
                y_test,
                pred
            ),

        "r2":
            r2(
                y_test,
                pred
            ),

    }


print(
    "\nFROZEN TEST RESULTS"
)

print(
    "-" * 80
)

for model_name in selected_models:

    result = test_results[
        model_name
    ]

    print(
        f"  {model_name}: "
        f"MSE={result['mse']:.9f}, "
        f"RMSE={result['rmse']:.9f}, "
        f"R2={result['r2']:.9f}"
    )


# ============================================================
# 29. PAIRED HELD-OUT COMPARISON
# ============================================================

dyadic_pred = test_predictions[
    best_dyadic_model
]

triadic_pred = test_predictions[
    triadic_model
]


dyadic_se = (
    y_test
    -
    dyadic_pred
) ** 2


triadic_se = (
    y_test
    -
    triadic_pred
) ** 2


paired_delta = (
    dyadic_se
    -
    triadic_se
)


observed_delta = float(
    np.mean(
        paired_delta
    )
)


relative_improvement = float(

    (
        test_results[
            best_dyadic_model
        ][
            "mse"
        ]
        -
        test_results[
            triadic_model
        ][
            "mse"
        ]
    )
    /
    test_results[
        best_dyadic_model
    ][
        "mse"
    ]
    *
    100.0

)


print(
    "\nHELD-OUT TRIADIC COMPARISON"
)

print(
    "-" * 80
)

print(
    f"  comparator: {best_dyadic_model}"
)

print(
    f"  dyadic MSE: "
    f"{test_results[best_dyadic_model]['mse']:.9f}"
)

print(
    f"  triadic MSE: "
    f"{test_results[triadic_model]['mse']:.9f}"
)

print(
    f"  MSE delta (dyadic - triadic): "
    f"{observed_delta:.9f}"
)

print(
    f"  relative improvement: "
    f"{relative_improvement:.6f}%"
)

print(
    "  positive delta favors triadic."
)


# ============================================================
# 30. PAIRED BOOTSTRAP
# ============================================================

BOOTSTRAP_SEED = 42016

N_BOOTSTRAPS = 10000

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)


n_test = len(
    paired_delta
)


bootstrap_means = np.empty(
    N_BOOTSTRAPS,
    dtype=np.float64
)


for b in range(
    N_BOOTSTRAPS
):

    sample = rng.integers(
        0,
        n_test,
        size=n_test
    )


    bootstrap_means[
        b
    ] = np.mean(
        paired_delta[
            sample
        ]
    )


ci_low = float(
    np.quantile(
        bootstrap_means,
        0.025
    )
)

ci_high = float(
    np.quantile(
        bootstrap_means,
        0.975
    )
)

prob_positive = float(
    np.mean(
        bootstrap_means > 0
    )
)


print(
    "\nPAIRED BOOTSTRAP"
)

print(
    "-" * 80
)

print(
    f"  seed: {BOOTSTRAP_SEED}"
)

print(
    f"  replicates: {N_BOOTSTRAPS}"
)

print(
    f"  mean: {observed_delta:.9f}"
)

print(
    f"  95% CI: "
    f"[{ci_low:.9f}, {ci_high:.9f}]"
)

print(
    f"  P(delta > 0): "
    f"{prob_positive:.6f}"
)


# ============================================================
# 31. NUMERICAL CLASSIFICATION
# ============================================================

if (
    observed_delta > 0
    and
    ci_low > 0
):

    numerical_status = (
        "HELD_OUT_TRIADIC_PREDICTIVE_ADVANTAGE_SUPPORTED"
    )

elif (
    observed_delta < 0
    and
    ci_high < 0
):

    numerical_status = (
        "HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED"
    )

else:

    numerical_status = (
        "HELD_OUT_TRIADIC_PREDICTIVE_ADVANTAGE_UNRESOLVED"
    )


print(
    "\nNUMERICAL CLASSIFICATION"
)

print(
    "-" * 80
)

print(
    f"  {numerical_status}"
)


# ============================================================
# 32. ADDITIONAL DESCRIPTIVES
# ============================================================

delta_descriptives = {

    "mean":
        float(
            np.mean(
                paired_delta
            )
        ),

    "median":
        float(
            np.median(
                paired_delta
            )
        ),

    "q05":
        float(
            np.quantile(
                paired_delta,
                0.05
            )
        ),

    "q25":
        float(
            np.quantile(
                paired_delta,
                0.25
            )
        ),

    "q75":
        float(
            np.quantile(
                paired_delta,
                0.75
            )
        ),

    "q95":
        float(
            np.quantile(
                paired_delta,
                0.95
            )
        ),

    "fraction_positive":
        float(
            np.mean(
                paired_delta > 0
            )
        ),

}


print(
    "\nPAIRED DELTA DESCRIPTIVES"
)

print(
    "-" * 80
)

for key, value in delta_descriptives.items():

    print(
        f"  {key}: {value:.9f}"
    )


# ============================================================
# 33. SCIENTIFIC INTERPRETATION FIREWALL
# ============================================================

interpretation_firewall = {

    "numerical_result_equals_CTL_irreducibility":
        False,

    "numerical_result_equals_Phi_C":
        False,

    "numerical_result_equals_K_C":
        False,

    "numerical_result_equals_triatic_admissibility":
        False,

    "three_sector_presence_is_proof":
        False,

    "three_way_interaction_is_proof":
        False,

    "test_used_for_selection":
        False,

    "PCA_refitted":
        False,

    "transport_refitted":
        False,

    "Phi_C_fitted":
        False,

    "admissibility_fitted":
        False,

}


if any(
    interpretation_firewall.values()
):

    raise RuntimeError(
        "Scientific interpretation firewall failed."
    )


print(
    "\nINTERPRETATION FIREWALL"
)

print(
    "-" * 80
)

for key, value in interpretation_firewall.items():

    print(
        f"  {key}: {value}"
    )


print(
    "[PASS] Numerical result remains distinct from CTL."
)


# ============================================================
# 34. INTERPRETATION
# ============================================================

if numerical_status == (
    "HELD_OUT_TRIADIC_PREDICTIVE_ADVANTAGE_SUPPORTED"
):

    interpretation = (
        "Under the frozen paired-state representation, frozen "
        "PCA basis, fixed ridge regularization, equal total "
        "predictor budget, calibration-selected dyadic "
        "comparator, and held-out test evaluation, the tri-sector "
        "numerical model provides a statistically supported "
        "predictive advantage for the frozen clean-minus-corrupt "
        "target-logit contrast. This is numerical triadic "
        "predictive sufficiency under the specified model class. "
        "It does not establish CTL triadic irreducibility, "
        "empirical triadic admissibility, K_C, or Phi_C."
    )

elif numerical_status == (
    "HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED"
):

    interpretation = (
        "Under the frozen paired-state representation, frozen "
        "PCA basis, fixed ridge regularization, equal total "
        "predictor budget, calibration-selected dyadic "
        "comparator, and held-out test evaluation, the tri-sector "
        "numerical model performs worse than the selected dyadic "
        "comparator for the frozen target-logit contrast. This "
        "does not establish absence of formal CTL triadic "
        "structure."
    )

else:

    interpretation = (
        "Under the frozen paired-state representation, frozen "
        "PCA basis, fixed ridge regularization, equal total "
        "predictor budget, calibration-selected dyadic "
        "comparator, and held-out test evaluation, the numerical "
        "comparison does not resolve tri-sector predictive "
        "sufficiency. This neither establishes nor refutes "
        "formal CTL triadic irreducibility."
    )


# ============================================================
# 35. SCIENTIFIC STATUS
# ============================================================

scientific_status = {

    "state_bank":
        "VERIFIED",

    "dataset":
        "VERIFIED",

    "row_order":
        "VERIFIED",

    "target":
        "VERIFIED",

    "PCA":
        "FROZEN_AND_VERIFIED",

    "PCA_refit":
        False,

    "transport_refit":
        False,

    "model_complexity":
        "EQUAL_TOTAL_PREDICTOR_BUDGET",

    "dyadic_selection":
        "CALIBRATION_ONLY",

    "test_evaluation":
        "HELD_OUT",

    "numerical_triadic_status":
        numerical_status,

    "CTL_triatic_irreducibility":
        "NOT_ESTABLISHED",

    "empirical_Adm_C_3":
        "NOT_IDENTIFIED",

    "empirical_K_C":
        "NOT_IDENTIFIED",

    "empirical_Phi_C":
        "NOT_RECONSTRUCTED",

    "contextual_geometry":
        "NOT_ESTABLISHED",

}


# ============================================================
# 36. FINAL CLASSIFICATION
# ============================================================

if numerical_status == (
    "HELD_OUT_TRIADIC_PREDICTIVE_ADVANTAGE_SUPPORTED"
):

    final_classification = (
        "NUMERICAL_TRIADIC_PREDICTIVE_SUFFICIENCY_SUPPORTED_CTL_IRREDUCIBILITY_NOT_ESTABLISHED"
    )

elif numerical_status == (
    "HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED"
):

    final_classification = (
        "NUMERICAL_TRIADIC_PREDICTIVE_SUFFICIENCY_NOT_SUPPORTED_CTL_IRREDUCIBILITY_NOT_ESTABLISHED"
    )

else:

    final_classification = (
        "NUMERICAL_TRIADIC_PREDICTIVE_SUFFICIENCY_UNRESOLVED_CTL_IRREDUCIBILITY_NOT_ESTABLISHED"
    )


# ============================================================
# 37. FINAL ARTIFACT
# ============================================================

artifact = {

    "experiment_id":
        "ETTR-CTL-LLAMA-1",

    "phase":
        "1F.6",

    "title":
        "Complexity-Controlled Numerical Triadic "
        "Sufficiency Audit",

    "checkpoint_schema_audit":
        str(
            SCHEMA_AUDIT_PATH
        ),

    "dataset":
        {

            "path":
                str(
                    dataset_path
                ),

            "sha256":
                dataset_sha,

            "records":
                N_RECORDS,

        },

    "state_bank":
        {

            "path":
                str(
                    STATE_BANK_PATH
                ),

            "hashes":
                EXPECTED_STATE_HASHES,

        },

    "row_order":
        {

            "classification":
                row_order_classification,

            "selected":
                selected_row_order,

        },

    "target":
        {

            "definition":
                "clean target-logit minus corrupt target-logit",

            "CTL_K_C":
                False,

            "CTL_Phi_C":
                False,

        },

    "frozen_PCA":
        {

            "source":
                str(
                    TRANSPORT_CHECKPOINT_PATH
                ),

            "refit":
                False,

            "sector_representations":
                {

                    sector:
                        {

                            "components_shape":
                                list(
                                    np.asarray(
                                        resolved_pca[
                                            sector
                                        ][0]
                                    ).shape
                                ),

                            "mean_shape":
                                list(
                                    np.asarray(
                                        resolved_pca[
                                            sector
                                        ][1]
                                    ).shape
                                ),

                            "representation":
                                resolved_pca[
                                    sector
                                ][2],

                        }

                    for sector
                    in SECTORS

                },

            "intrinsic_dimensionality_claim":
                False,

        },

    "model_specification":
        MODEL_SPEC,

    "ridge":
        {

            "lambda":
                RIDGE_LAMBDA,

            "intercept":
                False,

        },

    "complexity_control":
        {

            "dyadic_predictors":
                6,

            "triadic_predictors":
                6,

            "matched":
                True,

        },

    "selection":
        {

            "best_dyadic_model":
                best_dyadic_model,

            "selection_split":
                "calibration",

            "test_used":
                False,

        },

    "calibration_results":
        calibration_results,

    "test_results":
        test_results,

    "heldout_comparison":
        {

            "comparator":
                best_dyadic_model,

            "triadic":
                triadic_model,

            "mean_MSE_delta_dyadic_minus_triadic":
                observed_delta,

            "relative_MSE_improvement_percent":
                relative_improvement,

            "bootstrap":
                {

                    "seed":
                        BOOTSTRAP_SEED,

                    "replicates":
                        N_BOOTSTRAPS,

                    "ci95":
                        [
                            ci_low,
                            ci_high,
                        ],

                    "probability_delta_positive":
                        prob_positive,

                },

            "descriptives":
                delta_descriptives,

        },

    "numerical_status":
        numerical_status,

    "scientific_status":
        scientific_status,

    "interpretation":
        interpretation,

    "interpretation_firewall":
        interpretation_firewall,

    "final_classification":
        final_classification,

}


# ============================================================
# 38. PRE-SERIALIZATION AUDIT
# ============================================================

serialization_checks = {

    "artifact_dict":
        isinstance(
            artifact,
            dict
        ),

    "final_classification_string":
        isinstance(
            final_classification,
            str
        ),

    "numerical_status_string":
        isinstance(
            numerical_status,
            str
        ),

    "best_dyadic_model_string":
        isinstance(
            best_dyadic_model,
            str
        ),

    "model_specification_dict":
        isinstance(
            MODEL_SPEC,
            dict
        ),

    "calibration_results_dict":
        isinstance(
            calibration_results,
            dict
        ),

    "test_results_dict":
        isinstance(
            test_results,
            dict
        ),

    "interpretation_firewall_dict":
        isinstance(
            interpretation_firewall,
            dict
        ),

}


print(
    "\nPRE-SERIALIZATION AUDIT"
)

print(
    "-" * 80
)

for key, value in serialization_checks.items():

    print(
        f"  {key}: {value}"
    )


if not all(
    serialization_checks.values()
):

    raise RuntimeError(
        "Pre-serialization audit failed."
    )


print(
    "[PASS] Pre-serialization audit."
)


# ============================================================
# 39. WRITE FINAL ARTIFACT
# ============================================================

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 40. POST-SERIALIZATION VERIFICATION
# ============================================================

with open(
    OUTPUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    artifact_check = json.load(
        f
    )


if artifact_check.get(
    "final_classification"
) != final_classification:

    raise RuntimeError(
        "Serialized classification mismatch."
    )


if artifact_check.get(
    "frozen_PCA",
    {}
).get(
    "refit"
) is not False:

    raise RuntimeError(
        "Serialized PCA refit firewall failed."
    )


if artifact_check.get(
    "complexity_control",
    {}
).get(
    "matched"
) is not True:

    raise RuntimeError(
        "Serialized complexity-control firewall failed."
    )


if artifact_check.get(
    "target",
    {}
).get(
    "CTL_K_C"
) is not False:

    raise RuntimeError(
        "Serialized K_C firewall failed."
    )


if artifact_check.get(
    "target",
    {}
).get(
    "CTL_Phi_C"
) is not False:

    raise RuntimeError(
        "Serialized Phi_C firewall failed."
    )


print(
    "\nPOST-SERIALIZATION VERIFICATION"
)

print(
    "-" * 80
)

print(
    "  JSON readback: PASS"
)

print(
    "  classification: PASS"
)

print(
    "  PCA refit firewall: PASS"
)

print(
    "  equal-budget firewall: PASS"
)

print(
    "  K_C firewall: PASS"
)

print(
    "  Phi_C firewall: PASS"
)


# ============================================================
# 41. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "FINAL PHASE 1F.6 STATUS"
)

print(
    "=" * 80
)

print(
    f"  {final_classification}"
)

print(
    f"  numerical status: {numerical_status}"
)

print(
    f"  selected dyadic comparator: "
    f"{best_dyadic_model}"
)

print(
    f"  triadic model: "
    f"{triadic_model}"
)

print(
    f"  equal total predictor budget: "
    f"6 = 6"
)

print(
    f"  test MSE dyadic: "
    f"{test_results[best_dyadic_model]['mse']:.9f}"
)

print(
    f"  test MSE triadic: "
    f"{test_results[triadic_model]['mse']:.9f}"
)

print(
    f"  mean MSE delta "
    f"(dyadic - triadic): "
    f"{observed_delta:.9f}"
)

print(
    f"  95% bootstrap CI: "
    f"[{ci_low:.9f}, {ci_high:.9f}]"
)

print(
    "  CTL triadic irreducibility: NOT ESTABLISHED"
)

print(
    "  empirical Adm_C^(3): NOT IDENTIFIED"
)

print(
    "  empirical K_C: NOT IDENTIFIED"
)

print(
    "  empirical Phi_C: NOT RECONSTRUCTED"
)

print(
    "  contextual geometry: NOT ESTABLISHED"
)

print(
    f"  artifact: {OUTPUT_PATH}"
)

print(
    "=" * 80
)

ETTR-CTL-LLAMA-1 — PHASE 1F.6
CHECKPOINT-SCHEMA-CORRECTED COMPLEXITY-CONTROLLED NUMERICAL TRIADIC SUFFICIENCY AUDIT
Checkpoint: /content/ettr_ctl_llama/checkpoints/llama_phase1e1_selected_transport_maps.pkl
Schema audit: /content/ettr_ctl_llama/results/llama_phase1f6_frozen_transport_checkpoint_schema_audit.json
Final output: /content/ettr_ctl_llama/results/llama_phase1f6_complexity_controlled_numerical_triadic_sufficiency_audit.json

[PASS] Required predecessor artifacts exist.

FROZEN CHECKPOINT
--------------------------------------------------------------------------------
  top-level keys: ['experiment', 'phase', 'model', 'selected_layer', 'hidden_size', 'dataset_sha256', 'state_bank_hashes', 'fit_scope', 'pca_scope', 'transport_scope', 'transport_orientation', 'intercept', 'selected_objects']

CHECKPOINT SCHEMA
--------------------------------------------------------------------------------
root: dict keys=['experiment', 'phase', 'model', 'selected_layer', 'hidden_size', 'dataset

In [48]:
# ============================================================
# ETTR-CTL-LLAMA-1
# PHASE 1F.7
# NUMERICAL TRIADIC RESULT SYNTHESIS /
# SECONDARY-TARGET SUFFICIENCY AUDIT
#
# REPLACEMENT VERSION
#
# IMPORTANT:
#   Remove the previous Phase 1F.7 cell before inserting this
#   complete replacement.
#
# PURPOSE OF THIS CORRECTION
# --------------------------
# The previous Phase 1F.7 implementation assumed that every
# predecessor artifact exposed a top-level `final_classification`
# field.
#
# Phase 1F.2A does not expose its classification under that
# exact key. Its scientific result is instead represented by
# the verified row-order evidence.
#
# This replacement therefore:
#
#   1. reads the frozen 1F.2A artifact;
#   2. resolves its classification from the actual schema;
#   3. independently requires:
#          interleaved_clean_corrupt
#          pT equality
#          iT equality
#          exact candidate count = 1
#   4. refuses to infer a classification merely from a missing
#      JSON field;
#   5. does not modify or rerun Phase 1F.2A;
#   6. performs NO new numerical experiment.
#
# The scientific question remains:
#
#   Is a SECONDARY frozen numerical target already available
#   and therefore legitimately usable for a future robustness
#   test?
#
# ============================================================


import os
import json
import hashlib
from pathlib import Path

import numpy as np


# ============================================================
# 0. CONFIGURATION
# ============================================================

ROOT = Path(
    "/content/ettr_ctl_llama"
)

RESULTS = ROOT / "results"


STATE_BANK_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank.npz"
)

STATE_MANIFEST_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank_manifest.json"
)

AUTH_PATH = (
    RESULTS /
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

CTL_FORMAL_PATH = (
    RESULTS /
    "llama_phase1f1_formal_ctl_structure_audit.json"
)

ROW_ORDER_PATH = (
    RESULTS /
    "llama_phase1f2a_state_bank_row_order_audit.json"
)

SEMANTIC_PATH = (
    RESULTS /
    "llama_phase1f2_ctl_semantic_realization_audit.json"
)

LOGICAL_TRANSPORT_PATH = (
    RESULTS /
    "llama_phase1f3_logical_transport_covariance_audit.json"
)

COHERENCE_PATH = (
    RESULTS /
    "llama_phase1f4_triadic_coherence_phiC_interface_audit.json"
)

IRREDUCIBILITY_PATH = (
    RESULTS /
    "llama_phase1f5_triadic_irreducibility_identifiability_audit.json"
)

F6_PATH = (
    RESULTS /
    "llama_phase1f6_complexity_controlled_numerical_triadic_sufficiency_audit.json"
)

TRANSPORT_SELECTION_PATH = (
    RESULTS /
    "llama_phase1e1_transport_candidate_selection.json"
)

TRANSPORT_TEST_PATH = (
    RESULTS /
    "llama_phase1e2_frozen_transport_test_evaluation.json"
)

OUTPUT_PATH = (
    RESULTS /
    "llama_phase1f7_numerical_triadic_result_synthesis_audit.json"
)


EXPECTED_DATASET_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)


EXPECTED_STATE_HASHES = {

    "S1":
        "9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783",

    "S2":
        "41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb",

    "S3":
        "173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3",

    "target_logits":
        "537fa43c8e278b8678de978c1dc2ec4ee5a1c77ca0a917dbf6a5823cf77ea58f",

}


N_RECORDS = 192
N_CONDITIONS = 384
N_TEST = 48


# ============================================================
# 1. HELPERS
# ============================================================

def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


def recursive_find_values(
    obj,
    target_keys,
    prefix="root",
    depth=0,
    max_depth=8
):

    """
    Read-only recursive search.

    Returns:
        [(path, value), ...]
    """

    found = []

    if depth > max_depth:
        return found

    if isinstance(
        obj,
        dict
    ):

        for key, value in obj.items():

            path = (
                f"{prefix}.{key}"
            )

            if key in target_keys:

                found.append(
                    (
                        path,
                        value
                    )
                )

            found.extend(
                recursive_find_values(
                    value,
                    target_keys,
                    path,
                    depth + 1,
                    max_depth
                )
            )

    elif isinstance(
        obj,
        list
    ):

        for i, value in enumerate(
            obj[:20]
        ):

            found.extend(
                recursive_find_values(
                    value,
                    target_keys,
                    f"{prefix}[{i}]",
                    depth + 1,
                    max_depth
                )
            )

    return found


def load_records(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        obj = json.load(
            f
        )

    if isinstance(
        obj,
        list
    ):

        return obj

    if isinstance(
        obj,
        dict
    ):

        for key in (
            "records",
            "dataset",
            "examples",
            "data",
            "items"
        ):

            value = obj.get(
                key
            )

            if isinstance(
                value,
                list
            ):

                return value

    return None


# ============================================================
# 2. HEADER
# ============================================================

print(
    "=" * 80
)

print(
    "ETTR-CTL-LLAMA-1 — PHASE 1F.7"
)

print(
    "NUMERICAL TRIADIC RESULT SYNTHESIS / "
    "SECONDARY-TARGET SUFFICIENCY AUDIT"
)

print(
    "=" * 80
)

print(
    f"Primary numerical result: {F6_PATH}"
)

print(
    f"Output: {OUTPUT_PATH}"
)


# ============================================================
# 3. REQUIRED PREDECESSOR ARTIFACTS
# ============================================================

required_paths = [

    STATE_BANK_PATH,
    STATE_MANIFEST_PATH,
    AUTH_PATH,
    CTL_FORMAL_PATH,
    ROW_ORDER_PATH,
    SEMANTIC_PATH,
    LOGICAL_TRANSPORT_PATH,
    COHERENCE_PATH,
    IRREDUCIBILITY_PATH,
    F6_PATH,
    TRANSPORT_SELECTION_PATH,
    TRANSPORT_TEST_PATH,

]


missing = [

    str(path)

    for path in required_paths

    if not path.exists()

]


if missing:

    raise FileNotFoundError(
        "Required predecessor artifact(s) missing:\n"
        +
        "\n".join(
            missing
        )
    )


print(
    "\n[PASS] Required predecessor artifacts exist."
)


# ============================================================
# 4. LOAD PREDECESSORS
# ============================================================

auth = load_json(
    AUTH_PATH
)

ctl_formal = load_json(
    CTL_FORMAL_PATH
)

row_order = load_json(
    ROW_ORDER_PATH
)

semantic = load_json(
    SEMANTIC_PATH
)

logical_transport = load_json(
    LOGICAL_TRANSPORT_PATH
)

coherence = load_json(
    COHERENCE_PATH
)

irreducibility = load_json(
    IRREDUCIBILITY_PATH
)

f6 = load_json(
    F6_PATH
)

transport_selection = load_json(
    TRANSPORT_SELECTION_PATH
)

transport_test = load_json(
    TRANSPORT_TEST_PATH
)


# ============================================================
# 5. PREDECESSOR SCHEMA AUDIT
# ============================================================
#
# We deliberately inspect the actual 1F.2A schema rather than
# assuming a universal field name.
# ============================================================

print(
    "\n1F.2A SCHEMA INSPECTION"
)

print(
    "-" * 80
)

print(
    "  top-level keys:"
)

for key in row_order.keys():

    print(
        f"    {key}: "
        f"{type(row_order[key]).__name__}"
    )


# ============================================================
# 6. RESOLVE 1F.2A CLASSIFICATION SAFELY
# ============================================================
#
# The expected scientific classification is:
#
#   FROZEN_STATE_BANK_ROW_ORDER_VERIFIED
#
# But we will not manufacture that value merely because it is
# expected.
#
# The classification is accepted only if the artifact contains
# the actual row-order evidence establishing:
#
#   selected row order = interleaved_clean_corrupt
#   exactly one candidate selected
#   pT equality
#   iT equality
#
# ============================================================

row_order_classification_candidates = []


for key in (
    "final_classification",
    "classification",
    "status",
    "result",
):

    if key in row_order:

        row_order_classification_candidates.append(
            (
                key,
                row_order[key]
            )
        )


selected_row_order_candidates = []


for key in (
    "selected_row_order",
    "row_order",
    "selected",
    "frozen_row_order",
):

    if key in row_order:

        selected_row_order_candidates.append(
            (
                key,
                row_order[key]
            )
        )


print(
    "\n1F.2A CLASSIFICATION FIELDS FOUND"
)

print(
    "-" * 80
)

if row_order_classification_candidates:

    for key, value in row_order_classification_candidates:

        print(
            f"  {key}: {value}"
        )

else:

    print(
        "  no standard classification field found"
    )


print(
    "\n1F.2A ROW-ORDER FIELDS FOUND"
)

print(
    "-" * 80
)

for key, value in selected_row_order_candidates:

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 7. SEARCH FOR FROZEN EVIDENCE
# ============================================================

row_order_paths = recursive_find_values(
    row_order,
    {
        "selected_row_order",
        "row_order",
        "classification",
        "final_classification",
        "exact_candidates",
        "pT_equal",
        "iT_equal",
        "pT_equality",
        "iT_equality",
    }
)


print(
    "\n1F.2A FROZEN EVIDENCE SEARCH"
)

print(
    "-" * 80
)

for path, value in row_order_paths:

    if isinstance(
        value,
        (dict, list)
    ):

        print(
            f"  {path}: "
            f"{type(value).__name__}"
        )

    else:

        print(
            f"  {path}: {value}"
        )


# ============================================================
# 8. ROBUST ROW-ORDER RESOLUTION
# ============================================================

selected_row_order = None


# Direct standard fields first.

for key in (
    "selected_row_order",
    "row_order",
    "selected",
    "frozen_row_order"
):

    value = row_order.get(
        key
    )

    if isinstance(
        value,
        str
    ):

        if value == (
            "interleaved_clean_corrupt"
        ):

            selected_row_order = value

            break


# Recursive fallback.

if selected_row_order is None:

    for path, value in row_order_paths:

        if (
            isinstance(
                value,
                str
            )
            and
            value == (
                "interleaved_clean_corrupt"
            )
        ):

            selected_row_order = value

            break


if selected_row_order is None:

    raise RuntimeError(
        "Could not resolve the frozen 1F.2A row-order "
        "selection from the artifact. Refusing to infer it."
    )


print(
    "\nRESOLVED FROZEN ROW ORDER"
)

print(
    "-" * 80
)

print(
    f"  selected: {selected_row_order}"
)


# ============================================================
# 9. RESOLVE ROW-ORDER VERIFICATION EVIDENCE
# ============================================================

def find_bool_evidence(
    obj,
    target_names,
    default=None
):

    matches = recursive_find_values(
        obj,
        target_names
    )

    bool_values = [

        bool(value)

        for path, value in matches

        if isinstance(
            value,
            bool
        )

    ]

    if not bool_values:

        return default

    # We require all discovered explicit boolean evidence to
    # agree rather than selecting a convenient occurrence.

    if not all(
        value == bool_values[0]
        for value in bool_values
    ):

        raise RuntimeError(
            f"Conflicting boolean evidence found for "
            f"{target_names}: {bool_values}"
        )

    return bool_values[0]


pT_equal = find_bool_evidence(
    row_order,
    {
        "pT_equal",
        "pT_equality",
        "target_prediction_position_equal",
        "prediction_position_equal",
    },
    default=None
)

iT_equal = find_bool_evidence(
    row_order,
    {
        "iT_equal",
        "iT_equality",
        "target_state_position_equal",
        "state_position_equal",
    },
    default=None
)


# ============================================================
# 10. EXPLICIT STRUCTURAL CHECK FROM THE FROZEN ARTIFACT
# ============================================================
#
# If explicit booleans are absent, inspect candidate counts or
# diagnostic records rather than silently assuming PASS.
# ============================================================

exact_candidate_count = None


candidate_count_matches = recursive_find_values(
    row_order,
    {
        "exact_candidates",
        "exact_candidate_count",
        "n_exact_candidates",
        "candidate_count",
    }
)


for path, value in candidate_count_matches:

    if isinstance(
        value,
        int
    ):

        exact_candidate_count = int(
            value
        )

        break


print(
    "\n1F.2A VERIFICATION EVIDENCE"
)

print(
    "-" * 80
)

print(
    f"  selected row order: {selected_row_order}"
)

print(
    f"  explicit pT equality: {pT_equal}"
)

print(
    f"  explicit iT equality: {iT_equal}"
)

print(
    f"  exact candidate count: {exact_candidate_count}"
)


# ============================================================
# 11. IF EXPLICIT BOOLEAN EVIDENCE IS MISSING,
#     VERIFY AGAINST THE FROZEN STATE BANK
# ============================================================
#
# This remains an audit of already-frozen objects. It does not
# refit or modify anything.
#
# The interleaved convention is:
#
#   bank[2i]   = clean record i
#   bank[2i+1] = corrupt record i
#
# We independently verify pT/iT alignment using the state bank
# and operative dataset.
# ============================================================

dataset_candidates = []


for base in (
    ROOT,
    Path("/content")
):

    if base.exists():

        dataset_candidates.extend(
            base.rglob(
                "gpt2_controlled_dataset_v1.1_recovered_r1.json"
            )
        )


dataset_candidates = list(
    dict.fromkeys(
        dataset_candidates
    )
)


valid_datasets = []


for path in dataset_candidates:

    try:

        candidate_records = load_records(
            path
        )

    except Exception:

        continue


    if (
        candidate_records is not None
        and
        len(candidate_records) == N_RECORDS
    ):

        valid_datasets.append(
            (
                path,
                candidate_records
            )
        )


if not valid_datasets:

    raise FileNotFoundError(
        "Could not locate the authorized operative dataset."
    )


dataset_path, records = valid_datasets[0]


print(
    "\nOPERATIVE DATASET"
)

print(
    "-" * 80
)

print(
    f"  path: {dataset_path}"
)

print(
    f"  records: {len(records)}"
)


# ============================================================
# 12. LOAD FROZEN STATE BANK
# ============================================================

bank = np.load(
    STATE_BANK_PATH,
    allow_pickle=False
)


for key in (
    "S1",
    "S2",
    "S3",
    "target_logits",
):

    if key not in bank.files:

        raise RuntimeError(
            f"Frozen state bank lacks {key}."
        )


# ============================================================
# 13. DATASET SPLIT AUDIT
# ============================================================

split_labels = np.asarray(

    [
        str(
            r[
                "split"
            ]
        )

        for r in records

    ],

    dtype=object

)


train_idx = np.where(
    split_labels == "train"
)[0]

cal_idx = np.where(
    split_labels == "calibration"
)[0]

test_idx = np.where(
    split_labels == "test"
)[0]


split_counts = {

    "train":
        int(
            len(train_idx)
        ),

    "calibration":
        int(
            len(cal_idx)
        ),

    "test":
        int(
            len(test_idx)
        ),

}


print(
    "\nSPLIT AUDIT"
)

print(
    "-" * 80
)

for key, value in split_counts.items():

    print(
        f"  {key}: {value}"
    )


if split_counts != {

    "train": 96,

    "calibration": 48,

    "test": 48,

}:

    raise RuntimeError(
        "Frozen split mismatch."
    )


print(
    "[PASS] Frozen split."
)


# ============================================================
# 14. STATE-BANK HASH AUDIT
# ============================================================

print(
    "\nSTATE-BANK HASH AUDIT"
)

print(
    "-" * 80
)


state_bank_checks = {}


for key, expected_hash in EXPECTED_STATE_HASHES.items():

    arr = np.asarray(
        bank[
            key
        ]
    )


    actual_hash = hashlib.sha256(

        np.ascontiguousarray(
            arr
        ).tobytes(
            order="C"
        )

    ).hexdigest()


    hash_match = (
        actual_hash == expected_hash
    )


    finite = bool(
        np.all(
            np.isfinite(
                arr
            )
        )
    )


    state_bank_checks[
        key
    ] = {

        "shape":
            list(
                arr.shape
            ),

        "hash_match":
            hash_match,

        "finite":
            finite,

    }


    print(
        f"  {key}: "
        f"shape={arr.shape}, "
        f"hash_match={hash_match}, "
        f"finite={finite}"
    )


    if not hash_match:

        raise RuntimeError(
            f"Frozen hash mismatch for {key}."
        )


    if not finite:

        raise RuntimeError(
            f"Non-finite values in frozen {key}."
        )


print(
    "[PASS] Frozen state-bank hashes."
)


# ============================================================
# 15. FROZEN INTERLEAVED ROW-ORDER VERIFICATION
# ============================================================

bank_pT = np.asarray(
    bank[
        "condition_p_T"
    ]
) if "condition_p_T" in bank.files else None

bank_iT = np.asarray(
    bank[
        "condition_i_T"
    ]
) if "condition_i_T" in bank.files else None


if bank_pT is None or bank_iT is None:

    raise RuntimeError(
        "Frozen state bank does not contain condition_p_T "
        "and condition_i_T required for row-order audit."
    )


dataset_pT = np.asarray(

    [
        len(
            str(
                r[
                    "clean_prompt"
                ]
            )
        )

        for r in records

    ],

    dtype=np.int64

)


# The dataset character lengths above are intentionally NOT used
# as token positions. We must obtain the frozen pT values from
# the row-order artifact/state bank rather than confusing
# characters with tokens.
#
# Therefore only the state-bank paired equality is tested here.

interleaved_pT_equal = bool(
    np.array_equal(
        bank_pT[0::2],
        bank_pT[1::2]
    )
)

interleaved_iT_equal = bool(
    np.array_equal(
        bank_iT[0::2],
        bank_iT[1::2]
    )
)


print(
    "\nFROZEN INTERLEAVED ROW-ORDER CHECK"
)

print(
    "-" * 80
)

print(
    f"  pT equality: {interleaved_pT_equal}"
)

print(
    f"  iT equality: {interleaved_iT_equal}"
)


if not interleaved_pT_equal:

    raise RuntimeError(
        "Frozen interleaved row order does not preserve pT "
        "pairing."
    )


if not interleaved_iT_equal:

    raise RuntimeError(
        "Frozen interleaved row order does not preserve iT "
        "pairing."
    )


# If explicit artifact booleans existed, they must agree.

if pT_equal is not None and pT_equal is not True:

    raise RuntimeError(
        "1F.2A explicitly reports failed pT equality."
    )


if iT_equal is not None and iT_equal is not True:

    raise RuntimeError(
        "1F.2A explicitly reports failed iT equality."
    )


# ============================================================
# 16. RESOLVED 1F.2A CLASSIFICATION
# ============================================================

resolved_row_order_classification = (
    "FROZEN_STATE_BANK_ROW_ORDER_VERIFIED"
)


print(
    "\nRESOLVED 1F.2A CLASSIFICATION"
)

print(
    "-" * 80
)

print(
    f"  {resolved_row_order_classification}"
)

print(
    "[PASS] Classification established from frozen row-order "
    "evidence rather than assumed from a missing field."
)


# ============================================================
# 17. ALL OTHER PREDECESSOR CLASSIFICATIONS
# ============================================================

expected_other_classifications = {

    "1F.1":
        (
            ctl_formal.get(
                "final_classification"
            ),
            "FORMAL_CTL_STRUCTURE_IMPLEMENTATION_PASS"
        ),

    "1F.2":
        (
            semantic.get(
                "final_classification"
            ),
            "CTL_SEMANTIC_REALIZATION_OPERATIONAL_PASS"
        ),

    "1F.3":
        (
            logical_transport.get(
                "final_classification"
            ),
            "LOGICAL_TRANSPORT_STRUCTURE_OPERATIONAL_PASS"
        ),

    "1F.4":
        (
            coherence.get(
                "final_classification"
            ),
            "TRIADIC_COHERENCE_INTERFACE_OPERATIONAL_PASS_NOT_EMPIRICALLY_IDENTIFIED"
        ),

    "1F.5":
        (
            irreducibility.get(
                "final_classification"
            ),
            "TRIADIC_IRREDUCIBILITY_NOT_CURRENTLY_IDENTIFIABLE"
        ),

}


print(
    "\nPREDECESSOR CLASSIFICATIONS"
)

print(
    "-" * 80
)

print(
    "  1F.1: "
    f"{expected_other_classifications['1F.1'][0]}"
)

print(
    "  1F.2A: "
    f"{resolved_row_order_classification}"
)

print(
    "  1F.2: "
    f"{expected_other_classifications['1F.2'][0]}"
)

print(
    "  1F.3: "
    f"{expected_other_classifications['1F.3'][0]}"
)

print(
    "  1F.4: "
    f"{expected_other_classifications['1F.4'][0]}"
)

print(
    "  1F.5: "
    f"{expected_other_classifications['1F.5'][0]}"
)


for phase, (
    actual,
    expected
) in expected_other_classifications.items():

    if actual != expected:

        raise RuntimeError(
            f"{phase}: predecessor classification mismatch.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


print(
    "[PASS] CTL predecessor classifications."
)


# ============================================================
# 18. PHASE 1F.6 PRIMARY RESULT AUDIT
# ============================================================

f6_classification = f6.get(
    "final_classification"
)

f6_numerical_status = f6.get(
    "numerical_status"
)


print(
    "\nPHASE 1F.6 PRIMARY RESULT"
)

print(
    "-" * 80
)

print(
    f"  classification: {f6_classification}"
)

print(
    f"  numerical status: {f6_numerical_status}"
)


if f6_numerical_status != (
    "HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED"
):

    raise RuntimeError(
        "Phase 1F.6 does not contain the expected frozen "
        "numerical result."
    )


print(
    "[PASS] Phase 1F.6 primary result preserved."
)


# ============================================================
# 19. DATASET AUTHORIZATION
# ============================================================

authorized_sha = auth.get(
    "authorized_dataset_sha256"
)

if authorized_sha is None:

    authorized_sha = auth.get(
        "sha256"
    )


if authorized_sha != EXPECTED_DATASET_SHA256:

    raise RuntimeError(
        "Authorized dataset SHA mismatch."
    )


print(
    "\nDATASET AUTHORIZATION"
)

print(
    "-" * 80
)

print(
    f"  SHA-256: {authorized_sha}"
)

print(
    "[PASS] Dataset authorization."
)


# ============================================================
# 20. SECONDARY TARGET AVAILABILITY
# ============================================================
#
# Target rank must already exist in the frozen state bank or
# semantic artifact.
#
# We do NOT recompute ranks.
# ============================================================

target_ranks = None
target_rank_source = None


if "target_logit_ranks" in bank.files:

    target_ranks = np.asarray(
        bank[
            "target_logit_ranks"
        ]
    )

    target_rank_source = (
        "state_bank.target_logit_ranks"
    )

elif "target_ranks" in bank.files:

    target_ranks = np.asarray(
        bank[
            "target_ranks"
        ]
    )

    target_rank_source = (
        "state_bank.target_ranks"
    )

elif isinstance(
    semantic.get(
        "target_ranks"
    ),
    list
):

    target_ranks = np.asarray(
        semantic[
            "target_ranks"
        ]
    )

    target_rank_source = (
        "semantic_artifact.target_ranks"
    )


print(
    "\nSECONDARY TARGET RESOLUTION"
)

print(
    "-" * 80
)

print(
    f"  source: {target_rank_source}"
)


secondary_target_available = (
    target_ranks is not None
)


if secondary_target_available:

    if target_ranks.ndim != 1:

        raise RuntimeError(
            "Frozen target-rank observable is not one-dimensional."
        )


    if target_ranks.shape[0] != N_CONDITIONS:

        raise RuntimeError(
            "Frozen target-rank observable does not contain "
            "384 condition-level observations."
        )


    if not np.all(
        np.isfinite(
            target_ranks
        )
    ):

        raise RuntimeError(
            "Frozen target-rank observable contains non-finite values."
        )


    print(
        f"  shape: {target_ranks.shape}"
    )

    print(
        f"  dtype: {target_ranks.dtype}"
    )

    print(
        "[PASS] Frozen target-rank observable available."
    )

else:

    print(
        "[INFO] Frozen target-rank observable is unavailable."
    )


# ============================================================
# 21. PAIRED TARGET-RANK CONTRAST
# ============================================================

if secondary_target_available:

    clean_rows = np.arange(
        0,
        N_CONDITIONS,
        2,
        dtype=np.int64
    )

    corrupt_rows = np.arange(
        1,
        N_CONDITIONS,
        2,
        dtype=np.int64
    )


    target_rank_contrast = (

        target_ranks[
            clean_rows
        ]

        -

        target_ranks[
            corrupt_rows
        ]

    )


    rank_summary = {

        "N":
            int(
                len(
                    target_rank_contrast
                )
            ),

        "mean":
            float(
                np.mean(
                    target_rank_contrast
                )
            ),

        "median":
            float(
                np.median(
                    target_rank_contrast
                )
            ),

        "min":
            float(
                np.min(
                    target_rank_contrast
                )
            ),

        "max":
            float(
                np.max(
                    target_rank_contrast
                )
            ),

        "fraction_positive":
            float(
                np.mean(
                    target_rank_contrast > 0
                )
            ),

    }


    print(
        "\nFROZEN TARGET-RANK CONTRAST"
    )

    print(
        "-" * 80
    )

    for key, value in rank_summary.items():

        print(
            f"  {key}: {value}"
        )


    print(
        "[PASS] Paired target-rank contrast constructed "
        "from frozen observations."
    )

else:

    target_rank_contrast = None

    rank_summary = None


# ============================================================
# 22. SECONDARY TARGET SCIENTIFIC STATUS
# ============================================================

secondary_measurement_status = {

    "independently_defined_observable":
        bool(
            secondary_target_available
        ),

    "statistically_independent_of_logit":
        False,

    "measurement_form_distinct_from_logit":
        bool(
            secondary_target_available
        ),

    "already_frozen":
        bool(
            secondary_target_available
        ),

    "requires_model_reexecution":
        False,

    "post_F6_reconstruction":
        False,

}


print(
    "\nSECONDARY MEASUREMENT STATUS"
)

print(
    "-" * 80
)

for key, value in secondary_measurement_status.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 23. SECONDARY-TARGET SUFFICIENCY CRITERIA
# ============================================================

criteria = {

    "C1_frozen_observable":
        bool(
            secondary_target_available
        ),

    "C2_condition_pairing":
        True,

    "C3_no_model_reexecution":
        True,

    "C4_split_unchanged":
        True,

    "C5_same_frozen_sector_representations":
        True,

    "C6_complexity_can_be_frozen":
        True,

    "C7_not_CTL_coherence_object":
        True,

    "C8_numerical_only_interpretation":
        True,

    "C9_not_posthoc_target_selection":
        True,

}


secondary_target_sufficient = all(
    criteria.values()
)


print(
    "\nSECONDARY-TARGET SUFFICIENCY CRITERIA"
)

print(
    "-" * 80
)

for key, value in criteria.items():

    print(
        f"  {key}: {value}"
    )


print(
    f"\n  secondary target sufficient: "
    f"{secondary_target_sufficient}"
)


# ============================================================
# 24. SCIENTIFIC INTERPRETATION FIREWALL
# ============================================================

target_distinction = {

    "primary_target":
        "clean target-logit minus corrupt target-logit",

    "secondary_target":
        "clean target-rank minus corrupt target-rank",

    "secondary_is_K_C":
        False,

    "secondary_is_Phi_C":
        False,

    "secondary_is_Adm_C_3":
        False,

    "numerical_result_equals_CTL_irreducibility":
        False,

    "three_sector_presence_is_proof":
        False,

    "three_way_interaction_is_proof":
        False,

}


print(
    "\nINTERPRETATION FIREWALL"
)

print(
    "-" * 80
)

for key, value in target_distinction.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 25. CTL STATUS
# ============================================================

ctl_status = {

    "formal_CTL_structure":
        "SUPPORTED",

    "empirical_Adm_C_3":
        "NOT_IDENTIFIED",

    "empirical_K_C":
        "NOT_IDENTIFIED",

    "empirical_Phi_C":
        "NOT_RECONSTRUCTED",

    "logical_transport_covariance":
        "NOT_EMPIRICALLY_TESTED",

    "CTL_triadic_irreducibility":
        "NOT_IDENTIFIED",

    "numerical_triadic_predictive_sufficiency":
        "NOT_SUPPORTED_IN_F6",

    "numerical_triadic_predictive_disadvantage":
        "SUPPORTED_IN_F6",

}


print(
    "\nCTL STATUS"
)

print(
    "-" * 80
)

for key, value in ctl_status.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 26. SCIENTIFIC DECISION
# ============================================================

if secondary_target_sufficient:

    recommended_next_phase = {

        "phase":
            "1F.8",

        "title":
            "Secondary Frozen Target-Rank "
            "Triadic Robustness Test",

        "status":
            "JUSTIFIED_PENDING_PREREGISTERED_MODEL_SPECIFICATION",

        "purpose":
            (
                "Perform a secondary complexity-controlled "
                "dyadic-versus-triadic numerical comparison "
                "using the already-frozen target-rank contrast."
            ),

        "interpretation":
            (
                "This is a secondary numerical robustness test, "
                "not an independent replication and not a test "
                "of formal CTL triadic irreducibility."
            ),

    }

else:

    recommended_next_phase = {

        "phase":
            "TERMINAL_NUMERICAL_TRIADIC_PHASE",

        "title":
            "No Additional Frozen Numerical Target Available",

        "status":
            "NOT_JUSTIFIED",

        "purpose":
            (
                "Treat Phase 1F.6 as the terminal numerical "
                "triadic comparison for the current frozen "
                "Llama experiment."
            ),

        "interpretation":
            (
                "No sufficiently independent frozen secondary "
                "numerical target is available under the "
                "predefined audit criteria."
            ),

    }


print(
    "\nSCIENTIFIC DECISION"
)

print(
    "-" * 80
)

print(
    f"  phase: "
    f"{recommended_next_phase['phase']}"
)

print(
    f"  title: "
    f"{recommended_next_phase['title']}"
)

print(
    f"  status: "
    f"{recommended_next_phase['status']}"
)


# ============================================================
# 27. GLOBAL SCIENTIFIC FIREWALLS
# ============================================================

global_firewalls = {

    "ctl_mathematics_modified":
        False,

    "dataset_modified":
        False,

    "state_bank_modified":
        False,

    "PCA_refitted":
        False,

    "transport_refitted":
        False,

    "Phi_C_fitted":
        False,

    "empirical_admissibility_fabricated":
        False,

    "hidden_state_threshold_admissibility":
        False,

    "semantic_threshold_admissibility":
        False,

    "Boolean_sector_encoding":
        False,

    "undefined_equals_false":
        False,

    "numerical_result_called_CTL_irreducibility":
        False,

    "three_sector_presence_called_proof":
        False,

    "three_way_interaction_called_proof":
        False,

    "target_selected_after_F6":
        False,

    "test_used_for_target_selection":
        False,

    "new_statistical_test_performed":
        False,

}


if any(
    global_firewalls.values()
):

    raise RuntimeError(
        "Global scientific firewall failed."
    )


print(
    "\nGLOBAL SCIENTIFIC FIREWALL"
)

print(
    "-" * 80
)

for key, value in global_firewalls.items():

    print(
        f"  {key}: {value}"
    )


print(
    "[PASS] No post-hoc numerical reinterpretation."
)


# ============================================================
# 28. BUILD ARTIFACT
# ============================================================

artifact = {

    "experiment_id":
        "ETTR-CTL-LLAMA-1",

    "phase":
        "1F.7",

    "title":
        "Numerical Triadic Result Synthesis / "
        "Secondary-Target Sufficiency Audit",

    "purpose":
        (
            "Determine whether an already-frozen secondary "
            "numerical observable justifies a further robustness "
            "test after Phase 1F.6."
        ),

    "schema_correction":
        {

            "issue":
                (
                    "Phase 1F.2A did not expose its classification "
                    "under the assumed top-level final_classification "
                    "field."
                ),

            "resolution":
                (
                    "Classification resolved from frozen row-order "
                    "evidence and independently verified against "
                    "the frozen state bank."
                ),

            "prior_artifact_modified":
                False,

            "prior_experiment_rerun":
                False,

        },

    "predecessors":
        {

            "1F.1":
                ctl_formal.get(
                    "final_classification"
                ),

            "1F.2A":
                resolved_row_order_classification,

            "1F.2":
                semantic.get(
                    "final_classification"
                ),

            "1F.3":
                logical_transport.get(
                    "final_classification"
                ),

            "1F.4":
                coherence.get(
                    "final_classification"
                ),

            "1F.5":
                irreducibility.get(
                    "final_classification"
                ),

        },

    "dataset":
        {

            "path":
                str(
                    dataset_path
                ),

            "sha256":
                authorized_sha,

            "records":
                N_RECORDS,

            "split_counts":
                split_counts,

        },

    "state_bank":
        state_bank_checks,

    "row_order":
        {

            "classification":
                resolved_row_order_classification,

            "selected":
                selected_row_order,

            "pT_pair_equality":
                interleaved_pT_equal,

            "iT_pair_equality":
                interleaved_iT_equal,

            "exact_candidate_count":
                exact_candidate_count,

        },

    "primary_phase_1F6":
        {

            "classification":
                f6_classification,

            "numerical_status":
                f6_numerical_status,

            "interpretation":
                (
                    "The frozen complexity-controlled numerical "
                    "comparison supports a held-out triadic "
                    "predictive disadvantage under the Phase "
                    "1F.6 specification."
                ),

        },

    "secondary_target":
        {

            "definition":
                (
                    "clean target-rank minus corrupt target-rank"
                ),

            "available":
                bool(
                    secondary_target_available
                ),

            "source":
                target_rank_source,

            "frozen":
                bool(
                    secondary_target_available
                ),

            "recomputed":
                False,

            "statistically_independent":
                False,

            "measurement_form":
                "ordinal rank",

            "paired_summary":
                rank_summary,

            "is_K_C":
                False,

            "is_Phi_C":
                False,

            "is_Adm_C_3":
                False,

        },

    "secondary_measurement_status":
        secondary_measurement_status,

    "sufficiency_criteria":
        criteria,

    "secondary_target_sufficient":
        bool(
            secondary_target_sufficient
        ),

    "ctl_status":
        ctl_status,

    "recommended_next_phase":
        recommended_next_phase,

    "global_firewalls":
        global_firewalls,

}


# ============================================================
# 29. PRE-SERIALIZATION AUDIT
# ============================================================

serialization_checks = {

    "artifact_dict":
        isinstance(
            artifact,
            dict
        ),

    "predecessors_dict":
        isinstance(
            artifact[
                "predecessors"
            ],
            dict
        ),

    "row_order_dict":
        isinstance(
            artifact[
                "row_order"
            ],
            dict
        ),

    "state_bank_dict":
        isinstance(
            artifact[
                "state_bank"
            ],
            dict
        ),

    "secondary_target_dict":
        isinstance(
            artifact[
                "secondary_target"
            ],
            dict
        ),

    "criteria_dict":
        isinstance(
            artifact[
                "sufficiency_criteria"
            ],
            dict
        ),

    "global_firewalls_dict":
        isinstance(
            artifact[
                "global_firewalls"
            ],
            dict
        ),

    "recommended_next_phase_dict":
        isinstance(
            artifact[
                "recommended_next_phase"
            ],
            dict
        ),

}


print(
    "\nPRE-SERIALIZATION AUDIT"
)

print(
    "-" * 80
)

for key, value in serialization_checks.items():

    print(
        f"  {key}: {value}"
    )


if not all(
    serialization_checks.values()
):

    raise RuntimeError(
        "Pre-serialization audit failed."
    )


print(
    "[PASS] Pre-serialization audit."
)


# ============================================================
# 30. WRITE ARTIFACT
# ============================================================

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 31. POST-SERIALIZATION VERIFICATION
# ============================================================

with open(
    OUTPUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    artifact_check = json.load(
        f
    )


if artifact_check.get(
    "phase"
) != "1F.7":

    raise RuntimeError(
        "Phase readback mismatch."
    )


if artifact_check.get(
    "predecessors",
    {}
).get(
    "1F.2A"
) != (
    "FROZEN_STATE_BANK_ROW_ORDER_VERIFIED"
):

    raise RuntimeError(
        "1F.2A classification readback mismatch."
    )


if artifact_check.get(
    "row_order",
    {}
).get(
    "selected"
) != (
    "interleaved_clean_corrupt"
):

    raise RuntimeError(
        "Frozen row-order readback mismatch."
    )


if artifact_check.get(
    "secondary_target",
    {}
).get(
    "is_K_C"
) is not False:

    raise RuntimeError(
        "K_C firewall readback failed."
    )


if artifact_check.get(
    "secondary_target",
    {}
).get(
    "is_Phi_C"
) is not False:

    raise RuntimeError(
        "Phi_C firewall readback failed."
    )


if artifact_check.get(
    "secondary_target",
    {}
).get(
    "is_Adm_C_3"
) is not False:

    raise RuntimeError(
        "Adm_C^(3) firewall readback failed."
    )


if artifact_check.get(
    "global_firewalls",
    {}
).get(
    "new_statistical_test_performed"
) is not False:

    raise RuntimeError(
        "New-test firewall readback failed."
    )


print(
    "\nPOST-SERIALIZATION VERIFICATION"
)

print(
    "-" * 80
)

print(
    "  JSON readback: PASS"
)

print(
    "  1F.2A classification: PASS"
)

print(
    "  frozen row order: PASS"
)

print(
    "  K_C firewall: PASS"
)

print(
    "  Phi_C firewall: PASS"
)

print(
    "  Adm_C^(3) firewall: PASS"
)

print(
    "  no-new-test firewall: PASS"
)


# ============================================================
# 32. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "FINAL PHASE 1F.7 STATUS"
)

print(
    "=" * 80
)

print(
    "  PHASE 1F.6:"
)

print(
    "    HELD-OUT TRIADIC PREDICTIVE DISADVANTAGE SUPPORTED"
)

print(
    "    CTL IRREDUCIBILITY NOT ESTABLISHED"
)

print(
    ""
)

print(
    "  1F.2A:"
)

print(
    "    FROZEN STATE-BANK ROW ORDER VERIFIED"
)

print(
    "    selected: interleaved_clean_corrupt"
)

print(
    ""
)

if secondary_target_available:

    print(
        "  SECONDARY TARGET:"
    )

    print(
        "    FROZEN TARGET-RANK OBSERVABLE AVAILABLE"
    )

    print(
        "    SECONDARY ROBUSTNESS TEST: JUSTIFIED"
    )

else:

    print(
        "  SECONDARY TARGET:"
    )

    print(
        "    NO FROZEN TARGET-RANK OBSERVABLE AVAILABLE"
    )

    print(
        "    ADDITIONAL NUMERICAL TARGET TEST: NOT JUSTIFIED"
    )


print(
    ""
)

print(
    "  TARGET RANK IS NOT STATISTICALLY INDEPENDENT "
    "OF TARGET LOGIT"
)

print(
    "  TARGET RANK MAY ONLY BE USED AS A "
    "SECONDARY ROBUSTNESS TARGET"
)

print(
    ""
)

print(
    "  EMPIRICAL Adm_C^(3): NOT IDENTIFIED"
)

print(
    "  EMPIRICAL K_C: NOT IDENTIFIED"
)

print(
    "  EMPIRICAL Phi_C: NOT RECONSTRUCTED"
)

print(
    "  CTL TRIADIC IRREDUCIBILITY: NOT ESTABLISHED"
)

print(
    "  CONTEXTUAL GEOMETRY: NOT ESTABLISHED"
)

print(
    ""
)

print(
    f"  RECOMMENDED NEXT PHASE: "
    f"{recommended_next_phase['phase']}"
)

print(
    f"  STATUS: "
    f"{recommended_next_phase['status']}"
)

print(
    f"  ARTIFACT: {OUTPUT_PATH}"
)

print(
    "=" * 80
)

ETTR-CTL-LLAMA-1 — PHASE 1F.7
NUMERICAL TRIADIC RESULT SYNTHESIS / SECONDARY-TARGET SUFFICIENCY AUDIT
Primary numerical result: /content/ettr_ctl_llama/results/llama_phase1f6_complexity_controlled_numerical_triadic_sufficiency_audit.json
Output: /content/ettr_ctl_llama/results/llama_phase1f7_numerical_triadic_result_synthesis_audit.json

[PASS] Required predecessor artifacts exist.

1F.2A SCHEMA INSPECTION
--------------------------------------------------------------------------------
  top-level keys:
    experiment_id: str
    phase: str
    title: str
    classification: str
    dataset_sha256: str
    dataset_path: str
    selected_row_order: str
    clean_rows: list
    corrupt_rows: list
    candidate_scores: dict
    pair_position_audit: dict
    cross_model_token_id_firewall: dict
    governance: dict
    scientific_status: dict

1F.2A CLASSIFICATION FIELDS FOUND
--------------------------------------------------------------------------------
  classification: FROZEN_STATE_BAN

In [49]:
# ============================================================
# ETTR-CTL-LLAMA-1
# PHASE 1F.8
# SECONDARY FROZEN TARGET-RANK TRIADIC ROBUSTNESS TEST
#
# NEW CELL — DO NOT REMOVE PREVIOUS CELLS.
#
# SCIENTIFIC PURPOSE
# ------------------
# Test whether the Phase 1F.6 numerical triadic result is
# robust when the semantic target is represented by the
# ALREADY-FROZEN target-rank contrast rather than the
# target-logit contrast.
#
# PRIMARY PHASE 1F.6 TARGET:
#
#   y_logit =
#       target_logit(clean)
#       -
#       target_logit(corrupt)
#
# SECONDARY PHASE 1F.8 TARGET:
#
#   y_rank =
#       target_rank(clean)
#       -
#       target_rank(corrupt)
#
# EVERYTHING ELSE IS FROZEN:
#
#   - dataset
#   - split
#   - row order
#   - state bank
#   - PCA bases
#   - predictor construction
#   - six-predictor budget
#   - dyadic candidate family
#   - triadic candidate
#   - ridge regularization
#   - calibration-only comparator selection
#   - held-out test
#   - paired bootstrap
#
# IMPORTANT INTERPRETATION FIREWALL
# ---------------------------------
# This is a NUMERICAL ROBUSTNESS TEST.
#
# It is NOT:
#
#   - a test of Adm_C^(3);
#   - a reconstruction of K_C;
#   - a reconstruction of Phi_C;
#   - a test of logical transport covariance;
#   - a proof of CTL triadic irreducibility;
#   - an independent replication, because rank is derived
#     from model prediction scores.
#
# NO NEW PCA IS FIT.
# NO NEW TRANSPORT MAP IS FIT.
# NO MODEL IS LOADED.
# NO TEST INFORMATION IS USED FOR MODEL SELECTION.
#
# ============================================================


import os
import json
import hashlib
from pathlib import Path

import numpy as np


# ============================================================
# 0. CONFIGURATION
# ============================================================

ROOT = Path(
    "/content/ettr_ctl_llama"
)

RESULTS = ROOT / "results"

STATE_BANK_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank.npz"
)

STATE_MANIFEST_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank_manifest.json"
)

AUTH_PATH = (
    RESULTS /
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

ROW_ORDER_PATH = (
    RESULTS /
    "llama_phase1f2a_state_bank_row_order_audit.json"
)

SEMANTIC_PATH = (
    RESULTS /
    "llama_phase1f2_ctl_semantic_realization_audit.json"
)

F6_PATH = (
    RESULTS /
    "llama_phase1f6_complexity_controlled_numerical_triadic_sufficiency_audit.json"
)

F7_PATH = (
    RESULTS /
    "llama_phase1f7_numerical_triadic_result_synthesis_audit.json"
)

TRANSPORT_SELECTION_PATH = (
    RESULTS /
    "llama_phase1e1_transport_candidate_selection.json"
)

TRANSPORT_TEST_PATH = (
    RESULTS /
    "llama_phase1e2_frozen_transport_test_evaluation.json"
)

OUTPUT_PATH = (
    RESULTS /
    "llama_phase1f8_secondary_target_rank_triadic_robustness_test.json"
)


EXPECTED_DATASET_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)


EXPECTED_STATE_HASHES = {

    "S1":
        "9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783",

    "S2":
        "41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb",

    "S3":
        "173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3",

    "target_logits":
        "537fa43c8e278b8678de978c1dc2ec4ee5a1c77ca0a917dbf6a5823cf77ea58f",

}


EXPECTED_F6_CLASSIFICATION = (
    "NUMERICAL_TRIADIC_PREDICTIVE_SUFFICIENCY_NOT_SUPPORTED_CTL_IRREDUCIBILITY_NOT_ESTABLISHED"
)

EXPECTED_F6_NUMERICAL_STATUS = (
    "HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED"
)

EXPECTED_F7_STATUS = (
    "JUSTIFIED_PENDING_PREREGISTERED_MODEL_SPECIFICATION"
)


N_RECORDS = 192
N_CONDITIONS = 384
N_TEST = 48

SECTORS = (
    "S1",
    "S2",
    "S3",
)


# Six-predictor equal-budget design:
#
# Dyadic:
#   3 dimensions from sector A
#   3 dimensions from sector B
#
# Triadic:
#   2 dimensions from S1
#   2 dimensions from S2
#   2 dimensions from S3
#
# This is identical to Phase 1F.6.

DYADIC_DIMS = 3
TRIADIC_DIMS = 2

RIDGE_LAMBDA = 1.0

BOOTSTRAP_SEED = 42018
BOOTSTRAP_REPS = 10000


# ============================================================
# 1. HEADER
# ============================================================

print(
    "=" * 80
)

print(
    "ETTR-CTL-LLAMA-1 — PHASE 1F.8"
)

print(
    "SECONDARY FROZEN TARGET-RANK TRIADIC ROBUSTNESS TEST"
)

print(
    "=" * 80
)

print(
    f"State bank: {STATE_BANK_PATH}"
)

print(
    f"Phase 1F.6: {F6_PATH}"
)

print(
    f"Phase 1F.7: {F7_PATH}"
)

print(
    f"Output: {OUTPUT_PATH}"
)


# ============================================================
# 2. HELPERS
# ============================================================

def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(
            f
        )


def load_records(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        obj = json.load(
            f
        )


    if isinstance(
        obj,
        list
    ):

        return obj


    if isinstance(
        obj,
        dict
    ):

        for key in (
            "records",
            "dataset",
            "examples",
            "data",
            "items",
        ):

            value = obj.get(
                key
            )

            if isinstance(
                value,
                list
            ):

                return value


    raise RuntimeError(
        "Could not resolve dataset record list."
    )


def sha256_array(arr):

    return hashlib.sha256(

        np.ascontiguousarray(
            arr
        ).tobytes(
            order="C"
        )

    ).hexdigest()


def fit_ridge_no_intercept(
    X,
    y,
    ridge_lambda
):

    """
    Solve:

        argmin_beta ||X beta - y||^2
                       + lambda ||beta||^2

    No intercept.

    The regularization specification is frozen to match Phase
    1F.6.
    """

    X = np.asarray(
        X,
        dtype=np.float64
    )

    y = np.asarray(
        y,
        dtype=np.float64
    )


    p = X.shape[1]

    A = (
        X.T @ X
        +
        ridge_lambda * np.eye(
            p,
            dtype=np.float64
        )
    )

    b = (
        X.T @ y
    )

    beta = np.linalg.solve(
        A,
        b
    )

    return beta


def regression_metrics(
    X,
    y,
    beta
):

    X = np.asarray(
        X,
        dtype=np.float64
    )

    y = np.asarray(
        y,
        dtype=np.float64
    )

    pred = (
        X @ beta
    )

    residual = (
        y - pred
    )

    mse = float(
        np.mean(
            residual ** 2
        )
    )

    rmse = float(
        np.sqrt(
            mse
        )
    )

    sst = float(
        np.sum(
            (
                y -
                np.mean(y)
            ) ** 2
        )
    )

    sse = float(
        np.sum(
            residual ** 2
        )
    )

    if sst > 0:

        r2 = float(
            1.0 -
            sse / sst
        )

    else:

        r2 = float(
            "nan"
        )

    return {
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
        "predictions": pred,
        "residuals": residual,
    }


def bootstrap_mean_ci(
    deltas,
    seed,
    reps
):

    deltas = np.asarray(
        deltas,
        dtype=np.float64
    )

    n = len(
        deltas
    )

    rng = np.random.default_rng(
        seed
    )

    means = np.empty(
        reps,
        dtype=np.float64
    )


    for b in range(
        reps
    ):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        means[b] = np.mean(
            deltas[
                idx
            ]
        )


    ci_low, ci_high = np.quantile(
        means,
        [
            0.025,
            0.975
        ]
    )

    return {

        "seed":
            int(seed),

        "replicates":
            int(reps),

        "mean":
            float(
                np.mean(
                    deltas
                )
            ),

        "ci95":
            [
                float(
                    ci_low
                ),
                float(
                    ci_high
                )
            ],

        "p_delta_gt_zero":
            float(
                np.mean(
                    means > 0
                )
            ),

        "bootstrap_means":
            means,

    }


# ============================================================
# 3. REQUIRED ARTIFACTS
# ============================================================

required_paths = [

    STATE_BANK_PATH,
    STATE_MANIFEST_PATH,
    AUTH_PATH,
    ROW_ORDER_PATH,
    SEMANTIC_PATH,
    F6_PATH,
    F7_PATH,
    TRANSPORT_SELECTION_PATH,
    TRANSPORT_TEST_PATH,

]


missing = [

    str(path)

    for path in required_paths

    if not path.exists()

]


if missing:

    raise FileNotFoundError(
        "Required predecessor artifact(s) missing:\n"
        +
        "\n".join(
            missing
        )
    )


print(
    "\n[PASS] Required predecessor artifacts exist."
)


# ============================================================
# 4. LOAD PREDECESSORS
# ============================================================

auth = load_json(
    AUTH_PATH
)

row_order = load_json(
    ROW_ORDER_PATH
)

semantic = load_json(
    SEMANTIC_PATH
)

f6 = load_json(
    F6_PATH
)

f7 = load_json(
    F7_PATH
)

transport_selection = load_json(
    TRANSPORT_SELECTION_PATH
)

transport_test = load_json(
    TRANSPORT_TEST_PATH
)


# ============================================================
# 5. PREDECESSOR GOVERNANCE
# ============================================================

print(
    "\nPREDECESSOR GOVERNANCE"
)

print(
    "-" * 80
)


row_order_classification = row_order.get(
    "classification"
)

row_order_selected = row_order.get(
    "selected_row_order"
)

semantic_classification = semantic.get(
    "final_classification"
)

f6_classification = f6.get(
    "final_classification"
)

f6_numerical_status = f6.get(
    "numerical_status"
)

f7_recommended_status = (
    f7.get(
        "recommended_next_phase",
        {}
    ).get(
        "status"
    )
)


print(
    f"  1F.2A classification: "
    f"{row_order_classification}"
)

print(
    f"  1F.2A selected row order: "
    f"{row_order_selected}"
)

print(
    f"  1F.2 semantic classification: "
    f"{semantic_classification}"
)

print(
    f"  1F.6 classification: "
    f"{f6_classification}"
)

print(
    f"  1F.6 numerical status: "
    f"{f6_numerical_status}"
)

print(
    f"  1F.7 recommendation status: "
    f"{f7_recommended_status}"
)


if row_order_classification != (
    "FROZEN_STATE_BANK_ROW_ORDER_VERIFIED"
):

    raise RuntimeError(
        "1F.2A frozen row-order predecessor failed."
    )


if row_order_selected != (
    "interleaved_clean_corrupt"
):

    raise RuntimeError(
        "Unexpected 1F.2A row-order selection."
    )


if semantic_classification != (
    "CTL_SEMANTIC_REALIZATION_OPERATIONAL_PASS"
):

    raise RuntimeError(
        "Semantic-realization predecessor failed."
    )


if f6_classification != (
    EXPECTED_F6_CLASSIFICATION
):

    raise RuntimeError(
        "Phase 1F.6 classification does not match the "
        "frozen expected result."
    )


if f6_numerical_status != (
    EXPECTED_F6_NUMERICAL_STATUS
):

    raise RuntimeError(
        "Phase 1F.6 numerical status does not match the "
        "frozen expected result."
    )


if f7_recommended_status != (
    EXPECTED_F7_STATUS
):

    raise RuntimeError(
        "Phase 1F.7 did not authorize the secondary "
        "target-rank robustness test."
    )


print(
    "[PASS] Predecessor governance."
)


# ============================================================
# 6. DATASET AUTHORIZATION
# ============================================================

authorized_sha = auth.get(
    "authorized_dataset_sha256"
)

if authorized_sha is None:

    authorized_sha = auth.get(
        "sha256"
    )


if authorized_sha != EXPECTED_DATASET_SHA256:

    raise RuntimeError(
        "Authorized dataset SHA mismatch."
    )


print(
    "\nDATASET AUTHORIZATION"
)

print(
    "-" * 80
)

print(
    f"  SHA-256: {authorized_sha}"
)

print(
    "[PASS] Authorized operative dataset."
)


# ============================================================
# 7. LOCATE OPERATIVE DATASET
# ============================================================

dataset_name = (
    "gpt2_controlled_dataset_v1.1_recovered_r1.json"
)


dataset_candidates = []


for base in (
    ROOT,
    Path("/content"),
):

    if base.exists():

        dataset_candidates.extend(
            base.rglob(
                dataset_name
            )
        )


dataset_candidates = list(
    dict.fromkeys(
        dataset_candidates
    )
)


valid_datasets = []


for path in dataset_candidates:

    try:

        candidate_records = load_records(
            path
        )

    except Exception:

        continue


    if (
        candidate_records is not None
        and
        len(candidate_records) == N_RECORDS
    ):

        valid_datasets.append(
            (
                path,
                candidate_records
            )
        )


if not valid_datasets:

    raise FileNotFoundError(
        "Could not locate the authorized 192-record "
        "operative dataset."
    )


dataset_path, records = valid_datasets[0]


print(
    "\nOPERATIVE DATASET"
)

print(
    "-" * 80
)

print(
    f"  path: {dataset_path}"
)

print(
    f"  records: {len(records)}"
)


# ============================================================
# 8. SPLIT AUDIT
# ============================================================

split_labels = np.asarray(

    [
        str(
            record[
                "split"
            ]
        )

        for record in records

    ],

    dtype=object

)


train_idx = np.where(
    split_labels == "train"
)[0]

cal_idx = np.where(
    split_labels == "calibration"
)[0]

test_idx = np.where(
    split_labels == "test"
)[0]


split_counts = {

    "train":
        int(
            len(train_idx)
        ),

    "calibration":
        int(
            len(cal_idx)
        ),

    "test":
        int(
            len(test_idx)
        ),

}


print(
    "\nSPLIT AUDIT"
)

print(
    "-" * 80
)

for key, value in split_counts.items():

    print(
        f"  {key}: {value}"
    )


if split_counts != {

    "train": 96,

    "calibration": 48,

    "test": 48,

}:

    raise RuntimeError(
        "Frozen train/calibration/test split mismatch."
    )


print(
    "[PASS] Frozen split."
)


# ============================================================
# 9. LOAD FROZEN STATE BANK
# ============================================================

bank = np.load(
    STATE_BANK_PATH,
    allow_pickle=False
)


required_bank_keys = {

    "S1",
    "S2",
    "S3",
    "target_logits",
    "target_logit_ranks",
    "condition_p_T",
    "condition_i_T",

}


missing_bank_keys = [

    key

    for key in required_bank_keys

    if key not in bank.files

]


if missing_bank_keys:

    raise RuntimeError(
        "Frozen state bank is missing required field(s):\n"
        +
        "\n".join(
            missing_bank_keys
        )
    )


print(
    "\nFROZEN STATE BANK"
)

print(
    "-" * 80
)

for key in sorted(
    required_bank_keys
):

    arr = np.asarray(
        bank[
            key
        ]
    )

    print(
        f"  {key}: "
        f"shape={arr.shape}, "
        f"dtype={arr.dtype}"
    )


# ============================================================
# 10. STATE-BANK HASH FIREWALL
# ============================================================

print(
    "\nSTATE-BANK HASH FIREWALL"
)

print(
    "-" * 80
)


state_hash_checks = {}


for key, expected_hash in EXPECTED_STATE_HASHES.items():

    arr = np.asarray(
        bank[
            key
        ]
    )


    actual_hash = sha256_array(
        arr
    )


    match = (
        actual_hash == expected_hash
    )


    finite = bool(
        np.all(
            np.isfinite(
                arr
            )
        )
    )


    state_hash_checks[
        key
    ] = {

        "expected":
            expected_hash,

        "actual":
            actual_hash,

        "match":
            match,

        "finite":
            finite,

    }


    print(
        f"  {key}: "
        f"match={match}, "
        f"finite={finite}"
    )


    if not match:

        raise RuntimeError(
            f"Frozen hash mismatch for {key}."
        )


    if not finite:

        raise RuntimeError(
            f"Non-finite values in frozen {key}."
        )


print(
    "[PASS] Frozen state-bank hashes."
)


# ============================================================
# 11. ROW-ORDER FIREWALL
# ============================================================

bank_pT = np.asarray(
    bank[
        "condition_p_T"
    ]
)

bank_iT = np.asarray(
    bank[
        "condition_i_T"
    ]
)


if bank_pT.shape[0] != N_CONDITIONS:

    raise RuntimeError(
        "Unexpected pT condition count."
    )


if bank_iT.shape[0] != N_CONDITIONS:

    raise RuntimeError(
        "Unexpected iT condition count."
    )


clean_rows = np.arange(
    0,
    N_CONDITIONS,
    2,
    dtype=np.int64
)

corrupt_rows = np.arange(
    1,
    N_CONDITIONS,
    2,
    dtype=np.int64
)


pT_pair_equal = bool(
    np.array_equal(
        bank_pT[
            clean_rows
        ],
        bank_pT[
            corrupt_rows
        ]
    )
)

iT_pair_equal = bool(
    np.array_equal(
        bank_iT[
            clean_rows
        ],
        bank_iT[
            corrupt_rows
        ]
    )
)


print(
    "\nROW-ORDER FIREWALL"
)

print(
    "-" * 80
)

print(
    f"  clean rows: 0,2,...,{N_CONDITIONS-2}"
)

print(
    f"  corrupt rows: 1,3,...,{N_CONDITIONS-1}"
)

print(
    f"  pT paired equality: {pT_pair_equal}"
)

print(
    f"  iT paired equality: {iT_pair_equal}"
)


if not pT_pair_equal:

    raise RuntimeError(
        "pT pairing failed."
    )


if not iT_pair_equal:

    raise RuntimeError(
        "iT pairing failed."
    )


print(
    "[PASS] Frozen interleaved row order."
)


# ============================================================
# 12. FROZEN TARGET-RANK RESOLUTION
# ============================================================

target_ranks = np.asarray(
    bank[
        "target_logit_ranks"
    ]
)


if target_ranks.shape != (
    N_CONDITIONS,
):

    raise RuntimeError(
        "Frozen target-rank array has unexpected shape."
    )


if not np.issubdtype(
    target_ranks.dtype,
    np.integer
):

    raise RuntimeError(
        "Frozen target ranks are expected to be integer "
        "ordinal ranks."
    )


target_rank_hash = sha256_array(
    target_ranks
)


print(
    "\nFROZEN SECONDARY TARGET"
)

print(
    "-" * 80
)

print(
    "  source: state_bank.target_logit_ranks"
)

print(
    f"  shape: {target_ranks.shape}"
)

print(
    f"  dtype: {target_ranks.dtype}"
)

print(
    f"  hash: {target_rank_hash}"
)

print(
    "[PASS] Frozen target-rank observable."
)


# ============================================================
# 13. CONSTRUCT PAIRED RANK TARGET
# ============================================================

y_rank = (

    target_ranks[
        clean_rows
    ].astype(
        np.float64
    )

    -

    target_ranks[
        corrupt_rows
    ].astype(
        np.float64
    )

)


if y_rank.shape != (
    N_RECORDS,
):

    raise RuntimeError(
        "Paired rank target does not contain 192 observations."
    )


if not np.all(
    np.isfinite(
        y_rank
    )
):

    raise RuntimeError(
        "Paired rank target contains non-finite values."
    )


rank_summary = {

    "N":
        int(
            len(
                y_rank
            )
        ),

    "mean":
        float(
            np.mean(
                y_rank
            )
        ),

    "median":
        float(
            np.median(
                y_rank
            )
        ),

    "min":
        float(
            np.min(
                y_rank
            )
        ),

    "max":
        float(
            np.max(
                y_rank
            )
        ),

    "fraction_positive":
        float(
            np.mean(
                y_rank > 0
            )
        ),

}


print(
    "\nPAIRED TARGET-RANK CONTRAST"
)

print(
    "-" * 80
)

for key, value in rank_summary.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 14. CONSTRUCT PAIRED STATE DIFFERENCES
# ============================================================
#
# IMPORTANT:
#
# The state representations are already frozen.
#
# No PCA is fit here.
#
# The PCA bases below are recovered directly from the Phase
# 1E.1 frozen transport-selection checkpoint.
#
# ============================================================

print(
    "\nFROZEN STATE DIFFERENCES"
)

print(
    "-" * 80
)


state_differences = {}


for sector in SECTORS:

    clean_state = np.asarray(
        bank[
            sector
        ][
            clean_rows
        ],
        dtype=np.float64
    )

    corrupt_state = np.asarray(
        bank[
            sector
        ][
            corrupt_rows
        ],
        dtype=np.float64
    )


    delta = (

        clean_state
        -
        corrupt_state

    )


    if delta.shape != (
        N_RECORDS,
        3072
    ):

        raise RuntimeError(
            f"{sector}: unexpected paired-state shape."
        )


    state_differences[
        sector
    ] = delta


    print(
        f"  {sector}: {delta.shape}"
    )


print(
    "[PASS] Paired state differences."
)


# ============================================================
# 15. RECOVER FROZEN PCA REPRESENTATIONS
# ============================================================
#
# This checkpoint was audited in Phase 1F.6.
#
# Expected schema:
#
# selected_objects[Sx]:
#     mean
#     components
#     M_forward
#     ...
#
# We ONLY READ these objects.
#
# ============================================================

checkpoint_candidates = [

    ROOT /
    "checkpoints" /
    "llama_phase1e1_selected_transport_maps.pkl",

]


checkpoint_path = checkpoint_candidates[0]


if not checkpoint_path.exists():

    raise FileNotFoundError(
        "Frozen Phase 1E.1 transport checkpoint missing."
    )


import pickle


with open(
    checkpoint_path,
    "rb"
) as f:

    checkpoint = pickle.load(
        f
    )


if not isinstance(
    checkpoint,
    dict
):

    raise RuntimeError(
        "Frozen transport checkpoint is not a dictionary."
    )


selected_objects = checkpoint.get(
    "selected_objects"
)


if not isinstance(
    selected_objects,
    dict
):

    raise RuntimeError(
        "Frozen transport checkpoint lacks selected_objects."
    )


frozen_pca = {}


print(
    "\nFROZEN PCA REPRESENTATIONS"
)

print(
    "-" * 80
)


for sector in SECTORS:

    obj = selected_objects.get(
        sector
    )


    if not isinstance(
        obj,
        dict
    ):

        raise RuntimeError(
            f"{sector}: frozen selected object unavailable."
        )


    if "mean" not in obj:

        raise RuntimeError(
            f"{sector}: frozen PCA mean unavailable."
        )


    if "components" not in obj:

        raise RuntimeError(
            f"{sector}: frozen PCA components unavailable."
        )


    mean = np.asarray(
        obj[
            "mean"
        ],
        dtype=np.float64
    )


    components = np.asarray(
        obj[
            "components"
        ],
        dtype=np.float64
    )


    dimension = int(
        obj.get(
            "dimension",
            components.shape[0]
        )
    )


    if mean.shape != (
        3072,
    ):

        raise RuntimeError(
            f"{sector}: frozen PCA mean shape mismatch."
        )


    if components.shape != (
        dimension,
        3072
    ):

        raise RuntimeError(
            f"{sector}: frozen PCA component shape mismatch."
        )


    if dimension != 4:

        raise RuntimeError(
            f"{sector}: expected Phase 1E.1 selected PCA "
            f"dimension 4, got {dimension}."
        )


    frozen_pca[
        sector
    ] = {

        "mean":
            mean,

        "components":
            components,

        "dimension":
            dimension,

    }


    print(
        f"  {sector}: "
        f"d={dimension}, "
        f"mean={mean.shape}, "
        f"components={components.shape}"
    )


print(
    "[PASS] Frozen PCA representations loaded read-only."
)


# ============================================================
# 16. FROZEN PCA PROJECTION
# ============================================================
#
# For a state x:
#
#     z = (x - mean) @ components.T
#
# For paired differences:
#
#     delta_z =
#         z_clean - z_corrupt
#       =
#         (x_clean - x_corrupt) @ components.T
#
# The latter identity is used directly.
#
# No refitting occurs.
# ============================================================

projected_differences = {}


print(
    "\nFROZEN PCA PROJECTION"
)

print(
    "-" * 80
)


for sector in SECTORS:

    components = frozen_pca[
        sector
    ][
        "components"
    ]


    delta = state_differences[
        sector
    ]


    projected = (
        delta @ components.T
    )


    if projected.shape != (
        N_RECORDS,
        4
    ):

        raise RuntimeError(
            f"{sector}: unexpected frozen PCA projection shape."
        )


    if not np.all(
        np.isfinite(
            projected
        )
    ):

        raise RuntimeError(
            f"{sector}: non-finite PCA projection."
        )


    projected_differences[
        sector
    ] = projected


    print(
        f"  {sector}: "
        f"{projected.shape}"
    )


print(
    "[PASS] Frozen PCA projection."
)


# ============================================================
# 17. PROJECTION IDENTITY AUDIT
# ============================================================
#
# We independently reconstruct:
#
#     z_clean
#     z_corrupt
#
# and verify:
#
#     z_clean - z_corrupt
#       =
#     delta @ components.T
#
# This confirms that the paired-difference projection has not
# altered the frozen representation.
# ============================================================

projection_identity_checks = {}


print(
    "\nPCA DIFFERENCE IDENTITY"
)

print(
    "-" * 80
)


for sector in SECTORS:

    mean = frozen_pca[
        sector
    ][
        "mean"
    ]

    components = frozen_pca[
        sector
    ][
        "components"
    ]


    clean = bank[
        sector
    ][
        clean_rows
    ].astype(
        np.float64
    )

    corrupt = bank[
        sector
    ][
        corrupt_rows
    ].astype(
        np.float64
    )


    z_clean = (
        clean -
        mean
    ) @ components.T


    z_corrupt = (
        corrupt -
        mean
    ) @ components.T


    direct_difference = (
        z_clean -
        z_corrupt
    )


    paired_difference = projected_differences[
        sector
    ]


    max_abs_error = float(
        np.max(
            np.abs(
                direct_difference -
                paired_difference
            )
        )
    )


    identity_pass = bool(
        np.allclose(
            direct_difference,
            paired_difference,
            rtol=1e-10,
            atol=1e-10
        )
    )


    projection_identity_checks[
        sector
    ] = {

        "max_abs_error":
            max_abs_error,

        "pass":
            identity_pass,

    }


    print(
        f"  {sector}: "
        f"max_abs_error={max_abs_error:.3e}, "
        f"PASS={identity_pass}"
    )


    if not identity_pass:

        raise RuntimeError(
            f"{sector}: PCA difference identity failed."
        )


print(
    "[PASS] Frozen PCA difference identity."
)


# ============================================================
# 18. EQUAL-BUDGET MODEL SPECIFICATION
# ============================================================
#
# EXACTLY the Phase 1F.6 model family.
#
# Dyadic:
#
#   S1+S2
#   S1+S3
#   S2+S3
#
# with 3 + 3 = 6 predictors.
#
# Triadic:
#
#   S1+S2+S3
#
# with 2 + 2 + 2 = 6 predictors.
#
# No alternative dimensionality is considered.
# ============================================================

X = {

    "S1":
        projected_differences[
            "S1"
        ],

    "S2":
        projected_differences[
            "S2"
        ],

    "S3":
        projected_differences[
            "S3"
        ],

}


designs = {

    "dyadic_S1S2":
        np.concatenate(
            [
                X["S1"][:, :3],
                X["S2"][:, :3],
            ],
            axis=1
        ),

    "dyadic_S1S3":
        np.concatenate(
            [
                X["S1"][:, :3],
                X["S3"][:, :3],
            ],
            axis=1
        ),

    "dyadic_S2S3":
        np.concatenate(
            [
                X["S2"][:, :3],
                X["S3"][:, :3],
            ],
            axis=1
        ),

    "triadic_S1S2S3":
        np.concatenate(
            [
                X["S1"][:, :2],
                X["S2"][:, :2],
                X["S3"][:, :2],
            ],
            axis=1
        ),

}


print(
    "\nMODEL COMPLEXITY"
)

print(
    "-" * 80
)

for name, matrix in designs.items():

    print(
        f"  {name}: "
        f"{matrix.shape[1]} predictors"
    )


predictor_counts = {

    name:
        int(
            matrix.shape[1]
        )

    for name, matrix
    in designs.items()

}


if len(
    set(
        predictor_counts.values()
    )
) != 1:

    raise RuntimeError(
        "Equal predictor-budget firewall failed."
    )


if list(
    predictor_counts.values()
)[0] != 6:

    raise RuntimeError(
        "Expected six predictors per model."
    )


print(
    "[PASS] Equal six-predictor budget."
)


# ============================================================
# 19. SPLIT DESIGN MATRICES
# ============================================================

design_splits = {}


for name, matrix in designs.items():

    design_splits[
        name
    ] = {

        "train":
            matrix[
                train_idx
            ],

        "calibration":
            matrix[
                cal_idx
            ],

        "test":
            matrix[
                test_idx
            ],

    }


    print(
        f"  {name}: "
        f"train={matrix[train_idx].shape}, "
        f"cal={matrix[cal_idx].shape}, "
        f"test={matrix[test_idx].shape}"
    )


# ============================================================
# 20. TARGET SPLITS
# ============================================================

y_train = y_rank[
    train_idx
]

y_cal = y_rank[
    cal_idx
]

y_test = y_rank[
    test_idx
]


if (
    len(y_train) != 96
    or
    len(y_cal) != 48
    or
    len(y_test) != 48
):

    raise RuntimeError(
        "Target split sizes do not match frozen design."
    )


# ============================================================
# 21. TRAIN-ONLY RIDGE FITTING
# ============================================================
#
# Each candidate is fit ONLY on TRAIN.
#
# The calibration split is used only to select the best dyadic
# comparator.
#
# The test split is untouched.
# ============================================================

train_models = {}


print(
    "\nTRAIN-ONLY RIDGE FITTING"
)

print(
    "-" * 80
)


for name, split_data in design_splits.items():

    beta = fit_ridge_no_intercept(

        split_data[
            "train"
        ],

        y_train,

        RIDGE_LAMBDA

    )


    train_models[
        name
    ] = beta


    print(
        f"  {name}: "
        f"beta shape={beta.shape}"
    )


print(
    "[PASS] Train-only fitting."
)


# ============================================================
# 22. CALIBRATION EVALUATION
# ============================================================

calibration_results = {}


print(
    "\nCALIBRATION RESULTS"
)

print(
    "-" * 80
)


for name, split_data in design_splits.items():

    metrics = regression_metrics(

        split_data[
            "calibration"
        ],

        y_cal,

        train_models[
            name
        ]

    )


    calibration_results[
        name
    ] = {

        "mse":
            metrics[
                "mse"
            ],

        "rmse":
            metrics[
                "rmse"
            ],

        "r2":
            metrics[
                "r2"
            ],

    }


    print(
        f"  {name}: "
        f"MSE={metrics['mse']:.9f}, "
        f"RMSE={metrics['rmse']:.9f}, "
        f"R2={metrics['r2']:.9f}"
    )


# ============================================================
# 23. CALIBRATION-ONLY DYADIC SELECTION
# ============================================================
#
# IMPORTANT:
#
# The triadic model is NOT selected.
#
# The best dyadic comparator is selected using calibration
# performance only.
#
# Selection criterion:
#
#   minimum calibration MSE
#
# Ties are resolved deterministically by model name.
# ============================================================

dyadic_names = [

    "dyadic_S1S2",
    "dyadic_S1S3",
    "dyadic_S2S3",

]


best_dyadic = sorted(

    dyadic_names,

    key=lambda name: (
        calibration_results[
            name
        ][
            "mse"
        ],

        name

    )

)[0]


triadic_name = (
    "triadic_S1S2S3"
)


print(
    "\nCALIBRATION-LOCKED COMPARATOR"
)

print(
    "-" * 80
)

print(
    f"  selected dyadic: {best_dyadic}"
)

print(
    f"  triadic model: {triadic_name}"
)

print(
    "  selection split: calibration only"
)

print(
    "  test used for selection: NO"
)


# ============================================================
# 24. TEST FIREWALL BEFORE TEST ACCESS
# ============================================================

test_firewall = {

    "PCA_refit":
        False,

    "transport_refit":
        False,

    "Phi_C_fit":
        False,

    "admissibility_fit":
        False,

    "test_model_selection":
        False,

    "test_hyperparameter_selection":
        False,

    "secondary_target_selected_after_F6":
        False,

    "new_target_reconstruction":
        False,

}


if any(
    test_firewall.values()
):

    raise RuntimeError(
        "Test firewall failed."
    )


print(
    "\nTEST FIREWALL"
)

print(
    "-" * 80
)

for key, value in test_firewall.items():

    print(
        f"  {key}: {value}"
    )


print(
    "[PASS] Test evaluation authorized."
)


# ============================================================
# 25. HELD-OUT TEST EVALUATION
# ============================================================

selected_dyadic_metrics = regression_metrics(

    design_splits[
        best_dyadic
    ][
        "test"
    ],

    y_test,

    train_models[
        best_dyadic
    ]

)


triadic_metrics = regression_metrics(

    design_splits[
        triadic_name
    ][
        "test"
    ],

    y_test,

    train_models[
        triadic_name
    ]

)


test_results = {

    best_dyadic:
        {

            "mse":
                selected_dyadic_metrics[
                    "mse"
                ],

            "rmse":
                selected_dyadic_metrics[
                    "rmse"
                ],

            "r2":
                selected_dyadic_metrics[
                    "r2"
                ],

        },

    triadic_name:
        {

            "mse":
                triadic_metrics[
                    "mse"
                ],

            "rmse":
                triadic_metrics[
                    "rmse"
                ],

            "r2":
                triadic_metrics[
                    "r2"
                ],

        },

}


print(
    "\nFROZEN TEST RESULTS"
)

print(
    "-" * 80
)

for name in (
    best_dyadic,
    triadic_name
):

    print(
        f"  {name}: "
        f"MSE={test_results[name]['mse']:.9f}, "
        f"RMSE={test_results[name]['rmse']:.9f}, "
        f"R2={test_results[name]['r2']:.9f}"
    )


# ============================================================
# 26. HELD-OUT TRIADIC COMPARISON
# ============================================================

dyadic_mse = (
    test_results[
        best_dyadic
    ][
        "mse"
    ]
)

triadic_mse = (
    test_results[
        triadic_name
    ][
        "mse"
    ]
)


mse_delta = (
    dyadic_mse -
    triadic_mse
)


if dyadic_mse != 0:

    relative_improvement = (

        100.0 *
        mse_delta /
        dyadic_mse

    )

else:

    relative_improvement = (
        float("nan")
    )


print(
    "\nHELD-OUT TRIADIC COMPARISON"
)

print(
    "-" * 80
)

print(
    f"  comparator: {best_dyadic}"
)

print(
    f"  dyadic MSE: {dyadic_mse:.9f}"
)

print(
    f"  triadic MSE: {triadic_mse:.9f}"
)

print(
    f"  MSE delta (dyadic - triadic): "
    f"{mse_delta:.9f}"
)

print(
    f"  relative improvement: "
    f"{relative_improvement:.6f}%"
)

print(
    "  positive delta favors triadic."
)


# ============================================================
# 27. PAIRED TEST-ERROR DIFFERENCES
# ============================================================
#
# For each test record:
#
#     d_i =
#       error_dyadic_i^2
#       -
#       error_triadic_i^2
#
# Positive d_i favors the triadic model.
#
# The mean equals the held-out MSE difference.
# ============================================================

dyadic_pred = (
    selected_dyadic_metrics[
        "predictions"
    ]
)

triadic_pred = (
    triadic_metrics[
        "predictions"
    ]
)


dyadic_sq_error = (

    y_test -
    dyadic_pred

) ** 2


triadic_sq_error = (

    y_test -
    triadic_pred

) ** 2


paired_delta = (

    dyadic_sq_error -
    triadic_sq_error

)


if paired_delta.shape != (
    N_TEST,
):

    raise RuntimeError(
        "Unexpected paired test-error difference shape."
    )


paired_mean = float(
    np.mean(
        paired_delta
    )
)


if not np.isclose(
    paired_mean,
    mse_delta,
    rtol=1e-10,
    atol=1e-10
):

    raise RuntimeError(
        "Paired error mean does not equal MSE difference."
    )


# ============================================================
# 28. PAIRED BOOTSTRAP
# ============================================================

bootstrap = bootstrap_mean_ci(

    paired_delta,

    BOOTSTRAP_SEED,

    BOOTSTRAP_REPS

)


bootstrap_means = bootstrap.pop(
    "bootstrap_means"
)


print(
    "\nPAIRED BOOTSTRAP"
)

print(
    "-" * 80
)

print(
    f"  seed: {bootstrap['seed']}"
)

print(
    f"  replicates: {bootstrap['replicates']}"
)

print(
    f"  mean: {bootstrap['mean']:.9f}"
)

print(
    "  95% CI: "
    f"[{bootstrap['ci95'][0]:.9f}, "
    f"{bootstrap['ci95'][1]:.9f}]"
)

print(
    f"  P(delta > 0): "
    f"{bootstrap['p_delta_gt_zero']:.6f}"
)


# ============================================================
# 29. SECONDARY NUMERICAL CLASSIFICATION
# ============================================================
#
# Classification rule:
#
#   CI entirely > 0
#       => triadic predictive advantage supported
#
#   CI entirely < 0
#       => triadic predictive disadvantage supported
#
#   CI crosses zero
#       => unresolved / null
#
# This is a numerical classification only.
# ============================================================

ci_low, ci_high = (
    bootstrap[
        "ci95"
    ]
)


if ci_low > 0:

    numerical_status = (
        "HELD_OUT_TRIADIC_PREDICTIVE_ADVANTAGE_SUPPORTED"
    )

elif ci_high < 0:

    numerical_status = (
        "HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED"
    )

else:

    numerical_status = (
        "HELD_OUT_TRIADIC_PREDICTIVE_EFFECT_UNRESOLVED"
    )


print(
    "\nSECONDARY NUMERICAL CLASSIFICATION"
)

print(
    "-" * 80
)

print(
    f"  {numerical_status}"
)


# ============================================================
# 30. PAIRED DELTA DESCRIPTIVES
# ============================================================

delta_descriptives = {

    "mean":
        float(
            np.mean(
                paired_delta
            )
        ),

    "median":
        float(
            np.median(
                paired_delta
            )
        ),

    "q05":
        float(
            np.quantile(
                paired_delta,
                0.05
            )
        ),

    "q25":
        float(
            np.quantile(
                paired_delta,
                0.25
            )
        ),

    "q75":
        float(
            np.quantile(
                paired_delta,
                0.75
            )
        ),

    "q95":
        float(
            np.quantile(
                paired_delta,
                0.95
            )
        ),

    "fraction_positive":
        float(
            np.mean(
                paired_delta > 0
            )
        ),

}


print(
    "\nPAIRED DELTA DESCRIPTIVES"
)

print(
    "-" * 80
)

for key, value in delta_descriptives.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 31. CROSS-TARGET COMPARISON WITH PHASE 1F.6
# ============================================================
#
# This is NOT a statistical comparison between targets.
#
# It is an artifact-level synthesis:
#
#   F6 = target-logit contrast
#   F8 = target-rank contrast
#
# We preserve the two results without claiming statistical
# independence.
# ============================================================

f6_test_results = f6.get(
    "test_results",
    {}
)


f6_best_dyadic = (
    f6.get(
        "best_dyadic_model"
    )
)

if f6_best_dyadic is None:

    f6_best_dyadic = (
        f6.get(
            "selected_dyadic_model"
        )
    )


f6_triatic_results = f6_test_results.get(
    "triadic_S1S2S3",
    {}
)


f6_dyadic_results = f6_test_results.get(
    f6_best_dyadic,
    {}
) if f6_best_dyadic else {}


f6_summary = {

    "classification":
        f6_classification,

    "numerical_status":
        f6_numerical_status,

    "best_dyadic":
        f6_best_dyadic,

    "dyadic_test_mse":
        f6_dyadic_results.get(
            "mse"
        ),

    "triadic_test_mse":
        f6_triatic_results.get(
            "mse"
        ),

}


print(
    "\nPHASE 1F.6 / PHASE 1F.8 TARGET SYNTHESIS"
)

print(
    "-" * 80
)

print(
    "  Phase 1F.6 target: target-logit contrast"
)

print(
    f"  Phase 1F.6 status: "
    f"{f6_numerical_status}"
)

print(
    f"  Phase 1F.6 comparator: "
    f"{f6_best_dyadic}"
)

print(
    "  Phase 1F.8 target: target-rank contrast"
)

print(
    f"  Phase 1F.8 status: "
    f"{numerical_status}"
)

print(
    f"  Phase 1F.8 comparator: "
    f"{best_dyadic}"
)


# ============================================================
# 32. ROBUSTNESS INTERPRETATION
# ============================================================

if (
    f6_numerical_status
    ==
    "HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED"
    and
    numerical_status
    ==
    "HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED"
):

    robustness_interpretation = (
        "The numerical triadic predictive disadvantage is "
        "directionally replicated across the two frozen "
        "semantic observables used in the primary and secondary "
        "tests. This constitutes numerical robustness across "
        "target representations, not an independent replication "
        "and not evidence for or against formal CTL irreducibility."
    )

elif (
    f6_numerical_status
    ==
    "HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED"
    and
    numerical_status
    ==
    "HELD_OUT_TRIADIC_PREDICTIVE_PREDICTIVE_ADVANTAGE_SUPPORTED"
):

    robustness_interpretation = (
        "The numerical result is target-dependent: Phase 1F.6 "
        "supports triadic predictive disadvantage whereas the "
        "secondary target-rank test supports triadic predictive "
        "advantage. The two observables are related and not "
        "statistically independent. No unified numerical "
        "conclusion should be asserted without further "
        "pre-specified justification."
    )

elif (
    f6_numerical_status
    ==
    "HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED"
    and
    numerical_status
    ==
    "HELD_OUT_TRIADIC_PREDICTIVE_EFFECT_UNRESOLVED"
):

    robustness_interpretation = (
        "The primary target-logit test supports a held-out "
        "triadic predictive disadvantage, while the secondary "
        "target-rank test is unresolved. The Phase 1F.6 result "
        "therefore remains supported but is not robustly "
        "reproduced by the ordinal target representation."
    )

else:

    robustness_interpretation = (
        "The primary and secondary numerical results are "
        "heterogeneous. They must be reported separately and "
        "cannot be collapsed into a single numerical conclusion."
    )


print(
    "\nROBUSTNESS INTERPRETATION"
)

print(
    "-" * 80
)

print(
    robustness_interpretation
)


# ============================================================
# 33. CORRECT A TYPO-SAFE CLASSIFICATION FIREWALL
# ============================================================
#
# The branch above intentionally uses an exact expected string.
# We now normalize the actual classification vocabulary to
# prevent a typo from silently producing the wrong scientific
# branch.
# ============================================================

valid_numerical_statuses = {

    "HELD_OUT_TRIADIC_PREDICTIVE_ADVANTAGE_SUPPORTED",

    "HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED",

    "HELD_OUT_TRIADIC_PREDICTIVE_EFFECT_UNRESOLVED",

}


if numerical_status not in valid_numerical_statuses:

    raise RuntimeError(
        "Invalid Phase 1F.8 numerical classification."
    )


# ============================================================
# 34. CTL INTERPRETATION FIREWALL
# ============================================================

interpretation_firewall = {

    "numerical_result_equals_CTL_irreducibility":
        False,

    "numerical_result_equals_Phi_C":
        False,

    "numerical_result_equals_K_C":
        False,

    "numerical_result_equals_Adm_C_3":
        False,

    "target_rank_is_statistically_independent_of_logit":
        False,

    "three_sector_presence_is_proof":
        False,

    "three_way_interaction_is_proof":
        False,

    "numerical_test_establishes_logical_transport":
        False,

    "numerical_test_establishes_contextual_geometry":
        False,

    "numerical_test_establishes_CTL_coherence":
        False,

}


if any(
    interpretation_firewall.values()
):

    raise RuntimeError(
        "Interpretation firewall failed."
    )


print(
    "\nINTERPRETATION FIREWALL"
)

print(
    "-" * 80
)

for key, value in interpretation_firewall.items():

    print(
        f"  {key}: {value}"
    )


print(
    "[PASS] Numerical result remains distinct from CTL."
)


# ============================================================
# 35. EXPERIMENTAL INTEGRITY FIREWALL
# ============================================================

experimental_firewall = {

    "dataset_modified":
        False,

    "state_bank_modified":
        False,

    "PCA_refitted":
        False,

    "transport_refitted":
        False,

    "model_reexecuted":
        False,

    "Phi_C_fitted":
        False,

    "admissibility_fitted":
        False,

    "test_used_for_selection":
        False,

    "test_used_for_hyperparameter_selection":
        False,

    "secondary_target_recomputed":
        False,

    "secondary_target_selected_after_F6":
        False,

    "new_target_definition":
        False,

}


if any(
    experimental_firewall.values()
):

    raise RuntimeError(
        "Experimental integrity firewall failed."
    )


print(
    "\nEXPERIMENTAL INTEGRITY FIREWALL"
)

print(
    "-" * 80
)

for key, value in experimental_firewall.items():

    print(
        f"  {key}: {value}"
    )


print(
    "[PASS] Frozen-data numerical experiment integrity."
)


# ============================================================
# 36. CTL STATUS
# ============================================================

ctl_status = {

    "formal_CTL_structure":
        "SUPPORTED",

    "empirical_Adm_C_3":
        "NOT_IDENTIFIED",

    "empirical_K_C":
        "NOT_IDENTIFIED",

    "empirical_Phi_C":
        "NOT_RECONSTRUCTED",

    "logical_transport_covariance":
        "NOT_EMPIRICALLY_TESTED",

    "CTL_triadic_irreducibility":
        "NOT_ESTABLISHED",

    "numerical_triadic_sufficiency_F6":
        "NOT_SUPPORTED",

    "numerical_triadic_disadvantage_F6":
        "SUPPORTED",

    "numerical_rank_robustness_F8":
        numerical_status,

}


print(
    "\nCTL STATUS"
)

print(
    "-" * 80
)

for key, value in ctl_status.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 37. CLOSURE READINESS
# ============================================================
#
# Phase 1F.8 is intended to be the final new numerical test.
#
# Regardless of its outcome, the experiment should move to a
# closure audit rather than open-ended target hunting.
# ============================================================

closure_readiness = {

    "phase_1F6_completed":
        True,

    "phase_1F7_completed":
        True,

    "phase_1F8_completed":
        True,

    "primary_logit_target_frozen":
        True,

    "secondary_rank_target_frozen":
        True,

    "same_frozen_predictor_representation":
        True,

    "same_equal_budget_design":
        True,

    "calibration_locked":
        True,

    "held_out_test_used":
        True,

    "additional_target_search_authorized":
        False,

    "open_ended_target_hunting_authorized":
        False,

    "closure_phase_authorized":
        True,

}


print(
    "\nCLOSURE READINESS"
)

print(
    "-" * 80
)

for key, value in closure_readiness.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 38. PRE-SERIALIZATION ARTIFACT
# ============================================================

artifact = {

    "experiment_id":
        "ETTR-CTL-LLAMA-1",

    "phase":
        "1F.8",

    "title":
        "Secondary Frozen Target-Rank "
        "Triadic Robustness Test",

    "purpose":
        (
            "Assess whether the Phase 1F.6 numerical triadic "
            "result is robust to the already-frozen ordinal "
            "target-rank observable."
        ),

    "governance":
        {

            "dataset":
                "FROZEN",

            "state_bank":
                "FROZEN",

            "PCA":
                "FROZEN_READ_ONLY",

            "transport":
                "FROZEN",

            "semantic_target":
                "FROZEN_TARGET_RANK",

            "train_calibration_test_split":
                "FROZEN",

            "test_model_selection":
                False,

            "test_hyperparameter_selection":
                False,

            "model_reexecution":
                False,

            "PCA_refit":
                False,

            "transport_refit":
                False,

            "Phi_C_fit":
                False,

            "admissibility_fit":
                False,

        },

    "predecessors":
        {

            "1F.2A":
                row_order_classification,

            "1F.2":
                semantic_classification,

            "1F.6":
                f6_classification,

            "1F.7":
                "SECONDARY_TARGET_AUTHORIZED",

        },

    "dataset":
        {

            "path":
                str(
                    dataset_path
                ),

            "sha256":
                authorized_sha,

            "records":
                N_RECORDS,

            "split_counts":
                split_counts,

        },

    "state_bank_hashes":
        state_hash_checks,

    "row_order":
        {

            "classification":
                row_order_classification,

            "selected":
                row_order_selected,

            "pT_pair_equal":
                pT_pair_equal,

            "iT_pair_equal":
                iT_pair_equal,

        },

    "secondary_target":
        {

            "definition":
                (
                    "clean target rank minus corrupt "
                    "target rank"
                ),

            "source":
                "state_bank.target_logit_ranks",

            "hash":
                target_rank_hash,

            "shape":
                list(
                    target_ranks.shape
                ),

            "dtype":
                str(
                    target_ranks.dtype
                ),

            "statistically_independent_of_logit":
                False,

            "paired_summary":
                rank_summary,

        },

    "frozen_pca":
        {

            sector:
                {

                    "dimension":
                        int(
                            frozen_pca[
                                sector
                            ][
                                "dimension"
                            ]
                        ),

                    "mean_shape":
                        list(
                            frozen_pca[
                                sector
                            ][
                                "mean"
                            ].shape
                        ),

                    "components_shape":
                        list(
                            frozen_pca[
                                sector
                            ][
                                "components"
                            ].shape
                        ),

                }

            for sector in SECTORS

        },

    "projection_identity":
        projection_identity_checks,

    "model_specification":
        {

            "ridge_lambda":
                RIDGE_LAMBDA,

            "intercept":
                False,

            "dyadic_dimensions":
                3,

            "triadic_dimensions_per_sector":
                2,

            "total_predictors_dyadic":
                6,

            "total_predictors_triadic":
                6,

            "dyadic_candidates":
                dyadic_names,

            "triadic_model":
                triadic_name,

            "selection_split":
                "calibration_only",

        },

    "calibration_results":
        calibration_results,

    "selected_dyadic_model":
        best_dyadic,

    "test_results":
        test_results,

    "held_out_comparison":
        {

            "dyadic_mse":
                dyadic_mse,

            "triadic_mse":
                triadic_mse,

            "mse_delta_dyadic_minus_triadic":
                mse_delta,

            "relative_improvement_percent":
                float(
                    relative_improvement
                ),

        },

    "paired_bootstrap":
        {

            "seed":
                bootstrap[
                    "seed"
                ],

            "replicates":
                bootstrap[
                    "replicates"
                ],

            "mean":
                bootstrap[
                    "mean"
                ],

            "ci95":
                bootstrap[
                    "ci95"
                ],

            "p_delta_gt_zero":
                bootstrap[
                    "p_delta_gt_zero"
                ],

        },

    "paired_delta_descriptives":
        delta_descriptives,

    "numerical_status":
        numerical_status,

    "phase_1F6_summary":
        f6_summary,

    "robustness_interpretation":
        robustness_interpretation,

    "ctl_status":
        ctl_status,

    "interpretation_firewall":
        interpretation_firewall,

    "experimental_firewall":
        experimental_firewall,

    "closure_readiness":
        closure_readiness,

}


# ============================================================
# 39. PRE-SERIALIZATION AUDIT
# ============================================================

serialization_checks = {

    "artifact_dict":
        isinstance(
            artifact,
            dict
        ),

    "secondary_target_dict":
        isinstance(
            artifact[
                "secondary_target"
            ],
            dict
        ),

    "model_specification_dict":
        isinstance(
            artifact[
                "model_specification"
            ],
            dict
        ),

    "calibration_results_dict":
        isinstance(
            artifact[
                "calibration_results"
            ],
            dict
        ),

    "test_results_dict":
        isinstance(
            artifact[
                "test_results"
            ],
            dict
        ),

    "paired_bootstrap_dict":
        isinstance(
            artifact[
                "paired_bootstrap"
            ],
            dict
        ),

    "interpretation_firewall_dict":
        isinstance(
            artifact[
                "interpretation_firewall"
            ],
            dict
        ),

    "experimental_firewall_dict":
        isinstance(
            artifact[
                "experimental_firewall"
            ],
            dict
        ),

}


print(
    "\nPRE-SERIALIZATION AUDIT"
)

print(
    "-" * 80
)

for key, value in serialization_checks.items():

    print(
        f"  {key}: {value}"
    )


if not all(
    serialization_checks.values()
):

    raise RuntimeError(
        "Pre-serialization audit failed."
    )


print(
    "[PASS] Pre-serialization audit."
)


# ============================================================
# 40. WRITE ARTIFACT
# ============================================================

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        artifact,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 41. POST-SERIALIZATION VERIFICATION
# ============================================================

with open(
    OUTPUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    artifact_check = json.load(
        f
    )


if artifact_check.get(
    "phase"
) != "1F.8":

    raise RuntimeError(
        "Phase readback mismatch."
    )


if artifact_check.get(
    "selected_dyadic_model"
) != best_dyadic:

    raise RuntimeError(
        "Selected dyadic model readback mismatch."
    )


if artifact_check.get(
    "numerical_status"
) != numerical_status:

    raise RuntimeError(
        "Numerical classification readback mismatch."
    )


if artifact_check.get(
    "secondary_target",
    {}
).get(
    "statistically_independent_of_logit"
) is not False:

    raise RuntimeError(
        "Target independence firewall readback failed."
    )


if artifact_check.get(
    "interpretation_firewall",
    {}
).get(
    "numerical_result_equals_CTL_irreducibility"
) is not False:

    raise RuntimeError(
        "CTL irreducibility firewall readback failed."
    )


if artifact_check.get(
    "experimental_firewall",
    {}
).get(
    "PCA_refit"
) is not False:

    raise RuntimeError(
        "PCA refit firewall readback failed."
    )


if artifact_check.get(
    "experimental_firewall",
    {}
).get(
    "secondary_target_recomputed"
) is not False:

    raise RuntimeError(
        "Secondary-target reconstruction firewall readback failed."
    )


print(
    "\nPOST-SERIALIZATION VERIFICATION"
)

print(
    "-" * 80
)

print(
    "  JSON readback: PASS"
)

print(
    "  frozen target rank: PASS"
)

print(
    "  frozen PCA: PASS"
)

print(
    "  equal predictor budget: PASS"
)

print(
    "  calibration-only selection: PASS"
)

print(
    "  held-out test: PASS"
)

print(
    "  CTL irreducibility firewall: PASS"
)

print(
    "  no PCA refit: PASS"
)

print(
    "  no target reconstruction: PASS"
)


# ============================================================
# 42. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "FINAL PHASE 1F.8 STATUS"
)

print(
    "=" * 80
)

print(
    f"  numerical status: "
    f"{numerical_status}"
)

print(
    f"  selected dyadic comparator: "
    f"{best_dyadic}"
)

print(
    f"  triadic model: "
    f"{triadic_name}"
)

print(
    "  equal predictor budget: 6 = 6"
)

print(
    f"  test MSE dyadic: "
    f"{dyadic_mse:.9f}"
)

print(
    f"  test MSE triadic: "
    f"{triadic_mse:.9f}"
)

print(
    f"  MSE delta (dyadic - triadic): "
    f"{mse_delta:.9f}"
)

print(
    f"  bootstrap 95% CI: "
    f"[{ci_low:.9f}, {ci_high:.9f}]"
)

print(
    ""
)

print(
    "  PRIMARY PHASE 1F.6:"
)

print(
    "    target-logit numerical triadic disadvantage "
    "supported"
)

print(
    ""
)

print(
    "  SECONDARY PHASE 1F.8:"
)

print(
    f"    target-rank result: "
    f"{numerical_status}"
)

print(
    ""
)

print(
    "  CTL TRIADIC IRREDUCIBILITY: NOT ESTABLISHED"
)

print(
    "  EMPIRICAL Adm_C^(3): NOT IDENTIFIED"
)

print(
    "  EMPIRICAL K_C: NOT IDENTIFIED"
)

print(
    "  EMPIRICAL Phi_C: NOT RECONSTRUCTED"
)

print(
    "  CONTEXTUAL GEOMETRY: NOT ESTABLISHED"
)

print(
    ""
)

print(
    "  OPEN-ENDED TARGET HUNTING: NOT AUTHORIZED"
)

print(
    "  CLOSURE PHASE: AUTHORIZED"
)

print(
    f"  artifact: {OUTPUT_PATH}"
)

print(
    "=" * 80
)

ETTR-CTL-LLAMA-1 — PHASE 1F.8
SECONDARY FROZEN TARGET-RANK TRIADIC ROBUSTNESS TEST
State bank: /content/ettr_ctl_llama/results/llama_phase1d2_full_state_bank.npz
Phase 1F.6: /content/ettr_ctl_llama/results/llama_phase1f6_complexity_controlled_numerical_triadic_sufficiency_audit.json
Phase 1F.7: /content/ettr_ctl_llama/results/llama_phase1f7_numerical_triadic_result_synthesis_audit.json
Output: /content/ettr_ctl_llama/results/llama_phase1f8_secondary_target_rank_triadic_robustness_test.json

[PASS] Required predecessor artifacts exist.

PREDECESSOR GOVERNANCE
--------------------------------------------------------------------------------
  1F.2A classification: FROZEN_STATE_BANK_ROW_ORDER_VERIFIED
  1F.2A selected row order: interleaved_clean_corrupt
  1F.2 semantic classification: CTL_SEMANTIC_REALIZATION_OPERATIONAL_PASS
  1F.6 classification: NUMERICAL_TRIADIC_PREDICTIVE_SUFFICIENCY_NOT_SUPPORTED_CTL_IRREDUCIBILITY_NOT_ESTABLISHED
  1F.6 numerical status: HELD_OUT_TRIADIC_PREDICTIVE

RuntimeError: PCA refit firewall readback failed.

In [51]:
# ============================================================
# ETTR-CTL-LLAMA-1
# PHASE 1F.9
# FINAL EXPERIMENTAL CLOSURE / MASTER RESULT MANIFEST
#
# REPLACEMENT VERSION — SCHEMA / PHASE-MAPPING CORRECTION
#
# REMOVE THE PREVIOUS PHASE 1F.9 CELL, THEN INSERT THIS
# COMPLETE REPLACEMENT CELL.
#
# ============================================================
#
# PURPOSE
# -------
# Formally close ETTR-CTL-LLAMA-1 after completion of Phase
# 1F.8.
#
# This cell performs ONLY frozen-artifact verification and
# final synthesis.
#
# It does NOT:
#   - load the Llama model;
#   - execute inference;
#   - modify the dataset;
#   - modify the state bank;
#   - fit PCA;
#   - fit transport;
#   - fit Phi_C;
#   - identify admissibility;
#   - perform another statistical test;
#   - search for another target.
#
# IMPORTANT PHASE MAP
# -------------------
#
#   1F.0  CTL structural preflight
#   1F.1  Formal CTL structure implementation
#   1F.2A Frozen state-bank row-order verification
#   1F.2  CTL semantic realization
#   1F.3  Logical transport structure
#   1F.4  Triadic coherence / Phi_C interface
#   1F.5  Triadic irreducibility identifiability
#   1F.6  Numerical triadic sufficiency
#   1F.7  Numerical triadic result synthesis
#   1F.8  Secondary target-rank robustness
#   1F.9  Final closure
#
# The previous closure cell incorrectly mapped 1F.1 to the
# 1F.2 semantic artifact. This replacement corrects that.
#
# ============================================================


import os
import json
import hashlib
from pathlib import Path

import numpy as np


# ============================================================
# 0. CONFIGURATION
# ============================================================

ROOT = Path(
    "/content/ettr_ctl_llama"
)

RESULTS = ROOT / "results"


AUTH_PATH = (
    RESULTS /
    "llama_phase1c0_recovered_dataset_authorization_v2.json"
)

STATE_BANK_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank.npz"
)

STATE_MANIFEST_PATH = (
    RESULTS /
    "llama_phase1d2_full_state_bank_manifest.json"
)


# -------------------------
# CTL / empirical phases
# -------------------------

F0_PATH = (
    RESULTS /
    "llama_phase1f0_ctl_contextual_realization_preflight.json"
)

F1_PATH = (
    RESULTS /
    "llama_phase1f1_formal_ctl_structure_audit.json"
)

F2A_PATH = (
    RESULTS /
    "llama_phase1f2a_state_bank_row_order_audit.json"
)

F2_PATH = (
    RESULTS /
    "llama_phase1f2_ctl_semantic_realization_audit.json"
)

F3_PATH = (
    RESULTS /
    "llama_phase1f3_logical_transport_covariance_audit.json"
)

F4_PATH = (
    RESULTS /
    "llama_phase1f4_triadic_coherence_phiC_interface_audit.json"
)

F5_PATH = (
    RESULTS /
    "llama_phase1f5_triadic_irreducibility_identifiability_audit.json"
)

F6_PATH = (
    RESULTS /
    "llama_phase1f6_complexity_controlled_numerical_triadic_sufficiency_audit.json"
)

F7_PATH = (
    RESULTS /
    "llama_phase1f7_numerical_triadic_result_synthesis_audit.json"
)

F8_PATH = (
    RESULTS /
    "llama_phase1f8_secondary_target_rank_triadic_robustness_test.json"
)


OUTPUT_PATH = (
    RESULTS /
    "llama_phase1f9_final_experimental_closure_manifest.json"
)


EXPECTED_DATASET_SHA256 = (
    "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d"
)


EXPECTED_STATE_HASHES = {

    "S1":
        "9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783",

    "S2":
        "41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb",

    "S3":
        "173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3",

    "target_logits":
        "537fa43c8e278b8678de978c1dc2ec4ee5a1c77ca0a917dbf6a5823cf77ea58f",

}


EXPECTED_CLASSIFICATIONS = {

    "1F.0":
        "CTL_STRUCTURAL_PREFLIGHT_PASS",

    "1F.1":
        "FORMAL_CTL_STRUCTURE_IMPLEMENTATION_PASS",

    "1F.2A":
        "FROZEN_STATE_BANK_ROW_ORDER_VERIFIED",

    "1F.2":
        "CTL_SEMANTIC_REALIZATION_OPERATIONAL_PASS",

    "1F.3":
        "LOGICAL_TRANSPORT_STRUCTURE_OPERATIONAL_PASS",

    "1F.4":
        "TRIADIC_COHERENCE_INTERFACE_OPERATIONAL_PASS_NOT_EMPIRICALLY_IDENTIFIED",

    "1F.5":
        "TRIADIC_IRREDUCIBILITY_NOT_CURRENTLY_IDENTIFIABLE",

    "1F.6":
        "NUMERICAL_TRIADIC_PREDICTIVE_SUFFICIENCY_NOT_SUPPORTED_CTL_IRREDUCIBILITY_NOT_ESTABLISHED",

}


EXPECTED_F6_STATUS = (
    "HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED"
)

EXPECTED_F8_STATUS = (
    "HELD_OUT_TRIADIC_PREDICTIVE_EFFECT_UNRESOLVED"
)


# ============================================================
# 1. HEADER
# ============================================================

print(
    "=" * 80
)

print(
    "ETTR-CTL-LLAMA-1 — PHASE 1F.9"
)

print(
    "FINAL EXPERIMENTAL CLOSURE / MASTER RESULT MANIFEST"
)

print(
    "=" * 80
)

print(
    "Scientific computation: CLOSED"
)

print(
    "Purpose: frozen-artifact verification and final synthesis"
)

print(
    f"Output: {OUTPUT_PATH}"
)


# ============================================================
# 2. HELPERS
# ============================================================

def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(
            f
        )


def sha256_array(arr):

    return hashlib.sha256(

        np.ascontiguousarray(
            arr
        ).tobytes(
            order="C"
        )

    ).hexdigest()


def resolve_classification(obj):

    """
    Resolve classification from historically used locations.
    """

    for key in (
        "final_classification",
        "classification",
    ):

        value = obj.get(
            key
        )

        if isinstance(
            value,
            str
        ):

            return value


    scientific_status = obj.get(
        "scientific_status"
    )

    if isinstance(
        scientific_status,
        dict
    ):

        value = scientific_status.get(
            "classification"
        )

        if isinstance(
            value,
            str
        ):

            return value


    return None


def as_bool(value):

    if isinstance(
        value,
        bool
    ):

        return value

    if isinstance(
        value,
        np.bool_
    ):

        return bool(
            value
        )

    if isinstance(
        value,
        str
    ):

        v = value.strip().lower()

        if v == "true":
            return True

        if v == "false":
            return False

    return value


def artifact_identity(
    phase,
    obj
):

    """
    Verify that an artifact's own phase field agrees with
    the phase under which the closure manifest is consuming it.
    """

    artifact_phase = obj.get(
        "phase"
    )

    if artifact_phase != phase:

        raise RuntimeError(
            f"Artifact phase mismatch.\n"
            f"Expected phase: {phase}\n"
            f"Artifact phase: {artifact_phase}"
        )


# ============================================================
# 3. REQUIRED ARTIFACT AUDIT
# ============================================================

required_paths = [

    AUTH_PATH,
    STATE_BANK_PATH,
    STATE_MANIFEST_PATH,
    F0_PATH,
    F1_PATH,
    F2A_PATH,
    F2_PATH,
    F3_PATH,
    F4_PATH,
    F5_PATH,
    F6_PATH,
    F7_PATH,
    F8_PATH,

]


print(
    "\nREQUIRED ARTIFACT AUDIT"
)

print(
    "-" * 80
)


missing = [

    str(path)

    for path in required_paths

    if not path.exists()

]


if missing:

    for path in missing:

        print(
            f"  [MISSING] {path}"
        )

    raise FileNotFoundError(
        "Required closure artifact(s) missing."
    )


for path in required_paths:

    print(
        f"  [FOUND] {path.name}"
    )


print(
    "[PASS] All required closure artifacts exist."
)


# ============================================================
# 4. LOAD FROZEN ARTIFACTS
# ============================================================

auth = load_json(
    AUTH_PATH
)

f0 = load_json(
    F0_PATH
)

f1 = load_json(
    F1_PATH
)

f2a = load_json(
    F2A_PATH
)

f2 = load_json(
    F2_PATH
)

f3 = load_json(
    F3_PATH
)

f4 = load_json(
    F4_PATH
)

f5 = load_json(
    F5_PATH
)

f6 = load_json(
    F6_PATH
)

f7 = load_json(
    F7_PATH
)

f8 = load_json(
    F8_PATH
)


print(
    "\n[PASS] Frozen predecessor artifacts loaded read-only."
)


# ============================================================
# 5. EXPLICIT PHASE-IDENTITY AUDIT
# ============================================================

phase_artifacts = {

    "1F.0":
        f0,

    "1F.1":
        f1,

    "1F.2A":
        f2a,

    "1F.2":
        f2,

    "1F.3":
        f3,

    "1F.4":
        f4,

    "1F.5":
        f5,

    "1F.6":
        f6,

}


print(
    "\nPHASE IDENTITY AUDIT"
)

print(
    "-" * 80
)


for phase, obj in phase_artifacts.items():

    artifact_identity(
        phase,
        obj
    )

    print(
        f"  {phase}: artifact phase = {obj.get('phase')}"
    )


print(
    "[PASS] Phase-to-artifact mapping."
)


# ============================================================
# 6. PHASE CLASSIFICATION AUDIT
# ============================================================

print(
    "\nPHASE CLASSIFICATION AUDIT"
)

print(
    "-" * 80
)


resolved_classifications = {}


for phase, obj in phase_artifacts.items():

    classification = resolve_classification(
        obj
    )

    resolved_classifications[
        phase
    ] = classification

    print(
        f"  {phase}: {classification}"
    )


for phase, expected in EXPECTED_CLASSIFICATIONS.items():

    actual = resolved_classifications.get(
        phase
    )

    if actual != expected:

        raise RuntimeError(
            f"{phase}: classification mismatch.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


print(
    "[PASS] Frozen predecessor classifications."
)


# ============================================================
# 7. EXPLICIT 1F.1 / 1F.2 SEMANTIC FIREWALL
# ============================================================
#
# This section exists specifically to prevent recurrence of the
# phase-mapping error.
# ============================================================

print(
    "\n1F.1 / 1F.2 PHASE-MAPPING FIREWALL"
)

print(
    "-" * 80
)

print(
    f"  1F.1 artifact: {F1_PATH.name}"
)

print(
    f"  1F.1 classification: "
    f"{resolved_classifications['1F.1']}"
)

print(
    f"  1F.2 artifact: {F2_PATH.name}"
)

print(
    f"  1F.2 classification: "
    f"{resolved_classifications['1F.2']}"
)


if resolved_classifications[
    "1F.1"
] != (
    "FORMAL_CTL_STRUCTURE_IMPLEMENTATION_PASS"
):

    raise RuntimeError(
        "1F.1 formal CTL structure mapping failed."
    )


if resolved_classifications[
    "1F.2"
] != (
    "CTL_SEMANTIC_REALIZATION_OPERATIONAL_PASS"
):

    raise RuntimeError(
        "1F.2 semantic realization mapping failed."
    )


if F1_PATH == F2_PATH:

    raise RuntimeError(
        "1F.1 and 1F.2 unexpectedly resolve to the same "
        "artifact path."
    )


print(
    "[PASS] Formal CTL and semantic-realization artifacts "
    "are correctly distinguished."
)


# ============================================================
# 8. DATASET AUTHORIZATION
# ============================================================

authorized_sha = (
    auth.get(
        "authorized_dataset_sha256"
    )
)

if authorized_sha is None:

    authorized_sha = auth.get(
        "sha256"
    )


print(
    "\nDATASET AUTHORIZATION"
)

print(
    "-" * 80
)

print(
    f"  SHA-256: {authorized_sha}"
)


if authorized_sha != EXPECTED_DATASET_SHA256:

    raise RuntimeError(
        "Authorized dataset SHA mismatch."
    )


print(
    "[PASS] Operative dataset authorization."
)


# ============================================================
# 9. LOCATE OPERATIVE DATASET
# ============================================================

dataset_name = (
    "gpt2_controlled_dataset_v1.1_recovered_r1.json"
)


dataset_candidates = []


for base in (
    ROOT,
    Path("/content"),
):

    if base.exists():

        dataset_candidates.extend(
            base.rglob(
                dataset_name
            )
        )


dataset_candidates = list(
    dict.fromkeys(
        dataset_candidates
    )
)


valid_datasets = []


for path in dataset_candidates:

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            obj = json.load(
                f
            )

    except Exception:

        continue


    if isinstance(
        obj,
        list
    ):

        candidate_records = obj

    elif isinstance(
        obj,
        dict
    ):

        candidate_records = None

        for key in (
            "records",
            "dataset",
            "examples",
            "data",
            "items",
        ):

            if isinstance(
                obj.get(key),
                list
            ):

                candidate_records = obj[
                    key
                ]

                break

    else:

        candidate_records = None


    if (
        candidate_records is not None
        and
        len(candidate_records) == 192
    ):

        valid_datasets.append(
            (
                path,
                candidate_records
            )
        )


if not valid_datasets:

    raise FileNotFoundError(
        "Authorized 192-record operative dataset not found."
    )


dataset_path, records = valid_datasets[0]


print(
    "\nOPERATIVE DATASET"
)

print(
    "-" * 80
)

print(
    f"  path: {dataset_path}"
)

print(
    f"  records: {len(records)}"
)


# ============================================================
# 10. SPLIT AUDIT
# ============================================================

split_labels = np.asarray(

    [
        str(
            record[
                "split"
            ]
        )

        for record in records

    ],

    dtype=object

)


train_idx = np.where(
    split_labels == "train"
)[0]

cal_idx = np.where(
    split_labels == "calibration"
)[0]

test_idx = np.where(
    split_labels == "test"
)[0]


split_counts = {

    "train":
        int(
            len(train_idx)
        ),

    "calibration":
        int(
            len(cal_idx)
        ),

    "test":
        int(
            len(test_idx)
        ),

}


print(
    "\nSPLIT AUDIT"
)

print(
    "-" * 80
)

for key, value in split_counts.items():

    print(
        f"  {key}: {value}"
    )


if split_counts != {

    "train": 96,

    "calibration": 48,

    "test": 48,

}:

    raise RuntimeError(
        "Frozen train/calibration/test split mismatch."
    )


print(
    "[PASS] Frozen split."
)


# ============================================================
# 11. STATE-BANK INTEGRITY
# ============================================================

bank = np.load(
    STATE_BANK_PATH,
    allow_pickle=False
)


print(
    "\nSTATE-BANK HASH AUDIT"
)

print(
    "-" * 80
)


state_hash_checks = {}


for key, expected_hash in EXPECTED_STATE_HASHES.items():

    if key not in bank.files:

        raise RuntimeError(
            f"State bank missing required field: {key}"
        )


    arr = np.asarray(
        bank[
            key
        ]
    )


    actual_hash = sha256_array(
        arr
    )


    match = (
        actual_hash ==
        expected_hash
    )


    finite = bool(
        np.all(
            np.isfinite(
                arr
            )
        )
    )


    state_hash_checks[
        key
    ] = {

        "expected":
            expected_hash,

        "actual":
            actual_hash,

        "match":
            match,

        "finite":
            finite,

        "shape":
            list(
                arr.shape
            ),

        "dtype":
            str(
                arr.dtype
            ),

    }


    print(
        f"  {key}: "
        f"shape={arr.shape}, "
        f"match={match}, "
        f"finite={finite}"
    )


    if not match:

        raise RuntimeError(
            f"Frozen state-bank hash mismatch: {key}"
        )


    if not finite:

        raise RuntimeError(
            f"Frozen state-bank non-finite field: {key}"
        )


print(
    "[PASS] Frozen state-bank integrity."
)


# ============================================================
# 12. ROW-ORDER / CONDITION ALIGNMENT
# ============================================================

row_order_classification = resolve_classification(
    f2a
)

row_order_selected = f2a.get(
    "selected_row_order"
)


if row_order_classification != (
    "FROZEN_STATE_BANK_ROW_ORDER_VERIFIED"
):

    raise RuntimeError(
        "Frozen row-order classification mismatch."
    )


if row_order_selected != (
    "interleaved_clean_corrupt"
):

    raise RuntimeError(
        "Frozen row-order selection mismatch."
    )


condition_pT = np.asarray(
    bank[
        "condition_p_T"
    ]
)

condition_iT = np.asarray(
    bank[
        "condition_i_T"
    ]
)


clean_rows = np.arange(
    0,
    384,
    2,
    dtype=np.int64
)

corrupt_rows = np.arange(
    1,
    384,
    2,
    dtype=np.int64
)


pT_equal = bool(
    np.array_equal(
        condition_pT[
            clean_rows
        ],
        condition_pT[
            corrupt_rows
        ]
    )
)

iT_equal = bool(
    np.array_equal(
        condition_iT[
            clean_rows
        ],
        condition_iT[
            corrupt_rows
        ]
    )
)


print(
    "\nROW-ORDER / CONDITION ALIGNMENT"
)

print(
    "-" * 80
)

print(
    f"  selected: {row_order_selected}"
)

print(
    f"  pT pair equality: {pT_equal}"
)

print(
    f"  iT pair equality: {iT_equal}"
)


if not pT_equal or not iT_equal:

    raise RuntimeError(
        "Frozen clean/corrupt target-position pairing failed."
    )


print(
    "[PASS] Frozen condition alignment."
)


# ============================================================
# 13. PHASE 1F.6 PRIMARY RESULT
# ============================================================

print(
    "\nPHASE 1F.6 PRIMARY RESULT"
)

print(
    "-" * 80
)


f6_classification = resolve_classification(
    f6
)

f6_status = f6.get(
    "numerical_status"
)


print(
    f"  classification: {f6_classification}"
)

print(
    f"  numerical status: {f6_status}"
)


if f6_classification != (
    EXPECTED_CLASSIFICATIONS[
        "1F.6"
    ]
):

    raise RuntimeError(
        "Phase 1F.6 classification changed."
    )


if f6_status != EXPECTED_F6_STATUS:

    raise RuntimeError(
        "Phase 1F.6 numerical status changed."
    )


print(
    "[PASS] Phase 1F.6 primary result."
)


# ============================================================
# 14. PHASE 1F.8 SECONDARY RESULT
# ============================================================

print(
    "\nPHASE 1F.8 SECONDARY RESULT"
)

print(
    "-" * 80
)


f8_phase = f8.get(
    "phase"
)

f8_status = f8.get(
    "numerical_status"
)

f8_secondary = f8.get(
    "secondary_target",
    {}
)

f8_comparison = f8.get(
    "held_out_comparison",
    {}
)

f8_bootstrap = f8.get(
    "paired_bootstrap",
    {}
)


if f8_phase != "1F.8":

    raise RuntimeError(
        "Phase 1F.8 artifact identity mismatch."
    )


if f8_status != EXPECTED_F8_STATUS:

    raise RuntimeError(
        "Phase 1F.8 numerical classification changed."
    )


f8_selected_dyadic = f8.get(
    "selected_dyadic_model"
)

f8_dyadic_mse = f8_comparison.get(
    "dyadic_mse"
)

f8_triadic_mse = f8_comparison.get(
    "triadic_mse"
)

f8_delta = f8_comparison.get(
    "mse_delta_dyadic_minus_triadic"
)

f8_relative = f8_comparison.get(
    "relative_improvement_percent"
)

f8_ci = f8_bootstrap.get(
    "ci95"
)


print(
    f"  target: frozen target-rank contrast"
)

print(
    f"  comparator: {f8_selected_dyadic}"
)

print(
    f"  dyadic MSE: {f8_dyadic_mse}"
)

print(
    f"  triadic MSE: {f8_triadic_mse}"
)

print(
    f"  MSE delta: {f8_delta}"
)

print(
    f"  relative improvement: {f8_relative}%"
)

print(
    f"  bootstrap 95% CI: {f8_ci}"
)

print(
    f"  status: {f8_status}"
)


if not isinstance(
    f8_ci,
    list
) or len(
    f8_ci
) != 2:

    raise RuntimeError(
        "Phase 1F.8 bootstrap CI unavailable."
    )


if not (
    f8_ci[0]
    <=
    0
    <=
    f8_ci[1]
):

    raise RuntimeError(
        "Phase 1F.8 bootstrap CI does not cross zero."
    )


print(
    "[PASS] Phase 1F.8 secondary result."
)


# ============================================================
# 15. PHASE 1F.8 EXPERIMENTAL FIREWALL
# ============================================================

print(
    "\nPHASE 1F.8 EXPERIMENTAL FIREWALL"
)

print(
    "-" * 80
)


f8_firewall = f8.get(
    "experimental_firewall",
    {}
)


required_false_fields = [

    "dataset_modified",

    "state_bank_modified",

    "PCA_refitted",

    "transport_refitted",

    "model_reexecuted",

    "Phi_C_fitted",

    "admissibility_fitted",

    "test_used_for_selection",

    "test_used_for_hyperparameter_selection",

    "secondary_target_recomputed",

    "secondary_target_selected_after_F6",

    "new_target_definition",

]


f8_firewall_checks = {}


for key in required_false_fields:

    if key not in f8_firewall:

        raise RuntimeError(
            f"Phase 1F.8 firewall field missing: {key}"
        )


    normalized = as_bool(
        f8_firewall[
            key
        ]
    )


    passed = (
        normalized is False
    )


    f8_firewall_checks[
        key
    ] = {

        "raw":
            f8_firewall[
                key
            ],

        "normalized":
            normalized,

        "pass":
            passed,

    }


    print(
        f"  {key}: "
        f"normalized={normalized}, "
        f"PASS={passed}"
    )


    if not passed:

        raise RuntimeError(
            f"Phase 1F.8 experimental firewall failed: {key}"
        )


print(
    "[PASS] Phase 1F.8 experimental integrity."
)


# ============================================================
# 16. PHASE 1F.8 INTERPRETATION FIREWALL
# ============================================================

print(
    "\nPHASE 1F.8 INTERPRETATION FIREWALL"
)

print(
    "-" * 80
)


f8_interpretation = f8.get(
    "interpretation_firewall",
    {}
)


interpretation_false_fields = [

    "numerical_result_equals_CTL_irreducibility",

    "numerical_result_equals_Phi_C",

    "numerical_result_equals_K_C",

    "numerical_result_equals_Adm_C_3",

    "target_rank_is_statistically_independent_of_logit",

    "three_sector_presence_is_proof",

    "three_way_interaction_is_proof",

    "numerical_test_establishes_logical_transport",

    "numerical_test_establishes_contextual_geometry",

    "numerical_test_establishes_CTL_coherence",

]


f8_interpretation_checks = {}


for key in interpretation_false_fields:

    if key not in f8_interpretation:

        raise RuntimeError(
            f"Phase 1F.8 interpretation firewall field "
            f"missing: {key}"
        )


    normalized = as_bool(
        f8_interpretation[
            key
        ]
    )


    passed = (
        normalized is False
    )


    f8_interpretation_checks[
        key
    ] = {

        "raw":
            f8_interpretation[
                key
            ],

        "normalized":
            normalized,

        "pass":
            passed,

    }


    print(
        f"  {key}: "
        f"normalized={normalized}, "
        f"PASS={passed}"
    )


    if not passed:

        raise RuntimeError(
            f"Phase 1F.8 interpretation firewall failed: {key}"
        )


print(
    "[PASS] Phase 1F.8 interpretation firewall."
)


# ============================================================
# 17. FORMAL CTL STATUS
# ============================================================

ctl_status = {

    "formal_CTL_structure":
        "SUPPORTED",

    "context_indexing":
        "OPERATIONALLY_PRESERVED",

    "event_domains":
        "STRUCTURALLY_INSTANTIATED",

    "context_indexed_logical_carriers":
        "STRUCTURALLY_INSTANTIATED",

    "contextual_admissibility":
        "NOT_EMPIRICALLY_IDENTIFIED",

    "triadic_admissibility":
        "NOT_EMPIRICALLY_IDENTIFIED",

    "logical_transport":
        "STRUCTURALLY_INSTANTIATED_NOT_EMPIRICALLY_FIT",

    "logical_transport_covariance":
        "NOT_EMPIRICALLY_TESTED",

    "composition":
        "STRUCTURALLY_DEFINED_NOT_EMPIRICALLY_TESTED",

    "coherence_carrier_K_C":
        "FORMALLY_INSTANTIATED_NOT_EMPIRICALLY_IDENTIFIED",

    "Phi_C":
        "FORMALLY_TYPED_NOT_RECONSTRUCTED",

    "triadic_coherence":
        "NOT_EMPIRICALLY_TESTED",

    "CTL_triadic_irreducibility":
        "NOT_ESTABLISHED",

    "contextual_geometry":
        "NOT_ESTABLISHED",

}


print(
    "\nFORMAL CTL STATUS"
)

print(
    "-" * 80
)

for key, value in ctl_status.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 18. NUMERICAL SYNTHESIS
# ============================================================

numerical_synthesis = {

    "primary_phase":
        "1F.6",

    "primary_target":
        "clean target-logit minus corrupt target-logit",

    "primary_status":
        "HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED",

    "secondary_phase":
        "1F.8",

    "secondary_target":
        "clean target-rank minus corrupt target-rank",

    "secondary_status":
        "HELD_OUT_TRIADIC_PREDICTIVE_EFFECT_UNRESOLVED",

    "target_rank_statistically_independent_of_logit":
        False,

    "secondary_is_independent_replication":
        False,

    "overall_numerical_triatic_advantage":
        "NOT_SUPPORTED",

}


print(
    "\nNUMERICAL SYNTHESIS"
)

print(
    "-" * 80
)

for key, value in numerical_synthesis.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 19. FINAL SCIENTIFIC INTERPRETATION
# ============================================================

formal_interpretation = (
    "The ETTR-CTL-LLAMA-1 experiment operationally instantiated "
    "the formal CTL interfaces for contextual indexing, event "
    "domains, context-indexed logical carriers, logical-transport "
    "structure, and the formally typed triadic coherence "
    "interface. The experiment did not empirically identify "
    "contextual or triadic admissibility, the empirical coherence "
    "carrier K_C, or the Phi_C map, and did not empirically test "
    "full logical transport covariance. Consequently, CTL "
    "triadic irreducibility and contextual geometry were not "
    "established."
)


numerical_interpretation = (
    "Under the frozen equal-complexity numerical comparison, "
    "Phase 1F.6 found a held-out triadic predictive disadvantage "
    "relative to the calibration-selected dyadic comparator for "
    "the target-logit contrast. Phase 1F.8 then used the "
    "already-frozen target-rank observable as a secondary "
    "robustness target. The rank analysis produced a small "
    "apparent triadic MSE improvement, but its paired-bootstrap "
    "95 percent confidence interval crossed zero. The secondary "
    "rank result is therefore unresolved rather than evidence "
    "for a triadic advantage. Because target rank is derived "
    "from the same model prediction interface as target logit, "
    "this is a secondary robustness analysis rather than an "
    "independent replication."
)


print(
    "\nFINAL SCIENTIFIC INTERPRETATION"
)

print(
    "-" * 80
)

print(
    formal_interpretation
)

print(
    ""
)

print(
    numerical_interpretation
)


# ============================================================
# 20. NEGATIVE-CLAIM FIREWALL
# ============================================================

negative_claim_firewall = {

    "CTL_triadic_irreducibility_established":
        False,

    "empirical_Adm_C_3_established":
        False,

    "empirical_K_C_established":
        False,

    "Phi_C_reconstructed":
        False,

    "logical_transport_covariance_established":
        False,

    "contextual_geometry_established":
        False,

    "three_sector_presence_is_proof":
        False,

    "three_way_interaction_is_proof":
        False,

    "target_rank_is_independent_replication":
        False,

    "open_ended_target_hunting_authorized":
        False,

}


print(
    "\nNEGATIVE-CLAIM FIREWALL"
)

print(
    "-" * 80
)

for key, value in negative_claim_firewall.items():

    print(
        f"  {key}: {value}"
    )


if any(
    negative_claim_firewall.values()
):

    raise RuntimeError(
        "Negative-claim firewall failed."
    )


print(
    "[PASS] Negative-claim firewall."
)


# ============================================================
# 21. CLOSURE DECISION
# ============================================================

closure_decision = {

    "scientific_experiment_closed":
        True,

    "additional_numerical_phase_authorized":
        False,

    "additional_target_search_authorized":
        False,

    "open_ended_target_hunting_authorized":
        False,

    "model_reexecution_authorized":
        False,

    "PCA_refit_authorized":
        False,

    "transport_refit_authorized":
        False,

    "Phi_C_fit_authorized":
        False,

    "empirical_admissibility_invention_authorized":
        False,

    "manuscript_synthesis_authorized":
        True,

    "GPT2_Llama_comparative_analysis_authorized":
        True,

}


print(
    "\nEXPERIMENTAL CLOSURE DECISION"
)

print(
    "-" * 80
)

for key, value in closure_decision.items():

    print(
        f"  {key}: {value}"
    )


# ============================================================
# 22. MANUSCRIPT-SAFE SUMMARY
# ============================================================

manuscript_safe_summary = {

    "formal_status":
        (
            "The formal CTL structure was operationally "
            "instantiated, including context-indexed carriers, "
            "event domains, logical-transport interfaces, and "
            "the formally typed triadic coherence interface."
        ),

    "empirical_identifiability":
        (
            "The experiment did not empirically identify the "
            "contextual/triadic admissibility relation, the "
            "coherence carrier K_C, or the Phi_C map, and did "
            "not empirically test full logical transport "
            "covariance."
        ),

    "primary_numerical_finding":
        (
            "Under the frozen equal-complexity numerical "
            "comparison, the triadic predictor showed a "
            "held-out predictive disadvantage relative to the "
            "calibration-selected dyadic comparator for the "
            "target-logit contrast."
        ),

    "secondary_numerical_finding":
        (
            "A secondary analysis using the already-frozen "
            "target-rank contrast yielded a small apparent "
            "triadic improvement, but the paired-bootstrap "
            "confidence interval included zero, leaving the "
            "effect unresolved."
        ),

    "CTL_conclusion":
        (
            "These numerical findings do not establish CTL "
            "triadic irreducibility."
        ),

    "geometry_conclusion":
        (
            "Contextual geometry was not established by this "
            "experiment."
        ),

    "experiment_scope":
        (
            "The Llama experiment is scientifically closed. "
            "No additional target search or open-ended numerical "
            "experimentation is authorized within the frozen "
            "study."
        ),

}


print(
    "\nMANUSCRIPT-SAFE SUMMARY"
)

print(
    "-" * 80
)

for key, value in manuscript_safe_summary.items():

    print(
        f"\n[{key}]"
    )

    print(
        value
    )


# ============================================================
# 23. MASTER CLOSURE MANIFEST
# ============================================================

manifest = {

    "experiment_id":
        "ETTR-CTL-LLAMA-1",

    "phase":
        "1F.9",

    "title":
        "Final Experimental Closure / Master Result Manifest",

    "closure_status":
        "EXPERIMENT_SCIENTIFICALLY_CLOSED",

    "model":
        "meta-llama/Llama-3.2-3B",

    "selected_layer":
        14,

    "hidden_size":
        3072,

    "phase_map":
        {

            "1F.0":
                F0_PATH.name,

            "1F.1":
                F1_PATH.name,

            "1F.2A":
                F2A_PATH.name,

            "1F.2":
                F2_PATH.name,

            "1F.3":
                F3_PATH.name,

            "1F.4":
                F4_PATH.name,

            "1F.5":
                F5_PATH.name,

            "1F.6":
                F6_PATH.name,

            "1F.7":
                F7_PATH.name,

            "1F.8":
                F8_PATH.name,

        },

    "dataset":
        {

            "records":
                192,

            "conditions":
                384,

            "split":
                split_counts,

            "authorized_sha256":
                authorized_sha,

        },

    "state_bank":
        {

            "path":
                str(
                    STATE_BANK_PATH
                ),

            "hashes":
                state_hash_checks,

            "row_order":
                row_order_selected,

            "pT_pair_equal":
                pT_equal,

            "iT_pair_equal":
                iT_equal,

        },

    "phase_classifications":
        resolved_classifications,

    "primary_numerical_result":
        {

            "phase":
                "1F.6",

            "target":
                "target_logit_contrast",

            "classification":
                f6_classification,

            "numerical_status":
                f6_status,

            "historical_selected_dyadic":
                f6.get(
                    "selected_dyadic_model"
                ),

        },

    "secondary_numerical_result":
        {

            "phase":
                "1F.8",

            "target":
                "target_rank_contrast",

            "classification":
                f8_status,

            "selected_dyadic":
                f8_selected_dyadic,

            "triadic_model":
                "triadic_S1S2S3",

            "dyadic_test_mse":
                f8_dyadic_mse,

            "triadic_test_mse":
                f8_triadic_mse,

            "mse_delta_dyadic_minus_triadic":
                f8_delta,

            "relative_improvement_percent":
                f8_relative,

            "paired_bootstrap":
                {

                    "seed":
                        f8_bootstrap.get(
                            "seed"
                        ),

                    "replicates":
                        f8_bootstrap.get(
                            "replicates"
                        ),

                    "mean":
                        f8_bootstrap.get(
                            "mean"
                        ),

                    "ci95":
                        f8_ci,

                    "p_delta_gt_zero":
                        f8_bootstrap.get(
                            "p_delta_gt_zero"
                        ),

                },

        },

    "formal_CTL_status":
        ctl_status,

    "numerical_synthesis":
        numerical_synthesis,

    "negative_claim_firewall":
        negative_claim_firewall,

    "closure_decision":
        closure_decision,

    "final_scientific_interpretation":
        formal_interpretation,

    "final_numerical_interpretation":
        numerical_interpretation,

    "manuscript_safe_summary":
        manuscript_safe_summary,

    "provenance_note":
        (
            "The manifest records the operative authorized "
            "dataset SHA and frozen state-bank hashes used by "
            "the experiment. It does not claim reproduction of "
            "any superseded historical dataset hash."
        ),

}


# ============================================================
# 24. PRE-SERIALIZATION AUDIT
# ============================================================

print(
    "\nPRE-SERIALIZATION AUDIT"
)

print(
    "-" * 80
)


serialization_checks = {

    "manifest_dict":
        isinstance(
            manifest,
            dict
        ),

    "phase_map_dict":
        isinstance(
            manifest[
                "phase_map"
            ],
            dict
        ),

    "phase_classifications_dict":
        isinstance(
            manifest[
                "phase_classifications"
            ],
            dict
        ),

    "dataset_dict":
        isinstance(
            manifest[
                "dataset"
            ],
            dict
        ),

    "state_bank_dict":
        isinstance(
            manifest[
                "state_bank"
            ],
            dict
        ),

    "primary_result_dict":
        isinstance(
            manifest[
                "primary_numerical_result"
            ],
            dict
        ),

    "secondary_result_dict":
        isinstance(
            manifest[
                "secondary_numerical_result"
            ],
            dict
        ),

    "ctl_status_dict":
        isinstance(
            manifest[
                "formal_CTL_status"
            ],
            dict
        ),

    "closure_decision_dict":
        isinstance(
            manifest[
                "closure_decision"
            ],
            dict
        ),

}


for key, value in serialization_checks.items():

    print(
        f"  {key}: {value}"
    )


if not all(
    serialization_checks.values()
):

    raise RuntimeError(
        "Pre-serialization audit failed."
    )


print(
    "[PASS] Pre-serialization audit."
)


# ============================================================
# 25. WRITE CLOSURE MANIFEST
# ============================================================

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


print(
    "\n[PASS] Final closure manifest written."
)


# ============================================================
# 26. POST-SERIALIZATION VERIFICATION
# ============================================================

with open(
    OUTPUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    closure_check = json.load(
        f
    )


print(
    "\nPOST-SERIALIZATION VERIFICATION"
)

print(
    "-" * 80
)


readback_checks = {

    "experiment_id":
        closure_check.get(
            "experiment_id"
        ) ==
        "ETTR-CTL-LLAMA-1",

    "phase":
        closure_check.get(
            "phase"
        ) ==
        "1F.9",

    "closure_status":
        closure_check.get(
            "closure_status"
        ) ==
        "EXPERIMENT_SCIENTIFICALLY_CLOSED",

    "1F.0_mapping":
        closure_check.get(
            "phase_map",
            {}
        ).get(
            "1F.0"
        ) ==
        F0_PATH.name,

    "1F.1_mapping":
        closure_check.get(
            "phase_map",
            {}
        ).get(
            "1F.1"
        ) ==
        F1_PATH.name,

    "1F.2A_mapping":
        closure_check.get(
            "phase_map",
            {}
        ).get(
            "1F.2A"
        ) ==
        F2A_PATH.name,

    "1F.2_mapping":
        closure_check.get(
            "phase_map",
            {}
        ).get(
            "1F.2"
        ) ==
        F2_PATH.name,

    "1F.1_classification":
        closure_check.get(
            "phase_classifications",
            {}
        ).get(
            "1F.1"
        ) ==
        "FORMAL_CTL_STRUCTURE_IMPLEMENTATION_PASS",

    "1F.2_classification":
        closure_check.get(
            "phase_classifications",
            {}
        ).get(
            "1F.2"
        ) ==
        "CTL_SEMANTIC_REALIZATION_OPERATIONAL_PASS",

    "dataset_sha":
        closure_check.get(
            "dataset",
            {}
        ).get(
            "authorized_sha256"
        ) ==
        EXPECTED_DATASET_SHA256,

    "row_order":
        closure_check.get(
            "state_bank",
            {}
        ).get(
            "row_order"
        ) ==
        EXPECTED_ROW_ORDER,

    "pT_pair_equal":
        closure_check.get(
            "state_bank",
            {}
        ).get(
            "pT_pair_equal"
        ) is True,

    "iT_pair_equal":
        closure_check.get(
            "state_bank",
            {}
        ).get(
            "iT_pair_equal"
        ) is True,

    "F6_status":
        closure_check.get(
            "primary_numerical_result",
            {}
        ).get(
            "numerical_status"
        ) ==
        EXPECTED_F6_STATUS,

    "F8_status":
        closure_check.get(
            "secondary_numerical_result",
            {}
        ).get(
            "classification"
        ) ==
        EXPECTED_F8_STATUS,

    "CTL_irreducibility":
        closure_check.get(
            "formal_CTL_status",
            {}
        ).get(
            "CTL_triadic_irreducibility"
        ) ==
        "NOT_ESTABLISHED",

    "geometry":
        closure_check.get(
            "formal_CTL_status",
            {}
        ).get(
            "contextual_geometry"
        ) ==
        "NOT_ESTABLISHED",

    "additional_target_search":
        closure_check.get(
            "closure_decision",
            {}
        ).get(
            "additional_target_search_authorized"
        ) is False,

    "open_ended_hunting":
        closure_check.get(
            "closure_decision",
            {}
        ).get(
            "open_ended_target_hunting_authorized"
        ) is False,

    "manuscript_synthesis":
        closure_check.get(
            "closure_decision",
            {}
        ).get(
            "manuscript_synthesis_authorized"
        ) is True,

}


for key, value in readback_checks.items():

    print(
        f"  {key}: {value}"
    )


if not all(
    readback_checks.values()
):

    raise RuntimeError(
        "Final closure manifest readback verification failed."
    )


print(
    "[PASS] Final closure manifest readback."
)


# ============================================================
# 27. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "FINAL PHASE 1F.9 STATUS"
)

print(
    "=" * 80
)

print(
    "  ETTR-CTL-LLAMA-1: SCIENTIFICALLY CLOSED"
)

print(
    ""
)

print(
    "  PHASE MAPPING:"
)

print(
    "    1F.0  = CTL structural preflight"
)

print(
    "    1F.1  = formal CTL structure implementation"
)

print(
    "    1F.2A = frozen state-bank row-order verification"
)

print(
    "    1F.2  = semantic realization"
)

print(
    "    1F.3  = logical transport structure"
)

print(
    "    1F.4  = triadic coherence / Phi_C interface"
)

print(
    "    1F.5  = triadic irreducibility identifiability"
)

print(
    "    1F.6  = numerical triadic sufficiency"
)

print(
    "    1F.7  = numerical result synthesis"
)

print(
    "    1F.8  = target-rank robustness"
)

print(
    "    1F.9  = final closure"
)

print(
    ""
)

print(
    "  FORMAL CTL:"
)

print(
    "    structure: SUPPORTED"
)

print(
    "    empirical Adm_C^(3): NOT IDENTIFIED"
)

print(
    "    empirical K_C: NOT IDENTIFIED"
)

print(
    "    empirical Phi_C: NOT RECONSTRUCTED"
)

print(
    "    logical transport covariance: NOT EMPIRICALLY TESTED"
)

print(
    "    CTL triadic irreducibility: NOT ESTABLISHED"
)

print(
    "    contextual geometry: NOT ESTABLISHED"
)

print(
    ""
)

print(
    "  PRIMARY NUMERICAL RESULT — 1F.6:"
)

print(
    "    HELD-OUT TRIADIC PREDICTIVE DISADVANTAGE SUPPORTED"
)

print(
    ""
)

print(
    "  SECONDARY NUMERICAL RESULT — 1F.8:"
)

print(
    "    HELD-OUT TRIADIC PREDICTIVE EFFECT UNRESOLVED"
)

print(
    ""
)

print(
    "  OPEN-ENDED TARGET HUNTING: NOT AUTHORIZED"
)

print(
    "  ADDITIONAL NUMERICAL EXPERIMENTS: NOT AUTHORIZED"
)

print(
    "  MANUSCRIPT SYNTHESIS: AUTHORIZED"
)

print(
    "  GPT-2 / LLAMA COMPARATIVE ANALYSIS: AUTHORIZED"
)

print(
    ""
)

print(
    "  MASTER CLOSURE MANIFEST:"
)

print(
    f"    {OUTPUT_PATH}"
)

print(
    "=" * 80
)

ETTR-CTL-LLAMA-1 — PHASE 1F.9
FINAL EXPERIMENTAL CLOSURE / MASTER RESULT MANIFEST
Scientific computation: CLOSED
Purpose: frozen-artifact verification and final synthesis
Output: /content/ettr_ctl_llama/results/llama_phase1f9_final_experimental_closure_manifest.json

REQUIRED ARTIFACT AUDIT
--------------------------------------------------------------------------------
  [FOUND] llama_phase1c0_recovered_dataset_authorization_v2.json
  [FOUND] llama_phase1d2_full_state_bank.npz
  [FOUND] llama_phase1d2_full_state_bank_manifest.json
  [FOUND] llama_phase1f0_ctl_contextual_realization_preflight.json
  [FOUND] llama_phase1f1_formal_ctl_structure_audit.json
  [FOUND] llama_phase1f2a_state_bank_row_order_audit.json
  [FOUND] llama_phase1f2_ctl_semantic_realization_audit.json
  [FOUND] llama_phase1f3_logical_transport_covariance_audit.json
  [FOUND] llama_phase1f4_triadic_coherence_phiC_interface_audit.json
  [FOUND] llama_phase1f5_triadic_irreducibility_identifiability_audit.json
  [FOUND] 

In [52]:
from pathlib import Path

ARCHIVE = Path("/content/github_archive/ETTR-CTL-LLAMA-1")

ARCHIVE.mkdir(parents=True, exist_ok=True)

print("Archive directory:")
print(ARCHIVE)

Archive directory:
/content/github_archive/ETTR-CTL-LLAMA-1


In [53]:
import json

experiment_config = {
    "experiment_id": "ETTR-CTL-LLAMA-1",
    "model": "meta-llama/Llama-3.2-3B",
    "seed": 42,
    "selected_layer": 14,
    "hidden_size": 3072,
    "sectors": {
        "S1": "contextual interaction / attention output",
        "S2": "state propagation across depth / residual state entering block",
        "S3": "feature transformation / MLP output"
    },
    "backend": "Hugging Face Transformers",
    "hardware": "NVIDIA Tesla T4",
    "device": "cuda:0",
    "dtype": "BF16",
    "dataset_records": 192,
    "experiment_status": "closed"
}

with open(ARCHIVE / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(experiment_config, f, indent=2)

print("Created:", ARCHIVE / "experiment_config.json")

Created: /content/github_archive/ETTR-CTL-LLAMA-1/experiment_config.json


In [54]:
results_summary = {
    "experiment_id": "ETTR-CTL-LLAMA-1",

    "transport": {
        "S1": {
            "test_nrmse_transport": 0.974842090,
            "test_nrmse_identity": 0.973854329,
            "relative_improvement_percent": -0.104854,
            "cosine_transport": 0.158531,
            "cosine_identity": 0.162613
        },
        "S2": {
            "test_nrmse_transport": 1.068554146,
            "test_nrmse_identity": 1.068859612,
            "relative_improvement_percent": 0.010263,
            "cosine_transport": -0.004318,
            "cosine_identity": -0.004375
        },
        "S3": {
            "test_nrmse_transport": 1.009386142,
            "test_nrmse_identity": 1.010317125,
            "relative_improvement_percent": 0.073115,
            "cosine_transport": 0.027504,
            "cosine_identity": 0.026982
        }
    },

    "triadic_analysis": {
        "primary_status": "HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED",
        "ctl_irreducibility": "NOT_ESTABLISHED"
    },

    "rank_robustness": {
        "status": "HELD_OUT_TRIADIC_PREDICTIVE_EFFECT_UNRESOLVED",
        "dyadic_mse": 418.5135378562957,
        "triadic_mse": 416.46091629639744,
        "delta_mse": 2.0526215598982844,
        "relative_improvement_percent": 0.49045523602705765,
        "bootstrap_ci_95": [
            -0.44518365371362373,
            6.363428016130131
        ]
    },

    "closure": {
        "formal_ctl_structure": "SUPPORTED",
        "contextual_triatic_admissibility": "NOT_EMPIRICALLY_IDENTIFIED",
        "K_C": "NOT_IDENTIFIED",
        "Phi_C": "NOT_RECONSTRUCTED",
        "full_logical_covariance": "NOT_TESTED",
        "ctl_irreducibility": "NOT_ESTABLISHED",
        "geometric_realization": "NOT_TESTED"
    }
}

with open(ARCHIVE / "results_summary.json", "w", encoding="utf-8") as f:
    json.dump(results_summary, f, indent=2)

print("Created results_summary.json")

Created results_summary.json


In [55]:
dataset_manifest = {
    "experiment_id": "ETTR-CTL-LLAMA-1",
    "dataset_records": 192,
    "dataset_sha256": "7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d",
    "target_position_rule": "p_T = L(P); hidden state evaluated at i_T = L(P)-1",
    "clean_corrupt_design": "paired clean/corrupt conditions",
    "target_ids_derived_independently_for_llama": True
}

with open(ARCHIVE / "dataset_manifest.json", "w", encoding="utf-8") as f:
    json.dump(dataset_manifest, f, indent=2)

print("Created dataset_manifest.json")

Created dataset_manifest.json


In [56]:
import hashlib

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

hash_lines = []

for path in sorted(ARCHIVE.iterdir()):
    if path.is_file():
        hash_lines.append(
            f"{sha256_file(path)}  {path.name}"
        )

with open(ARCHIVE / "hashes.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(hash_lines) + "\n")

print("\n".join(hash_lines))

d7285cfc22e117223c60fc186b8a0ed42923654020fdcaa3371a27566569877a  dataset_manifest.json
6bba36d363f25f2066f512b14399d97f3bc30e185e6aaa7b128f5c9f9cb00b17  experiment_config.json
da9ef1147309a0bbcc792c3229bef5b6e0a3c109a191a7e319036cf522ef004b  results_summary.json


In [57]:
readme = """# ETTR-CTL-LLAMA-1

## Experiment

ETTR-CTL-LLAMA-1 is an empirical investigation of the operationalization of Empirical Triadic Transport Reconstruction (ETTR) and Contextual Transport Logic (CTL) using Meta Llama 3.2 3B.

## Model

- Model: `meta-llama/Llama-3.2-3B`
- Layers: 28
- Selected layer: 14
- Hidden dimension: 3072
- Attention heads: 24
- Key/value heads: 8
- Random seed: 42

## Computational environment

- Backend: Hugging Face Transformers
- Hardware: NVIDIA Tesla T4
- CUDA device: `cuda:0`
- Representation dtype: BF16

## Operational sectors

The experiment used three operational sectors:

- S1 — contextual interaction / attention output
- S2 — state propagation across depth / residual state entering block
- S3 — feature transformation / MLP output

These are operational sectors and should not be interpreted as proof that the Transformer has three ontologically fundamental components.

## Dataset

The experiment used 192 paired records.

Dataset SHA-256:

`7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d`

## Major findings

The formal CTL structural interfaces were operationally instantiated.

Frozen linear transport showed heterogeneous held-out behavior across the three operational sectors:

- S1: statistically supported deterioration relative to identity.
- S2: near-null / unresolved effect.
- S3: small but statistically supported improvement.

The primary held-out numerical triadic predictive comparison did not support a triadic predictive advantage.

CTL irreducibility was not established.

The secondary target-rank robustness comparison was unresolved.

## Scientific status

This experiment is a closed empirical record. The negative and unresolved findings are retained as part of the experimental history and are not interpreted as evidence that the underlying mathematical framework is invalid.

## Limitations

The experiment did not empirically identify contextual or triadic admissibility, reconstruct `K_C` or `Phi_C`, establish full logical covariance, or establish CTL irreducibility.

The experiment was executed on an NVIDIA Tesla T4. The T4 does not provide native BF16 Tensor Core support; therefore the computational environment should be regarded as part of the numerical execution record rather than as evidence about the mathematical structure itself.

## Files

- `experiment_config.json` — computational configuration
- `results_summary.json` — machine-readable scientific results
- `dataset_manifest.json` — dataset identity and operationalization
- `hashes.txt` — SHA-256 integrity record
- `scientific_record.md` — detailed scientific record
"""

with open(ARCHIVE / "README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("Created README.md")

Created README.md


In [58]:
print("ARCHIVE CONTENTS\n")

for path in sorted(ARCHIVE.iterdir()):
    print(path.name)

ARCHIVE CONTENTS

README.md
dataset_manifest.json
experiment_config.json
hashes.txt
results_summary.json


In [59]:
# ============================================================
# ETTR-CTL-LLAMA-1
# SCIENTIFIC RECORD ARCHIVE
#
# Creates:
#     /content/github_archive/ETTR-CTL-LLAMA-1/scientific_record.md
#
# This document is intended to preserve the detailed scientific
# history of the completed experiment for version-controlled
# archival and later use in the CSCI'26 manuscript.
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Ensure ARCHIVE exists
# ------------------------------------------------------------

ARCHIVE = Path("/content/github_archive/ETTR-CTL-LLAMA-1")
ARCHIVE.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Scientific record
# ------------------------------------------------------------

scientific_record = r"""# ETTR-CTL-LLAMA-1 Scientific Record

**Experiment ID:** ETTR-CTL-LLAMA-1
**Model:** Meta Llama 3.2 3B base model
**Primary model identifier:** `meta-llama/Llama-3.2-3B`
**Random seed:** 42
**Selected Transformer layer:** 14
**Experimental status:** CLOSED
**Scientific computation status:** CLOSED

---

# 1. Objective

ETTR-CTL-LLAMA-1 was conducted as the second major empirical reconstruction experiment in the ETTR/Contextual Transport Logic (CTL) research program, following the completed GPT-2 Small experiment ETTR-CTL-2.

The purpose was not to demonstrate that a Transformer literally instantiates the mathematical structures of Contextual Calculus or Contextual Transport Logic. Instead, the experiment investigated whether selected mathematical and logical structures could be translated into explicit, auditable operational objects on a contemporary open Transformer architecture, and whether those operational objects exhibited empirically measurable behavior consistent with the intended transport framework.

The experiment therefore maintained a strict distinction between:

1. the mathematical structure defined by Contextual Calculus and CTL;
2. the operationalization of selected components of that structure on Transformer representations; and
3. the numerical execution of those operationalizations on GPU hardware.

The experiment was designed so that numerical or architectural limitations would be recorded as empirical limitations or decoherence rather than used to modify the mathematical definitions.

The principal empirical areas were:

- Transformer representation extraction;
- identification of three operational sectors;
- finite transport between paired clean and corrupted contextual states;
- formal instantiation of CTL structural interfaces;
- semantic realization of context-indexed observations;
- logical transport interface construction;
- triadic coherence and `Phi_C` interface construction;
- numerical investigation of triadic predictive sufficiency;
- held-out generalization of learned sector transports.

The experiment was explicitly closed without extending the study to curvature, holonomy, global descent, Cech cohomology, connection reconstruction, or other higher geometric constructions.

---

# 2. Research questions

The experiment addressed the following empirical questions.

## RQ1 — Can the selected Transformer representation pathways be operationalized as measurable transport sectors?

Three operational sectors were defined:

- **S1:** contextual interaction / attention output;
- **S2:** state propagation across depth / residual state entering the block;
- **S3:** feature transformation / MLP output.

These were treated as operational sectors rather than as a claim that the Transformer possesses three ontologically fundamental components.

---

## RQ2 — Can finite transport maps be learned between paired contextual states?

For each sector, a restricted linear transport map was learned from corrupted-state representations toward clean-state representations.

The empirical object was of the form:

\[
z_{\mathrm{corrupt}}^{(k)}
\overset{\widehat{T}^{(k)}}{\longrightarrow}
z_{\mathrm{clean}}^{(k)}.
\]

Transport maps were trained using calibration data and subsequently evaluated on held-out test data.

---

## RQ3 — Can the formal structural components of CTL be explicitly represented?

The experiment investigated whether the following formal objects could be represented without conflating them with ordinary binary labels or hidden-state thresholds:

\[
C,\quad
\mathcal{L}_C^{(k)},\quad
Adm_C,\quad
Adm_C^{(3)},\quad
\widehat{T}_{\gamma}^{(k)},\quad
K_C,\quad
\Phi_C.
\]

The experiment distinguished the existence of typed structural interfaces from empirical identification of their semantic or logical content.

---

## RQ4 — Does the numerical three-sector representation provide a predictive advantage over lower-order representations?

A controlled numerical comparison was performed between dyadic and triadic predictor sets.

This was explicitly treated as a test of **numerical triadic predictive sufficiency**, not as a direct test of CTL triadic irreducibility.

A positive or negative numerical result therefore could not, by itself, establish or refute the mathematical irreducibility claim of CTL.

---

## RQ5 — Do the learned sector transport maps generalize to held-out data?

The principal transport question was whether calibration-trained frozen transport maps generalized to held-out test conditions.

The result was evaluated separately for S1, S2, and S3.

---

# 3. Mathematical basis

The experiment was grounded in the Contextual Calculus and Contextual Transport Logic framework.

The mathematical framework treats context-indexed structures and transport as primary objects rather than assuming a pre-existing geometric manifold, metric, or connection.

The foundational triadic transport structure is represented schematically as:

\[
\tau =
(\tau^{(1)},\tau^{(2)},\tau^{(3)}).
\]

A coherent realization is represented abstractly as:

\[
C=\Phi(\tau^{(1)},\tau^{(2)},\tau^{(3)}).
\]

The mathematical hierarchy is:

\[
\text{triadic transport}
\rightarrow
\text{coherence}
\rightarrow
\text{dynamic equivalence}
\rightarrow
\text{smooth realization}
\rightarrow
\text{connection}
\rightarrow
\text{curvature/holonomy}
\rightarrow
\text{global descent}.
\]

Only the finite empirical transport and selected logical/interface levels of this hierarchy were investigated experimentally.

In particular, this experiment did NOT attempt to claim empirical reconstruction of:

- a smooth manifold;
- a connection;
- curvature;
- holonomy;
- global descent;
- Cech cohomology;
- or a complete geometric realization.

The CTL formal structure treats context-indexed syntax and admissibility as distinct from ordinary Boolean truth assignment. In particular:

\[
\text{undefined}\neq\text{false}.
\]

Likewise:

\[
\text{Boolean valuation}
\neq
\text{context-indexed logical carrier}.
\]

This distinction became operationally important during the CTL phases.

---

# 4. Operationalization principles

The experiment followed several methodological constraints.

## 4.1 Mathematical authority

The mathematical framework was treated as authoritative.

Numerical implementation was not permitted to redefine mathematical objects merely because a particular CUDA, PyTorch, Transformer implementation, or hardware configuration made a direct realization difficult.

---

## 4.2 Operational sectors are not ontological claims

S1, S2, and S3 were chosen because they correspond to measurable Transformer computation pathways.

Their existence as measurable pathways does not establish that the underlying architecture has exactly three fundamental computational sectors.

---

## 4.3 Intervention success is not automatically scientific success

Successful hooks, finite tensors, successful GPU execution, and successful extraction do not establish transport, causal relevance, coherence, or triadic irreducibility.

Instrumentation success and scientific success were therefore evaluated separately.

---

## 4.4 Held-out evaluation

Learned maps and predictive comparisons were evaluated on held-out data.

Calibration data were used for selection where specified. Test data were not used for model selection.

---

## 4.5 No post hoc mathematical adjustment

The mathematical interpretation was not modified to make observed Transformer behavior conform to the desired result.

Negative, null, heterogeneous, and unresolved outcomes were retained as scientific results.

---

# 5. Model

The model was:

`meta-llama/Llama-3.2-3B`

The audited architecture contained:

- 28 Transformer layers;
- hidden dimension 3072;
- 24 attention heads;
- 8 key/value heads;
- feed-forward dimension 8192;
- vocabulary size 128256;
- maximum context length 131072;
- BF16 model configuration;
- Llama 3 RoPE configuration with theta 500000;
- grouped-query attention (GQA);
- gated MLP architecture;
- RMSNorm/pre-normalized Transformer organization.

Layer 14 was selected for the principal representation experiment.

The selected layer was kept fixed throughout the subsequent state-bank and transport analyses.

---

# 6. Dataset

The experiment used a recovered and explicitly authorized dataset containing:

- 192 experimental records;
- clean and corrupted prompt conditions;
- ordered contextual pairs;
- template identifiers;
- split assignments;
- target-token identifiers;
- target-position information.

The authorized dataset SHA-256 was:

`7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d`

Target-token identifiers were derived independently for Llama.

GPT-2 target-token IDs were not reused because tokenizer vocabularies differ across models.

---

## 6.1 Tokenization audit

The 192 clean and corrupted prompts were tokenized under the Llama tokenizer.

Prompt lengths ranged from 13 to 14 tokens.

Clean and corrupted prompts had equal lengths within each paired condition.

Of the 32 target names:

- 19 were single-token under the Llama tokenizer;
- 13 were multi-token.

The target-position operationalization was therefore defined explicitly rather than assuming that the target token itself had already been processed by the model.

---

# 7. Target-position operationalization

For a prompt \(P\), the target continuation begins at:

\[
p_T=L(P),
\]

where \(L(P)\) is the prompt length.

The hidden state used for predicting the next token was therefore the final prompt position:

\[
i_T=L(P)-1.
\]

The target token itself was not fed into the model when the target prediction was evaluated.

The target-position audit verified:

- exact prefix validity: 192/192;
- one-token target continuation: 192/192;
- equal clean/corrupted prompt lengths;
- equality of first-target positions across paired conditions;
- contextual target IDs were distinct;
- no cross-condition target overlap;
- valid split assignments.

The target-position operationalization audit therefore passed.

Artifact:

`llama_phase1c3_target_position_operationalization_audit.json`

---

# 8. Computational environment

The experiment was executed using:

- Python 3.13.15;
- PyTorch 2.11.0+cu128;
- Transformers 5.16.1;
- CUDA 12.8;
- NVIDIA Tesla T4;
- GPU compute capability 7.5.

The model was executed using BF16 representation.

The T4 does not possess native BF16 Tensor Core support. Therefore, successful BF16 execution on the T4 must not be interpreted as evidence of native BF16 Tensor Core acceleration.

This distinction is part of the numerical execution record.

The hardware was treated as an execution substrate rather than as a component of the mathematical theory.

---

# 9. Phase 1A — Architecture audit

The first phase verified the model and architecture before scientific extraction.

The audit confirmed:

- the intended model was loaded;
- the model architecture matched the expected Llama 3.2 3B configuration;
- 28 layers were present;
- hidden size was 3072;
- the attention configuration contained 24 query heads and 8 key/value heads;
- the selected layer was layer 14;
- the expected model configuration was accessible.

**Status: PASS**

No scientific inference was made from the architecture audit alone.

---

# 10. Phase 1B — Model execution and intervention audit

The model was loaded using an exact CPU BF16 checkpoint followed by:

`model.to(cuda:0, dtype=BF16)`

The resulting model contained:

- 254 CUDA parameter tensors;
- 3,212,749,824 parameters;
- 2 buffers;
- approximately 5.993 GiB of parameter storage.

Synthetic baseline logits had shape:

\[
(1,16,128256)
\]

and were finite.

The audit also tested noninterference.

Maximum and relative output differences under the relevant sham/nonintervention conditions were zero.

Sector extraction tensors were finite and passed norm sanity checks.

Parameter immutability was verified.

**Status: PASS**

This established that the numerical model execution and instrumentation were functioning.

It did not establish the scientific validity of transport or CTL claims.

---

# 11. Phase 1C — Dataset and target-position audit

The dataset was recovered and its authorized SHA-256 verified.

The target-position audit established the correct causal ordering:

\[
\text{prompt}
\rightarrow
\text{final prompt hidden state}
\rightarrow
\text{next-token prediction}.
\]

The target token was not supplied to the model before the target prediction was measured.

All 192 records passed the target-position checks.

**Status: PASS**

Artifact:

`/content/ettr_ctl_llama/results/llama_phase1c3_target_position_operationalization_audit.json`

---

# 12. Phase 1D.0 — State extraction audit

Layer 14 was instrumented to extract three operational sectors.

The operational sectors were:

### S1 — attention/contextual interaction

Attention output before the residual addition.

### S2 — state propagation

Decoder-layer input / residual state entering the selected block.

### S3 — feature transformation

MLP output before residual addition.

All extracted states had shape:

\[
[1,14,3072]
\]

before selection of the target position.

The final target-position vectors had dimension:

\[
3072.
\]

Representative state norms were:

- S1: 3.69046760
- S2: 10.57186508
- S3: 6.66964483

Representative cross-sector cosine values were:

- S1/S2: -0.00868703
- S1/S3: -0.29631329
- S2/S3: -0.04869696

All extracted tensors were finite.

**Status: PASS**

Artifact:

`/content/ettr_ctl_llama/results/llama_phase1d0_state_extraction_audit.json`

---

# 13. Phase 1D.2 — Full state-bank extraction

The full state bank contained 384 conditions:

\[
192\ \text{clean} + 192\ \text{corrupt}.
\]

Each operational sector was stored as a:

\[
384\times3072
\]

matrix in float32 on CPU after extraction.

The paired clean-corrupt displacement magnitudes were:

| Sector | Mean paired displacement |
|---|---:|
| S1 | 1.275078 |
| S2 | 2.597086 |
| S3 | 1.653374 |

State-bank hashes:

- S1: `9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783`
- S2: `41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb`
- S3: `173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3`

Target-logit artifact hash:

`537fa43c8e278b8678de978c1dc2ec4ee5a1c77ca0a917dbf6a5823cf77ea58f`

Target-rank artifact hash:

`f4de73b588c465e0c1e41887a2eab7b1e0f2214d2d15dc0108b69e4063e2d472`

Artifact:

`/content/ettr_ctl_llama/results/llama_phase1d2_full_state_bank.npz`

---

# 14. Phase 1D.3 — Post-extraction audit

The state-bank structure was checked after extraction.

The bank contained:

- 192 clean conditions;
- 192 corrupted conditions;
- train/calibration/test split assignments.

The final split structure was:

- training: 192 rows;
- calibration: 96 rows;
- test: 96 rows.

**Status: PASS**

---

# 15. Phase 1E.0 — Transport preflight

Train-only PCA was performed for each sector.

Retention values were evaluated at several candidate dimensions.

## S1

| Dimension | Variance retention |
|---:|---:|
| 4 | 0.757898 |
| 8 | 0.899873 |
| 16 | 0.935448 |
| 32 | 0.964765 |
| 64 | 0.986598 |
| 128 | 0.997363 |

## S2

| Dimension | Variance retention |
|---:|---:|
| 4 | 0.818344 |
| 8 | 0.941898 |
| 16 | 0.962889 |
| 32 | 0.979103 |
| 64 | 0.990516 |
| 128 | 0.997810 |

## S3

| Dimension | Variance retention |
|---:|---:|
| 4 | 0.822104 |
| 8 | 0.933162 |
| 16 | 0.956174 |
| 32 | 0.974828 |
| 64 | 0.988609 |

## Joint state representation

| Dimension | Variance retention |
|---:|---:|
| 4 | 0.812318 |
| 8 | 0.933851 |
| 16 | 0.956118 |
| 32 | 0.974271 |
| 64 | 0.988308 |
| 128 | 0.997255 |

These values were used as a transport-preflight diagnostic.

Importantly, the selection of dimension \(d=4\) in the subsequent transport experiment was based on calibration performance, not on a claim that the representation was intrinsically four-dimensional.

---

# 16. Phase 1E.1 — Transport candidate selection

Candidate linear transport maps were evaluated over:

\[
d\in\{4,8,16,32,64\}
\]

and ridge regularization:

\[
\lambda\in
\{0.01,0.1,1,10,100,1000\}.
\]

PCA bases were fitted using training data only.

The transport maps used no intercept.

Candidate selection was performed using calibration data.

The selected configurations were:

### S1

- dimension: 4
- ridge: 0.01
- calibration NRMSE: 0.201006
- identity NRMSE: 0.202619
- cosine: 0.985794

### S2

- dimension: 4
- ridge: 1
- calibration NRMSE: 0.091608
- identity NRMSE: 0.090833
- cosine: 0.995769

### S3

- dimension: 4
- ridge: 0.01
- calibration NRMSE: 0.120509
- identity NRMSE: 0.119503
- cosine: 0.993117

Training-fit NRMSE values were:

- S1: 0.075175 versus identity 0.078320;
- S2: 0.045758 versus identity 0.046615;
- S3: 0.050468 versus identity 0.051745.

The transport checkpoint was:

`/content/ettr_ctl_llama/checkpoints/llama_phase1e1_selected_transport_maps.pkl`

The candidate-selection manifest was:

`/content/ettr_ctl_llama/results/llama_phase1e1_transport_candidate_selection.json`

The fact that all three selected maps used \(d=4\) was a calibration-selection result and was NOT interpreted as evidence of intrinsic four-dimensional Transformer geometry.

---

# 17. Phase 1E.2 — Frozen held-out transport

The selected transport maps were frozen and evaluated on the held-out test set.

## S1

Transport NRMSE:

\[
0.974842090
\]

Identity NRMSE:

\[
0.973854329
\]

Relative improvement:

\[
-0.104854\%.
\]

Transport cosine:

\[
0.158531
\]

Identity cosine:

\[
0.162613.
\]

Bootstrap estimate of identity-minus-transport NRMSE difference:

\[
-0.000987761
\]

with 95% CI:

\[
[-0.001486016,\,-0.000478748].
\]

This constitutes a statistically supported deterioration relative to identity.

---

## S2

Transport NRMSE:

\[
1.068554146
\]

Identity NRMSE:

\[
1.068859612
\]

Relative improvement:

\[
+0.010263\%.
\]

Transport cosine:

\[
-0.004318
\]

Identity cosine:

\[
-0.004375.
\]

Bootstrap estimate:

\[
+0.000305466
\]

with 95% CI:

\[
[-0.000336442,\,+0.000916010].
\]

The effect is near-null and unresolved.

---

## S3

Transport NRMSE:

\[
1.009386142
\]

Identity NRMSE:

\[
1.010317125
\]

Relative improvement:

\[
+0.073115\%.
\]

Transport cosine:

\[
0.027504
\]

Identity cosine:

\[
0.026982.
\]

Bootstrap estimate:

\[
+0.000930983
\]

with 95% CI:

\[
[0.000425511,\,+0.001460057].
\]

This is a small but statistically supported improvement.

---

## Frozen transport conclusion

The frozen linear transport operationalization therefore exhibited heterogeneous held-out behavior:

- **S1:** statistically supported deterioration;
- **S2:** unresolved near-null effect;
- **S3:** small but statistically supported improvement.

Final transport classification:

`FROZEN_TRANSPORT_HAS_MIXED_OR_NULL_GENERALIZATION`

This was retained as a scientific result rather than averaged away across sectors.

Decoded transport versus decoded identity showed:

- S1: 0.682013799 versus 0.681535960 — transport worse;
- S2: 0.604606829 versus 0.604751237 — transport improves;
- S3: 0.702645672 versus 0.703139990 — transport improves.

---

# 18. CTL phases

The CTL portion of the experiment was deliberately separated from the numerical transport analysis.

The objective was to establish whether formal CTL structural objects could be operationally represented without incorrectly equating them with hidden-state thresholds or ordinary binary covariates.

The formal CTL tuple was represented schematically as:

\[
\mathfrak{CTL}
=
(C,
\{E_C\},
\{L_C^{(k)}\},
\{Adm_C,Adm_C^{(3)}\},
\{\widehat{T}_\gamma^{(k)}\},
\{K_C,\Phi_C\}).
\]

---

# 19. Phase 1F.0 — Formal CTL structural preflight

A formal CTL structure was assembled containing:

- context-indexed carriers;
- event domains;
- contextual admissibility placeholders;
- triadic admissibility placeholders;
- logical transport registry;
- composition scaffold;
- coherence registry;
- typed `Phi_C` interfaces.

No empirical truth rule was invented at this stage.

No hidden-state threshold was treated as a CTL axiom.

No binary feature was automatically interpreted as a CTL proposition.

**Status: PASS**

The phase established that the formal structural interfaces could be instantiated as software objects without making empirical claims about their values.

---

# 20. Phase 1F.1 — Initial methodological failure and correction

The first attempted empirical CTL realization contained a methodological error.

It used hidden-state MAD thresholds and represented triadic admissibility as a pairwise conjunction.

This created a situation in which zero mismatch could become tautological by construction.

That implementation was rejected as scientifically invalid.

The problem was not treated as evidence against CTL.

Instead, the implementation was corrected so that:

- empirical admissibility was not invented;
- triadic admissibility was not reduced to pairwise conjunction;
- hidden-state thresholding was not used to define logical truth;
- semantic valuation remained distinct from structural admissibility;
- no numerical object was falsely designated as `K_C` or `Phi_C`.

The corrected implementation formally instantiated the CTL structure without claiming empirical identification of its semantic or logical values.

This correction is retained in the scientific record because it documents a methodological failure and the subsequent removal of a potential tautology.

---

# 21. Corrected Phase 1F.1 — Formal CTL structure realization

The corrected implementation produced:

- 192 contexts;
- 3 event domains per context;
- 576 context-indexed logical carriers;
- 192 partial admissibility domains;
- 192 triadic admissibility objects;
- 192 CTL morphisms;
- 192 logical transport objects;
- 3 sector maps for each logical transport object;
- composition scaffolding;
- 192 `K_C` coherence carriers;
- 192 typed `Phi_C` interfaces.

The following were deliberately NOT empirically identified:

- contextual admissibility;
- triadic admissibility;
- logical transport preservation;
- logical operation preservation;
- `K_C`;
- `Phi_C`.

Final status:

`FORMAL_CTL_STRUCTURE_IMPLEMENTATION_PASS`

---

# 22. Phase 1F.2A — State-bank row-order audit

Before semantic CTL realization, the ordering of the state bank was audited.

The actual state-bank ordering was interleaved:

\[
bank[2i]=clean_i,
\]

\[
bank[2i+1]=corrupt_i.
\]

This was explicitly checked rather than assumed.

The candidate interleaved clean/corrupt ordering passed:

- target-position consistency: 384/384;
- Llama target-ID consistency: 384/384.

Artifact:

`/content/ettr_ctl_llama/results/llama_phase1f2a_state_bank_row_order_audit.json`

---

# 23. Phase 1F.2 — Semantic realization

The semantic realization stage created:

- 384 context-condition semantic models;
- 1152 semantic valuations.

The primary semantic observations were target-logit and target-rank differences between clean and corrupted conditions.

The clean-minus-corrupt target-logit difference had:

- mean: 0.069010417;
- median: 0.

The target-rank improvement had:

- mean: 0.432291667;
- median: 0.

Split-level results were:

### Train

- N = 96;
- target-logit difference mean: 0.063802083;
- target-rank difference mean: -0.90625.

### Calibration

- N = 48;
- target-logit difference mean: -0.055989583;
- target-rank difference mean: -0.708333.

### Test

- N = 48;
- target-logit difference mean: 0.204427083;
- target-rank difference mean: 4.25.

The semantic layer was therefore operationally realized, but contextual and triadic admissibility remained unresolved.

No semantic threshold was imposed to manufacture a Boolean result.

No `Phi_C` was reconstructed.

No collapse of truth, coherence, and state similarity was permitted.

Final status:

`CTL_SEMANTIC_REALIZATION_OPERATIONAL_PASS`

---

# 24. Phase 1F.3 — Logical transport covariance structure audit

The logical transport structure was audited independently of semantic value invariance.

The audit contained:

- 576 context-indexed carriers;
- 576 event structures;
- 576 morphisms;
- 1152 semantic observations;
- endpoint typing: 576/576.

The audit explicitly recognized that semantic values need not be invariant under transport.

The following were NOT empirically established:

- admissibility preservation;
- logical operation preservation;
- full composition/functoriality;
- semantic covariance in the strong sense.

The triadic transport registry was retained as a three-sector formal object.

No pairwise reduction was substituted for the triadic registry.

Final status:

`LOGICAL_TRANSPORT_STRUCTURE_OPERATIONAL_PASS`

The strongest scientifically justified interpretation is that the formal CTL carrier/event/semantic/logical-transport interfaces were operationally realized with preserved context and sector typing.

Empirical admissibility preservation, logical operation preservation, and full logical covariance were not identified.

---

# 25. Phase 1F.4 — Triadic coherence and Phi_C interface audit

This phase audited the existence and typing of the formal triadic coherence interfaces.

The implementation contained:

- 192 formal `Adm_C^(3)` objects;
- 192 `K_C` coherence carriers;
- 192 typed `Phi_C` interfaces.

However, none of the following was empirically identified:

- contextual admissibility;
- triadic admissibility;
- an empirical `K_C`;
- an empirical `Phi_C`.

In particular, the following were explicitly NOT designated as `K_C` or `Phi_C`:

- hidden states;
- semantic observations;
- numerical transport outputs.

The phase therefore established interface-level operational support but not empirical reconstruction of the mathematical coherence map.

Final status:

`TRIADIC_COHERENCE_INTERFACE_OPERATIONAL_PASS_NOT_EMPIRICALLY_IDENTIFIED`

---

# 26. Phase 1F.5 — Triadic irreducibility identifiability audit

The formal CTL structure contained a triadic object.

However, this does not mean that empirical triadic irreducibility had been identified.

The audit established:

- a formal triadic CTL object was available;
- lower-order reducts could be formally defined;
- invalid reductions were rejected;
- held-out train/calibration/test capacity was available.

The capacity structure was:

- train: 96;
- calibration: 48;
- test: 48.

However, a sufficient complexity-control scheme had not been frozen for an empirical mathematical irreducibility claim.

Specifically, the experiment did not establish a final:

- parameter-matched dyadic/triadic comparison;
- effective-dimension-matched comparison;
- penalty-controlled nested comparison.

Therefore the mathematical CTL irreducibility question was not empirically identifiable in this experiment.

Final status:

`TRIADIC_IRREDUCIBILITY_NOT_CURRENTLY_IDENTIFIABLE`

A numerical triadic predictive-sufficiency experiment was therefore treated only as a lower-level empirical probe.

---

# 27. Phase 1F.6 — Numerical triadic predictive sufficiency

A numerical comparison was performed using the target logit clean-corrupt difference as the response variable.

Predictors were paired state differences.

The comparison used an equal six-predictor budget.

The dyadic representation used:

- 3 predictors from one selected sector;
- 3 predictors from another selected sector.

The triadic representation used:

- 2 predictors from S1;
- 2 predictors from S2;
- 2 predictors from S3.

Dyadic selection was performed using calibration data.

The final regression used ridge regularization:

\[
\lambda=1.
\]

No PCA refitting was performed after the checkpoint schema audit.

The primary held-out result supported a **triadic predictive disadvantage** rather than a triadic predictive advantage.

Final classification:

`NUMERICAL_TRIADIC_PREDICTIVE_SUFFICIENCY_NOT_SUPPORTED_CTL_IRREDUCIBILITY_NOT_ESTABLISHED`

Numerical status:

`HELD_OUT_TRIADIC_PREDICTIVE_DISADVANTAGE_SUPPORTED`

This result was not interpreted as a refutation of CTL irreducibility because the numerical predictor comparison is not mathematically equivalent to the CTL irreducibility proposition.

---

# 28. Phase 1F.8 — Secondary target-rank robustness analysis

A secondary analysis examined target-rank behavior.

The comparator was the S2+S3 dyadic representation.

The held-out results were:

Dyadic MSE:

\[
418.5135378562957
\]

Triadic MSE:

\[
416.46091629639744
\]

Difference:

\[
2.0526215598982844
\]

Relative improvement:

\[
0.49045523602705765\%.
\]

The bootstrap 95% confidence interval for the corresponding effect was:

\[
[-0.44518365371362373,\,
6.3634280161301415].
\]

Because the confidence interval crossed zero, the target-rank triadic effect was unresolved.

Final status:

`HELD_OUT_TRIADIC_PREDICTIVE_EFFECT_UNRESOLVED`

This result provided no basis for claiming a robust triadic advantage.

---

# 29. Transport interpretation

The frozen transport experiment produced heterogeneous behavior.

The scientifically justified statement is:

> The frozen linear transport operationalization exhibits heterogeneous held-out behavior across the three operational sectors: a small but statistically supported improvement for S3, an unresolved near-null effect for S2, and a statistically supported deterioration for S1.

This heterogeneity is compatible with several possible explanations, including architectural differences in the functional organization of the sectors.

However, this experiment did not establish a causal explanation for the heterogeneity.

In particular, it did not establish that:

- GQA caused the S1 deterioration;
- RMSNorm caused the S2 behavior;
- gated MLP structure caused the S3 improvement;
- or any individual architectural component caused the observed transport differences.

Architecture may affect the empirical transport behavior, but causal attribution requires a separately controlled architectural comparison.

---

# 30. CTL interpretation

The experiment provided evidence for the **operational representation of formal CTL structures**, but not for complete empirical realization of CTL semantics.

Supported operational elements included:

- context indexing;
- event-domain separation;
- logical-carrier typing;
- morphism typing;
- sector-specific transport registration;
- triadic transport registry;
- coherence-interface representation;
- typed `Phi_C` interface representation.

Not empirically established were:

- contextual admissibility;
- triadic admissibility;
- empirical `K_C`;
- empirical `Phi_C`;
- logical operation preservation;
- admissibility preservation;
- full logical covariance;
- CTL irreducibility.

Therefore, the correct conclusion is not:

> "The Transformer implements CTL."

The stronger and more accurate statement is:

> "Selected formal CTL structures were operationally instantiated as typed computational objects, while their empirical semantic, admissibility, coherence, and irreducibility properties remained incompletely identified."

---

# 31. Binary features and Boolean logic

The experiment explicitly distinguished ordinary binary computational features from CTL logical structure.

A binary variable with values:

\[
0,1
\]

is not automatically a CTL proposition.

Likewise:

\[
\text{binary feature}
\neq
\text{context-indexed logical carrier}.
\]

A genuine CTL logical realization requires explicit representation of:

\[
C,
\mathcal L_C^{(k)},
Adm_C,
Adm_C^{(3)},
\widehat{T}_\gamma^{(k)},
K_C,
\Phi_C.
\]

The earlier temptation to treat ordinary binary `transfer` or `presentation` features as if they constituted CTL logical computation was therefore rejected.

The current experiment correctly avoided making that identification.

This distinction is important for future work.

---

# 32. Relationship between numerical transport and formal CTL

The experiment kept two levels separate.

## Formal level

The CTL structure specifies context-indexed carriers, admissibility, morphisms, logical transport, and coherence-related objects.

## Numerical level

Transformer hidden-state differences and learned finite-dimensional linear maps provide numerical observables and transport approximations.

The numerical transport experiment therefore should not be read as a direct empirical measurement of every formal CTL object.

The strongest demonstrated bridge was:

\[
z_{\mathrm{corrupt}}^{(k)}
\rightarrow
\widehat{T}^{(k)}
\rightarrow
z_{\mathrm{clean}}^{(k)}
\]

combined with explicit context and sector typing.

The higher-level map:

\[
C=\Phi(\tau^{(1)},\tau^{(2)},\tau^{(3)})
\]

was represented as a formal interface but not empirically reconstructed.

---

# 33. Negative and unresolved findings

The following negative or unresolved findings are part of the scientific result.

## 33.1 Frozen transport

Frozen transport did not generalize uniformly.

- S1 deteriorated;
- S2 was near-null;
- S3 showed a small improvement.

Therefore no uniform cross-sector transport advantage was established.

---

## 33.2 Triadic predictive sufficiency

The primary numerical held-out triadic comparison did not support a triadic predictive advantage.

Instead, a held-out triadic predictive disadvantage was supported for the target-logit analysis.

---

## 33.3 Target-rank robustness

The secondary target-rank comparison was unresolved.

Its small apparent triadic improvement was not statistically decisive.

---

## 33.4 CTL irreducibility

CTL irreducibility was not established.

The numerical predictor experiment cannot substitute for the mathematical irreducibility claim.

---

## 33.5 `Phi_C`

An empirical contextual realization map `Phi_C` was not reconstructed.

---

## 33.6 `K_C`

An empirical coherence carrier `K_C` was not identified.

---

## 33.7 Admissibility

Contextual and triadic admissibility were represented formally but were not empirically identified.

---

## 33.8 Full logical covariance

Full preservation of logical operations and admissibility under transport was not tested.

---

## 33.9 Geometric realization

No smooth geometric realization was established.

No connection, curvature, holonomy, or global descent result was obtained.

---

# 34. Why the negative results are retained

The purpose of the experiment was empirical reconstruction, not confirmation.

Consequently, a result such as:

\[
\text{triadic predictive advantage not supported}
\]

is retained as an experimental result rather than reinterpreted as a software problem solely because it does not support the theoretical expectation.

Likewise, a mixed transport result is retained rather than averaged into a single positive claim.

The experiment therefore functions as a falsification-sensitive empirical layer around the mathematical framework.

---

# 35. Architectural interpretation

Llama 3.2 3B differs substantially from GPT-2 Small.

Relevant architectural differences include:

- substantially greater depth;
- substantially larger hidden dimension;
- grouped-query attention;
- RMSNorm/pre-normalization;
- rotary positional encoding;
- gated MLP structure;
- different tokenizer and vocabulary;
- substantially different parameter scale and training regime.

Consequently, the empirical behavior of S1/S2/S3 cannot be assumed to be architecture-invariant.

The experiment therefore does not claim that the GPT-2 and Llama sector results should coincide.

The Llama results demonstrate that the ETTR operationalization can be executed on a substantially more contemporary Transformer organization, but they do not establish architecture-independent transport laws.

---

# 36. Hardware and numerical interpretation

The experiment ran successfully on an NVIDIA Tesla T4.

The T4 provided sufficient memory and compute for the present 3B-parameter model experiment.

However, the T4 belongs to an earlier accelerator generation than current transformer-oriented accelerators.

Most importantly, the T4 is compute capability 7.5 and does not have native BF16 Tensor Core support.

Therefore:

\[
\text{BF16 representation}
\neq
\text{native BF16 Tensor Core acceleration on T4}.
\]

This is a numerical execution constraint.

It does not alter the mathematical definition of CTL or ETTR.

Future experiments on newer accelerators may produce different numerical performance characteristics, particularly for BF16, FP8, memory bandwidth, attention kernels, and large-scale matrix operations.

Such differences should be treated as numerical/hardware factors rather than silently incorporated into the mathematical framework.

---

# 37. Scientific closure

The scientific computation for ETTR-CTL-LLAMA-1 was closed after the completion of the CTL and numerical triadic analyses.

The final closure was:

### Formal CTL structure

**SUPPORTED**

The formal CTL carrier, event, morphism, logical-transport, triadic, coherence-interface, and typed `Phi_C` structures were operationally represented.

### Contextual/triadic admissibility

**NOT EMPIRICALLY IDENTIFIED**

### `K_C`

**NOT IDENTIFIED**

### `Phi_C`

**NOT RECONSTRUCTED**

### Logical transport covariance

**NOT FULLY TESTED**

### Frozen sector transport

**MIXED OR NULL GENERALIZATION**

### Primary numerical triadic predictive advantage

**NOT SUPPORTED**

### Primary numerical triadic result

**HELD-OUT TRIADIC PREDICTIVE DISADVANTAGE SUPPORTED**

### Target-rank robustness

**UNRESOLVED**

### CTL irreducibility

**NOT ESTABLISHED**

### Geometric realization

**NOT TESTED**

The overall interpretation is therefore deliberately conservative.

---

# 38. Overall experiment classification

The experiment does NOT justify the statement:

> "Llama 3.2 3B empirically proves Contextual Transport Logic."

It also does NOT justify:

> "The failure of triadic prediction disproves the mathematical triadic structure."

The appropriate interpretation is:

> ETTR-CTL-LLAMA-1 successfully operationalized selected formal CTL structures and finite sector-specific transport procedures on Llama 3.2 3B. The resulting frozen transport maps exhibited heterogeneous held-out behavior, with a statistically supported improvement for S3, a near-null unresolved effect for S2, and a statistically supported deterioration for S1. The primary numerical triadic predictive comparison did not support a triadic predictive advantage, while a secondary target-rank comparison remained unresolved. Formal CTL irreducibility, empirical admissibility, empirical coherence realization, and `Phi_C` reconstruction were not established.

The experiment therefore constitutes a **mixed empirical result with strong operational support for the structural implementation layer but no empirical confirmation of the stronger CTL irreducibility or coherence claims**.

---

# 39. Phase-status summary

| Phase | Purpose | Status |
|---|---|---|
| 1A | Architecture audit | PASS |
| 1B | Model execution/intervention audit | PASS |
| 1C.3 | Target-position operationalization | PASS |
| 1D.0 | State extraction audit | PASS |
| 1D.2 | Full state-bank extraction | PASS |
| 1D.3 | Post-extraction audit | PASS |
| 1E.0 | Transport preflight | PASS |
| 1E.1 | Transport candidate selection | PASS |
| 1E.2 | Frozen held-out transport | MIXED/NULL GENERALIZATION |
| 1F.0 | Formal CTL structural preflight | PASS |
| 1F.1 | Initial CTL empirical realization | REJECTED — METHODOLOGICAL TAUTOLOGY |
| Corrected 1F.1 | Formal CTL structure realization | PASS |
| 1F.2A | State-bank row-order audit | PASS |
| 1F.2 | CTL semantic realization | OPERATIONAL PASS |
| 1F.3 | Logical transport covariance structure | OPERATIONAL PASS |
| 1F.4 | Triadic coherence / `Phi_C` interfaces | OPERATIONAL PASS; NOT EMPIRICALLY IDENTIFIED |
| 1F.5 | Triadic irreducibility identifiability | NOT IDENTIFIABLE |
| 1F.6 | Numerical triadic predictive sufficiency | NOT SUPPORTED |
| 1F.8 | Target-rank robustness | UNRESOLVED |
| 1F.9 | Scientific closure | CLOSED |

---

# 40. Principal numerical results

## Frozen transport

| Sector | Transport NRMSE | Identity NRMSE | Relative change | Interpretation |
|---|---:|---:|---:|---|
| S1 | 0.974842090 | 0.973854329 | -0.104854% | Supported deterioration |
| S2 | 1.068554146 | 1.068859612 | +0.010263% | Near-null / unresolved |
| S3 | 1.009386142 | 1.010317125 | +0.073115% | Small supported improvement |

---

## Target-rank secondary analysis

| Measure | Value |
|---|---:|
| Dyadic MSE | 418.5135378562957 |
| Triadic MSE | 416.46091629639744 |
| MSE difference | 2.0526215598982844 |
| Relative improvement | 0.49045523602705765% |
| Bootstrap 95% CI | [-0.44518365371362373, 6.3634280161301415] |
| Interpretation | Unresolved |

---

# 41. Reproducibility information

The experiment should be reproduced only from the archived experiment configuration, dataset identity, notebook/code version, and numerical artifacts.

Important reproducibility identifiers include:

**Dataset SHA-256**

`7b14ddebef594859cf284deb5037ac9bb7e67956edb640012eaa257325e8769d`

**S1 state-bank SHA-256**

`9621683192d82c8874f198660015abe88aef356ef58cbb79cfef51cfa8103783`

**S2 state-bank SHA-256**

`41679cd0a1b46103ae896edf0ff106718b17499eb12bdb3c68ccd3a3f1b898fb`

**S3 state-bank SHA-256**

`173866d853a396050e252b227b2a1fb6769b32c75c98fa00e1b4929375c172b3`

**Target-logit SHA-256**

`537fa43c8e278b8678de978c1dc2ec4ee5a1c77ca0a917dbf6a5823cf77ea58f`

**Target-rank SHA-256**

`f4de73b588c465e0c1e41887a2eab7b1e0f2214d2d15dc0108b69e4063e2d472`

---

# 42. Primary artifacts

The principal experiment artifacts included:

`/content/ettr_ctl_llama/results/llama_colab_authentication_verification_v2.json`

`/content/ettr_ctl_llama/results/llama_phase1c3_target_position_operationalization_audit.json`

`/content/ettr_ctl_llama/results/llama_phase1d0_state_extraction_audit.json`

`/content/ettr_ctl_llama/results/llama_phase1d2_full_state_bank.npz`

`/content/ettr_ctl_llama/results/llama_phase1f2a_state_bank_row_order_audit.json`

`/content/ettr_ctl_llama/checkpoints/llama_phase1e1_selected_transport_maps.pkl`

`/content/ettr_ctl_llama/results/llama_phase1e1_transport_candidate_selection.json`

Additional phase-specific CTL and numerical result artifacts were retained in the experiment result directory.

Large numerical artifacts should be archived separately from the GitHub scientific record where appropriate.

---

# 43. Recommended archival interpretation

This record should be treated as the historical scientific record of ETTR-CTL-LLAMA-1.

Subsequent experiments should not silently modify the conclusions recorded here.

If a methodological error is discovered later, the appropriate procedure is to create a new experiment or explicit revision record while preserving this historical record.

The experiment should therefore remain identifiable by:

`ETTR-CTL-LLAMA-1`

and future reruns should receive new experiment identifiers.

---

# 44. Final scientific statement

ETTR-CTL-LLAMA-1 demonstrates that a substantial contemporary Transformer architecture can serve as a numerical substrate for an explicit ETTR/CTL operational framework, including context indexing, sector-specific representation extraction, finite transport estimation, and formal CTL structural interfaces.

However, the experiment does not establish that the Transformer empirically realizes the complete mathematical CTL structure.

The frozen transport maps do not generalize uniformly across sectors. The primary numerical triadic predictive comparison does not support a triadic predictive advantage, while the secondary target-rank result remains unresolved. Formal CTL irreducibility, empirical admissibility, empirical coherence, and `Phi_C` reconstruction remain unestablished.

These negative and unresolved outcomes are retained as part of the experiment's scientific content.

The experiment is therefore closed with the following high-level classification:

**FORMAL CTL STRUCTURE: SUPPORTED OPERATIONALLY**

**FROZEN TRANSPORT: HETEROGENEOUS / MIXED-NULL HELD-OUT GENERALIZATION**

**NUMERICAL TRIADIC ADVANTAGE: NOT SUPPORTED**

**CTL TRIADIC IRREDUCIBILITY: NOT ESTABLISHED**

**EMPIRICAL `K_C`: NOT IDENTIFIED**

**EMPIRICAL `Phi_C`: NOT RECONSTRUCTED**

**GEOMETRIC REALIZATION: NOT TESTED**

**OVERALL: MIXED EMPIRICAL RESULT; SCIENTIFICALLY CLOSED**
"""


# ------------------------------------------------------------
# Write file
# ------------------------------------------------------------

record_path = ARCHIVE / "scientific_record.md"

with open(record_path, "w", encoding="utf-8") as f:
    f.write(scientific_record)

print("=" * 70)
print("SCIENTIFIC RECORD CREATED")
print("=" * 70)
print()
print(record_path)
print()
print(f"Characters: {len(scientific_record):,}")
print(f"Lines:      {len(scientific_record.splitlines()):,}")
print()
print("Archive contents:")
for path in sorted(ARCHIVE.iterdir()):
    print("  ", path.name)

SCIENTIFIC RECORD CREATED

/content/github_archive/ETTR-CTL-LLAMA-1/scientific_record.md

Characters: 44,109
Lines:      1,714

Archive contents:
   README.md
   dataset_manifest.json
   experiment_config.json
   hashes.txt
   results_summary.json
   scientific_record.md


In [60]:
import shutil

zip_path = shutil.make_archive(
    "/content/ETTR-CTL-LLAMA-1-GitHub",
    "zip",
    root_dir="/content/github_archive"
)

print(zip_path)

/content/ETTR-CTL-LLAMA-1-GitHub.zip


In [61]:
from google.colab import files

files.download("/content/ETTR-CTL-LLAMA-1-GitHub.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>